# Notebook 06 — Player Data Processing and Identity Reconciliation

This notebook prepares player-level performance datasets from Cricsheet ODI records and Afghanistan scorecard data. It builds a unified CricketIQ player identity system, validates player references and performance records, and prepares versioned datasets for downstream analysis.

In [1]:
import sys

print(sys.executable)

c:\CricIQ\venv\Scripts\python.exe


In [2]:
import pandas as pd
import numpy as np
import json
import zipfile
from pathlib import Path

PROJECT_DIR = Path(r"C:\CricIQ")

CRICSHEET_DIR = PROJECT_DIR / "data" / "processed" / "cricsheet_odi"
AFGHANISTAN_DIR = PROJECT_DIR / "data" / "processed" / "afghanistan_odi"

# Load existing Cricsheet datasets
matches_df = pd.read_csv(
    CRICSHEET_DIR / "cricsheet_odi_matches.csv"
)

innings_df = pd.read_csv(
    CRICSHEET_DIR / "cricsheet_odi_innings.csv"
)

deliveries_df = pd.read_csv(
    CRICSHEET_DIR / "cricsheet_odi_deliveries.csv"
)

# Load Afghanistan player scorecards
afg_batting_df = pd.read_csv(
    AFGHANISTAN_DIR / "afghanistan_odi_batting_clean.csv"
)

afg_bowling_df = pd.read_csv(
    AFGHANISTAN_DIR / "afghanistan_odi_bowling_clean.csv"
)

print("CRICSHEET DATA")
print("Matches:", matches_df.shape)
print("Innings:", innings_df.shape)
print("Deliveries:", deliveries_df.shape)

print("\nAFGHANISTAN PLAYER DATA")
print("Batting:", afg_batting_df.shape)
print("Bowling:", afg_bowling_df.shape)

CRICSHEET DATA
Matches: (2600, 17)
Innings: (5143, 7)
Deliveries: (1379686, 13)

AFGHANISTAN PLAYER DATA
Batting: (2719, 13)
Bowling: (1765, 15)


In [3]:
print("CRICSHEET DELIVERIES")
print(deliveries_df.columns.tolist())

print("\nCRICSHEET INNINGS")
print(innings_df.columns.tolist())

print("\nAFGHANISTAN BATTING")
print(afg_batting_df.columns.tolist())

print("\nAFGHANISTAN BOWLING")
print(afg_bowling_df.columns.tolist())

CRICSHEET DELIVERIES
['match_id', 'innings_number', 'batting_team', 'over', 'delivery', 'batter', 'bowler', 'non_striker', 'batter_runs', 'extra_runs', 'total_runs', 'extras', 'wickets']

CRICSHEET INNINGS
['match_id', 'innings_number', 'batting_team', 'total_runs', 'wickets_lost', 'delivery_records', 'target_runs']

AFGHANISTAN BATTING
['Batter', 'Dismissal', 'R', 'B', '4s', '6s', 'SR', 'innings_number', 'date', 'match_url', 'runs', 'calculated_sr', 'sr_difference']

AFGHANISTAN BOWLING
['Bowler', 'O', 'M', 'R', 'W', 'Econ', 'WD', 'NB', 'innings_number', 'date', 'match_url', 'total_balls', 'overs_decimal', 'calculated_economy', 'economy_difference']


In [4]:
print("CRICSHEET DELIVERY SAMPLE")
display(deliveries_df.head(5))

print("\nAFGHANISTAN BATTING SAMPLE")
display(afg_batting_df.head(5))

print("\nAFGHANISTAN BOWLING SAMPLE")
display(afg_bowling_df.head(5))

CRICSHEET DELIVERY SAMPLE


,match_id,innings_number,batting_team,over,delivery,batter,bowler,non_striker,batter_runs,extra_runs,total_runs,extras,wickets
0,1000887,1,Australia,0,0.1,DA Warner,Mohammad Amir,TM Head,0,0,0,NaN,NaN
1,1000887,1,Australia,0,0.2,DA Warner,Mohammad Amir,TM Head,0,0,0,NaN,NaN
2,1000887,1,Australia,0,0.3,DA Warner,Mohammad Amir,TM Head,0,0,0,NaN,NaN
3,1000887,1,Australia,0,0.4,DA Warner,Mohammad Amir,TM Head,0,0,0,NaN,NaN
4,1000887,1,Australia,0,0.5,DA Warner,Mohammad Amir,TM Head,0,1,1,{'wides': 1},NaN



AFGHANISTAN BATTING SAMPLE


,Batter,Dismissal,R,B,4s,6s,SR,innings_number,date,match_url,runs,calculated_sr,sr_difference
0,Paul Stirling,bowled b Khaliq Dad,6,12,1,0,50.00,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,6,50.000000,0.000000
1,James Hall,caught Karim Sadiq b Shapoor Zadran,15,29,2,0,51.72,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,15,51.724138,0.004138
2,Alex Cusack,caught Shabir Noori b Khaliq Dad,2,17,0,0,11.76,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,2,11.764706,0.004706
3,Kevin O'Brien,caught Hamid Hassan b Samiullah Shinwari,44,68,5,0,64.70,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,44,64.705882,0.005882
4,Andrew Poynter,caught Karim Sadiq b Hamid Hassan,78,109,6,1,71.55,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,78,71.559633,0.009633



AFGHANISTAN BOWLING SAMPLE


,Bowler,O,M,R,W,Econ,WD,NB,innings_number,date,match_url,total_balls,overs_decimal,calculated_economy,economy_difference
0,Shapoor Zadran,10.0,3,39,1,3.90,1,1,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,60,10.0,3.900,0.000
1,Khaliq Dad,8.0,1,27,2,3.37,4,1,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,48,8.0,3.375,0.005
2,Hamid Hassan,8.0,0,44,3,5.50,0,1,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,48,8.0,5.500,0.000
3,Mohammad Nabi,10.0,0,55,1,5.50,1,0,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,60,10.0,5.500,0.000
4,Samiullah Shinwari,10.0,0,52,1,5.20,0,0,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,60,10.0,5.200,0.000


## 1. Cricsheet Player and Performance Data Extraction

Extract player identities and match participation from Cricsheet ODI JSON files. Construct player lookups and derive batting, bowling, and dismissal-related performance records.

In [5]:
# Cricsheet player information
cricsheet_players = pd.concat([
    deliveries_df["batter"],
    deliveries_df["bowler"],
    deliveries_df["non_striker"]
]).dropna()

print("CRICSHEET PLAYER INFORMATION")
print("Unique player names:", cricsheet_players.nunique())
print("Total player references:", len(cricsheet_players))

print("\nMissing values in Cricsheet player columns:")
print(deliveries_df[
    ["batter", "bowler", "non_striker"]
].isnull().sum())

# Afghanistan player information
print("\nAFGHANISTAN PLAYER INFORMATION")
print("Unique batters:", afg_batting_df["Batter"].nunique())
print("Unique bowlers:", afg_bowling_df["Bowler"].nunique())

print("\nMissing values in Afghanistan batting:")
print(afg_batting_df[["Batter", "R", "B", "4s", "6s"]].isnull().sum())

print("\nMissing values in Afghanistan bowling:")
print(afg_bowling_df[["Bowler", "O", "R", "W"]].isnull().sum())

CRICSHEET PLAYER INFORMATION
Unique player names: 2008
Total player references: 4139058

Missing values in Cricsheet player columns:
batter         0
bowler         0
non_striker    0
dtype: int64

AFGHANISTAN PLAYER INFORMATION
Unique batters: 403
Unique bowlers: 303

Missing values in Afghanistan batting:
Batter    0
R         0
B         0
4s        0
6s        0
dtype: int64

Missing values in Afghanistan bowling:
Bowler    0
O         0
R         0
W         0
dtype: int64


In [6]:
import json
import zipfile

ZIP_FILE = PROJECT_DIR / "data" / "raw" / "cricsheet_json" / "odis_json.zip"

with zipfile.ZipFile(ZIP_FILE, "r") as z:
    json_files = [
        name for name in z.namelist()
        if name.endswith(".json")
    ]

    sample_file = json_files[0]

    with z.open(sample_file) as file:
        sample_match = json.load(file)

print("Sample JSON file:", sample_file)

print("\nTop-level JSON sections:")
print(sample_match.keys())

print("\nPlayer registry:")
print(sample_match.get("meta", {}).get("data_version"))

registry = sample_match.get("info", {}).get("registry", {})
print("\nRegistry structure:")
print(registry)

Sample JSON file: 1000887.json

Top-level JSON sections:
dict_keys(['meta', 'info', 'innings'])

Player registry:
1.2.0

Registry structure:
{'people': {'Azhar Ali': '211de69f', 'B Stanlake': '6834d1f2', 'Babar Azam': '8a75e999', 'C Shamshuddin': '0f9d921b', 'CA Lynn': '45eda7c8', 'CB Gaffaney': 'd5ac41d8', 'DA Warner': 'dcce6f09', 'GJ Maxwell': 'b681e71e', 'Hasan Ali': '2911de16', 'Imad Wasim': '9cb8d7a6', 'JJ Crowe': '2e760301', 'JP Faulkner': '808f425a', 'MA Starc': '3fb19989', 'MD Martell': '487c072e', 'MR Marsh': '3d8feaf8', 'MS Wade': 'afa7e784', 'Mohammad Amir': 'e174dadd', 'Mohammad Hafeez': '9ab63e7b', 'Mohammad Nawaz (3)': '3086f7a4', 'Mohammad Rizwan': '2f26ac1a', 'PJ Cummins': 'ded9240e', 'SJ Nogajski': '9b3f9323', 'SPD Smith': '30a45b23', 'Sharjeel Khan': '193ef196', 'TM Head': '12b610c2', 'Umar Akmal': 'fd2bf2a0', 'Wahab Riaz': 'b3118300'}}


In [7]:
# Extract player registries from all Cricsheet ODI JSON files

player_registry_records = []

with zipfile.ZipFile(ZIP_FILE, "r") as z:

    for file_name in json_files:

        with z.open(file_name) as file:
            match_data = json.load(file)

        match_id = Path(file_name).stem

        registry = (
            match_data
            .get("info", {})
            .get("registry", {})
            .get("people", {})
        )

        for player_name, player_id in registry.items():

            player_registry_records.append({
                "match_id": match_id,
                "player_name": player_name,
                "player_id": player_id
            })

player_registry_df = pd.DataFrame(player_registry_records)

print("Registry extraction completed!")
print("Total registry records:", len(player_registry_df))
print("Unique player names:", player_registry_df["player_name"].nunique())
print("Unique player IDs:", player_registry_df["player_id"].nunique())

display(player_registry_df.head(10))

Registry extraction completed!
Total registry records: 86666
Unique player names: 3282
Unique player IDs: 3287


,match_id,player_name,player_id
0,1000887,Azhar Ali,211de69f
1,1000887,B Stanlake,6834d1f2
2,1000887,Babar Azam,8a75e999
3,1000887,C Shamshuddin,0f9d921b
4,1000887,CA Lynn,45eda7c8
5,1000887,CB Gaffaney,d5ac41d8
6,1000887,DA Warner,dcce6f09
7,1000887,GJ Maxwell,b681e71e
8,1000887,Hasan Ali,2911de16
9,1000887,Imad Wasim,9cb8d7a6


In [8]:
# Check whether one player name has multiple IDs

name_id_counts = (
    player_registry_df.groupby("player_name")["player_id"]
    .nunique()
    .sort_values(ascending=False)
)

multiple_ids = name_id_counts[name_id_counts > 1]

# Check whether one player ID has multiple names

id_name_counts = (
    player_registry_df.groupby("player_id")["player_name"]
    .nunique()
    .sort_values(ascending=False)
)

multiple_names = id_name_counts[id_name_counts > 1]

print("Player names associated with multiple IDs:", len(multiple_ids))
print("Player IDs associated with multiple names:", len(multiple_names))

print("\nNames with multiple IDs:")
display(multiple_ids)

print("\nIDs associated with multiple names:")
display(multiple_names)

Player names associated with multiple IDs: 12
Player IDs associated with multiple names: 7

Names with multiple IDs:


player_name
D Joseph           2
Abdul Muqeet       2
Junaid Siddique    2
Haider Ali         2
C de Lange         2
A Sharma           2
Ali Khan           2
RP Singh           2
Rahat Ali          2
SR Taylor          2
Shoaib Khan        2
Tanvir Ahmed       2
Name: player_id, dtype: int64


IDs associated with multiple names:


player_id
7c0055c1    2
5230a33d    2
3086f7a4    2
a9231c3f    2
bafb9238    2
e237b28c    2
f3a18a0c    2
Name: player_name, dtype: int64

In [9]:
# Display all conflicting name-ID combinations

conflicting_names = player_registry_df[
    player_registry_df["player_name"].isin(multiple_ids.index)
].drop_duplicates().sort_values("player_name")

print("PLAYER NAMES WITH MULTIPLE IDs")
display(conflicting_names)

# Display all conflicting ID-name combinations

conflicting_ids = player_registry_df[
    player_registry_df["player_id"].isin(multiple_names.index)
].drop_duplicates().sort_values("player_id")

print("\nPLAYER IDs WITH MULTIPLE NAMES")
display(conflicting_ids)

PLAYER NAMES WITH MULTIPLE IDs


,match_id,player_name,player_id
37957,1481531,A Sharma,5661ef90
68871,594901,A Sharma,a3ead8da
38036,1481534,A Sharma,5661ef90
27021,1377005,A Sharma,5661ef90
43173,1546002,A Sharma,5661ef90
...,...,...,...
25258,1351399,Tanvir Ahmed,ea2df1f1
25201,1351397,Tanvir Ahmed,ea2df1f1
24183,1340847,Tanvir Ahmed,ea2df1f1
24132,1340845,Tanvir Ahmed,ea2df1f1



PLAYER IDs WITH MULTIPLE NAMES


,match_id,player_name,player_id
18,1000887,Mohammad Nawaz (3),3086f7a4
28665,1384399,Mohammad Nawaz,3086f7a4
40508,1502168,Mohammad Nawaz,3086f7a4
40481,1502167,Mohammad Nawaz,3086f7a4
28742,1384403,Mohammad Nawaz,3086f7a4
...,...,...,...
11369,1168023,NR Sciver,f3a18a0c
11341,1168022,NR Sciver,f3a18a0c
11286,1168017,NR Sciver,f3a18a0c
34459,1448394,NR Sciver-Brunt,f3a18a0c


In [10]:
# Create a unique player master table

player_master_df = (
    player_registry_df
    .groupby("player_id")
    .agg(
        player_names=("player_name", lambda x: sorted(set(x))),
        name_count=("player_name", "nunique"),
        registry_match_count=("match_id", "nunique")
    )
    .reset_index()
)

# Create a preferred display name for convenience
player_master_df["player_name"] = (
    player_master_df["player_names"].apply(lambda x: x[-1])
)

print("Player master table created!")

print("Unique player IDs:", len(player_master_df))
print("Players with multiple registered names:",
      (player_master_df["name_count"] > 1).sum())

print("\nPlayer master table sample:")
display(player_master_df.head(10))

Player master table created!
Unique player IDs: 3287
Players with multiple registered names: 7

Player master table sample:


,player_id,player_names,name_count,registry_match_count,player_name
0,004c9e85,[PJ Hughes],1,25,PJ Hughes
1,005f0561,[Panna Ghosh],1,12,Panna Ghosh
2,006de9ca,[Zulfiqar Jan],1,3,Zulfiqar Jan
3,0070ed27,[Glen Walklin],1,1,Glen Walklin
4,00812511,[Lui Els],1,1,Lui Els
5,008589a4,[CM Wright],1,7,CM Wright
6,00d69fa4,[SO Tikolo],1,39,SO Tikolo
7,00ea847a,[MA Agarwal],1,5,MA Agarwal
8,00ea9791,[HP Tillakaratne],1,16,HP Tillakaratne
9,012829ff,[JW Hastings],1,29,JW Hastings


In [11]:
# Create player data output directory

PLAYER_DIR = PROJECT_DIR / "data" / "processed" / "player_data"
PLAYER_DIR.mkdir(parents=True, exist_ok=True)

# Save player master table
PLAYER_MASTER_FILE = PLAYER_DIR / "cricsheet_player_master.csv"

player_master_df.to_csv(PLAYER_MASTER_FILE, index=False)

print("Player master table saved!")
print("Location:", PLAYER_MASTER_FILE)
print("Records:", len(player_master_df))

Player master table saved!
Location: C:\CricIQ\data\processed\player_data\cricsheet_player_master.csv
Records: 3287


In [12]:
# Create player name-to-ID lookup

player_lookup = (
    player_registry_df[
        ["player_name", "player_id"]
    ]
    .drop_duplicates()
)

# Check whether each delivery player name has a unique ID
lookup_id_counts = (
    player_lookup.groupby("player_name")["player_id"]
    .nunique()
)

ambiguous_names = lookup_id_counts[lookup_id_counts > 1]

print("Unique delivery player names:", cricsheet_players.nunique())
print("Names with multiple registry IDs:", len(ambiguous_names))

# Check unmatched player names
registry_names = set(player_lookup["player_name"])

delivery_names = set(cricsheet_players.unique())

unmatched_names = delivery_names - registry_names

print("Unmatched delivery player names:", len(unmatched_names))
print("\nUnmatched names:")
print(sorted(unmatched_names)[:30])

Unique delivery player names: 2008
Names with multiple registry IDs: 12
Unmatched delivery player names: 0

Unmatched names:
[]


In [13]:
# Create match-specific player lookup

match_player_lookup = (
    player_registry_df[
        ["match_id", "player_name", "player_id"]
    ]
    .drop_duplicates()
)

# Check whether each match-player combination has one ID
match_player_id_counts = (
    match_player_lookup
    .groupby(["match_id", "player_name"])["player_id"]
    .nunique()
)

ambiguous_match_players = match_player_id_counts[
    match_player_id_counts > 1
]

print("Match-specific registry records:", len(match_player_lookup))

print(
    "Match-player combinations with multiple IDs:",
    len(ambiguous_match_players)
)

print("\nSample lookup records:")
display(match_player_lookup.head(10))# Create match-specific player lookup

match_player_lookup = (
    player_registry_df[
        ["match_id", "player_name", "player_id"]
    ]
    .drop_duplicates()
)

# Check whether each match-player combination has one ID
match_player_id_counts = (
    match_player_lookup
    .groupby(["match_id", "player_name"])["player_id"]
    .nunique()
)

ambiguous_match_players = match_player_id_counts[
    match_player_id_counts > 1
]

print("Match-specific registry records:", len(match_player_lookup))

print(
    "Match-player combinations with multiple IDs:",
    len(ambiguous_match_players)
)

print("\nSample lookup records:")
display(match_player_lookup.head(10))

Match-specific registry records: 86666
Match-player combinations with multiple IDs: 0

Sample lookup records:


,match_id,player_name,player_id
0,1000887,Azhar Ali,211de69f
1,1000887,B Stanlake,6834d1f2
2,1000887,Babar Azam,8a75e999
3,1000887,C Shamshuddin,0f9d921b
4,1000887,CA Lynn,45eda7c8
5,1000887,CB Gaffaney,d5ac41d8
6,1000887,DA Warner,dcce6f09
7,1000887,GJ Maxwell,b681e71e
8,1000887,Hasan Ali,2911de16
9,1000887,Imad Wasim,9cb8d7a6


Match-specific registry records: 86666
Match-player combinations with multiple IDs: 0

Sample lookup records:


,match_id,player_name,player_id
0,1000887,Azhar Ali,211de69f
1,1000887,B Stanlake,6834d1f2
2,1000887,Babar Azam,8a75e999
3,1000887,C Shamshuddin,0f9d921b
4,1000887,CA Lynn,45eda7c8
5,1000887,CB Gaffaney,d5ac41d8
6,1000887,DA Warner,dcce6f09
7,1000887,GJ Maxwell,b681e71e
8,1000887,Hasan Ali,2911de16
9,1000887,Imad Wasim,9cb8d7a6


In [14]:
# Create separate lookup tables
batter_lookup = match_player_lookup.rename(
    columns={"player_name": "batter", "player_id": "batter_id"}
)

bowler_lookup = match_player_lookup.rename(
    columns={"player_name": "bowler", "player_id": "bowler_id"}
)

non_striker_lookup = match_player_lookup.rename(
    columns={"player_name": "non_striker", "player_id": "non_striker_id"}
)

# Convert match IDs to strings for consistent merging
deliveries_df["match_id"] = deliveries_df["match_id"].astype(str)

for lookup in [batter_lookup, bowler_lookup, non_striker_lookup]:
    lookup["match_id"] = lookup["match_id"].astype(str)

# Attach player IDs
deliveries_df = deliveries_df.merge(
    batter_lookup,
    on=["match_id", "batter"],
    how="left",
    validate="many_to_one"
)

deliveries_df = deliveries_df.merge(
    bowler_lookup,
    on=["match_id", "bowler"],
    how="left",
    validate="many_to_one"
)

deliveries_df = deliveries_df.merge(
    non_striker_lookup,
    on=["match_id", "non_striker"],
    how="left",
    validate="many_to_one"
)

print("Player IDs attached successfully!")

print("\nMissing player IDs:")
print(
    deliveries_df[
        ["batter_id", "bowler_id", "non_striker_id"]
    ].isnull().sum()
)

print("\nUpdated delivery dataset shape:", deliveries_df.shape)

display(
    deliveries_df[
        ["match_id", "batter", "batter_id",
         "bowler", "bowler_id",
         "non_striker", "non_striker_id"]
    ].head(10)
)

Player IDs attached successfully!

Missing player IDs:
batter_id         0
bowler_id         0
non_striker_id    0
dtype: int64

Updated delivery dataset shape: (1379686, 16)


,match_id,batter,batter_id,bowler,bowler_id,non_striker,non_striker_id
0,1000887,DA Warner,dcce6f09,Mohammad Amir,e174dadd,TM Head,12b610c2
1,1000887,DA Warner,dcce6f09,Mohammad Amir,e174dadd,TM Head,12b610c2
2,1000887,DA Warner,dcce6f09,Mohammad Amir,e174dadd,TM Head,12b610c2
3,1000887,DA Warner,dcce6f09,Mohammad Amir,e174dadd,TM Head,12b610c2
4,1000887,DA Warner,dcce6f09,Mohammad Amir,e174dadd,TM Head,12b610c2
5,1000887,DA Warner,dcce6f09,Mohammad Amir,e174dadd,TM Head,12b610c2
6,1000887,DA Warner,dcce6f09,Mohammad Amir,e174dadd,TM Head,12b610c2
7,1000887,TM Head,12b610c2,Mohammad Hafeez,9ab63e7b,DA Warner,dcce6f09
8,1000887,TM Head,12b610c2,Mohammad Hafeez,9ab63e7b,DA Warner,dcce6f09
9,1000887,DA Warner,dcce6f09,Mohammad Hafeez,9ab63e7b,TM Head,12b610c2


In [15]:
# Extract player participation from delivery records

batter_participation = deliveries_df[
    ["match_id", "batting_team", "batter", "batter_id"]
].rename(columns={
    "batter": "player_name",
    "batter_id": "player_id"
})

non_striker_participation = deliveries_df[
    ["match_id", "batting_team", "non_striker", "non_striker_id"]
].rename(columns={
    "non_striker": "player_name",
    "non_striker_id": "player_id"
})

bowler_participation = deliveries_df[
    ["match_id", "bowler", "bowler_id"]
].rename(columns={
    "bowler": "player_name",
    "bowler_id": "player_id"
})

# Combine all player roles
player_participation_df = pd.concat([
    batter_participation,
    non_striker_participation,
    bowler_participation
], ignore_index=True)

# Remove duplicate player appearances within a match
player_participation_df = (
    player_participation_df
    .drop_duplicates(subset=["match_id", "player_id"])
    .reset_index(drop=True)
)

print("Player participation dataset created!")
print("Unique player-match records:", len(player_participation_df))
print("Unique players:", player_participation_df["player_id"].nunique())
print("Unique matches:", player_participation_df["match_id"].nunique())

print("\nMissing values:")
print(player_participation_df.isnull().sum())

display(player_participation_df.head(10))

Player participation dataset created!
Unique player-match records: 55172
Unique players: 2008
Unique matches: 2600

Missing values:
match_id           0
batting_team    9284
player_name        0
player_id          0
dtype: int64


,match_id,batting_team,player_name,player_id
0,1000887,Australia,DA Warner,dcce6f09
1,1000887,Australia,TM Head,12b610c2
2,1000887,Australia,SPD Smith,30a45b23
3,1000887,Australia,CA Lynn,45eda7c8
4,1000887,Australia,MR Marsh,3d8feaf8
5,1000887,Australia,GJ Maxwell,b681e71e
6,1000887,Australia,MS Wade,afa7e784
7,1000887,Australia,JP Faulkner,808f425a
8,1000887,Australia,PJ Cummins,ded9240e
9,1000887,Australia,MA Starc,3fb19989


In [16]:
# Build match and batting-team lookup
match_teams = matches_df[
    ["match_id", "team_1", "team_2"]
].copy()

match_teams["match_id"] = match_teams["match_id"].astype(str)

# Create innings-level batting team lookup
innings_team_lookup = innings_df[
    ["match_id", "innings_number", "batting_team"]
].copy()

innings_team_lookup["match_id"] = (
    innings_team_lookup["match_id"].astype(str)
)

# Attach batting team to delivery records
delivery_team_info = deliveries_df[
    ["match_id", "innings_number", "batting_team",
     "bowler_id"]
].drop_duplicates()

# Find the bowling team from the two match teams
delivery_team_info = delivery_team_info.merge(
    match_teams,
    on="match_id",
    how="left"
)

delivery_team_info["bowling_team"] = np.where(
    delivery_team_info["batting_team"] == delivery_team_info["team_1"],
    delivery_team_info["team_2"],
    delivery_team_info["team_1"]
)

print("Delivery team information created.")

print("Missing bowling teams:",
      delivery_team_info["bowling_team"].isnull().sum())

display(delivery_team_info.head())

Delivery team information created.
Missing bowling teams: 0


,match_id,innings_number,batting_team,bowler_id,team_1,team_2,bowling_team
0,1000887,1,Australia,e174dadd,Australia,Pakistan,Pakistan
1,1000887,1,Australia,9ab63e7b,Australia,Pakistan,Pakistan
2,1000887,1,Australia,2911de16,Australia,Pakistan,Pakistan
3,1000887,1,Australia,b3118300,Australia,Pakistan,Pakistan
4,1000887,1,Australia,9cb8d7a6,Australia,Pakistan,Pakistan


In [17]:
# 1. Batter participation
batter_participation = deliveries_df[
    ["match_id", "batting_team", "batter", "batter_id"]
].rename(columns={
    "batter": "player_name",
    "batter_id": "player_id"
})

# 2. Non-striker participation
non_striker_participation = deliveries_df[
    ["match_id", "batting_team", "non_striker", "non_striker_id"]
].rename(columns={
    "non_striker": "player_name",
    "non_striker_id": "player_id"
})

# 3. Bowler participation with fielding team
bowler_participation = deliveries_df[
    ["match_id", "bowler", "bowler_id", "batting_team"]
].merge(
    delivery_team_info[
        ["match_id", "bowler_id", "batting_team", "bowling_team"]
    ].drop_duplicates(),
    on=["match_id", "bowler_id", "batting_team"],
    how="left",
    validate="many_to_one"
)

bowler_participation = bowler_participation[
    ["match_id", "bowling_team", "bowler", "bowler_id"]
].rename(columns={
    "bowling_team": "batting_team",
    "bowler": "player_name",
    "bowler_id": "player_id"
})

# 4. Combine all roles
player_participation_df = pd.concat([
    batter_participation,
    non_striker_participation,
    bowler_participation
], ignore_index=True)

# 5. Remove duplicate player appearances within a match
player_participation_df = (
    player_participation_df
    .drop_duplicates(subset=["match_id", "player_id"])
    .reset_index(drop=True)
)

print("Updated player participation dataset:")
print("Shape:", player_participation_df.shape)
print("Unique players:", player_participation_df["player_id"].nunique())
print("Unique matches:", player_participation_df["match_id"].nunique())

print("\nMissing values:")
print(player_participation_df.isnull().sum())

print("\nPlayers by team:")
display(
    player_participation_df.groupby("batting_team")["player_id"]
    .nunique()
    .sort_values(ascending=False)
    .head(10)
)

Updated player participation dataset:
Shape: (55172, 4)
Unique players: 2008
Unique matches: 2600

Missing values:
match_id        0
batting_team    0
player_name     0
player_id       0
dtype: int64

Players by team:


batting_team
Pakistan        147
West Indies     142
India           137
England         133
Sri Lanka       133
Australia       127
South Africa    122
New Zealand     118
Zimbabwe        115
Bangladesh      109
Name: player_id, dtype: int64

In [18]:
# Check whether a player is assigned to multiple teams in one match

player_team_counts = (
    player_participation_df
    .groupby(["match_id", "player_id"])["batting_team"]
    .nunique()
)

conflicting_assignments = player_team_counts[
    player_team_counts > 1
]

print("Player-match records:", len(player_participation_df))
print("Conflicting player-team assignments:", len(conflicting_assignments))

if len(conflicting_assignments) > 0:
    print("\nExamples of conflicts:")
    conflict_keys = conflicting_assignments.index
    display(
        player_participation_df.set_index(
            ["match_id", "player_id"]
        ).loc[conflict_keys].head(20)
    )
else:
    print("All players have consistent team assignments within each match.")

Player-match records: 55172
Conflicting player-team assignments: 0
All players have consistent team assignments within each match.


In [19]:
# Create a working copy
batting_deliveries = deliveries_df.copy()

# Identify wides and no-balls from the extras dictionary
def get_extra_type(extras, extra_type):
    if isinstance(extras, dict):
        return extras.get(extra_type, 0)
    return 0

batting_deliveries["wides"] = batting_deliveries["extras"].apply(
    lambda x: get_extra_type(x, "wides")
)

batting_deliveries["no_balls"] = batting_deliveries["extras"].apply(
    lambda x: get_extra_type(x, "noballs")
)

# A batter faces a ball unless it is a wide
batting_deliveries["balls_faced"] = (
    batting_deliveries["wides"] == 0
).astype(int)

# Calculate boundaries
batting_deliveries["fours"] = (
    (batting_deliveries["batter_runs"] == 4)
).astype(int)

batting_deliveries["sixes"] = (
    (batting_deliveries["batter_runs"] == 6)
).astype(int)

# Aggregate batting performance
player_batting_df = (
    batting_deliveries
    .groupby([
        "match_id",
        "innings_number",
        "batting_team",
        "batter",
        "batter_id"
    ])
    .agg(
        runs=("batter_runs", "sum"),
        balls_faced=("balls_faced", "sum"),
        fours=("fours", "sum"),
        sixes=("sixes", "sum")
    )
    .reset_index()
)

# Calculate strike rate
player_batting_df["strike_rate"] = np.where(
    player_batting_df["balls_faced"] > 0,
    player_batting_df["runs"] / player_batting_df["balls_faced"] * 100,
    np.nan
)

print("Player batting dataset created!")
print("Shape:", player_batting_df.shape)

print("\nMissing values:")
print(player_batting_df.isnull().sum())

display(player_batting_df.head(10))

Player batting dataset created!
Shape: (45432, 10)

Missing values:
match_id          0
innings_number    0
batting_team      0
batter            0
batter_id         0
runs              0
balls_faced       0
fours             0
sixes             0
strike_rate       0
dtype: int64


,match_id,innings_number,batting_team,batter,batter_id,runs,balls_faced,fours,sixes,strike_rate
0,1000887,1,Australia,B Stanlake,6834d1f2,1,6,0,0,16.666667
1,1000887,1,Australia,CA Lynn,45eda7c8,16,14,1,1,114.285714
2,1000887,1,Australia,DA Warner,dcce6f09,7,19,1,0,36.842105
3,1000887,1,Australia,GJ Maxwell,b681e71e,60,58,7,0,103.448276
4,1000887,1,Australia,JP Faulkner,808f425a,5,13,0,0,38.461538
5,1000887,1,Australia,MA Starc,3fb19989,10,8,0,1,125.000000
6,1000887,1,Australia,MR Marsh,3d8feaf8,4,17,0,0,23.529412
7,1000887,1,Australia,MS Wade,afa7e784,100,102,7,2,98.039216
8,1000887,1,Australia,PJ Cummins,ded9240e,15,33,0,0,45.454545
9,1000887,1,Australia,SPD Smith,30a45b23,0,1,0,0,0.000000


In [21]:
# Standardise match IDs before merging
innings_df["match_id"] = innings_df["match_id"].astype(str)

calculated_batting_totals["match_id"] = (
    calculated_batting_totals["match_id"].astype(str)
)

# Compare against Cricsheet innings totals
batting_validation_df = innings_df.merge(
    calculated_batting_totals,
    on=["match_id", "innings_number"],
    how="left",
    validate="one_to_one"
)

# Calculate difference between innings total and batter runs
batting_validation_df["runs_difference"] = (
    batting_validation_df["total_runs"]
    - batting_validation_df["calculated_batter_runs"]
)

print("Total innings checked:", len(batting_validation_df))

print(
    "Missing calculated batting totals:",
    batting_validation_df["calculated_batter_runs"].isnull().sum()
)

print(
    "Innings with zero difference:",
    (batting_validation_df["runs_difference"] == 0).sum()
)

print(
    "Innings with non-zero difference:",
    (batting_validation_df["runs_difference"] != 0).sum()
)

print("\nSample discrepancies:")
display(
    batting_validation_df[
        batting_validation_df["runs_difference"] != 0
    ].head(10)
)

Total innings checked: 5143
Missing calculated batting totals: 0
Innings with zero difference: 24
Innings with non-zero difference: 5119

Sample discrepancies:


,match_id,innings_number,batting_team,total_runs,wickets_lost,delivery_records,target_runs,calculated_batter_runs,runs_difference
0,1000887,1,Australia,268,9,310,NaN,257,11
1,1000887,2,Pakistan,176,10,261,269.0,162,14
2,1000889,1,Australia,220,10,297,NaN,202,18
3,1000889,2,Pakistan,221,4,293,221.0,208,13
4,1000891,1,Pakistan,263,7,308,NaN,248,15
5,1000891,2,Australia,265,3,276,264.0,257,8
6,1000893,1,Australia,353,6,302,NaN,343,10
7,1000893,2,Pakistan,267,10,267,354.0,258,9
8,1000895,1,Australia,369,7,306,NaN,358,11
9,1000895,2,Pakistan,312,9,307,370.0,297,15


In [22]:
# Calculate total batter runs and extras from deliveries

delivery_totals = (
    deliveries_df
    .groupby(["match_id", "innings_number"])
    .agg(
        calculated_batter_runs=("batter_runs", "sum"),
        calculated_extras=("extra_runs", "sum")
    )
    .reset_index()
)

# Standardise match IDs
delivery_totals["match_id"] = delivery_totals["match_id"].astype(str)

# Merge with official innings totals
batting_validation_df = innings_df.merge(
    delivery_totals,
    on=["match_id", "innings_number"],
    how="left",
    validate="one_to_one"
)

# Calculate reconstructed innings total
batting_validation_df["reconstructed_total"] = (
    batting_validation_df["calculated_batter_runs"]
    + batting_validation_df["calculated_extras"]
)

# Compare reconstructed and official totals
batting_validation_df["total_difference"] = (
    batting_validation_df["total_runs"]
    - batting_validation_df["reconstructed_total"]
)

print("Total innings checked:", len(batting_validation_df))

print("\nExact matches:")
print(
    (batting_validation_df["total_difference"] == 0).sum()
)

print("\nMismatches:")
print(
    (batting_validation_df["total_difference"] != 0).sum()
)

print("\nMissing reconstructed totals:")
print(
    batting_validation_df["reconstructed_total"].isnull().sum()
)

print("\nSample mismatches:")
display(
    batting_validation_df[
        batting_validation_df["total_difference"] != 0
    ].head(10)
)

Total innings checked: 5143

Exact matches:
5143

Mismatches:
0

Missing reconstructed totals:
0

Sample mismatches:


,match_id,innings_number,batting_team,total_runs,wickets_lost,delivery_records,target_runs,calculated_batter_runs,calculated_extras,reconstructed_total,total_difference


In [23]:
print("Extras column data type:")
print(deliveries_df["extras"].dtype)

print("\nSample extras values:")
print(deliveries_df["extras"].dropna().head(15).tolist())

print("\nUnique extras representations:")
print(deliveries_df["extras"].dropna().apply(type).value_counts())

Extras column data type:
object

Sample extras values:
["{'wides': 1}", "{'wides': 1}", "{'wides': 1}", "{'wides': 1}", "{'wides': 1}", "{'wides': 1}", "{'wides': 1}", "{'legbyes': 1}", "{'wides': 1}", "{'noballs': 1}", "{'wides': 1}", "{'wides': 1}", "{'byes': 4}", "{'wides': 1}", "{'legbyes': 1}"]

Unique extras representations:
extras
<class 'str'>    53120
Name: count, dtype: int64


In [24]:
import ast

def parse_extras(value):
    if pd.isna(value):
        return {}
    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return {}

deliveries_df["extras_parsed"] = deliveries_df["extras"].apply(parse_extras)

print("Parsed extras samples:")
print(deliveries_df.loc[
    deliveries_df["extras"].notna(),
    ["extras", "extras_parsed"]
].head(10))

print("\nParsed extras data types:")
print(deliveries_df["extras_parsed"].apply(type).value_counts())

Parsed extras samples:
             extras   extras_parsed
4      {'wides': 1}    {'wides': 1}
35     {'wides': 1}    {'wides': 1}
50     {'wides': 1}    {'wides': 1}
107    {'wides': 1}    {'wides': 1}
181    {'wides': 1}    {'wides': 1}
186    {'wides': 1}    {'wides': 1}
199    {'wides': 1}    {'wides': 1}
236  {'legbyes': 1}  {'legbyes': 1}
263    {'wides': 1}    {'wides': 1}
266  {'noballs': 1}  {'noballs': 1}

Parsed extras data types:
extras_parsed
<class 'dict'>    1379686
Name: count, dtype: int64


In [25]:
batting_deliveries = deliveries_df.copy()

# Extract wides and no-balls from parsed extras
batting_deliveries["wides"] = batting_deliveries["extras_parsed"].apply(
    lambda x: x.get("wides", 0)
)

batting_deliveries["no_balls"] = batting_deliveries["extras_parsed"].apply(
    lambda x: x.get("noballs", 0)
)

# A wide does not count as a ball faced.
# A no-ball does count as a ball faced.
batting_deliveries["balls_faced"] = (
    batting_deliveries["wides"] == 0
).astype(int)

# Boundaries scored off the bat
batting_deliveries["fours"] = (
    batting_deliveries["batter_runs"] == 4
).astype(int)

batting_deliveries["sixes"] = (
    batting_deliveries["batter_runs"] == 6
).astype(int)

# Aggregate individual batting performance
player_batting_df = (
    batting_deliveries
    .groupby(
        ["match_id", "innings_number", "batting_team", "batter", "batter_id"]
    )
    .agg(
        runs=("batter_runs", "sum"),
        balls_faced=("balls_faced", "sum"),
        fours=("fours", "sum"),
        sixes=("sixes", "sum")
    )
    .reset_index()
)

# Calculate strike rate
player_batting_df["strike_rate"] = np.where(
    player_batting_df["balls_faced"] > 0,
    player_batting_df["runs"] / player_batting_df["balls_faced"] * 100,
    np.nan
)

print("Player batting dataset shape:", player_batting_df.shape)
print("\nSample records:")
display(player_batting_df.head(10))

print("\nTotal wides:", batting_deliveries["wides"].sum())
print("Total no-balls:", batting_deliveries["no_balls"].sum())
print("Total player batting runs:", player_batting_df["runs"].sum())

Player batting dataset shape: (45432, 10)

Sample records:


,match_id,innings_number,batting_team,batter,batter_id,runs,balls_faced,fours,sixes,strike_rate
0,1000887,1,Australia,B Stanlake,6834d1f2,1,6,0,0,16.666667
1,1000887,1,Australia,CA Lynn,45eda7c8,16,12,1,1,133.333333
2,1000887,1,Australia,DA Warner,dcce6f09,7,18,1,0,38.888889
3,1000887,1,Australia,GJ Maxwell,b681e71e,60,56,7,0,107.142857
4,1000887,1,Australia,JP Faulkner,808f425a,5,12,0,0,41.666667
5,1000887,1,Australia,MA Starc,3fb19989,10,8,0,1,125.000000
6,1000887,1,Australia,MR Marsh,3d8feaf8,4,17,0,0,23.529412
7,1000887,1,Australia,MS Wade,afa7e784,100,100,7,2,100.000000
8,1000887,1,Australia,PJ Cummins,ded9240e,15,32,0,0,46.875000
9,1000887,1,Australia,SPD Smith,30a45b23,0,1,0,0,0.000000



Total wides: 38377
Total no-balls: 5449
Total player batting runs: 1099490


In [26]:
# Aggregate player runs by match and innings
player_runs_validation = (
    player_batting_df
    .groupby(["match_id", "innings_number"])["runs"]
    .sum()
    .reset_index(name="player_runs")
)

# Aggregate batter runs directly from deliveries
delivery_runs_validation = (
    deliveries_df
    .groupby(["match_id", "innings_number"])["batter_runs"]
    .sum()
    .reset_index(name="delivery_batter_runs")
)

# Standardise match IDs
player_runs_validation["match_id"] = (
    player_runs_validation["match_id"].astype(str)
)
delivery_runs_validation["match_id"] = (
    delivery_runs_validation["match_id"].astype(str)
)

# Compare both calculations
runs_check = player_runs_validation.merge(
    delivery_runs_validation,
    on=["match_id", "innings_number"],
    how="outer",
    validate="one_to_one"
)

runs_check["difference"] = (
    runs_check["player_runs"] - runs_check["delivery_batter_runs"]
)

print("Total innings checked:", len(runs_check))
print("Exact matches:", (runs_check["difference"] == 0).sum())
print("Mismatches:", (runs_check["difference"] != 0).sum())
print("Missing values:", runs_check.isna().sum().sum())

print("\nSample mismatches:")
display(runs_check[runs_check["difference"] != 0].head())

Total innings checked: 5143
Exact matches: 5143
Mismatches: 0
Missing values: 0

Sample mismatches:


,match_id,innings_number,player_runs,delivery_batter_runs,difference


In [27]:
print("Wickets column data type:")
print(deliveries_df["wickets"].dtype)

print("\nSample wicket records:")
print(deliveries_df.loc[
    deliveries_df["wickets"].notna(),
    ["match_id", "innings_number", "batter", "wickets"]
].head(15).to_string(index=False))

print("\nWickets value types:")
print(deliveries_df["wickets"].dropna().apply(type).value_counts())

Wickets column data type:
object

Sample wicket records:
match_id  innings_number             batter                                                                                                            wickets
 1000887               1          DA Warner                                                                    [{'kind': 'bowled', 'player_out': 'DA Warner'}]
 1000887               1          SPD Smith                         [{'kind': 'caught', 'player_out': 'SPD Smith', 'fielders': [{'name': 'Mohammad Rizwan'}]}]
 1000887               1            CA Lynn                           [{'kind': 'caught', 'player_out': 'CA Lynn', 'fielders': [{'name': 'Mohammad Rizwan'}]}]
 1000887               1            TM Head                           [{'kind': 'caught', 'player_out': 'TM Head', 'fielders': [{'name': 'Mohammad Rizwan'}]}]
 1000887               1           MR Marsh                          [{'kind': 'caught', 'player_out': 'MR Marsh', 'fielders': [{'name': 'Mohammad R

In [28]:
def parse_wickets(value):
    if pd.isna(value):
        return []
    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return []

deliveries_df["wickets_parsed"] = deliveries_df["wickets"].apply(
    parse_wickets
)

print("Sample parsed wicket records:")

display(
    deliveries_df.loc[
        deliveries_df["wickets_parsed"].apply(len) > 0,
        ["batter", "wickets", "wickets_parsed"]
    ].head(10)
)

print("\nParsed value types:")
print(
    deliveries_df["wickets_parsed"]
    .apply(type)
    .value_counts()
)

print("\nTotal deliveries containing wicket records:")
print(
    (deliveries_df["wickets_parsed"].apply(len) > 0).sum()
)

Sample parsed wicket records:


,batter,wickets,wickets_parsed
27,DA Warner,"[{'kind': 'bowled', 'player_out': 'DA Warner'}]","[{'kind': 'bowled', 'player_out': 'DA Warner'}]"
28,SPD Smith,"[{'kind': 'caught', 'player_out': 'SPD Smith',...","[{'kind': 'caught', 'player_out': 'SPD Smith',..."
58,CA Lynn,"[{'kind': 'caught', 'player_out': 'CA Lynn', '...","[{'kind': 'caught', 'player_out': 'CA Lynn', '..."
80,TM Head,"[{'kind': 'caught', 'player_out': 'TM Head', '...","[{'kind': 'caught', 'player_out': 'TM Head', '..."
100,MR Marsh,"[{'kind': 'caught', 'player_out': 'MR Marsh', ...","[{'kind': 'caught', 'player_out': 'MR Marsh', ..."
191,GJ Maxwell,"[{'kind': 'caught', 'player_out': 'GJ Maxwell'...","[{'kind': 'caught', 'player_out': 'GJ Maxwell'..."
210,JP Faulkner,"[{'kind': 'caught', 'player_out': 'JP Faulkner...","[{'kind': 'caught', 'player_out': 'JP Faulkner..."
264,PJ Cummins,"[{'kind': 'run out', 'player_out': 'PJ Cummins...","[{'kind': 'run out', 'player_out': 'PJ Cummins..."
282,MA Starc,"[{'kind': 'bowled', 'player_out': 'MA Starc'}]","[{'kind': 'bowled', 'player_out': 'MA Starc'}]"
361,Sharjeel Khan,"[{'kind': 'caught', 'player_out': 'Sharjeel Kh...","[{'kind': 'caught', 'player_out': 'Sharjeel Kh..."



Parsed value types:
wickets_parsed
<class 'list'>    1379686
Name: count, dtype: int64

Total deliveries containing wicket records:
37770


In [29]:
dismissal_records = []

for _, row in deliveries_df.iterrows():
    for wicket in row["wickets_parsed"]:
        dismissal_records.append({
            "match_id": row["match_id"],
            "innings_number": row["innings_number"],
            "batting_team": row["batting_team"],
            "batter": row["batter"],
            "batter_id": row["batter_id"],
            "player_out": wicket.get("player_out"),
            "dismissal_type": wicket.get("kind"),
            "fielders": wicket.get("fielders", [])
        })

dismissals_df = pd.DataFrame(dismissal_records)

print("Dismissal dataset shape:", dismissals_df.shape)

print("\nDismissal types:")
print(dismissals_df["dismissal_type"].value_counts())

print("\nSample dismissals:")
display(dismissals_df.head(10))

Dismissal dataset shape: (37772, 8)

Dismissal types:
dismissal_type
caught                   21628
bowled                    7087
lbw                       4188
run out                   2729
caught and bowled         1138
stumped                    898
retired hurt                61
hit wicket                  36
obstructing the field        6
timed out                    1
Name: count, dtype: int64

Sample dismissals:


,match_id,innings_number,batting_team,batter,batter_id,player_out,dismissal_type,fielders
0,1000887,1,Australia,DA Warner,dcce6f09,DA Warner,bowled,[]
1,1000887,1,Australia,SPD Smith,30a45b23,SPD Smith,caught,[{'name': 'Mohammad Rizwan'}]
2,1000887,1,Australia,CA Lynn,45eda7c8,CA Lynn,caught,[{'name': 'Mohammad Rizwan'}]
3,1000887,1,Australia,TM Head,12b610c2,TM Head,caught,[{'name': 'Mohammad Rizwan'}]
4,1000887,1,Australia,MR Marsh,3d8feaf8,MR Marsh,caught,[{'name': 'Mohammad Rizwan'}]
5,1000887,1,Australia,GJ Maxwell,b681e71e,GJ Maxwell,caught,[{'name': 'Mohammad Hafeez'}]
6,1000887,1,Australia,JP Faulkner,808f425a,JP Faulkner,caught,[{'name': 'Mohammad Hafeez'}]
7,1000887,1,Australia,PJ Cummins,ded9240e,PJ Cummins,run out,"[{'name': 'Umar Akmal'}, {'name': 'Mohammad Am..."
8,1000887,1,Australia,MA Starc,3fb19989,MA Starc,bowled,[]
9,1000887,2,Pakistan,Sharjeel Khan,193ef196,Sharjeel Khan,caught,[{'name': 'GJ Maxwell'}]


In [30]:
# Check whether player_out matches the batter on the delivery
dismissals_df["same_as_batter"] = (
    dismissals_df["player_out"] == dismissals_df["batter"]
)

print("Total dismissal events:", len(dismissals_df))

print("\nPlayer-out matches delivery batter:")
print(dismissals_df["same_as_batter"].value_counts(dropna=False))

print("\nRecords where player_out differs from batter:")
display(
    dismissals_df.loc[
        ~dismissals_df["same_as_batter"],
        [
            "match_id",
            "innings_number",
            "batter",
            "batter_id",
            "player_out",
            "dismissal_type"
        ]
    ].head(15)
)

Total dismissal events: 37772

Player-out matches delivery batter:
same_as_batter
True     36447
False     1325
Name: count, dtype: int64

Records where player_out differs from batter:


,match_id,innings_number,batter,batter_id,player_out,dismissal_type
79,1001371,1,SPD Smith,30a45b23,MR Marsh,run out
136,1004283,2,Ehsan Khan,d3851cd8,Anshuman Rath,run out
141,1004285,1,CS MacLeod,9c376401,CD de Lange,run out
151,1004285,2,Tanwir Afzal,5f5ca1a7,Waqas Khan,run out
199,1019973,2,Mosaddek Hossain,8919756f,Mushfiqur Rahim,retired hurt
214,1019975,2,Imrul Kayes,5fbe14fc,Sabbir Rahman,run out
233,1019977,2,KS Williamson,d027ba9f,MJ Guptill,retired hurt
253,1020013,2,MP Stoinis,d9273ee7,JR Hazlewood,run out
296,1020033,2,D Pretorius,11614d87,CH Morris,run out
329,1020039,1,DG Brownlie,22bd731a,KS Williamson,run out


In [31]:
# Build match-specific player name-to-ID lookup
player_out_lookup = match_player_lookup.rename(
    columns={
        "player_name": "player_out",
        "player_id": "player_out_id"
    }
)

# Attach the correct player ID to each dismissal
dismissals_df["match_id"] = dismissals_df["match_id"].astype(str)
player_out_lookup["match_id"] = player_out_lookup["match_id"].astype(str)

dismissals_df = dismissals_df.merge(
    player_out_lookup,
    on=["match_id", "player_out"],
    how="left",
    validate="many_to_one"
)

print("Total dismissal events:", len(dismissals_df))

print(
    "Missing dismissed-player IDs:",
    dismissals_df["player_out_id"].isna().sum()
)

print("\nSample mapped dismissals:")
display(
    dismissals_df[
        [
            "batter",
            "player_out",
            "player_out_id",
            "dismissal_type"
        ]
    ].head(10)
)

Total dismissal events: 37772
Missing dismissed-player IDs: 0

Sample mapped dismissals:


,batter,player_out,player_out_id,dismissal_type
0,DA Warner,DA Warner,dcce6f09,bowled
1,SPD Smith,SPD Smith,30a45b23,caught
2,CA Lynn,CA Lynn,45eda7c8,caught
3,TM Head,TM Head,12b610c2,caught
4,MR Marsh,MR Marsh,3d8feaf8,caught
5,GJ Maxwell,GJ Maxwell,b681e71e,caught
6,JP Faulkner,JP Faulkner,808f425a,caught
7,PJ Cummins,PJ Cummins,ded9240e,run out
8,MA Starc,MA Starc,3fb19989,bowled
9,Sharjeel Khan,Sharjeel Khan,193ef196,caught


In [32]:
bowler_credited_types = [
    "bowled",
    "caught",
    "caught and bowled",
    "lbw",
    "stumped",
    "hit wicket"
]

dismissals_df["bowler_credited"] = (
    dismissals_df["dismissal_type"].isin(bowler_credited_types)
)

dismissals_df["batter_dismissed"] = (
    dismissals_df["dismissal_type"] != "retired hurt"
)

print("Bowler-credited dismissals:")
print(dismissals_df["bowler_credited"].value_counts())

print("\nBatter dismissal status:")
print(dismissals_df["batter_dismissed"].value_counts())

print("\nClassification by dismissal type:")
display(
    dismissals_df.groupby("dismissal_type").agg(
        total_events=("dismissal_type", "size"),
        bowler_credited=("bowler_credited", "sum"),
        batter_dismissals=("batter_dismissed", "sum")
    ).sort_values("total_events", ascending=False)
)

Bowler-credited dismissals:
bowler_credited
True     34975
False     2797
Name: count, dtype: int64

Batter dismissal status:
batter_dismissed
True     37711
False       61
Name: count, dtype: int64

Classification by dismissal type:


,total_events,bowler_credited,batter_dismissals
dismissal_type,,,
caught,21628,21628,21628
bowled,7087,7087,7087
lbw,4188,4188,4188
run out,2729,0,2729
caught and bowled,1138,1138,1138
stumped,898,898,898
retired hurt,61,0,0
hit wicket,36,36,36
obstructing the field,6,0,6


In [33]:
# Keep only actual batter dismissals
valid_dismissals = dismissals_df[
    dismissals_df["batter_dismissed"]
].copy()

# Aggregate dismissal statistics by player and match
player_dismissal_df = (
    valid_dismissals
    .groupby(
        [
            "match_id",
            "innings_number",
            "batting_team",
            "player_out",
            "player_out_id"
        ]
    )
    .agg(
        dismissals=("dismissal_type", "count")
    )
    .reset_index()
)

print("Player dismissal dataset shape:", player_dismissal_df.shape)

print("\nSample records:")
display(player_dismissal_df.head(10))

print(
    "\nTotal batter dismissals:",
    player_dismissal_df["dismissals"].sum()
)

print(
    "\nPlayers with multiple dismissals in the same innings:",
    (player_dismissal_df["dismissals"] > 1).sum()
)

Player dismissal dataset shape: (37711, 6)

Sample records:


,match_id,innings_number,batting_team,player_out,player_out_id,dismissals
0,1000887,1,Australia,CA Lynn,45eda7c8,1
1,1000887,1,Australia,DA Warner,dcce6f09,1
2,1000887,1,Australia,GJ Maxwell,b681e71e,1
3,1000887,1,Australia,JP Faulkner,808f425a,1
4,1000887,1,Australia,MA Starc,3fb19989,1
5,1000887,1,Australia,MR Marsh,3d8feaf8,1
6,1000887,1,Australia,PJ Cummins,ded9240e,1
7,1000887,1,Australia,SPD Smith,30a45b23,1
8,1000887,1,Australia,TM Head,12b610c2,1
9,1000887,2,Pakistan,Azhar Ali,211de69f,1



Total batter dismissals: 37711

Players with multiple dismissals in the same innings: 0


In [34]:
# Inspect bowling-related delivery data
print("Sample delivery records:")

display(
    deliveries_df[
        [
            "match_id",
            "innings_number",
            "bowler",
            "batter_runs",
            "extra_runs",
            "total_runs",
            "extras_parsed"
        ]
    ].head(15)
)

print("\nTotal delivery records:", len(deliveries_df))

print("\nBowling extras summary:")
print(
    deliveries_df["extras_parsed"]
    .apply(lambda x: sum(x.values()))
    .sum()
)

Sample delivery records:


,match_id,innings_number,bowler,batter_runs,extra_runs,total_runs,extras_parsed
0,1000887,1,Mohammad Amir,0,0,0,{}
1,1000887,1,Mohammad Amir,0,0,0,{}
2,1000887,1,Mohammad Amir,0,0,0,{}
3,1000887,1,Mohammad Amir,0,0,0,{}
4,1000887,1,Mohammad Amir,0,1,1,{'wides': 1}
5,1000887,1,Mohammad Amir,0,0,0,{}
6,1000887,1,Mohammad Amir,0,0,0,{}
7,1000887,1,Mohammad Hafeez,0,0,0,{}
8,1000887,1,Mohammad Hafeez,1,0,1,{}
9,1000887,1,Mohammad Hafeez,0,0,0,{}



Total delivery records: 1379686

Bowling extras summary:
67397


In [35]:
bowling_deliveries = deliveries_df.copy()

# Extract individual extras
bowling_deliveries["wides"] = bowling_deliveries[
    "extras_parsed"
].apply(lambda x: x.get("wides", 0))

bowling_deliveries["no_balls"] = bowling_deliveries[
    "extras_parsed"
].apply(lambda x: x.get("noballs", 0))

bowling_deliveries["byes"] = bowling_deliveries[
    "extras_parsed"
].apply(lambda x: x.get("byes", 0))

bowling_deliveries["legbyes"] = bowling_deliveries[
    "extras_parsed"
].apply(lambda x: x.get("legbyes", 0))

# Calculate runs charged to the bowler
bowling_deliveries["bowler_runs_conceded"] = (
    bowling_deliveries["total_runs"]
    - bowling_deliveries["byes"]
    - bowling_deliveries["legbyes"]
)

print("Total runs charged to bowlers:",
      bowling_deliveries["bowler_runs_conceded"].sum())

print("\nTotal byes:",
      bowling_deliveries["byes"].sum())

print("Total leg byes:",
      bowling_deliveries["legbyes"].sum())

print("Total wides:",
      bowling_deliveries["wides"].sum())

print("Total no-balls:",
      bowling_deliveries["no_balls"].sum())

print("\nSample calculations:")
display(
    bowling_deliveries[
        [
            "bowler",
            "total_runs",
            "byes",
            "legbyes",
            "wides",
            "no_balls",
            "bowler_runs_conceded"
        ]
    ].head(15)
)

Total runs charged to bowlers: 1143411

Total byes: 4480
Total leg byes: 18996
Total wides: 38377
Total no-balls: 5449

Sample calculations:


,bowler,total_runs,byes,legbyes,wides,no_balls,bowler_runs_conceded
0,Mohammad Amir,0,0,0,0,0,0
1,Mohammad Amir,0,0,0,0,0,0
2,Mohammad Amir,0,0,0,0,0,0
3,Mohammad Amir,0,0,0,0,0,0
4,Mohammad Amir,1,0,0,1,0,1
5,Mohammad Amir,0,0,0,0,0,0
6,Mohammad Amir,0,0,0,0,0,0
7,Mohammad Hafeez,0,0,0,0,0,0
8,Mohammad Hafeez,1,0,0,0,0,1
9,Mohammad Hafeez,0,0,0,0,0,0


In [36]:
# Dismissals credited to bowlers
bowler_dismissals = dismissals_df[
    dismissals_df["bowler_credited"]
].copy()

# Count wickets by match, innings and bowler
player_bowling_wickets = (
    bowler_dismissals
    .groupby(
        ["match_id", "innings_number", "batting_team"]
    )
    .size()
    .reset_index(name="wickets")
)

# Attach bowler identity using the delivery records
bowler_lookup = (
    deliveries_df[
        ["match_id", "innings_number", "bowler", "bowler_id"]
    ]
    .drop_duplicates()
)

player_bowling_wickets = player_bowling_wickets.merge(
    bowler_lookup,
    on=["match_id", "innings_number"],
    how="left"
)

print("Bowler-credited dismissal events:",
      len(bowler_dismissals))

print("Total bowler-credited wickets:",
      bowler_dismissals["bowler_credited"].sum())

print("\nSample bowler wicket records:")
display(player_bowling_wickets.head(10))

Bowler-credited dismissal events: 34975
Total bowler-credited wickets: 34975

Sample bowler wicket records:


,match_id,innings_number,batting_team,wickets,bowler,bowler_id
0,1000887,1,Australia,8,Mohammad Amir,e174dadd
1,1000887,1,Australia,8,Mohammad Hafeez,9ab63e7b
2,1000887,1,Australia,8,Hasan Ali,2911de16
3,1000887,1,Australia,8,Wahab Riaz,b3118300
4,1000887,1,Australia,8,Imad Wasim,9cb8d7a6
5,1000887,1,Australia,8,Mohammad Nawaz (3),3086f7a4
6,1000887,2,Pakistan,10,MA Starc,3fb19989
7,1000887,2,Pakistan,10,B Stanlake,6834d1f2
8,1000887,2,Pakistan,10,JP Faulkner,808f425a
9,1000887,2,Pakistan,10,PJ Cummins,ded9240e


In [38]:
# Extract the bowler responsible for each dismissal delivery
dismissal_bowler_records = []

for _, row in deliveries_df.iterrows():
    for wicket in row["wickets_parsed"]:
        dismissal_bowler_records.append({
            "match_id": str(row["match_id"]),
            "innings_number": row["innings_number"],
            "bowler": row["bowler"],
            "bowler_id": row["bowler_id"],
            "player_out": wicket.get("player_out"),
            "dismissal_type": wicket.get("kind")
        })

dismissal_bowler_df = pd.DataFrame(dismissal_bowler_records)

# Keep only bowler-credited dismissals
dismissal_bowler_df = dismissal_bowler_df[
    dismissal_bowler_df["dismissal_type"].isin(
        bowler_credited_types
    )
].copy()

# Aggregate wickets by the actual bowler
player_bowling_wickets = (
    dismissal_bowler_df
    .groupby(
        [
            "match_id",
            "innings_number",
            "bowler",
            "bowler_id"
        ]
    )
    .size()
    .reset_index(name="wickets")
)

print("Bowling wicket dataset shape:", player_bowling_wickets.shape)

print("\nTotal bowler-credited wickets:",
      player_bowling_wickets["wickets"].sum())

print("\nSample records:")
display(player_bowling_wickets.head(10))

Bowling wicket dataset shape: (19220, 5)

Total bowler-credited wickets: 34975

Sample records:


,match_id,innings_number,bowler,bowler_id,wickets
0,1000887,1,Hasan Ali,2911de16,3
1,1000887,1,Imad Wasim,9cb8d7a6,2
2,1000887,1,Mohammad Amir,e174dadd,2
3,1000887,1,Mohammad Nawaz (3),3086f7a4,1
4,1000887,2,JP Faulkner,808f425a,4
5,1000887,2,MA Starc,3fb19989,2
6,1000887,2,MR Marsh,3d8feaf8,1
7,1000887,2,PJ Cummins,ded9240e,3
8,1000889,1,Hasan Ali,2911de16,1
9,1000889,1,Imad Wasim,9cb8d7a6,2


In [39]:
player_bowling_runs = (
    bowling_deliveries
    .groupby(
        ["match_id", "innings_number", "bowler", "bowler_id"]
    )
    .agg(
        runs_conceded=("bowler_runs_conceded", "sum"),
        wides=("wides", "sum"),
        no_balls=("no_balls", "sum")
    )
    .reset_index()
)

# Standardise match IDs
player_bowling_runs["match_id"] = (
    player_bowling_runs["match_id"].astype(str)
)

player_bowling_wickets["match_id"] = (
    player_bowling_wickets["match_id"].astype(str)
)

# Combine runs and wickets
player_bowling_df = player_bowling_runs.merge(
    player_bowling_wickets,
    on=["match_id", "innings_number", "bowler", "bowler_id"],
    how="left",
    validate="one_to_one"
)

# Bowlers without a wicket receive zero
player_bowling_df["wickets"] = (
    player_bowling_df["wickets"].fillna(0).astype(int)
)

print("Player bowling dataset shape:", player_bowling_df.shape)

print("\nTotal runs conceded:",
      player_bowling_df["runs_conceded"].sum())

print("Total wickets:",
      player_bowling_df["wickets"].sum())

print("\nSample records:")
display(player_bowling_df.head(10))

Player bowling dataset shape: (31186, 8)

Total runs conceded: 1143411
Total wickets: 34975

Sample records:


,match_id,innings_number,bowler,bowler_id,runs_conceded,wides,no_balls,wickets
0,1000887,1,Hasan Ali,2911de16,65,2,0,3
1,1000887,1,Imad Wasim,9cb8d7a6,35,0,0,2
2,1000887,1,Mohammad Amir,e174dadd,54,3,0,2
3,1000887,1,Mohammad Hafeez,9ab63e7b,23,1,0,0
4,1000887,1,Mohammad Nawaz (3),3086f7a4,48,0,0,1
5,1000887,1,Wahab Riaz,b3118300,42,3,1,0
6,1000887,2,B Stanlake,6834d1f2,13,1,0,0
7,1000887,2,JP Faulkner,808f425a,32,0,0,4
8,1000887,2,MA Starc,3fb19989,34,2,0,2
9,1000887,2,MR Marsh,3d8feaf8,27,0,0,1


In [40]:
print("Bowling runs records:", len(player_bowling_runs))
print("Bowling wicket records:", len(player_bowling_wickets))
print("Combined records:", len(player_bowling_df))

print("\nDuplicate keys in bowling runs:")
print(
    player_bowling_runs.duplicated(
        ["match_id", "innings_number", "bowler", "bowler_id"]
    ).sum()
)

print("\nDuplicate keys in bowling wickets:")
print(
    player_bowling_wickets.duplicated(
        ["match_id", "innings_number", "bowler", "bowler_id"]
    ).sum()
)

print("\nMissing values in combined dataset:")
print(player_bowling_df.isna().sum())

print("\nRecords with missing bowling statistics:")
display(
    player_bowling_df[
        player_bowling_df.isna().any(axis=1)
    ].head(10)
)

Bowling runs records: 31186
Bowling wicket records: 19220
Combined records: 31186

Duplicate keys in bowling runs:
0

Duplicate keys in bowling wickets:
0

Missing values in combined dataset:
match_id          0
innings_number    0
bowler            0
bowler_id         0
runs_conceded     0
wides             0
no_balls          0
wickets           0
dtype: int64

Records with missing bowling statistics:


,match_id,innings_number,bowler,bowler_id,runs_conceded,wides,no_balls,wickets


In [41]:
# Calculate legal balls bowled
bowling_deliveries["legal_ball"] = (
    (bowling_deliveries["wides"] == 0) &
    (bowling_deliveries["no_balls"] == 0)
).astype(int)

# Aggregate legal balls by bowler and innings
bowler_balls = (
    bowling_deliveries
    .groupby(
        ["match_id", "innings_number", "bowler", "bowler_id"]
    )
    .agg(
        legal_balls=("legal_ball", "sum")
    )
    .reset_index()
)

# Merge legal balls into bowling dataset
player_bowling_df = player_bowling_df.merge(
    bowler_balls,
    on=["match_id", "innings_number", "bowler", "bowler_id"],
    how="left",
    validate="one_to_one"
)

# Convert legal balls into overs (display format)
player_bowling_df["overs"] = (
    player_bowling_df["legal_balls"] // 6
    + (player_bowling_df["legal_balls"] % 6) / 10
)

# Calculate economy rate
player_bowling_df["economy"] = np.where(
    player_bowling_df["legal_balls"] > 0,
    player_bowling_df["runs_conceded"] /
    (player_bowling_df["legal_balls"] / 6),
    np.nan
)

print("Total legal balls:", player_bowling_df["legal_balls"].sum())

print("\nTotal overs (legal balls / 6):",
      player_bowling_df["legal_balls"].sum() / 6)

print("\nSample bowling figures:")
display(
    player_bowling_df[
        [
            "bowler",
            "runs_conceded",
            "wickets",
            "legal_balls",
            "overs",
            "economy"
        ]
    ].head(10)
)

Total legal balls: 1342575

Total overs (legal balls / 6): 223762.5

Sample bowling figures:


,bowler,runs_conceded,wickets,legal_balls,overs,economy
0,Hasan Ali,65,3,54,9.0,7.222222
1,Imad Wasim,35,2,60,10.0,3.500000
2,Mohammad Amir,54,2,60,10.0,5.400000
3,Mohammad Hafeez,23,0,42,7.0,3.285714
4,Mohammad Nawaz (3),48,1,48,8.0,6.000000
5,Wahab Riaz,42,0,36,6.0,7.000000
6,B Stanlake,13,0,18,3.0,4.333333
7,JP Faulkner,32,4,42,7.0,4.571429
8,MA Starc,34,2,48,8.0,4.250000
9,MR Marsh,27,1,36,6.0,4.500000


In [42]:
# Validate total runs conceded
delivery_bowler_runs = bowling_deliveries[
    "bowler_runs_conceded"
].sum()

player_bowler_runs = player_bowling_df[
    "runs_conceded"
].sum()

# Validate total bowler-credited wickets
delivery_bowler_wickets = len(
    dismissal_bowler_df
)

player_bowler_wickets = player_bowling_df[
    "wickets"
].sum()

print("RUNS VALIDATION")
print("Delivery-level runs:", delivery_bowler_runs)
print("Player-level runs:", player_bowler_runs)
print("Difference:", delivery_bowler_runs - player_bowler_runs)

print("\nWICKETS VALIDATION")
print("Dismissal-level wickets:", delivery_bowler_wickets)
print("Player-level wickets:", player_bowler_wickets)
print("Difference:", delivery_bowler_wickets - player_bowler_wickets)

RUNS VALIDATION
Delivery-level runs: 1143411
Player-level runs: 1143411
Difference: 0

WICKETS VALIDATION
Dismissal-level wickets: 34975
Player-level wickets: 34975
Difference: 0


In [43]:
# Aggregate bowling runs by innings
bowling_innings_validation = (
    player_bowling_df
    .groupby(["match_id", "innings_number"])
    ["runs_conceded"]
    .sum()
    .reset_index(name="bowler_runs")
)

# Aggregate byes and leg byes by innings
non_bowler_extras = (
    bowling_deliveries
    .groupby(["match_id", "innings_number"])
    .agg(
        byes=("byes", "sum"),
        legbyes=("legbyes", "sum")
    )
    .reset_index()
)

# Reconstruct innings totals
innings_validation = innings_df.copy()
innings_validation["match_id"] = (
    innings_validation["match_id"].astype(str)
)

innings_validation = innings_validation.merge(
    bowling_innings_validation,
    on=["match_id", "innings_number"],
    how="left"
)

innings_validation = innings_validation.merge(
    non_bowler_extras,
    on=["match_id", "innings_number"],
    how="left"
)

innings_validation["reconstructed_total"] = (
    innings_validation["bowler_runs"]
    + innings_validation["byes"]
    + innings_validation["legbyes"]
)

innings_validation["difference"] = (
    innings_validation["total_runs"]
    - innings_validation["reconstructed_total"]
)

print("Total innings checked:", len(innings_validation))
print("Exact matches:", (innings_validation["difference"] == 0).sum())
print("Mismatches:", (innings_validation["difference"] != 0).sum())

print("\nSample mismatches:")
display(
    innings_validation[
        innings_validation["difference"] != 0
    ].head(10)
)

Total innings checked: 5143
Exact matches: 5143
Mismatches: 0

Sample mismatches:


,match_id,innings_number,batting_team,total_runs,wickets_lost,delivery_records,target_runs,bowler_runs,byes,legbyes,reconstructed_total,difference


In [45]:
print("AFGHANISTAN BATTING DATA")
print("Shape:", afg_batting_df.shape)
display(afg_batting_df.head(10))

print("\nAFGHANISTAN BOWLING DATA")
print("Shape:", afg_bowling_df.shape)
display(afg_bowling_df.head(10))

print("\nUnique batting players:", afg_batting_df["Batter"].nunique())
print("Unique bowling players:", afg_bowling_df["Bowler"].nunique())

print("\nUnique match URLs:")
print("Batting:", afg_batting_df["match_url"].nunique())
print("Bowling:", afg_bowling_df["match_url"].nunique())

AFGHANISTAN BATTING DATA
Shape: (2719, 13)


,Batter,Dismissal,R,B,4s,6s,SR,innings_number,date,match_url,runs,calculated_sr,sr_difference
0,Paul Stirling,bowled b Khaliq Dad,6,12,1,0,50.00,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,6,50.000000,0.000000
1,James Hall,caught Karim Sadiq b Shapoor Zadran,15,29,2,0,51.72,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,15,51.724138,0.004138
2,Alex Cusack,caught Shabir Noori b Khaliq Dad,2,17,0,0,11.76,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,2,11.764706,0.004706
3,Kevin O'Brien,caught Hamid Hassan b Samiullah Shinwari,44,68,5,0,64.70,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,44,64.705882,0.005882
4,Andrew Poynter,caught Karim Sadiq b Hamid Hassan,78,109,6,1,71.55,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,78,71.559633,0.009633
5,Andrew White,lbw b Hamid Hassan,31,35,1,1,88.57,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,31,88.571429,0.001429
6,John Mooney,caught Mohammad Nabi b Hamid Hassan,2,2,0,0,100.00,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,2,100.000000,0.000000
7,Trent Johnston,not out,42*,28,2,2,150.00,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,42,150.000000,0.000000
8,Nigel Jones,caught Nawroz Mangal b Mohammad Nabi,2,4,0,0,50.00,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,2,50.000000,0.000000
9,Rory McCann,run out Mohammad Nabi b Mohammad Nabi,0,0,0,0,0.00,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,0,NaN,NaN



AFGHANISTAN BOWLING DATA
Shape: (1765, 15)


,Bowler,O,M,R,W,Econ,WD,NB,innings_number,date,match_url,total_balls,overs_decimal,calculated_economy,economy_difference
0,Shapoor Zadran,10.0,3,39,1,3.90,1,1,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,60,10.000000,3.900000,0.000000
1,Khaliq Dad,8.0,1,27,2,3.37,4,1,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,48,8.000000,3.375000,0.005000
2,Hamid Hassan,8.0,0,44,3,5.50,0,1,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,48,8.000000,5.500000,0.000000
3,Mohammad Nabi,10.0,0,55,1,5.50,1,0,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,60,10.000000,5.500000,0.000000
4,Samiullah Shinwari,10.0,0,52,1,5.20,0,0,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,60,10.000000,5.200000,0.000000
5,Nawroz Mangal,4.0,0,18,0,4.50,0,1,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,24,4.000000,4.500000,0.000000
6,Kevin O'Brien,7.0,0,35,1,5.00,2,0,2,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,42,7.000000,5.000000,0.000000
7,Trent Johnston,9.0,1,34,1,3.77,0,0,2,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,54,9.000000,3.777778,0.007778
8,Alex Cusack,8.1,1,20,5,2.44,0,0,2,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,49,8.166667,2.448980,0.008980
9,George Dockrell,10.0,2,37,1,3.70,0,0,2,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,60,10.000000,3.700000,0.000000



Unique batting players: 403
Unique bowling players: 303

Unique match URLs:
Batting: 148
Bowling: 148


In [46]:
# Extract unique player names
batting_players = set(
    afg_batting_df["Batter"].dropna().str.strip()
)

bowling_players = set(
    afg_bowling_df["Bowler"].dropna().str.strip()
)

# Compare the two sets
common_players = batting_players.intersection(bowling_players)

batting_only = batting_players - bowling_players
bowling_only = bowling_players - batting_players

print("Unique batting players:", len(batting_players))
print("Unique bowling players:", len(bowling_players))

print("\nPlayers appearing in both datasets:", len(common_players))
print("Players appearing only in batting:", len(batting_only))
print("Players appearing only in bowling:", len(bowling_only))

print("\nSample common players:")
print(sorted(common_players)[:20])

print("\nSample batting-only players:")
print(sorted(batting_only)[:20])

print("\nSample bowling-only players:")
print(sorted(bowling_only)[:20])

Unique batting players: 403
Unique bowling players: 303

Players appearing in both datasets: 252
Players appearing only in batting: 151
Players appearing only in bowling: 51

Sample common players:
['AM Ghazanfar', 'Abdul Rahman (Afghanistan, 2023)', 'Abdur Razzak', 'Abu Hider', 'Adil Rashid', 'Afif Hossain', 'Aftab Alam', 'Aiden Markram', 'Akila Dananjaya', 'Al Islam', 'Alex Cusack', 'Alzarri Joseph', 'Amir Hamza', 'Andile Phehlukwayo', 'Andrew White', 'Andy Balbirnie', 'Andy McBrine', 'Angelo Mathews', 'Anwar Ali', 'Arafat Sunny']

Sample batting-only players:
['Aaron Finch', 'Abdul Malik', 'Abdullah Shafique', 'Afsar Zazai', 'Ahmed Shehzad', 'Ajinkya Rahane', 'Alex Hales', 'Ambati Rayudu', 'Anamul Haque', 'Andrew Poynter', 'Asad Shafiq', 'Asif Ali', 'Avishka Fernando', 'Babar Azam', 'Ben Calitz', 'Ben Curran', 'Ben Duckett', 'Brad Haddin', 'Brandon King', 'Brendan Taylor']

Sample bowling-only players:
['Adam Milne', 'Adam Zampa', 'Ajantha Mendis', 'Amit Mishra', 'Asitha Fernando', 

In [47]:
afg_player_batting_df = afg_batting_df[
    [
        "Batter",
        "Dismissal",
        "runs",
        "B",
        "4s",
        "6s",
        "innings_number",
        "date",
        "match_url"
    ]
].copy()

# Standardise column names
afg_player_batting_df = afg_player_batting_df.rename(
    columns={
        "Batter": "player_name",
        "Dismissal": "dismissal",
        "runs": "runs_scored",
        "B": "balls_faced",
        "4s": "fours",
        "6s": "sixes"
    }
)

# Ensure correct data types
afg_player_batting_df["date"] = pd.to_datetime(
    afg_player_batting_df["date"]
)

numeric_columns = [
    "runs_scored",
    "balls_faced",
    "fours",
    "sixes",
    "innings_number"
]

for col in numeric_columns:
    afg_player_batting_df[col] = pd.to_numeric(
        afg_player_batting_df[col],
        errors="coerce"
    )

# Calculate strike rate
afg_player_batting_df["strike_rate"] = np.where(
    afg_player_batting_df["balls_faced"] > 0,
    afg_player_batting_df["runs_scored"] /
    afg_player_batting_df["balls_faced"] * 100,
    np.nan
)

print("Afghanistan player batting dataset shape:",
      afg_player_batting_df.shape)

print("\nMissing values:")
print(afg_player_batting_df.isna().sum())

print("\nSample records:")
display(afg_player_batting_df.head(10))

Afghanistan player batting dataset shape: (2719, 10)

Missing values:
player_name        0
dismissal          0
runs_scored        0
balls_faced        0
fours              0
sixes              0
innings_number     0
date               0
match_url          0
strike_rate       28
dtype: int64

Sample records:


,player_name,dismissal,runs_scored,balls_faced,fours,sixes,innings_number,date,match_url,strike_rate
0,Paul Stirling,bowled b Khaliq Dad,6,12,1,0,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,50.000000
1,James Hall,caught Karim Sadiq b Shapoor Zadran,15,29,2,0,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,51.724138
2,Alex Cusack,caught Shabir Noori b Khaliq Dad,2,17,0,0,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,11.764706
3,Kevin O'Brien,caught Hamid Hassan b Samiullah Shinwari,44,68,5,0,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,64.705882
4,Andrew Poynter,caught Karim Sadiq b Hamid Hassan,78,109,6,1,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,71.559633
5,Andrew White,lbw b Hamid Hassan,31,35,1,1,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,88.571429
6,John Mooney,caught Mohammad Nabi b Hamid Hassan,2,2,0,0,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,100.000000
7,Trent Johnston,not out,42,28,2,2,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,150.000000
8,Nigel Jones,caught Nawroz Mangal b Mohammad Nabi,2,4,0,0,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,50.000000
9,Rory McCann,run out Mohammad Nabi b Mohammad Nabi,0,0,0,0,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,NaN


## 2. Afghanistan Player Data Processing

Process Afghanistan scorecard-derived batting and bowling records. Connect player performances with match metadata and validate dates, teams, opponents, and innings assignments.

In [48]:
afg_player_bowling_df = afg_bowling_df[
    [
        "Bowler",
        "O",
        "M",
        "R",
        "W",
        "WD",
        "NB",
        "total_balls",
        "innings_number",
        "date",
        "match_url"
    ]
].copy()

# Standardise column names
afg_player_bowling_df = afg_player_bowling_df.rename(
    columns={
        "Bowler": "player_name",
        "O": "overs_display",
        "M": "maidens",
        "R": "runs_conceded",
        "W": "wickets",
        "WD": "wides",
        "NB": "no_balls",
        "total_balls": "legal_balls"
    }
)

# Convert date
afg_player_bowling_df["date"] = pd.to_datetime(
    afg_player_bowling_df["date"]
)

# Ensure numeric columns
numeric_columns = [
    "maidens",
    "runs_conceded",
    "wickets",
    "wides",
    "no_balls",
    "legal_balls",
    "innings_number"
]

for col in numeric_columns:
    afg_player_bowling_df[col] = pd.to_numeric(
        afg_player_bowling_df[col],
        errors="coerce"
    )

# Calculate economy using legal balls
afg_player_bowling_df["economy"] = np.where(
    afg_player_bowling_df["legal_balls"] > 0,
    afg_player_bowling_df["runs_conceded"] /
    (afg_player_bowling_df["legal_balls"] / 6),
    np.nan
)

print("Afghanistan player bowling dataset shape:",
      afg_player_bowling_df.shape)

print("\nMissing values:")
print(afg_player_bowling_df.isna().sum())

print("\nSample records:")
display(afg_player_bowling_df.head(10))

Afghanistan player bowling dataset shape: (1765, 12)

Missing values:
player_name       0
overs_display     0
maidens           0
runs_conceded     0
wickets           0
wides             0
no_balls          0
legal_balls       0
innings_number    0
date              0
match_url         0
economy           1
dtype: int64

Sample records:


,player_name,overs_display,maidens,runs_conceded,wickets,wides,no_balls,legal_balls,innings_number,date,match_url,economy
0,Shapoor Zadran,10.0,3,39,1,1,1,60,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,3.900000
1,Khaliq Dad,8.0,1,27,2,4,1,48,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,3.375000
2,Hamid Hassan,8.0,0,44,3,0,1,48,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,5.500000
3,Mohammad Nabi,10.0,0,55,1,1,0,60,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,5.500000
4,Samiullah Shinwari,10.0,0,52,1,0,0,60,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,5.200000
5,Nawroz Mangal,4.0,0,18,0,0,1,24,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,4.500000
6,Kevin O'Brien,7.0,0,35,1,2,0,42,2,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,5.000000
7,Trent Johnston,9.0,1,34,1,0,0,54,2,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,3.777778
8,Alex Cusack,8.1,1,20,5,0,0,49,2,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,2.448980
9,George Dockrell,10.0,2,37,1,0,0,60,2,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,3.700000


In [49]:
missing_economy = afg_player_bowling_df[
    afg_player_bowling_df["economy"].isna()
]

print("Records with missing economy:", len(missing_economy))

display(
    missing_economy[
        [
            "player_name",
            "overs_display",
            "runs_conceded",
            "wickets",
            "wides",
            "no_balls",
            "legal_balls",
            "date",
            "match_url"
        ]
    ]
)

Records with missing economy: 1


,player_name,overs_display,runs_conceded,wickets,wides,no_balls,legal_balls,date,match_url
1515,Rahmanullah Gurbaz,0.0,1,0,1,0,0,2024-09-22,https://crickrida.com/matches/afghanistan-sout...


In [51]:
# Make sure both date columns use the same data type
afg_player_bowling_df["date"] = pd.to_datetime(
    afg_player_bowling_df["date"]
)

afg_bowling_df["date"] = pd.to_datetime(
    afg_bowling_df["date"]
)

# Compare calculated economy with original scorecard economy
afg_economy_check = afg_player_bowling_df.merge(
    afg_bowling_df[
        ["Bowler", "Econ", "date", "match_url", "innings_number"]
    ],
    left_on=[
        "player_name", "date", "match_url", "innings_number"
    ],
    right_on=[
        "Bowler", "date", "match_url", "innings_number"
    ],
    how="left",
    validate="one_to_one"
)

afg_economy_check["economy_difference"] = (
    afg_economy_check["economy"] -
    afg_economy_check["Econ"]
).abs()

print("Total records checked:", len(afg_economy_check))

print(
    "Records within 0.01 tolerance:",
    (afg_economy_check["economy_difference"] <= 0.01).sum()
)

print(
    "Records outside 0.01 tolerance:",
    (afg_economy_check["economy_difference"] > 0.01).sum()
)

print(
    "Missing original economy values:",
    afg_economy_check["Econ"].isna().sum()
)

print("\nRecords outside tolerance:")
display(
    afg_economy_check[
        afg_economy_check["economy_difference"] > 0.01
    ][
        [
            "player_name",
            "date",
            "runs_conceded",
            "legal_balls",
            "economy",
            "Econ",
            "economy_difference"
        ]
    ].head(10)
)

Total records checked: 1765
Records within 0.01 tolerance: 1764
Records outside 0.01 tolerance: 0
Missing original economy values: 0

Records outside tolerance:


,player_name,date,runs_conceded,legal_balls,economy,Econ,economy_difference


In [53]:
# Standardise date formats
afg_player_batting_df["date"] = pd.to_datetime(
    afg_player_batting_df["date"]
)

afg_batting_df["date"] = pd.to_datetime(
    afg_batting_df["date"]
)

# Compare calculated strike rates with original scorecards
afg_batting_check = afg_player_batting_df.merge(
    afg_batting_df[
        ["Batter", "SR", "date", "match_url", "innings_number"]
    ],
    left_on=[
        "player_name", "date", "match_url", "innings_number"
    ],
    right_on=[
        "Batter", "date", "match_url", "innings_number"
    ],
    how="left",
    validate="one_to_one"
)

afg_batting_check["strike_rate_difference"] = (
    afg_batting_check["strike_rate"] -
    afg_batting_check["SR"]
).abs()

print("Total records checked:", len(afg_batting_check))

print(
    "Records within 0.01 tolerance:",
    (afg_batting_check["strike_rate_difference"] <= 0.01).sum()
)

print(
    "Records outside 0.01 tolerance:",
    (afg_batting_check["strike_rate_difference"] > 0.01).sum()
)

print(
    "Missing original strike rates:",
    afg_batting_check["SR"].isna().sum()
)

print("\nRecords outside tolerance:")
display(
    afg_batting_check[
        afg_batting_check["strike_rate_difference"] > 0.01
    ][
        [
            "player_name",
            "date",
            "runs_scored",
            "balls_faced",
            "strike_rate",
            "SR",
            "strike_rate_difference"
        ]
    ].head(10)
)

Total records checked: 2719
Records within 0.01 tolerance: 2691
Records outside 0.01 tolerance: 0
Missing original strike rates: 0

Records outside tolerance:


,player_name,date,runs_scored,balls_faced,strike_rate,SR,strike_rate_difference


In [54]:
batting_matches = set(afg_player_batting_df["match_url"])
bowling_matches = set(afg_player_bowling_df["match_url"])

print("Unique batting matches:", len(batting_matches))
print("Unique bowling matches:", len(bowling_matches))

print("\nMatches in both datasets:",
      len(batting_matches.intersection(bowling_matches)))

print("Matches only in batting:",
      len(batting_matches - bowling_matches))

print("Matches only in bowling:",
      len(bowling_matches - batting_matches))

print("\nBatting-only match URLs:")
print(sorted(batting_matches - bowling_matches))

print("\nBowling-only match URLs:")
print(sorted(bowling_matches - batting_matches))

Unique batting matches: 148
Unique bowling matches: 148

Matches in both datasets: 148
Matches only in batting: 0
Matches only in bowling: 0

Batting-only match URLs:
[]

Bowling-only match URLs:
[]


In [56]:
AFGHANISTAN_MATCH_FILE = (
    AFGHANISTAN_DIR / "afghanistan_odi_match_level.csv"
)

afg_match_df = pd.read_csv(AFGHANISTAN_MATCH_FILE)

print("Afghanistan match-level shape:", afg_match_df.shape)
print("\nAvailable columns:")
print(afg_match_df.columns.tolist())

display(afg_match_df.head())

Afghanistan match-level shape: (153, 45)

Available columns:
['date', 'match', 'format', 'result', 'match_url', 'batting_innings_count', 'bowling_innings_count', 'scorecard_status', 'venue', 'city', 'toss_winner', 'toss_decision', 'result_type', 'result_margin', 'team_1', 'team_2', 'afghanistan_innings_number', 'afghanistan_runs', 'afghanistan_wickets', 'afghanistan_extras', 'afghanistan_batter_runs', 'afghanistan_balls_faced', 'afghanistan_fours', 'afghanistan_sixes', 'afghanistan_legal_balls', 'opponent_innings_number', 'opponent', 'opponent_runs', 'opponent_wickets', 'opponent_extras', 'opponent_batter_runs', 'opponent_balls_faced', 'opponent_fours', 'opponent_sixes', 'opponent_legal_balls', 'innings_status', 'afghanistan_won', 'venue_source', 'venue_status', 'result_margin_type', 'toss_source', 'toss_status', 'toss_source_url', 'toss_validation', 'toss_decision_validation']


,date,match,format,result,match_url,batting_innings_count,bowling_innings_count,scorecard_status,venue,city,...,innings_status,afghanistan_won,venue_source,venue_status,result_margin_type,toss_source,toss_status,toss_source_url,toss_validation,toss_decision_validation
0,2010-07-03,Afghanistan v Ireland,ODI · Men,Ireland won by 39 runs,https://crickrida.com/matches/afghanistan-irel...,2,2,Available,Hazelaarweg,Rotterdam,...,Two innings recorded,0.0,Cricbuzz,Verified,runs,IPL.com,Available,NaN,Valid,Valid
1,2012-02-10,Afghanistan v Pakistan,ODI · Men,Pakistan won by 7 wickets,https://crickrida.com/matches/afghanistan-paki...,2,2,Available,Sharjah Cricket Stadium,Sharjah,...,Two innings recorded,0.0,Pakistan Cricket Board,Verified,wickets,NaN,Available,NaN,No toss / missing,Missing / not applicable
2,2012-07-05,Afghanistan v Ireland,ODI · Men,Ireland won by 59 runs,https://crickrida.com/matches/afghanistan-irel...,2,2,Available,Clontarf Cricket Club Ground,Dublin,...,Two innings recorded,0.0,CricketArchive,Verified,runs,NaN,Available,NaN,No toss / missing,Missing / not applicable
3,2012-08-25,Afghanistan v Australia,ODI · Men,Australia won by 66 runs,https://crickrida.com/matches/afghanistan-aust...,2,2,Available,Sharjah Cricket Stadium,Sharjah,...,Two innings recorded,0.0,Crickrida,Verified,runs,NaN,Available,NaN,No toss / missing,Missing / not applicable
4,2014-02-27,Afghanistan v Pakistan,ODI · Men,Pakistan won by 72 runs,https://crickrida.com/matches/afghanistan-paki...,2,2,Available,Khan Shaheb Osman Ali Stadium,Fatullah,...,Two innings recorded,0.0,Pakistan Cricket Board,Verified,runs,NaN,Available,NaN,No toss / missing,Missing / not applicable


In [57]:
# Inspect Afghanistan match-level columns
print("Afghanistan match-level shape:", afg_match_df.shape)
print("\nAvailable columns:")
print(afg_match_df.columns.tolist())

Afghanistan match-level shape: (153, 45)

Available columns:
['date', 'match', 'format', 'result', 'match_url', 'batting_innings_count', 'bowling_innings_count', 'scorecard_status', 'venue', 'city', 'toss_winner', 'toss_decision', 'result_type', 'result_margin', 'team_1', 'team_2', 'afghanistan_innings_number', 'afghanistan_runs', 'afghanistan_wickets', 'afghanistan_extras', 'afghanistan_batter_runs', 'afghanistan_balls_faced', 'afghanistan_fours', 'afghanistan_sixes', 'afghanistan_legal_balls', 'opponent_innings_number', 'opponent', 'opponent_runs', 'opponent_wickets', 'opponent_extras', 'opponent_batter_runs', 'opponent_balls_faced', 'opponent_fours', 'opponent_sixes', 'opponent_legal_balls', 'innings_status', 'afghanistan_won', 'venue_source', 'venue_status', 'result_margin_type', 'toss_source', 'toss_status', 'toss_source_url', 'toss_validation', 'toss_decision_validation']


In [58]:
match_urls = set(afg_match_df["match_url"].dropna())

batting_urls = set(afg_player_batting_df["match_url"].dropna())
bowling_urls = set(afg_player_bowling_df["match_url"].dropna())

print("Match-level URLs:", len(match_urls))
print("Batting URLs:", len(batting_urls))
print("Bowling URLs:", len(bowling_urls))

print("\nBatting URLs missing from match-level data:",
      len(batting_urls - match_urls))

print("Bowling URLs missing from match-level data:",
      len(bowling_urls - match_urls))

Match-level URLs: 153
Batting URLs: 148
Bowling URLs: 148

Batting URLs missing from match-level data: 0
Bowling URLs missing from match-level data: 0


In [59]:
afg_innings_mapping = pd.concat([
    afg_match_df[
        ["match_url", "date", "afghanistan_innings_number"]
    ].rename(columns={
        "afghanistan_innings_number": "innings_number"
    }).assign(
        batting_team="Afghanistan"
    ),

    afg_match_df[
        ["match_url", "date", "opponent", "opponent_innings_number"]
    ].rename(columns={
        "opponent_innings_number": "innings_number",
        "opponent": "batting_team"
    })
], ignore_index=True)

afg_innings_mapping["innings_number"] = pd.to_numeric(
    afg_innings_mapping["innings_number"],
    errors="coerce"
)

afg_innings_mapping = afg_innings_mapping.dropna(
    subset=["innings_number"]
)

afg_innings_mapping["innings_number"] = (
    afg_innings_mapping["innings_number"].astype(int)
)

print("Innings mapping shape:", afg_innings_mapping.shape)

display(afg_innings_mapping.head(10))

Innings mapping shape: (294, 4)


,match_url,date,innings_number,batting_team
0,https://crickrida.com/matches/afghanistan-irel...,2010-07-03,2,Afghanistan
1,https://crickrida.com/matches/afghanistan-paki...,2012-02-10,1,Afghanistan
2,https://crickrida.com/matches/afghanistan-irel...,2012-07-05,2,Afghanistan
3,https://crickrida.com/matches/afghanistan-aust...,2012-08-25,2,Afghanistan
4,https://crickrida.com/matches/afghanistan-paki...,2014-02-27,2,Afghanistan
5,https://crickrida.com/matches/afghanistan-bang...,2014-03-01,1,Afghanistan
6,https://crickrida.com/matches/afghanistan-sri-...,2014-03-03,2,Afghanistan
7,https://crickrida.com/matches/afghanistan-indi...,2014-03-05,1,Afghanistan
8,https://crickrida.com/matches/afghanistan-zimb...,2014-07-18,1,Afghanistan
9,https://crickrida.com/matches/afghanistan-zimb...,2014-07-20,1,Afghanistan


In [60]:
mapping_duplicates = afg_innings_mapping.duplicated(
    subset=["match_url", "date", "innings_number"]
).sum()

print("Duplicate match-innings mappings:", mapping_duplicates)

print("\nInnings counts by batting team:")
display(
    afg_innings_mapping["batting_team"]
    .value_counts()
    .head(15)
)

print("\nMissing values:")
display(afg_innings_mapping.isna().sum())

Duplicate match-innings mappings: 0

Innings counts by batting team:


batting_team
Afghanistan     147
Ireland          35
Zimbabwe         33
Bangladesh       23
Sri Lanka        15
Pakistan          8
West Indies       8
India             7
South Africa      6
Australia         5
England           4
New Zealand       3
Name: count, dtype: int64


Missing values:


match_url         0
date              0
innings_number    0
batting_team      0
dtype: int64

In [61]:
afg_player_batting_df = afg_player_batting_df.merge(
    afg_innings_mapping,
    on=["match_url", "date", "innings_number"],
    how="left",
    validate="many_to_one"
)

print("Batting dataset shape:", afg_player_batting_df.shape)

print("\nMissing batting team assignments:",
      afg_player_batting_df["batting_team"].isna().sum())

print("\nBatting records by team:")
display(afg_player_batting_df["batting_team"].value_counts())

display(afg_player_batting_df.head())

ValueError: You are trying to merge on datetime64[ns] and object columns for key 'date'. If you wish to proceed you should use pd.concat

In [62]:
# Standardise date formats in both datasets
afg_player_batting_df["date"] = pd.to_datetime(
    afg_player_batting_df["date"]
)

afg_innings_mapping["date"] = pd.to_datetime(
    afg_innings_mapping["date"]
)

# Attach batting team to each player record
afg_player_batting_df = afg_player_batting_df.merge(
    afg_innings_mapping,
    on=["match_url", "date", "innings_number"],
    how="left",
    validate="many_to_one"
)

print("Batting dataset shape:", afg_player_batting_df.shape)

print("\nMissing batting team assignments:",
      afg_player_batting_df["batting_team"].isna().sum())

print("\nBatting records by team:")
display(afg_player_batting_df["batting_team"].value_counts())

display(afg_player_batting_df.head())

Batting dataset shape: (2719, 11)

Missing batting team assignments: 0

Batting records by team:


batting_team
Afghanistan     1395
Ireland          348
Zimbabwe         316
Bangladesh       220
Sri Lanka        118
Pakistan          73
West Indies       70
India             48
South Africa      45
Australia         34
England           33
New Zealand       19
Name: count, dtype: int64

,player_name,dismissal,runs_scored,balls_faced,fours,sixes,innings_number,date,match_url,strike_rate,batting_team
0,Paul Stirling,bowled b Khaliq Dad,6,12,1,0,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,50.000000,Ireland
1,James Hall,caught Karim Sadiq b Shapoor Zadran,15,29,2,0,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,51.724138,Ireland
2,Alex Cusack,caught Shabir Noori b Khaliq Dad,2,17,0,0,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,11.764706,Ireland
3,Kevin O'Brien,caught Hamid Hassan b Samiullah Shinwari,44,68,5,0,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,64.705882,Ireland
4,Andrew Poynter,caught Karim Sadiq b Hamid Hassan,78,109,6,1,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,71.559633,Ireland


In [63]:
# Standardise date formats
afg_player_bowling_df["date"] = pd.to_datetime(
    afg_player_bowling_df["date"]
)

afg_innings_mapping["date"] = pd.to_datetime(
    afg_innings_mapping["date"]
)

# Attach the team being bowled to
afg_player_bowling_df = afg_player_bowling_df.merge(
    afg_innings_mapping,
    on=["match_url", "date", "innings_number"],
    how="left",
    validate="many_to_one"
)

# The bowling team is the opposite of the batting team
afg_player_bowling_df["bowling_team"] = np.where(
    afg_player_bowling_df["batting_team"] == "Afghanistan",
    afg_player_bowling_df["opponent"]
    if "opponent" in afg_player_bowling_df.columns
    else "CHECK",
    "Afghanistan"
)

print("Bowling dataset shape:", afg_player_bowling_df.shape)

print("\nMissing batting team assignments:",
      afg_player_bowling_df["batting_team"].isna().sum())

display(afg_player_bowling_df.head())

Bowling dataset shape: (1765, 14)

Missing batting team assignments: 0


,player_name,overs_display,maidens,runs_conceded,wickets,wides,no_balls,legal_balls,innings_number,date,match_url,economy,batting_team,bowling_team
0,Shapoor Zadran,10.0,3,39,1,1,1,60,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,3.900,Ireland,Afghanistan
1,Khaliq Dad,8.0,1,27,2,4,1,48,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,3.375,Ireland,Afghanistan
2,Hamid Hassan,8.0,0,44,3,0,1,48,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,5.500,Ireland,Afghanistan
3,Mohammad Nabi,10.0,0,55,1,1,0,60,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,5.500,Ireland,Afghanistan
4,Samiullah Shinwari,10.0,0,52,1,0,0,60,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,5.200,Ireland,Afghanistan


In [64]:
# Check bowling team assignments against match-level opponent data

afg_bowling_validation = afg_player_bowling_df.merge(
    afg_match_df[["match_url", "date", "opponent"]],
    on=["match_url", "date"],
    how="left",
    validate="many_to_one"
)

# Expected bowling team is opposite to the batting team
afg_bowling_validation["expected_bowling_team"] = np.where(
    afg_bowling_validation["batting_team"] == "Afghanistan",
    afg_bowling_validation["opponent"],
    "Afghanistan"
)

mismatches = (
    afg_bowling_validation["bowling_team"] !=
    afg_bowling_validation["expected_bowling_team"]
)

print("Total bowling records:", len(afg_bowling_validation))
print("Bowling team mismatches:", mismatches.sum())
print("Missing opponent values:",
      afg_bowling_validation["opponent"].isna().sum())

print("\nMismatch examples:")
display(
    afg_bowling_validation.loc[
        mismatches,
        ["player_name", "date", "batting_team",
         "bowling_team", "expected_bowling_team"]
    ].head(10)
)

ValueError: You are trying to merge on datetime64[ns] and object columns for key 'date'. If you wish to proceed you should use pd.concat

In [65]:
# Standardise dates in both DataFrames
afg_player_bowling_df["date"] = pd.to_datetime(
    afg_player_bowling_df["date"]
)

afg_match_df["date"] = pd.to_datetime(
    afg_match_df["date"]
)

# Merge bowling records with match-level opponent information
afg_bowling_validation = afg_player_bowling_df.merge(
    afg_match_df[["match_url", "date", "opponent"]],
    on=["match_url", "date"],
    how="left",
    validate="many_to_one"
)

# Calculate the expected bowling team
afg_bowling_validation["expected_bowling_team"] = np.where(
    afg_bowling_validation["batting_team"] == "Afghanistan",
    afg_bowling_validation["opponent"],
    "Afghanistan"
)

mismatches = (
    afg_bowling_validation["bowling_team"] !=
    afg_bowling_validation["expected_bowling_team"]
)

print("Total bowling records:", len(afg_bowling_validation))
print("Bowling team mismatches:", mismatches.sum())
print(
    "Missing opponent values:",
    afg_bowling_validation["opponent"].isna().sum()
)

print("\nMismatch examples:")
display(
    afg_bowling_validation.loc[
        mismatches,
        [
            "player_name", "date", "batting_team",
            "bowling_team", "expected_bowling_team"
        ]
    ].head(10)
)

Total bowling records: 1765
Bowling team mismatches: 899
Missing opponent values: 0

Mismatch examples:


,player_name,date,batting_team,bowling_team,expected_bowling_team
6,Kevin O'Brien,2010-07-03,Afghanistan,CHECK,Ireland
7,Trent Johnston,2010-07-03,Afghanistan,CHECK,Ireland
8,Alex Cusack,2010-07-03,Afghanistan,CHECK,Ireland
9,George Dockrell,2010-07-03,Afghanistan,CHECK,Ireland
10,Paul Stirling,2010-07-03,Afghanistan,CHECK,Ireland
11,Andrew White,2010-07-03,Afghanistan,CHECK,Ireland
12,Umar Gul,2012-02-10,Afghanistan,CHECK,Pakistan
13,Wahab Riaz,2012-02-10,Afghanistan,CHECK,Pakistan
14,Saeed Ajmal,2012-02-10,Afghanistan,CHECK,Pakistan
15,Shahid Afridi,2012-02-10,Afghanistan,CHECK,Pakistan


In [66]:
# Create a match-level opponent lookup
opponent_lookup = afg_match_df[
    ["match_url", "date", "opponent"]
].drop_duplicates()

# Merge opponent information into bowling records
afg_player_bowling_df = afg_player_bowling_df.drop(
    columns=["opponent"],
    errors="ignore"
).merge(
    opponent_lookup,
    on=["match_url", "date"],
    how="left",
    validate="many_to_one"
)

# Assign the actual bowling team
afg_player_bowling_df["bowling_team"] = np.where(
    afg_player_bowling_df["batting_team"] == "Afghanistan",
    afg_player_bowling_df["opponent"],
    "Afghanistan"
)

print("Total bowling records:", len(afg_player_bowling_df))

print("\nBowling team distribution:")
display(afg_player_bowling_df["bowling_team"].value_counts())

print("\nMissing bowling team assignments:",
      afg_player_bowling_df["bowling_team"].isna().sum())

print("\nRemaining CHECK labels:",
      (afg_player_bowling_df["bowling_team"] == "CHECK").sum())

Total bowling records: 1765

Bowling team distribution:


bowling_team
Afghanistan     866
Ireland         218
Zimbabwe        203
Bangladesh      143
Sri Lanka        91
West Indies      50
Pakistan         47
India            40
South Africa     34
Australia        32
England          24
New Zealand      17
Name: count, dtype: int64


Missing bowling team assignments: 0

Remaining CHECK labels: 0


In [67]:
# Calculate expected bowling team
expected_bowling_team = np.where(
    afg_player_bowling_df["batting_team"] == "Afghanistan",
    afg_player_bowling_df["opponent"],
    "Afghanistan"
)

mismatches = (
    afg_player_bowling_df["bowling_team"] != expected_bowling_team
)

print("Total bowling records:", len(afg_player_bowling_df))
print("Bowling team mismatches:", mismatches.sum())

print("\nMissing bowling teams:",
      afg_player_bowling_df["bowling_team"].isna().sum())

print("\nMismatch examples:")
display(
    afg_player_bowling_df.loc[
        mismatches,
        ["player_name", "batting_team", "bowling_team", "opponent"]
    ].head(10)
)

Total bowling records: 1765
Bowling team mismatches: 0

Missing bowling teams: 0

Mismatch examples:


,player_name,batting_team,bowling_team,opponent


In [68]:
# Check batting player-team consistency
batting_conflicts = (
    afg_player_batting_df
    .groupby(["match_url", "date", "player_name"])["batting_team"]
    .nunique()
)

# Check bowling player-team consistency
bowling_conflicts = (
    afg_player_bowling_df
    .groupby(["match_url", "date", "player_name"])["bowling_team"]
    .nunique()
)

print(
    "Batting player-team conflicts:",
    (batting_conflicts > 1).sum()
)

print(
    "Bowling player-team conflicts:",
    (bowling_conflicts > 1).sum()
)

Batting player-team conflicts: 0
Bowling player-team conflicts: 0


In [69]:
PLAYER_DATA_DIR = PROJECT_DIR / "data" / "processed" / "player_data"
PLAYER_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Save Afghanistan player datasets
afg_player_batting_df.to_csv(
    PLAYER_DATA_DIR / "afghanistan_player_batting.csv",
    index=False
)

afg_player_bowling_df.to_csv(
    PLAYER_DATA_DIR / "afghanistan_player_bowling.csv",
    index=False
)

# Save innings-to-team mapping
afg_innings_mapping.to_csv(
    PLAYER_DATA_DIR / "afghanistan_innings_team_mapping.csv",
    index=False
)

print("Saved files:")
for file in [
    "afghanistan_player_batting.csv",
    "afghanistan_player_bowling.csv",
    "afghanistan_innings_team_mapping.csv"
]:
    path = PLAYER_DATA_DIR / file
    print(f"{file}: {path.stat().st_size:,} bytes")

Saved files:
afghanistan_player_batting.csv: 441,818 bytes
afghanistan_player_bowling.csv: 278,947 bytes
afghanistan_innings_team_mapping.csv: 28,243 bytes


In [70]:
# Reload saved Afghanistan player datasets
afg_batting_saved = pd.read_csv(
    PLAYER_DATA_DIR / "afghanistan_player_batting.csv"
)

afg_bowling_saved = pd.read_csv(
    PLAYER_DATA_DIR / "afghanistan_player_bowling.csv"
)

afg_mapping_saved = pd.read_csv(
    PLAYER_DATA_DIR / "afghanistan_innings_team_mapping.csv"
)

print("BATTING DATASET")
print("Shape:", afg_batting_saved.shape)
print("Missing values:", afg_batting_saved.isna().sum().sum())

print("\nBOWLING DATASET")
print("Shape:", afg_bowling_saved.shape)
print("Missing values:", afg_bowling_saved.isna().sum().sum())

print("\nINNINGS MAPPING")
print("Shape:", afg_mapping_saved.shape)
print("Missing values:", afg_mapping_saved.isna().sum().sum())

print("\nRecord count verification:")
print(
    "Batting records preserved:",
    len(afg_batting_saved) == len(afg_player_batting_df)
)
print(
    "Bowling records preserved:",
    len(afg_bowling_saved) == len(afg_player_bowling_df)
)
print(
    "Mapping records preserved:",
    len(afg_mapping_saved) == len(afg_innings_mapping)
)

BATTING DATASET
Shape: (2719, 11)
Missing values: 28

BOWLING DATASET
Shape: (1765, 15)
Missing values: 1

INNINGS MAPPING
Shape: (294, 4)
Missing values: 0

Record count verification:
Batting records preserved: True
Bowling records preserved: True
Mapping records preserved: True


In [71]:
print("Files in player data directory:\n")

for file in sorted(PLAYER_DATA_DIR.iterdir()):
    if file.is_file():
        print(file.name)

Files in player data directory:

afghanistan_innings_team_mapping.csv
afghanistan_player_batting.csv
afghanistan_player_bowling.csv
cricsheet_player_master.csv


In [74]:
print("Available dismissal-related DataFrames:\n")

for name in list(globals().keys()):
    if "dismiss" in name.lower():
        value = globals().get(name)

        if isinstance(value, pd.DataFrame):
            print(f"{name}: {value.shape}")

Available dismissal-related DataFrames:

dismissals_df: (37772, 12)
valid_dismissals: (37711, 12)
player_dismissal_df: (37711, 6)
bowler_dismissals: (34975, 12)
dismissal_bowler_df: (34975, 6)


In [75]:
# Save valid player dismissal records
player_dismissal_df.to_csv(
    PLAYER_DATA_DIR / "cricsheet_player_dismissals.csv",
    index=False
)

# Save bowler-credited wicket records
dismissal_bowler_df.to_csv(
    PLAYER_DATA_DIR / "cricsheet_bowler_wickets.csv",
    index=False
)

print("Dismissal datasets saved successfully!")

print("\nPlayer dismissals:", player_dismissal_df.shape)
print("Bowler-credited wickets:", dismissal_bowler_df.shape)

Dismissal datasets saved successfully!

Player dismissals: (37711, 6)
Bowler-credited wickets: (34975, 6)


## 3. Initial Cross-Source Player Identity Mapping

Establish the initial identity mapping between Cricsheet and Crickrida. Create CricketIQ player IDs, preserve source-specific identities, attach IDs to performance records, and establish canonical player names.

In [76]:
player_batting_df.to_csv(
    PLAYER_DATA_DIR / "cricsheet_player_batting.csv",
    index=False
)

player_bowling_df.to_csv(
    PLAYER_DATA_DIR / "cricsheet_player_bowling.csv",
    index=False
)

print("Cricsheet batting and bowling datasets saved.")

print("Batting:", player_batting_df.shape)
print("Bowling:", player_bowling_df.shape)

Cricsheet batting and bowling datasets saved.
Batting: (45432, 10)
Bowling: (31186, 11)


In [77]:
player_files = {
    "Cricsheet player master": "cricsheet_player_master.csv",
    "Cricsheet batting": "cricsheet_player_batting.csv",
    "Cricsheet bowling": "cricsheet_player_bowling.csv",
    "Cricsheet player dismissals": "cricsheet_player_dismissals.csv",
    "Cricsheet bowler wickets": "cricsheet_bowler_wickets.csv",
    "Afghanistan batting": "afghanistan_player_batting.csv",
    "Afghanistan bowling": "afghanistan_player_bowling.csv",
    "Afghanistan innings mapping": "afghanistan_innings_team_mapping.csv"
}

print("FINAL PLAYER DATASET VERIFICATION\n")

for label, filename in player_files.items():
    file_path = PLAYER_DATA_DIR / filename

    df = pd.read_csv(file_path)

    print(f"{label}")
    print(f"  Shape: {df.shape}")
    print(f"  Missing values: {df.isna().sum().sum()}")
    print(f"  File exists: {file_path.exists()}")
    print()

FINAL PLAYER DATASET VERIFICATION

Cricsheet player master
  Shape: (3287, 5)
  Missing values: 0
  File exists: True

Cricsheet batting
  Shape: (45432, 10)
  Missing values: 9
  File exists: True

Cricsheet bowling
  Shape: (31186, 11)
  Missing values: 2
  File exists: True

Cricsheet player dismissals
  Shape: (37711, 6)
  Missing values: 0
  File exists: True

Cricsheet bowler wickets
  Shape: (34975, 6)
  Missing values: 0
  File exists: True

Afghanistan batting
  Shape: (2719, 11)
  Missing values: 28
  File exists: True

Afghanistan bowling
  Shape: (1765, 15)
  Missing values: 1
  File exists: True

Afghanistan innings mapping
  Shape: (294, 4)
  Missing values: 0
  File exists: True



In [78]:
import re
import unicodedata

def normalize_player_name(name):
    if pd.isna(name):
        return ""

    name = unicodedata.normalize("NFKD", str(name))
    name = "".join(
        char for char in name
        if not unicodedata.combining(char)
    )

    name = name.lower().strip()
    name = re.sub(r"[^a-z0-9]", "", name)

    return name


# Cricsheet names and their registry IDs
cricsheet_names = (
    player_registry_df[
        ["player_name", "player_id"]
    ]
    .drop_duplicates()
    .copy()
)

# Afghanistan scorecard names from both batting and bowling
afg_names = pd.concat([
    afg_player_batting_df[["player_name"]],
    afg_player_bowling_df[["player_name"]]
]).drop_duplicates().copy()

# Normalise names
cricsheet_names["normalized_name"] = (
    cricsheet_names["player_name"].apply(normalize_player_name)
)

afg_names["normalized_name"] = (
    afg_names["player_name"].apply(normalize_player_name)
)

# Find exact normalized matches
exact_matches = cricsheet_names.merge(
    afg_names,
    on="normalized_name",
    how="inner",
    suffixes=("_cricsheet", "_afghanistan")
)

exact_matches = exact_matches[
    [
        "player_name_cricsheet",
        "player_id",
        "player_name_afghanistan",
        "normalized_name"
    ]
].drop_duplicates()

print("Unique Cricsheet names:",
      cricsheet_names["player_name"].nunique())

print("Unique Afghanistan scorecard names:",
      afg_names["player_name"].nunique())

print("Exact normalized name matches:",
      len(exact_matches))

display(exact_matches.head(50))

Unique Cricsheet names: 3282
Unique Afghanistan scorecard names: 454
Exact normalized name matches: 98


,player_name_cricsheet,player_id,player_name_afghanistan,normalized_name
0,Babar Azam,8a75e999,Babar Azam,babarazam
1,Hasan Ali,2911de16,Hasan Ali,hasanali
2,Imad Wasim,9cb8d7a6,Imad Wasim,imadwasim
3,Mohammad Amir,e174dadd,Mohammad Amir,mohammadamir
4,Mohammad Hafeez,9ab63e7b,Mohammad Hafeez,mohammadhafeez
5,Mohammad Rizwan,2f26ac1a,Mohammad Rizwan,mohammadrizwan
6,Sharjeel Khan,193ef196,Sharjeel Khan,sharjeelkhan
7,Umar Akmal,fd2bf2a0,Umar Akmal,umarakmal
8,Wahab Riaz,b3118300,Wahab Riaz,wahabriaz
9,Asad Shafiq,50250426,Asad Shafiq,asadshafiq


In [79]:
# Check whether any matched name maps to multiple Cricsheet player IDs

exact_match_ambiguity = (
    exact_matches
    .groupby("normalized_name")
    .agg(
        cricsheet_id_count=("player_id", "nunique"),
        cricsheet_names=("player_name_cricsheet", "nunique"),
        afghanistan_names=("player_name_afghanistan", "nunique")
    )
    .reset_index()
)

ambiguous_matches = exact_match_ambiguity[
    exact_match_ambiguity["cricsheet_id_count"] > 1
]

print("Total exact name matches:", len(exact_matches))
print("Unique matched normalized names:",
      exact_match_ambiguity.shape[0])

print("Names with multiple Cricsheet IDs:",
      len(ambiguous_matches))

display(ambiguous_matches)

Total exact name matches: 98
Unique matched normalized names: 98
Names with multiple Cricsheet IDs: 0


,normalized_name,cricsheet_id_count,cricsheet_names,afghanistan_names


In [80]:
# Create a cross-source identity mapping for exact matches

player_identity_map = exact_matches.copy()

player_identity_map = player_identity_map.rename(columns={
    "player_name_cricsheet": "cricsheet_name",
    "player_name_afghanistan": "crickrida_name"
})

# Create a unified internal ID using the Cricsheet registry ID
player_identity_map["cricketiq_player_id"] = (
    "CRIQ_" + player_identity_map["player_id"].astype(str)
)

player_identity_map["match_status"] = "exact_name_match"

player_identity_map = player_identity_map[
    [
        "cricketiq_player_id",
        "player_id",
        "cricsheet_name",
        "crickrida_name",
        "normalized_name",
        "match_status"
    ]
].drop_duplicates()

print("Unified player identities:", len(player_identity_map))
print("\nDuplicate unified IDs:",
      player_identity_map["cricketiq_player_id"].duplicated().sum())

display(player_identity_map.head(20))

Unified player identities: 98

Duplicate unified IDs: 0


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,normalized_name,match_status
0,CRIQ_8a75e999,8a75e999,Babar Azam,Babar Azam,babarazam,exact_name_match
1,CRIQ_2911de16,2911de16,Hasan Ali,Hasan Ali,hasanali,exact_name_match
2,CRIQ_9cb8d7a6,9cb8d7a6,Imad Wasim,Imad Wasim,imadwasim,exact_name_match
3,CRIQ_e174dadd,e174dadd,Mohammad Amir,Mohammad Amir,mohammadamir,exact_name_match
4,CRIQ_9ab63e7b,9ab63e7b,Mohammad Hafeez,Mohammad Hafeez,mohammadhafeez,exact_name_match
5,CRIQ_2f26ac1a,2f26ac1a,Mohammad Rizwan,Mohammad Rizwan,mohammadrizwan,exact_name_match
6,CRIQ_193ef196,193ef196,Sharjeel Khan,Sharjeel Khan,sharjeelkhan,exact_name_match
7,CRIQ_fd2bf2a0,fd2bf2a0,Umar Akmal,Umar Akmal,umarakmal,exact_name_match
8,CRIQ_b3118300,b3118300,Wahab Riaz,Wahab Riaz,wahabriaz,exact_name_match
9,CRIQ_50250426,50250426,Asad Shafiq,Asad Shafiq,asadshafiq,exact_name_match


In [81]:
# Identify Afghanistan names without an exact Cricsheet match

matched_names = set(
    exact_matches["normalized_name"]
)

unmatched_afg_names = afg_names[
    ~afg_names["normalized_name"].isin(matched_names)
].copy()

print("Total Afghanistan scorecard names:",
      len(afg_names))

print("Exact matches:", len(matched_names))

print("Unmatched Afghanistan names:",
      len(unmatched_afg_names))

display(
    unmatched_afg_names[
        ["player_name", "normalized_name"]
    ].sort_values("player_name").head(100)
)

Total Afghanistan scorecard names: 454
Exact matches: 98
Unmatched Afghanistan names: 356


,player_name,normalized_name
2259,AM Ghazanfar,amghazanfar
308,Aaron Finch,aaronfinch
2308,Abdul Malik,abdulmalik
1944,"Abdul Rahman (Afghanistan, 2023)",abdulrahmanafghanistan2023
229,Adam Milne,adammilne
...,...,...
2685,Ernest Masuku,ernestmasuku
891,Evin Lewis,evinlewis
1573,Fabian Allen,fabianallen
750,Fareed Ahmad,fareedahmad


In [82]:
from difflib import SequenceMatcher

# Get unique Cricsheet names
cricsheet_unique_names = (
    cricsheet_names[
        ["player_name", "player_id", "normalized_name"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

candidate_records = []

for _, afg_row in unmatched_afg_names.iterrows():

    afg_name = afg_row["player_name"]
    afg_normalized = afg_row["normalized_name"]

    similarities = []

    for _, cricsheet_row in cricsheet_unique_names.iterrows():

        cricsheet_normalized = cricsheet_row["normalized_name"]

        score = SequenceMatcher(
            None,
            afg_normalized,
            cricsheet_normalized
        ).ratio()

        similarities.append({
            "afghanistan_name": afg_name,
            "cricsheet_name": cricsheet_row["player_name"],
            "cricsheet_id": cricsheet_row["player_id"],
            "similarity": round(score, 3)
        })

    # Keep the three closest candidates
    top_candidates = sorted(
        similarities,
        key=lambda x: x["similarity"],
        reverse=True
    )[:3]

    candidate_records.extend(top_candidates)


name_candidates_df = pd.DataFrame(candidate_records)

print("Candidate comparisons generated:",
      len(name_candidates_df))

print("\nTop potential name variations:")

display(
    name_candidates_df
    .sort_values("similarity", ascending=False)
    .head(50)
)

Candidate comparisons generated: 1068

Top potential name variations:


,afghanistan_name,cricsheet_name,cricsheet_id,similarity
1065,Nitish Kumar Reddy,Nithish Kumar Reddy,aad0c365,0.970
1059,Prasidh Krishna,M Prasidh Krishna,85e0cf10,0.966
486,Sarfaraz Ahmed,Sarfraz Ahmed,2254ab79,0.960
480,Najmul Hossain Shanto,Nazmul Hossain Shanto,1274d7ab,0.947
477,Litton Das,Liton Das,0404d43c,0.941
714,Sam Curran,SM Curran,e94915e6,0.941
1035,Ebadot Hossain,Ebadat Hossain,cb5100f1,0.923
555,Joe Root,JE Root,a343262c,0.923
45,Mohammad Nabi,Mohammad Naim,7923a51d,0.917
885,Bashir Ahmad,Basir Ahamad,218c7eab,0.909


In [83]:
 high_similarity_candidates = (
    name_candidates_df[
        name_candidates_df["similarity"] >= 0.90
    ]
    .sort_values(
        ["afghanistan_name", "similarity"],
        ascending=[True, False]
    )
    .drop_duplicates(
        subset=["afghanistan_name", "cricsheet_id"]
    )
    .reset_index(drop=True)
)

print("High-similarity candidate pairs:",
      len(high_similarity_candidates))

print(
    "Unique Afghanistan names:",
    high_similarity_candidates["afghanistan_name"].nunique()
)

display(high_similarity_candidates)

High-similarity candidate pairs: 12
Unique Afghanistan names: 12


,afghanistan_name,cricsheet_name,cricsheet_id,similarity
0,Bashir Ahmad,Basir Ahamad,218c7eab,0.909
1,Ebadot Hossain,Ebadat Hossain,cb5100f1,0.923
2,Joe Root,JE Root,a343262c,0.923
3,Liam McCarthy,AM McCarthy,7f148ce7,0.909
4,Litton Das,Liton Das,0404d43c,0.941
5,Mohammad Nabi,Mohammad Naim,7923a51d,0.917
6,Najmul Hossain Shanto,Nazmul Hossain Shanto,1274d7ab,0.947
7,Nitish Kumar Reddy,Nithish Kumar Reddy,aad0c365,0.970
8,Prasidh Krishna,M Prasidh Krishna,85e0cf10,0.966
9,Sam Curran,SM Curran,e94915e6,0.941


In [84]:
# Prepare candidate names
candidate_names = high_similarity_candidates[
    ["afghanistan_name", "cricsheet_name", "cricsheet_id"]
].drop_duplicates()

# Afghanistan player appearances
afg_appearances = pd.concat([
    afg_player_batting_df[
        ["player_name", "date", "batting_team"]
    ].rename(columns={"batting_team": "team"}),

    afg_player_bowling_df[
        ["player_name", "date", "bowling_team"]
    ].rename(columns={"bowling_team": "team"})
]).drop_duplicates()

# Cricsheet player appearances
cricsheet_appearances = player_registry_df[
    ["match_id", "player_name", "player_id"]
].drop_duplicates()

cricsheet_appearances["match_id"] = (
    cricsheet_appearances["match_id"].astype(str)
)

match_info = matches_df[
    ["match_id", "date", "team_1", "team_2"]
].copy()

match_info["match_id"] = match_info["match_id"].astype(str)
match_info["date"] = pd.to_datetime(match_info["date"])

cricsheet_appearances = cricsheet_appearances.merge(
    match_info,
    on="match_id",
    how="left",
    validate="many_to_one"
)

# Show appearance information for each candidate
for _, candidate in candidate_names.iterrows():

    afg_name = candidate["afghanistan_name"]
    cricsheet_name = candidate["cricsheet_name"]
    cricsheet_id = candidate["cricsheet_id"]

    print("\n" + "=" * 65)
    print(f"Afghanistan name: {afg_name}")
    print(f"Cricsheet name: {cricsheet_name}")
    print(f"Cricsheet ID: {cricsheet_id}")

    print("\nAfghanistan scorecard appearances:")
    display(
        afg_appearances[
            afg_appearances["player_name"] == afg_name
        ].drop_duplicates().head(5)
    )

    print("Cricsheet appearances:")
    display(
        cricsheet_appearances[
            cricsheet_appearances["player_id"] == cricsheet_id
        ][
            ["date", "team_1", "team_2"]
        ].drop_duplicates().head(5)
    )


Afghanistan name: Bashir Ahmad
Cricsheet name: Basir Ahamad
Cricsheet ID: 218c7eab

Afghanistan scorecard appearances:


,player_name,date,team
2502,Bashir Ahmad,2025-10-11,Afghanistan
1611,Bashir Ahmad,2025-10-08,Afghanistan


Cricsheet appearances:


,date,team_1,team_2
22338,2022-07-11,Namibia,Nepal
22364,2022-07-13,Scotland,Nepal
22418,2022-07-16,Namibia,Nepal
38381,2025-06-02,Scotland,Nepal
38488,2025-06-10,Nepal,Netherlands



Afghanistan name: Ebadot Hossain
Cricsheet name: Ebadat Hossain
Cricsheet ID: cb5100f1

Afghanistan scorecard appearances:


,player_name,date,team
1238,Ebadot Hossain,2023-07-08,Bangladesh


... (38 outputs left out)


Afghanistan name: Zia-ur-Rehman
Cricsheet name: Ziaur Rahman
Cricsheet ID: c27518d2

Afghanistan scorecard appearances:


,player_name,date,team
1945,Zia-ur-Rehman,2023-07-11,Afghanistan


Cricsheet appearances:


,date,team_1,team_2
69356,2013-03-23,Sri Lanka,Bangladesh
69383,2013-03-25,Sri Lanka,Bangladesh
69410,2013-03-28,Sri Lanka,Bangladesh
69515,2013-05-03,Zimbabwe,Bangladesh
69542,2013-05-05,Zimbabwe,Bangladesh


In [85]:
# Manually reviewed and approved aliases
verified_aliases = [
    ("Ebadot Hossain", "Ebadat Hossain", "cb5100f1"),
    ("Joe Root", "JE Root", "a343262c"),
    ("Litton Das", "Liton Das", "0404d43c"),
    ("Najmul Hossain Shanto", "Nazmul Hossain Shanto", "1274d7ab"),
    ("Nitish Kumar Reddy", "Nithish Kumar Reddy", "aad0c365"),
    ("Prasidh Krishna", "M Prasidh Krishna", "85e0cf10"),
    ("Sam Curran", "SM Curran", "e94915e6"),
    ("Sarfaraz Ahmed", "Sarfraz Ahmed", "2254ab79")
]

alias_records = []

for afg_name, cricsheet_name, cricsheet_id in verified_aliases:
    alias_records.append({
        "cricketiq_player_id": "CRIQ_" + cricsheet_id,
        "player_id": cricsheet_id,
        "cricsheet_name": cricsheet_name,
        "crickrida_name": afg_name,
        "normalized_name": normalize_player_name(afg_name),
        "match_status": "verified_alias"
    })

verified_alias_df = pd.DataFrame(alias_records)

# Add aliases to the original exact-match mapping
player_identity_map = pd.concat(
    [player_identity_map, verified_alias_df],
    ignore_index=True
).drop_duplicates(
    subset=["cricketiq_player_id", "crickrida_name"]
)

print("Total unified player identities:", len(player_identity_map))

print("\nIdentity status:")
display(player_identity_map["match_status"].value_counts())

print("\nDuplicate unified IDs:",
      player_identity_map["cricketiq_player_id"].duplicated().sum())

display(verified_alias_df)

Total unified player identities: 106

Identity status:


match_status
exact_name_match    98
verified_alias       8
Name: count, dtype: int64


Duplicate unified IDs: 0


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,normalized_name,match_status
0,CRIQ_cb5100f1,cb5100f1,Ebadat Hossain,Ebadot Hossain,ebadothossain,verified_alias
1,CRIQ_a343262c,a343262c,JE Root,Joe Root,joeroot,verified_alias
2,CRIQ_0404d43c,0404d43c,Liton Das,Litton Das,littondas,verified_alias
3,CRIQ_1274d7ab,1274d7ab,Nazmul Hossain Shanto,Najmul Hossain Shanto,najmulhossainshanto,verified_alias
4,CRIQ_aad0c365,aad0c365,Nithish Kumar Reddy,Nitish Kumar Reddy,nitishkumarreddy,verified_alias
5,CRIQ_85e0cf10,85e0cf10,M Prasidh Krishna,Prasidh Krishna,prasidhkrishna,verified_alias
6,CRIQ_e94915e6,e94915e6,SM Curran,Sam Curran,samcurran,verified_alias
7,CRIQ_2254ab79,2254ab79,Sarfraz Ahmed,Sarfaraz Ahmed,sarfarazahmed,verified_alias


In [86]:
# Names already represented in the cross-source mapping
mapped_afg_names = set(
    player_identity_map["crickrida_name"]
)

mapped_cricsheet_names = set(
    player_identity_map["cricsheet_name"]
)

# Remaining Afghanistan names
remaining_afg_names = afg_names[
    ~afg_names["player_name"].isin(mapped_afg_names)
].copy()

# Remaining Cricsheet names
remaining_cricsheet_names = cricsheet_names[
    ~cricsheet_names["player_name"].isin(mapped_cricsheet_names)
].copy()

print("Mapped cross-source identities:",
      len(player_identity_map))

print("Remaining Afghanistan names:",
      len(remaining_afg_names))

print("Remaining Cricsheet names:",
      len(remaining_cricsheet_names))

print("\nRemaining Afghanistan examples:")
display(remaining_afg_names.head(15))

print("\nRemaining Cricsheet examples:")
display(remaining_cricsheet_names.head(15))

Mapped cross-source identities: 106
Remaining Afghanistan names: 348
Remaining Cricsheet names: 3188

Remaining Afghanistan examples:


,player_name,normalized_name
0,Paul Stirling,paulstirling
1,James Hall,jameshall
2,Alex Cusack,alexcusack
3,Kevin O'Brien,kevinobrien
4,Andrew Poynter,andrewpoynter
5,Andrew White,andrewwhite
6,John Mooney,johnmooney
7,Trent Johnston,trentjohnston
8,Nigel Jones,nigeljones
9,Rory McCann,rorymccann



Remaining Cricsheet examples:


,player_name,player_id,normalized_name
0,Azhar Ali,211de69f,azharali
1,B Stanlake,6834d1f2,bstanlake
3,C Shamshuddin,0f9d921b,cshamshuddin
4,CA Lynn,45eda7c8,calynn
5,CB Gaffaney,d5ac41d8,cbgaffaney
6,DA Warner,dcce6f09,dawarner
7,GJ Maxwell,b681e71e,gjmaxwell
10,JJ Crowe,2e760301,jjcrowe
11,JP Faulkner,808f425a,jpfaulkner
12,MA Starc,3fb19989,mastarc


In [87]:
print("CROSS-SOURCE IDENTITY MAPPING CHECK\n")

print(
    "Unique CricketIQ IDs:",
    player_identity_map["cricketiq_player_id"].nunique()
)

print(
    "Unique Cricsheet IDs:",
    player_identity_map["player_id"].nunique()
)

print(
    "Unique Cricsheet names:",
    player_identity_map["cricsheet_name"].nunique()
)

print(
    "Unique Crickrida names:",
    player_identity_map["crickrida_name"].nunique()
)

print("\nDuplicate Cricsheet IDs:")
display(
    player_identity_map[
        player_identity_map["player_id"].duplicated(keep=False)
    ].sort_values("player_id")
)

print("\nDuplicate Crickrida names:")
display(
    player_identity_map[
        player_identity_map["crickrida_name"].duplicated(keep=False)
    ].sort_values("crickrida_name")
)

CROSS-SOURCE IDENTITY MAPPING CHECK

Unique CricketIQ IDs: 106
Unique Cricsheet IDs: 106
Unique Cricsheet names: 106
Unique Crickrida names: 106

Duplicate Cricsheet IDs:


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,normalized_name,match_status



Duplicate Crickrida names:


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,normalized_name,match_status


In [88]:
# Create source-specific identities for unmatched Cricsheet players
cricsheet_only = remaining_cricsheet_names.copy()

cricsheet_only["cricketiq_player_id"] = (
    "CS_" + cricsheet_only["player_id"].astype(str)
)

cricsheet_only["source"] = "Cricsheet"
cricsheet_only["match_status"] = "source_only"

# Create source-specific identities for unmatched Crickrida players
crickrida_only = remaining_afg_names.copy()

crickrida_only["cricketiq_player_id"] = (
    "CR_" + crickrida_only["normalized_name"].astype(str)
)

crickrida_only["source"] = "Crickrida"
crickrida_only["match_status"] = "source_only"

print("Cricsheet-only identities:", len(cricsheet_only))
print("Crickrida-only identities:", len(crickrida_only))

print("\nDuplicate Cricsheet-only IDs:",
      cricsheet_only["cricketiq_player_id"].duplicated().sum())

print("Duplicate Crickrida-only IDs:",
      crickrida_only["cricketiq_player_id"].duplicated().sum())

Cricsheet-only identities: 3188
Crickrida-only identities: 348

Duplicate Cricsheet-only IDs: 6
Duplicate Crickrida-only IDs: 0


In [89]:
# Find Cricsheet IDs associated with multiple display names

duplicate_cricsheet_ids = cricsheet_only[
    cricsheet_only["cricketiq_player_id"].duplicated(keep=False)
].sort_values("cricketiq_player_id")

print("Duplicate ID records:", len(duplicate_cricsheet_ids))

display(
    duplicate_cricsheet_ids[
        [
            "cricketiq_player_id",
            "player_id",
            "player_name",
            "normalized_name"
        ]
    ]
)

Duplicate ID records: 12


,cricketiq_player_id,player_id,player_name,normalized_name
2355,CS_5230a33d,5230a33d,L Winfield,lwinfield
15402,CS_5230a33d,5230a33d,L Winfield-Hill,lwinfieldhill
6445,CS_7c0055c1,7c0055c1,KE Ebrahim,keebrahim
68957,CS_7c0055c1,7c0055c1,KE Broadmore,kebroadmore
12249,CS_a9231c3f,a9231c3f,SR Taylor,srtaylor
34248,CS_a9231c3f,a9231c3f,Steven Ryan Taylor,stevenryantaylor
33686,CS_bafb9238,bafb9238,Usman Khan,usmankhan
33743,CS_bafb9238,bafb9238,Usman Khan (2),usmankhan2
43685,CS_e237b28c,e237b28c,Yousuf Youhana,yousufyouhana
44578,CS_e237b28c,e237b28c,Mohammad Yousuf,mohammadyousuf


In [90]:
# One identity per unique Cricsheet player ID
cricsheet_only_master = (
    cricsheet_only
    .groupby("player_id", as_index=False)
    .agg(
        player_names=("player_name", lambda x: sorted(set(x))),
        normalized_names=("normalized_name", lambda x: sorted(set(x)))
    )
)

cricsheet_only_master["cricketiq_player_id"] = (
    "CS_" + cricsheet_only_master["player_id"].astype(str)
)

cricsheet_only_master["source"] = "Cricsheet"
cricsheet_only_master["match_status"] = "source_only"

print("Cricsheet-only player identities:",
      len(cricsheet_only_master))

print("Duplicate CricketIQ IDs:",
      cricsheet_only_master["cricketiq_player_id"].duplicated().sum())

print("\nPlayers with multiple registered names:")
display(
    cricsheet_only_master[
        cricsheet_only_master["player_names"].apply(len) > 1
    ]
)

Cricsheet-only player identities: 3182
Duplicate CricketIQ IDs: 0

Players with multiple registered names:


,player_id,player_names,normalized_names,cricketiq_player_id,source,match_status
987,5230a33d,"[L Winfield, L Winfield-Hill]","[lwinfield, lwinfieldhill]",CS_5230a33d,Cricsheet,source_only
1516,7c0055c1,"[KE Broadmore, KE Ebrahim]","[kebroadmore, keebrahim]",CS_7c0055c1,Cricsheet,source_only
2110,a9231c3f,"[SR Taylor, Steven Ryan Taylor]","[srtaylor, stevenryantaylor]",CS_a9231c3f,Cricsheet,source_only
2325,bafb9238,"[Usman Khan, Usman Khan (2)]","[usmankhan, usmankhan2]",CS_bafb9238,Cricsheet,source_only
2813,e237b28c,"[Mohammad Yousuf, Yousuf Youhana]","[mohammadyousuf, yousufyouhana]",CS_e237b28c,Cricsheet,source_only
3012,f3a18a0c,"[NR Sciver, NR Sciver-Brunt]","[nrsciver, nrsciverbrunt]",CS_f3a18a0c,Cricsheet,source_only


In [91]:
# Check whether different Crickrida names produce the same normalized name

duplicate_crickrida_names = crickrida_only[
    crickrida_only["normalized_name"].duplicated(keep=False)
].sort_values("normalized_name")

print("Crickrida-only records:", len(crickrida_only))
print("Unique normalized names:",
      crickrida_only["normalized_name"].nunique())
print("Duplicate normalized names:",
      len(duplicate_crickrida_names))

display(
    duplicate_crickrida_names[
        ["player_name", "normalized_name"]
    ]
)

Crickrida-only records: 348
Unique normalized names: 348
Duplicate normalized names: 0


,player_name,normalized_name


In [92]:
# Create unique provisional IDs for Crickrida-only players

crickrida_only_master = crickrida_only.copy()

crickrida_only_master["cricketiq_player_id"] = (
    "CR_" + crickrida_only_master["normalized_name"]
)

crickrida_only_master["source"] = "Crickrida"
crickrida_only_master["match_status"] = "source_only"

print("Crickrida-only identities:",
      len(crickrida_only_master))

print("Duplicate CricketIQ IDs:",
      crickrida_only_master["cricketiq_player_id"].duplicated().sum())

print("Missing CricketIQ IDs:",
      crickrida_only_master["cricketiq_player_id"].isna().sum())

display(crickrida_only_master.head(10))

Crickrida-only identities: 348
Duplicate CricketIQ IDs: 0
Missing CricketIQ IDs: 0


,player_name,normalized_name,cricketiq_player_id,source,match_status
0,Paul Stirling,paulstirling,CR_paulstirling,Crickrida,source_only
1,James Hall,jameshall,CR_jameshall,Crickrida,source_only
2,Alex Cusack,alexcusack,CR_alexcusack,Crickrida,source_only
3,Kevin O'Brien,kevinobrien,CR_kevinobrien,Crickrida,source_only
4,Andrew Poynter,andrewpoynter,CR_andrewpoynter,Crickrida,source_only
5,Andrew White,andrewwhite,CR_andrewwhite,Crickrida,source_only
6,John Mooney,johnmooney,CR_johnmooney,Crickrida,source_only
7,Trent Johnston,trentjohnston,CR_trentjohnston,Crickrida,source_only
8,Nigel Jones,nigeljones,CR_nigeljones,Crickrida,source_only
9,Rory McCann,rorymccann,CR_rorymccann,Crickrida,source_only


In [93]:
# Prepare cross-source identities
cross_source_master = player_identity_map[
    [
        "cricketiq_player_id",
        "player_id",
        "cricsheet_name",
        "crickrida_name",
        "match_status"
    ]
].copy()

cross_source_master["source"] = "Both"

# Prepare Cricsheet-only identities
cricsheet_master = cricsheet_only_master[
    [
        "cricketiq_player_id",
        "player_id",
        "player_names",
        "match_status",
        "source"
    ]
].copy()

# Prepare Crickrida-only identities
crickrida_master = crickrida_only_master[
    [
        "cricketiq_player_id",
        "player_name",
        "match_status",
        "source"
    ]
].copy()

# Combine into one registry
player_registry_final = pd.concat(
    [
        cross_source_master,
        cricsheet_master,
        crickrida_master
    ],
    ignore_index=True,
    sort=False
)

print("Final registry shape:", player_registry_final.shape)
print("\nIdentity status:")
print(player_registry_final["match_status"].value_counts())

print("\nSource:")
print(player_registry_final["source"].value_counts())

print("\nDuplicate CricketIQ IDs:",
      player_registry_final["cricketiq_player_id"].duplicated().sum())

print("Missing CricketIQ IDs:",
      player_registry_final["cricketiq_player_id"].isna().sum())

Final registry shape: (3636, 8)

Identity status:
match_status
source_only         3530
exact_name_match      98
verified_alias         8
Name: count, dtype: int64

Source:
source
Cricsheet    3182
Crickrida     348
Both          106
Name: count, dtype: int64

Duplicate CricketIQ IDs: 0
Missing CricketIQ IDs: 0


In [94]:
# Save the unified player identity registry

PLAYER_DATA_DIR.mkdir(parents=True, exist_ok=True)

registry_path = PLAYER_DATA_DIR / "cricketiq_player_identity_registry.csv"

player_registry_final.to_csv(registry_path, index=False)

# Reload and verify
registry_check = pd.read_csv(registry_path)

print("Saved:", registry_path)
print("Reloaded shape:", registry_check.shape)

print("Duplicate IDs after reload:",
      registry_check["cricketiq_player_id"].duplicated().sum())

print("Missing IDs after reload:",
      registry_check["cricketiq_player_id"].isna().sum())

print("All records preserved:",
      len(registry_check) == len(player_registry_final))

Saved: C:\CricIQ\data\processed\player_data\cricketiq_player_identity_registry.csv
Reloaded shape: (3636, 8)
Duplicate IDs after reload: 0
Missing IDs after reload: 0
All records preserved: True


In [95]:
# Create a lookup from Cricsheet player IDs to CricketIQ IDs

cricsheet_id_lookup = (
    player_registry_final[
        player_registry_final["source"].isin(["Cricsheet", "Both"])
    ][
        ["player_id", "cricketiq_player_id"]
    ]
    .dropna(subset=["player_id"])
    .drop_duplicates()
)

print("Cricsheet ID lookup shape:", cricsheet_id_lookup.shape)

print("Duplicate source player IDs:",
      cricsheet_id_lookup["player_id"].duplicated().sum())

print("Missing CricketIQ IDs:",
      cricsheet_id_lookup["cricketiq_player_id"].isna().sum())

display(cricsheet_id_lookup.head(10))

Cricsheet ID lookup shape: (3288, 2)
Duplicate source player IDs: 1
Missing CricketIQ IDs: 0


,player_id,cricketiq_player_id
0,8a75e999,CRIQ_8a75e999
1,2911de16,CRIQ_2911de16
2,9cb8d7a6,CRIQ_9cb8d7a6
3,e174dadd,CRIQ_e174dadd
4,9ab63e7b,CRIQ_9ab63e7b
5,2f26ac1a,CRIQ_2f26ac1a
6,193ef196,CRIQ_193ef196
7,fd2bf2a0,CRIQ_fd2bf2a0
8,b3118300,CRIQ_b3118300
9,50250426,CRIQ_50250426


In [96]:
# Find duplicate Cricsheet source IDs in the lookup

duplicate_source_ids = cricsheet_id_lookup[
    cricsheet_id_lookup["player_id"].duplicated(keep=False)
]

print("Conflicting records:")
display(duplicate_source_ids)

# Inspect all registry records associated with the duplicate ID
conflicting_id = duplicate_source_ids.iloc[0]["player_id"]

print("\nFull registry details:")
display(
    player_registry_final[
        player_registry_final["player_id"] == conflicting_id
    ]
)

Conflicting records:


,player_id,cricketiq_player_id
64,3086f7a4,CRIQ_3086f7a4
664,3086f7a4,CS_3086f7a4



Full registry details:


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,match_status,source,player_names,player_name
64,CRIQ_3086f7a4,3086f7a4,Mohammad Nawaz,Mohammad Nawaz,exact_name_match,Both,NaN,NaN
664,CS_3086f7a4,3086f7a4,NaN,NaN,source_only,Cricsheet,[Mohammad Nawaz (3)],NaN


In [97]:
# Remove the incorrectly classified source-only record
player_registry_final = player_registry_final[
    ~(
        (player_registry_final["player_id"] == "3086f7a4") &
        (player_registry_final["cricketiq_player_id"] == "CS_3086f7a4")
    )
].copy()

print("Corrected registry shape:", player_registry_final.shape)

print("\nMohammad Nawaz identity:")
display(
    player_registry_final[
        player_registry_final["player_id"] == "3086f7a4"
    ]
)

print("\nDuplicate CricketIQ IDs:",
      player_registry_final["cricketiq_player_id"].duplicated().sum())

print("Duplicate Cricsheet player IDs:",
      player_registry_final[
          player_registry_final["source"].isin(["Cricsheet", "Both"])
      ]["player_id"].duplicated().sum())

Corrected registry shape: (3635, 8)

Mohammad Nawaz identity:


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,match_status,source,player_names,player_name
64,CRIQ_3086f7a4,3086f7a4,Mohammad Nawaz,Mohammad Nawaz,exact_name_match,Both,NaN,NaN



Duplicate CricketIQ IDs: 0
Duplicate Cricsheet player IDs: 0


In [98]:
# Rebuild Cricsheet player ID lookup

cricsheet_id_lookup = (
    player_registry_final[
        player_registry_final["source"].isin(["Cricsheet", "Both"])
    ][["player_id", "cricketiq_player_id"]]
    .dropna(subset=["player_id"])
    .copy()
)

print("Lookup shape:", cricsheet_id_lookup.shape)

print("Unique Cricsheet IDs:",
      cricsheet_id_lookup["player_id"].nunique())

print("Duplicate source IDs:",
      cricsheet_id_lookup["player_id"].duplicated().sum())

print("Duplicate CricketIQ IDs:",
      cricsheet_id_lookup["cricketiq_player_id"].duplicated().sum())

print("Missing CricketIQ IDs:",
      cricsheet_id_lookup["cricketiq_player_id"].isna().sum())

Lookup shape: (3287, 2)
Unique Cricsheet IDs: 3287
Duplicate source IDs: 0
Duplicate CricketIQ IDs: 0
Missing CricketIQ IDs: 0


In [99]:
# Attach CricketIQ IDs to Cricsheet batting records

player_batting_df = player_batting_df.merge(
    cricsheet_id_lookup.rename(columns={"player_id": "batter_id"}),
    on="batter_id",
    how="left",
    validate="many_to_one"
)

print("Batting dataset shape:", player_batting_df.shape)

print("Missing CricketIQ IDs:",
      player_batting_df["cricketiq_player_id"].isna().sum())

print("Total batting records:", len(player_batting_df))

display(player_batting_df.head())

Batting dataset shape: (45432, 11)
Missing CricketIQ IDs: 0
Total batting records: 45432


,match_id,innings_number,batting_team,batter,batter_id,runs,balls_faced,fours,sixes,strike_rate,cricketiq_player_id
0,1000887,1,Australia,B Stanlake,6834d1f2,1,6,0,0,16.666667,CS_6834d1f2
1,1000887,1,Australia,CA Lynn,45eda7c8,16,12,1,1,133.333333,CS_45eda7c8
2,1000887,1,Australia,DA Warner,dcce6f09,7,18,1,0,38.888889,CS_dcce6f09
3,1000887,1,Australia,GJ Maxwell,b681e71e,60,56,7,0,107.142857,CS_b681e71e
4,1000887,1,Australia,JP Faulkner,808f425a,5,12,0,0,41.666667,CS_808f425a


In [100]:
# Attach CricketIQ IDs to Cricsheet bowling records

player_bowling_df = player_bowling_df.merge(
    cricsheet_id_lookup.rename(columns={"player_id": "bowler_id"}),
    on="bowler_id",
    how="left",
    validate="many_to_one"
)

print("Bowling dataset shape:", player_bowling_df.shape)

print("Missing CricketIQ IDs:",
      player_bowling_df["cricketiq_player_id"].isna().sum())

print("Total bowling records:", len(player_bowling_df))

display(player_bowling_df.head())

Bowling dataset shape: (31186, 12)
Missing CricketIQ IDs: 0
Total bowling records: 31186


,match_id,innings_number,bowler,bowler_id,runs_conceded,wides,no_balls,wickets,legal_balls,overs,economy,cricketiq_player_id
0,1000887,1,Hasan Ali,2911de16,65,2,0,3,54,9.0,7.222222,CRIQ_2911de16
1,1000887,1,Imad Wasim,9cb8d7a6,35,0,0,2,60,10.0,3.500000,CRIQ_9cb8d7a6
2,1000887,1,Mohammad Amir,e174dadd,54,3,0,2,60,10.0,5.400000,CRIQ_e174dadd
3,1000887,1,Mohammad Hafeez,9ab63e7b,23,1,0,0,42,7.0,3.285714,CRIQ_9ab63e7b
4,1000887,1,Mohammad Nawaz (3),3086f7a4,48,0,0,1,48,8.0,6.000000,CRIQ_3086f7a4


In [101]:
# Attach CricketIQ IDs to dismissal records

player_dismissal_df = player_dismissal_df.merge(
    cricsheet_id_lookup.rename(columns={"player_id": "player_out_id"}),
    on="player_out_id",
    how="left",
    validate="many_to_one"
)

print("Dismissal dataset shape:", player_dismissal_df.shape)

print("Missing CricketIQ IDs:",
      player_dismissal_df["cricketiq_player_id"].isna().sum())

print("Total dismissal records:", len(player_dismissal_df))

display(player_dismissal_df.head())

Dismissal dataset shape: (37711, 7)
Missing CricketIQ IDs: 0
Total dismissal records: 37711


,match_id,innings_number,batting_team,player_out,player_out_id,dismissals,cricketiq_player_id
0,1000887,1,Australia,CA Lynn,45eda7c8,1,CS_45eda7c8
1,1000887,1,Australia,DA Warner,dcce6f09,1,CS_dcce6f09
2,1000887,1,Australia,GJ Maxwell,b681e71e,1,CS_b681e71e
3,1000887,1,Australia,JP Faulkner,808f425a,1,CS_808f425a
4,1000887,1,Australia,MA Starc,3fb19989,1,CS_3fb19989


In [102]:
# Attach CricketIQ IDs to bowler-wicket records

dismissal_bowler_df = dismissal_bowler_df.merge(
    cricsheet_id_lookup.rename(columns={"player_id": "bowler_id"}),
    on="bowler_id",
    how="left",
    validate="many_to_one"
)

print("Bowler-wicket dataset shape:", dismissal_bowler_df.shape)

print("Missing CricketIQ IDs:",
      dismissal_bowler_df["cricketiq_player_id"].isna().sum())

print("Total bowler-wicket records:", len(dismissal_bowler_df))

display(dismissal_bowler_df.head())

Bowler-wicket dataset shape: (34975, 7)
Missing CricketIQ IDs: 0
Total bowler-wicket records: 34975


,match_id,innings_number,bowler,bowler_id,player_out,dismissal_type,cricketiq_player_id
0,1000887,1,Mohammad Amir,e174dadd,DA Warner,bowled,CRIQ_e174dadd
1,1000887,1,Mohammad Amir,e174dadd,SPD Smith,caught,CRIQ_e174dadd
2,1000887,1,Hasan Ali,2911de16,CA Lynn,caught,CRIQ_2911de16
3,1000887,1,Imad Wasim,9cb8d7a6,TM Head,caught,CRIQ_9cb8d7a6
4,1000887,1,Imad Wasim,9cb8d7a6,MR Marsh,caught,CRIQ_9cb8d7a6


In [103]:
# Build Crickrida player name-to-CricketIQ ID lookup

crickrida_id_lookup = player_registry_final[
    player_registry_final["source"].isin(["Both", "Crickrida"])
][
    ["crickrida_name", "player_name", "cricketiq_player_id"]
].copy()

# Use the Crickrida name for shared identities,
# and the source-only name for Crickrida-only identities.
crickrida_id_lookup["lookup_name"] = (
    crickrida_id_lookup["crickrida_name"]
    .fillna(crickrida_id_lookup["player_name"])
)

crickrida_id_lookup = crickrida_id_lookup[
    ["lookup_name", "cricketiq_player_id"]
].dropna()

print("Crickrida lookup shape:", crickrida_id_lookup.shape)

print("Duplicate lookup names:",
      crickrida_id_lookup["lookup_name"].duplicated().sum())

print("Missing CricketIQ IDs:",
      crickrida_id_lookup["cricketiq_player_id"].isna().sum())

display(crickrida_id_lookup.head(10))

Crickrida lookup shape: (454, 2)
Duplicate lookup names: 0
Missing CricketIQ IDs: 0


,lookup_name,cricketiq_player_id
0,Babar Azam,CRIQ_8a75e999
1,Hasan Ali,CRIQ_2911de16
2,Imad Wasim,CRIQ_9cb8d7a6
3,Mohammad Amir,CRIQ_e174dadd
4,Mohammad Hafeez,CRIQ_9ab63e7b
5,Mohammad Rizwan,CRIQ_2f26ac1a
6,Sharjeel Khan,CRIQ_193ef196
7,Umar Akmal,CRIQ_fd2bf2a0
8,Wahab Riaz,CRIQ_b3118300
9,Asad Shafiq,CRIQ_50250426


In [104]:
# Attach CricketIQ IDs to Afghanistan batting records

afg_player_batting_df = afg_player_batting_df.merge(
    crickrida_id_lookup.rename(
        columns={"lookup_name": "player_name"}
    ),
    on="player_name",
    how="left",
    validate="many_to_one"
)

print("Afghanistan batting shape:", afg_player_batting_df.shape)

print("Missing CricketIQ IDs:",
      afg_player_batting_df["cricketiq_player_id"].isna().sum())

print("Total batting records:", len(afg_player_batting_df))

print("\nIdentity source distribution:")
print(
    afg_player_batting_df["cricketiq_player_id"]
    .str.split("_")
    .str[0]
    .value_counts()
)

display(afg_player_batting_df.head())

Afghanistan batting shape: (2719, 12)
Missing CricketIQ IDs: 0
Total batting records: 2719

Identity source distribution:
cricketiq_player_id
CR      2312
CRIQ     407
Name: count, dtype: int64


,player_name,dismissal,runs_scored,balls_faced,fours,sixes,innings_number,date,match_url,strike_rate,batting_team,cricketiq_player_id
0,Paul Stirling,bowled b Khaliq Dad,6,12,1,0,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,50.000000,Ireland,CR_paulstirling
1,James Hall,caught Karim Sadiq b Shapoor Zadran,15,29,2,0,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,51.724138,Ireland,CR_jameshall
2,Alex Cusack,caught Shabir Noori b Khaliq Dad,2,17,0,0,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,11.764706,Ireland,CR_alexcusack
3,Kevin O'Brien,caught Hamid Hassan b Samiullah Shinwari,44,68,5,0,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,64.705882,Ireland,CR_kevinobrien
4,Andrew Poynter,caught Karim Sadiq b Hamid Hassan,78,109,6,1,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,71.559633,Ireland,CR_andrewpoynter


In [105]:
# Attach CricketIQ IDs to Afghanistan bowling records

afg_player_bowling_df = afg_player_bowling_df.merge(
    crickrida_id_lookup.rename(
        columns={"lookup_name": "player_name"}
    ),
    on="player_name",
    how="left",
    validate="many_to_one"
)

print("Afghanistan bowling shape:", afg_player_bowling_df.shape)

print("Missing CricketIQ IDs:",
      afg_player_bowling_df["cricketiq_player_id"].isna().sum())

print("Total bowling records:", len(afg_player_bowling_df))

print("\nIdentity source distribution:")
print(
    afg_player_bowling_df["cricketiq_player_id"]
    .str.split("_")
    .str[0]
    .value_counts()
)

display(afg_player_bowling_df.head())

Afghanistan bowling shape: (1765, 16)
Missing CricketIQ IDs: 0
Total bowling records: 1765

Identity source distribution:
cricketiq_player_id
CR      1536
CRIQ     229
Name: count, dtype: int64


,player_name,overs_display,maidens,runs_conceded,wickets,wides,no_balls,legal_balls,innings_number,date,match_url,economy,batting_team,bowling_team,opponent,cricketiq_player_id
0,Shapoor Zadran,10.0,3,39,1,1,1,60,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,3.900,Ireland,Afghanistan,Ireland,CR_shapoorzadran
1,Khaliq Dad,8.0,1,27,2,4,1,48,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,3.375,Ireland,Afghanistan,Ireland,CR_khaliqdad
2,Hamid Hassan,8.0,0,44,3,0,1,48,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,5.500,Ireland,Afghanistan,Ireland,CR_hamidhassan
3,Mohammad Nabi,10.0,0,55,1,1,0,60,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,5.500,Ireland,Afghanistan,Ireland,CR_mohammadnabi
4,Samiullah Shinwari,10.0,0,52,1,0,0,60,1,2010-07-03,https://crickrida.com/matches/afghanistan-irel...,5.200,Ireland,Afghanistan,Ireland,CR_samiullahshinwari


In [106]:
# Collect all mapped player datasets

player_datasets = {
    "Cricsheet batting": (player_batting_df, "batter_id"),
    "Cricsheet bowling": (player_bowling_df, "bowler_id"),
    "Cricsheet dismissals": (player_dismissal_df, "player_out_id"),
    "Cricsheet bowler wickets": (dismissal_bowler_df, "bowler_id"),
    "Afghanistan batting": (afg_player_batting_df, "player_name"),
    "Afghanistan bowling": (afg_player_bowling_df, "player_name")
}

# Check missing IDs and identity consistency
for dataset_name, (df, source_id_col) in player_datasets.items():
    print(f"\n{dataset_name}")
    print("Records:", len(df))
    print("Missing CricketIQ IDs:",
          df["cricketiq_player_id"].isna().sum())

# Verify cross-source identities retain the same ID
shared_names = set(
    player_identity_map["cricsheet_name"]
) & set(
    player_identity_map["crickrida_name"]
)

print("\nShared identity count:", len(shared_names))

print("Registry duplicate CricketIQ IDs:",
      player_registry_final["cricketiq_player_id"].duplicated().sum())


Cricsheet batting
Records: 45432
Missing CricketIQ IDs: 0

Cricsheet bowling
Records: 31186
Missing CricketIQ IDs: 0

Cricsheet dismissals
Records: 37711
Missing CricketIQ IDs: 0

Cricsheet bowler wickets
Records: 34975
Missing CricketIQ IDs: 0

Afghanistan batting
Records: 2719
Missing CricketIQ IDs: 0

Afghanistan bowling
Records: 1765
Missing CricketIQ IDs: 0

Shared identity count: 98
Registry duplicate CricketIQ IDs: 0


In [107]:
# Verify that all shared identities have consistent IDs

identity_check = player_identity_map[
    [
        "cricketiq_player_id",
        "cricsheet_name",
        "crickrida_name",
        "match_status"
    ]
].copy()

# Check that each identity has a unique ID
print("Verified cross-source identities:", len(identity_check))

print("Unique CricketIQ IDs:",
      identity_check["cricketiq_player_id"].nunique())

print("Missing Cricsheet names:",
      identity_check["cricsheet_name"].isna().sum())

print("Missing Crickrida names:",
      identity_check["crickrida_name"].isna().sum())

print("\nMatch status distribution:")
print(identity_check["match_status"].value_counts())

print("\nVerified alias examples:")
display(
    identity_check[
        identity_check["match_status"] == "verified_alias"
    ]
)

Verified cross-source identities: 106
Unique CricketIQ IDs: 106
Missing Cricsheet names: 0
Missing Crickrida names: 0

Match status distribution:
match_status
exact_name_match    98
verified_alias       8
Name: count, dtype: int64

Verified alias examples:


,cricketiq_player_id,cricsheet_name,crickrida_name,match_status
98,CRIQ_cb5100f1,Ebadat Hossain,Ebadot Hossain,verified_alias
99,CRIQ_a343262c,JE Root,Joe Root,verified_alias
100,CRIQ_0404d43c,Liton Das,Litton Das,verified_alias
101,CRIQ_1274d7ab,Nazmul Hossain Shanto,Najmul Hossain Shanto,verified_alias
102,CRIQ_aad0c365,Nithish Kumar Reddy,Nitish Kumar Reddy,verified_alias
103,CRIQ_85e0cf10,M Prasidh Krishna,Prasidh Krishna,verified_alias
104,CRIQ_e94915e6,SM Curran,Sam Curran,verified_alias
105,CRIQ_2254ab79,Sarfraz Ahmed,Sarfaraz Ahmed,verified_alias


In [108]:
# Build source-to-CricketIQ lookup dictionaries
cricsheet_id_map = cricsheet_id_lookup.set_index(
    "player_id"
)["cricketiq_player_id"]

crickrida_name_map = crickrida_id_lookup.set_index(
    "lookup_name"
)["cricketiq_player_id"]

# Define all six validation checks
validation_checks = {
    "Cricsheet batting": (
        player_batting_df,
        "batter_id",
        cricsheet_id_map
    ),
    "Cricsheet bowling": (
        player_bowling_df,
        "bowler_id",
        cricsheet_id_map
    ),
    "Cricsheet dismissals": (
        player_dismissal_df,
        "player_out_id",
        cricsheet_id_map
    ),
    "Cricsheet bowler wickets": (
        dismissal_bowler_df,
        "bowler_id",
        cricsheet_id_map
    ),
    "Afghanistan batting": (
        afg_player_batting_df,
        "player_name",
        crickrida_name_map
    ),
    "Afghanistan bowling": (
        afg_player_bowling_df,
        "player_name",
        crickrida_name_map
    )
}

# Compare assigned IDs with expected IDs
for name, (df, source_column, lookup) in validation_checks.items():
    expected_ids = df[source_column].map(lookup)

    mismatches = (
        df["cricketiq_player_id"] != expected_ids
    ).sum()

    print(f"\n{name}")
    print("Records:", len(df))
    print("Missing expected mappings:", expected_ids.isna().sum())
    print("Incorrect CricketIQ IDs:", mismatches)
    print("Validation:", "PASS" if mismatches == 0 and expected_ids.notna().all() else "CHECK")


Cricsheet batting
Records: 45432
Missing expected mappings: 0
Incorrect CricketIQ IDs: 0
Validation: PASS

Cricsheet bowling
Records: 31186
Missing expected mappings: 0
Incorrect CricketIQ IDs: 0
Validation: PASS

Cricsheet dismissals
Records: 37711
Missing expected mappings: 0
Incorrect CricketIQ IDs: 0
Validation: PASS

Cricsheet bowler wickets
Records: 34975
Missing expected mappings: 0
Incorrect CricketIQ IDs: 0
Validation: PASS

Afghanistan batting
Records: 2719
Missing expected mappings: 0
Incorrect CricketIQ IDs: 0
Validation: PASS

Afghanistan bowling
Records: 1765
Missing expected mappings: 0
Incorrect CricketIQ IDs: 0
Validation: PASS


In [109]:
# Save the corrected player identity registry
player_registry_final.to_csv(
    PLAYER_DATA_DIR / "cricketiq_player_identity_registry.csv",
    index=False
)

# Save updated Cricsheet player datasets
player_batting_df.to_csv(
    PLAYER_DATA_DIR / "cricsheet_player_batting.csv",
    index=False
)

player_bowling_df.to_csv(
    PLAYER_DATA_DIR / "cricsheet_player_bowling.csv",
    index=False
)

player_dismissal_df.to_csv(
    PLAYER_DATA_DIR / "cricsheet_player_dismissals.csv",
    index=False
)

dismissal_bowler_df.to_csv(
    PLAYER_DATA_DIR / "cricsheet_bowler_wickets.csv",
    index=False
)

# Save updated Afghanistan player datasets
afg_player_batting_df.to_csv(
    PLAYER_DATA_DIR / "afghanistan_player_batting.csv",
    index=False
)

afg_player_bowling_df.to_csv(
    PLAYER_DATA_DIR / "afghanistan_player_bowling.csv",
    index=False
)

# Reload and verify saved files
files_to_check = {
    "Player identity registry": "cricketiq_player_identity_registry.csv",
    "Cricsheet batting": "cricsheet_player_batting.csv",
    "Cricsheet bowling": "cricsheet_player_bowling.csv",
    "Cricsheet dismissals": "cricsheet_player_dismissals.csv",
    "Cricsheet bowler wickets": "cricsheet_bowler_wickets.csv",
    "Afghanistan batting": "afghanistan_player_batting.csv",
    "Afghanistan bowling": "afghanistan_player_bowling.csv"
}

for name, filename in files_to_check.items():
    saved_df = pd.read_csv(PLAYER_DATA_DIR / filename)

    print(
        f"{name}: {saved_df.shape} | "
        f"Missing CricketIQ IDs: "
        f"{saved_df['cricketiq_player_id'].isna().sum()}"
    )

Player identity registry: (3635, 8) | Missing CricketIQ IDs: 0
Cricsheet batting: (45432, 11) | Missing CricketIQ IDs: 0
Cricsheet bowling: (31186, 12) | Missing CricketIQ IDs: 0
Cricsheet dismissals: (37711, 7) | Missing CricketIQ IDs: 0
Cricsheet bowler wickets: (34975, 7) | Missing CricketIQ IDs: 0
Afghanistan batting: (2719, 12) | Missing CricketIQ IDs: 0
Afghanistan bowling: (1765, 16) | Missing CricketIQ IDs: 0


In [110]:
print("Registry columns:")
print(player_registry_final.columns.tolist())

display(
    player_registry_final[
        [
            "cricketiq_player_id",
            "cricsheet_name",
            "crickrida_name",
            "player_names",
            "player_name",
            "match_status",
            "source"
        ]
    ].head(15)
)

Registry columns:
['cricketiq_player_id', 'player_id', 'cricsheet_name', 'crickrida_name', 'match_status', 'source', 'player_names', 'player_name']


,cricketiq_player_id,cricsheet_name,crickrida_name,player_names,player_name,match_status,source
0,CRIQ_8a75e999,Babar Azam,Babar Azam,NaN,NaN,exact_name_match,Both
1,CRIQ_2911de16,Hasan Ali,Hasan Ali,NaN,NaN,exact_name_match,Both
2,CRIQ_9cb8d7a6,Imad Wasim,Imad Wasim,NaN,NaN,exact_name_match,Both
3,CRIQ_e174dadd,Mohammad Amir,Mohammad Amir,NaN,NaN,exact_name_match,Both
4,CRIQ_9ab63e7b,Mohammad Hafeez,Mohammad Hafeez,NaN,NaN,exact_name_match,Both
5,CRIQ_2f26ac1a,Mohammad Rizwan,Mohammad Rizwan,NaN,NaN,exact_name_match,Both
6,CRIQ_193ef196,Sharjeel Khan,Sharjeel Khan,NaN,NaN,exact_name_match,Both
7,CRIQ_fd2bf2a0,Umar Akmal,Umar Akmal,NaN,NaN,exact_name_match,Both
8,CRIQ_b3118300,Wahab Riaz,Wahab Riaz,NaN,NaN,exact_name_match,Both
9,CRIQ_50250426,Asad Shafiq,Asad Shafiq,NaN,NaN,exact_name_match,Both


In [111]:
# Create canonical player names

player_registry_final["canonical_player_name"] = (
    player_registry_final["crickrida_name"]
    .fillna(player_registry_final["cricsheet_name"])
    .fillna(player_registry_final["player_name"])
)

# For Cricsheet-only players with multiple registered names,
# use the first alphabetically sorted registered name.
mask = (
    player_registry_final["source"].eq("Cricsheet") &
    player_registry_final["canonical_player_name"].isna()
)

player_registry_final.loc[mask, "canonical_player_name"] = (
    player_registry_final.loc[mask, "player_names"]
    .apply(lambda names: names[0] if isinstance(names, list) and names else None)
)

print("Missing canonical names:",
      player_registry_final["canonical_player_name"].isna().sum())

print("Duplicate CricketIQ IDs:",
      player_registry_final["cricketiq_player_id"].duplicated().sum())

print("\nVerified alias canonical names:")

display(
    player_registry_final[
        player_registry_final["match_status"] == "verified_alias"
    ][
        [
            "cricketiq_player_id",
            "cricsheet_name",
            "crickrida_name",
            "canonical_player_name"
        ]
    ]
)

Missing canonical names: 0
Duplicate CricketIQ IDs: 0

Verified alias canonical names:


,cricketiq_player_id,cricsheet_name,crickrida_name,canonical_player_name
98,CRIQ_cb5100f1,Ebadat Hossain,Ebadot Hossain,Ebadot Hossain
99,CRIQ_a343262c,JE Root,Joe Root,Joe Root
100,CRIQ_0404d43c,Liton Das,Litton Das,Litton Das
101,CRIQ_1274d7ab,Nazmul Hossain Shanto,Najmul Hossain Shanto,Najmul Hossain Shanto
102,CRIQ_aad0c365,Nithish Kumar Reddy,Nitish Kumar Reddy,Nitish Kumar Reddy
103,CRIQ_85e0cf10,M Prasidh Krishna,Prasidh Krishna,Prasidh Krishna
104,CRIQ_e94915e6,SM Curran,Sam Curran,Sam Curran
105,CRIQ_2254ab79,Sarfraz Ahmed,Sarfaraz Ahmed,Sarfaraz Ahmed


In [112]:
# Create a unique CricketIQ ID-to-canonical-name lookup

canonical_name_lookup = player_registry_final[
    ["cricketiq_player_id", "canonical_player_name"]
].copy()

print("Lookup records:", len(canonical_name_lookup))
print(
    "Duplicate CricketIQ IDs:",
    canonical_name_lookup["cricketiq_player_id"].duplicated().sum()
)
print(
    "Missing canonical names:",
    canonical_name_lookup["canonical_player_name"].isna().sum()
)

display(canonical_name_lookup.head(10))

Lookup records: 3635
Duplicate CricketIQ IDs: 0
Missing canonical names: 0


,cricketiq_player_id,canonical_player_name
0,CRIQ_8a75e999,Babar Azam
1,CRIQ_2911de16,Hasan Ali
2,CRIQ_9cb8d7a6,Imad Wasim
3,CRIQ_e174dadd,Mohammad Amir
4,CRIQ_9ab63e7b,Mohammad Hafeez
5,CRIQ_2f26ac1a,Mohammad Rizwan
6,CRIQ_193ef196,Sharjeel Khan
7,CRIQ_fd2bf2a0,Umar Akmal
8,CRIQ_b3118300,Wahab Riaz
9,CRIQ_50250426,Asad Shafiq


In [113]:
# Attach canonical names to Cricsheet batting records

player_batting_df = player_batting_df.merge(
    canonical_name_lookup,
    on="cricketiq_player_id",
    how="left",
    validate="many_to_one"
)

print("Shape:", player_batting_df.shape)
print("Missing canonical names:",
      player_batting_df["canonical_player_name"].isna().sum())

display(
    player_batting_df[
        ["batter", "cricketiq_player_id", "canonical_player_name"]
    ].head(10)
)

Shape: (45432, 12)
Missing canonical names: 0


,batter,cricketiq_player_id,canonical_player_name
0,B Stanlake,CS_6834d1f2,B Stanlake
1,CA Lynn,CS_45eda7c8,CA Lynn
2,DA Warner,CS_dcce6f09,DA Warner
3,GJ Maxwell,CS_b681e71e,GJ Maxwell
4,JP Faulkner,CS_808f425a,JP Faulkner
5,MA Starc,CS_3fb19989,MA Starc
6,MR Marsh,CS_3d8feaf8,MR Marsh
7,MS Wade,CS_afa7e784,MS Wade
8,PJ Cummins,CS_ded9240e,PJ Cummins
9,SPD Smith,CS_30a45b23,SPD Smith


In [114]:
# Attach canonical names to Cricsheet bowling records

player_bowling_df = player_bowling_df.merge(
    canonical_name_lookup,
    on="cricketiq_player_id",
    how="left",
    validate="many_to_one"
)

print("Shape:", player_bowling_df.shape)

print("Missing canonical names:",
      player_bowling_df["canonical_player_name"].isna().sum())

display(
    player_bowling_df[
        ["bowler", "cricketiq_player_id", "canonical_player_name"]
    ].head(10)
)

Shape: (31186, 13)
Missing canonical names: 0


,bowler,cricketiq_player_id,canonical_player_name
0,Hasan Ali,CRIQ_2911de16,Hasan Ali
1,Imad Wasim,CRIQ_9cb8d7a6,Imad Wasim
2,Mohammad Amir,CRIQ_e174dadd,Mohammad Amir
3,Mohammad Hafeez,CRIQ_9ab63e7b,Mohammad Hafeez
4,Mohammad Nawaz (3),CRIQ_3086f7a4,Mohammad Nawaz
5,Wahab Riaz,CRIQ_b3118300,Wahab Riaz
6,B Stanlake,CS_6834d1f2,B Stanlake
7,JP Faulkner,CS_808f425a,JP Faulkner
8,MA Starc,CS_3fb19989,MA Starc
9,MR Marsh,CS_3d8feaf8,MR Marsh


In [115]:
# Attach canonical names to Cricsheet dismissal records

player_dismissal_df = player_dismissal_df.merge(
    canonical_name_lookup,
    on="cricketiq_player_id",
    how="left",
    validate="many_to_one"
)

print("Shape:", player_dismissal_df.shape)

print("Missing canonical names:",
      player_dismissal_df["canonical_player_name"].isna().sum())

display(
    player_dismissal_df[
        ["player_out", "cricketiq_player_id", "canonical_player_name"]
    ].head(10)
)

Shape: (37711, 8)
Missing canonical names: 0


,player_out,cricketiq_player_id,canonical_player_name
0,CA Lynn,CS_45eda7c8,CA Lynn
1,DA Warner,CS_dcce6f09,DA Warner
2,GJ Maxwell,CS_b681e71e,GJ Maxwell
3,JP Faulkner,CS_808f425a,JP Faulkner
4,MA Starc,CS_3fb19989,MA Starc
5,MR Marsh,CS_3d8feaf8,MR Marsh
6,PJ Cummins,CS_ded9240e,PJ Cummins
7,SPD Smith,CS_30a45b23,SPD Smith
8,TM Head,CS_12b610c2,TM Head
9,Azhar Ali,CS_211de69f,Azhar Ali


In [116]:
# Attach canonical names to Cricsheet bowler-wicket records

dismissal_bowler_df = dismissal_bowler_df.merge(
    canonical_name_lookup,
    on="cricketiq_player_id",
    how="left",
    validate="many_to_one"
)

print("Shape:", dismissal_bowler_df.shape)

print("Missing canonical names:",
      dismissal_bowler_df["canonical_player_name"].isna().sum())

display(
    dismissal_bowler_df[
        ["bowler", "cricketiq_player_id", "canonical_player_name"]
    ].head(10)
)

Shape: (34975, 8)
Missing canonical names: 0


,bowler,cricketiq_player_id,canonical_player_name
0,Mohammad Amir,CRIQ_e174dadd,Mohammad Amir
1,Mohammad Amir,CRIQ_e174dadd,Mohammad Amir
2,Hasan Ali,CRIQ_2911de16,Hasan Ali
3,Imad Wasim,CRIQ_9cb8d7a6,Imad Wasim
4,Imad Wasim,CRIQ_9cb8d7a6,Imad Wasim
5,Hasan Ali,CRIQ_2911de16,Hasan Ali
6,Mohammad Nawaz (3),CRIQ_3086f7a4,Mohammad Nawaz
7,Hasan Ali,CRIQ_2911de16,Hasan Ali
8,JP Faulkner,CS_808f425a,JP Faulkner
9,JP Faulkner,CS_808f425a,JP Faulkner


In [117]:
# Attach canonical names to Afghanistan batting records

afg_player_batting_df = afg_player_batting_df.merge(
    canonical_name_lookup,
    on="cricketiq_player_id",
    how="left",
    validate="many_to_one"
)

print("Shape:", afg_player_batting_df.shape)

print("Missing canonical names:",
      afg_player_batting_df["canonical_player_name"].isna().sum())

display(
    afg_player_batting_df[
        ["player_name", "cricketiq_player_id", "canonical_player_name"]
    ].head(10)
)

Shape: (2719, 13)
Missing canonical names: 0


,player_name,cricketiq_player_id,canonical_player_name
0,Paul Stirling,CR_paulstirling,Paul Stirling
1,James Hall,CR_jameshall,James Hall
2,Alex Cusack,CR_alexcusack,Alex Cusack
3,Kevin O'Brien,CR_kevinobrien,Kevin O'Brien
4,Andrew Poynter,CR_andrewpoynter,Andrew Poynter
5,Andrew White,CR_andrewwhite,Andrew White
6,John Mooney,CR_johnmooney,John Mooney
7,Trent Johnston,CR_trentjohnston,Trent Johnston
8,Nigel Jones,CR_nigeljones,Nigel Jones
9,Rory McCann,CR_rorymccann,Rory McCann


In [118]:
# Attach canonical names to Afghanistan bowling records

afg_player_bowling_df = afg_player_bowling_df.merge(
    canonical_name_lookup,
    on="cricketiq_player_id",
    how="left",
    validate="many_to_one"
)

print("Shape:", afg_player_bowling_df.shape)

print("Missing canonical names:",
      afg_player_bowling_df["canonical_player_name"].isna().sum())

display(
    afg_player_bowling_df[
        ["player_name", "cricketiq_player_id", "canonical_player_name"]
    ].head(10)
)

Shape: (1765, 17)
Missing canonical names: 0


,player_name,cricketiq_player_id,canonical_player_name
0,Shapoor Zadran,CR_shapoorzadran,Shapoor Zadran
1,Khaliq Dad,CR_khaliqdad,Khaliq Dad
2,Hamid Hassan,CR_hamidhassan,Hamid Hassan
3,Mohammad Nabi,CR_mohammadnabi,Mohammad Nabi
4,Samiullah Shinwari,CR_samiullahshinwari,Samiullah Shinwari
5,Nawroz Mangal,CR_nawrozmangal,Nawroz Mangal
6,Kevin O'Brien,CR_kevinobrien,Kevin O'Brien
7,Trent Johnston,CR_trentjohnston,Trent Johnston
8,Alex Cusack,CR_alexcusack,Alex Cusack
9,George Dockrell,CR_georgedockrell,George Dockrell


In [119]:
# Collect all six updated datasets
player_datasets = {
    "Cricsheet batting": player_batting_df,
    "Cricsheet bowling": player_bowling_df,
    "Cricsheet dismissals": player_dismissal_df,
    "Cricsheet bowler wickets": dismissal_bowler_df,
    "Afghanistan batting": afg_player_batting_df,
    "Afghanistan bowling": afg_player_bowling_df
}

# Check completeness and ID-to-name consistency
for name, df in player_datasets.items():
    print(f"\n{name}")
    print("Records:", len(df))
    print("Missing CricketIQ IDs:",
          df["cricketiq_player_id"].isna().sum())
    print("Missing canonical names:",
          df["canonical_player_name"].isna().sum())

# Combine identity pairs from all six datasets
identity_pairs = pd.concat(
    [
        df[["cricketiq_player_id", "canonical_player_name"]]
        for df in player_datasets.values()
    ],
    ignore_index=True
)

# Check whether one ID has multiple canonical names
names_per_id = identity_pairs.groupby(
    "cricketiq_player_id"
)["canonical_player_name"].nunique()

print("\nTotal performance records:", len(identity_pairs))
print("IDs with multiple canonical names:",
      (names_per_id > 1).sum())

print("Unique IDs across performance datasets:",
      identity_pairs["cricketiq_player_id"].nunique())


Cricsheet batting
Records: 45432
Missing CricketIQ IDs: 0
Missing canonical names: 0

Cricsheet bowling
Records: 31186
Missing CricketIQ IDs: 0
Missing canonical names: 0

Cricsheet dismissals
Records: 37711
Missing CricketIQ IDs: 0
Missing canonical names: 0

Cricsheet bowler wickets
Records: 34975
Missing CricketIQ IDs: 0
Missing canonical names: 0

Afghanistan batting
Records: 2719
Missing CricketIQ IDs: 0
Missing canonical names: 0

Afghanistan bowling
Records: 1765
Missing CricketIQ IDs: 0
Missing canonical names: 0

Total performance records: 153788
IDs with multiple canonical names: 0
Unique IDs across performance datasets: 2356


In [120]:
# Save updated player identity registry
player_registry_final.to_csv(
    PLAYER_DATA_DIR / "cricketiq_player_identity_registry.csv",
    index=False
)

# Save all six datasets with canonical names
updated_files = {
    "cricsheet_player_batting.csv": player_batting_df,
    "cricsheet_player_bowling.csv": player_bowling_df,
    "cricsheet_player_dismissals.csv": player_dismissal_df,
    "cricsheet_bowler_wickets.csv": dismissal_bowler_df,
    "afghanistan_player_batting.csv": afg_player_batting_df,
    "afghanistan_player_bowling.csv": afg_player_bowling_df
}

for filename, df in updated_files.items():
    df.to_csv(PLAYER_DATA_DIR / filename, index=False)

    print(f"Saved: {filename} | Shape: {df.shape}")

Saved: cricsheet_player_batting.csv | Shape: (45432, 12)
Saved: cricsheet_player_bowling.csv | Shape: (31186, 13)
Saved: cricsheet_player_dismissals.csv | Shape: (37711, 8)
Saved: cricsheet_bowler_wickets.csv | Shape: (34975, 8)
Saved: afghanistan_player_batting.csv | Shape: (2719, 13)
Saved: afghanistan_player_bowling.csv | Shape: (1765, 17)


In [121]:
# Reload and verify all saved player datasets

files_to_verify = {
    "Player identity registry": "cricketiq_player_identity_registry.csv",
    "Cricsheet batting": "cricsheet_player_batting.csv",
    "Cricsheet bowling": "cricsheet_player_bowling.csv",
    "Cricsheet dismissals": "cricsheet_player_dismissals.csv",
    "Cricsheet bowler wickets": "cricsheet_bowler_wickets.csv",
    "Afghanistan batting": "afghanistan_player_batting.csv",
    "Afghanistan bowling": "afghanistan_player_bowling.csv"
}

for name, filename in files_to_verify.items():
    saved_df = pd.read_csv(PLAYER_DATA_DIR / filename)

    print(f"\n{name}")
    print("Shape:", saved_df.shape)
    print("Has CricketIQ ID:",
          "cricketiq_player_id" in saved_df.columns)
    print("Has canonical name:",
          "canonical_player_name" in saved_df.columns)

    if "canonical_player_name" in saved_df.columns:
        print("Missing canonical names:",
              saved_df["canonical_player_name"].isna().sum())


Player identity registry
Shape: (3635, 9)
Has CricketIQ ID: True
Has canonical name: True
Missing canonical names: 0

Cricsheet batting
Shape: (45432, 12)
Has CricketIQ ID: True
Has canonical name: True
Missing canonical names: 0

Cricsheet bowling
Shape: (31186, 13)
Has CricketIQ ID: True
Has canonical name: True
Missing canonical names: 0

Cricsheet dismissals
Shape: (37711, 8)
Has CricketIQ ID: True
Has canonical name: True
Missing canonical names: 0

Cricsheet bowler wickets
Shape: (34975, 8)
Has CricketIQ ID: True
Has canonical name: True
Missing canonical names: 0

Afghanistan batting
Shape: (2719, 13)
Has CricketIQ ID: True
Has canonical name: True
Missing canonical names: 0

Afghanistan bowling
Shape: (1765, 17)
Has CricketIQ ID: True
Has canonical name: True
Missing canonical names: 0


In [122]:
print("CRICKRIDA BATTING")
print(afg_player_batting_df.columns.tolist())

print("\nCRICKRIDA BOWLING")
print(afg_player_bowling_df.columns.tolist())

print("\nCRICSHEET BATTING")
print(player_batting_df.columns.tolist())

print("\nCRICSHEET BOWLING")
print(player_bowling_df.columns.tolist())

print("\nCRICSHEET MATCHES")
print(matches_df.columns.tolist())

print("\nCRICKRIDA MATCHES")
print(afg_match_df.columns.tolist())

CRICKRIDA BATTING
['player_name', 'dismissal', 'runs_scored', 'balls_faced', 'fours', 'sixes', 'innings_number', 'date', 'match_url', 'strike_rate', 'batting_team', 'cricketiq_player_id', 'canonical_player_name']

CRICKRIDA BOWLING
['player_name', 'overs_display', 'maidens', 'runs_conceded', 'wickets', 'wides', 'no_balls', 'legal_balls', 'innings_number', 'date', 'match_url', 'economy', 'batting_team', 'bowling_team', 'opponent', 'cricketiq_player_id', 'canonical_player_name']

CRICSHEET BATTING
['match_id', 'innings_number', 'batting_team', 'batter', 'batter_id', 'runs', 'balls_faced', 'fours', 'sixes', 'strike_rate', 'cricketiq_player_id', 'canonical_player_name']

CRICSHEET BOWLING
['match_id', 'innings_number', 'bowler', 'bowler_id', 'runs_conceded', 'wides', 'no_balls', 'wickets', 'legal_balls', 'overs', 'economy', 'cricketiq_player_id', 'canonical_player_name']

CRICSHEET MATCHES
['match_id', 'date', 'team_1', 'team_2', 'venue', 'city', 'season', 'event', 'toss_winner', 'toss_dec

## 4. Player Identity Discovery and Manual Verification

Investigate potential cross-source player matches using names, appearance histories, career dates, match records, and country context. Review ambiguous candidates before approving identity mappings; similarity alone is not treated as proof of identity.

In [123]:
# Inspect Cricsheet innings structure
print("CRICSHEET INNINGS COLUMNS")
print(innings_df.columns.tolist())

print("\nCRICSHEET INNINGS SAMPLE")
display(innings_df.head())

# Inspect date formats
print("\nDATE INFORMATION")

print("Crickrida date dtype:", afg_player_batting_df["date"].dtype)
print("Cricsheet date dtype:", matches_df["date"].dtype)

print("\nCrickrida date range:")
print(
    afg_player_batting_df["date"].min(),
    "to",
    afg_player_batting_df["date"].max()
)

print("\nCricsheet date range:")
print(
    matches_df["date"].min(),
    "to",
    matches_df["date"].max()
)

# List countries represented in Crickrida scorecards
crickrida_teams = sorted(
    set(afg_match_df["team_1"].dropna()) |
    set(afg_match_df["team_2"].dropna())
)

cricsheet_teams = sorted(
    set(matches_df["team_1"].dropna()) |
    set(matches_df["team_2"].dropna())
)

print("\nCRICKRIDA COUNTRIES")
print(crickrida_teams)

print("\nCRICSHEET COUNTRIES")
print(cricsheet_teams)

print("\nNumber of Crickrida teams:", len(crickrida_teams))
print("Number of Cricsheet teams:", len(cricsheet_teams))

CRICSHEET INNINGS COLUMNS
['match_id', 'innings_number', 'batting_team', 'total_runs', 'wickets_lost', 'delivery_records', 'target_runs']

CRICSHEET INNINGS SAMPLE


,match_id,innings_number,batting_team,total_runs,wickets_lost,delivery_records,target_runs
0,1000887,1,Australia,268,9,310,NaN
1,1000887,2,Pakistan,176,10,261,269.0
2,1000889,1,Australia,220,10,297,NaN
3,1000889,2,Pakistan,221,4,293,221.0
4,1000891,1,Pakistan,263,7,308,NaN



DATE INFORMATION
Crickrida date dtype: datetime64[ns]
Cricsheet date dtype: object

Crickrida date range:
2010-07-03 00:00:00 to 2026-10-23 00:00:00

Cricsheet date range:
2002-06-27 to 2026-10-25

CRICKRIDA COUNTRIES
['Afghanistan', 'Australia', 'Bangladesh', 'England', 'India', 'Ireland', 'New Zealand', 'Pakistan', 'South Africa', 'Sri Lanka', 'West Indies', 'Zimbabwe']

CRICSHEET COUNTRIES
['Australia', 'Bangladesh', 'Bermuda', 'Canada', 'England', 'Hong Kong', 'India', 'Ireland', 'Jersey', 'Kenya', 'Namibia', 'Nepal', 'Netherlands', 'New Zealand', 'Oman', 'Pakistan', 'Papua New Guinea', 'Scotland', 'South Africa', 'Sri Lanka', 'United Arab Emirates', 'United States of America', 'West Indies', 'Zimbabwe']

Number of Crickrida teams: 12
Number of Cricsheet teams: 24


In [124]:
print("CRICSHEET INNINGS COLUMNS")
print(innings_df.columns.tolist())

print("\nCRICSHEET INNINGS SAMPLE")
display(innings_df.head(10))

print("\nCRICSHEET MATCH DATE SAMPLE")
display(matches_df[["match_id", "date", "team_1", "team_2"]].head())

CRICSHEET INNINGS COLUMNS
['match_id', 'innings_number', 'batting_team', 'total_runs', 'wickets_lost', 'delivery_records', 'target_runs']

CRICSHEET INNINGS SAMPLE


,match_id,innings_number,batting_team,total_runs,wickets_lost,delivery_records,target_runs
0,1000887,1,Australia,268,9,310,NaN
1,1000887,2,Pakistan,176,10,261,269.0
2,1000889,1,Australia,220,10,297,NaN
3,1000889,2,Pakistan,221,4,293,221.0
4,1000891,1,Pakistan,263,7,308,NaN
5,1000891,2,Australia,265,3,276,264.0
6,1000893,1,Australia,353,6,302,NaN
7,1000893,2,Pakistan,267,10,267,354.0
8,1000895,1,Australia,369,7,306,NaN
9,1000895,2,Pakistan,312,9,307,370.0



CRICSHEET MATCH DATE SAMPLE


,match_id,date,team_1,team_2
0,1000887,2017-01-13,Australia,Pakistan
1,1000889,2017-01-15,Australia,Pakistan
2,1000891,2017-01-19,Australia,Pakistan
3,1000893,2017-01-22,Australia,Pakistan
4,1000895,2017-01-26,Australia,Pakistan


In [126]:
import pandas as pd

# Standardise match IDs before merging
player_batting_df["match_id"] = (
    player_batting_df["match_id"].astype(str)
)

player_bowling_df["match_id"] = (
    player_bowling_df["match_id"].astype(str)
)

matches_df["match_id"] = matches_df["match_id"].astype(str)
innings_df["match_id"] = innings_df["match_id"].astype(str)

# Convert Cricsheet match dates
cricsheet_matches_dates = matches_df[
    ["match_id", "date", "team_1", "team_2"]
].copy()

cricsheet_matches_dates["date"] = pd.to_datetime(
    cricsheet_matches_dates["date"]
)

# 1. Batting appearances
cricsheet_batting_appearances = player_batting_df[
    ["match_id", "batter", "batter_id", "batting_team"]
].copy()

cricsheet_batting_appearances = cricsheet_batting_appearances.merge(
    cricsheet_matches_dates,
    on="match_id",
    how="left",
    validate="many_to_one"
)

cricsheet_batting_appearances = cricsheet_batting_appearances.rename(
    columns={
        "batter": "player_name",
        "batter_id": "source_player_id"
    }
)

cricsheet_batting_appearances["role"] = "batting"
cricsheet_batting_appearances["team"] = (
    cricsheet_batting_appearances["batting_team"]
)

# 2. Bowling appearances
cricsheet_bowling_appearances = player_bowling_df[
    ["match_id", "innings_number", "bowler", "bowler_id"]
].copy()

cricsheet_bowling_appearances = cricsheet_bowling_appearances.merge(
    innings_df[["match_id", "innings_number", "batting_team"]],
    on=["match_id", "innings_number"],
    how="left",
    validate="many_to_one"
)

cricsheet_bowling_appearances = cricsheet_bowling_appearances.merge(
    cricsheet_matches_dates,
    on="match_id",
    how="left",
    validate="many_to_one"
)

cricsheet_bowling_appearances["team"] = (
    cricsheet_bowling_appearances.apply(
        lambda row: (
            row["team_2"] if row["batting_team"] == row["team_1"]
            else row["team_1"] if row["batting_team"] == row["team_2"]
            else None
        ),
        axis=1
    )
)

cricsheet_bowling_appearances = cricsheet_bowling_appearances.rename(
    columns={
        "bowler": "player_name",
        "bowler_id": "source_player_id"
    }
)

cricsheet_bowling_appearances["role"] = "bowling"

# 3. Combine appearances
appearance_columns = [
    "date", "team", "player_name",
    "source_player_id", "match_id", "role"
]

cricsheet_appearances = pd.concat(
    [
        cricsheet_batting_appearances[appearance_columns],
        cricsheet_bowling_appearances[appearance_columns]
    ],
    ignore_index=True
)

cricsheet_appearances["source"] = "Cricsheet"
cricsheet_appearances["date"] = pd.to_datetime(
    cricsheet_appearances["date"]
).dt.normalize()

# 4. Validation
print("CRICSHEET APPEARANCE DATASET")
print("Shape:", cricsheet_appearances.shape)

print("\nMissing values:")
print(
    cricsheet_appearances[
        ["date", "team", "player_name", "source_player_id", "match_id"]
    ].isna().sum()
)

print("\nUnique players:", cricsheet_appearances["source_player_id"].nunique())
print("Unique teams:", cricsheet_appearances["team"].nunique())
print("Unique matches:", cricsheet_appearances["match_id"].nunique())

print("\nRole distribution:")
print(cricsheet_appearances["role"].value_counts())

print("\nSample records:")
display(cricsheet_appearances.head(10))

CRICSHEET APPEARANCE DATASET
Shape: (76618, 7)

Missing values:
date                0
team                0
player_name         0
source_player_id    0
match_id            0
dtype: int64

Unique players: 2008
Unique teams: 24
Unique matches: 2600

Role distribution:
role
batting    45432
bowling    31186
Name: count, dtype: int64

Sample records:


,date,team,player_name,source_player_id,match_id,role,source
0,2017-01-13,Australia,B Stanlake,6834d1f2,1000887,batting,Cricsheet
1,2017-01-13,Australia,CA Lynn,45eda7c8,1000887,batting,Cricsheet
2,2017-01-13,Australia,DA Warner,dcce6f09,1000887,batting,Cricsheet
3,2017-01-13,Australia,GJ Maxwell,b681e71e,1000887,batting,Cricsheet
4,2017-01-13,Australia,JP Faulkner,808f425a,1000887,batting,Cricsheet
5,2017-01-13,Australia,MA Starc,3fb19989,1000887,batting,Cricsheet
6,2017-01-13,Australia,MR Marsh,3d8feaf8,1000887,batting,Cricsheet
7,2017-01-13,Australia,MS Wade,afa7e784,1000887,batting,Cricsheet
8,2017-01-13,Australia,PJ Cummins,ded9240e,1000887,batting,Cricsheet
9,2017-01-13,Australia,SPD Smith,30a45b23,1000887,batting,Cricsheet


In [127]:
# 1. Crickrida batting appearances
crickrida_batting_appearances = afg_player_batting_df[
    [
        "date",
        "batting_team",
        "player_name",
        "cricketiq_player_id",
        "match_url"
    ]
].copy()

crickrida_batting_appearances = (
    crickrida_batting_appearances.rename(
        columns={
            "batting_team": "team",
            "cricketiq_player_id": "source_player_id",
            "match_url": "match_id"
        }
    )
)

crickrida_batting_appearances["role"] = "batting"


# 2. Crickrida bowling appearances
crickrida_bowling_appearances = afg_player_bowling_df[
    [
        "date",
        "bowling_team",
        "player_name",
        "cricketiq_player_id",
        "match_url"
    ]
].copy()

crickrida_bowling_appearances = (
    crickrida_bowling_appearances.rename(
        columns={
            "bowling_team": "team",
            "cricketiq_player_id": "source_player_id",
            "match_url": "match_id"
        }
    )
)

crickrida_bowling_appearances["role"] = "bowling"


# 3. Combine both roles
appearance_columns = [
    "date",
    "team",
    "player_name",
    "source_player_id",
    "match_id",
    "role"
]

crickrida_appearances = pd.concat(
    [
        crickrida_batting_appearances[appearance_columns],
        crickrida_bowling_appearances[appearance_columns]
    ],
    ignore_index=True
)

crickrida_appearances["source"] = "Crickrida"

crickrida_appearances["date"] = pd.to_datetime(
    crickrida_appearances["date"]
).dt.normalize()


# 4. Validation
print("CRICKRIDA APPEARANCE DATASET")
print("Shape:", crickrida_appearances.shape)

print("\nMissing values:")
print(
    crickrida_appearances[
        ["date", "team", "player_name", "source_player_id", "match_id"]
    ].isna().sum()
)

print("\nUnique players:", crickrida_appearances["source_player_id"].nunique())
print("Unique teams:", crickrida_appearances["team"].nunique())
print("Unique matches:", crickrida_appearances["match_id"].nunique())

print("\nCountry distribution:")
print(
    crickrida_appearances.groupby("team")["player_name"]
    .nunique()
    .sort_values(ascending=False)
)

print("\nRole distribution:")
print(crickrida_appearances["role"].value_counts())

print("\nSample records:")
display(crickrida_appearances.head(10))

CRICKRIDA APPEARANCE DATASET
Shape: (4484, 7)

Missing values:
date                0
team                0
player_name         0
source_player_id    0
match_id            0
dtype: int64

Unique players: 454
Unique teams: 12
Unique matches: 148

Country distribution:
team
Afghanistan     58
Zimbabwe        52
Bangladesh      51
Ireland         45
Sri Lanka       40
Pakistan        39
India           36
Australia       29
England         28
West Indies     27
South Africa    26
New Zealand     23
Name: player_name, dtype: int64

Role distribution:
role
batting    2719
bowling    1765
Name: count, dtype: int64

Sample records:


,date,team,player_name,source_player_id,match_id,role,source
0,2010-07-03,Ireland,Paul Stirling,CR_paulstirling,https://crickrida.com/matches/afghanistan-irel...,batting,Crickrida
1,2010-07-03,Ireland,James Hall,CR_jameshall,https://crickrida.com/matches/afghanistan-irel...,batting,Crickrida
2,2010-07-03,Ireland,Alex Cusack,CR_alexcusack,https://crickrida.com/matches/afghanistan-irel...,batting,Crickrida
3,2010-07-03,Ireland,Kevin O'Brien,CR_kevinobrien,https://crickrida.com/matches/afghanistan-irel...,batting,Crickrida
4,2010-07-03,Ireland,Andrew Poynter,CR_andrewpoynter,https://crickrida.com/matches/afghanistan-irel...,batting,Crickrida
5,2010-07-03,Ireland,Andrew White,CR_andrewwhite,https://crickrida.com/matches/afghanistan-irel...,batting,Crickrida
6,2010-07-03,Ireland,John Mooney,CR_johnmooney,https://crickrida.com/matches/afghanistan-irel...,batting,Crickrida
7,2010-07-03,Ireland,Trent Johnston,CR_trentjohnston,https://crickrida.com/matches/afghanistan-irel...,batting,Crickrida
8,2010-07-03,Ireland,Nigel Jones,CR_nigeljones,https://crickrida.com/matches/afghanistan-irel...,batting,Crickrida
9,2010-07-03,Ireland,Rory McCann,CR_rorymccann,https://crickrida.com/matches/afghanistan-irel...,batting,Crickrida


In [128]:
import re
import unicodedata

# Normalization is used only as a comparison key
def normalize_comparison_name(name):
    name = unicodedata.normalize("NFKD", str(name))
    name = "".join(
        char for char in name
        if not unicodedata.combining(char)
    )
    name = name.lower().strip()
    return re.sub(r"[^a-z0-9]", "", name)


# Keep one player record per source, country, date and match
crickrida_unique = crickrida_appearances.drop_duplicates(
    subset=["date", "team", "player_name", "match_id"]
).copy()

cricsheet_unique = cricsheet_appearances.drop_duplicates(
    subset=["date", "team", "player_name", "match_id"]
).copy()

for df in [crickrida_unique, cricsheet_unique]:
    df["normalized_name"] = df["player_name"].apply(
        normalize_comparison_name
    )


# Countries available in both sources
shared_countries = sorted(
    set(crickrida_unique["team"]) &
    set(cricsheet_unique["team"])
)

print("SHARED COUNTRIES:", len(shared_countries))
print(shared_countries)


# Compare players on the same date and for the same country
comparison_rows = []

for country in shared_countries:

    cr_dates = crickrida_unique[
        crickrida_unique["team"] == country
    ]

    cs_dates = cricsheet_unique[
        cricsheet_unique["team"] == country
    ]

    shared_dates = sorted(
        set(cr_dates["date"]) & set(cs_dates["date"])
    )

    for match_date in shared_dates:

        cr_day = cr_dates[cr_dates["date"] == match_date]
        cs_day = cs_dates[cs_dates["date"] == match_date]

        cr_names = set(cr_day["normalized_name"])
        cs_names = set(cs_day["normalized_name"])

        exact_names = cr_names & cs_names

        comparison_rows.append({
            "date": match_date,
            "team": country,
            "crickrida_players": len(cr_names),
            "cricsheet_players": len(cs_names),
            "exact_name_matches": len(exact_names),
            "crickrida_unmatched": len(cr_names - cs_names),
            "cricsheet_unmatched": len(cs_names - cr_names)
        })


date_team_comparison = pd.DataFrame(comparison_rows)


print("\nDATE + COUNTRY COMPARISON")
print("Shared country-date combinations:", len(date_team_comparison))

if not date_team_comparison.empty:
    print(
        "Total exact name matches:",
        date_team_comparison["exact_name_matches"].sum()
    )

    print(
        "Country-date combinations with exact matches:",
        (date_team_comparison["exact_name_matches"] > 0).sum()
    )

    print("\nCOUNTRY SUMMARY")

    country_summary = (
        date_team_comparison.groupby("team")
        .agg(
            shared_dates=("date", "count"),
            exact_matches=("exact_name_matches", "sum"),
            crickrida_unmatched=("crickrida_unmatched", "sum"),
            cricsheet_unmatched=("cricsheet_unmatched", "sum")
        )
        .sort_values("exact_matches", ascending=False)
    )

    display(country_summary)

    print("\nSAMPLE DATE COMPARISONS")
    display(date_team_comparison.head(20))
else:
    print("No exact country-date overlaps found.")

SHARED COUNTRIES: 11
['Australia', 'Bangladesh', 'England', 'India', 'Ireland', 'New Zealand', 'Pakistan', 'South Africa', 'Sri Lanka', 'West Indies', 'Zimbabwe']

DATE + COUNTRY COMPARISON
Shared country-date combinations: 0
No exact country-date overlaps found.


In [129]:
# Build unique player histories for each source and country

cr_player_history = (
    crickrida_unique.groupby(
        ["team", "normalized_name", "player_name"],
        as_index=False
    )
    .agg(
        crickrida_first_date=("date", "min"),
        crickrida_last_date=("date", "max"),
        crickrida_matches=("match_id", "nunique")
    )
)

cs_player_history = (
    cricsheet_unique.groupby(
        ["team", "normalized_name", "player_name", "source_player_id"],
        as_index=False
    )
    .agg(
        cricsheet_first_date=("date", "min"),
        cricsheet_last_date=("date", "max"),
        cricsheet_matches=("match_id", "nunique")
    )
)

# Exact normalized-name matches within the same country
exact_country_matches = cr_player_history.merge(
    cs_player_history,
    on=["team", "normalized_name"],
    how="inner",
    suffixes=("_crickrida", "_cricsheet")
)

print("EXACT COUNTRY-LEVEL PLAYER MATCHES")
print("Total matched records:", len(exact_country_matches))
print(
    "Unique Crickrida players:",
    exact_country_matches["normalized_name"].nunique()
)

print(
    "Unique Cricsheet player IDs:",
    exact_country_matches["source_player_id"].nunique()
)

print("\nCOUNTRY-WISE EXACT MATCHES")

if not exact_country_matches.empty:
    display(
        exact_country_matches.groupby("team")
        .agg(
            matched_players=("normalized_name", "nunique"),
            matched_records=("source_player_id", "count")
        )
        .sort_values("matched_players", ascending=False)
    )

    print("\nMATCHED PLAYER DETAILS")
    display(
        exact_country_matches.sort_values(
            ["team", "player_name_crickrida"]
        ).head(50)
    )
else:
    print("No exact normalized-name matches found within the same country.")


# Check if any exact name maps to multiple Cricsheet IDs
print("\nAMBIGUOUS EXACT NAME MATCHES")

if not exact_country_matches.empty:
    ambiguous_exact = (
        exact_country_matches.groupby(
            ["team", "normalized_name"]
        )["source_player_id"]
        .nunique()
        .reset_index(name="cricsheet_id_count")
    )

    ambiguous_exact = ambiguous_exact[
        ambiguous_exact["cricsheet_id_count"] > 1
    ]

    display(ambiguous_exact)
else:
    print("No exact matches to test for ambiguity.")

EXACT COUNTRY-LEVEL PLAYER MATCHES
Total matched records: 96
Unique Crickrida players: 96
Unique Cricsheet player IDs: 96

COUNTRY-WISE EXACT MATCHES


,matched_players,matched_records
team,,
Bangladesh,45,45
Pakistan,37,37
India,11,11
Ireland,1,1
South Africa,1,1
Zimbabwe,1,1



MATCHED PLAYER DETAILS


,team,normalized_name,player_name_crickrida,crickrida_first_date,crickrida_last_date,crickrida_matches,player_name_cricsheet,source_player_id,cricsheet_first_date,cricsheet_last_date,cricsheet_matches
0,Bangladesh,abdurrazzak,Abdur Razzak,2014-03-01,2014-03-01,1,Abdur Razzak,888e32bf,2004-07-17,2014-08-25,121
1,Bangladesh,abuhider,Abu Hider,2018-09-20,2018-09-20,1,Abu Hider,31803fac,2018-10-26,2018-10-26,1
2,Bangladesh,afifhossain,Afif Hossain,2022-02-23,2023-09-03,6,Afif Hossain,37138918,2020-03-06,2026-04-17,27
3,Bangladesh,anamulhaque,Anamul Haque,2014-03-01,2015-02-18,2,Anamul Haque,72b245dd,2012-11-30,2024-03-18,44
4,Bangladesh,arafatsunny,Arafat Sunny,2014-03-01,2014-03-01,1,Arafat Sunny,1f7adc8a,2014-02-17,2015-11-11,15
5,Bangladesh,hasanmahmud,Hasan Mahmud,2023-07-05,2025-10-14,5,Hasan Mahmud,80ad39ac,2021-01-20,2025-07-05,20
6,Bangladesh,imrulkayes,Imrul Kayes,2016-09-25,2018-09-23,2,Imrul Kayes,5fbe14fc,2008-11-07,2018-12-11,72
7,Bangladesh,jakerali,Jaker Ali,2024-11-09,2025-10-11,4,Jaker Ali,cd56a813,2024-12-08,2025-07-08,8
8,Bangladesh,mahmudullah,Mahmudullah,2015-02-18,2024-11-11,14,Mahmudullah,bf1d7d3e,2007-07-25,2025-02-24,205
9,Bangladesh,mashrafemortaza,Mashrafe Mortaza,2015-02-18,2019-06-24,7,Mashrafe Mortaza,e186f49c,2003-02-11,2020-03-06,173



AMBIGUOUS EXACT NAME MATCHES


,team,normalized_name,cricsheet_id_count


In [130]:
from difflib import SequenceMatcher

# Exclude exact normalized-name matches
cr_unmatched = cr_player_history.merge(
    exact_country_matches[["team", "normalized_name"]].drop_duplicates(),
    on=["team", "normalized_name"],
    how="left",
    indicator=True
)

cr_unmatched = cr_unmatched[
    cr_unmatched["_merge"] == "left_only"
].drop(columns="_merge")


cs_unmatched = cs_player_history.merge(
    exact_country_matches[
        ["team", "normalized_name"]
    ].drop_duplicates(),
    on=["team", "normalized_name"],
    how="left",
    indicator=True
)

cs_unmatched = cs_unmatched[
    cs_unmatched["_merge"] == "left_only"
].drop(columns="_merge")


# Calculate the gap between two career date ranges
def career_date_gap(row):
    if row["crickrida_last_date"] < row["cricsheet_first_date"]:
        return (
            row["cricsheet_first_date"] -
            row["crickrida_last_date"]
        ).days

    if row["cricsheet_last_date"] < row["crickrida_first_date"]:
        return (
            row["crickrida_first_date"] -
            row["cricsheet_last_date"]
        ).days

    return 0


# Generate candidates within the same country
candidate_rows = []

for country in shared_countries:

    cr_country = cr_unmatched[
        cr_unmatched["team"] == country
    ]

    cs_country = cs_unmatched[
        cs_unmatched["team"] == country
    ]

    for _, cr_row in cr_country.iterrows():

        possible_candidates = []

        for _, cs_row in cs_country.iterrows():

            similarity = SequenceMatcher(
                None,
                cr_row["normalized_name"],
                cs_row["normalized_name"]
            ).ratio()

            possible_candidates.append({
                "team": country,
                "crickrida_name": cr_row["player_name"],
                "crickrida_normalized": cr_row["normalized_name"],
                "crickrida_first_date": cr_row["crickrida_first_date"],
                "crickrida_last_date": cr_row["crickrida_last_date"],
                "crickrida_matches": cr_row["crickrida_matches"],
                "cricsheet_name": cs_row["player_name"],
                "cricsheet_normalized": cs_row["normalized_name"],
                "cricsheet_player_id": cs_row["source_player_id"],
                "cricsheet_first_date": cs_row["cricsheet_first_date"],
                "cricsheet_last_date": cs_row["cricsheet_last_date"],
                "cricsheet_matches": cs_row["cricsheet_matches"],
                "similarity": similarity
            })

        # Keep the three closest candidates for this player
        possible_candidates = sorted(
            possible_candidates,
            key=lambda x: x["similarity"],
            reverse=True
        )[:3]

        candidate_rows.extend(possible_candidates)


fuzzy_candidates = pd.DataFrame(candidate_rows)

if not fuzzy_candidates.empty:

    fuzzy_candidates["career_date_gap_days"] = (
        fuzzy_candidates.apply(career_date_gap, axis=1)
    )

    fuzzy_candidates["career_periods_overlap"] = (
        fuzzy_candidates["career_date_gap_days"] == 0
    )

    # Show candidates with similarity >= 0.70
    fuzzy_shortlist = fuzzy_candidates[
        fuzzy_candidates["similarity"] >= 0.70
    ].copy()

    fuzzy_shortlist = fuzzy_shortlist.sort_values(
        ["similarity", "career_date_gap_days"],
        ascending=[False, True]
    )

    print("UNMATCHED CRICKRIDA PLAYERS:", len(cr_unmatched))
    print("UNMATCHED CRICSHEET PLAYER RECORDS:", len(cs_unmatched))
    print("ALL TOP-3 CANDIDATES:", len(fuzzy_candidates))
    print("CANDIDATES WITH SIMILARITY >= 0.70:", len(fuzzy_shortlist))

    print("\nCANDIDATES WITH OVERLAPPING CAREER PERIODS:")
    display(
        fuzzy_shortlist[
            fuzzy_shortlist["career_periods_overlap"]
        ].head(100)
    )

    print("\nALL HIGH-SIMILARITY CANDIDATES:")
    display(fuzzy_shortlist.head(100))

else:
    print("No fuzzy candidates generated.")

UNMATCHED CRICKRIDA PLAYERS: 358
UNMATCHED CRICSHEET PLAYER RECORDS: 1929
ALL TOP-3 CANDIDATES: 900
CANDIDATES WITH SIMILARITY >= 0.70: 291

CANDIDATES WITH OVERLAPPING CAREER PERIODS:


,team,crickrida_name,crickrida_normalized,crickrida_first_date,crickrida_last_date,crickrida_matches,cricsheet_name,cricsheet_normalized,cricsheet_player_id,cricsheet_first_date,cricsheet_last_date,cricsheet_matches,similarity,career_date_gap_days,career_periods_overlap
225,India,Nitish Kumar Reddy,nitishkumarreddy,2026-06-13,2026-06-20,2,Nithish Kumar Reddy,nithishkumarreddy,aad0c365,2025-10-19,2026-10-03,6,0.969697,0,True
228,India,Prasidh Krishna,prasidhkrishna,2026-06-13,2026-06-20,2,M Prasidh Krishna,mprasidhkrishna,85e0cf10,2021-03-23,2026-09-30,28,0.965517,0,True
468,Pakistan,Sarfaraz Ahmed,sarfarazahmed,2018-09-21,2019-06-29,2,Sarfraz Ahmed,sarfrazahmed,2254ab79,2008-06-24,2021-04-07,89,0.960000,0,True
99,Bangladesh,Najmul Hossain Shanto,najmulhossainshanto,2018-09-20,2026-10-21,13,Nazmul Hossain Shanto,nazmulhossainshanto,1274d7ab,2018-09-21,2026-10-17,58,0.947368,0,True
93,Bangladesh,Litton Das,littondas,2018-09-20,2023-10-07,10,Liton Das,litondas,0404d43c,2015-06-18,2026-06-14,93,0.941176,0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
567,Sri Lanka,Dasun Shanaka,dasunshanaka,2018-09-17,2023-09-05,6,D Madushanka,dmadushanka,de7d833e,2023-01-10,2026-10-22,29,0.782609,0,True
597,Sri Lanka,Janith Liyanage,janithliyanage,2024-02-09,2024-02-14,3,J Liyanage,jliyanage,84212ffb,2024-01-06,2026-10-22,36,0.782609,0,True
636,Sri Lanka,Pathum Nissanka,pathumnissanka,2022-11-25,2024-02-14,11,P Nissanka,pnissanka,8ee36b18,2021-03-10,2026-09-27,70,0.782609,0,True
744,West Indies,Shimron Hetmyer,shimronhetmyer,2018-03-15,2019-11-11,6,SO Hetmyer,sohetmyer,48a1d7b7,2017-12-20,2026-07-21,48,0.782609,0,True



ALL HIGH-SIMILARITY CANDIDATES:


,team,crickrida_name,crickrida_normalized,crickrida_first_date,crickrida_last_date,crickrida_matches,cricsheet_name,cricsheet_normalized,cricsheet_player_id,cricsheet_first_date,cricsheet_last_date,cricsheet_matches,similarity,career_date_gap_days,career_periods_overlap
225,India,Nitish Kumar Reddy,nitishkumarreddy,2026-06-13,2026-06-20,2,Nithish Kumar Reddy,nithishkumarreddy,aad0c365,2025-10-19,2026-10-03,6,0.969697,0,True
228,India,Prasidh Krishna,prasidhkrishna,2026-06-13,2026-06-20,2,M Prasidh Krishna,mprasidhkrishna,85e0cf10,2021-03-23,2026-09-30,28,0.965517,0,True
468,Pakistan,Sarfaraz Ahmed,sarfarazahmed,2018-09-21,2019-06-29,2,Sarfraz Ahmed,sarfrazahmed,2254ab79,2008-06-24,2021-04-07,89,0.960000,0,True
99,Bangladesh,Najmul Hossain Shanto,najmulhossainshanto,2018-09-20,2026-10-21,13,Nazmul Hossain Shanto,nazmulhossainshanto,1274d7ab,2018-09-21,2026-10-17,58,0.947368,0,True
93,Bangladesh,Litton Das,littondas,2018-09-20,2023-10-07,10,Liton Das,litondas,0404d43c,2015-06-18,2026-06-14,93,0.941176,0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
576,Sri Lanka,Dilshan Madushanka,dilshanmadushanka,2023-10-30,2024-02-14,4,DSK Madushanka,dskmadushanka,248698f4,2018-01-27,2018-01-27,1,0.800000,2102,False
577,Sri Lanka,Dilshan Madushanka,dilshanmadushanka,2023-10-30,2024-02-14,4,D Madushanka,dmadushanka,de7d833e,2023-01-10,2026-10-22,29,0.785714,0,True
624,Sri Lanka,Maheesh Theekshana,maheeshtheekshana,2022-11-25,2024-02-11,9,M Theekshana,mtheekshana,f24c6701,2021-09-07,2026-06-03,51,0.785714,0,True
879,Zimbabwe,Tinashe Panyangara,tinashepanyangara,2014-07-18,2015-10-24,8,T Panyangara,tpanyangara,3b417422,2004-04-20,2016-11-14,50,0.785714,0,True


In [131]:
from difflib import SequenceMatcher
import numpy as np

all_candidate_rows = []

for country in shared_countries:

    cr_country = cr_unmatched[
        cr_unmatched["team"] == country
    ]

    cs_country = cs_unmatched[
        cs_unmatched["team"] == country
    ]

    for _, cr_row in cr_country.iterrows():

        cr_dates = crickrida_unique.loc[
            (crickrida_unique["team"] == country) &
            (crickrida_unique["normalized_name"] == cr_row["normalized_name"]),
            "date"
        ].drop_duplicates()

        for _, cs_row in cs_country.iterrows():

            similarity = SequenceMatcher(
                None,
                cr_row["normalized_name"],
                cs_row["normalized_name"]
            ).ratio()

            if similarity < 0.65:
                continue

            cs_dates = cricsheet_unique.loc[
                (cricsheet_unique["team"] == country) &
                (cricsheet_unique["normalized_name"] == cs_row["normalized_name"]),
                "date"
            ].drop_duplicates()

            # Closest actual appearance date between the two sources
            date_gaps = [
                abs((cr_date - cs_date).days)
                for cr_date in cr_dates
                for cs_date in cs_dates
            ]

            nearest_date_gap = min(date_gaps) if date_gaps else np.nan

            all_candidate_rows.append({
                "team": country,
                "crickrida_name": cr_row["player_name"],
                "crickrida_normalized": cr_row["normalized_name"],
                "crickrida_first_date": cr_row["crickrida_first_date"],
                "crickrida_last_date": cr_row["crickrida_last_date"],
                "crickrida_matches": cr_row["crickrida_matches"],
                "cricsheet_name": cs_row["player_name"],
                "cricsheet_normalized": cs_row["normalized_name"],
                "cricsheet_player_id": cs_row["source_player_id"],
                "cricsheet_first_date": cs_row["cricsheet_first_date"],
                "cricsheet_last_date": cs_row["cricsheet_last_date"],
                "cricsheet_matches": cs_row["cricsheet_matches"],
                "similarity": similarity,
                "nearest_appearance_gap_days": nearest_date_gap
            })


all_fuzzy_candidates = pd.DataFrame(all_candidate_rows)

all_fuzzy_candidates = all_fuzzy_candidates.sort_values(
    ["similarity", "nearest_appearance_gap_days"],
    ascending=[False, True]
).reset_index(drop=True)


print("ALL CANDIDATES WITH SIMILARITY >= 0.65:", len(all_fuzzy_candidates))

print("\nCANDIDATES BY SIMILARITY BAND")
all_fuzzy_candidates["similarity_band"] = pd.cut(
    all_fuzzy_candidates["similarity"],
    bins=[0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 1.001],
    right=False
)

print(all_fuzzy_candidates["similarity_band"].value_counts().sort_index())


print("\nVERY STRONG CANDIDATES (>= 0.85)")
display(
    all_fuzzy_candidates[
        all_fuzzy_candidates["similarity"] >= 0.85
    ].head(100)
)


print("\nSTRONG CANDIDATES (0.75 TO 0.85)")
display(
    all_fuzzy_candidates[
        (all_fuzzy_candidates["similarity"] >= 0.75) &
        (all_fuzzy_candidates["similarity"] < 0.85)
    ].head(100)
)

ALL CANDIDATES WITH SIMILARITY >= 0.65: 355

CANDIDATES BY SIMILARITY BAND
similarity_band
[0.65, 0.7)      61
[0.7, 0.75)     114
[0.75, 0.8)      84
[0.8, 0.85)      68
[0.85, 0.9)      20
[0.9, 1.001)      8
Name: count, dtype: int64

VERY STRONG CANDIDATES (>= 0.85)


,team,crickrida_name,crickrida_normalized,crickrida_first_date,crickrida_last_date,crickrida_matches,cricsheet_name,cricsheet_normalized,cricsheet_player_id,cricsheet_first_date,cricsheet_last_date,cricsheet_matches,similarity,nearest_appearance_gap_days,similarity_band
0,India,Nitish Kumar Reddy,nitishkumarreddy,2026-06-13,2026-06-20,2,Nithish Kumar Reddy,nithishkumarreddy,aad0c365,2025-10-19,2026-10-03,6,0.969697,99,"[0.9, 1.001)"
1,India,Prasidh Krishna,prasidhkrishna,2026-06-13,2026-06-20,2,M Prasidh Krishna,mprasidhkrishna,85e0cf10,2021-03-23,2026-09-30,28,0.965517,24,"[0.9, 1.001)"
2,Pakistan,Sarfaraz Ahmed,sarfarazahmed,2018-09-21,2019-06-29,2,Sarfraz Ahmed,sarfrazahmed,2254ab79,2008-06-24,2021-04-07,89,0.960000,2,"[0.9, 1.001)"
3,Bangladesh,Najmul Hossain Shanto,najmulhossainshanto,2018-09-20,2026-10-21,13,Nazmul Hossain Shanto,nazmulhossainshanto,1274d7ab,2018-09-21,2026-10-17,58,0.947368,1,"[0.9, 1.001)"
4,Bangladesh,Litton Das,littondas,2018-09-20,2023-10-07,10,Liton Das,litondas,0404d43c,2015-06-18,2026-06-14,93,0.941176,1,"[0.9, 1.001)"
5,England,Sam Curran,samcurran,2023-10-15,2023-10-15,1,SM Curran,smcurran,e94915e6,2018-06-24,2026-10-25,46,0.941176,5,"[0.9, 1.001)"
6,England,Joe Root,joeroot,2019-06-18,2025-02-26,3,JE Root,jeroot,a343262c,2013-01-11,2026-10-25,187,0.923077,3,"[0.9, 1.001)"
7,Bangladesh,Ebadot Hossain,ebadothossain,2023-07-08,2023-07-08,1,Ebadat Hossain,ebadathossain,cb5100f1,2022-08-10,2023-05-14,10,0.923077,55,"[0.9, 1.001)"
8,New Zealand,Tom Latham,tomlatham,2019-06-08,2023-10-18,2,TWM Latham,twmlatham,e824e6ee,2012-02-03,2026-07-21,156,0.888889,3,"[0.85, 0.9)"
9,Ireland,Jai Moondra,jaimoondra,2026-08-07,2026-08-15,3,J Moondra,jmoondra,3bea7ea9,2026-10-22,2026-10-24,2,0.888889,68,"[0.85, 0.9)"



STRONG CANDIDATES (0.75 TO 0.85)


,team,crickrida_name,crickrida_normalized,crickrida_first_date,crickrida_last_date,crickrida_matches,cricsheet_name,cricsheet_normalized,cricsheet_player_id,cricsheet_first_date,cricsheet_last_date,cricsheet_matches,similarity,nearest_appearance_gap_days,similarity_band
28,New Zealand,Kane Williamson,kanewilliamson,2015-03-08,2019-06-08,2,KS Williamson,kswilliamson,d027ba9f,2010-08-10,2025-10-29,168,0.846154,3,"[0.8, 0.85)"
29,Sri Lanka,Dunith Wellalage,dunithwellalage,2022-11-30,2024-02-14,3,DN Wellalage,dnwellalage,736123bb,2022-06-14,2026-10-22,35,0.846154,4,"[0.8, 0.85)"
30,Zimbabwe,Elton Chigumbura,eltonchigumbura,2014-07-18,2016-01-06,12,E Chigumbura,echigumbura,d2babdd5,2004-04-20,2018-10-26,155,0.846154,5,"[0.8, 0.85)"
31,Ireland,Theo van Woerkom,theovanwoerkom,2024-03-07,2024-03-12,2,TF van Woerkom,tfvanwoerkom,70f2f922,2023-09-26,2023-12-17,2,0.846154,81,"[0.8, 0.85)"
32,South Africa,Beuran Hendricks,beuranhendricks,2019-06-15,2019-06-15,1,BE Hendricks,behendricks,b56dc5f7,2019-01-25,2023-12-21,7,0.846154,139,"[0.8, 0.85)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123,Australia,Brad Haddin,bradhaddin,2015-03-04,2015-03-04,1,BJ Haddin,bjhaddin,b0c772ee,2004-01-29,2015-03-26,107,0.777778,4,"[0.75, 0.8)"
124,Australia,Nathan Ellis,nathanellis,2025-02-28,2025-02-28,1,NT Ellis,ntellis,9eb1455b,2022-03-29,2026-09-27,26,0.777778,4,"[0.75, 0.8)"
125,Ireland,Andrew White,andrewwhite,2010-07-03,2010-07-03,1,AR White,arwhite,f27f89e8,2006-06-13,2014-09-12,24,0.777778,4,"[0.75, 0.8)"
126,Ireland,Boyd Rankin,boydrankin,2017-12-05,2019-05-21,13,WB Rankin,wbrankin,29b89ae8,2007-03-15,2020-01-09,41,0.777778,4,"[0.75, 0.8)"


In [132]:
# Focus on stronger candidates
review_candidates = all_fuzzy_candidates[
    all_fuzzy_candidates["similarity"] >= 0.80
].copy()

# Count competing Cricsheet candidates for each Crickrida player
cr_candidate_counts = (
    review_candidates.groupby(
        ["team", "crickrida_normalized"]
    )["cricsheet_player_id"]
    .nunique()
    .reset_index(name="possible_cricsheet_ids")
)

# Count Crickrida candidates for each Cricsheet ID
cs_candidate_counts = (
    review_candidates.groupby(
        ["team", "cricsheet_player_id"]
    )["crickrida_normalized"]
    .nunique()
    .reset_index(name="possible_crickrida_players")
)

review_candidates = review_candidates.merge(
    cr_candidate_counts,
    on=["team", "crickrida_normalized"],
    how="left"
)

review_candidates = review_candidates.merge(
    cs_candidate_counts,
    on=["team", "cricsheet_player_id"],
    how="left"
)

# Find the closest actual appearance dates for each candidate
closest_date_rows = []

for _, candidate in review_candidates.iterrows():

    cr_dates = crickrida_unique.loc[
        (crickrida_unique["team"] == candidate["team"]) &
        (
            crickrida_unique["normalized_name"] ==
            candidate["crickrida_normalized"]
        ),
        ["date", "match_id"]
    ].drop_duplicates()

    cs_dates = cricsheet_unique.loc[
        (cricsheet_unique["team"] == candidate["team"]) &
        (
            cricsheet_unique["source_player_id"] ==
            candidate["cricsheet_player_id"]
        ),
        ["date", "match_id"]
    ].drop_duplicates()

    pairs = cr_dates.merge(cs_dates, how="cross", suffixes=("_cr", "_cs"))

    pairs["gap_days"] = (
        pairs["date_cr"] - pairs["date_cs"]
    ).abs().dt.days

    closest = pairs.sort_values("gap_days").iloc[0]

    closest_date_rows.append({
        "team": candidate["team"],
        "crickrida_name": candidate["crickrida_name"],
        "cricsheet_name": candidate["cricsheet_name"],
        "cricsheet_player_id": candidate["cricsheet_player_id"],
        "similarity": candidate["similarity"],
        "possible_cricsheet_ids": candidate["possible_cricsheet_ids"],
        "possible_crickrida_players": candidate["possible_crickrida_players"],
        "closest_crickrida_date": closest["date_cr"],
        "closest_cricsheet_date": closest["date_cs"],
        "date_gap_days": closest["gap_days"]
    })

candidate_review_df = pd.DataFrame(closest_date_rows)

candidate_review_df = candidate_review_df.sort_values(
    ["similarity", "date_gap_days"],
    ascending=[False, True]
).reset_index(drop=True)

print("CANDIDATES REVIEWED:", len(candidate_review_df))

print("\nCANDIDATES WITH MULTIPLE POSSIBLE CRICSHEET IDs")
display(
    candidate_review_df[
        candidate_review_df["possible_cricsheet_ids"] > 1
    ]
)

print("\nCRICSHEET IDs PROPOSED FOR MULTIPLE CRICKRIDA PLAYERS")
display(
    candidate_review_df[
        candidate_review_df["possible_crickrida_players"] > 1
    ]
)

print("\nSTRONG CANDIDATES WITH CLOSE APPEARANCE DATES")
display(
    candidate_review_df[
        (candidate_review_df["similarity"] >= 0.85) &
        (candidate_review_df["date_gap_days"] <= 10)
    ]
)

print("\nTOP 50 CANDIDATES FOR MANUAL REVIEW")
display(candidate_review_df.head(50))

CANDIDATES REVIEWED: 96

CANDIDATES WITH MULTIPLE POSSIBLE CRICSHEET IDs


,team,crickrida_name,cricsheet_name,cricsheet_player_id,similarity,possible_cricsheet_ids,possible_crickrida_players,closest_crickrida_date,closest_cricsheet_date,date_gap_days
21,Ireland,Ed Joyce,EC Joyce,a48fc79d,0.857143,2,1,2015-01-10,2015-01-12,2
27,Ireland,Ed Joyce,DI Joyce,fd67a507,0.857143,2,1,2012-07-05,2007-06-24,1838
89,South Africa,Reeza Hendricks,RR Hendricks,b8cc58c9,0.800000,2,1,2024-09-22,2024-10-07,15
94,South Africa,Reeza Hendricks,BE Hendricks,b56dc5f7,0.800000,2,2,2024-09-18,2023-12-21,272



CRICSHEET IDs PROPOSED FOR MULTIPLE CRICKRIDA PLAYERS


,team,crickrida_name,cricsheet_name,cricsheet_player_id,similarity,possible_cricsheet_ids,possible_crickrida_players,closest_crickrida_date,closest_cricsheet_date,date_gap_days
32,South Africa,Beuran Hendricks,BE Hendricks,b56dc5f7,0.846154,1,2,2019-06-15,2019-01-27,139
94,South Africa,Reeza Hendricks,BE Hendricks,b56dc5f7,0.800000,2,2,2024-09-18,2023-12-21,272



STRONG CANDIDATES WITH CLOSE APPEARANCE DATES


,team,crickrida_name,cricsheet_name,cricsheet_player_id,similarity,possible_cricsheet_ids,possible_crickrida_players,closest_crickrida_date,closest_cricsheet_date,date_gap_days
2,Pakistan,Sarfaraz Ahmed,Sarfraz Ahmed,2254ab79,0.960000,1,1,2018-09-21,2018-09-19,2
3,Bangladesh,Najmul Hossain Shanto,Nazmul Hossain Shanto,1274d7ab,0.947368,1,1,2018-09-20,2018-09-21,1
4,Bangladesh,Litton Das,Liton Das,0404d43c,0.941176,1,1,2018-09-20,2018-09-21,1
5,England,Sam Curran,SM Curran,e94915e6,0.941176,1,1,2023-10-15,2023-10-10,5
6,England,Joe Root,JE Root,a343262c,0.923077,1,1,2019-06-18,2019-06-21,3
8,New Zealand,Tom Latham,TWM Latham,e824e6ee,0.888889,1,1,2019-06-08,2019-06-05,3
11,Bangladesh,Mehidy Hasan Miraz,Mehedi Hasan Miraz,c58b0108,0.875000,1,1,2018-09-20,2018-09-21,1
12,Ireland,Mark Adair,MR Adair,fc55ec67,0.875000,1,1,2019-05-19,2019-05-15,4
13,Ireland,Andy Balbirnie,A Balbirnie,6ef13460,0.869565,1,1,2015-01-10,2015-01-12,2
14,Australia,Ben Dwarshuis,BJ Dwarshuis,e1b9f3a9,0.869565,1,1,2025-02-28,2025-03-04,4



TOP 50 CANDIDATES FOR MANUAL REVIEW


,team,crickrida_name,cricsheet_name,cricsheet_player_id,similarity,possible_cricsheet_ids,possible_crickrida_players,closest_crickrida_date,closest_cricsheet_date,date_gap_days
0,India,Nitish Kumar Reddy,Nithish Kumar Reddy,aad0c365,0.969697,1,1,2026-06-20,2026-09-27,99
1,India,Prasidh Krishna,M Prasidh Krishna,85e0cf10,0.965517,1,1,2026-06-20,2026-07-14,24
2,Pakistan,Sarfaraz Ahmed,Sarfraz Ahmed,2254ab79,0.960000,1,1,2018-09-21,2018-09-19,2
3,Bangladesh,Najmul Hossain Shanto,Nazmul Hossain Shanto,1274d7ab,0.947368,1,1,2018-09-20,2018-09-21,1
4,Bangladesh,Litton Das,Liton Das,0404d43c,0.941176,1,1,2018-09-20,2018-09-21,1
5,England,Sam Curran,SM Curran,e94915e6,0.941176,1,1,2023-10-15,2023-10-10,5
6,England,Joe Root,JE Root,a343262c,0.923077,1,1,2019-06-18,2019-06-21,3
7,Bangladesh,Ebadot Hossain,Ebadat Hossain,cb5100f1,0.923077,1,1,2023-07-08,2023-05-14,55
8,New Zealand,Tom Latham,TWM Latham,e824e6ee,0.888889,1,1,2019-06-08,2019-06-05,3
9,Ireland,Jai Moondra,J Moondra,3bea7ea9,0.888889,1,1,2026-08-15,2026-10-22,68


In [133]:
print("PLAYER REGISTRY COLUMNS")
print(player_registry_df.columns.tolist())

print("\nPLAYER REGISTRY SAMPLE")
display(player_registry_df.head(10))

# Inspect registered names for the strongest candidate pairs
strong_candidate_ids = (
    candidate_review_df[
        (candidate_review_df["similarity"] >= 0.80) &
        (candidate_review_df["possible_cricsheet_ids"] == 1) &
        (candidate_review_df["possible_crickrida_players"] == 1)
    ][
        ["team", "crickrida_name", "cricsheet_name", "cricsheet_player_id"]
    ]
    .drop_duplicates()
)

registry_aliases = (
    player_registry_df.groupby("player_id")["player_name"]
    .agg(lambda names: sorted(set(names)))
    .reset_index(name="registered_names")
)

registry_review = strong_candidate_ids.merge(
    registry_aliases,
    left_on="cricsheet_player_id",
    right_on="player_id",
    how="left"
)

registry_review["crickrida_name_normalized"] = (
    registry_review["crickrida_name"].apply(normalize_comparison_name)
)

registry_review["registered_names_normalized"] = (
    registry_review["registered_names"].apply(
        lambda names: [
            normalize_comparison_name(name)
            for name in names
        ] if isinstance(names, list) else []
    )
)

registry_review["full_name_in_registry"] = (
    registry_review.apply(
        lambda row:
            row["crickrida_name_normalized"]
            in row["registered_names_normalized"],
        axis=1
    )
)

print("\nSTRONG CANDIDATES WITH CRICSHEET REGISTERED ALIASES")
display(
    registry_review[
        [
            "team",
            "crickrida_name",
            "cricsheet_name",
            "cricsheet_player_id",
            "registered_names",
            "full_name_in_registry"
        ]
    ].head(100)
)

print("\nFULL CRICKRIDA NAME FOUND IN CRICSHEET REGISTRY:")
print(registry_review["full_name_in_registry"].value_counts())

PLAYER REGISTRY COLUMNS
['match_id', 'player_name', 'player_id']

PLAYER REGISTRY SAMPLE


,match_id,player_name,player_id
0,1000887,Azhar Ali,211de69f
1,1000887,B Stanlake,6834d1f2
2,1000887,Babar Azam,8a75e999
3,1000887,C Shamshuddin,0f9d921b
4,1000887,CA Lynn,45eda7c8
5,1000887,CB Gaffaney,d5ac41d8
6,1000887,DA Warner,dcce6f09
7,1000887,GJ Maxwell,b681e71e
8,1000887,Hasan Ali,2911de16
9,1000887,Imad Wasim,9cb8d7a6



STRONG CANDIDATES WITH CRICSHEET REGISTERED ALIASES


,team,crickrida_name,cricsheet_name,cricsheet_player_id,registered_names,full_name_in_registry
0,India,Nitish Kumar Reddy,Nithish Kumar Reddy,aad0c365,[Nithish Kumar Reddy],False
1,India,Prasidh Krishna,M Prasidh Krishna,85e0cf10,[M Prasidh Krishna],False
2,Pakistan,Sarfaraz Ahmed,Sarfraz Ahmed,2254ab79,[Sarfraz Ahmed],False
3,Bangladesh,Najmul Hossain Shanto,Nazmul Hossain Shanto,1274d7ab,[Nazmul Hossain Shanto],False
4,Bangladesh,Litton Das,Liton Das,0404d43c,[Liton Das],False
...,...,...,...,...,...,...
86,Zimbabwe,Dion Myers,D Myers,d96165b5,[D Myers],False
87,Zimbabwe,Shingi Masakadza,H Masakadza,31fbf891,[H Masakadza],False
88,Sri Lanka,Asitha Fernando,WIA Fernando,9550470d,[WIA Fernando],False
89,Ireland,Byron McDonough,BD McDonough,d2c250c8,[BD McDonough],False



FULL CRICKRIDA NAME FOUND IN CRICSHEET REGISTRY:
full_name_in_registry
False    91
Name: count, dtype: int64


In [134]:
# All names recorded for each Cricsheet player ID
cricsheet_registered_names = (
    player_registry_df.groupby("player_id")
    .agg(
        registered_names=("player_name", lambda x: sorted(set(x))),
        registered_name_count=("player_name", "nunique"),
        registered_match_count=("match_id", "nunique")
    )
    .reset_index()
)

# Attach registered names to candidate review
candidate_registry_audit = candidate_review_df.merge(
    cricsheet_registered_names,
    left_on="cricsheet_player_id",
    right_on="player_id",
    how="left"
)

# Check whether the same Cricsheet name is linked to multiple IDs
cricsheet_name_id_counts = (
    player_registry_df.groupby("player_name")["player_id"]
    .nunique()
    .reset_index(name="player_id_count")
)

ambiguous_cricsheet_names = cricsheet_name_id_counts[
    cricsheet_name_id_counts["player_id_count"] > 1
]

print("CANDIDATE ID NAME VARIATIONS")
print(
    "Candidate IDs with multiple registered names:",
    (
        candidate_registry_audit["registered_name_count"] > 1
    ).sum()
)

print("\nCANDIDATES WITH MULTIPLE REGISTERED NAMES")
display(
    candidate_registry_audit[
        candidate_registry_audit["registered_name_count"] > 1
    ][
        [
            "team",
            "crickrida_name",
            "cricsheet_name",
            "cricsheet_player_id",
            "registered_names",
            "registered_name_count",
            "registered_match_count"
        ]
    ]
)

print("\nCRICSHEET NAMES MAPPED TO MULTIPLE PLAYER IDs")
display(ambiguous_cricsheet_names)

print("\nCANDIDATE REGISTRY AUDIT SAMPLE")
display(
    candidate_registry_audit[
        [
            "team",
            "crickrida_name",
            "cricsheet_name",
            "cricsheet_player_id",
            "registered_names",
            "registered_name_count",
            "registered_match_count"
        ]
    ].head(30)
)

CANDIDATE ID NAME VARIATIONS
Candidate IDs with multiple registered names: 0

CANDIDATES WITH MULTIPLE REGISTERED NAMES


,team,crickrida_name,cricsheet_name,cricsheet_player_id,registered_names,registered_name_count,registered_match_count



CRICSHEET NAMES MAPPED TO MULTIPLE PLAYER IDs


,player_name,player_id_count
68,A Sharma,2
257,Abdul Muqeet,2
311,Ali Khan,2
531,C de Lange,2
664,D Joseph,2
1112,Haider Ali,2
1434,Junaid Siddique,2
2485,RP Singh,2
2528,Rahat Ali,2
2800,SR Taylor,2



CANDIDATE REGISTRY AUDIT SAMPLE


,team,crickrida_name,cricsheet_name,cricsheet_player_id,registered_names,registered_name_count,registered_match_count
0,India,Nitish Kumar Reddy,Nithish Kumar Reddy,aad0c365,[Nithish Kumar Reddy],1,6
1,India,Prasidh Krishna,M Prasidh Krishna,85e0cf10,[M Prasidh Krishna],1,28
2,Pakistan,Sarfaraz Ahmed,Sarfraz Ahmed,2254ab79,[Sarfraz Ahmed],1,115
3,Bangladesh,Najmul Hossain Shanto,Nazmul Hossain Shanto,1274d7ab,[Nazmul Hossain Shanto],1,62
4,Bangladesh,Litton Das,Liton Das,0404d43c,[Liton Das],1,94
5,England,Sam Curran,SM Curran,e94915e6,[SM Curran],1,46
6,England,Joe Root,JE Root,a343262c,[JE Root],1,192
7,Bangladesh,Ebadot Hossain,Ebadat Hossain,cb5100f1,[Ebadat Hossain],1,11
8,New Zealand,Tom Latham,TWM Latham,e824e6ee,[TWM Latham],1,170
9,Ireland,Jai Moondra,J Moondra,3bea7ea9,[J Moondra],1,2


In [ ]:
print("PLAYER REGISTRY COLUMNS")
print(player_registry_df.columns.tolist())

print("\nPLAYER REGISTRY SAMPLE")
display(player_registry_df.head(10))

# Inspect registered names for the strongest candidate pairs
strong_candidate_ids = (
    candidate_review_df[
        (candidate_review_df["similarity"] >= 0.80) &
        (candidate_review_df["possible_cricsheet_ids"] == 1) &
        (candidate_review_df["possible_crickrida_players"] == 1)
    ][
        ["team", "crickrida_name", "cricsheet_name", "cricsheet_player_id"]
    ]
    .drop_duplicates()
)

registry_aliases = (
    player_registry_df.groupby("player_id")["player_name"]
    .agg(lambda names: sorted(set(names)))
    .reset_index(name="registered_names")
)

registry_review = strong_candidate_ids.merge(
    registry_aliases,
    left_on="cricsheet_player_id",
    right_on="player_id",
    how="left"
)

registry_review["crickrida_name_normalized"] = (
    registry_review["crickrida_name"].apply(normalize_comparison_name)
)

registry_review["registered_names_normalized"] = (
    registry_review["registered_names"].apply(
        lambda names: [
            normalize_comparison_name(name)
            for name in names
        ] if isinstance(names, list) else []
    )
)

registry_review["full_name_in_registry"] = (
    registry_review.apply(
        lambda row:
            row["crickrida_name_normalized"]
            in row["registered_names_normalized"],
        axis=1
    )
)

print("\nSTRONG CANDIDATES WITH CRICSHEET REGISTERED ALIASES")
display(
    registry_review[
        [
            "team",
            "crickrida_name",
            "cricsheet_name",
            "cricsheet_player_id",
            "registered_names",
            "full_name_in_registry"
        ]
    ].head(100)
)

print("\nFULL CRICKRIDA NAME FOUND IN CRICSHEET REGISTRY:")
print(registry_review["full_name_in_registry"].value_counts())

PLAYER REGISTRY COLUMNS
['match_id', 'player_name', 'player_id']

PLAYER REGISTRY SAMPLE


,match_id,player_name,player_id
0,1000887,Azhar Ali,211de69f
1,1000887,B Stanlake,6834d1f2
2,1000887,Babar Azam,8a75e999
3,1000887,C Shamshuddin,0f9d921b
4,1000887,CA Lynn,45eda7c8
5,1000887,CB Gaffaney,d5ac41d8
6,1000887,DA Warner,dcce6f09
7,1000887,GJ Maxwell,b681e71e
8,1000887,Hasan Ali,2911de16
9,1000887,Imad Wasim,9cb8d7a6



STRONG CANDIDATES WITH CRICSHEET REGISTERED ALIASES


,team,crickrida_name,cricsheet_name,cricsheet_player_id,registered_names,full_name_in_registry
0,India,Nitish Kumar Reddy,Nithish Kumar Reddy,aad0c365,[Nithish Kumar Reddy],False
1,India,Prasidh Krishna,M Prasidh Krishna,85e0cf10,[M Prasidh Krishna],False
2,Pakistan,Sarfaraz Ahmed,Sarfraz Ahmed,2254ab79,[Sarfraz Ahmed],False
3,Bangladesh,Najmul Hossain Shanto,Nazmul Hossain Shanto,1274d7ab,[Nazmul Hossain Shanto],False
4,Bangladesh,Litton Das,Liton Das,0404d43c,[Liton Das],False
...,...,...,...,...,...,...
86,Zimbabwe,Dion Myers,D Myers,d96165b5,[D Myers],False
87,Zimbabwe,Shingi Masakadza,H Masakadza,31fbf891,[H Masakadza],False
88,Sri Lanka,Asitha Fernando,WIA Fernando,9550470d,[WIA Fernando],False
89,Ireland,Byron McDonough,BD McDonough,d2c250c8,[BD McDonough],False



FULL CRICKRIDA NAME FOUND IN CRICSHEET REGISTRY:
full_name_in_registry
False    91
Name: count, dtype: int64


In [ ]:
print("PLAYER REGISTRY COLUMNS")
print(player_registry_df.columns.tolist())

print("\nPLAYER REGISTRY SAMPLE")
display(player_registry_df.head(10))

# Inspect registered names for the strongest candidate pairs
strong_candidate_ids = (
    candidate_review_df[
        (candidate_review_df["similarity"] >= 0.80) &
        (candidate_review_df["possible_cricsheet_ids"] == 1) &
        (candidate_review_df["possible_crickrida_players"] == 1)
    ][
        ["team", "crickrida_name", "cricsheet_name", "cricsheet_player_id"]
    ]
    .drop_duplicates()
)

registry_aliases = (
    player_registry_df.groupby("player_id")["player_name"]
    .agg(lambda names: sorted(set(names)))
    .reset_index(name="registered_names")
)

registry_review = strong_candidate_ids.merge(
    registry_aliases,
    left_on="cricsheet_player_id",
    right_on="player_id",
    how="left"
)

registry_review["crickrida_name_normalized"] = (
    registry_review["crickrida_name"].apply(normalize_comparison_name)
)

registry_review["registered_names_normalized"] = (
    registry_review["registered_names"].apply(
        lambda names: [
            normalize_comparison_name(name)
            for name in names
        ] if isinstance(names, list) else []
    )
)

registry_review["full_name_in_registry"] = (
    registry_review.apply(
        lambda row:
            row["crickrida_name_normalized"]
            in row["registered_names_normalized"],
        axis=1
    )
)

print("\nSTRONG CANDIDATES WITH CRICSHEET REGISTERED ALIASES")
display(
    registry_review[
        [
            "team",
            "crickrida_name",
            "cricsheet_name",
            "cricsheet_player_id",
            "registered_names",
            "full_name_in_registry"
        ]
    ].head(100)
)

print("\nFULL CRICKRIDA NAME FOUND IN CRICSHEET REGISTRY:")
print(registry_review["full_name_in_registry"].value_counts())

PLAYER REGISTRY COLUMNS
['match_id', 'player_name', 'player_id']

PLAYER REGISTRY SAMPLE


,match_id,player_name,player_id
0,1000887,Azhar Ali,211de69f
1,1000887,B Stanlake,6834d1f2
2,1000887,Babar Azam,8a75e999
3,1000887,C Shamshuddin,0f9d921b
4,1000887,CA Lynn,45eda7c8
5,1000887,CB Gaffaney,d5ac41d8
6,1000887,DA Warner,dcce6f09
7,1000887,GJ Maxwell,b681e71e
8,1000887,Hasan Ali,2911de16
9,1000887,Imad Wasim,9cb8d7a6



STRONG CANDIDATES WITH CRICSHEET REGISTERED ALIASES


,team,crickrida_name,cricsheet_name,cricsheet_player_id,registered_names,full_name_in_registry
0,India,Nitish Kumar Reddy,Nithish Kumar Reddy,aad0c365,[Nithish Kumar Reddy],False
1,India,Prasidh Krishna,M Prasidh Krishna,85e0cf10,[M Prasidh Krishna],False
2,Pakistan,Sarfaraz Ahmed,Sarfraz Ahmed,2254ab79,[Sarfraz Ahmed],False
3,Bangladesh,Najmul Hossain Shanto,Nazmul Hossain Shanto,1274d7ab,[Nazmul Hossain Shanto],False
4,Bangladesh,Litton Das,Liton Das,0404d43c,[Liton Das],False
...,...,...,...,...,...,...
86,Zimbabwe,Dion Myers,D Myers,d96165b5,[D Myers],False
87,Zimbabwe,Shingi Masakadza,H Masakadza,31fbf891,[H Masakadza],False
88,Sri Lanka,Asitha Fernando,WIA Fernando,9550470d,[WIA Fernando],False
89,Ireland,Byron McDonough,BD McDonough,d2c250c8,[BD McDonough],False



FULL CRICKRIDA NAME FOUND IN CRICSHEET REGISTRY:
full_name_in_registry
False    91
Name: count, dtype: int64


In [138]:
print("candidate_review_df columns:")
print(candidate_review_df.columns.tolist())

print("\nreview_80 columns:")
print(review_80.columns.tolist())

print("\nAvailable date-related columns:")
print([
    col for col in candidate_review_df.columns
    if "date" in col.lower() or "gap" in col.lower()
])

candidate_review_df columns:
['team', 'crickrida_name', 'cricsheet_name', 'cricsheet_player_id', 'similarity', 'possible_cricsheet_ids', 'possible_crickrida_players', 'closest_crickrida_date', 'closest_cricsheet_date', 'date_gap_days']

review_80 columns:
['team', 'crickrida_name', 'cricsheet_name', 'cricsheet_player_id', 'similarity', 'closest_crickrida_date', 'closest_cricsheet_date', 'date_gap_days', 'possible_cricsheet_ids', 'possible_crickrida_names']

Available date-related columns:
['closest_crickrida_date', 'closest_cricsheet_date', 'date_gap_days']


In [139]:
# Candidates with similarity >= 0.80
review_80 = candidate_review_df[
    candidate_review_df["similarity"] >= 0.80
].copy()

# Count possible Cricsheet IDs for each Crickrida player
crickrida_candidate_counts = (
    review_80.groupby(["team", "crickrida_name"])
    ["cricsheet_player_id"]
    .nunique()
    .reset_index(name="possible_cricsheet_ids")
)

# Count possible Crickrida names for each Cricsheet ID
cricsheet_candidate_counts = (
    review_80.groupby(["team", "cricsheet_player_id"])
    ["crickrida_name"]
    .nunique()
    .reset_index(name="possible_crickrida_names")
)

# Attach both conflict counts
review_80 = review_80.drop(
    columns=[
        "possible_cricsheet_ids",
        "possible_crickrida_players"
    ],
    errors="ignore"
).merge(
    crickrida_candidate_counts,
    on=["team", "crickrida_name"],
    how="left"
).merge(
    cricsheet_candidate_counts,
    on=["team", "cricsheet_player_id"],
    how="left"
)

# One Crickrida name points to multiple Cricsheet IDs
crickrida_conflicts = review_80[
    review_80["possible_cricsheet_ids"] > 1
].sort_values(
    ["team", "crickrida_name", "similarity"],
    ascending=[True, True, False]
)

# One Cricsheet ID is proposed for multiple Crickrida names
cricsheet_conflicts = review_80[
    review_80["possible_crickrida_names"] > 1
].sort_values(
    ["team", "cricsheet_player_id", "similarity"],
    ascending=[True, True, False]
)

print("TOTAL CANDIDATE PAIRS:", len(review_80))

print(
    "\nCRICKRIDA NAMES WITH MULTIPLE CRICSHEET IDs:",
    crickrida_conflicts[
        ["team", "crickrida_name"]
    ].drop_duplicates().shape[0]
)

print(
    "CRICSHEET IDs WITH MULTIPLE CRICKRIDA NAMES:",
    cricsheet_conflicts[
        ["team", "cricsheet_player_id"]
    ].drop_duplicates().shape[0]
)

print("\n1. CRICKRIDA NAME CONFLICTS")
display(
    crickrida_conflicts[
        [
            "team", "crickrida_name", "cricsheet_name",
            "cricsheet_player_id", "similarity",
            "closest_crickrida_date", "closest_cricsheet_date",
            "date_gap_days", "possible_cricsheet_ids"
        ]
    ]
)

print("\n2. CRICSHEET ID CONFLICTS")
display(
    cricsheet_conflicts[
        [
            "team", "crickrida_name", "cricsheet_name",
            "cricsheet_player_id", "similarity",
            "closest_crickrida_date", "closest_cricsheet_date",
            "date_gap_days", "possible_crickrida_names"
        ]
    ]
)

print("\n3. UNCONFLICTED CANDIDATES")
display(
    review_80[
        (review_80["possible_cricsheet_ids"] == 1) &
        (review_80["possible_crickrida_names"] == 1)
    ].sort_values(
        ["similarity", "date_gap_days"],
        ascending=[False, True]
    )[
        [
            "team", "crickrida_name", "cricsheet_name",
            "cricsheet_player_id", "similarity",
            "closest_crickrida_date", "closest_cricsheet_date",
            "date_gap_days"
        ]
    ]
)

TOTAL CANDIDATE PAIRS: 96

CRICKRIDA NAMES WITH MULTIPLE CRICSHEET IDs: 2
CRICSHEET IDs WITH MULTIPLE CRICKRIDA NAMES: 1

1. CRICKRIDA NAME CONFLICTS


,team,crickrida_name,cricsheet_name,cricsheet_player_id,similarity,closest_crickrida_date,closest_cricsheet_date,date_gap_days,possible_cricsheet_ids
21,Ireland,Ed Joyce,EC Joyce,a48fc79d,0.857143,2015-01-10,2015-01-12,2,2
27,Ireland,Ed Joyce,DI Joyce,fd67a507,0.857143,2012-07-05,2007-06-24,1838,2
89,South Africa,Reeza Hendricks,RR Hendricks,b8cc58c9,0.800000,2024-09-22,2024-10-07,15,2
94,South Africa,Reeza Hendricks,BE Hendricks,b56dc5f7,0.800000,2024-09-18,2023-12-21,272,2



2. CRICSHEET ID CONFLICTS


,team,crickrida_name,cricsheet_name,cricsheet_player_id,similarity,closest_crickrida_date,closest_cricsheet_date,date_gap_days,possible_crickrida_names
32,South Africa,Beuran Hendricks,BE Hendricks,b56dc5f7,0.846154,2019-06-15,2019-01-27,139,2
94,South Africa,Reeza Hendricks,BE Hendricks,b56dc5f7,0.800000,2024-09-18,2023-12-21,272,2



3. UNCONFLICTED CANDIDATES


,team,crickrida_name,cricsheet_name,cricsheet_player_id,similarity,closest_crickrida_date,closest_cricsheet_date,date_gap_days
0,India,Nitish Kumar Reddy,Nithish Kumar Reddy,aad0c365,0.969697,2026-06-20,2026-09-27,99
1,India,Prasidh Krishna,M Prasidh Krishna,85e0cf10,0.965517,2026-06-20,2026-07-14,24
2,Pakistan,Sarfaraz Ahmed,Sarfraz Ahmed,2254ab79,0.960000,2018-09-21,2018-09-19,2
3,Bangladesh,Najmul Hossain Shanto,Nazmul Hossain Shanto,1274d7ab,0.947368,2018-09-20,2018-09-21,1
4,Bangladesh,Litton Das,Liton Das,0404d43c,0.941176,2018-09-20,2018-09-21,1
...,...,...,...,...,...,...,...,...
90,Zimbabwe,Dion Myers,D Myers,d96165b5,0.800000,2024-12-17,2024-11-28,19
91,Zimbabwe,Shingi Masakadza,H Masakadza,31fbf891,0.800000,2014-07-24,2014-08-17,24
92,Sri Lanka,Asitha Fernando,WIA Fernando,9550470d,0.800000,2024-02-14,2024-03-13,28
93,Ireland,Byron McDonough,BD McDonough,d2c250c8,0.800000,2026-08-12,2026-10-24,73


In [140]:
# Conflicting identity candidates to inspect
conflict_checks = [
    {
        "team": "Ireland",
        "crickrida_name": "Ed Joyce",
        "cricsheet_name": "EC Joyce",
        "cricsheet_id": "a48fc79d"
    },
    {
        "team": "Ireland",
        "crickrida_name": "Ed Joyce",
        "cricsheet_name": "DI Joyce",
        "cricsheet_id": "fd67a507"
    },
    {
        "team": "South Africa",
        "crickrida_name": "Reeza Hendricks",
        "cricsheet_name": "RR Hendricks",
        "cricsheet_id": "b8cc58c9"
    },
    {
        "team": "South Africa",
        "crickrida_name": "Reeza Hendricks",
        "cricsheet_name": "BE Hendricks",
        "cricsheet_id": "b56dc5f7"
    },
    {
        "team": "South Africa",
        "crickrida_name": "Beuran Hendricks",
        "cricsheet_name": "BE Hendricks",
        "cricsheet_id": "b56dc5f7"
    }
]

# Ensure date fields are datetime for consistent sorting
crickrida_appearances["date"] = pd.to_datetime(
    crickrida_appearances["date"]
)
cricsheet_appearances["date"] = pd.to_datetime(
    cricsheet_appearances["date"]
)

for check in conflict_checks:
    print("\n" + "=" * 90)
    print(
        f"{check['team']} | "
        f"{check['crickrida_name']} ↔ {check['cricsheet_name']}"
    )
    print("Cricsheet ID:", check["cricsheet_id"])

    # Crickrida appearances for this player and team
    cr_rows = crickrida_appearances[
        (crickrida_appearances["team"] == check["team"]) &
        (crickrida_appearances["player_name"] == check["crickrida_name"])
    ].copy()

    # Cricsheet appearances for this exact player ID and team
    cs_rows = cricsheet_appearances[
        (cricsheet_appearances["team"] == check["team"]) &
        (cricsheet_appearances["source_player_id"] == check["cricsheet_id"])
    ].copy()

    print("\nCRICKRIDA HISTORY")
    display(
        cr_rows[
            ["date", "team", "player_name", "match_url", "role"]
        ].sort_values("date").drop_duplicates()
        if "match_url" in cr_rows.columns
        else cr_rows[
            ["date", "team", "player_name", "match_id", "role"]
        ].sort_values("date").drop_duplicates()
    )

    print("\nCRICSHEET HISTORY")
    display(
        cs_rows[
            ["date", "team", "player_name", "match_id", "role"]
        ].sort_values("date").drop_duplicates()
    )

    print("\nAPPEARANCE COUNTS")
    print("Crickrida records:", len(cr_rows))
    print("Cricsheet records:", len(cs_rows))
    print(
        "Crickrida date range:",
        cr_rows["date"].min(),
        "to",
        cr_rows["date"].max()
    )
    print(
        "Cricsheet date range:",
        cs_rows["date"].min(),
        "to",
        cs_rows["date"].max()
    )


Ireland | Ed Joyce ↔ EC Joyce
Cricsheet ID: a48fc79d

CRICKRIDA HISTORY


,date,team,player_name,match_id,role
39,2012-07-05,Ireland,Ed Joyce,https://crickrida.com/matches/afghanistan-irel...,batting
238,2015-01-10,Ireland,Ed Joyce,https://crickrida.com/matches/afghanistan-irel...,batting
257,2015-01-17,Ireland,Ed Joyce,https://crickrida.com/matches/afghanistan-irel...,batting
561,2016-07-12,Ireland,Ed Joyce,https://crickrida.com/matches/afghanistan-irel...,batting
583,2016-07-14,Ireland,Ed Joyce,https://crickrida.com/matches/afghanistan-irel...,batting
598,2016-07-17,Ireland,Ed Joyce,https://crickrida.com/matches/afghanistan-irel...,batting
609,2016-07-19,Ireland,Ed Joyce,https://crickrida.com/matches/afghanistan-irel...,batting
798,2017-03-15,Ireland,Ed Joyce,https://crickrida.com/matches/afghanistan-irel...,batting
820,2017-03-17,Ireland,Ed Joyce,https://crickrida.com/matches/afghanistan-irel...,batting
841,2017-03-19,Ireland,Ed Joyce,https://crickrida.com/matches/afghanistan-irel...,batting



CRICSHEET HISTORY


,date,team,player_name,match_id,role
29523,2011-02-25,Ireland,EC Joyce,433566,batting
29638,2011-03-02,Ireland,EC Joyce,433572,batting
29719,2011-03-06,Ireland,EC Joyce,433578,batting
29843,2011-03-11,Ireland,EC Joyce,433584,batting
29966,2011-03-15,Ireland,EC Joyce,433591,batting
30044,2011-03-18,Ireland,EC Joyce,433595,batting
31914,2011-08-25,Ireland,EC Joyce,483126,batting
33052,2011-09-19,Ireland,EC Joyce,523804,batting
33072,2011-09-20,Ireland,EC Joyce,523805,batting
33888,2012-02-18,Ireland,EC Joyce,543881,batting



APPEARANCE COUNTS
Crickrida records: 12
Cricsheet records: 43
Crickrida date range: 2012-07-05 00:00:00 to 2017-03-24 00:00:00
Cricsheet date range: 2011-02-25 00:00:00 to 2018-03-16 00:00:00

Ireland | Ed Joyce ↔ DI Joyce
Cricsheet ID: fd67a507

CRICKRIDA HISTORY


,date,team,player_name,match_id,role
39,2012-07-05,Ireland,Ed Joyce,https://crickrida.com/matches/afghanistan-irel...,batting
238,2015-01-10,Ireland,Ed Joyce,https://crickrida.com/matches/afghanistan-irel...,batting
257,2015-01-17,Ireland,Ed Joyce,https://crickrida.com/matches/afghanistan-irel...,batting
561,2016-07-12,Ireland,Ed Joyce,https://crickrida.com/matches/afghanistan-irel...,batting
583,2016-07-14,Ireland,Ed Joyce,https://crickrida.com/matches/afghanistan-irel...,batting
598,2016-07-17,Ireland,Ed Joyce,https://crickrida.com/matches/afghanistan-irel...,batting
609,2016-07-19,Ireland,Ed Joyce,https://crickrida.com/matches/afghanistan-irel...,batting
798,2017-03-15,Ireland,Ed Joyce,https://crickrida.com/matches/afghanistan-irel...,batting
820,2017-03-17,Ireland,Ed Joyce,https://crickrida.com/matches/afghanistan-irel...,batting
841,2017-03-19,Ireland,Ed Joyce,https://crickrida.com/matches/afghanistan-irel...,batting


... (11 outputs left out)

,date,team,player_name,match_id,role
3654,2019-06-15,South Africa,Beuran Hendricks,https://crickrida.com/matches/afghanistan-sout...,bowling



CRICSHEET HISTORY


,date,team,player_name,match_id,role
47916,2019-01-25,South Africa,BE Hendricks,1144158,bowling
3653,2019-01-27,South Africa,BE Hendricks,1144159,batting
47932,2019-01-27,South Africa,BE Hendricks,1144159,bowling
49411,2020-02-04,South Africa,BE Hendricks,1185310,bowling
5824,2020-02-09,South Africa,BE Hendricks,1185312,batting
49436,2020-02-09,South Africa,BE Hendricks,1185312,bowling
7408,2021-04-07,South Africa,BE Hendricks,1251574,batting
50502,2021-04-07,South Africa,BE Hendricks,1251574,bowling
55061,2023-12-19,South Africa,BE Hendricks,1387601,bowling
14163,2023-12-21,South Africa,BE Hendricks,1387602,batting



APPEARANCE COUNTS
Crickrida records: 1
Cricsheet records: 11
Crickrida date range: 2019-06-15 00:00:00 to 2019-06-15 00:00:00
Cricsheet date range: 2019-01-25 00:00:00 to 2023-12-21 00:00:00


In [141]:
# Start with one-to-one candidates only
clean_candidates = review_80[
    (review_80["possible_cricsheet_ids"] == 1) &
    (review_80["possible_crickrida_names"] == 1)
].copy()

# Conservative shortlist
strong_shortlist = clean_candidates[
    (clean_candidates["similarity"] >= 0.85) &
    (clean_candidates["date_gap_days"] <= 10)
].copy()

# Exclude candidates already reviewed in the previous identity-matching stage
previously_approved_aliases = {
    ("Ebadot Hossain", "Ebadat Hossain"),
    ("Joe Root", "JE Root"),
    ("Litton Das", "Liton Das"),
    ("Najmul Hossain Shanto", "Nazmul Hossain Shanto"),
    ("Nitish Kumar Reddy", "Nithish Kumar Reddy"),
    ("Prasidh Krishna", "M Prasidh Krishna"),
    ("Sam Curran", "SM Curran"),
    ("Sarfaraz Ahmed", "Sarfraz Ahmed")
}

strong_shortlist["previously_approved"] = strong_shortlist.apply(
    lambda row: (
        row["crickrida_name"],
        row["cricsheet_name"]
    ) in previously_approved_aliases,
    axis=1
)

new_candidates_to_review = strong_shortlist[
    ~strong_shortlist["previously_approved"]
].copy()

print("Strong one-to-one candidates:", len(strong_shortlist))
print("Previously approved aliases:", strong_shortlist["previously_approved"].sum())
print("New candidates requiring review:", len(new_candidates_to_review))

print("\nNEW CANDIDATES BY COUNTRY")
display(
    new_candidates_to_review.groupby("team").size()
    .reset_index(name="candidate_count")
    .sort_values("candidate_count", ascending=False)
)

print("\nNEW CANDIDATES FOR REVIEW")
display(
    new_candidates_to_review.sort_values(
        ["team", "similarity", "date_gap_days"],
        ascending=[True, False, True]
    )[
        [
            "team",
            "crickrida_name",
            "cricsheet_name",
            "cricsheet_player_id",
            "similarity",
            "closest_crickrida_date",
            "closest_cricsheet_date",
            "date_gap_days"
        ]
    ].to_string(index=False)
)

Strong one-to-one candidates: 17
Previously approved aliases: 5
New candidates requiring review: 12

NEW CANDIDATES BY COUNTRY


,team,candidate_count
2,England,3
3,Ireland,3
4,New Zealand,2
1,Bangladesh,1
0,Australia,1
5,South Africa,1
6,West Indies,1



NEW CANDIDATES FOR REVIEW


'        team      crickrida_name     cricsheet_name cricsheet_player_id  similarity closest_crickrida_date closest_cricsheet_date  date_gap_days\n   Australia       Ben Dwarshuis       BJ Dwarshuis            e1b9f3a9    0.869565             2025-02-28             2025-03-04              4\n  Bangladesh  Mehidy Hasan Miraz Mehedi Hasan Miraz            c58b0108    0.875000             2018-09-20             2018-09-21              1\n     England      James Anderson        JM Anderson            d12143bf    0.869565             2015-03-13             2015-03-09              4\n     England    Liam Livingstone     LS Livingstone            50c6bc2b    0.857143             2025-02-26             2025-03-01              3\n     England           Mark Wood            MA Wood            8d92a2c3    0.857143             2019-06-18             2019-06-21              3\n     Ireland          Mark Adair           MR Adair            fc55ec67    0.875000             2019-05-19             2019

In [142]:
# Inspect both sources' full histories for the 12 candidates
for _, candidate in new_candidates_to_review.iterrows():

    team = candidate["team"]
    cr_name = candidate["crickrida_name"]
    cs_name = candidate["cricsheet_name"]
    cs_id = candidate["cricsheet_player_id"]

    print("\n" + "=" * 100)
    print(f"{team}: {cr_name}  <->  {cs_name}")
    print(
        f"Similarity: {candidate['similarity']:.3f} | "
        f"Closest date gap: {candidate['date_gap_days']} days"
    )
    print(f"Cricsheet ID: {cs_id}")

    cr_rows = crickrida_appearances[
        (crickrida_appearances["team"] == team) &
        (crickrida_appearances["player_name"] == cr_name)
    ].copy()

    cs_rows = cricsheet_appearances[
        (cricsheet_appearances["team"] == team) &
        (cricsheet_appearances["source_player_id"] == cs_id)
    ].copy()

    print("\nCRICKRIDA HISTORY")
    display(
        cr_rows[
            ["date", "team", "player_name", "match_id", "role"]
        ].sort_values(["date", "role"]).drop_duplicates()
    )

    print("\nCRICSHEET HISTORY")
    display(
        cs_rows[
            ["date", "team", "player_name", "match_id", "role"]
        ].sort_values(["date", "role"]).drop_duplicates()
    )

    print("\nCOUNTS")
    print("Crickrida appearances:", len(cr_rows))
    print("Cricsheet appearances:", len(cs_rows))
    print("Crickrida date range:", cr_rows["date"].min(), "to", cr_rows["date"].max())
    print("Cricsheet date range:", cs_rows["date"].min(), "to", cs_rows["date"].max())


New Zealand: Tom Latham  <->  TWM Latham
Similarity: 0.889 | Closest date gap: 3 days
Cricsheet ID: e824e6ee

CRICKRIDA HISTORY


,date,team,player_name,match_id,role
1473,2019-06-08,New Zealand,Tom Latham,https://crickrida.com/matches/afghanistan-new-...,batting
2110,2023-10-18,New Zealand,Tom Latham,https://crickrida.com/matches/afghanistan-new-...,batting



CRICSHEET HISTORY


,date,team,player_name,match_id,role
32732,2012-02-03,New Zealand,TWM Latham,520592,batting
32751,2012-02-06,New Zealand,TWM Latham,520593,batting
32769,2012-02-09,New Zealand,TWM Latham,520594,batting
33952,2012-07-07,New Zealand,TWM Latham,560924,batting
33963,2012-07-11,New Zealand,TWM Latham,560925,batting
...,...,...,...,...,...
19030,2026-07-11,New Zealand,TWM Latham,1538624,batting
19053,2026-07-13,New Zealand,TWM Latham,1538625,batting
19070,2026-07-16,New Zealand,TWM Latham,1538626,batting
19092,2026-07-19,New Zealand,TWM Latham,1538627,batting



COUNTS
Crickrida appearances: 2
Cricsheet appearances: 156
Crickrida date range: 2019-06-08 00:00:00 to 2023-10-18 00:00:00
Cricsheet date range: 2012-02-03 00:00:00 to 2026-07-21 00:00:00

Bangladesh: Mehidy Hasan Miraz  <->  Mehedi Hasan Miraz
Similarity: 0.875 | Closest date gap: 1 days
Cricsheet ID: c58b0108

CRICKRIDA HISTORY


,date,team,player_name,match_id,role
1236,2018-09-20,Bangladesh,Mehidy Hasan Miraz,https://crickrida.com/matches/afghanistan-bang...,batting
3496,2018-09-20,Bangladesh,Mehidy Hasan Miraz,https://crickrida.com/matches/afghanistan-bang...,bowling
1265,2018-09-23,Bangladesh,Mehidy Hasan Miraz,https://crickrida.com/matches/afghanistan-bang...,batting
3529,2018-09-23,Bangladesh,Mehidy Hasan Miraz,https://crickrida.com/matches/afghanistan-bang...,bowling
3696,2019-06-24,Bangladesh,Mehidy Hasan Miraz,https://crickrida.com/matches/afghanistan-bang...,bowling
1715,2022-02-23,Bangladesh,Mehidy Hasan Miraz,https://crickrida.com/matches/afghanistan-bang...,batting
3802,2022-02-23,Bangladesh,Mehidy Hasan Miraz,https://crickrida.com/matches/afghanistan-bang...,bowling
3821,2022-02-25,Bangladesh,Mehidy Hasan Miraz,https://crickrida.com/matches/afghanistan-bang...,bowling
1740,2022-02-28,Bangladesh,Mehidy Hasan Miraz,https://crickrida.com/matches/afghanistan-bang...,batting
3834,2022-02-28,Bangladesh,Mehidy Hasan Miraz,https://crickrida.com/matches/afghanistan-bang...,bowling


... (39 outputs left out)

,date,team,player_name,match_id,role
1097,2018-03-15,West Indies,Carlos Brathwaite,https://crickrida.com/matches/afghanistan-west...,batting
3426,2018-03-15,West Indies,Carlos Brathwaite,https://crickrida.com/matches/afghanistan-west...,bowling
1132,2018-03-25,West Indies,Carlos Brathwaite,https://crickrida.com/matches/afghanistan-west...,batting
3448,2018-03-25,West Indies,Carlos Brathwaite,https://crickrida.com/matches/afghanistan-west...,bowling
1572,2019-07-04,West Indies,Carlos Brathwaite,https://crickrida.com/matches/afghanistan-west...,batting
3721,2019-07-04,West Indies,Carlos Brathwaite,https://crickrida.com/matches/afghanistan-west...,bowling



CRICSHEET HISTORY


,date,team,player_name,match_id,role
33331,2011-10-18,West Indies,CR Brathwaite,531985,batting
41912,2015-01-21,West Indies,CR Brathwaite,722343,batting
74164,2015-01-21,West Indies,CR Brathwaite,722343,bowling
41936,2015-01-25,West Indies,CR Brathwaite,722345,batting
74171,2015-01-25,West Indies,CR Brathwaite,722345,bowling
...,...,...,...,...,...
48327,2019-07-01,West Indies,CR Brathwaite,1144521,bowling
6083,2019-08-11,West Indies,CR Brathwaite,1188625,batting
49609,2019-08-11,West Indies,CR Brathwaite,1188625,bowling
6094,2019-08-14,West Indies,CR Brathwaite,1188626,batting



COUNTS
Crickrida appearances: 6
Cricsheet appearances: 71
Crickrida date range: 2018-03-15 00:00:00 to 2019-07-04 00:00:00
Cricsheet date range: 2011-10-18 00:00:00 to 2019-08-14 00:00:00


In [143]:
# Previously approved aliases
previous_aliases = [
    ("Ebadot Hossain", "Ebadat Hossain", "Bangladesh"),
    ("Joe Root", "JE Root", "England"),
    ("Litton Das", "Liton Das", "Bangladesh"),
    ("Najmul Hossain Shanto", "Nazmul Hossain Shanto", "Bangladesh"),
    ("Nitish Kumar Reddy", "Nithish Kumar Reddy", "India"),
    ("Prasidh Krishna", "M Prasidh Krishna", "India"),
    ("Sam Curran", "SM Curran", "England"),
    ("Sarfaraz Ahmed", "Sarfraz Ahmed", "Pakistan")
]

# Newly approved aliases from the 12-candidate review
newly_approved_aliases = [
    ("Ben Dwarshuis", "BJ Dwarshuis", "Australia"),
    ("Mehidy Hasan Miraz", "Mehedi Hasan Miraz", "Bangladesh"),
    ("James Anderson", "JM Anderson", "England"),
    ("Liam Livingstone", "LS Livingstone", "England"),
    ("Mark Wood", "MA Wood", "England"),
    ("Mark Adair", "MR Adair", "Ireland"),
    ("Andy Balbirnie", "A Balbirnie", "Ireland"),
    ("James McCollum", "JA McCollum", "Ireland"),
    ("Tom Latham", "TWM Latham", "New Zealand"),
    ("Colin de Grandhomme", "C de Grandhomme", "New Zealand"),
    ("Andile Phehlukwayo", "AL Phehlukwayo", "South Africa"),
    ("Carlos Brathwaite", "CR Brathwaite", "West Indies")
]

# Conflict cases resolved through appearance-history review
resolved_conflict_aliases = [
    ("Ed Joyce", "EC Joyce", "Ireland"),
    ("Reeza Hendricks", "RR Hendricks", "South Africa"),
    ("Beuran Hendricks", "BE Hendricks", "South Africa")
]

approved_aliases_df = pd.DataFrame(
    previous_aliases + newly_approved_aliases + resolved_conflict_aliases,
    columns=[
        "crickrida_name",
        "cricsheet_name",
        "team"
    ]
)

approved_aliases_df["match_status"] = "verified_alias"

print("APPROVED ALIAS COUNT:", len(approved_aliases_df))
print(
    "Unique Crickrida names:",
    approved_aliases_df["crickrida_name"].nunique()
)
print(
    "Duplicate Crickrida names:",
    approved_aliases_df["crickrida_name"].duplicated().sum()
)
print(
    "Duplicate Cricsheet names:",
    approved_aliases_df["cricsheet_name"].duplicated().sum()
)

display(
    approved_aliases_df.sort_values(
        ["team", "crickrida_name"]
    ).reset_index(drop=True)
)

APPROVED ALIAS COUNT: 23
Unique Crickrida names: 23
Duplicate Crickrida names: 0
Duplicate Cricsheet names: 0


,crickrida_name,cricsheet_name,team,match_status
0,Ben Dwarshuis,BJ Dwarshuis,Australia,verified_alias
1,Ebadot Hossain,Ebadat Hossain,Bangladesh,verified_alias
2,Litton Das,Liton Das,Bangladesh,verified_alias
3,Mehidy Hasan Miraz,Mehedi Hasan Miraz,Bangladesh,verified_alias
4,Najmul Hossain Shanto,Nazmul Hossain Shanto,Bangladesh,verified_alias
5,James Anderson,JM Anderson,England,verified_alias
6,Joe Root,JE Root,England,verified_alias
7,Liam Livingstone,LS Livingstone,England,verified_alias
8,Mark Wood,MA Wood,England,verified_alias
9,Sam Curran,SM Curran,England,verified_alias


In [144]:
print("CRICKRIDA LOOKUP COLUMNS:")
print(crickrida_id_lookup.columns.tolist())

print("\nCRICSHEET LOOKUP COLUMNS:")
print(cricsheet_id_lookup.columns.tolist())

print("\nEXISTING IDENTITY MAP COLUMNS:")
print(player_identity_map.columns.tolist())

# Attach the Crickrida CricketIQ ID
alias_reconciliation = approved_aliases_df.merge(
    crickrida_id_lookup[
        ["player_name", "cricketiq_player_id"]
    ].rename(columns={
        "player_name": "crickrida_name"
    }),
    on="crickrida_name",
    how="left"
)

# Attach the Cricsheet source ID
alias_reconciliation = alias_reconciliation.merge(
    cricsheet_id_lookup[
        ["player_name", "player_id"]
    ].rename(columns={
        "player_name": "cricsheet_name",
        "player_id": "cricsheet_player_id"
    }),
    on="cricsheet_name",
    how="left"
)

# Check against the existing cross-source identity map
existing_pairs = player_identity_map[
    ["crickrida_name", "cricsheet_name", "cricketiq_player_id", "player_id"]
].copy()

alias_reconciliation = alias_reconciliation.merge(
    existing_pairs.rename(columns={
        "cricketiq_player_id": "existing_cricketiq_player_id",
        "player_id": "existing_cricsheet_player_id"
    }),
    on=["crickrida_name", "cricsheet_name"],
    how="left"
)

alias_reconciliation["cricketiq_id_matches_existing"] = (
    alias_reconciliation["cricketiq_player_id"] ==
    alias_reconciliation["existing_cricketiq_player_id"]
)

alias_reconciliation["cricsheet_id_matches_existing"] = (
    alias_reconciliation["cricsheet_player_id"] ==
    alias_reconciliation["existing_cricsheet_player_id"]
)

print("\nALIAS RECONCILIATION")
display(alias_reconciliation)

print("\nMISSING CRICKRIDA IDs:",
      alias_reconciliation["cricketiq_player_id"].isna().sum())

print("MISSING CRICSHEET IDs:",
      alias_reconciliation["cricsheet_player_id"].isna().sum())

print("ALREADY IN EXISTING IDENTITY MAP:",
      alias_reconciliation["existing_cricketiq_player_id"].notna().sum())

print("\nCRICKETIQ ID DUPLICATES:",
      alias_reconciliation["cricketiq_player_id"].duplicated().sum())

print("CRICSHEET ID DUPLICATES:",
      alias_reconciliation["cricsheet_player_id"].duplicated().sum())

CRICKRIDA LOOKUP COLUMNS:
['lookup_name', 'cricketiq_player_id']

CRICSHEET LOOKUP COLUMNS:
['player_id', 'cricketiq_player_id']

EXISTING IDENTITY MAP COLUMNS:
['cricketiq_player_id', 'player_id', 'cricsheet_name', 'crickrida_name', 'normalized_name', 'match_status']


KeyError: "['player_name'] not in index"

In [145]:
print("CRICKRIDA ID LOOKUP")
print("Shape:", crickrida_id_lookup.shape)
print("Columns:", crickrida_id_lookup.columns.tolist())
display(crickrida_id_lookup.head())

print("\nCRICSHEET ID LOOKUP")
print("Shape:", cricsheet_id_lookup.shape)
print("Columns:", cricsheet_id_lookup.columns.tolist())
display(cricsheet_id_lookup.head())

print("\nEXISTING IDENTITY MAP")
print("Shape:", player_identity_map.shape)
print("Columns:", player_identity_map.columns.tolist())
display(player_identity_map.head())

CRICKRIDA ID LOOKUP
Shape: (454, 2)
Columns: ['lookup_name', 'cricketiq_player_id']


,lookup_name,cricketiq_player_id
0,Babar Azam,CRIQ_8a75e999
1,Hasan Ali,CRIQ_2911de16
2,Imad Wasim,CRIQ_9cb8d7a6
3,Mohammad Amir,CRIQ_e174dadd
4,Mohammad Hafeez,CRIQ_9ab63e7b



CRICSHEET ID LOOKUP
Shape: (3287, 2)
Columns: ['player_id', 'cricketiq_player_id']


,player_id,cricketiq_player_id
0,8a75e999,CRIQ_8a75e999
1,2911de16,CRIQ_2911de16
2,9cb8d7a6,CRIQ_9cb8d7a6
3,e174dadd,CRIQ_e174dadd
4,9ab63e7b,CRIQ_9ab63e7b



EXISTING IDENTITY MAP
Shape: (106, 6)
Columns: ['cricketiq_player_id', 'player_id', 'cricsheet_name', 'crickrida_name', 'normalized_name', 'match_status']


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,normalized_name,match_status
0,CRIQ_8a75e999,8a75e999,Babar Azam,Babar Azam,babarazam,exact_name_match
1,CRIQ_2911de16,2911de16,Hasan Ali,Hasan Ali,hasanali,exact_name_match
2,CRIQ_9cb8d7a6,9cb8d7a6,Imad Wasim,Imad Wasim,imadwasim,exact_name_match
3,CRIQ_e174dadd,e174dadd,Mohammad Amir,Mohammad Amir,mohammadamir,exact_name_match
4,CRIQ_9ab63e7b,9ab63e7b,Mohammad Hafeez,Mohammad Hafeez,mohammadhafeez,exact_name_match


In [146]:
# Attach Crickrida IDs
alias_reconciliation = approved_aliases_df.merge(
    crickrida_id_lookup[
        ["lookup_name", "cricketiq_player_id"]
    ].rename(columns={"lookup_name": "crickrida_name"}),
    on="crickrida_name",
    how="left"
)

# Get the Cricsheet player ID from the existing identity map
alias_reconciliation = alias_reconciliation.merge(
    player_identity_map[
        ["cricsheet_name", "player_id"]
    ].drop_duplicates(),
    on="cricsheet_name",
    how="left"
)

# Attach the Cricsheet ID lookup's CricketIQ ID
alias_reconciliation = alias_reconciliation.merge(
    cricsheet_id_lookup[
        ["player_id", "cricketiq_player_id"]
    ].rename(columns={
        "cricketiq_player_id": "cricsheet_lookup_cricketiq_id"
    }),
    on="player_id",
    how="left"
)

# Check whether each alias already exists in the identity map
existing_pairs = player_identity_map[
    ["crickrida_name", "cricsheet_name", "cricketiq_player_id", "player_id"]
].rename(columns={
    "cricketiq_player_id": "existing_cricketiq_player_id",
    "player_id": "existing_player_id"
})

alias_reconciliation = alias_reconciliation.merge(
    existing_pairs,
    on=["crickrida_name", "cricsheet_name"],
    how="left"
)

alias_reconciliation["already_in_identity_map"] = (
    alias_reconciliation["existing_player_id"].notna()
)

alias_reconciliation["ids_agree"] = (
    alias_reconciliation["cricketiq_player_id"] ==
    alias_reconciliation["cricsheet_lookup_cricketiq_id"]
)

display(alias_reconciliation)

print("Total approved aliases:", len(alias_reconciliation))
print("Missing Crickrida IDs:",
      alias_reconciliation["cricketiq_player_id"].isna().sum())
print("Missing Cricsheet IDs:",
      alias_reconciliation["player_id"].isna().sum())
print("Missing Cricsheet lookup IDs:",
      alias_reconciliation["cricsheet_lookup_cricketiq_id"].isna().sum())
print("Already in identity map:",
      alias_reconciliation["already_in_identity_map"].sum())
print("Crickrida and Cricsheet IDs agree:",
      alias_reconciliation["ids_agree"].sum())
print("Duplicate Crickrida IDs:",
      alias_reconciliation["cricketiq_player_id"].duplicated().sum())
print("Duplicate Cricsheet IDs:",
      alias_reconciliation["player_id"].duplicated().sum())

,crickrida_name,cricsheet_name,team,match_status,cricketiq_player_id,player_id,cricsheet_lookup_cricketiq_id,existing_cricketiq_player_id,existing_player_id,already_in_identity_map,ids_agree
0,Ebadot Hossain,Ebadat Hossain,Bangladesh,verified_alias,CRIQ_cb5100f1,cb5100f1,CRIQ_cb5100f1,CRIQ_cb5100f1,cb5100f1,True,True
1,Joe Root,JE Root,England,verified_alias,CRIQ_a343262c,a343262c,CRIQ_a343262c,CRIQ_a343262c,a343262c,True,True
2,Litton Das,Liton Das,Bangladesh,verified_alias,CRIQ_0404d43c,0404d43c,CRIQ_0404d43c,CRIQ_0404d43c,0404d43c,True,True
3,Najmul Hossain Shanto,Nazmul Hossain Shanto,Bangladesh,verified_alias,CRIQ_1274d7ab,1274d7ab,CRIQ_1274d7ab,CRIQ_1274d7ab,1274d7ab,True,True
4,Nitish Kumar Reddy,Nithish Kumar Reddy,India,verified_alias,CRIQ_aad0c365,aad0c365,CRIQ_aad0c365,CRIQ_aad0c365,aad0c365,True,True
5,Prasidh Krishna,M Prasidh Krishna,India,verified_alias,CRIQ_85e0cf10,85e0cf10,CRIQ_85e0cf10,CRIQ_85e0cf10,85e0cf10,True,True
6,Sam Curran,SM Curran,England,verified_alias,CRIQ_e94915e6,e94915e6,CRIQ_e94915e6,CRIQ_e94915e6,e94915e6,True,True
7,Sarfaraz Ahmed,Sarfraz Ahmed,Pakistan,verified_alias,CRIQ_2254ab79,2254ab79,CRIQ_2254ab79,CRIQ_2254ab79,2254ab79,True,True
8,Ben Dwarshuis,BJ Dwarshuis,Australia,verified_alias,CR_bendwarshuis,NaN,NaN,NaN,NaN,False,False
9,Mehidy Hasan Miraz,Mehedi Hasan Miraz,Bangladesh,verified_alias,CR_mehidyhasanmiraz,NaN,NaN,NaN,NaN,False,False


Total approved aliases: 23
Missing Crickrida IDs: 0
Missing Cricsheet IDs: 15
Missing Cricsheet lookup IDs: 15
Already in identity map: 8
Crickrida and Cricsheet IDs agree: 8
Duplicate Crickrida IDs: 0
Duplicate Cricsheet IDs: 14


In [147]:
new_aliases = alias_reconciliation[
    ~alias_reconciliation["already_in_identity_map"]
].copy()

candidate_ids = candidate_review_df[
    ["team", "crickrida_name", "cricsheet_name", "cricsheet_player_id"]
].drop_duplicates()

new_alias_evidence = new_aliases.merge(
    candidate_ids,
    on=["team", "crickrida_name", "cricsheet_name"],
    how="left"
)

display(new_alias_evidence)

print("New aliases:", len(new_alias_evidence))
print("Missing candidate IDs:",
      new_alias_evidence["cricsheet_player_id"].isna().sum())
print("Duplicate candidate IDs:",
      new_alias_evidence["cricsheet_player_id"].duplicated().sum())
print("Unique candidate IDs:",
      new_alias_evidence["cricsheet_player_id"].nunique())

,crickrida_name,cricsheet_name,team,match_status,cricketiq_player_id,player_id,cricsheet_lookup_cricketiq_id,existing_cricketiq_player_id,existing_player_id,already_in_identity_map,ids_agree,cricsheet_player_id
0,Ben Dwarshuis,BJ Dwarshuis,Australia,verified_alias,CR_bendwarshuis,NaN,NaN,NaN,NaN,False,False,e1b9f3a9
1,Mehidy Hasan Miraz,Mehedi Hasan Miraz,Bangladesh,verified_alias,CR_mehidyhasanmiraz,NaN,NaN,NaN,NaN,False,False,c58b0108
2,James Anderson,JM Anderson,England,verified_alias,CR_jamesanderson,NaN,NaN,NaN,NaN,False,False,d12143bf
3,Liam Livingstone,LS Livingstone,England,verified_alias,CR_liamlivingstone,NaN,NaN,NaN,NaN,False,False,50c6bc2b
4,Mark Wood,MA Wood,England,verified_alias,CR_markwood,NaN,NaN,NaN,NaN,False,False,8d92a2c3
5,Mark Adair,MR Adair,Ireland,verified_alias,CR_markadair,NaN,NaN,NaN,NaN,False,False,fc55ec67
6,Andy Balbirnie,A Balbirnie,Ireland,verified_alias,CR_andybalbirnie,NaN,NaN,NaN,NaN,False,False,6ef13460
7,James McCollum,JA McCollum,Ireland,verified_alias,CR_jamesmccollum,NaN,NaN,NaN,NaN,False,False,c7105dab
8,Tom Latham,TWM Latham,New Zealand,verified_alias,CR_tomlatham,NaN,NaN,NaN,NaN,False,False,e824e6ee
9,Colin de Grandhomme,C de Grandhomme,New Zealand,verified_alias,CR_colindegrandhomme,NaN,NaN,NaN,NaN,False,False,94d7f855


New aliases: 15
Missing candidate IDs: 0
Duplicate candidate IDs: 0
Unique candidate IDs: 15


In [148]:
new_identity_records = new_alias_evidence[
    ["crickrida_name", "cricsheet_name", "team",
     "cricsheet_player_id"]
].copy()

new_identity_records = new_identity_records.rename(
    columns={"cricsheet_player_id": "player_id"}
)

new_identity_records["cricketiq_player_id"] = (
    "CRIQ_" + new_identity_records["player_id"].astype(str)
)

new_identity_records["normalized_name"] = (
    new_identity_records["crickrida_name"].apply(normalize_player_name)
)

new_identity_records["match_status"] = "verified_alias"

# Keep the same six-column structure as player_identity_map
new_identity_records = new_identity_records[
    [
        "cricketiq_player_id",
        "player_id",
        "cricsheet_name",
        "crickrida_name",
        "normalized_name",
        "match_status"
    ]
]

display(new_identity_records)

print("New identity records:", len(new_identity_records))
print("Unique CricketIQ IDs:",
      new_identity_records["cricketiq_player_id"].nunique())
print("Unique Cricsheet IDs:",
      new_identity_records["player_id"].nunique())
print("Missing values:", new_identity_records.isna().sum().sum())

,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,normalized_name,match_status
0,CRIQ_e1b9f3a9,e1b9f3a9,BJ Dwarshuis,Ben Dwarshuis,bendwarshuis,verified_alias
1,CRIQ_c58b0108,c58b0108,Mehedi Hasan Miraz,Mehidy Hasan Miraz,mehidyhasanmiraz,verified_alias
2,CRIQ_d12143bf,d12143bf,JM Anderson,James Anderson,jamesanderson,verified_alias
3,CRIQ_50c6bc2b,50c6bc2b,LS Livingstone,Liam Livingstone,liamlivingstone,verified_alias
4,CRIQ_8d92a2c3,8d92a2c3,MA Wood,Mark Wood,markwood,verified_alias
5,CRIQ_fc55ec67,fc55ec67,MR Adair,Mark Adair,markadair,verified_alias
6,CRIQ_6ef13460,6ef13460,A Balbirnie,Andy Balbirnie,andybalbirnie,verified_alias
7,CRIQ_c7105dab,c7105dab,JA McCollum,James McCollum,jamesmccollum,verified_alias
8,CRIQ_e824e6ee,e824e6ee,TWM Latham,Tom Latham,tomlatham,verified_alias
9,CRIQ_94d7f855,94d7f855,C de Grandhomme,Colin de Grandhomme,colindegrandhomme,verified_alias


New identity records: 15
Unique CricketIQ IDs: 15
Unique Cricsheet IDs: 15
Missing values: 0


In [149]:
# Combine the existing 106 identities with 15 new verified aliases
player_identity_map_updated = pd.concat(
    [player_identity_map, new_identity_records],
    ignore_index=True
)

print("UPDATED IDENTITY MAP")
print("Total records:", len(player_identity_map_updated))
print("Unique CricketIQ IDs:",
      player_identity_map_updated["cricketiq_player_id"].nunique())
print("Unique Cricsheet IDs:",
      player_identity_map_updated["player_id"].nunique())
print("Unique Crickrida names:",
      player_identity_map_updated["crickrida_name"].nunique())

print("\nDUPLICATE CHECKS")
print("Duplicate CricketIQ IDs:",
      player_identity_map_updated["cricketiq_player_id"].duplicated().sum())
print("Duplicate Cricsheet IDs:",
      player_identity_map_updated["player_id"].duplicated().sum())
print("Duplicate Crickrida names:",
      player_identity_map_updated["crickrida_name"].duplicated().sum())
print("Missing values:",
      player_identity_map_updated.isna().sum().sum())

print("\nMATCH STATUS")
display(
    player_identity_map_updated["match_status"]
    .value_counts()
    .rename_axis("match_status")
    .to_frame("count")
)

print("\nNEW TOTAL CROSS-SOURCE IDENTITIES:",
      len(player_identity_map_updated))

UPDATED IDENTITY MAP
Total records: 121
Unique CricketIQ IDs: 121
Unique Cricsheet IDs: 121
Unique Crickrida names: 121

DUPLICATE CHECKS
Duplicate CricketIQ IDs: 0
Duplicate Cricsheet IDs: 0
Duplicate Crickrida names: 0
Missing values: 0

MATCH STATUS


,count
match_status,
exact_name_match,98
verified_alias,23



NEW TOTAL CROSS-SOURCE IDENTITIES: 121


In [150]:
print("CURRENT MASTER REGISTRY")
print("Shape:", player_registry_final.shape)
print("Columns:", player_registry_final.columns.tolist())
display(player_registry_final.head())

print("\nCRICSHEET-ONLY MASTER")
print("Shape:", cricsheet_only_master.shape)
print("Columns:", cricsheet_only_master.columns.tolist())
display(cricsheet_only_master.head())

print("\nCRICKRIDA-ONLY MASTER")
print("Shape:", crickrida_only_master.shape)
print("Columns:", crickrida_only_master.columns.tolist())
display(crickrida_only_master.head())

print("\nCURRENT REGISTRY SOURCE COUNTS")
display(player_registry_final["source"].value_counts(dropna=False))

print("\nCURRENT REGISTRY MATCH STATUS")
display(player_registry_final["match_status"].value_counts(dropna=False))

CURRENT MASTER REGISTRY
Shape: (3635, 9)
Columns: ['cricketiq_player_id', 'player_id', 'cricsheet_name', 'crickrida_name', 'match_status', 'source', 'player_names', 'player_name', 'canonical_player_name']


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,match_status,source,player_names,player_name,canonical_player_name
0,CRIQ_8a75e999,8a75e999,Babar Azam,Babar Azam,exact_name_match,Both,NaN,NaN,Babar Azam
1,CRIQ_2911de16,2911de16,Hasan Ali,Hasan Ali,exact_name_match,Both,NaN,NaN,Hasan Ali
2,CRIQ_9cb8d7a6,9cb8d7a6,Imad Wasim,Imad Wasim,exact_name_match,Both,NaN,NaN,Imad Wasim
3,CRIQ_e174dadd,e174dadd,Mohammad Amir,Mohammad Amir,exact_name_match,Both,NaN,NaN,Mohammad Amir
4,CRIQ_9ab63e7b,9ab63e7b,Mohammad Hafeez,Mohammad Hafeez,exact_name_match,Both,NaN,NaN,Mohammad Hafeez



CRICSHEET-ONLY MASTER
Shape: (3182, 6)
Columns: ['player_id', 'player_names', 'normalized_names', 'cricketiq_player_id', 'source', 'match_status']


,player_id,player_names,normalized_names,cricketiq_player_id,source,match_status
0,004c9e85,[PJ Hughes],[pjhughes],CS_004c9e85,Cricsheet,source_only
1,005f0561,[Panna Ghosh],[pannaghosh],CS_005f0561,Cricsheet,source_only
2,006de9ca,[Zulfiqar Jan],[zulfiqarjan],CS_006de9ca,Cricsheet,source_only
3,0070ed27,[Glen Walklin],[glenwalklin],CS_0070ed27,Cricsheet,source_only
4,00812511,[Lui Els],[luiels],CS_00812511,Cricsheet,source_only



CRICKRIDA-ONLY MASTER
Shape: (348, 5)
Columns: ['player_name', 'normalized_name', 'cricketiq_player_id', 'source', 'match_status']


,player_name,normalized_name,cricketiq_player_id,source,match_status
0,Paul Stirling,paulstirling,CR_paulstirling,Crickrida,source_only
1,James Hall,jameshall,CR_jameshall,Crickrida,source_only
2,Alex Cusack,alexcusack,CR_alexcusack,Crickrida,source_only
3,Kevin O'Brien,kevinobrien,CR_kevinobrien,Crickrida,source_only
4,Andrew Poynter,andrewpoynter,CR_andrewpoynter,Crickrida,source_only



CURRENT REGISTRY SOURCE COUNTS


source
Cricsheet    3181
Crickrida     348
Both          106
Name: count, dtype: int64


CURRENT REGISTRY MATCH STATUS


match_status
source_only         3529
exact_name_match      98
verified_alias         8
Name: count, dtype: int64

In [151]:
# IDs and names of the 15 newly matched players
new_cricsheet_ids = set(new_identity_records["player_id"])
new_crickrida_names = set(new_identity_records["crickrida_name"])

# 1. Remove newly matched players from Cricsheet-only records
cricsheet_only_updated = cricsheet_only_master[
    ~cricsheet_only_master["player_id"].isin(new_cricsheet_ids)
].copy()

# 2. Remove newly matched players from Crickrida-only records
crickrida_only_updated = crickrida_only_master[
    ~crickrida_only_master["player_name"].isin(new_crickrida_names)
].copy()

# 3. Prepare new shared identity records
new_both_records = new_identity_records.copy()

new_both_records["source"] = "Both"
new_both_records["player_names"] = None
new_both_records["player_name"] = None

# Crickrida's full name becomes the canonical display name
new_both_records["canonical_player_name"] = (
    new_both_records["crickrida_name"]
)

# 4. Add canonical names to existing source-only records
cricsheet_only_updated["cricsheet_name"] = None
cricsheet_only_updated["crickrida_name"] = None
cricsheet_only_updated["canonical_player_name"] = (
    cricsheet_only_updated["player_names"].apply(
        lambda names: names[0] if isinstance(names, list) and len(names) > 0 else None
    )
)

crickrida_only_updated["player_id"] = None
crickrida_only_updated["cricsheet_name"] = None
crickrida_only_updated["crickrida_name"] = (
    crickrida_only_updated["player_name"]
)
crickrida_only_updated["player_names"] = None
crickrida_only_updated["canonical_player_name"] = (
    crickrida_only_updated["player_name"]
)

# 5. Combine all records
player_registry_updated = pd.concat(
    [
        player_registry_final,
        new_both_records
    ],
    ignore_index=True
)

# Remove the 15 newly matched players from the old source-only records
player_registry_updated = player_registry_updated[
    ~(
        (player_registry_updated["source"] == "Cricsheet") &
        (player_registry_updated["player_id"].isin(new_cricsheet_ids))
    )
]

player_registry_updated = player_registry_updated[
    ~(
        (player_registry_updated["source"] == "Crickrida") &
        (player_registry_updated["crickrida_name"].isin(new_crickrida_names))
    )
]

# 6. Validate
print("UPDATED MASTER REGISTRY")
print("Shape:", player_registry_updated.shape)

print("\nSOURCE COUNTS")
display(player_registry_updated["source"].value_counts())

print("\nMATCH STATUS COUNTS")
display(player_registry_updated["match_status"].value_counts())

print("\nDUPLICATE CHECKS")
print("Duplicate CricketIQ IDs:",
      player_registry_updated["cricketiq_player_id"].duplicated().sum())

print("Missing CricketIQ IDs:",
      player_registry_updated["cricketiq_player_id"].isna().sum())

print("Missing canonical names:",
      player_registry_updated["canonical_player_name"].isna().sum())

print("\nNEWLY MATCHED PLAYERS PRESENT:")
print(
    player_registry_updated[
        player_registry_updated["cricketiq_player_id"].isin(
            new_identity_records["cricketiq_player_id"]
        )
    ][
        ["cricketiq_player_id", "cricsheet_name",
         "crickrida_name", "canonical_player_name", "source"]
    ].to_string(index=False)
)

UPDATED MASTER REGISTRY
Shape: (3635, 10)

SOURCE COUNTS


source
Cricsheet    3166
Crickrida     348
Both          121
Name: count, dtype: int64


MATCH STATUS COUNTS


match_status
source_only         3514
exact_name_match      98
verified_alias        23
Name: count, dtype: int64


DUPLICATE CHECKS
Duplicate CricketIQ IDs: 0
Missing CricketIQ IDs: 0
Missing canonical names: 0

NEWLY MATCHED PLAYERS PRESENT:
cricketiq_player_id     cricsheet_name      crickrida_name canonical_player_name source
      CRIQ_e1b9f3a9       BJ Dwarshuis       Ben Dwarshuis         Ben Dwarshuis   Both
      CRIQ_c58b0108 Mehedi Hasan Miraz  Mehidy Hasan Miraz    Mehidy Hasan Miraz   Both
      CRIQ_d12143bf        JM Anderson      James Anderson        James Anderson   Both
      CRIQ_50c6bc2b     LS Livingstone    Liam Livingstone      Liam Livingstone   Both
      CRIQ_8d92a2c3            MA Wood           Mark Wood             Mark Wood   Both
      CRIQ_fc55ec67           MR Adair          Mark Adair            Mark Adair   Both
      CRIQ_6ef13460        A Balbirnie      Andy Balbirnie        Andy Balbirnie   Both
      CRIQ_c7105dab        JA McCollum      James McCollum        James McCollum   Both
      CRIQ_e824e6ee         TWM Latham          Tom Latham            Tom Latha

In [152]:
duplicate_crickrida_records = player_registry_updated[
    (player_registry_updated["source"] == "Crickrida") &
    (player_registry_updated["crickrida_name"].isin(new_crickrida_names))
].copy()

print("Remaining duplicate source-only records:",
      len(duplicate_crickrida_records))

display(
    duplicate_crickrida_records[
        ["cricketiq_player_id", "crickrida_name",
         "canonical_player_name", "source", "match_status"]
    ]
)

Remaining duplicate source-only records: 0


,cricketiq_player_id,crickrida_name,canonical_player_name,source,match_status


In [153]:
print("ORIGINAL CRICKRIDA-ONLY MASTER")
print("Rows:", len(crickrida_only_master))
print("Unique player names:",
      crickrida_only_master["player_name"].nunique())

print("\nUPDATED REGISTRY: CRICKRIDA SOURCE")
crickrida_source = player_registry_updated[
    player_registry_updated["source"] == "Crickrida"
].copy()

print("Rows:", len(crickrida_source))
print("Unique Crickrida names:",
      crickrida_source["crickrida_name"].nunique())

print("\nCHECK NEWLY MATCHED NAMES IN ORIGINAL MASTER")
matched_names_check = crickrida_only_master[
    crickrida_only_master["player_name"].isin(new_crickrida_names)
]

print("Newly matched names still in original Crickrida-only master:",
      len(matched_names_check))

display(matched_names_check)

print("\nCHECK SOURCE-ONLY RECORDS IN UPDATED REGISTRY")
display(
    crickrida_source[
        ["cricketiq_player_id", "crickrida_name",
         "canonical_player_name", "source", "match_status"]
    ].head(20)
)

ORIGINAL CRICKRIDA-ONLY MASTER
Rows: 348
Unique player names: 348

UPDATED REGISTRY: CRICKRIDA SOURCE
Rows: 348
Unique Crickrida names: 0

CHECK NEWLY MATCHED NAMES IN ORIGINAL MASTER
Newly matched names still in original Crickrida-only master: 15


,player_name,normalized_name,cricketiq_player_id,source,match_status
39,Ed Joyce,edjoyce,CR_edjoyce,Crickrida,source_only
242,Andy Balbirnie,andybalbirnie,CR_andybalbirnie,Crickrida,source_only
1097,Carlos Brathwaite,carlosbrathwaite,CR_carlosbrathwaite,Crickrida,source_only
1236,Mehidy Hasan Miraz,mehidyhasanmiraz,CR_mehidyhasanmiraz,Crickrida,source_only
1299,James McCollum,jamesmccollum,CR_jamesmccollum,Crickrida,source_only
1386,Mark Adair,markadair,CR_markadair,Crickrida,source_only
1473,Tom Latham,tomlatham,CR_tomlatham,Crickrida,source_only
1487,Andile Phehlukwayo,andilephehlukwayo,CR_andilephehlukwayo,Crickrida,source_only
2100,Liam Livingstone,liamlivingstone,CR_liamlivingstone,Crickrida,source_only
2104,Mark Wood,markwood,CR_markwood,Crickrida,source_only



CHECK SOURCE-ONLY RECORDS IN UPDATED REGISTRY


,cricketiq_player_id,crickrida_name,canonical_player_name,source,match_status
3287,CR_paulstirling,NaN,Paul Stirling,Crickrida,source_only
3288,CR_jameshall,NaN,James Hall,Crickrida,source_only
3289,CR_alexcusack,NaN,Alex Cusack,Crickrida,source_only
3290,CR_kevinobrien,NaN,Kevin O'Brien,Crickrida,source_only
3291,CR_andrewpoynter,NaN,Andrew Poynter,Crickrida,source_only
3292,CR_andrewwhite,NaN,Andrew White,Crickrida,source_only
3293,CR_johnmooney,NaN,John Mooney,Crickrida,source_only
3294,CR_trentjohnston,NaN,Trent Johnston,Crickrida,source_only
3295,CR_nigeljones,NaN,Nigel Jones,Crickrida,source_only
3296,CR_rorymccann,NaN,Rory McCann,Crickrida,source_only


In [154]:
# 1. Filter Cricsheet-only players
cricsheet_only_updated = cricsheet_only_master[
    ~cricsheet_only_master["player_id"].isin(
        player_identity_map_updated["player_id"]
    )
].copy()

# 2. Filter Crickrida-only players
crickrida_only_updated = crickrida_only_master[
    ~crickrida_only_master["player_name"].isin(
        player_identity_map_updated["crickrida_name"]
    )
].copy()

# 3. Prepare the 121 shared identities
both_updated = player_identity_map_updated.copy()

both_updated["source"] = "Both"
both_updated["player_names"] = None
both_updated["player_name"] = None
both_updated["canonical_player_name"] = both_updated["crickrida_name"]

# 4. Prepare Cricsheet-only registry records
cs_updated = pd.DataFrame({
    "cricketiq_player_id": cricsheet_only_updated["cricketiq_player_id"],
    "player_id": cricsheet_only_updated["player_id"],
    "cricsheet_name": None,
    "crickrida_name": None,
    "match_status": "source_only",
    "source": "Cricsheet",
    "player_names": cricsheet_only_updated["player_names"],
    "player_name": None,
    "canonical_player_name": cricsheet_only_updated["player_names"].apply(
        lambda x: x[0] if isinstance(x, list) and len(x) else None
    )
})

# 5. Prepare Crickrida-only registry records
cr_updated = pd.DataFrame({
    "cricketiq_player_id": crickrida_only_updated["cricketiq_player_id"],
    "player_id": None,
    "cricsheet_name": None,
    "crickrida_name": crickrida_only_updated["player_name"],
    "match_status": "source_only",
    "source": "Crickrida",
    "player_names": None,
    "player_name": crickrida_only_updated["player_name"],
    "canonical_player_name": crickrida_only_updated["player_name"]
})

# 6. Combine all three groups
player_registry_final_updated = pd.concat(
    [both_updated, cs_updated, cr_updated],
    ignore_index=True
)

# 7. Validate
print("FINAL REGISTRY SHAPE:", player_registry_final_updated.shape)

print("\nSOURCE COUNTS")
display(player_registry_final_updated["source"].value_counts())

print("\nMATCH STATUS COUNTS")
display(player_registry_final_updated["match_status"].value_counts())

print("\nVALIDATION")
print("Duplicate CricketIQ IDs:",
      player_registry_final_updated["cricketiq_player_id"].duplicated().sum())
print("Missing CricketIQ IDs:",
      player_registry_final_updated["cricketiq_player_id"].isna().sum())
print("Missing canonical names:",
      player_registry_final_updated["canonical_player_name"].isna().sum())
print("New shared identities:",
      (player_registry_final_updated["source"] == "Both").sum())

print("\nEXPECTED TOTAL: 3620")

FINAL REGISTRY SHAPE: (3620, 10)

SOURCE COUNTS


source
Cricsheet    3166
Crickrida     333
Both          121
Name: count, dtype: int64


MATCH STATUS COUNTS


match_status
source_only         3499
exact_name_match      98
verified_alias        23
Name: count, dtype: int64


VALIDATION
Duplicate CricketIQ IDs: 0
Missing CricketIQ IDs: 0
Missing canonical names: 0
New shared identities: 121

EXPECTED TOTAL: 3620


In [155]:
# Cricsheet: map every source player ID to its CricketIQ ID
cricsheet_id_lookup_updated = (
    player_registry_final_updated[
        player_registry_final_updated["player_id"].notna()
    ][["player_id", "cricketiq_player_id"]]
    .copy()
    .reset_index(drop=True)
)

# Crickrida: map every source player name to its CricketIQ ID
crickrida_id_lookup_updated = (
    player_registry_final_updated[
        player_registry_final_updated["crickrida_name"].notna()
    ][["crickrida_name", "cricketiq_player_id"]]
    .rename(columns={"crickrida_name": "lookup_name"})
    .copy()
    .reset_index(drop=True)
)

# Validate Cricsheet lookup
print("CRICSHEET LOOKUP")
print("Shape:", cricsheet_id_lookup_updated.shape)
print("Unique source IDs:",
      cricsheet_id_lookup_updated["player_id"].nunique())
print("Duplicate source IDs:",
      cricsheet_id_lookup_updated["player_id"].duplicated().sum())
print("Missing CricketIQ IDs:",
      cricsheet_id_lookup_updated["cricketiq_player_id"].isna().sum())

# Validate Crickrida lookup
print("\nCRICKRIDA LOOKUP")
print("Shape:", crickrida_id_lookup_updated.shape)
print("Unique source names:",
      crickrida_id_lookup_updated["lookup_name"].nunique())
print("Duplicate source names:",
      crickrida_id_lookup_updated["lookup_name"].duplicated().sum())
print("Missing CricketIQ IDs:",
      crickrida_id_lookup_updated["cricketiq_player_id"].isna().sum())

print("\nEXPECTED LOOKUP SIZES")
print("Cricsheet: 3287")
print("Crickrida: 454")

CRICSHEET LOOKUP
Shape: (3287, 2)
Unique source IDs: 3287
Duplicate source IDs: 0
Missing CricketIQ IDs: 0

CRICKRIDA LOOKUP
Shape: (454, 2)
Unique source names: 454
Duplicate source names: 0
Missing CricketIQ IDs: 0

EXPECTED LOOKUP SIZES
Cricsheet: 3287
Crickrida: 454


In [156]:
# Create a canonical-name lookup from the validated registry
canonical_lookup = player_registry_final_updated[
    ["cricketiq_player_id", "canonical_player_name"]
].copy()

def update_player_identity(df, source_key, lookup_df, lookup_key):
    # Remove old identity columns before refreshing them
    df = df.drop(
        columns=["cricketiq_player_id", "canonical_player_name"],
        errors="ignore"
    )

    # Attach the updated CricketIQ ID
    df = df.merge(
        lookup_df,
        left_on=source_key,
        right_on=lookup_key,
        how="left",
        validate="many_to_one"
    )

    if source_key != lookup_key:
        df = df.drop(columns=[lookup_key])

    # Attach the canonical display name
    df = df.merge(
        canonical_lookup,
        on="cricketiq_player_id",
        how="left",
        validate="many_to_one"
    )

    return df


# 1. Cricsheet batting
player_batting_updated = update_player_identity(
    player_batting_df,
    "batter_id",
    cricsheet_id_lookup_updated,
    "player_id"
)

# 2. Cricsheet bowling
player_bowling_updated = update_player_identity(
    player_bowling_df,
    "bowler_id",
    cricsheet_id_lookup_updated,
    "player_id"
)

# 3. Cricsheet dismissals
player_dismissal_updated = update_player_identity(
    player_dismissal_df,
    "player_out_id",
    cricsheet_id_lookup_updated,
    "player_id"
)

# 4. Cricsheet bowler wickets
dismissal_bowler_updated = update_player_identity(
    dismissal_bowler_df,
    "bowler_id",
    cricsheet_id_lookup_updated,
    "player_id"
)

# 5. Afghanistan / Crickrida batting
afg_player_batting_updated = update_player_identity(
    afg_player_batting_df,
    "player_name",
    crickrida_id_lookup_updated,
    "lookup_name"
)

# 6. Afghanistan / Crickrida bowling
afg_player_bowling_updated = update_player_identity(
    afg_player_bowling_df,
    "player_name",
    crickrida_id_lookup_updated,
    "lookup_name"
)


# Validate all six datasets
updated_datasets = {
    "Cricsheet batting": player_batting_updated,
    "Cricsheet bowling": player_bowling_updated,
    "Cricsheet dismissals": player_dismissal_updated,
    "Cricsheet bowler wickets": dismissal_bowler_updated,
    "Afghanistan batting": afg_player_batting_updated,
    "Afghanistan bowling": afg_player_bowling_updated
}

for name, df in updated_datasets.items():
    print(f"\n{name}")
    print("Rows:", len(df))
    print("Missing CricketIQ IDs:",
          df["cricketiq_player_id"].isna().sum())
    print("Missing canonical names:",
          df["canonical_player_name"].isna().sum())


Cricsheet batting
Rows: 45432
Missing CricketIQ IDs: 0
Missing canonical names: 0

Cricsheet bowling
Rows: 31186
Missing CricketIQ IDs: 0
Missing canonical names: 0

Cricsheet dismissals
Rows: 37711
Missing CricketIQ IDs: 0
Missing canonical names: 0

Cricsheet bowler wickets
Rows: 34975
Missing CricketIQ IDs: 0
Missing canonical names: 0

Afghanistan batting
Rows: 2719
Missing CricketIQ IDs: 0
Missing canonical names: 0

Afghanistan bowling
Rows: 1765
Missing CricketIQ IDs: 0
Missing canonical names: 0


In [157]:
# IDs of the 15 newly matched players
new_shared_ids = set(
    new_identity_records["cricketiq_player_id"]
)

cross_source_validation = []

for _, player in new_identity_records.iterrows():
    player_id = player["cricketiq_player_id"]

    # Count records in Afghanistan / Crickrida datasets
    afg_batting_count = (
        afg_player_batting_updated["cricketiq_player_id"] == player_id
    ).sum()

    afg_bowling_count = (
        afg_player_bowling_updated["cricketiq_player_id"] == player_id
    ).sum()

    # Count records in Cricsheet datasets
    cs_batting_count = (
        player_batting_updated["cricketiq_player_id"] == player_id
    ).sum()

    cs_bowling_count = (
        player_bowling_updated["cricketiq_player_id"] == player_id
    ).sum()

    cross_source_validation.append({
        "player": player["crickrida_name"],
        "cricketiq_player_id": player_id,
        "afghanistan_batting_records": afg_batting_count,
        "afghanistan_bowling_records": afg_bowling_count,
        "cricsheet_batting_records": cs_batting_count,
        "cricsheet_bowling_records": cs_bowling_count
    })

cross_source_validation_df = pd.DataFrame(
    cross_source_validation
)

cross_source_validation_df["present_in_both_sources"] = (
    (
        cross_source_validation_df[
            ["afghanistan_batting_records",
             "afghanistan_bowling_records"]
        ].sum(axis=1) > 0
    ) &
    (
        cross_source_validation_df[
            ["cricsheet_batting_records",
             "cricsheet_bowling_records"]
        ].sum(axis=1) > 0
    )
)

display(cross_source_validation_df)

print("New shared identities:", len(cross_source_validation_df))
print("Present in both sources:",
      cross_source_validation_df["present_in_both_sources"].sum())
print("Not found in both sources:",
      (~cross_source_validation_df["present_in_both_sources"]).sum())

,player,cricketiq_player_id,afghanistan_batting_records,afghanistan_bowling_records,cricsheet_batting_records,cricsheet_bowling_records,present_in_both_sources
0,Ben Dwarshuis,CRIQ_e1b9f3a9,0,1,4,6,True
1,Mehidy Hasan Miraz,CRIQ_c58b0108,15,17,79,106,True
2,James Anderson,CRIQ_d12143bf,0,1,64,177,True
3,Liam Livingstone,CRIQ_50c6bc2b,2,2,34,29,True
4,Mark Wood,CRIQ_8d92a2c3,2,3,22,65,True
5,Mark Adair,CRIQ_fc55ec67,8,8,34,47,True
6,Andy Balbirnie,CRIQ_6ef13460,28,2,85,1,True
7,James McCollum,CRIQ_c7105dab,4,0,6,0,True
8,Tom Latham,CRIQ_e824e6ee,2,0,156,0,True
9,Colin de Grandhomme,CRIQ_94d7f855,0,1,35,44,True


New shared identities: 15
Present in both sources: 15
Not found in both sources: 0


In [158]:
from pathlib import Path

player_data_dir = Path(r"C:\CricIQ\data\processed\player_data")
player_data_dir.mkdir(parents=True, exist_ok=True)

files_to_save = {
    "cricketiq_player_identity_registry.csv": player_registry_final_updated,
    "cricsheet_id_lookup.csv": cricsheet_id_lookup_updated,
    "crickrida_id_lookup.csv": crickrida_id_lookup_updated,
    "cricsheet_player_batting.csv": player_batting_updated,
    "cricsheet_player_bowling.csv": player_bowling_updated,
    "cricsheet_player_dismissals.csv": player_dismissal_updated,
    "cricsheet_bowler_wickets.csv": dismissal_bowler_updated,
    "afghanistan_player_batting.csv": afg_player_batting_updated,
    "afghanistan_player_bowling.csv": afg_player_bowling_updated
}

for filename, df in files_to_save.items():
    df.to_csv(player_data_dir / filename, index=False)
    print(f"Saved: {filename} | Shape: {df.shape}")

print("\nAll updated player identity files saved.")

Saved: cricketiq_player_identity_registry.csv | Shape: (3620, 10)
Saved: cricsheet_id_lookup.csv | Shape: (3287, 2)
Saved: crickrida_id_lookup.csv | Shape: (454, 2)
Saved: cricsheet_player_batting.csv | Shape: (45432, 12)
Saved: cricsheet_player_bowling.csv | Shape: (31186, 13)
Saved: cricsheet_player_dismissals.csv | Shape: (37711, 8)
Saved: cricsheet_bowler_wickets.csv | Shape: (34975, 8)
Saved: afghanistan_player_batting.csv | Shape: (2719, 13)
Saved: afghanistan_player_bowling.csv | Shape: (1765, 17)

All updated player identity files saved.


In [159]:
# Reload all saved datasets
reloaded_files = {}

for filename in files_to_save:
    reloaded_files[filename] = pd.read_csv(
        player_data_dir / filename
    )

    print(
        f"{filename}: "
        f"{reloaded_files[filename].shape}"
    )

# Validate the master registry
registry_check = reloaded_files[
    "cricketiq_player_identity_registry.csv"
]

print("\nREGISTRY VALIDATION")
print("Total players:", len(registry_check))
print("Unique CricketIQ IDs:",
      registry_check["cricketiq_player_id"].nunique())
print("Duplicate IDs:",
      registry_check["cricketiq_player_id"].duplicated().sum())
print("Missing canonical names:",
      registry_check["canonical_player_name"].isna().sum())

# Validate all six performance datasets
performance_files = [
    "cricsheet_player_batting.csv",
    "cricsheet_player_bowling.csv",
    "cricsheet_player_dismissals.csv",
    "cricsheet_bowler_wickets.csv",
    "afghanistan_player_batting.csv",
    "afghanistan_player_bowling.csv"
]

print("\nPERFORMANCE DATASET VALIDATION")

for filename in performance_files:
    df = reloaded_files[filename]

    print(f"\n{filename}")
    print("Rows:", len(df))
    print("Missing IDs:", df["cricketiq_player_id"].isna().sum())
    print("Missing canonical names:",
          df["canonical_player_name"].isna().sum())

# Verify all 15 new shared identities survived saving
new_ids = set(new_identity_records["cricketiq_player_id"])

saved_shared = registry_check[
    registry_check["cricketiq_player_id"].isin(new_ids)
]

print("\nNEW SHARED IDENTITY VALIDATION")
print("Expected:", len(new_ids))
print("Found in saved registry:", len(saved_shared))
print("All source labels are Both:",
      (saved_shared["source"] == "Both").all())

cricketiq_player_identity_registry.csv: (3620, 10)
cricsheet_id_lookup.csv: (3287, 2)
crickrida_id_lookup.csv: (454, 2)
cricsheet_player_batting.csv: (45432, 12)
cricsheet_player_bowling.csv: (31186, 13)
cricsheet_player_dismissals.csv: (37711, 8)
cricsheet_bowler_wickets.csv: (34975, 8)
afghanistan_player_batting.csv: (2719, 13)
afghanistan_player_bowling.csv: (1765, 17)

REGISTRY VALIDATION
Total players: 3620
Unique CricketIQ IDs: 3620
Duplicate IDs: 0
Missing canonical names: 0

PERFORMANCE DATASET VALIDATION

cricsheet_player_batting.csv
Rows: 45432
Missing IDs: 0
Missing canonical names: 0

cricsheet_player_bowling.csv
Rows: 31186
Missing IDs: 0
Missing canonical names: 0

cricsheet_player_dismissals.csv
Rows: 37711
Missing IDs: 0
Missing canonical names: 0

cricsheet_bowler_wickets.csv
Rows: 34975
Missing IDs: 0
Missing canonical names: 0

afghanistan_player_batting.csv
Rows: 2719
Missing IDs: 0
Missing canonical names: 0

afghanistan_player_bowling.csv
Rows: 1765
Missing IDs: 0

In [160]:
# Load the saved player datasets
afg_bat = pd.read_csv(
    player_data_dir / "afghanistan_player_batting.csv"
)

cs_bat = pd.read_csv(
    player_data_dir / "cricsheet_player_batting.csv"
)

afg_bowl = pd.read_csv(
    player_data_dir / "afghanistan_player_bowling.csv"
)

cs_bowl = pd.read_csv(
    player_data_dir / "cricsheet_player_bowling.csv"
)

# Find players appearing in both batting datasets
shared_batting_ids = (
    set(afg_bat["cricketiq_player_id"])
    & set(cs_bat["cricketiq_player_id"])
)

# Find players appearing in both bowling datasets
shared_bowling_ids = (
    set(afg_bowl["cricketiq_player_id"])
    & set(cs_bowl["cricketiq_player_id"])
)

# Inspect the unified player identities
shared_players = player_registry_final_updated[
    player_registry_final_updated["cricketiq_player_id"].isin(
        shared_batting_ids | shared_bowling_ids
    )
].copy()

shared_players = shared_players[
    ["cricketiq_player_id", "crickrida_name",
     "cricsheet_name", "canonical_player_name", "match_status"]
].sort_values("canonical_player_name")

display(shared_players)

print("Shared batting identities:", len(shared_batting_ids))
print("Shared bowling identities:", len(shared_bowling_ids))
print("Shared identities appearing in either batting or bowling:",
      len(shared_players))

,cricketiq_player_id,crickrida_name,cricsheet_name,canonical_player_name,match_status
66,CRIQ_fc2fffb5,Abdullah Shafique,Abdullah Shafique,Abdullah Shafique,exact_name_match
88,CRIQ_888e32bf,Abdur Razzak,Abdur Razzak,Abdur Razzak,exact_name_match
51,CRIQ_31803fac,Abu Hider,Abu Hider,Abu Hider,exact_name_match
55,CRIQ_37138918,Afif Hossain,Afif Hossain,Afif Hossain,exact_name_match
29,CRIQ_f8fd6e3f,Ahmed Shehzad,Ahmed Shehzad,Ahmed Shehzad,exact_name_match
...,...,...,...,...,...
8,CRIQ_b3118300,Wahab Riaz,Wahab Riaz,Wahab Riaz,exact_name_match
43,CRIQ_f19ccfad,Washington Sundar,Washington Sundar,Washington Sundar,exact_name_match
85,CRIQ_33cb3411,Younis Khan,Younis Khan,Younis Khan,exact_name_match
76,CRIQ_e3e09091,Zakir Hasan,Zakir Hasan,Zakir Hasan,exact_name_match


Shared batting identities: 104
Shared bowling identities: 80
Shared identities appearing in either batting or bowling: 120


In [161]:
# Load the saved player datasets
afg_bat = pd.read_csv(
    player_data_dir / "afghanistan_player_batting.csv"
)

cs_bat = pd.read_csv(
    player_data_dir / "cricsheet_player_batting.csv"
)

afg_bowl = pd.read_csv(
    player_data_dir / "afghanistan_player_bowling.csv"
)

cs_bowl = pd.read_csv(
    player_data_dir / "cricsheet_player_bowling.csv"
)

# Find players appearing in both batting datasets
shared_batting_ids = (
    set(afg_bat["cricketiq_player_id"])
    & set(cs_bat["cricketiq_player_id"])
)

# Find players appearing in both bowling datasets
shared_bowling_ids = (
    set(afg_bowl["cricketiq_player_id"])
    & set(cs_bowl["cricketiq_player_id"])
)

# Inspect the unified player identities
shared_players = player_registry_final_updated[
    player_registry_final_updated["cricketiq_player_id"].isin(
        shared_batting_ids | shared_bowling_ids
    )
].copy()

shared_players = shared_players[
    ["cricketiq_player_id", "crickrida_name",
     "cricsheet_name", "canonical_player_name", "match_status"]
].sort_values("canonical_player_name")

display(shared_players)

print("Shared batting identities:", len(shared_batting_ids))
print("Shared bowling identities:", len(shared_bowling_ids))
print("Shared identities appearing in either batting or bowling:",
      len(shared_players))

,cricketiq_player_id,crickrida_name,cricsheet_name,canonical_player_name,match_status
66,CRIQ_fc2fffb5,Abdullah Shafique,Abdullah Shafique,Abdullah Shafique,exact_name_match
88,CRIQ_888e32bf,Abdur Razzak,Abdur Razzak,Abdur Razzak,exact_name_match
51,CRIQ_31803fac,Abu Hider,Abu Hider,Abu Hider,exact_name_match
55,CRIQ_37138918,Afif Hossain,Afif Hossain,Afif Hossain,exact_name_match
29,CRIQ_f8fd6e3f,Ahmed Shehzad,Ahmed Shehzad,Ahmed Shehzad,exact_name_match
...,...,...,...,...,...
8,CRIQ_b3118300,Wahab Riaz,Wahab Riaz,Wahab Riaz,exact_name_match
43,CRIQ_f19ccfad,Washington Sundar,Washington Sundar,Washington Sundar,exact_name_match
85,CRIQ_33cb3411,Younis Khan,Younis Khan,Younis Khan,exact_name_match
76,CRIQ_e3e09091,Zakir Hasan,Zakir Hasan,Zakir Hasan,exact_name_match


Shared batting identities: 104
Shared bowling identities: 80
Shared identities appearing in either batting or bowling: 120


In [162]:
# All 121 shared identities from the registry
registry_shared_ids = set(
    player_registry_final_updated.loc[
        player_registry_final_updated["source"] == "Both",
        "cricketiq_player_id"
    ]
)

# Identities appearing in either batting or bowling datasets
performance_shared_ids = (
    shared_batting_ids | shared_bowling_ids
)

# Find identities not represented in either source's performance data
missing_performance_ids = (
    registry_shared_ids - performance_shared_ids
)

missing_player_check = player_registry_final_updated[
    player_registry_final_updated["cricketiq_player_id"].isin(
        missing_performance_ids
    )
][
    ["cricketiq_player_id", "crickrida_name",
     "cricsheet_name", "canonical_player_name", "match_status"]
]

print("Shared identities in registry:", len(registry_shared_ids))
print("Identities in performance datasets:", len(performance_shared_ids))
print("Missing from performance datasets:", len(missing_performance_ids))

display(missing_player_check)

Shared identities in registry: 121
Identities in performance datasets: 120
Missing from performance datasets: 1


,cricketiq_player_id,crickrida_name,cricsheet_name,canonical_player_name,match_status
70,CRIQ_c290033d,Ihsanullah,Ihsanullah,Ihsanullah,exact_name_match


In [163]:
ihsanullah_id = "CRIQ_c290033d"

# Inspect Afghanistan / Crickrida records
ihsanullah_afg_bat = afg_bat[
    afg_bat["cricketiq_player_id"] == ihsanullah_id
]

ihsanullah_afg_bowl = afg_bowl[
    afg_bowl["cricketiq_player_id"] == ihsanullah_id
]

# Inspect Cricsheet records
ihsanullah_cs_bat = cs_bat[
    cs_bat["cricketiq_player_id"] == ihsanullah_id
]

ihsanullah_cs_bowl = cs_bowl[
    cs_bowl["cricketiq_player_id"] == ihsanullah_id
]

print("AFGHANISTAN BATTING RECORDS:", len(ihsanullah_afg_bat))
display(ihsanullah_afg_bat)

print("\nAFGHANISTAN BOWLING RECORDS:", len(ihsanullah_afg_bowl))
display(ihsanullah_afg_bowl)

print("\nCRICSHEET BATTING RECORDS:", len(ihsanullah_cs_bat))
display(ihsanullah_cs_bat)

print("\nCRICSHEET BOWLING RECORDS:", len(ihsanullah_cs_bowl))
display(ihsanullah_cs_bowl)

# Check raw source participation history
print("\nCRICKRIDA APPEARANCES")
display(
    crickrida_appearances[
        crickrida_appearances["player_name"] == "Ihsanullah"
    ]
)

print("\nCRICSHEET APPEARANCES")
display(
    cricsheet_appearances[
        cricsheet_appearances["source_player_id"] == "c290033d"
    ]
)

AFGHANISTAN BATTING RECORDS: 14


,player_name,dismissal,runs_scored,balls_faced,fours,sixes,innings_number,date,match_url,strike_rate,batting_team,cricketiq_player_id,canonical_player_name
752,Ihsanullah,caught Tarisai Musakanda b Tendai Chatara,0,20,0,0,1,2017-02-24,https://crickrida.com/matches/afghanistan-zimb...,0.000000,Afghanistan,CRIQ_c290033d,Ihsanullah
922,Ihsanullah,caught Niall O'Brien b Boyd Rankin,10,15,0,1,1,2017-12-05,https://crickrida.com/matches/afghanistan-irel...,66.666667,Afghanistan,CRIQ_c290033d,Ihsanullah
954,Ihsanullah,bowled b Barry McCarthy,20,29,1,1,2,2017-12-07,https://crickrida.com/matches/afghanistan-irel...,68.965517,Afghanistan,CRIQ_c290033d,Ihsanullah
983,Ihsanullah,caught Solomon Mire b Graeme Cremer,54,53,9,0,1,2018-02-09,https://crickrida.com/matches/afghanistan-zimb...,101.886792,Afghanistan,CRIQ_c290033d,Ihsanullah
1007,Ihsanullah,caught Brian Vitori b Tendai Chatara,2,6,0,0,2,2018-02-11,https://crickrida.com/matches/afghanistan-zimb...,33.333333,Afghanistan,CRIQ_c290033d,Ihsanullah
1029,Ihsanullah,caught Malcolm Waller b Tendai Chatara,12,16,2,0,2,2018-02-13,https://crickrida.com/matches/afghanistan-zimb...,75.000000,Afghanistan,CRIQ_c290033d,Ihsanullah
1046,Ihsanullah,not out,51,53,6,2,2,2018-02-16,https://crickrida.com/matches/afghanistan-zimb...,96.226415,Afghanistan,CRIQ_c290033d,Ihsanullah
1080,Ihsanullah,lbw b Brian Vitori,1,9,0,0,2,2018-03-06,https://crickrida.com/matches/afghanistan-zimb...,11.111111,Afghanistan,CRIQ_c290033d,Ihsanullah
1142,Ihsanullah,caught Tim Murtagh b Tim Murtagh,9,24,1,0,1,2018-08-27,https://crickrida.com/matches/afghanistan-irel...,37.500000,Afghanistan,CRIQ_c290033d,Ihsanullah
1195,Ihsanullah,not out,57,62,6,2,2,2018-08-31,https://crickrida.com/matches/afghanistan-irel...,91.935484,Afghanistan,CRIQ_c290033d,Ihsanullah



AFGHANISTAN BOWLING RECORDS: 0


,player_name,overs_display,maidens,runs_conceded,wickets,wides,no_balls,legal_balls,innings_number,date,match_url,economy,batting_team,bowling_team,opponent,cricketiq_player_id,canonical_player_name



CRICSHEET BATTING RECORDS: 0


,match_id,innings_number,batting_team,batter,batter_id,runs,balls_faced,fours,sixes,strike_rate,cricketiq_player_id,canonical_player_name


... (2 outputs left out)


CRICKRIDA APPEARANCES


,date,team,player_name,source_player_id,match_id,role,source
752,2017-02-24,Afghanistan,Ihsanullah,CRIQ_c290033d,https://crickrida.com/matches/afghanistan-zimb...,batting,Crickrida
922,2017-12-05,Afghanistan,Ihsanullah,CRIQ_c290033d,https://crickrida.com/matches/afghanistan-irel...,batting,Crickrida
954,2017-12-07,Afghanistan,Ihsanullah,CRIQ_c290033d,https://crickrida.com/matches/afghanistan-irel...,batting,Crickrida
983,2018-02-09,Afghanistan,Ihsanullah,CRIQ_c290033d,https://crickrida.com/matches/afghanistan-zimb...,batting,Crickrida
1007,2018-02-11,Afghanistan,Ihsanullah,CRIQ_c290033d,https://crickrida.com/matches/afghanistan-zimb...,batting,Crickrida
1029,2018-02-13,Afghanistan,Ihsanullah,CRIQ_c290033d,https://crickrida.com/matches/afghanistan-zimb...,batting,Crickrida
1046,2018-02-16,Afghanistan,Ihsanullah,CRIQ_c290033d,https://crickrida.com/matches/afghanistan-zimb...,batting,Crickrida
1080,2018-03-06,Afghanistan,Ihsanullah,CRIQ_c290033d,https://crickrida.com/matches/afghanistan-zimb...,batting,Crickrida
1142,2018-08-27,Afghanistan,Ihsanullah,CRIQ_c290033d,https://crickrida.com/matches/afghanistan-irel...,batting,Crickrida
1195,2018-08-31,Afghanistan,Ihsanullah,CRIQ_c290033d,https://crickrida.com/matches/afghanistan-irel...,batting,Crickrida



CRICSHEET APPEARANCES


,date,team,player_name,source_player_id,match_id,role,source
52902,2023-04-29,Pakistan,Ihsanullah,c290033d,1339620,bowling,Cricsheet


In [164]:
# Recalculate shared performance IDs from the actual datasets
afg_ids = (
    set(afg_bat["cricketiq_player_id"].dropna())
    | set(afg_bowl["cricketiq_player_id"].dropna())
)

cs_ids = (
    set(cs_bat["cricketiq_player_id"].dropna())
    | set(cs_bowl["cricketiq_player_id"].dropna())
)

actual_shared_ids = afg_ids & cs_ids

missing_ids = registry_shared_ids - actual_shared_ids

print("Shared identities in registry:", len(registry_shared_ids))
print("Shared identities in performance datasets:", len(actual_shared_ids))
print("Missing identities:", len(missing_ids))

print("\nIHSANULLAH CHECK")
print("Afghanistan batting:",
      (afg_bat["cricketiq_player_id"] == ihsanullah_id).sum())
print("Afghanistan bowling:",
      (afg_bowl["cricketiq_player_id"] == ihsanullah_id).sum())
print("Cricsheet batting:",
      (cs_bat["cricketiq_player_id"] == ihsanullah_id).sum())
print("Cricsheet bowling:",
      (cs_bowl["cricketiq_player_id"] == ihsanullah_id).sum())

Shared identities in registry: 121
Shared identities in performance datasets: 121
Missing identities: 0

IHSANULLAH CHECK
Afghanistan batting: 14
Afghanistan bowling: 0
Cricsheet batting: 0
Cricsheet bowling: 1


In [165]:
# Search both source datasets and the identity registry
warner_check = player_registry_final_updated[
    player_registry_final_updated.apply(
        lambda row: any(
            "warner" in str(row[col]).lower()
            for col in ["cricsheet_name", "crickrida_name",
                        "canonical_player_name"]
        ),
        axis=1
    )
]

display(
    warner_check[
        ["cricketiq_player_id", "cricsheet_name",
         "crickrida_name", "canonical_player_name",
         "match_status", "source"]
    ]
)

,cricketiq_player_id,cricsheet_name,crickrida_name,canonical_player_name,match_status,source
2848,CS_dcce6f09,None,None,DA Warner,source_only,Cricsheet
3319,CR_davidwarner,None,David Warner,David Warner,source_only,Crickrida


In [166]:
# Cricsheet appearance history for DA Warner
da_warner_history = cricsheet_appearances[
    cricsheet_appearances["source_player_id"].astype(str) == "dcce6f09"
].sort_values("date")

# Crickrida appearance history for David Warner
david_warner_history = crickrida_appearances[
    crickrida_appearances["source_player_id"].astype(str)
    == "CR_davidwarner"
].sort_values("date")

print("CRICSHEET — DA WARNER")
print("Records:", len(da_warner_history))
display(da_warner_history)

print("\nCRICKRIDA — DAVID WARNER")
print("Records:", len(david_warner_history))
display(david_warner_history)

print("\nTEAM COMPARISON")
print("Cricsheet teams:",
      sorted(da_warner_history["team"].dropna().unique()))

print("Crickrida teams:",
      sorted(david_warner_history["team"].dropna().unique()))

print("\nDATE RANGES")
print("Cricsheet:",
      da_warner_history["date"].min(),
      "to",
      da_warner_history["date"].max())

print("Crickrida:",
      david_warner_history["date"].min(),
      "to",
      david_warner_history["date"].max())

CRICSHEET — DA WARNER
Records: 155


,date,team,player_name,source_player_id,match_id,role,source
26728,2009-01-18,Australia,DA Warner,dcce6f09,351685,batting,Cricsheet
26744,2009-01-23,Australia,DA Warner,dcce6f09,351686,batting,Cricsheet
26766,2009-01-26,Australia,DA Warner,dcce6f09,351687,batting,Cricsheet
26788,2009-01-30,Australia,DA Warner,dcce6f09,351688,batting,Cricsheet
26798,2009-02-01,Australia,DA Warner,dcce6f09,351689,batting,Cricsheet
...,...,...,...,...,...,...,...
13705,2023-10-28,Australia,DA Warner,dcce6f09,1384418,batting,Cricsheet
13833,2023-11-04,Australia,DA Warner,dcce6f09,1384427,batting,Cricsheet
13939,2023-11-11,Australia,DA Warner,dcce6f09,1384434,batting,Cricsheet
14010,2023-11-16,Australia,DA Warner,dcce6f09,1384438,batting,Cricsheet



CRICKRIDA — DAVID WARNER
Records: 4


,date,team,player_name,source_player_id,match_id,role,source
60,2012-08-25,Australia,David Warner,CR_davidwarner,https://crickrida.com/matches/afghanistan-aust...,batting,Crickrida
307,2015-03-04,Australia,David Warner,CR_davidwarner,https://crickrida.com/matches/afghanistan-aust...,batting,Crickrida
1432,2019-06-01,Australia,David Warner,CR_davidwarner,https://crickrida.com/matches/afghanistan-aust...,batting,Crickrida
2160,2023-11-07,Australia,David Warner,CR_davidwarner,https://crickrida.com/matches/afghanistan-aust...,batting,Crickrida



TEAM COMPARISON
Cricsheet teams: ['Australia']
Crickrida teams: ['Australia']

DATE RANGES
Cricsheet: 2009-01-18 00:00:00 to 2023-11-19 00:00:00
Crickrida: 2012-08-25 00:00:00 to 2023-11-07 00:00:00


In [167]:
david_warner_record = pd.DataFrame([{
    "cricketiq_player_id": "CRIQ_dcce6f09",
    "player_id": "dcce6f09",
    "cricsheet_name": "DA Warner",
    "crickrida_name": "David Warner",
    "normalized_name": "davidwarner",
    "match_status": "verified_alias"
}])

player_identity_map_updated = pd.concat(
    [player_identity_map_updated, david_warner_record],
    ignore_index=True
)

print("UPDATED SHARED IDENTITY MAP")
print("Total identities:", len(player_identity_map_updated))
print("Unique CricketIQ IDs:",
      player_identity_map_updated["cricketiq_player_id"].nunique())
print("Duplicate CricketIQ IDs:",
      player_identity_map_updated["cricketiq_player_id"].duplicated().sum())
print("Duplicate Cricsheet IDs:",
      player_identity_map_updated["player_id"].duplicated().sum())
print("Duplicate Crickrida names:",
      player_identity_map_updated["crickrida_name"].duplicated().sum())

display(
    player_identity_map_updated[
        player_identity_map_updated["crickrida_name"] == "David Warner"
    ]
)

UPDATED SHARED IDENTITY MAP
Total identities: 122
Unique CricketIQ IDs: 122
Duplicate CricketIQ IDs: 0
Duplicate Cricsheet IDs: 0
Duplicate Crickrida names: 0


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,normalized_name,match_status
121,CRIQ_dcce6f09,dcce6f09,DA Warner,David Warner,davidwarner,verified_alias


In [168]:
# All opponent players recorded in Afghanistan scorecards
# compared with Cricsheet ODI player records.

# Exclude Afghanistan: we are auditing its opponents.
opponent_crickrida = crickrida_appearances[
    crickrida_appearances["team"] != "Afghanistan"
].copy()

# Keep Cricsheet records for the same opponent countries.
opponent_teams = sorted(opponent_crickrida["team"].dropna().unique())

opponent_cricsheet = cricsheet_appearances[
    cricsheet_appearances["team"].isin(opponent_teams)
].copy()

# Summarise each source by country.
crickrida_summary = (
    opponent_crickrida.groupby("team")
    .agg(
        crickrida_players=("source_player_id", "nunique"),
        crickrida_appearances=("match_id", "count"),
        crickrida_first_date=("date", "min"),
        crickrida_last_date=("date", "max")
    )
)

cricsheet_summary = (
    opponent_cricsheet.groupby("team")
    .agg(
        cricsheet_players=("source_player_id", "nunique"),
        cricsheet_appearances=("match_id", "count"),
        cricsheet_first_date=("date", "min"),
        cricsheet_last_date=("date", "max")
    )
)

opponent_comparison_summary = (
    crickrida_summary
    .join(cricsheet_summary, how="outer")
    .fillna({
        "crickrida_players": 0,
        "crickrida_appearances": 0,
        "cricsheet_players": 0,
        "cricsheet_appearances": 0
    })
    .reset_index()
    .sort_values("team")
)

print("OPPONENT TEAMS:", len(opponent_teams))
print("Crickrida opponent player records:", len(opponent_crickrida))
print("Cricsheet opponent player records:", len(opponent_cricsheet))

display(opponent_comparison_summary)

OPPONENT TEAMS: 11
Crickrida opponent player records: 2223
Cricsheet opponent player records: 62933


,team,crickrida_players,crickrida_appearances,crickrida_first_date,crickrida_last_date,cricsheet_players,cricsheet_appearances,cricsheet_first_date,cricsheet_last_date
0,Australia,29,66,2012-08-25,2025-02-28,127,6892,2003-01-09,2026-09-30
1,Bangladesh,51,363,2014-03-01,2026-10-21,109,5023,2003-02-11,2026-10-17
2,England,28,57,2015-03-13,2025-02-26,133,6526,2002-06-27,2026-10-25
3,India,36,88,2014-03-05,2026-06-20,137,7325,2002-06-29,2026-10-03
4,Ireland,45,566,2010-07-03,2026-08-15,68,2044,2006-06-13,2026-10-24
5,New Zealand,23,36,2015-03-08,2023-10-18,118,5746,2002-12-29,2026-07-21
6,Pakistan,39,120,2012-02-10,2023-10-23,147,6174,2003-02-11,2026-10-25
7,South Africa,26,79,2019-06-15,2025-02-21,122,5482,2003-02-09,2026-09-30
8,Sri Lanka,40,209,2014-03-03,2024-02-14,133,7313,2002-06-27,2026-10-22
9,West Indies,27,120,2017-06-09,2019-11-11,142,5907,2003-02-09,2026-10-03


In [169]:
# Build distinct player lists for both sources.

crickrida_players_all = (
    opponent_crickrida[
        ["team", "player_name", "source_player_id"]
    ]
    .drop_duplicates()
    .copy()
)

cricsheet_players_all = (
    opponent_cricsheet[
        ["team", "player_name", "source_player_id"]
    ]
    .drop_duplicates()
    .copy()
)

# Normalize names for comparison while preserving original names.
crickrida_players_all["normalized_name"] = (
    crickrida_players_all["player_name"].apply(normalize_player_name)
)

cricsheet_players_all["normalized_name"] = (
    cricsheet_players_all["player_name"].apply(normalize_player_name)
)

# Exact matches within the same country.
exact_country_matches = crickrida_players_all.merge(
    cricsheet_players_all,
    on=["team", "normalized_name"],
    how="inner",
    suffixes=("_crickrida", "_cricsheet")
)

# Names without an exact normalized match in the other source.
crickrida_exact_unmatched = (
    crickrida_players_all.merge(
        cricsheet_players_all[
            ["team", "normalized_name"]
        ].drop_duplicates(),
        on=["team", "normalized_name"],
        how="left",
        indicator=True
    )
    .query("_merge == 'left_only'")
    .drop(columns="_merge")
)

cricsheet_exact_unmatched = (
    cricsheet_players_all.merge(
        crickrida_players_all[
            ["team", "normalized_name"]
        ].drop_duplicates(),
        on=["team", "normalized_name"],
        how="left",
        indicator=True
    )
    .query("_merge == 'left_only'")
    .drop(columns="_merge")
)

print("DISTINCT PLAYER RECORDS")
print("Crickrida:", len(crickrida_players_all))
print("Cricsheet:", len(cricsheet_players_all))

print("\nEXACT NORMALIZED MATCHES")
print("Matched name pairs:", len(exact_country_matches))
print("Unique Crickrida names matched:",
      exact_country_matches["player_name_crickrida"].nunique())
print("Unique Cricsheet names matched:",
      exact_country_matches["player_name_cricsheet"].nunique())

print("\nUNMATCHED NAMES")
print("Crickrida unmatched:", len(crickrida_exact_unmatched))
print("Cricsheet unmatched:", len(cricsheet_exact_unmatched))

print("\nEXACT MATCHES BY COUNTRY")
display(
    exact_country_matches.groupby("team")
    .agg(
        matched_pairs=("normalized_name", "size"),
        crickrida_names=("player_name_crickrida", "nunique"),
        cricsheet_names=("player_name_cricsheet", "nunique")
    )
    .reset_index()
)

print("\nPOSSIBLE AMBIGUOUS EXACT MATCHES")
display(
    exact_country_matches.groupby(
        ["team", "normalized_name"]
    ).agg(
        crickrida_names=("player_name_crickrida", "nunique"),
        cricsheet_ids=("source_player_id_cricsheet", "nunique"),
        cricsheet_names=("player_name_cricsheet", "nunique")
    )
    .query("cricsheet_ids > 1 or crickrida_names > 1")
    .reset_index()
)

DISTINCT PLAYER RECORDS
Crickrida: 396
Cricsheet: 1354

EXACT NORMALIZED MATCHES
Matched name pairs: 96
Unique Crickrida names matched: 96
Unique Cricsheet names matched: 96

UNMATCHED NAMES
Crickrida unmatched: 300
Cricsheet unmatched: 1258

EXACT MATCHES BY COUNTRY


,team,matched_pairs,crickrida_names,cricsheet_names
0,Bangladesh,45,45,45
1,India,11,11,11
2,Ireland,1,1,1
3,Pakistan,37,37,37
4,South Africa,1,1,1
5,Zimbabwe,1,1,1



POSSIBLE AMBIGUOUS EXACT MATCHES


,team,normalized_name,crickrida_names,cricsheet_ids,cricsheet_names


In [171]:
print("CRICKRIDA COLUMNS")
print(opponent_crickrida.columns.tolist())

print("\nCRICSHEET COLUMNS")
print(opponent_cricsheet.columns.tolist())

print("\nCRICKRIDA SAMPLE")
display(opponent_crickrida.head(3))

print("\nCRICSHEET SAMPLE")
display(opponent_cricsheet.head(3))

CRICKRIDA COLUMNS
['date', 'team', 'player_name', 'source_player_id', 'match_id', 'role', 'source']

CRICSHEET COLUMNS
['date', 'team', 'player_name', 'source_player_id', 'match_id', 'role', 'source']

CRICKRIDA SAMPLE


,date,team,player_name,source_player_id,match_id,role,source
0,2010-07-03,Ireland,Paul Stirling,CR_paulstirling,https://crickrida.com/matches/afghanistan-irel...,batting,Crickrida
1,2010-07-03,Ireland,James Hall,CR_jameshall,https://crickrida.com/matches/afghanistan-irel...,batting,Crickrida
2,2010-07-03,Ireland,Alex Cusack,CR_alexcusack,https://crickrida.com/matches/afghanistan-irel...,batting,Crickrida



CRICSHEET SAMPLE


,date,team,player_name,source_player_id,match_id,role,source
0,2017-01-13,Australia,B Stanlake,6834d1f2,1000887,batting,Cricsheet
1,2017-01-13,Australia,CA Lynn,45eda7c8,1000887,batting,Cricsheet
2,2017-01-13,Australia,DA Warner,dcce6f09,1000887,batting,Cricsheet


In [175]:
from difflib import SequenceMatcher

# 1. Build appearance histories.
crickrida_history = (
    opponent_crickrida.groupby(
        ["team", "player_name", "source_player_id"]
    )
    .agg(
        crickrida_first_date=("date", "min"),
        crickrida_last_date=("date", "max"),
        crickrida_matches=("match_id", "nunique")
    )
    .reset_index()
)

cricsheet_history = (
    opponent_cricsheet.groupby(
        ["team", "player_name", "source_player_id"]
    )
    .agg(
        cricsheet_first_date=("date", "min"),
        cricsheet_last_date=("date", "max"),
        cricsheet_matches=("match_id", "nunique")
    )
    .reset_index()
)

# 2. Get unmatched names from both sources.
cr_names = crickrida_exact_unmatched[
    ["team", "player_name", "source_player_id", "normalized_name"]
].drop_duplicates()

cs_names = cricsheet_exact_unmatched[
    ["team", "player_name", "source_player_id", "normalized_name"]
].drop_duplicates()

# 3. Generate candidates within the same country.
candidate_rows = []

for team in opponent_teams:

    cr_team = cr_names[cr_names["team"] == team]
    cs_team = cs_names[cs_names["team"] == team]

    for _, cr in cr_team.iterrows():

        for _, cs in cs_team.iterrows():

            similarity = SequenceMatcher(
                None,
                cr["normalized_name"],
                cs["normalized_name"]
            ).ratio()

            if similarity >= 0.55:
                candidate_rows.append({
                    "team": team,
                    "crickrida_name": cr["player_name"],
                    "crickrida_id": cr["source_player_id"],
                    "cricsheet_name": cs["player_name"],
                    "cricsheet_id": cs["source_player_id"],
                    "similarity": round(similarity, 3)
                })

# 4. Create candidate DataFrame.
candidate_review_all = pd.DataFrame(candidate_rows)

# 5. Attach Crickrida appearance history.
candidate_review_all = candidate_review_all.merge(
    crickrida_history,
    left_on=["team", "crickrida_name", "crickrida_id"],
    right_on=["team", "player_name", "source_player_id"],
    how="left"
).drop(columns=["player_name", "source_player_id"])

# 6. Attach Cricsheet appearance history.
candidate_review_all = candidate_review_all.merge(
    cricsheet_history,
    left_on=["team", "cricsheet_name", "cricsheet_id"],
    right_on=["team", "player_name", "source_player_id"],
    how="left"
).drop(columns=["player_name", "source_player_id"])

# 7. Calculate preliminary date gap.
candidate_review_all["date_gap_days"] = (
    pd.to_datetime(candidate_review_all["crickrida_first_date"])
    - pd.to_datetime(candidate_review_all["cricsheet_last_date"])
).dt.days.abs()

# 8. Display summary.
print("Candidate pairs:", len(candidate_review_all))
print(
    "Crickrida players with candidates:",
    candidate_review_all["crickrida_name"].nunique()
)

print("\nSimilarity distribution:")
display(
    candidate_review_all["similarity"].describe()
)

print("\nTop 60 candidates:")
display(
    candidate_review_all.sort_values(
        ["team", "crickrida_name", "similarity"],
        ascending=[True, True, False]
    ).head(60)
)

Candidate pairs: 599
Crickrida players with candidates: 295

Similarity distribution:


count    599.000000
mean       0.689290
std        0.096682
min        0.552000
25%        0.600000
50%        0.696000
75%        0.762000
max        0.970000
Name: similarity, dtype: float64


Top 60 candidates:


,team,crickrida_name,crickrida_id,cricsheet_name,cricsheet_id,similarity,crickrida_first_date,crickrida_last_date,crickrida_matches,cricsheet_first_date,cricsheet_last_date,cricsheet_matches,date_gap_days
20,Australia,Aaron Finch,CR_aaronfinch,AJ Finch,b8d490fd,0.706,2015-03-04,2019-06-01,2,2013-01-11,2022-09-11,139,2748
21,Australia,Aaron Finch,CR_aaronfinch,L Ronchi,0fa5042b,0.588,2015-03-04,2019-06-01,2,2008-07-04,2008-07-06,2,2432
43,Australia,Adam Zampa,CR_adamzampa,A Zampa,14f96089,0.800,2019-06-01,2025-02-28,3,2016-02-06,2026-09-30,124,2678
46,Australia,Ben Dwarshuis,CR_bendwarshuis,BJ Dwarshuis,e1b9f3a9,0.870,2025-02-28,2025-02-28,1,2024-09-19,2026-06-14,6,471
47,Australia,Ben Dwarshuis,CR_bendwarshuis,J Edwards,f787e0eb,0.600,2025-02-28,2025-02-28,1,2026-09-24,2026-09-27,2,576
27,Australia,Brad Haddin,CR_bradhaddin,BJ Haddin,b0c772ee,0.778,2015-03-04,2015-03-04,1,2004-01-29,2015-03-26,107,22
26,Australia,Brad Haddin,CR_bradhaddin,AM Hardie,d405c0d5,0.556,2015-03-04,2015-03-04,1,2023-09-09,2025-08-22,14,3824
12,Australia,David Hussey,CR_davidhussey,DJ Hussey,fd835ab3,0.737,2012-08-25,2012-08-25,1,2008-07-04,2013-01-23,62,151
11,Australia,David Hussey,CR_davidhussey,MEK Hussey,48fd7349,0.600,2012-08-25,2012-08-25,1,2004-02-01,2012-09-03,148,9
10,Australia,David Hussey,CR_davidhussey,TH David,f1f99156,0.556,2012-08-25,2012-08-25,1,2023-09-09,2023-09-17,4,4040


In [176]:
# Identify already-approved source name pairs.
approved_pairs = set(
    zip(
        player_identity_map_updated["crickrida_name"],
        player_identity_map_updated["cricsheet_name"]
    )
)

strong_candidates = candidate_review_all[
    candidate_review_all["similarity"] >= 0.70
].copy()

strong_candidates["already_approved"] = strong_candidates.apply(
    lambda row: (
        row["crickrida_name"],
        row["cricsheet_name"]
    ) in approved_pairs,
    axis=1
)

# Remove previously approved aliases from the new review list.
new_strong_candidates = strong_candidates[
    ~strong_candidates["already_approved"]
].copy()

# Calculate whether the two appearance-history intervals overlap.
new_strong_candidates["career_overlap"] = (
    pd.to_datetime(new_strong_candidates["crickrida_first_date"])
    <= pd.to_datetime(new_strong_candidates["cricsheet_last_date"])
) & (
    pd.to_datetime(new_strong_candidates["crickrida_last_date"])
    >= pd.to_datetime(new_strong_candidates["cricsheet_first_date"])
)

print("Strong candidate pairs:", len(strong_candidates))
print("Previously approved pairs:", strong_candidates["already_approved"].sum())
print("New candidates for review:", len(new_strong_candidates))

print("\nNEW STRONG CANDIDATES")
display(
    new_strong_candidates[
        [
            "team",
            "crickrida_name",
            "cricsheet_name",
            "cricsheet_id",
            "similarity",
            "crickrida_first_date",
            "crickrida_last_date",
            "cricsheet_first_date",
            "cricsheet_last_date",
            "career_overlap"
        ]
    ].sort_values(
        ["team", "crickrida_name", "similarity"],
        ascending=[True, True, False]
    )
)

Strong candidate pairs: 294
Previously approved pairs: 24
New candidates for review: 270

NEW STRONG CANDIDATES


,team,crickrida_name,cricsheet_name,cricsheet_id,similarity,crickrida_first_date,crickrida_last_date,cricsheet_first_date,cricsheet_last_date,career_overlap
20,Australia,Aaron Finch,AJ Finch,b8d490fd,0.706,2015-03-04,2019-06-01,2013-01-11,2022-09-11,True
43,Australia,Adam Zampa,A Zampa,14f96089,0.800,2019-06-01,2025-02-28,2016-02-06,2026-09-30,True
27,Australia,Brad Haddin,BJ Haddin,b0c772ee,0.778,2015-03-04,2015-03-04,2004-01-29,2015-03-26,True
12,Australia,David Hussey,DJ Hussey,fd835ab3,0.737,2012-08-25,2012-08-25,2008-07-04,2013-01-23,True
13,Australia,George Bailey,GJ Bailey,5056011d,0.700,2012-08-25,2012-08-25,2012-03-16,2016-12-09,True
...,...,...,...,...,...,...,...,...,...,...
552,Zimbabwe,Tinashe Panyangara,T Panyangara,3b417422,0.786,2014-07-18,2015-10-24,2004-04-20,2016-11-14,True
557,Zimbabwe,Tinotenda Mutombodzi,CT Mutombodzi,26f12506,0.710,2015-10-16,2015-10-24,2013-02-26,2020-03-06,True
593,Zimbabwe,Trevor Gwandu,T Gwandu,d1d83db3,0.737,2024-12-19,2024-12-21,2024-11-24,2025-08-29,True
532,Zimbabwe,Vusi Sibanda,V Sibanda,0fdabe05,0.842,2014-07-18,2014-07-18,2003-11-23,2016-06-15,True


In [177]:
# Build detailed appearance-date histories for each candidate player.

cr_dates = (
    opponent_crickrida.groupby(
        ["team", "player_name", "source_player_id"]
    )
    .agg(
        cr_dates=("date", lambda x: set(pd.to_datetime(x).dt.normalize())),
        cr_batting=("role", lambda x: (x == "batting").any()),
        cr_bowling=("role", lambda x: (x == "bowling").any())
    )
    .reset_index()
)

cs_dates = (
    opponent_cricsheet.groupby(
        ["team", "player_name", "source_player_id"]
    )
    .agg(
        cs_dates=("date", lambda x: set(pd.to_datetime(x).dt.normalize())),
        cs_batting=("role", lambda x: (x == "batting").any()),
        cs_bowling=("role", lambda x: (x == "bowling").any())
    )
    .reset_index()
)

# Attach date sets and role information to candidates.
date_review = new_strong_candidates.merge(
    cr_dates,
    left_on=["team", "crickrida_name", "crickrida_id"],
    right_on=["team", "player_name", "source_player_id"],
    how="left"
).drop(columns=["player_name", "source_player_id"])

date_review = date_review.merge(
    cs_dates,
    left_on=["team", "cricsheet_name", "cricsheet_id"],
    right_on=["team", "player_name", "source_player_id"],
    how="left"
).drop(columns=["player_name", "source_player_id"])

# Compare every Crickrida appearance date with every Cricsheet appearance date.
def compare_appearance_dates(row):
    cr = row["cr_dates"]
    cs = row["cs_dates"]

    if not isinstance(cr, set) or not isinstance(cs, set):
        return pd.Series({
            "exact_date_matches": 0,
            "within_7_days": 0,
            "within_30_days": 0,
            "within_90_days": 0,
            "within_365_days": 0,
            "nearest_date_gap": None
        })

    gaps = [
        abs((d1 - d2).days)
        for d1 in cr
        for d2 in cs
    ]

    return pd.Series({
        "exact_date_matches": sum(g == 0 for g in gaps),
        "within_7_days": sum(g <= 7 for g in gaps),
        "within_30_days": sum(g <= 30 for g in gaps),
        "within_90_days": sum(g <= 90 for g in gaps),
        "within_365_days": sum(g <= 365 for g in gaps),
        "nearest_date_gap": min(gaps) if gaps else None
    })

date_metrics = date_review.apply(compare_appearance_dates, axis=1)

date_review = pd.concat(
    [date_review.reset_index(drop=True), date_metrics],
    axis=1
)

# Role compatibility is supporting evidence, not proof.
date_review["role_compatible"] = (
    (date_review["cr_batting"] & date_review["cs_batting"]) |
    (date_review["cr_bowling"] & date_review["cs_bowling"])
)

print("CANDIDATES ANALYSED:", len(date_review))

print("\nDATE PROXIMITY SUMMARY")
display(
    date_review[
        [
            "exact_date_matches",
            "within_7_days",
            "within_30_days",
            "within_90_days",
            "within_365_days"
        ]
    ].sum().to_frame("candidate_date_pair_count")
)

print("\nCLOSEST DATE MATCHES")
display(
    date_review.sort_values(
        ["exact_date_matches", "within_7_days",
         "within_30_days", "similarity"],
        ascending=False
    )[
        [
            "team",
            "crickrida_name",
            "cricsheet_name",
            "cricsheet_id",
            "similarity",
            "exact_date_matches",
            "within_7_days",
            "within_30_days",
            "nearest_date_gap",
            "role_compatible"
        ]
    ].head(60)
)

CANDIDATES ANALYSED: 270

DATE PROXIMITY SUMMARY


,candidate_date_pair_count
exact_date_matches,0
within_7_days,506
within_30_days,2242
within_90_days,4884
within_365_days,14068



CLOSEST DATE MATCHES


,team,crickrida_name,cricsheet_name,cricsheet_id,similarity,exact_date_matches,within_7_days,within_30_days,nearest_date_gap,role_compatible
158,Sri Lanka,Angelo Mathews,AD Mathews,896d78ad,0.727,0,11,24,2,True
66,Ireland,Paul Stirling,PR Stirling,cf494ffe,0.818,0,9,40,2,True
68,Ireland,Kevin O'Brien,KJ O'Brien,243431b5,0.737,0,9,31,2,True
162,Sri Lanka,Kusal Mendis,BKG Mendis,5d1e7582,0.700,0,8,40,2,True
206,West Indies,Jason Holder,JO Holder,0f721006,0.842,0,8,26,3,True
202,West Indies,Shai Hope,SD Hope,1fc6ef83,0.714,0,8,24,3,True
57,India,Ravindra Jadeja,RA Jadeja,fe93fd9d,0.727,0,8,20,2,True
212,West Indies,Shimron Hetmyer,SO Hetmyer,48a1d7b7,0.783,0,8,16,3,True
0,Australia,Glenn Maxwell,GJ Maxwell,b681e71e,0.762,0,7,28,3,True
50,India,Rohit Sharma,RG Sharma,740742ef,0.737,0,7,26,3,True


In [178]:
# Focus on candidates with similarity >= 0.70.
review_candidates = date_review[
    date_review["similarity"] >= 0.70
].copy()

# Count possible Cricsheet identities per Crickrida player.
crickrida_candidate_counts = (
    review_candidates.groupby(
        ["team", "crickrida_name", "crickrida_id"]
    )
    .agg(
        candidate_count=("cricsheet_id", "nunique"),
        candidate_names=("cricsheet_name", lambda x: list(x)),
        candidate_ids=("cricsheet_id", lambda x: list(x))
    )
    .reset_index()
)

# Count Crickrida players proposed for each Cricsheet ID.
cricsheet_candidate_counts = (
    review_candidates.groupby(
        ["team", "cricsheet_name", "cricsheet_id"]
    )
    .agg(
        crickrida_candidate_count=("crickrida_id", "nunique"),
        crickrida_names=("crickrida_name", lambda x: list(x))
    )
    .reset_index()
)

ambiguous_crickrida = crickrida_candidate_counts[
    crickrida_candidate_counts["candidate_count"] > 1
]

ambiguous_cricsheet = cricsheet_candidate_counts[
    cricsheet_candidate_counts["crickrida_candidate_count"] > 1
]

# Candidate pairs that have a one-to-one relationship in this shortlist.
one_to_one_candidates = review_candidates.merge(
    crickrida_candidate_counts[
        ["team", "crickrida_id", "candidate_count"]
    ],
    on=["team", "crickrida_id"],
    how="left"
).merge(
    cricsheet_candidate_counts[
        ["team", "cricsheet_id", "crickrida_candidate_count"]
    ],
    on=["team", "cricsheet_id"],
    how="left"
)

one_to_one_candidates = one_to_one_candidates[
    (one_to_one_candidates["candidate_count"] == 1) &
    (one_to_one_candidates["crickrida_candidate_count"] == 1)
]

print("STRONG CANDIDATE PAIRS:", len(review_candidates))
print("AMBIGUOUS CRICKRIDA PLAYERS:", len(ambiguous_crickrida))
print("AMBIGUOUS CRICSHEET IDENTITIES:", len(ambiguous_cricsheet))
print("ONE-TO-ONE CANDIDATE PAIRS:", len(one_to_one_candidates))

print("\nCRICKRIDA PLAYERS WITH MULTIPLE CANDIDATES")
display(ambiguous_crickrida)

print("\nCRICSHEET IDENTITIES PROPOSED FOR MULTIPLE PLAYERS")
display(ambiguous_cricsheet)

print("\nONE-TO-ONE CANDIDATES")
display(
    one_to_one_candidates[
        [
            "team",
            "crickrida_name",
            "cricsheet_name",
            "cricsheet_id",
            "similarity",
            "nearest_date_gap",
            "role_compatible"
        ]
    ].sort_values(
        ["team", "crickrida_name"]
    )
)

STRONG CANDIDATE PAIRS: 270
AMBIGUOUS CRICKRIDA PLAYERS: 32
AMBIGUOUS CRICSHEET IDENTITIES: 15
ONE-TO-ONE CANDIDATE PAIRS: 190

CRICKRIDA PLAYERS WITH MULTIPLE CANDIDATES


,team,crickrida_name,crickrida_id,candidate_count,candidate_names,candidate_ids
19,Bangladesh,Al Islam,CR_alislam,3,"[Sunzamul Islam, Jahurul Islam, Manjural Islam]","[0f348f58, 6752c57b, 2cfa6534]"
25,England,Ben Stokes,CR_benstokes,2,"[BA Stokes, BT Foakes]","[e087956b, fd770945]"
35,England,Jamie Overton,CR_jamieoverton,2,"[C Overton, J Overton]","[e9987a94, 59559bc2]"
46,India,Deepak Chahar,CR_deepakchahar,2,"[DL Chahar, RD Chahar]","[23eeb873, 2ed569a0]"
48,India,Hardik Pandya,CR_hardikpandya,2,"[HH Pandya, KH Pandya]","[dbe50b21, 5b8c830e]"
52,India,Rohit Sharma,CR_rohitsharma,3,"[RG Sharma, I Sharma, R Sharma]","[740742ef, 5bb1a1c4, 5d9a1a73]"
53,India,Shikhar Dhawan,CR_shikhardhawan,2,"[S Dhawan, R Dhawan]","[0a476045, 26a85969]"
58,Ireland,Andrew Poynter,CR_andrewpoynter,2,"[SW Poynter, AD Poynter]","[cdf59953, b98cdbf0]"
61,Ireland,Barry McCarthy,CR_barrymccarthy,2,"[BJ McCarthy, L McCarthy]","[3766056b, 8eb2e2e6]"
76,Ireland,John Mooney,CR_johnmooney,2,"[JF Mooney, PJK Mooney]","[aae9ce90, 94edecfe]"



CRICSHEET IDENTITIES PROPOSED FOR MULTIPLE PLAYERS


,team,cricsheet_name,cricsheet_id,crickrida_candidate_count,crickrida_names
66,Ireland,AD Poynter,b98cdbf0,2,"[Andrew Poynter, Stuart Poynter]"
71,Ireland,BJ McCarthy,3766056b,2,"[Barry McCarthy, Liam McCarthy]"
89,Ireland,L McCarthy,8eb2e2e6,2,"[Barry McCarthy, Liam McCarthy]"
92,Ireland,NJ O'Brien,84826f48,2,"[Kevin O'Brien, Niall O'Brien]"
98,Ireland,SW Poynter,cdf59953,2,"[Andrew Poynter, Stuart Poynter]"
149,Sri Lanka,AM Fernando,de3d549a,2,"[Avishka Fernando, Asitha Fernando]"
154,Sri Lanka,D Madushanka,de7d833e,3,"[Dasun Shanaka, Dilshan Madushanka, Pramod Mad..."
157,Sri Lanka,DSK Madushanka,248698f4,2,"[Dasun Shanaka, Dilshan Madushanka]"
169,Sri Lanka,LD Madushanka,de063a39,2,"[Dasun Shanaka, Dilshan Madushanka]"
184,Sri Lanka,WIA Fernando,9550470d,2,"[Avishka Fernando, Asitha Fernando]"



ONE-TO-ONE CANDIDATES


,team,crickrida_name,cricsheet_name,cricsheet_id,similarity,nearest_date_gap,role_compatible
5,Australia,Aaron Finch,AJ Finch,b8d490fd,0.706,4,True
17,Australia,Adam Zampa,A Zampa,14f96089,0.800,3,True
7,Australia,Brad Haddin,BJ Haddin,b0c772ee,0.778,4,True
2,Australia,David Hussey,DJ Hussey,fd835ab3,0.737,3,True
3,Australia,George Bailey,GJ Bailey,5056011d,0.700,3,True
...,...,...,...,...,...,...,...
237,Zimbabwe,Tinashe Panyangara,T Panyangara,3b417422,0.786,5,True
241,Zimbabwe,Tinotenda Mutombodzi,CT Mutombodzi,26f12506,0.710,3,True
265,Zimbabwe,Trevor Gwandu,T Gwandu,d1d83db3,0.737,23,True
225,Zimbabwe,Vusi Sibanda,V Sibanda,0fdabe05,0.842,30,True


In [179]:
# 1. Collect all Cricsheet IDs involved in ambiguous matches.
ambiguous_ids = set(
    ambiguous_cricsheet["cricsheet_id"].dropna()
)

# 2. Inspect the full registry records for those IDs.
registry_columns = [
    col for col in [
        "player_id",
        "cricsheet_name",
        "player_names",
        "cricketiq_player_id",
        "match_status"
    ]
    if col in player_registry_final_updated.columns
]

ambiguous_registry = (
    player_registry_final_updated[
        player_registry_final_updated["player_id"].isin(ambiguous_ids)
    ][registry_columns]
    .drop_duplicates()
    .sort_values("player_id")
)

print("AMBIGUOUS CRICSHEET IDs:", len(ambiguous_ids))
print("REGISTRY RECORDS:", len(ambiguous_registry))

print("\nCRICSHEET REGISTRY DETAILS")
display(ambiguous_registry)

# 3. Show every candidate relationship for those IDs.
print("\nALL AMBIGUOUS CANDIDATE RELATIONSHIPS")

display(
    review_candidates[
        review_candidates["cricsheet_id"].isin(ambiguous_ids)
    ][
        [
            "team",
            "crickrida_name",
            "crickrida_id",
            "cricsheet_name",
            "cricsheet_id",
            "similarity",
            "nearest_date_gap"
        ]
    ].sort_values(
        ["team", "cricsheet_id", "similarity"],
        ascending=[True, True, False]
    )
)

TypeError: unhashable type: 'list'

In [180]:
# Collect all Cricsheet IDs involved in ambiguous matches.
ambiguous_ids = set(
    ambiguous_cricsheet["cricsheet_id"].dropna()
)

# Select available registry columns.
registry_columns = [
    col for col in [
        "player_id",
        "cricsheet_name",
        "player_names",
        "cricketiq_player_id",
        "match_status"
    ]
    if col in player_registry_final_updated.columns
]

# Remove duplicates using scalar columns only.
dedup_columns = [
    col for col in [
        "player_id",
        "cricsheet_name",
        "cricketiq_player_id",
        "match_status"
    ]
    if col in registry_columns
]

ambiguous_registry = (
    player_registry_final_updated[
        player_registry_final_updated["player_id"].isin(ambiguous_ids)
    ][registry_columns]
    .drop_duplicates(subset=dedup_columns)
    .sort_values("player_id")
)

print("AMBIGUOUS CRICSHEET IDs:", len(ambiguous_ids))
print("REGISTRY RECORDS:", len(ambiguous_registry))

print("\nCRICSHEET REGISTRY DETAILS")
display(ambiguous_registry)

print("\nALL AMBIGUOUS CANDIDATE RELATIONSHIPS")

display(
    review_candidates[
        review_candidates["cricsheet_id"].isin(ambiguous_ids)
    ][
        [
            "team",
            "crickrida_name",
            "crickrida_id",
            "cricsheet_name",
            "cricsheet_id",
            "similarity",
            "nearest_date_gap"
        ]
    ].sort_values(
        ["team", "cricsheet_id", "similarity"],
        ascending=[True, True, False]
    )
)

AMBIGUOUS CRICSHEET IDs: 15
REGISTRY RECORDS: 15

CRICSHEET REGISTRY DETAILS


,player_id,cricsheet_name,player_names,cricketiq_player_id,match_status
526,248698f4,None,[DSK Madushanka],CS_248698f4,source_only
693,31fbf891,None,[H Masakadza],CS_31fbf891,source_only
721,33a50ecf,None,[RL Powell],CS_33a50ecf,source_only
772,3766056b,None,[BJ McCarthy],CS_3766056b,source_only
933,449f4e3c,None,[B Muzarabani],CS_449f4e3c,source_only
1353,650d5e49,None,[R Powell],CS_650d5e49,source_only
1741,84826f48,None,[NJ O'Brien],CS_84826f48,source_only
1876,8eb2e2e6,None,[L McCarthy],CS_8eb2e2e6,source_only
1957,9550470d,None,[WIA Fernando],CS_9550470d,source_only
2420,b98cdbf0,None,[AD Poynter],CS_b98cdbf0,source_only



ALL AMBIGUOUS CANDIDATE RELATIONSHIPS


,team,crickrida_name,crickrida_id,cricsheet_name,cricsheet_id,similarity,nearest_date_gap
89,Ireland,Barry McCarthy,CR_barrymccarthy,BJ McCarthy,3766056b,0.783,3
105,Ireland,Liam McCarthy,CR_liammccarthy,BJ McCarthy,3766056b,0.727,444
69,Ireland,Kevin O'Brien,CR_kevinobrien,NJ O'Brien,84826f48,0.737,2
80,Ireland,Niall O'Brien,CR_niallobrien,NJ O'Brien,84826f48,0.737,2
106,Ireland,Liam McCarthy,CR_liammccarthy,L McCarthy,8eb2e2e6,0.857,68
90,Ireland,Barry McCarthy,CR_barrymccarthy,L McCarthy,8eb2e2e6,0.727,435
71,Ireland,Andrew Poynter,CR_andrewpoynter,AD Poynter,b98cdbf0,0.818,4
87,Ireland,Stuart Poynter,CR_stuartpoynter,AD Poynter,b98cdbf0,0.727,669
70,Ireland,Andrew Poynter,CR_andrewpoynter,SW Poynter,cdf59953,0.727,1530
86,Ireland,Stuart Poynter,CR_stuartpoynter,SW Poynter,cdf59953,0.727,24


In [181]:
# Focus on the ambiguous Crickrida players.
ambiguous_crickrida_ids = set(
    ambiguous_crickrida["crickrida_id"]
)

# Get all Crickrida appearances for those players.
crickrida_conflict_history = (
    opponent_crickrida[
        opponent_crickrida["source_player_id"].isin(
            ambiguous_crickrida_ids
        )
    ][
        ["team", "date", "player_name", "source_player_id",
         "role", "match_id"]
    ]
    .drop_duplicates()
    .sort_values(["team", "player_name", "date"])
)

# Get all Cricsheet appearances for the ambiguous IDs.
cricsheet_conflict_history = (
    opponent_cricsheet[
        opponent_cricsheet["source_player_id"].isin(ambiguous_ids)
    ][
        ["team", "date", "player_name", "source_player_id",
         "role", "match_id"]
    ]
    .drop_duplicates()
    .sort_values(["team", "player_name", "date"])
)

print("CRICKRIDA AMBIGUOUS PLAYER APPEARANCES")
display(crickrida_conflict_history)

print("\nCRICSHEET AMBIGUOUS PLAYER APPEARANCES")
display(cricsheet_conflict_history)

CRICKRIDA AMBIGUOUS PLAYER APPEARANCES


,team,date,player_name,source_player_id,role,match_id
2695,Bangladesh,2026-10-21,Al Islam,CR_alislam,batting,https://crickrida.com/matches/afghanistan-bang...
4468,Bangladesh,2026-10-21,Al Islam,CR_alislam,bowling,https://crickrida.com/matches/afghanistan-bang...
1493,England,2019-06-18,Ben Stokes,CR_benstokes,batting,https://crickrida.com/matches/afghanistan-engl...
3673,England,2019-06-18,Ben Stokes,CR_benstokes,bowling,https://crickrida.com/matches/afghanistan-engl...
2456,England,2025-02-26,Jamie Overton,CR_jamieoverton,batting,https://crickrida.com/matches/afghanistan-engl...
...,...,...,...,...,...,...
222,Zimbabwe,2014-07-24,Shingi Masakadza,CR_shingimasakadza,batting,https://crickrida.com/matches/afghanistan-zimb...
2866,Zimbabwe,2014-07-24,Shingi Masakadza,CR_shingimasakadza,bowling,https://crickrida.com/matches/afghanistan-zimb...
466,Zimbabwe,2015-12-25,Taurai Muzarabani,CR_tauraimuzarabani,batting,https://crickrida.com/matches/afghanistan-zimb...
3033,Zimbabwe,2015-12-25,Taurai Muzarabani,CR_tauraimuzarabani,bowling,https://crickrida.com/matches/afghanistan-zimb...



CRICSHEET AMBIGUOUS PLAYER APPEARANCES


,team,date,player_name,source_player_id,role,match_id
30807,Ireland,2010-07-07,AD Poynter,b98cdbf0,batting,450379
31434,Ireland,2010-09-06,AD Poynter,b98cdbf0,batting,464723
41716,Ireland,2014-02-23,AD Poynter,b98cdbf0,batting,702139
42036,Ireland,2014-09-08,AD Poynter,b98cdbf0,batting,727935
42052,Ireland,2014-09-10,AD Poynter,b98cdbf0,batting,727937
...,...,...,...,...,...,...
6056,Zimbabwe,2019-06-21,H Masakadza,31fbf891,batting,1188378
5484,Zimbabwe,2019-07-01,H Masakadza,31fbf891,batting,1168517
5514,Zimbabwe,2019-07-04,H Masakadza,31fbf891,batting,1168518
5525,Zimbabwe,2019-07-07,H Masakadza,31fbf891,batting,1168519


In [183]:
# Build date sets for each player identity and country.

cr_date_sets = (
    opponent_crickrida
    .groupby(["team", "player_name", "source_player_id"])["date"]
    .agg(lambda x: set(pd.to_datetime(x).dt.normalize()))
    .to_dict()
)

cs_date_sets = (
    opponent_cricsheet
    .groupby(["team", "player_name", "source_player_id"])["date"]
    .agg(lambda x: set(pd.to_datetime(x).dt.normalize()))
    .to_dict()
)

# Calculate the closest date pair for each candidate.
def get_closest_dates(row):
    cr_key = (
        row["team"],
        row["crickrida_name"],
        row["crickrida_id"]
    )

    cs_key = (
        row["team"],
        row["cricsheet_name"],
        row["cricsheet_id"]
    )

    cr_dates = cr_date_sets.get(cr_key, set())
    cs_dates = cs_date_sets.get(cs_key, set())

    if not cr_dates or not cs_dates:
        return pd.Series({
            "closest_crickrida_date": None,
            "closest_cricsheet_date": None,
            "nearest_gap_days": None
        })

    closest = min(
        (
            (abs((d1 - d2).days), d1, d2)
            for d1 in cr_dates
            for d2 in cs_dates
        ),
        key=lambda x: x[0]
    )

    gap, cr_date, cs_date = closest

    return pd.Series({
        "closest_crickrida_date": cr_date.date(),
        "closest_cricsheet_date": cs_date.date(),
        "nearest_gap_days": gap
    })


# Use only ambiguous Crickrida identities.
ambiguous_review = review_candidates[
    review_candidates["crickrida_id"].isin(
        set(ambiguous_crickrida["crickrida_id"])
    )
].copy()

closest_dates = ambiguous_review.apply(
    get_closest_dates,
    axis=1
)

ambiguous_review = pd.concat(
    [
        ambiguous_review.reset_index(drop=True),
        closest_dates.reset_index(drop=True)
    ],
    axis=1
)

ambiguous_review_output = ambiguous_review[
    [
        "team",
        "crickrida_name",
        "crickrida_id",
        "cricsheet_name",
        "cricsheet_id",
        "similarity",
        "closest_crickrida_date",
        "closest_cricsheet_date",
        "nearest_gap_days"
    ]
].sort_values(
    ["team", "crickrida_name", "nearest_gap_days"]
)

print("Ambiguous candidate relationships:", len(ambiguous_review_output))

display(ambiguous_review_output)

Ambiguous candidate relationships: 76


,team,crickrida_name,crickrida_id,cricsheet_name,cricsheet_id,similarity,closest_crickrida_date,closest_cricsheet_date,nearest_gap_days
0,Bangladesh,Al Islam,CR_alislam,Sunzamul Islam,0f348f58,0.700,2026-10-21,2018-01-23,3193
1,Bangladesh,Al Islam,CR_alislam,Jahurul Islam,6752c57b,0.737,2026-10-21,2013-05-08,4914
2,Bangladesh,Al Islam,CR_alislam,Manjural Islam,2cfa6534,0.700,2026-10-21,2003-03-01,8635
3,England,Ben Stokes,CR_benstokes,BA Stokes,e087956b,0.824,2019-06-18,2019-06-21,3
4,England,Ben Stokes,CR_benstokes,BT Foakes,fd770945,0.706,2019-06-18,2019-05-03,46
...,...,...,...,...,...,...,...,...,...
64,West Indies,Shai Hope,CR_shaihope,KA Hope,c59d8f8a,0.714,2017-06-11,2017-06-30,19
72,Zimbabwe,Shingi Masakadza,CR_shingimasakadza,H Masakadza,31fbf891,0.800,2014-07-24,2014-08-17,24
73,Zimbabwe,Shingi Masakadza,CR_shingimasakadza,SW Masakadza,2182013e,0.769,2014-07-24,2014-08-17,24
74,Zimbabwe,Taurai Muzarabani,CR_tauraimuzarabani,T Muzarabani,a945bb28,0.815,2015-12-25,2015-11-11,44


In [184]:
with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", 220,
    "display.max_colwidth", 35
):
    display(
        ambiguous_review_output.reset_index(drop=True)
    )

,team,crickrida_name,crickrida_id,cricsheet_name,cricsheet_id,similarity,closest_crickrida_date,closest_cricsheet_date,nearest_gap_days
0,Bangladesh,Al Islam,CR_alislam,Sunzamul Islam,0f348f58,0.700,2026-10-21,2018-01-23,3193
1,Bangladesh,Al Islam,CR_alislam,Jahurul Islam,6752c57b,0.737,2026-10-21,2013-05-08,4914
2,Bangladesh,Al Islam,CR_alislam,Manjural Islam,2cfa6534,0.700,2026-10-21,2003-03-01,8635
3,England,Ben Stokes,CR_benstokes,BA Stokes,e087956b,0.824,2019-06-18,2019-06-21,3
4,England,Ben Stokes,CR_benstokes,BT Foakes,fd770945,0.706,2019-06-18,2019-05-03,46
5,England,Jamie Overton,CR_jamieoverton,J Overton,59559bc2,0.800,2025-02-26,2025-03-01,3
6,England,Jamie Overton,CR_jamieoverton,C Overton,e9987a94,0.700,2025-02-26,2022-07-17,955
7,India,Deepak Chahar,CR_deepakchahar,DL Chahar,23eeb873,0.700,2018-09-25,2019-12-15,446
8,India,Deepak Chahar,CR_deepakchahar,RD Chahar,2ed569a0,0.700,2018-09-25,2021-07-23,1032
9,India,Hardik Pandya,CR_hardikpandya,HH Pandya,dbe50b21,0.700,2023-10-11,2023-10-08,3


In [185]:
# Aggregate complete appearance histories for each candidate identity.

cr_history_detail = (
    opponent_crickrida
    .groupby(["team", "player_name", "source_player_id"])
    .agg(
        crickrida_dates=("date", lambda x: sorted(
            pd.to_datetime(x).dt.strftime("%Y-%m-%d").unique()
        )),
        crickrida_roles=("role", lambda x: sorted(x.unique())),
        crickrida_match_count=("match_id", "nunique")
    )
    .reset_index()
)

cs_history_detail = (
    opponent_cricsheet
    .groupby(["team", "player_name", "source_player_id"])
    .agg(
        cricsheet_dates=("date", lambda x: sorted(
            pd.to_datetime(x).dt.strftime("%Y-%m-%d").unique()
        )),
        cricsheet_roles=("role", lambda x: sorted(x.unique())),
        cricsheet_match_count=("match_id", "nunique")
    )
    .reset_index()
)

# Join both complete histories to the candidate relationships.
candidate_history_detail = (
    ambiguous_review_output
    .merge(
        cr_history_detail,
        left_on=["team", "crickrida_name", "crickrida_id"],
        right_on=["team", "player_name", "source_player_id"],
        how="left"
    )
    .drop(columns=["player_name", "source_player_id"])
    .merge(
        cs_history_detail,
        left_on=["team", "cricsheet_name", "cricsheet_id"],
        right_on=["team", "player_name", "source_player_id"],
        how="left"
    )
    .drop(columns=["player_name", "source_player_id"])
)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", 100
):
    display(candidate_history_detail)

,team,crickrida_name,crickrida_id,cricsheet_name,cricsheet_id,similarity,closest_crickrida_date,closest_cricsheet_date,nearest_gap_days,crickrida_dates,crickrida_roles,crickrida_match_count,cricsheet_dates,cricsheet_roles,cricsheet_match_count
0,Bangladesh,Al Islam,CR_alislam,Sunzamul Islam,0f348f58,0.700,2026-10-21,2018-01-23,3193,[2026-10-21],"[batting, bowling]",1,"[2017-05-19, 2018-01-15, 2018-01-23]","[batting, bowling]",3
1,Bangladesh,Al Islam,CR_alislam,Jahurul Islam,6752c57b,0.737,2026-10-21,2013-05-08,4914,[2026-10-21],"[batting, bowling]",1,"[2010-06-21, 2010-07-10, 2010-07-12, 2010-07-16, 2012-03-11, 2012-03-16, 2012-03-20, 2012-03-22,...",[batting],11
2,Bangladesh,Al Islam,CR_alislam,Manjural Islam,2cfa6534,0.700,2026-10-21,2003-03-01,8635,[2026-10-21],"[batting, bowling]",1,"[2003-02-11, 2003-02-18, 2003-02-22, 2003-02-26, 2003-03-01]","[batting, bowling]",5
3,England,Ben Stokes,CR_benstokes,BA Stokes,e087956b,0.824,2019-06-18,2019-06-21,3,[2019-06-18],"[batting, bowling]",1,"[2011-08-25, 2011-09-09, 2011-09-11, 2013-09-03, 2013-09-08, 2013-09-14, 2013-09-16, 2014-01-12,...","[batting, bowling]",108
4,England,Ben Stokes,CR_benstokes,BT Foakes,fd770945,0.706,2019-06-18,2019-05-03,46,[2019-06-18],"[batting, bowling]",1,[2019-05-03],[batting],1
5,England,Jamie Overton,CR_jamieoverton,J Overton,59559bc2,0.800,2025-02-26,2025-03-01,3,[2025-02-26],"[batting, bowling]",1,"[2024-10-31, 2024-11-06, 2025-02-09, 2025-03-01, 2025-05-29, 2025-09-07, 2025-10-26, 2025-10-29,...","[batting, bowling]",18
6,England,Jamie Overton,CR_jamieoverton,C Overton,e9987a94,0.700,2025-02-26,2022-07-17,955,[2025-02-26],"[batting, bowling]",1,"[2018-06-21, 2021-07-08, 2021-07-10, 2021-07-13, 2022-07-12, 2022-07-14, 2022-07-17]","[batting, bowling]",7
7,India,Deepak Chahar,CR_deepakchahar,DL Chahar,23eeb873,0.700,2018-09-25,2019-12-15,446,[2018-09-25],"[batting, bowling]",1,"[2019-12-15, 2019-12-18, 2021-07-18, 2021-07-20, 2022-01-23, 2022-02-11, 2022-08-18, 2022-08-22,...","[batting, bowling]",11
8,India,Deepak Chahar,CR_deepakchahar,RD Chahar,2ed569a0,0.700,2018-09-25,2021-07-23,1032,[2018-09-25],"[batting, bowling]",1,[2021-07-23],"[batting, bowling]",1
9,India,Hardik Pandya,CR_hardikpandya,HH Pandya,dbe50b21,0.700,2023-10-11,2023-10-08,3,"[2019-06-22, 2023-10-11]","[batting, bowling]",2,"[2016-10-16, 2016-10-20, 2016-10-23, 2016-10-26, 2017-01-15, 2017-01-19, 2017-01-22, 2017-06-04,...","[batting, bowling]",91


In [186]:
# List the unresolved identity candidates.
unresolved_pairs = [
    ("Deepak Chahar", "DL Chahar", "23eeb873"),
    ("Kevin O'Brien", "KJ O'Brien", "243431b5"),
    ("Kevin O'Brien", "NJ O'Brien", "84826f48"),
    ("Asitha Fernando", "WIA Fernando", "9550470d"),
    ("Asitha Fernando", "AM Fernando", "de3d549a"),
    ("Avishka Fernando", "WIA Fernando", "9550470d"),
    ("Avishka Fernando", "AM Fernando", "de3d549a"),
    ("Shehan Jayasuriya", "NGRP Jayasuriya", "127bc5ca"),
    ("Shehan Jayasuriya", "GSNFG Jayasuriya", "b58b7db6"),
    ("Brandon King", "BA King", "7fca84b7"),
    ("Kieran Powell", "KOA Powell", "e012ad13"),
]

unresolved_ids = {item[2] for item in unresolved_pairs}

unresolved_match_evidence = (
    opponent_cricsheet[
        opponent_cricsheet["source_player_id"].isin(unresolved_ids)
    ][
        ["date", "team", "player_name", "source_player_id", "match_id", "role"]
    ]
    .drop_duplicates()
    .sort_values(["team", "player_name", "date"])
)

# Add match opponents and match-level metadata.
match_metadata = matches_df[
    [
        "match_id", "date", "team_1", "team_2",
        "venue", "city", "winner"
    ]
].copy()

match_metadata["match_id"] = match_metadata["match_id"].astype(str)

unresolved_match_evidence["match_id"] = (
    unresolved_match_evidence["match_id"].astype(str)
)

unresolved_match_evidence = unresolved_match_evidence.merge(
    match_metadata,
    on="match_id",
    how="left",
    suffixes=("", "_match")
)

print("Unresolved Cricsheet appearance records:",
      len(unresolved_match_evidence))

display(unresolved_match_evidence)

Unresolved Cricsheet appearance records: 449


,date,team,player_name,source_player_id,match_id,role,date_match,team_1,team_2,venue,city,winner
0,2019-12-15,India,DL Chahar,23eeb873,1187021,batting,2019-12-15,India,West Indies,"MA Chidambaram Stadium, Chepauk",Chennai,West Indies
1,2019-12-15,India,DL Chahar,23eeb873,1187021,bowling,2019-12-15,India,West Indies,"MA Chidambaram Stadium, Chepauk",Chennai,West Indies
2,2019-12-18,India,DL Chahar,23eeb873,1187022,bowling,2019-12-18,India,West Indies,Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket St...,Visakhapatnam,India
3,2021-07-18,India,DL Chahar,23eeb873,1262755,bowling,2021-07-18,Sri Lanka,India,"R Premadasa Stadium, Colombo",Colombo,India
4,2021-07-20,India,DL Chahar,23eeb873,1262756,batting,2021-07-20,Sri Lanka,India,"R Premadasa Stadium, Colombo",Colombo,India
...,...,...,...,...,...,...,...,...,...,...,...,...
444,2018-10-24,West Indies,KOA Powell,e012ad13,1157755,batting,2018-10-24,India,West Indies,Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket St...,Visakhapatnam,NaN
445,2018-10-27,West Indies,KOA Powell,e012ad13,1157756,batting,2018-10-27,West Indies,India,Maharashtra Cricket Association Stadium,Pune,West Indies
446,2018-10-29,West Indies,KOA Powell,e012ad13,1157757,batting,2018-10-29,India,West Indies,Brabourne Stadium,Mumbai,India
447,2018-11-01,West Indies,KOA Powell,e012ad13,1157758,batting,2018-11-01,West Indies,India,Greenfield International Stadium,Thiruvananthapuram,India


In [191]:
# Build Crickrida summary.
cr_unresolved_summary = (
    opponent_crickrida[
        opponent_crickrida["player_name"].isin(
            [item[0] for item in unresolved_pairs]
        )
    ]
    .groupby(["team", "player_name", "source_player_id"])
    .agg(
        cr_dates=("date", lambda x: sorted(
            pd.to_datetime(x).dt.strftime("%Y-%m-%d").unique()
        )),
        cr_match_count=("match_id", "nunique"),
        cr_roles=("role", lambda x: sorted(x.unique()))
    )
    .reset_index()
)

# Build Cricsheet summary.
cs_unresolved_summary = (
    opponent_cricsheet[
        opponent_cricsheet["source_player_id"].isin(unresolved_ids)
    ]
    .groupby(["team", "player_name", "source_player_id"])
    .agg(
        cs_dates=("date", lambda x: sorted(
            pd.to_datetime(x).dt.strftime("%Y-%m-%d").unique()
        )),
        cs_match_count=("match_id", "nunique"),
        cs_roles=("role", lambda x: sorted(x.unique()))
    )
    .reset_index()
)

# Build candidate relationship table with country included.
unresolved_comparison = pd.DataFrame(
    unresolved_pairs,
    columns=["crickrida_name", "cricsheet_name", "cricsheet_id"]
)

# Attach the country using the Crickrida appearance records.
cr_team_lookup = (
    cr_unresolved_summary[
        ["player_name", "team"]
    ]
    .drop_duplicates()
)

unresolved_comparison = unresolved_comparison.merge(
    cr_team_lookup,
    left_on="crickrida_name",
    right_on="player_name",
    how="left"
).drop(columns="player_name")

# Attach Crickrida histories.
unresolved_comparison = unresolved_comparison.merge(
    cr_unresolved_summary,
    left_on=["team", "crickrida_name"],
    right_on=["team", "player_name"],
    how="left"
).drop(columns="player_name")

# Attach Cricsheet histories.
unresolved_comparison = unresolved_comparison.merge(
    cs_unresolved_summary,
    left_on=["team", "cricsheet_name", "cricsheet_id"],
    right_on=["team", "player_name", "source_player_id"],
    how="left",
    suffixes=("", "_cs")
).drop(columns=["player_name", "source_player_id"])

# Display only required columns.
unresolved_comparison = unresolved_comparison[
    [
        "team",
        "crickrida_name",
        "cricsheet_name",
        "cricsheet_id",
        "cr_dates",
        "cr_match_count",
        "cr_roles",
        "cs_dates",
        "cs_match_count",
        "cs_roles"
    ]
]

print("Candidate comparison rows:", len(unresolved_comparison))

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", 100
):
    display(unresolved_comparison)

Candidate comparison rows: 11


,team,crickrida_name,cricsheet_name,cricsheet_id,cr_dates,cr_match_count,cr_roles,cs_dates,cs_match_count,cs_roles
0,India,Deepak Chahar,DL Chahar,23eeb873,[2018-09-25],1,"[batting, bowling]","[2019-12-15, 2019-12-18, 2021-07-18, 2021-07-20, 2022-01-23, 2022-02-11, 2022-08-18, 2022-08-22,...",11,"[batting, bowling]"
1,Ireland,Kevin O'Brien,KJ O'Brien,243431b5,"[2010-07-03, 2012-07-05, 2015-01-10, 2015-01-17, 2016-07-12, 2016-07-14, 2016-07-17, 2016-07-19,...",25,"[batting, bowling]","[2006-06-13, 2007-03-15, 2007-03-17, 2007-03-23, 2007-03-30, 2007-04-09, 2007-04-13, 2007-04-15,...",88,"[batting, bowling]"
2,Ireland,Kevin O'Brien,NJ O'Brien,84826f48,"[2010-07-03, 2012-07-05, 2015-01-10, 2015-01-17, 2016-07-12, 2016-07-14, 2016-07-17, 2016-07-19,...",25,"[batting, bowling]","[2007-03-15, 2007-03-17, 2007-03-23, 2007-03-30, 2007-04-03, 2007-04-09, 2007-04-13, 2007-04-15,...",63,[batting]
3,Sri Lanka,Asitha Fernando,WIA Fernando,9550470d,"[2022-11-30, 2024-02-11, 2024-02-14]",3,[bowling],"[2016-08-31, 2019-03-06, 2019-03-10, 2019-03-13, 2019-03-16, 2019-05-21, 2019-06-21, 2019-06-28,...",49,[batting]
4,Sri Lanka,Asitha Fernando,AM Fernando,de3d549a,"[2022-11-30, 2024-02-11, 2024-02-14]",3,[bowling],"[2017-07-08, 2021-03-14, 2021-07-01, 2021-07-04, 2024-08-02, 2024-08-04, 2024-08-07, 2024-10-20,...",33,"[batting, bowling]"
5,Sri Lanka,Avishka Fernando,WIA Fernando,9550470d,"[2024-02-09, 2024-02-11, 2024-02-14]",3,[batting],"[2016-08-31, 2019-03-06, 2019-03-10, 2019-03-13, 2019-03-16, 2019-05-21, 2019-06-21, 2019-06-28,...",49,[batting]
6,Sri Lanka,Avishka Fernando,AM Fernando,de3d549a,"[2024-02-09, 2024-02-11, 2024-02-14]",3,[batting],"[2017-07-08, 2021-03-14, 2021-07-01, 2021-07-04, 2024-08-02, 2024-08-04, 2024-08-07, 2024-10-20,...",33,"[batting, bowling]"
7,Sri Lanka,Shehan Jayasuriya,NGRP Jayasuriya,127bc5ca,[2018-09-17],1,"[batting, bowling]","[2018-08-01, 2018-08-05]",2,"[batting, bowling]"
8,Sri Lanka,Shehan Jayasuriya,GSNFG Jayasuriya,b58b7db6,[2018-09-17],1,"[batting, bowling]","[2015-11-01, 2015-11-04, 2015-11-07, 2016-11-14, 2016-11-16, 2016-11-23, 2018-07-29, 2018-08-01,...",11,"[batting, bowling]"
9,West Indies,Brandon King,BA King,7fca84b7,[2019-11-11],1,[batting],"[2020-01-07, 2020-01-09, 2020-01-12, 2022-02-06, 2022-02-09, 2022-02-11, 2022-05-31, 2022-06-02,...",57,[batting]


In [192]:
# Four newly approved aliases.
new_approved_aliases = pd.DataFrame([
    {
        "cricketiq_player_id": "CRIQ_23eeb873",
        "player_id": "23eeb873",
        "cricsheet_name": "DL Chahar",
        "crickrida_name": "Deepak Chahar",
        "normalized_name": "deepakchahar",
        "match_status": "verified_alias"
    },
    {
        "cricketiq_player_id": "CRIQ_243431b5",
        "player_id": "243431b5",
        "cricsheet_name": "KJ O'Brien",
        "crickrida_name": "Kevin O'Brien",
        "normalized_name": "kevinobrien",
        "match_status": "verified_alias"
    },
    {
        "cricketiq_player_id": "CRIQ_7fca84b7",
        "player_id": "7fca84b7",
        "cricsheet_name": "BA King",
        "crickrida_name": "Brandon King",
        "normalized_name": "brandonking",
        "match_status": "verified_alias"
    },
    {
        "cricketiq_player_id": "CRIQ_e012ad13",
        "player_id": "e012ad13",
        "cricsheet_name": "KOA Powell",
        "crickrida_name": "Kieran Powell",
        "normalized_name": "kieranpowell",
        "match_status": "verified_alias"
    }
])

# Append to the existing identity map.
player_identity_map_updated = pd.concat(
    [player_identity_map_updated, new_approved_aliases],
    ignore_index=True
)

# Validate uniqueness.
print("Total identities:", len(player_identity_map_updated))
print(
    "Unique CricketIQ IDs:",
    player_identity_map_updated["cricketiq_player_id"].nunique()
)
print(
    "Duplicate CricketIQ IDs:",
    player_identity_map_updated["cricketiq_player_id"].duplicated().sum()
)
print(
    "Duplicate Cricsheet IDs:",
    player_identity_map_updated["player_id"].duplicated().sum()
)
print(
    "Duplicate Crickrida names:",
    player_identity_map_updated["crickrida_name"].duplicated().sum()
)

display(new_approved_aliases)

Total identities: 126
Unique CricketIQ IDs: 126
Duplicate CricketIQ IDs: 0
Duplicate Cricsheet IDs: 0
Duplicate Crickrida names: 0


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,normalized_name,match_status
0,CRIQ_23eeb873,23eeb873,DL Chahar,Deepak Chahar,deepakchahar,verified_alias
1,CRIQ_243431b5,243431b5,KJ O'Brien,Kevin O'Brien,kevinobrien,verified_alias
2,CRIQ_7fca84b7,7fca84b7,BA King,Brandon King,brandonking,verified_alias
3,CRIQ_e012ad13,e012ad13,KOA Powell,Kieran Powell,kieranpowell,verified_alias


In [193]:
sri_lankan_candidate_ids = [
    "9550470d",  # WIA Fernando
    "de3d549a",  # AM Fernando
    "127bc5ca",  # NGRP Jayasuriya
    "b58b7db6"   # GSNFG Jayasuriya
]

sri_lankan_candidate_history = (
    opponent_cricsheet[
        opponent_cricsheet["source_player_id"].isin(
            sri_lankan_candidate_ids
        )
    ]
    .groupby(
        ["team", "player_name", "source_player_id"]
    )
    .agg(
        first_date=("date", "min"),
        last_date=("date", "max"),
        matches=("match_id", "nunique"),
        roles=("role", lambda x: sorted(x.unique()))
    )
    .reset_index()
    .sort_values(["player_name", "first_date"])
)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None
):
    display(sri_lankan_candidate_history)

print("\nCrickrida Sri Lankan players:")
display(
    cr_unresolved_summary[
        cr_unresolved_summary["team"].eq("Sri Lanka")
    ][["team", "player_name", "cr_dates", "cr_match_count", "cr_roles"]]
)

,team,player_name,source_player_id,first_date,last_date,matches,roles
0,Sri Lanka,AM Fernando,de3d549a,2017-07-08,2026-10-22,33,"[batting, bowling]"
1,Sri Lanka,GSNFG Jayasuriya,b58b7db6,2015-11-01,2019-10-02,11,"[batting, bowling]"
2,Sri Lanka,NGRP Jayasuriya,127bc5ca,2018-08-01,2018-08-05,2,"[batting, bowling]"
3,Sri Lanka,WIA Fernando,9550470d,2016-08-31,2026-10-22,49,[batting]



Crickrida Sri Lankan players:


,team,player_name,cr_dates,cr_match_count,cr_roles
2,Sri Lanka,Asitha Fernando,"[2022-11-30, 2024-02-11, 2024-02-14]",3,[bowling]
3,Sri Lanka,Avishka Fernando,"[2024-02-09, 2024-02-11, 2024-02-14]",3,[batting]
4,Sri Lanka,Shehan Jayasuriya,[2018-09-17],1,"[batting, bowling]"


In [194]:
sri_lanka_match_context = (
    opponent_cricsheet[
        opponent_cricsheet["source_player_id"].isin(
            sri_lankan_candidate_ids
        )
    ][
        [
            "date",
            "team",
            "player_name",
            "source_player_id",
            "match_id",
            "role"
        ]
    ]
    .merge(
        matches_df[
            [
                "match_id",
                "team_1",
                "team_2",
                "venue",
                "city"
            ]
        ],
        on="match_id",
        how="left"
    )
    .sort_values(["source_player_id", "date"])
)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", 60
):
    display(sri_lanka_match_context)

,date,team,player_name,source_player_id,match_id,role,team_1,team_2,venue,city
4,2018-08-01,Sri Lanka,NGRP Jayasuriya,127bc5ca,1142585,batting,Sri Lanka,South Africa,Rangiri Dambulla International Stadium,NaN
80,2018-08-01,Sri Lanka,NGRP Jayasuriya,127bc5ca,1142585,bowling,Sri Lanka,South Africa,Rangiri Dambulla International Stadium,NaN
5,2018-08-05,Sri Lanka,NGRP Jayasuriya,127bc5ca,1142586,batting,South Africa,Sri Lanka,Pallekele International Cricket Stadium,NaN
81,2018-08-05,Sri Lanka,NGRP Jayasuriya,127bc5ca,1142586,bowling,South Africa,Sri Lanka,Pallekele International Cricket Stadium,NaN
73,2016-08-31,Sri Lanka,WIA Fernando,9550470d,995463,batting,Sri Lanka,Australia,Rangiri Dambulla International Stadium,NaN
6,2019-03-06,Sri Lanka,WIA Fernando,9550470d,1144168,batting,South Africa,Sri Lanka,SuperSport Park,Centurion
7,2019-03-10,Sri Lanka,WIA Fernando,9550470d,1144169,batting,South Africa,Sri Lanka,Kingsmead,Durban
8,2019-03-13,Sri Lanka,WIA Fernando,9550470d,1144170,batting,South Africa,Sri Lanka,St George's Park,Port Elizabeth
9,2019-03-16,Sri Lanka,WIA Fernando,9550470d,1144171,batting,South Africa,Sri Lanka,Newlands,Cape Town
14,2019-05-21,Sri Lanka,WIA Fernando,9550470d,1169332,batting,Scotland,Sri Lanka,"Grange Cricket Club, Raeburn Place",Edinburgh


In [195]:
sri_lanka_crickrida_records = (
    afg_player_batting_df[
        afg_player_batting_df["player_name"].isin([
            "Asitha Fernando",
            "Avishka Fernando",
            "Shehan Jayasuriya"
        ])
    ][
        [
            "date",
            "player_name",
            "runs_scored",
            "balls_faced",
            "fours",
            "sixes",
            "dismissal",
            "match_url"
        ]
    ]
    .sort_values(["player_name", "date"])
)

sri_lanka_crickrida_bowling = (
    afg_player_bowling_df[
        afg_player_bowling_df["player_name"].isin([
            "Asitha Fernando",
            "Avishka Fernando",
            "Shehan Jayasuriya"
        ])
    ][
        [
            "date",
            "player_name",
            "overs_display",
            "maidens",
            "runs_conceded",
            "wickets",
            "economy",
            "match_url"
        ]
    ]
    .sort_values(["player_name", "date"])
)

print("CRICKRIDA BATTING RECORDS")
display(sri_lanka_crickrida_records)

print("\nCRICKRIDA BOWLING RECORDS")
display(sri_lanka_crickrida_bowling)

CRICKRIDA BATTING RECORDS


,date,player_name,runs_scored,balls_faced,fours,sixes,dismissal,match_url
2188,2024-02-09,Avishka Fernando,88,88,8,3,caught Ibrahim Zadran b Fareed Ahmad,https://crickrida.com/matches/afghanistan-sri-...
2201,2024-02-11,Avishka Fernando,5,23,0,0,caught Gulbadin Naib b Fazalhaq Farooqi,https://crickrida.com/matches/afghanistan-sri-...
2230,2024-02-14,Avishka Fernando,91,66,10,5,caught Sharafuddin Ashraf b Qais Ahmad,https://crickrida.com/matches/afghanistan-sri-...
1214,2018-09-17,Shehan Jayasuriya,14,28,0,0,"run out Rashid Khan, Mujeeb Ur Rahman b Mujeeb...",https://crickrida.com/matches/afghanistan-sri-...



CRICKRIDA BOWLING RECORDS


,date,player_name,overs_display,maidens,runs_conceded,wickets,economy,match_url
1182,2022-11-30,Asitha Fernando,10.0,1,64,1,6.400000,https://crickrida.com/matches/afghanistan-sri-...
1439,2024-02-11,Asitha Fernando,6.0,0,23,2,3.833333,https://crickrida.com/matches/afghanistan-sri-...
1445,2024-02-14,Asitha Fernando,9.0,0,44,2,4.888889,https://crickrida.com/matches/afghanistan-sri-...
769,2018-09-17,Shehan Jayasuriya,6.0,0,22,1,3.666667,https://crickrida.com/matches/afghanistan-sri-...


In [197]:
print("CRICSHEET BATTING COLUMNS")
print(player_batting_updated.columns.tolist())

print("\nCRICSHEET BOWLING COLUMNS")
print(player_bowling_updated.columns.tolist())

print("\nAFGHANISTAN BATTING COLUMNS")
print(afg_player_batting_updated.columns.tolist())

print("\nAFGHANISTAN BOWLING COLUMNS")
print(afg_player_bowling_updated.columns.tolist())

CRICSHEET BATTING COLUMNS
['match_id', 'innings_number', 'batting_team', 'batter', 'batter_id', 'runs', 'balls_faced', 'fours', 'sixes', 'strike_rate', 'cricketiq_player_id', 'canonical_player_name']

CRICSHEET BOWLING COLUMNS
['match_id', 'innings_number', 'bowler', 'bowler_id', 'runs_conceded', 'wides', 'no_balls', 'wickets', 'legal_balls', 'overs', 'economy', 'cricketiq_player_id', 'canonical_player_name']

AFGHANISTAN BATTING COLUMNS
['player_name', 'dismissal', 'runs_scored', 'balls_faced', 'fours', 'sixes', 'innings_number', 'date', 'match_url', 'strike_rate', 'batting_team', 'cricketiq_player_id', 'canonical_player_name']

AFGHANISTAN BOWLING COLUMNS
['player_name', 'overs_display', 'maidens', 'runs_conceded', 'wickets', 'wides', 'no_balls', 'legal_balls', 'innings_number', 'date', 'match_url', 'economy', 'batting_team', 'bowling_team', 'opponent', 'cricketiq_player_id', 'canonical_player_name']


In [198]:
# Candidate CricketIQ IDs
sri_lanka_identity_ids = [
    "CRIQ_de3d549a",  # AM Fernando
    "CRIQ_9550470d",  # WIA Fernando
    "CRIQ_127bc5ca",  # NGRP Jayasuriya
    "CRIQ_b58b7db6"   # GSNFG Jayasuriya
]

# Cricsheet batting records
sl_cs_batting = player_batting_updated[
    player_batting_updated["cricketiq_player_id"].isin(
        sri_lanka_identity_ids
    )
].copy()

# Cricsheet bowling records
sl_cs_bowling = player_bowling_updated[
    player_bowling_updated["cricketiq_player_id"].isin(
        sri_lanka_identity_ids
    )
].copy()

print("CRICSHEET BATTING RECORDS")
display(
    sl_cs_batting[
        [
            "match_id",
            "innings_number",
            "batting_team",
            "batter",
            "batter_id",
            "runs",
            "balls_faced",
            "fours",
            "sixes",
            "strike_rate",
            "cricketiq_player_id",
            "canonical_player_name"
        ]
    ].sort_values(["canonical_player_name", "match_id"])
)

print("\nCRICSHEET BOWLING RECORDS")
display(
    sl_cs_bowling[
        [
            "match_id",
            "innings_number",
            "bowler",
            "bowler_id",
            "runs_conceded",
            "wickets",
            "legal_balls",
            "economy",
            "cricketiq_player_id",
            "canonical_player_name"
        ]
    ].sort_values(["canonical_player_name", "match_id"])
)

CRICSHEET BATTING RECORDS


,match_id,innings_number,batting_team,batter,batter_id,runs,balls_faced,fours,sixes,strike_rate,cricketiq_player_id,canonical_player_name



CRICSHEET BOWLING RECORDS


,match_id,innings_number,bowler,bowler_id,runs_conceded,wickets,legal_balls,economy,cricketiq_player_id,canonical_player_name


In [199]:
candidate_cricsheet_ids = [
    "de3d549a",
    "9550470d",
    "127bc5ca",
    "b58b7db6"
]

print("IDENTITY REGISTRY")

display(
    player_registry_final[
        player_registry_final["player_id"].isin(
            candidate_cricsheet_ids
        )
    ]
)

print("\nIDENTITY MAP")

display(
    player_identity_map_updated[
        player_identity_map_updated["player_id"].isin(
            candidate_cricsheet_ids
        )
    ]
)

print("\nCRICSHEET MASTER REGISTRY")

display(
    player_master_df[
        player_master_df["player_id"].isin(
            candidate_cricsheet_ids
        )
    ]
)

IDENTITY REGISTRY


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,match_status,source,player_names,player_name,canonical_player_name
321,CS_127bc5ca,127bc5ca,NaN,NaN,source_only,Cricsheet,[NGRP Jayasuriya],NaN,NGRP Jayasuriya
1947,CS_9550470d,9550470d,NaN,NaN,source_only,Cricsheet,[WIA Fernando],NaN,WIA Fernando
2371,CS_b58b7db6,b58b7db6,NaN,NaN,source_only,Cricsheet,[GSNFG Jayasuriya],NaN,GSNFG Jayasuriya
2863,CS_de3d549a,de3d549a,NaN,NaN,source_only,Cricsheet,[AM Fernando],NaN,AM Fernando



IDENTITY MAP


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,normalized_name,match_status



CRICSHEET MASTER REGISTRY


,player_id,player_names,name_count,registry_match_count,player_name
225,127bc5ca,[NGRP Jayasuriya],1,2,NGRP Jayasuriya
1904,9550470d,[WIA Fernando],1,51,WIA Fernando
2345,b58b7db6,[GSNFG Jayasuriya],1,22,GSNFG Jayasuriya
2850,de3d549a,[AM Fernando],1,33,AM Fernando


In [200]:
candidate_ids = [
    "9550470d",
    "de3d549a",
    "127bc5ca",
    "b58b7db6"
]

print("CRICSHEET BATTING RECORDS")

display(
    player_batting_df[
        player_batting_df["batter_id"].isin(candidate_ids)
    ].sort_values(["batter", "match_id"])
)

print("\nCRICSHEET BOWLING RECORDS")

display(
    player_bowling_df[
        player_bowling_df["bowler_id"].isin(candidate_ids)
    ].sort_values(["bowler", "match_id"])
)

CRICSHEET BATTING RECORDS


,match_id,innings_number,batting_team,batter,batter_id,runs,balls_faced,fours,sixes,strike_rate,cricketiq_player_id,canonical_player_name
7146,1239535,1,Sri Lanka,AM Fernando,de3d549a,1,1,0,0,100.000000,CS_de3d549a,AM Fernando
7161,1239536,1,Sri Lanka,AM Fernando,de3d549a,0,5,0,0,0.000000,CS_de3d549a,AM Fernando
15465,1443546,1,Sri Lanka,AM Fernando,de3d549a,2,8,0,0,25.000000,CS_de3d549a,AM Fernando
15489,1443547,2,Sri Lanka,AM Fernando,de3d549a,0,2,0,0,0.000000,CS_de3d549a,AM Fernando
16319,1459908,1,Sri Lanka,AM Fernando,de3d549a,0,3,0,0,0.000000,CS_de3d549a,AM Fernando
...,...,...,...,...,...,...,...,...,...,...,...,...
16201,1456444,2,Sri Lanka,WIA Fernando,9550470d,5,9,1,0,55.555556,CS_9550470d,WIA Fernando
16329,1459908,1,Sri Lanka,WIA Fernando,9550470d,1,4,0,0,25.000000,CS_9550470d,WIA Fernando
19439,1550713,2,Sri Lanka,WIA Fernando,9550470d,5,9,1,0,55.555556,CS_9550470d,WIA Fernando
19475,1550715,2,Sri Lanka,WIA Fernando,9550470d,97,83,11,2,116.867470,CS_9550470d,WIA Fernando



CRICSHEET BOWLING RECORDS


,match_id,innings_number,bowler,bowler_id,runs_conceded,wides,no_balls,wickets,legal_balls,overs,economy,cricketiq_player_id,canonical_player_name
1323,1104481,2,AM Fernando,de3d549a,22,0,0,0,12,2.0,11.000000,CS_de3d549a,AM Fernando
4903,1239535,2,AM Fernando,de3d549a,54,0,0,0,60,10.0,5.400000,CS_de3d549a,AM Fernando
5177,1252071,2,AM Fernando,de3d549a,62,2,1,0,48,8.0,7.750000,CS_de3d549a,AM Fernando
10505,1442990,2,AM Fernando,de3d549a,34,2,0,1,36,6.0,5.666667,CS_de3d549a,AM Fernando
10518,1442991,2,AM Fernando,de3d549a,31,1,0,0,42,7.0,4.428571,CS_de3d549a,AM Fernando
10529,1442992,2,AM Fernando,de3d549a,29,0,0,1,30,5.0,5.800000,CS_de3d549a,AM Fernando
10539,1443546,2,AM Fernando,de3d549a,33,2,0,0,36,6.0,5.500000,CS_de3d549a,AM Fernando
10544,1443547,1,AM Fernando,de3d549a,54,1,0,1,42,7.0,7.714286,CS_de3d549a,AM Fernando
10563,1443548,2,AM Fernando,de3d549a,26,6,0,3,42,7.0,3.714286,CS_de3d549a,AM Fernando
10966,1451817,1,AM Fernando,de3d549a,28,2,0,0,24,4.0,7.000000,CS_de3d549a,AM Fernando


In [202]:
registry_variables = [
    name for name, obj in globals().items()
    if isinstance(obj, pd.DataFrame)
    and (
        "registry" in name.lower()
        or "master" in name.lower()
        or "identity" in name.lower()
    )
]

print("Available registry-related DataFrames:")

for name in registry_variables:
    obj = globals()[name]
    print(f"{name}: {obj.shape} | {obj.columns.tolist()}")

Available registry-related DataFrames:
player_registry_df: (86666, 3) | ['match_id', 'player_name', 'player_id']
player_master_df: (3287, 5) | ['player_id', 'player_names', 'name_count', 'registry_match_count', 'player_name']
player_identity_map: (106, 6) | ['cricketiq_player_id', 'player_id', 'cricsheet_name', 'crickrida_name', 'normalized_name', 'match_status']
cricsheet_only_master: (3182, 6) | ['player_id', 'player_names', 'normalized_names', 'cricketiq_player_id', 'source', 'match_status']
crickrida_only_master: (348, 5) | ['player_name', 'normalized_name', 'cricketiq_player_id', 'source', 'match_status']
cross_source_master: (106, 6) | ['cricketiq_player_id', 'player_id', 'cricsheet_name', 'crickrida_name', 'match_status', 'source']
cricsheet_master: (3182, 5) | ['cricketiq_player_id', 'player_id', 'player_names', 'match_status', 'source']
crickrida_master: (348, 4) | ['cricketiq_player_id', 'player_name', 'match_status', 'source']
player_registry_final: (3635, 9) | ['cricketiq_p

In [203]:
sri_lanka_ids = {
    "9550470d",
    "de3d549a",
    "127bc5ca",
    "b58b7db6"
}

sri_lanka_registry_records = (
    player_registry_df[
        player_registry_df["player_id"].isin(sri_lanka_ids)
    ]
    .sort_values(["player_id", "player_name"])
    .drop_duplicates(["player_id", "player_name"])
)

print("Original Cricsheet registry records:")
display(sri_lanka_registry_records)

print("\nRegistered names and match counts:")

for player_id in sorted(sri_lanka_ids):
    player_records = player_registry_df[
        player_registry_df["player_id"].eq(player_id)
    ]

    print(
        f"\nID: {player_id}",
        f"\nRegistered names: {sorted(player_records['player_name'].dropna().unique())}",
        f"\nRegistry records: {len(player_records)}",
        f"\nMatches: {player_records['match_id'].nunique()}"
    )

Original Cricsheet registry records:


,match_id,player_name,player_id
7705,1142585,NGRP Jayasuriya,127bc5ca
8204,1144168,WIA Fernando,9550470d
2404,1059710,GSNFG Jayasuriya,b58b7db6
4842,1104481,AM Fernando,de3d549a



Registered names and match counts:

ID: 127bc5ca 
Registered names: ['NGRP Jayasuriya'] 
Registry records: 2 
Matches: 2

ID: 9550470d 
Registered names: ['WIA Fernando'] 
Registry records: 51 
Matches: 51

ID: b58b7db6 
Registered names: ['GSNFG Jayasuriya'] 
Registry records: 22 
Matches: 22

ID: de3d549a 
Registered names: ['AM Fernando'] 
Registry records: 33 
Matches: 33


In [204]:
jayasuriya_ids = ["127bc5ca", "b58b7db6"]

jayasuriya_matches = (
    player_registry_df[
        player_registry_df["player_id"].isin(jayasuriya_ids)
    ][["match_id", "player_name", "player_id"]]
    .drop_duplicates()
    .merge(
        matches_df[
            ["match_id", "date", "team_1", "team_2", "venue"]
        ],
        on="match_id",
        how="left"
    )
    .sort_values(["player_id", "date"])
)

print("JAYASURIYA MATCH HISTORY")
display(jayasuriya_matches)

print("\nJAYASURIYA BATTING")
display(
    player_batting_df[
        player_batting_df["batter_id"].isin(jayasuriya_ids)
    ].sort_values(["batter_id", "match_id"])
)

print("\nJAYASURIYA BOWLING")
display(
    player_bowling_df[
        player_bowling_df["bowler_id"].isin(jayasuriya_ids)
    ].sort_values(["bowler_id", "match_id"])
)

JAYASURIYA MATCH HISTORY


,match_id,player_name,player_id,date,team_1,team_2,venue
5,1142585,NGRP Jayasuriya,127bc5ca,2018-08-01,Sri Lanka,South Africa,Rangiri Dambulla International Stadium
6,1142586,NGRP Jayasuriya,127bc5ca,2018-08-05,South Africa,Sri Lanka,Pallekele International Cricket Stadium
18,915777,GSNFG Jayasuriya,b58b7db6,2015-11-01,Sri Lanka,West Indies,R Premadasa Stadium
19,915779,GSNFG Jayasuriya,b58b7db6,2015-11-04,Sri Lanka,West Indies,R Premadasa Stadium
20,915781,GSNFG Jayasuriya,b58b7db6,2015-11-07,Sri Lanka,West Indies,Pallekele International Cricket Stadium
0,1059710,GSNFG Jayasuriya,b58b7db6,2016-11-14,Zimbabwe,Sri Lanka,Harare Sports Club
1,1059711,GSNFG Jayasuriya,b58b7db6,2016-11-16,Sri Lanka,West Indies,Harare Sports Club
2,1059714,GSNFG Jayasuriya,b58b7db6,2016-11-23,Sri Lanka,West Indies,Queens Sports Club
3,1142584,GSNFG Jayasuriya,b58b7db6,2018-07-29,Sri Lanka,South Africa,Rangiri Dambulla International Stadium
4,1142585,GSNFG Jayasuriya,b58b7db6,2018-08-01,Sri Lanka,South Africa,Rangiri Dambulla International Stadium



JAYASURIYA BATTING


,match_id,innings_number,batting_team,batter,batter_id,runs,balls_faced,fours,sixes,strike_rate,cricketiq_player_id,canonical_player_name
3407,1142585,1,Sri Lanka,NGRP Jayasuriya,127bc5ca,11,11,1,0,100.000000,CS_127bc5ca,NGRP Jayasuriya
3434,1142586,2,Sri Lanka,NGRP Jayasuriya,127bc5ca,0,2,0,0,0.000000,CS_127bc5ca,NGRP Jayasuriya
1291,1059711,2,Sri Lanka,GSNFG Jayasuriya,b58b7db6,31,59,2,0,52.542373,CS_b58b7db6,GSNFG Jayasuriya
1324,1059714,1,Sri Lanka,GSNFG Jayasuriya,b58b7db6,9,12,0,0,75.000000,CS_b58b7db6,GSNFG Jayasuriya
3387,1142584,1,Sri Lanka,GSNFG Jayasuriya,b58b7db6,0,8,0,0,0.000000,CS_b58b7db6,GSNFG Jayasuriya
3404,1142585,1,Sri Lanka,GSNFG Jayasuriya,b58b7db6,18,27,1,0,66.666667,CS_b58b7db6,GSNFG Jayasuriya
6146,1193506,1,Sri Lanka,GSNFG Jayasuriya,b58b7db6,13,7,1,1,185.714286,CS_b58b7db6,GSNFG Jayasuriya
6395,1198487,2,Sri Lanka,GSNFG Jayasuriya,b58b7db6,96,109,7,1,88.073394,CS_b58b7db6,GSNFG Jayasuriya
6404,1198488,1,Sri Lanka,GSNFG Jayasuriya,b58b7db6,3,6,0,0,50.000000,CS_b58b7db6,GSNFG Jayasuriya
18682,1531703,2,United States of America,GSNFG Jayasuriya,b58b7db6,9,13,0,0,69.230769,CS_b58b7db6,GSNFG Jayasuriya



JAYASURIYA BOWLING


,match_id,innings_number,bowler,bowler_id,runs_conceded,wides,no_balls,wickets,legal_balls,overs,economy,cricketiq_player_id,canonical_player_name
2335,1142585,2,NGRP Jayasuriya,127bc5ca,53,0,0,0,60,10.0,5.300000,CS_127bc5ca,NGRP Jayasuriya
2341,1142586,1,NGRP Jayasuriya,127bc5ca,42,0,0,0,36,6.0,7.000000,CS_127bc5ca,NGRP Jayasuriya
869,1059710,1,GSNFG Jayasuriya,b58b7db6,24,0,0,0,18,3.0,8.000000,CS_b58b7db6,GSNFG Jayasuriya
881,1059711,1,GSNFG Jayasuriya,b58b7db6,19,0,0,0,24,4.0,4.750000,CS_b58b7db6,GSNFG Jayasuriya
916,1059714,2,GSNFG Jayasuriya,b58b7db6,20,1,0,0,24,4.0,5.000000,CS_b58b7db6,GSNFG Jayasuriya
2323,1142584,2,GSNFG Jayasuriya,b58b7db6,23,1,0,0,24,4.0,5.750000,CS_b58b7db6,GSNFG Jayasuriya
2334,1142585,2,GSNFG Jayasuriya,b58b7db6,16,0,0,0,18,3.0,5.333333,CS_b58b7db6,GSNFG Jayasuriya
4234,1193506,2,GSNFG Jayasuriya,b58b7db6,40,0,0,0,36,6.0,6.666667,CS_b58b7db6,GSNFG Jayasuriya
4390,1198487,1,GSNFG Jayasuriya,b58b7db6,48,0,0,0,60,10.0,4.800000,CS_b58b7db6,GSNFG Jayasuriya
4408,1198488,2,GSNFG Jayasuriya,b58b7db6,39,0,0,1,42,7.0,5.571429,CS_b58b7db6,GSNFG Jayasuriya


In [205]:
# Summarise each candidate's Cricsheet career statistics.
jayasuriya_batting_summary = (
    player_batting_df[
        player_batting_df["batter_id"].isin(jayasuriya_ids)
    ]
    .groupby(["batter_id", "batter"])
    .agg(
        innings=("match_id", "nunique"),
        total_runs=("runs", "sum"),
        total_balls=("balls_faced", "sum"),
        total_fours=("fours", "sum"),
        total_sixes=("sixes", "sum")
    )
    .reset_index()
)

jayasuriya_bowling_summary = (
    player_bowling_df[
        player_bowling_df["bowler_id"].isin(jayasuriya_ids)
    ]
    .groupby(["bowler_id", "bowler"])
    .agg(
        innings=("match_id", "nunique"),
        total_runs_conceded=("runs_conceded", "sum"),
        total_wickets=("wickets", "sum"),
        total_legal_balls=("legal_balls", "sum")
    )
    .reset_index()
)

print("BATTING CAREER SUMMARY")
display(jayasuriya_batting_summary)

print("\nBOWLING CAREER SUMMARY")
display(jayasuriya_bowling_summary)

BATTING CAREER SUMMARY


,batter_id,batter,innings,total_runs,total_balls,total_fours,total_sixes
0,127bc5ca,NGRP Jayasuriya,2,11,13,1,0
1,b58b7db6,GSNFG Jayasuriya,20,682,827,59,12



BOWLING CAREER SUMMARY


,bowler_id,bowler,innings,total_runs_conceded,total_wickets,total_legal_balls
0,127bc5ca,NGRP Jayasuriya,2,95,0,96
1,b58b7db6,GSNFG Jayasuriya,15,378,6,402


In [206]:
# Compare the three players' appearance histories and roles.

jayasuriya_names = [
    "Shehan Jayasuriya",
    "GSNFG Jayasuriya",
    "NGRP Jayasuriya"
]

print("CRICKRIDA: SHEHAN JAYASURIYA")
display(
    opponent_crickrida[
        opponent_crickrida["player_name"].eq("Shehan Jayasuriya")
    ][
        ["date", "team", "player_name", "role", "match_id"]
    ].sort_values("date")
)

print("\nCRICSHEET: JAYASURIYA CANDIDATES")
display(
    opponent_cricsheet[
        opponent_cricsheet["player_name"].isin(
            ["GSNFG Jayasuriya", "NGRP Jayasuriya"]
        )
    ][
        ["date", "team", "player_name", "role", "match_id"]
    ].sort_values(["player_name", "date"])
)

CRICKRIDA: SHEHAN JAYASURIYA


,date,team,player_name,role,match_id
1214,2018-09-17,Sri Lanka,Shehan Jayasuriya,batting,https://crickrida.com/matches/afghanistan-sri-...
3488,2018-09-17,Sri Lanka,Shehan Jayasuriya,bowling,https://crickrida.com/matches/afghanistan-sri-...



CRICSHEET: JAYASURIYA CANDIDATES


,date,team,player_name,role,match_id
44712,2015-11-01,Sri Lanka,GSNFG Jayasuriya,batting,915777
76124,2015-11-01,Sri Lanka,GSNFG Jayasuriya,bowling,915777
76136,2015-11-04,Sri Lanka,GSNFG Jayasuriya,bowling,915779
44748,2015-11-07,Sri Lanka,GSNFG Jayasuriya,batting,915781
46301,2016-11-14,Sri Lanka,GSNFG Jayasuriya,bowling,1059710
1291,2016-11-16,Sri Lanka,GSNFG Jayasuriya,batting,1059711
46313,2016-11-16,Sri Lanka,GSNFG Jayasuriya,bowling,1059711
1324,2016-11-23,Sri Lanka,GSNFG Jayasuriya,batting,1059714
46348,2016-11-23,Sri Lanka,GSNFG Jayasuriya,bowling,1059714
3387,2018-07-29,Sri Lanka,GSNFG Jayasuriya,batting,1142584


In [207]:
fernando_ids = ["9550470d", "de3d549a"]

fernando_batting_summary = (
    player_batting_df[
        player_batting_df["batter_id"].isin(fernando_ids)
    ]
    .groupby(["batter_id", "batter"])
    .agg(
        innings=("match_id", "nunique"),
        total_runs=("runs", "sum"),
        total_balls=("balls_faced", "sum")
    )
    .reset_index()
)

fernando_bowling_summary = (
    player_bowling_df[
        player_bowling_df["bowler_id"].isin(fernando_ids)
    ]
    .groupby(["bowler_id", "bowler"])
    .agg(
        innings=("match_id", "nunique"),
        total_runs_conceded=("runs_conceded", "sum"),
        total_wickets=("wickets", "sum"),
        total_legal_balls=("legal_balls", "sum")
    )
    .reset_index()
)

print("CRICSHEET BATTING")
display(fernando_batting_summary)

print("\nCRICSHEET BOWLING")
display(fernando_bowling_summary)

CRICSHEET BATTING


,batter_id,batter,innings,total_runs,total_balls
0,9550470d,WIA Fernando,49,1507,1702
1,de3d549a,AM Fernando,14,21,54



CRICSHEET BOWLING


,bowler_id,bowler,innings,total_runs_conceded,total_wickets,total_legal_balls
0,de3d549a,AM Fernando,32,1430,45,1401


In [208]:
fernando_ids = ["9550470d", "de3d549a"]

fernando_history = (
    opponent_cricsheet[
        opponent_cricsheet["source_player_id"].isin(fernando_ids)
    ]
    .sort_values(["source_player_id", "date"])
)

display(
    fernando_history[
        [
            "date",
            "team",
            "player_name",
            "role",
            "match_id"
        ]
    ].to_string(index=False)
)

'      date      team  player_name    role match_id\n2016-08-31 Sri Lanka WIA Fernando batting   995463\n2019-03-06 Sri Lanka WIA Fernando batting  1144168\n2019-03-10 Sri Lanka WIA Fernando batting  1144169\n2019-03-13 Sri Lanka WIA Fernando batting  1144170\n2019-03-16 Sri Lanka WIA Fernando batting  1144171\n2019-05-21 Sri Lanka WIA Fernando batting  1169332\n2019-06-21 Sri Lanka WIA Fernando batting  1144509\n2019-06-28 Sri Lanka WIA Fernando batting  1144517\n2019-07-01 Sri Lanka WIA Fernando batting  1144521\n2019-07-06 Sri Lanka WIA Fernando batting  1144526\n2019-07-26 Sri Lanka WIA Fernando batting  1193504\n2019-07-28 Sri Lanka WIA Fernando batting  1193505\n2019-07-31 Sri Lanka WIA Fernando batting  1193506\n2019-09-30 Sri Lanka WIA Fernando batting  1198487\n2019-10-02 Sri Lanka WIA Fernando batting  1198488\n2020-02-22 Sri Lanka WIA Fernando batting  1213871\n2020-02-26 Sri Lanka WIA Fernando batting  1213872\n2020-03-01 Sri Lanka WIA Fernando batting  1213873\n2021-07-01 

In [209]:
fernando_ids = ["9550470d", "de3d549a"]

fernando_registry_details = (
    player_registry_df[
        player_registry_df["player_id"].isin(fernando_ids)
    ]
    .groupby("player_id")
    .agg(
        registered_names=("player_name", lambda x: sorted(x.unique())),
        registered_match_count=("match_id", "nunique")
    )
    .reset_index()
)

display(fernando_registry_details)

,player_id,registered_names,registered_match_count
0,9550470d,[WIA Fernando],51
1,de3d549a,[AM Fernando],33


In [210]:
# Count one-to-one candidate pairs by country.
print("ONE-TO-ONE CANDIDATES BY COUNTRY")
display(
    one_to_one_candidates["team"]
    .value_counts()
    .rename_axis("team")
    .reset_index(name="candidate_pairs")
)

# Inspect all Australian candidates with their matching evidence.
australia_candidates = (
    date_review[
        (date_review["team"] == "Australia")
        & (date_review["similarity"] >= 0.70)
    ]
    .merge(
        one_to_one_candidates[
            ["team", "crickrida_name", "cricsheet_name", "cricsheet_id"]
        ],
        on=["team", "crickrida_name", "cricsheet_name", "cricsheet_id"],
        how="inner"
    )
    .sort_values(
        ["similarity", "within_7_days", "nearest_date_gap"],
        ascending=[False, False, True]
    )
)

print("\nAUSTRALIAN ONE-TO-ONE CANDIDATES")
display(
    australia_candidates[
        [
            "crickrida_name",
            "cricsheet_name",
            "cricsheet_id",
            "similarity",
            "within_7_days",
            "within_30_days",
            "nearest_date_gap",
            "crickrida_first_date",
            "crickrida_last_date",
            "cricsheet_first_date",
            "cricsheet_last_date"
        ]
    ].to_string(index=False)
)

ONE-TO-ONE CANDIDATES BY COUNTRY


,team,candidate_pairs
0,Zimbabwe,39
1,Ireland,29
2,South Africa,22
3,England,20
4,New Zealand,19
5,Australia,19
6,Sri Lanka,17
7,West Indies,14
8,India,9
9,Bangladesh,2



AUSTRALIAN ONE-TO-ONE CANDIDATES


'     crickrida_name  cricsheet_name cricsheet_id  similarity  within_7_days  within_30_days  nearest_date_gap crickrida_first_date crickrida_last_date cricsheet_first_date cricsheet_last_date\n        Pat Cummins      PJ Cummins     ded9240e       0.842              3              17                 3           2019-06-01          2023-11-07           2011-10-19          2026-09-30\n     Josh Hazlewood    JR Hazlewood     03806cf8       0.833              2              14                 3           2015-03-04          2023-11-07           2010-06-22          2026-09-27\n Marnus Labuschagne   M Labuschagne     fa433be6       0.828              1               9                 3           2023-11-07          2023-11-07           2020-01-17          2026-06-14\n         Adam Zampa         A Zampa     14f96089       0.800              5              17                 3           2019-06-01          2025-02-28           2016-02-06          2026-09-30\n    James Pattinson    JL Pattinso

In [211]:
# Inspect Crickrida and Cricsheet histories for Australian candidates.

australian_pairs = australia_candidates[
    ["crickrida_name", "cricsheet_name", "cricsheet_id"]
].drop_duplicates()

for _, pair in australian_pairs.iterrows():
    cr_name = pair["crickrida_name"]
    cs_name = pair["cricsheet_name"]
    cs_id = pair["cricsheet_id"]

    cr_history = opponent_crickrida[
        (opponent_crickrida["team"] == "Australia")
        & (opponent_crickrida["player_name"] == cr_name)
    ]

    cs_history = opponent_cricsheet[
        (opponent_cricsheet["team"] == "Australia")
        & (opponent_cricsheet["source_player_id"] == cs_id)
    ]

    print(f"\n{cr_name}  <->  {cs_name} ({cs_id})")
    print(
        "Crickrida:",
        sorted(pd.to_datetime(cr_history["date"]).dt.strftime("%Y-%m-%d").unique())
    )
    print(
        "Cricsheet:",
        sorted(pd.to_datetime(cs_history["date"]).dt.strftime("%Y-%m-%d").unique())
    )
    print(
        "Crickrida roles:", sorted(cr_history["role"].unique()),
        "| Cricsheet roles:", sorted(cs_history["role"].unique())
    )


Pat Cummins  <->  PJ Cummins (ded9240e)
Crickrida: ['2019-06-01', '2023-11-07']
Cricsheet: ['2011-10-19', '2011-10-23', '2011-10-28', '2012-06-23', '2012-06-29', '2014-11-21', '2014-11-23', '2015-01-16', '2015-01-18', '2015-01-23', '2015-02-28', '2015-03-14', '2015-08-27', '2015-09-03', '2015-09-05', '2015-09-08', '2015-09-11', '2015-09-13', '2016-12-04', '2016-12-06', '2016-12-09', '2017-01-13', '2017-01-15', '2017-01-19', '2017-01-22', '2017-01-26', '2017-01-30', '2017-02-05', '2017-06-02', '2017-06-05', '2017-06-10', '2017-09-17', '2017-09-21', '2017-09-24', '2017-09-28', '2017-10-01', '2018-01-14', '2018-01-21', '2018-01-26', '2018-11-04', '2018-11-09', '2018-11-11', '2019-03-02', '2019-03-05', '2019-03-08', '2019-03-10', '2019-03-13', '2019-03-27', '2019-06-06', '2019-06-09', '2019-06-12', '2019-06-15', '2019-06-20', '2019-06-25', '2019-06-29', '2019-07-06', '2019-07-11', '2020-01-14', '2020-01-17', '2020-01-19', '2020-02-29', '2020-03-04', '2020-03-13', '2020-09-11', '2020-09-13

In [212]:
# Find Australia vs Afghanistan, 1 June 2019 in Cricsheet.

target_match = matches_df[
    (pd.to_datetime(matches_df["date"]).dt.strftime("%Y-%m-%d") == "2019-06-01")
    & (
        (
            (matches_df["team_1"] == "Australia")
            & (matches_df["team_2"] == "Afghanistan")
        )
        |
        (
            (matches_df["team_1"] == "Afghanistan")
            & (matches_df["team_2"] == "Australia")
        )
    )
]

display(
    target_match[
        ["match_id", "date", "team_1", "team_2", "venue"]
    ]
)

,match_id,date,team_1,team_2,venue


In [213]:
target_match = matches_df[
    (pd.to_datetime(matches_df["date"]).dt.strftime("%Y-%m-%d") == "2019-06-20")
    & (
        (
            (matches_df["team_1"] == "Australia")
            & (matches_df["team_2"] == "Bangladesh")
        )
        |
        (
            (matches_df["team_1"] == "Bangladesh")
            & (matches_df["team_2"] == "Australia")
        )
    )
]

display(
    target_match[
        ["match_id", "date", "team_1", "team_2", "venue"]
    ]
)

,match_id,date,team_1,team_2,venue
237,1144508,2019-06-20,Australia,Bangladesh,Trent Bridge


In [214]:
# Extract Australia's registered players for the selected Cricsheet match.

australia_xi_cricsheet = (
    player_registry_df[
        (player_registry_df["match_id"].astype(str) == "1144508")
        & (player_registry_df["player_name"].notna())
    ][["player_id", "player_name"]]
    .drop_duplicates()
    .sort_values("player_name")
    .reset_index(drop=True)
)

print("AUSTRALIA PLAYERS IN CRICSHEET REGISTRY")
print("Player count:", len(australia_xi_cricsheet))
display(australia_xi_cricsheet)

AUSTRALIA PLAYERS IN CRICSHEET REGISTRY
Player count: 27


,player_id,player_name
0,14f96089,A Zampa
1,b8d490fd,AJ Finch
2,69d03465,AT Carey
3,f02778d1,Aleem Dar
4,dcce6f09,DA Warner
5,b681e71e,GJ Maxwell
6,2e760301,JJ Crowe
7,0404d43c,Liton Das
8,76752ac8,MA Gough
9,3fb19989,MA Starc


In [215]:
# Find Australia's 4 November 2023 ODI against England.

target_match_2023 = matches_df[
    (pd.to_datetime(matches_df["date"]).dt.strftime("%Y-%m-%d") == "2023-11-04")
    & (
        (
            (matches_df["team_1"] == "Australia")
            & (matches_df["team_2"] == "England")
        )
        |
        (
            (matches_df["team_1"] == "England")
            & (matches_df["team_2"] == "Australia")
        )
    )
]

display(
    target_match_2023[
        ["match_id", "date", "team_1", "team_2", "venue"]
    ]
)

,match_id,date,team_1,team_2,venue
784,1384427,2023-11-04,Australia,England,"Narendra Modi Stadium, Ahmedabad"


In [216]:
australia_xi_2023_cricsheet = (
    player_registry_df[
        (player_registry_df["match_id"].astype(str) == "1384427")
        & (player_registry_df["player_name"].notna())
    ][["player_id", "player_name"]]
    .drop_duplicates()
    .sort_values("player_name")
    .reset_index(drop=True)
)

print("AUSTRALIA PLAYERS IN CRICSHEET REGISTRY")
print("Player count:", len(australia_xi_2023_cricsheet))
display(australia_xi_2023_cricsheet)

AUSTRALIA PLAYERS IN CRICSHEET REGISTRY
Player count: 28


,player_id,player_name
0,14f96089,A Zampa
1,6fa3258c,AT Holdstock
2,249d60c9,AU Rashid
3,e0e5b209,Ahsan Raza
4,e087956b,BA Stokes
5,eaa76d3c,C Green
6,d5ac41d8,CB Gaffaney
7,4c5d73db,CR Woakes
8,dcce6f09,DA Warner
9,ad9c32a2,DJ Malan


In [217]:
players_to_check = [
    "James Pattinson",
    "Brad Haddin",
    "Xavier Doherty",
    "David Hussey",
    "Michael Hussey",
    "James Faulkner",
    "Matthew Short",
    "Nathan Ellis"
]

candidate_fixtures = (
    opponent_crickrida[
        opponent_crickrida["player_name"].isin(players_to_check)
    ]
    .groupby(["date", "player_name", "match_id"])
    .agg(
        roles=("role", lambda x: ", ".join(sorted(set(x)))),
        team=("team", "first")
    )
    .reset_index()
    .sort_values(["date", "player_name"])
)

display(candidate_fixtures.to_string(index=False))

'      date     player_name                                                                match_id            roles      team\n2012-08-25    David Hussey  https://crickrida.com/matches/afghanistan-australia-2012-08-25-571947/ batting, bowling Australia\n2012-08-25 James Pattinson  https://crickrida.com/matches/afghanistan-australia-2012-08-25-571947/ batting, bowling Australia\n2012-08-25  Michael Hussey  https://crickrida.com/matches/afghanistan-australia-2012-08-25-571947/          batting Australia\n2012-08-25  Xavier Doherty  https://crickrida.com/matches/afghanistan-australia-2012-08-25-571947/          bowling Australia\n2015-03-04     Brad Haddin  https://crickrida.com/matches/afghanistan-australia-2015-03-04-656449/          batting Australia\n2015-03-04  James Faulkner  https://crickrida.com/matches/afghanistan-australia-2015-03-04-656449/ batting, bowling Australia\n2025-02-28   Matthew Short https://crickrida.com/matches/afghanistan-australia-2025-02-28-1466423/ batting, bo

In [218]:
# Find Australia's Cricsheet ODIs between August and September 2012.

australia_2012_matches = matches_df[
    (pd.to_datetime(matches_df["date"]) >= "2012-08-25")
    & (pd.to_datetime(matches_df["date"]) <= "2012-09-30")
    & (
        (matches_df["team_1"] == "Australia")
        | (matches_df["team_2"] == "Australia")
    )
].sort_values("date")

display(
    australia_2012_matches[
        ["match_id", "date", "team_1", "team_2", "venue"]
    ].to_string(index=False)
)

'match_id       date    team_1   team_2                   venue\n  571145 2012-08-28 Australia Pakistan Sharjah Cricket Stadium\n  571146 2012-08-31 Australia Pakistan    Sheikh Zayed Stadium\n  571147 2012-09-03 Australia Pakistan Sharjah Cricket Stadium'

In [219]:
australia_xi_2012_cricsheet = (
    player_registry_df[
        (player_registry_df["match_id"].astype(str) == "571145")
        & (player_registry_df["player_name"].notna())
    ][["player_id", "player_name"]]
    .drop_duplicates()
    .sort_values("player_name")
    .reset_index(drop=True)
)

print("AUSTRALIA VS PAKISTAN — 28 AUGUST 2012")
print("Registered players:", len(australia_xi_2012_cricsheet))
display(australia_xi_2012_cricsheet)

AUSTRALIA VS PAKISTAN — 28 AUGUST 2012
Registered players: 27


,player_id,player_name
0,5441db8c,Aizaz Cheema
1,50250426,Asad Shafiq
2,211de69f,Azhar Ali
3,efc0a862,BF Bowden
4,dcce6f09,DA Warner
5,fd835ab3,DJ Hussey
6,2a2e6343,DT Christian
7,5056011d,GJ Bailey
8,b681e71e,GJ Maxwell
9,bad31fac,J Srinath


In [220]:
# Find Australia's Cricsheet ODIs from March to April 2015.

australia_2015_matches = matches_df[
    (pd.to_datetime(matches_df["date"]) >= "2015-03-04")
    & (pd.to_datetime(matches_df["date"]) <= "2015-04-30")
    & (
        (matches_df["team_1"] == "Australia")
        | (matches_df["team_2"] == "Australia")
    )
].sort_values("date")

display(
    australia_2015_matches[
        ["match_id", "date", "team_1", "team_2", "venue"]
    ].to_string(index=False)
)

'match_id       date    team_1      team_2                    venue\n  656461 2015-03-08 Australia   Sri Lanka    Sydney Cricket Ground\n  656477 2015-03-14 Australia    Scotland           Bellerive Oval\n  656487 2015-03-20 Australia    Pakistan            Adelaide Oval\n  656493 2015-03-26 Australia       India    Sydney Cricket Ground\n  656495 2015-03-29 Australia New Zealand Melbourne Cricket Ground'

In [221]:
australia_xi_2015_cricsheet = (
    player_registry_df[
        (player_registry_df["match_id"].astype(str) == "656461")
        & (player_registry_df["player_name"].notna())
    ][["player_id", "player_name"]]
    .drop_duplicates()
    .sort_values("player_name")
    .reset_index(drop=True)
)

print("AUSTRALIA VS SRI LANKA — 8 MARCH 2015")
print("Registered players:", len(australia_xi_2015_cricsheet))
display(australia_xi_2015_cricsheet)

AUSTRALIA VS SRI LANKA — 8 MARCH 2015
Registered players: 27


,player_id,player_name
0,896d78ad,AD Mathews
1,b8d490fd,AJ Finch
2,f02778d1,Aleem Dar
3,b0c772ee,BJ Haddin
4,dcce6f09,DA Warner
5,d18f9182,DPMD Jayawardene
6,b681e71e,GJ Maxwell
7,c5928dec,HDRL Thirimanne
8,a9dfdf6c,IJ Gould
9,2e760301,JJ Crowe


In [222]:
australia_2025_matches = matches_df[
    (pd.to_datetime(matches_df["date"]) >= "2025-02-28")
    & (pd.to_datetime(matches_df["date"]) <= "2025-03-31")
    & (
        (matches_df["team_1"] == "Australia")
        | (matches_df["team_2"] == "Australia")
    )
].sort_values("date")

display(
    australia_2025_matches[
        ["match_id", "date", "team_1", "team_2", "venue"]
    ].to_string(index=False)
)

'match_id       date    team_1 team_2                               venue\n 1466426 2025-03-04 Australia  India Dubai International Cricket Stadium'

In [223]:
australia_xi_2025_cricsheet = (
    player_registry_df[
        (player_registry_df["match_id"].astype(str) == "1466426")
        & (player_registry_df["player_name"].notna())
    ][["player_id", "player_name"]]
    .drop_duplicates()
    .sort_values("player_name")
    .reset_index(drop=True)
)

print("AUSTRALIA VS INDIA — 4 MARCH 2025")
print("Registered players:", len(australia_xi_2025_cricsheet))
display(australia_xi_2025_cricsheet)

AUSTRALIA VS INDIA — 4 MARCH 2025
Registered players: 27


,player_id,player_name
0,14f96089,A Zampa
1,685d3f80,AJ Pycroft
2,2e171977,AR Patel
3,69d03465,AT Carey
4,6fa3258c,AT Holdstock
5,e1b9f3a9,BJ Dwarshuis
6,fe366f34,C Connolly
7,d5ac41d8,CB Gaffaney
8,5b7ab5a9,CV Varun
9,b681e71e,GJ Maxwell


In [224]:
short_id = "a90e53ec"

short_history = (
    opponent_cricsheet[
        opponent_cricsheet["source_player_id"] == short_id
    ]
    .sort_values("date")
)

print("MATTHEW SHORT CANDIDATE HISTORY")
display(
    short_history[
        ["date", "team", "player_name", "role", "match_id"]
    ].to_string(index=False)
)

MATTHEW SHORT CANDIDATE HISTORY


'      date      team player_name    role match_id\n2023-09-22 Australia    MW Short bowling  1389388\n2023-09-22 Australia    MW Short batting  1389388\n2023-09-24 Australia    MW Short bowling  1389389\n2023-09-24 Australia    MW Short batting  1389389\n2024-02-02 Australia    MW Short bowling  1375847\n2024-02-04 Australia    MW Short batting  1375848\n2024-09-19 Australia    MW Short bowling  1385700\n2024-09-21 Australia    MW Short batting  1385701\n2024-09-21 Australia    MW Short bowling  1385701\n2024-09-24 Australia    MW Short batting  1385702\n2024-09-24 Australia    MW Short bowling  1385702\n2024-09-29 Australia    MW Short bowling  1385704\n2024-09-29 Australia    MW Short batting  1385704\n2024-11-04 Australia    MW Short batting  1426549\n2024-11-04 Australia    MW Short bowling  1426549\n2024-11-08 Australia    MW Short batting  1426550\n2024-11-10 Australia    MW Short batting  1426551\n2025-02-12 Australia    MW Short bowling  1459908\n2025-02-12 Australia    MW Sho

In [225]:
short_identity_check = player_identity_map_updated[
    (player_identity_map_updated["player_id"] == "a90e53ec")
    | (player_identity_map_updated["crickrida_name"] == "Matthew Short")
]

display(short_identity_check)

,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,normalized_name,match_status


In [226]:
matthew_short_record = pd.DataFrame([{
    "cricketiq_player_id": "CRIQ_a90e53ec",
    "player_id": "a90e53ec",
    "cricsheet_name": "MW Short",
    "crickrida_name": "Matthew Short",
    "normalized_name": "matthewshort",
    "match_status": "verified_alias"
}])

player_identity_map_updated = pd.concat(
    [player_identity_map_updated, matthew_short_record],
    ignore_index=True
)

print("Total identities:", len(player_identity_map_updated))
print("Unique CricketIQ IDs:", player_identity_map_updated["cricketiq_player_id"].nunique())
print("Duplicate CricketIQ IDs:", player_identity_map_updated["cricketiq_player_id"].duplicated().sum())
print("Duplicate Cricsheet IDs:", player_identity_map_updated["player_id"].duplicated().sum())
print("Duplicate Crickrida names:", player_identity_map_updated["crickrida_name"].duplicated().sum())

Total identities: 127
Unique CricketIQ IDs: 127
Duplicate CricketIQ IDs: 0
Duplicate Cricsheet IDs: 0
Duplicate Crickrida names: 0


In [227]:
ireland_candidates = (
    date_review[
        (date_review["team"] == "Ireland")
        & (date_review["similarity"] >= 0.70)
    ]
    .merge(
        one_to_one_candidates[
            ["team", "crickrida_name", "cricsheet_name", "cricsheet_id"]
        ],
        on=["team", "crickrida_name", "cricsheet_name", "cricsheet_id"],
        how="inner"
    )
    .sort_values(
        ["similarity", "within_7_days", "nearest_date_gap"],
        ascending=[False, False, True]
    )
)

display(
    ireland_candidates[
        [
            "crickrida_name",
            "cricsheet_name",
            "cricsheet_id",
            "similarity",
            "within_7_days",
            "nearest_date_gap",
            "crickrida_first_date",
            "crickrida_last_date",
            "cricsheet_first_date",
            "cricsheet_last_date"
        ]
    ].to_string(index=False)
)

'     crickrida_name  cricsheet_name cricsheet_id  similarity  within_7_days  nearest_date_gap crickrida_first_date crickrida_last_date cricsheet_first_date cricsheet_last_date\n        Jai Moondra       J Moondra     3bea7ea9       0.889              0                68           2026-08-07          2026-08-15           2026-10-22          2026-10-24\n    Cade Carmichael    C Carmichael     f3b158d9       0.880              0                68           2026-08-07          2026-08-15           2025-05-21          2026-10-24\n      John Anderson      J Anderson     d5bbb6a2       0.857              0                24           2016-07-12          2016-07-12           2014-09-08          2016-09-27\n           Ed Joyce        DI Joyce     fd67a507       0.857              0              1838           2012-07-05          2017-03-24           2006-06-13          2007-06-24\n   Theo van Woerkom  TF van Woerkom     70f2f922       0.846              0                81           2024-03-07

In [228]:
import pickle
import gzip
from pathlib import Path
from datetime import datetime
import types

# Create a separate checkpoint folder.
checkpoint_dir = (
    Path(r"C:\CricIQ")
    / "backups"
    / "notebook_checkpoints"
)
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Create a timestamped checkpoint filename.
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
checkpoint_path = checkpoint_dir / f"notebook06_checkpoint_{timestamp}.pkl.gz"

# Collect serializable notebook variables.
checkpoint_data = {}
skipped_variables = []

excluded_names = {
    "In", "Out", "get_ipython", "exit", "quit"
}

for name, value in list(globals().items()):
    if (
        name.startswith("_")
        or name in excluded_names
        or any(word in name.lower() for word in
               ["password", "secret", "token", "api_key", "apikey"])
        or isinstance(value, (types.ModuleType, types.FunctionType, type))
    ):
        continue

    try:
        pickle.dumps(value, protocol=pickle.HIGHEST_PROTOCOL)
        checkpoint_data[name] = value
    except Exception:
        skipped_variables.append(name)

# Save compressed checkpoint.
with gzip.open(checkpoint_path, "wb") as file:
    pickle.dump(
        checkpoint_data,
        file,
        protocol=pickle.HIGHEST_PROTOCOL
    )

print("CHECKPOINT SAVED")
print("Location:", checkpoint_path)
print("Variables saved:", len(checkpoint_data))
print("Variables skipped:", len(skipped_variables))
print("Identity map shape:",
      checkpoint_data["player_identity_map_updated"].shape)
print("Identity count:",
      checkpoint_data["player_identity_map_updated"]
      ["cricketiq_player_id"].nunique())
print("Checkpoint size (MB):",
      round(checkpoint_path.stat().st_size / (1024 ** 2), 2))


CHECKPOINT SAVED
Location: c:\CricIQ\notebooks\data\processed\player_data\checkpoints\notebook06_checkpoint_20261115_004228.pkl.gz
Variables saved: 441
Variables skipped: 1
Identity map shape: (127, 6)
Identity count: 127
Checkpoint size (MB): 62.46


In [1]:
import pickle
import gzip
from pathlib import Path

checkpoint_dir = (
    Path(r"C:\CricIQ")
    / "backups"
    / "notebook_checkpoints"
)

checkpoint_files = sorted(
    checkpoint_dir.glob("notebook06_checkpoint_*.pkl.gz")
)

if not checkpoint_files:
    raise FileNotFoundError("No Notebook 06 checkpoint found.")

latest_checkpoint = checkpoint_files[-1]

with gzip.open(latest_checkpoint, "rb") as file:
    restored_data = pickle.load(file)

globals().update(restored_data)

print("CHECKPOINT RESTORED")
print("File:", latest_checkpoint.name)
print("Variables restored:", len(restored_data))
print("Identity map:", player_identity_map_updated.shape)
print("Unique identities:",
      player_identity_map_updated["cricketiq_player_id"].nunique())

CHECKPOINT RESTORED
File: notebook06_checkpoint_20261115_004228.pkl.gz
Variables restored: 441
Identity map: (127, 6)
Unique identities: 127


In [3]:
# Find Ireland's Cricsheet matches around 3 July 2010.
import pandas as pd
cs_ireland_window = opponent_cricsheet[
    (opponent_cricsheet["team"] == "Ireland") &
    (pd.to_datetime(opponent_cricsheet["date"]) >= "2010-06-20") &
    (pd.to_datetime(opponent_cricsheet["date"]) <= "2010-07-20")
].copy()

# Match-level summary.
ireland_nearby_matches = (
    cs_ireland_window[
        ["date", "match_id"]
    ]
    .drop_duplicates()
    .sort_values("date")
)

print("Ireland Cricsheet matches between 20 June and 20 July 2010:")
display(ireland_nearby_matches)

# Registered players for those matches.
nearby_match_ids = ireland_nearby_matches["match_id"].tolist()

ireland_registered_players = (
    player_registry_df[
        player_registry_df["match_id"].isin(nearby_match_ids)
    ][["match_id", "player_name", "player_id"]]
    .drop_duplicates()
    .sort_values(["match_id", "player_name"])
)

print("\nRegistered players in these matches:")
display(ireland_registered_players)

Ireland Cricsheet matches between 20 June and 20 July 2010:


,date,match_id
30806,2010-07-07,450379
30778,2010-07-16,450107



Registered players in these matches:


,match_id,player_name,player_id
61249,450107,AR Cusack,568e5a4a
61250,450107,AR White,f27f89e8
61251,450107,Abdur Razzak,888e32bf
61252,450107,DT Johnston,e01895bb
61253,450107,DW Ward,e33923eb
61254,450107,Faisal Hossain,89ffd337
61255,450107,GC Wilson,05de1a5a
61256,450107,GH Dockrell,99cddc5e
61257,450107,HDPK Dharmasena,323e4c16
61258,450107,Imrul Kayes,5fbe14fc


In [4]:
ireland_review_ids = {
    "DT Johnston": "e01895bb",
    "PR Stirling": "cf494ffe",
    "AR Cusack": "568e5a4a",
    "KJ O'Brien": "243431b5",
    "AR White": "f27f89e8",
    "AD Poynter": "b98cdbf0",
    "NG Jones": "c5326dd0",
    "RD McCann": "92c43fac",
    "GH Dockrell": "99cddc5e",
    "JF Mooney": "aae9ce90"
}

ireland_history_review = (
    opponent_cricsheet[
        opponent_cricsheet["source_player_id"].isin(
            ireland_review_ids.values()
        )
    ]
    .groupby(
        ["player_name", "source_player_id", "team"]
    )
    .agg(
        first_appearance=("date", "min"),
        last_appearance=("date", "max"),
        matches=("match_id", "nunique"),
        roles=("role", lambda x: sorted(x.unique()))
    )
    .reset_index()
    .sort_values("player_name")
)

with pd.option_context("display.max_rows", None):
    display(ireland_history_review)

,player_name,source_player_id,team,first_appearance,last_appearance,matches,roles
0,AD Poynter,b98cdbf0,Ireland,2010-07-07,2014-09-12,6,[batting]
1,AR Cusack,568e5a4a,Ireland,2007-06-24,2015-03-15,29,"[batting, bowling]"
2,AR White,f27f89e8,Ireland,2006-06-13,2014-09-12,24,"[batting, bowling]"
3,DT Johnston,e01895bb,Ireland,2006-06-13,2013-09-06,33,"[batting, bowling]"
4,GH Dockrell,99cddc5e,Ireland,2010-07-07,2026-10-24,92,"[batting, bowling]"
5,JF Mooney,aae9ce90,Ireland,2006-06-13,2015-10-11,35,"[batting, bowling]"
6,KJ O'Brien,243431b5,Ireland,2006-06-13,2021-06-07,88,"[batting, bowling]"
7,NG Jones,c5326dd0,Ireland,2010-07-07,2011-08-25,4,"[batting, bowling]"
8,PR Stirling,cf494ffe,Ireland,2008-08-25,2026-10-24,112,"[batting, bowling]"
9,RD McCann,92c43fac,Ireland,2010-07-07,2010-09-06,2,[batting]


In [8]:
existing_ids = set(player_identity_map_updated["player_id"])
existing_names = set(player_identity_map_updated["crickrida_name"])

ireland_alias_check = new_ireland_aliases.copy()

ireland_alias_check["id_already_exists"] = (
    ireland_alias_check["player_id"].isin(existing_ids)
)

ireland_alias_check["name_already_exists"] = (
    ireland_alias_check["crickrida_name"].isin(existing_names)
)

ireland_alias_check["status"] = ireland_alias_check.apply(
    lambda row:
        "Already exists"
        if row["id_already_exists"] and row["name_already_exists"]
        else "Conflict - investigate"
        if row["id_already_exists"] or row["name_already_exists"]
        else "New alias",
    axis=1
)

display(
    ireland_alias_check[
        [
            "crickrida_name",
            "cricsheet_name",
            "player_id",
            "status"
        ]
    ]
)

print("\nSummary:")
print(ireland_alias_check["status"].value_counts())

,crickrida_name,cricsheet_name,player_id,status
0,Trent Johnston,DT Johnston,e01895bb,New alias
1,Paul Stirling,PR Stirling,cf494ffe,New alias
2,Alex Cusack,AR Cusack,568e5a4a,New alias
3,Kevin O'Brien,KJ O'Brien,243431b5,Already exists
4,Andrew White,AR White,f27f89e8,New alias
5,Andrew Poynter,AD Poynter,b98cdbf0,New alias
6,Nigel Jones,NG Jones,c5326dd0,New alias
7,Rory McCann,RD McCann,92c43fac,New alias
8,George Dockrell,GH Dockrell,99cddc5e,New alias
9,John Mooney,JF Mooney,aae9ce90,New alias



Summary:
status
New alias         9
Already exists    1
Name: count, dtype: int64


In [9]:
# Select only genuinely new aliases.
ireland_new_only = new_ireland_aliases[
    ~new_ireland_aliases["player_id"].isin(
        set(player_identity_map_updated["player_id"])
    )
].copy()

# Add them to the identity map.
player_identity_map_updated = pd.concat(
    [player_identity_map_updated, ireland_new_only],
    ignore_index=True
)

# Validate the updated map.
print("New aliases added:", len(ireland_new_only))
print("Identity map shape:", player_identity_map_updated.shape)
print(
    "Unique CricketIQ IDs:",
    player_identity_map_updated["cricketiq_player_id"].nunique()
)
print(
    "Duplicate CricketIQ IDs:",
    player_identity_map_updated["cricketiq_player_id"].duplicated().sum()
)
print(
    "Duplicate Cricsheet IDs:",
    player_identity_map_updated["player_id"].duplicated().sum()
)
print(
    "Duplicate Crickrida names:",
    player_identity_map_updated["crickrida_name"].duplicated().sum()
)

New aliases added: 9
Identity map shape: (136, 6)
Unique CricketIQ IDs: 136
Duplicate CricketIQ IDs: 0
Duplicate Cricsheet IDs: 0
Duplicate Crickrida names: 0


In [10]:
ireland_remaining = {
    "TF van Woerkom": "70f2f922",
    "J Anderson": "d5bbb6a2",
    "SR Thompson": "7f3ad1ed",
    "TJ Murtagh": "c89474d1",
    "CA Young": "bd0a4a7d",
    "J Little": "9061a703",
    "C Campher": "478a63e7",
    "HT Tector": "59ea70a4",
    "L Tucker": "3979b5e0",
    "GJ Delany": "33b3d2df"
}

ireland_remaining_history = (
    opponent_cricsheet[
        opponent_cricsheet["source_player_id"].isin(
            ireland_remaining.values()
        )
    ]
    .groupby(["player_name", "source_player_id", "team"])
    .agg(
        first_appearance=("date", "min"),
        last_appearance=("date", "max"),
        matches=("match_id", "nunique"),
        roles=("role", lambda x: sorted(x.unique()))
    )
    .reset_index()
    .sort_values("player_name")
)

display(ireland_remaining_history)

,player_name,source_player_id,team,first_appearance,last_appearance,matches,roles
0,C Campher,478a63e7,Ireland,2020-07-30,2026-10-24,38,"[batting, bowling]"
1,CA Young,bd0a4a7d,Ireland,2014-09-08,2024-10-07,40,"[batting, bowling]"
2,GJ Delany,33b3d2df,Ireland,2020-01-07,2023-07-04,18,"[batting, bowling]"
3,HT Tector,59ea70a4,Ireland,2020-07-30,2026-10-24,47,"[batting, bowling]"
4,J Anderson,d5bbb6a2,Ireland,2014-09-08,2016-09-27,6,"[batting, bowling]"
5,J Little,9061a703,Ireland,2019-05-03,2025-05-25,39,"[batting, bowling]"
6,L Tucker,3979b5e0,Ireland,2019-05-03,2026-10-24,45,[batting]
7,SR Thompson,7f3ad1ed,Ireland,2014-02-23,2017-05-12,13,"[batting, bowling]"
8,TF van Woerkom,70f2f922,Ireland,2023-09-26,2023-12-17,2,[bowling]
9,TJ Murtagh,c89474d1,Ireland,2013-05-23,2019-07-07,31,"[batting, bowling]"


In [11]:
ireland_candidate_names = [
    "Curtis Campher",
    "Craig Young",
    "Gareth Delany",
    "Harry Tector",
    "John Anderson",
    "Josh Little",
    "Lorcan Tucker",
    "Stuart Thompson",
    "Theo van Woerkom",
    "Tim Murtagh"
]

ireland_crickrida_history = (
    opponent_crickrida[
        opponent_crickrida["player_name"].isin(
            ireland_candidate_names
        )
    ]
    .groupby(["player_name", "team"])
    .agg(
        first_appearance=("date", "min"),
        last_appearance=("date", "max"),
        matches=("match_id", "nunique"),
        roles=("role", lambda x: sorted(x.unique())),
        match_dates=("date", lambda x: sorted(
            pd.to_datetime(x).dt.strftime("%Y-%m-%d").unique()
        ))
    )
    .reset_index()
    .sort_values("player_name")
)

with pd.option_context(
    "display.max_rows", None,
    "display.max_colwidth", 200
):
    display(ireland_crickrida_history)

,player_name,team,first_appearance,last_appearance,matches,roles,match_dates
0,Craig Young,Ireland,2015-01-10,2024-03-07,6,"[batting, bowling]","[2015-01-10, 2015-01-17, 2021-01-21, 2021-01-24, 2021-01-26, 2024-03-07]"
1,Curtis Campher,Ireland,2021-01-21,2026-08-15,9,"[batting, bowling]","[2021-01-21, 2021-01-24, 2021-01-26, 2024-03-07, 2024-03-12, 2026-08-07, 2026-08-10, 2026-08-12, 2026-08-15]"
2,Gareth Delany,Ireland,2021-01-21,2021-01-24,2,"[batting, bowling]","[2021-01-21, 2021-01-24]"
3,Harry Tector,Ireland,2021-01-21,2026-08-15,9,"[batting, bowling]","[2021-01-21, 2021-01-24, 2021-01-26, 2024-03-07, 2024-03-12, 2026-08-07, 2026-08-10, 2026-08-12, 2026-08-15]"
4,John Anderson,Ireland,2016-07-12,2016-07-12,1,[batting],[2016-07-12]
5,Josh Little,Ireland,2021-01-26,2021-01-26,1,"[batting, bowling]",[2021-01-26]
6,Lorcan Tucker,Ireland,2021-01-21,2026-08-15,9,[batting],"[2021-01-21, 2021-01-24, 2021-01-26, 2024-03-07, 2024-03-12, 2026-08-07, 2026-08-10, 2026-08-12, 2026-08-15]"
7,Stuart Thompson,Ireland,2017-03-15,2017-03-24,5,"[batting, bowling]","[2017-03-15, 2017-03-17, 2017-03-19, 2017-03-22, 2017-03-24]"
8,Theo van Woerkom,Ireland,2024-03-07,2024-03-12,2,"[batting, bowling]","[2024-03-07, 2024-03-12]"
9,Tim Murtagh,Ireland,2012-07-05,2019-05-21,24,"[batting, bowling]","[2012-07-05, 2016-07-12, 2016-07-14, 2016-07-17, 2016-07-19, 2017-03-15, 2017-03-17, 2017-03-19, 2017-03-22, 2017-03-24, 2017-12-05, 2017-12-07, 2017-12-10, 2018-03-23, 2018-08-27, 2018-08-29, 201..."


In [12]:
# Inspect all Cricsheet players with Anderson in their registered name.

anderson_registry = (
    player_registry_df[
        player_registry_df["player_name"]
        .str.contains("Anderson", case=False, na=False)
    ]
    .groupby(["player_name", "player_id"])
    .agg(
        matches=("match_id", "nunique")
    )
    .reset_index()
    .sort_values("player_name")
)

display(anderson_registry)

# Inspect the candidate's full Ireland match history.
j_anderson_matches = (
    opponent_cricsheet[
        opponent_cricsheet["source_player_id"] == "d5bbb6a2"
    ][["date", "match_id", "player_name", "team", "role"]]
    .drop_duplicates()
    .sort_values("date")
)

display(j_anderson_matches)

,player_name,player_id,matches
0,CJ Anderson,8abdf100,48
1,J Anderson,d5bbb6a2,6
2,JM Anderson,d12143bf,179
3,OV Anderson,da984dff,1
4,TL Anderson,f52ddf8d,23


,date,match_id,player_name,team,role
42037,2014-09-08,727935,J Anderson,Ireland,batting
42054,2014-09-10,727937,J Anderson,Ireland,batting
74242,2014-09-10,727937,J Anderson,Ireland,bowling
42065,2014-09-12,727939,J Anderson,Ireland,batting
74265,2014-09-12,727939,J Anderson,Ireland,bowling
44344,2016-06-18,913621,J Anderson,Ireland,batting
45105,2016-09-25,936131,J Anderson,Ireland,batting
45116,2016-09-27,936133,J Anderson,Ireland,batting


In [13]:
j_anderson_match_ids = (
    opponent_cricsheet.loc[
        opponent_cricsheet["source_player_id"] == "d5bbb6a2",
        "match_id"
    ]
    .unique()
)

j_anderson_teammates = (
    opponent_cricsheet[
        (opponent_cricsheet["match_id"].isin(j_anderson_match_ids)) &
        (opponent_cricsheet["team"] == "Ireland") &
        (opponent_cricsheet["source_player_id"] != "d5bbb6a2")
    ]
    .groupby(["player_name", "source_player_id"])
    .agg(
        matches_together=("match_id", "nunique"),
        first_seen=("date", "min"),
        last_seen=("date", "max")
    )
    .reset_index()
    .sort_values(
        ["matches_together", "player_name"],
        ascending=[False, True]
    )
)

display(j_anderson_teammates)

,player_name,source_player_id,matches_together,first_seen,last_seen
11,KJ O'Brien,243431b5,6,2014-09-08,2016-09-27
17,SW Poynter,cdf59953,5,2014-09-10,2016-09-27
5,CA Young,bd0a4a7d,4,2014-09-08,2016-09-25
0,A Balbirnie,6ef13460,3,2014-09-08,2014-09-12
1,AD Poynter,b98cdbf0,3,2014-09-08,2014-09-12
2,AR McBrine,e957b38f,3,2014-09-08,2016-06-18
7,GC Wilson,05de1a5a,3,2016-06-18,2016-09-27
10,JF Mooney,aae9ce90,3,2014-09-08,2014-09-12
12,MC Sorensen,3f18724b,3,2014-09-08,2014-09-12
14,PR Stirling,cf494ffe,3,2016-06-18,2016-09-27


In [14]:
# Inspect the Crickrida John Anderson match.
john_anderson_crickrida = opponent_crickrida[
    (opponent_crickrida["player_name"] == "John Anderson") &
    (opponent_crickrida["date"].astype(str).str.startswith("2016-07-12"))
]

display(
    john_anderson_crickrida[
        ["date", "match_id", "player_name", "team", "role"]
    ]
)

# Identify all Crickrida players in that match.
john_anderson_match_ids = (
    john_anderson_crickrida["match_id"].unique()
)

john_anderson_crickrida_teammates = (
    opponent_crickrida[
        (opponent_crickrida["match_id"].isin(john_anderson_match_ids)) &
        (opponent_crickrida["team"] == "Ireland")
    ][["player_name", "role"]]
    .drop_duplicates()
    .sort_values("player_name")
)

print("Crickrida Ireland players in the same match:")
display(john_anderson_crickrida_teammates)

,date,match_id,player_name,team,role
562,2016-07-12,https://crickrida.com/matches/afghanistan-irel...,John Anderson,Ireland,batting


Crickrida Ireland players in the same match:


,player_name,role
3097,Andy McBrine,bowling
566,Andy McBrine,batting
3094,Barry McCarthy,bowling
568,Barry McCarthy,batting
561,Ed Joyce,batting
567,George Dockrell,batting
3095,George Dockrell,bowling
562,John Anderson,batting
564,Kevin O'Brien,batting
3096,Kevin O'Brien,bowling


In [15]:
# Inspect Ireland's Cricsheet matches around 12 July 2016.

july_2016_ireland = opponent_cricsheet[
    (opponent_cricsheet["team"] == "Ireland") &
    (pd.to_datetime(opponent_cricsheet["date"]) >= "2016-07-01") &
    (pd.to_datetime(opponent_cricsheet["date"]) <= "2016-07-25")
].copy()

july_2016_matches = (
    july_2016_ireland[
        ["date", "match_id"]
    ]
    .drop_duplicates()
    .sort_values("date")
)

display(july_2016_matches)

# Show Ireland players in those matches.
july_2016_match_ids = july_2016_matches["match_id"].tolist()

july_2016_players = (
    player_registry_df[
        player_registry_df["match_id"].isin(july_2016_match_ids)
    ][["match_id", "player_name", "player_id"]]
    .drop_duplicates()
    .sort_values(["match_id", "player_name"])
)

display(july_2016_players)

,date,match_id


,match_id,player_name,player_id


In [16]:
ireland_remaining_aliases = [
    ("Curtis Campher", "C Campher", "478a63e7"),
    ("Craig Young", "CA Young", "bd0a4a7d"),
    ("Gareth Delany", "GJ Delany", "33b3d2df"),
    ("Harry Tector", "HT Tector", "59ea70a4"),
    ("Josh Little", "J Little", "9061a703"),
    ("Lorcan Tucker", "L Tucker", "3979b5e0"),
    ("Stuart Thompson", "SR Thompson", "7f3ad1ed"),
    ("Theo van Woerkom", "TF van Woerkom", "70f2f922"),
    ("Tim Murtagh", "TJ Murtagh", "c89474d1")
]

existing_ids = set(player_identity_map_updated["player_id"])
existing_names = set(player_identity_map_updated["crickrida_name"])

for crickrida_name, cricsheet_name, player_id in ireland_remaining_aliases:
    if player_id in existing_ids or crickrida_name in existing_names:
        status = "Already exists / investigate"
    else:
        status = "New candidate"

    print(f"{crickrida_name:20} | {cricsheet_name:15} | {status}")

Curtis Campher       | C Campher       | New candidate
Craig Young          | CA Young        | New candidate
Gareth Delany        | GJ Delany       | New candidate
Harry Tector         | HT Tector       | New candidate
Josh Little          | J Little        | New candidate
Lorcan Tucker        | L Tucker        | New candidate
Stuart Thompson      | SR Thompson     | New candidate
Theo van Woerkom     | TF van Woerkom  | New candidate
Tim Murtagh          | TJ Murtagh      | New candidate


In [17]:
ireland_remaining_new = pd.DataFrame([
    {
        "cricketiq_player_id": f"CRIQ_{player_id}",
        "player_id": player_id,
        "cricsheet_name": cricsheet_name,
        "crickrida_name": crickrida_name,
        "normalized_name": normalize_player_name(crickrida_name),
        "match_status": "verified_alias"
    }
    for crickrida_name, cricsheet_name, player_id
    in ireland_remaining_aliases
])

player_identity_map_updated = pd.concat(
    [player_identity_map_updated, ireland_remaining_new],
    ignore_index=True
)

print("New aliases added:", len(ireland_remaining_new))
print("Updated identity map:", player_identity_map_updated.shape)
print("Unique CricketIQ IDs:",
      player_identity_map_updated["cricketiq_player_id"].nunique())
print("Duplicate CricketIQ IDs:",
      player_identity_map_updated["cricketiq_player_id"].duplicated().sum())
print("Duplicate Cricsheet IDs:",
      player_identity_map_updated["player_id"].duplicated().sum())
print("Duplicate Crickrida names:",
      player_identity_map_updated["crickrida_name"].duplicated().sum())

New aliases added: 9
Updated identity map: (145, 6)
Unique CricketIQ IDs: 145
Duplicate CricketIQ IDs: 0
Duplicate Cricsheet IDs: 0
Duplicate Crickrida names: 0


In [18]:
ireland_recent_candidates = {
    "BD McDonough": "d2c250c8",
    "J Moondra": "3bea7ea9",
    "C Carmichael": "f3b158d9",
    "G Hoey": "19d0a172",
    "J Neill": "8bb34d86"
}

ireland_recent_history = (
    opponent_cricsheet[
        opponent_cricsheet["source_player_id"].isin(
            ireland_recent_candidates.values()
        )
    ]
    .groupby(["player_name", "source_player_id", "team"])
    .agg(
        first_appearance=("date", "min"),
        last_appearance=("date", "max"),
        matches=("match_id", "nunique"),
        roles=("role", lambda x: sorted(x.unique()))
    )
    .reset_index()
    .sort_values("player_name")
)

display(ireland_recent_history)

,player_name,source_player_id,team,first_appearance,last_appearance,matches,roles
0,BD McDonough,d2c250c8,Ireland,2026-10-24,2026-10-24,1,"[batting, bowling]"
1,C Carmichael,f3b158d9,Ireland,2025-05-21,2026-10-24,4,[batting]
2,G Hoey,19d0a172,Ireland,2024-10-02,2026-10-24,4,"[batting, bowling]"
3,J Moondra,3bea7ea9,Ireland,2026-10-22,2026-10-24,2,"[batting, bowling]"
4,J Neill,8bb34d86,Ireland,2025-05-25,2025-05-25,1,[bowling]


In [19]:
ireland_recent_names = [
    "Byron McDonough",
    "Cade Carmichael",
    "Gavin Hoey",
    "Jai Moondra",
    "Jordan Neill"
]

ireland_recent_crickrida = (
    opponent_crickrida[
        opponent_crickrida["player_name"].isin(
            ireland_recent_names
        )
    ]
    .groupby(["player_name", "team"])
    .agg(
        first_appearance=("date", "min"),
        last_appearance=("date", "max"),
        matches=("match_id", "nunique"),
        roles=("role", lambda x: sorted(x.unique())),
        match_dates=("date", lambda x: sorted(
            pd.to_datetime(x).dt.strftime("%Y-%m-%d").unique()
        ))
    )
    .reset_index()
    .sort_values("player_name")
)

with pd.option_context(
    "display.max_rows", None,
    "display.max_colwidth", 200
):
    display(ireland_recent_crickrida)

,player_name,team,first_appearance,last_appearance,matches,roles,match_dates
0,Byron McDonough,Ireland,2026-08-07,2026-08-12,3,"[batting, bowling]","[2026-08-07, 2026-08-10, 2026-08-12]"
1,Cade Carmichael,Ireland,2026-08-07,2026-08-15,4,[batting],"[2026-08-07, 2026-08-10, 2026-08-12, 2026-08-15]"
2,Gavin Hoey,Ireland,2026-08-07,2026-08-15,4,"[batting, bowling]","[2026-08-07, 2026-08-10, 2026-08-12, 2026-08-15]"
3,Jai Moondra,Ireland,2026-08-07,2026-08-15,3,"[batting, bowling]","[2026-08-07, 2026-08-10, 2026-08-15]"
4,Jordan Neill,Ireland,2026-08-15,2026-08-15,1,"[batting, bowling]",[2026-08-15]


In [20]:
recent_ireland_ids = {
    "BD McDonough": "d2c250c8",
    "C Carmichael": "f3b158d9",
    "G Hoey": "19d0a172",
    "J Moondra": "3bea7ea9",
    "J Neill": "8bb34d86"
}

recent_ireland_match_ids = opponent_cricsheet[
    opponent_cricsheet["source_player_id"].isin(
        recent_ireland_ids.values()
    )
]["match_id"].unique()

recent_ireland_teammates = (
    opponent_cricsheet[
        (opponent_cricsheet["match_id"].isin(recent_ireland_match_ids)) &
        (opponent_cricsheet["team"] == "Ireland")
    ]
    .groupby(["player_name", "source_player_id"])
    .agg(
        matches_together=("match_id", "nunique"),
        first_seen=("date", "min"),
        last_seen=("date", "max")
    )
    .reset_index()
    .sort_values(
        ["matches_together", "player_name"],
        ascending=[False, True]
    )
)

display(recent_ireland_teammates)

,player_name,source_player_id,matches_together,first_seen,last_seen
0,A Balbirnie,6ef13460,6,2024-10-02,2026-10-24
1,AR McBrine,e957b38f,6,2024-10-02,2026-10-24
8,GH Dockrell,99cddc5e,6,2024-10-02,2026-10-24
10,HT Tector,59ea70a4,6,2024-10-02,2026-10-24
17,PR Stirling,cf494ffe,6,2024-10-02,2026-10-24
4,C Campher,478a63e7,4,2024-10-02,2026-10-24
5,C Carmichael,f3b158d9,4,2025-05-21,2026-10-24
7,G Hoey,19d0a172,4,2024-10-02,2026-10-24
15,L Tucker,3979b5e0,4,2025-05-21,2026-10-24
14,L McCarthy,8eb2e2e6,3,2025-05-21,2026-10-22


In [21]:
ireland_recent_approved = [
    ("Byron McDonough", "BD McDonough", "d2c250c8"),
    ("Cade Carmichael", "C Carmichael", "f3b158d9"),
    ("Gavin Hoey", "G Hoey", "19d0a172"),
    ("Jai Moondra", "J Moondra", "3bea7ea9")
]

recent_alias_check = pd.DataFrame([
    {
        "crickrida_name": cr_name,
        "cricsheet_name": cs_name,
        "player_id": player_id,
        "status": (
            "Already exists"
            if player_id in set(player_identity_map_updated["player_id"])
            or cr_name in set(player_identity_map_updated["crickrida_name"])
            else "New candidate"
        )
    }
    for cr_name, cs_name, player_id in ireland_recent_approved
])

display(recent_alias_check)

,crickrida_name,cricsheet_name,player_id,status
0,Byron McDonough,BD McDonough,d2c250c8,New candidate
1,Cade Carmichael,C Carmichael,f3b158d9,New candidate
2,Gavin Hoey,G Hoey,19d0a172,New candidate
3,Jai Moondra,J Moondra,3bea7ea9,New candidate


In [22]:
ireland_recent_new = pd.DataFrame([
    {
        "cricketiq_player_id": f"CRIQ_{player_id}",
        "player_id": player_id,
        "cricsheet_name": cricsheet_name,
        "crickrida_name": crickrida_name,
        "normalized_name": normalize_player_name(crickrida_name),
        "match_status": "verified_alias"
    }
    for crickrida_name, cricsheet_name, player_id
    in ireland_recent_approved
])

# Final conflict checks
assert not set(ireland_recent_new["player_id"]) & set(
    player_identity_map_updated["player_id"]
)
assert not set(ireland_recent_new["crickrida_name"]) & set(
    player_identity_map_updated["crickrida_name"]
)

player_identity_map_updated = pd.concat(
    [player_identity_map_updated, ireland_recent_new],
    ignore_index=True
)

print("New aliases added:", len(ireland_recent_new))
print("Identity map shape:", player_identity_map_updated.shape)
print("Unique CricketIQ IDs:",
      player_identity_map_updated["cricketiq_player_id"].nunique())
print("Duplicate CricketIQ IDs:",
      player_identity_map_updated["cricketiq_player_id"].duplicated().sum())
print("Duplicate Cricsheet IDs:",
      player_identity_map_updated["player_id"].duplicated().sum())
print("Duplicate Crickrida names:",
      player_identity_map_updated["crickrida_name"].duplicated().sum())

New aliases added: 4
Identity map shape: (149, 6)
Unique CricketIQ IDs: 149
Duplicate CricketIQ IDs: 0
Duplicate Cricsheet IDs: 0
Duplicate Crickrida names: 0


In [23]:
# Identify Ireland Crickrida names not yet in the identity map.

mapped_ireland_names = set(
    player_identity_map_updated["crickrida_name"]
)

unmapped_ireland = (
    opponent_crickrida[
        (opponent_crickrida["team"] == "Ireland") &
        (~opponent_crickrida["player_name"].isin(mapped_ireland_names))
    ]
    .groupby("player_name")
    .agg(
        first_appearance=("date", "min"),
        last_appearance=("date", "max"),
        matches=("match_id", "nunique"),
        roles=("role", lambda x: sorted(x.unique()))
    )
    .reset_index()
    .sort_values("player_name")
)

print("Unmapped Ireland Crickrida players:", len(unmapped_ireland))
display(unmapped_ireland)

Unmapped Ireland Crickrida players: 17


,player_name,first_appearance,last_appearance,matches,roles
0,Andy McBrine,2016-07-12,2024-03-12,23,"[batting, bowling]"
1,Barry McCarthy,2016-07-12,2024-03-12,12,"[batting, bowling]"
2,Ben Calitz,2026-08-07,2026-08-12,3,[batting]
3,Boyd Rankin,2017-12-05,2019-05-21,13,"[batting, bowling]"
4,Gary Wilson,2012-07-05,2019-05-21,19,[batting]
5,Graham Hume,2024-03-07,2024-03-12,2,"[batting, bowling]"
6,Jacob Mulder,2017-03-15,2017-03-24,3,"[batting, bowling]"
7,James Cameron-Dow,2019-03-02,2019-03-10,4,"[batting, bowling]"
8,James Hall,2010-07-03,2010-07-03,1,[batting]
9,John Anderson,2016-07-12,2016-07-12,1,[batting]


In [24]:
# Search the complete Cricsheet registry for possible name matches.

surname_search = {
    "Ben Calitz": ["Calitz"],
    "James Cameron-Dow": ["Cameron", "Dow"],
    "James Hall": ["Hall"],
    "John Anderson": ["Anderson"],
    "Jordan Neill": ["Neill"]
}

registry_search_results = []

for crickrida_name, terms in surname_search.items():

    for term in terms:
        matches = player_registry_df[
            player_registry_df["player_name"].str.contains(
                term,
                case=False,
                na=False,
                regex=False
            )
        ][["player_name", "player_id"]].drop_duplicates()

        for _, row in matches.iterrows():
            registry_search_results.append({
                "crickrida_name": crickrida_name,
                "search_term": term,
                "cricsheet_name": row["player_name"],
                "player_id": row["player_id"]
            })

registry_search_df = pd.DataFrame(registry_search_results)

if registry_search_df.empty:
    print("No surname matches found in the Cricsheet registry.")
else:
    registry_search_df = registry_search_df.drop_duplicates()
    display(registry_search_df.sort_values(
        ["crickrida_name", "cricsheet_name"]
    ))

,crickrida_name,search_term,cricsheet_name,player_id
2,James Cameron-Dow,Dow,AJT Dowdalls,f06852e3
1,James Cameron-Dow,Cameron,JE Cameron,5ebc3941
3,James Cameron-Dow,Dow,LR Down,19147fe0
5,James Cameron-Dow,Dow,MP O'Dowd,b699df47
7,James Cameron-Dow,Dow,RT Widows,108a0d3c
0,James Cameron-Dow,Cameron,SJ Cameron,1a8ad45c
4,James Cameron-Dow,Dow,SO Dowrich,dc2f959a
6,James Cameron-Dow,Dow,TM Dowlin,24ecc358
11,James Hall,Hall,AJ Hall,6b0c3a20
10,James Hall,Hall,BM Halliday,efb6cb2a


In [25]:
candidate_ids = {
    "JE Cameron": "5ebc3941",
    "AJ Hall": "6b0c3a20",
    "J Anderson": "d5bbb6a2",
    "J Neill": "8bb34d86"
}

candidate_history = (
    opponent_cricsheet[
        opponent_cricsheet["source_player_id"].isin(
            candidate_ids.values()
        )
    ]
    .groupby(["player_name", "source_player_id", "team"])
    .agg(
        first_appearance=("date", "min"),
        last_appearance=("date", "max"),
        matches=("match_id", "nunique"),
        roles=("role", lambda x: sorted(x.unique()))
    )
    .reset_index()
    .sort_values("player_name")
)

display(candidate_history)

,player_name,source_player_id,team,first_appearance,last_appearance,matches,roles
0,AJ Hall,6b0c3a20,South Africa,2003-02-22,2007-07-01,54,"[batting, bowling]"
1,J Anderson,d5bbb6a2,Ireland,2014-09-08,2016-09-27,6,"[batting, bowling]"
2,J Neill,8bb34d86,Ireland,2025-05-25,2025-05-25,1,[bowling]


In [26]:
# Find Jordan Neill's only Cricsheet match.
j_neill_match_ids = opponent_cricsheet.loc[
    opponent_cricsheet["source_player_id"] == "8bb34d86",
    "match_id"
].unique()

# Show match details.
j_neill_match_details = matches_df[
    matches_df["match_id"].astype(str).isin(
        [str(x) for x in j_neill_match_ids]
    )
]

display(j_neill_match_details)

# Show all registered players in that match.
j_neill_match_players = (
    player_registry_df[
        player_registry_df["match_id"].isin(j_neill_match_ids)
    ][["match_id", "player_name", "player_id"]]
    .drop_duplicates()
    .sort_values("player_name")
)

display(j_neill_match_players)

,match_id,date,team_1,team_2,venue,city,season,event,toss_winner,toss_decision,winner,win_by_runs,win_by_wickets,gender,match_type,team_type,result_type
950,1472513,2025-05-25,West Indies,Ireland,"Castle Avenue, Dublin",Dublin,2025,West Indies tour of Ireland,Ireland,field,West Indies,197.0,NaN,male,ODI,international,Win


,match_id,player_name,player_id
36498,1472513,A Balbirnie,6ef13460
36499,1472513,AA Jangoo,88626ed2
36500,1472513,AR McBrine,e957b38f
36501,1472513,AS Joseph,b0946605
36502,1472513,Aidan Seaver,862c0b16
36503,1472513,BA King,7fca84b7
36504,1472513,BJ McCarthy,3766056b
36505,1472513,C Carmichael,f3b158d9
36506,1472513,DA Cosker,aac9f784
36507,1472513,E Lewis,0ebfb1ad


In [27]:
# Dates to investigate
target_dates = ["2010-07-01", "2016-06-16"]

# Find all Cricsheet matches on these dates
target_matches = matches_df[
    pd.to_datetime(matches_df["date"]).dt.strftime("%Y-%m-%d").isin(
        target_dates
    )
].copy()

print("Cricsheet matches found:", len(target_matches))

display(
    target_matches[
        [
            "match_id",
            "date",
            "team_1",
            "team_2",
            "venue",
            "city",
            "winner"
        ]
    ].sort_values(["date", "match_id"])
)

# Extract all registered players from these matches
target_match_ids = target_matches["match_id"].astype(str).tolist()

target_match_players = (
    player_registry_df[
        player_registry_df["match_id"].astype(str).isin(target_match_ids)
    ][["match_id", "player_name", "player_id"]]
    .drop_duplicates()
    .sort_values(["match_id", "player_name"])
)

print("\nRegistered players in the selected matches:")
display(target_match_players)

Cricsheet matches found: 0


,match_id,date,team_1,team_2,venue,city,winner



Registered players in the selected matches:


,match_id,player_name,player_id


In [28]:
# Search wider windows around the two dates.
date_ranges = [
    ("2010-06-25", "2010-07-10"),
    ("2016-06-10", "2016-06-25")
]

for start_date, end_date in date_ranges:

    print(f"\nMatches from {start_date} to {end_date}")

    mask = (
        (pd.to_datetime(matches_df["date"]) >= start_date) &
        (pd.to_datetime(matches_df["date"]) <= end_date)
    )

    window_matches = matches_df.loc[
        mask,
        ["match_id", "date", "team_1", "team_2", "venue", "city"]
    ].sort_values("date")

    display(window_matches)

    print("Total matches:", len(window_matches))


Matches from 2010-06-25 to 2010-07-10


,match_id,date,team_1,team_2,venue,city
1628,426389,2010-06-27,England,Australia,Old Trafford,Manchester
1629,426390,2010-06-30,England,Australia,Kennington Oval,London
1630,426391,2010-07-03,England,Australia,Lord's,London
1732,450379,2010-07-07,Canada,Ireland,VRA Ground,Amstelveen
1631,426403,2010-07-08,England,Bangladesh,Trent Bridge,Nottingham
1632,426404,2010-07-10,England,Bangladesh,County Ground,Bristol


Total matches: 6

Matches from 2016-06-10 to 2016-06-25


,match_id,date,team_1,team_2,venue,city
10,1007649,2016-06-11,Zimbabwe,India,Harare Sports Club,NaN
2556,932855,2016-06-11,Australia,South Africa,"Warner Park, Basseterre",St Kitts
11,1007651,2016-06-13,Zimbabwe,India,Harare Sports Club,NaN
2557,932857,2016-06-13,West Indies,Australia,"Warner Park, Basseterre",St Kitts
12,1007653,2016-06-15,Zimbabwe,India,Harare Sports Club,NaN
2558,932859,2016-06-15,West Indies,South Africa,"Warner Park, Basseterre",St Kitts
2519,913621,2016-06-18,Ireland,Sri Lanka,"The Village, Malahide",Dublin
2559,932861,2016-06-19,Australia,South Africa,"Kensington Oval, Bridgetown",Barbados
2520,913623,2016-06-21,England,Sri Lanka,Trent Bridge,Nottingham
2560,932863,2016-06-21,West Indies,Australia,"Kensington Oval, Bridgetown",Barbados


Total matches: 12


In [29]:
john_id = "d5bbb6a2"

john_matches = player_registry_df[
    player_registry_df["player_id"] == john_id
]["match_id"].unique()

display(
    matches_df[
        matches_df["match_id"].isin(john_matches)
    ][["match_id", "date", "team_1", "team_2", "venue", "city"]]
    .sort_values("date")
)

print("John Anderson's Cricsheet match IDs:", john_matches)

,match_id,date,team_1,team_2,venue,city
2385,727935,2014-09-08,Ireland,Scotland,"The Village, Malahide",Dublin
2386,727937,2014-09-10,Ireland,Scotland,"The Village, Malahide",Dublin
2387,727939,2014-09-12,Ireland,Scotland,"The Village, Malahide",Dublin
2519,913621,2016-06-18,Ireland,Sri Lanka,"The Village, Malahide",Dublin
2563,936131,2016-09-25,South Africa,Ireland,Willowmoore Park,Benoni
2564,936133,2016-09-27,Australia,Ireland,Willowmoore Park,Benoni


John Anderson's Cricsheet match IDs: ['727935' '727937' '727939' '913621' '936131' '936133']


In [30]:
new_mapping = {
    "crickrida_player_name": "John Anderson",
    "cricsheet_player_name": "J Anderson",
    "cricsheet_player_id": "d5bbb6a2",
    "mapping_method": "verified_match_and_career",
    "confidence": "high",
    "notes": "Matched using Ireland career appearances and the 18 June 2016 Ireland vs Sri Lanka scorecard."
}

player_identity_map_updated = pd.concat(
    [player_identity_map_updated, pd.DataFrame([new_mapping])],
    ignore_index=True
)

print("Updated identity map:", player_identity_map_updated.shape)

Updated identity map: (150, 12)


In [31]:
john_mapping = player_identity_map_updated[
    player_identity_map_updated["crickrida_player_name"]
    .str.casefold()
    == "john anderson"
]

display(john_mapping)

print("Total rows:", len(player_identity_map_updated))
print(
    "Duplicate Cricsheet IDs:",
    player_identity_map_updated["cricsheet_player_id"].duplicated().sum()
)
print(
    "Duplicate Crickrida names:",
    player_identity_map_updated["crickrida_player_name"].duplicated().sum()
)

,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,normalized_name,match_status,crickrida_player_name,cricsheet_player_name,cricsheet_player_id,mapping_method,confidence,notes
149,NaN,NaN,NaN,NaN,NaN,NaN,John Anderson,J Anderson,d5bbb6a2,verified_match_and_career,high,Matched using Ireland career appearances and t...


Total rows: 150
Duplicate Cricsheet IDs: 148
Duplicate Crickrida names: 148


In [32]:
player_identity_map_updated = player_identity_map_updated[
    player_identity_map_updated["cricketiq_player_id"] != 149
].copy()

print(player_identity_map_updated.shape)

(150, 12)


In [33]:
display(player_identity_map_updated.columns.tolist())
display(player_identity_map_updated.tail(3))

['cricketiq_player_id',
 'player_id',
 'cricsheet_name',
 'crickrida_name',
 'normalized_name',
 'match_status',
 'crickrida_player_name',
 'cricsheet_player_name',
 'cricsheet_player_id',
 'mapping_method',
 'confidence',
 'notes']

,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,normalized_name,match_status,crickrida_player_name,cricsheet_player_name,cricsheet_player_id,mapping_method,confidence,notes
147,CRIQ_19d0a172,19d0a172,G Hoey,Gavin Hoey,gavinhoey,verified_alias,NaN,NaN,NaN,NaN,NaN,NaN
148,CRIQ_3bea7ea9,3bea7ea9,J Moondra,Jai Moondra,jaimoondra,verified_alias,NaN,NaN,NaN,NaN,NaN,NaN
149,NaN,NaN,NaN,NaN,NaN,NaN,John Anderson,J Anderson,d5bbb6a2,verified_match_and_career,high,Matched using Ireland career appearances and t...


In [34]:
mask = player_identity_map_updated["crickrida_player_name"].eq(
    "John Anderson"
)

player_identity_map_updated.loc[mask, [
    "cricketiq_player_id",
    "player_id",
    "cricsheet_name",
    "crickrida_name",
    "normalized_name",
    "match_status",
    "crickrida_player_name",
    "cricsheet_player_name",
    "cricsheet_player_id",
    "mapping_method",
    "confidence",
    "notes"
]] = [
    "CRIQ_d5bbb6a2",
    "d5bbb6a2",
    "J Anderson",
    "John Anderson",
    "johnanderson",
    "verified_match_and_career",
    "John Anderson",
    "J Anderson",
    "d5bbb6a2",
    "verified_match_and_career",
    "high",
    "Matched using Ireland career appearances and the 18 June 2016 Ireland vs Sri Lanka scorecard."
]

display(
    player_identity_map_updated[
        player_identity_map_updated["crickrida_player_name"].eq("John Anderson")
    ]
)

print("Shape:", player_identity_map_updated.shape)
print("Missing CricketIQ IDs:", player_identity_map_updated["cricketiq_player_id"].isna().sum())
print("Duplicate CricketIQ IDs:", player_identity_map_updated["cricketiq_player_id"].duplicated().sum())

,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,normalized_name,match_status,crickrida_player_name,cricsheet_player_name,cricsheet_player_id,mapping_method,confidence,notes
149,CRIQ_d5bbb6a2,d5bbb6a2,J Anderson,John Anderson,johnanderson,verified_match_and_career,John Anderson,J Anderson,d5bbb6a2,verified_match_and_career,high,Matched using Ireland career appearances and t...


Shape: (150, 12)
Missing CricketIQ IDs: 0
Duplicate CricketIQ IDs: 0


In [36]:
# Find existing DataFrames related to player participation
for name, obj in globals().copy().items():
    if isinstance(obj, pd.DataFrame):
        if any(word in name.lower() for word in ["participation", "appearance", "player"]):
            print(f"{name}: {obj.shape}")
            

player_registry_df: (86666, 3)
player_master_df: (3287, 5)
player_lookup: (3294, 2)
match_player_lookup: (86666, 3)
batter_participation: (1379686, 4)
non_striker_participation: (1379686, 4)
bowler_participation: (1379686, 4)
player_participation_df: (55172, 4)
player_batting_df: (45432, 12)
player_runs_validation: (5143, 3)
player_out_lookup: (86666, 3)
player_dismissal_df: (37711, 8)
player_bowling_wickets: (19220, 5)
player_bowling_runs: (31186, 7)
player_bowling_df: (31186, 13)
afg_player_batting_df: (2719, 13)
afg_player_bowling_df: (1765, 17)
player_identity_map: (106, 6)
afg_appearances: (3171, 3)
cricsheet_appearances: (76618, 7)
player_registry_final: (3635, 9)
cricsheet_batting_appearances: (45432, 9)
cricsheet_bowling_appearances: (31186, 10)
crickrida_batting_appearances: (2719, 6)
crickrida_bowling_appearances: (1765, 6)
crickrida_appearances: (4484, 7)
cr_player_history: (454, 6)
cs_player_history: (2025, 7)
player_identity_map_updated: (150, 12)
player_registry_updated: 

In [38]:
# Inspect the existing comprehensive appearance dataset
print("Columns:", cricsheet_appearances.columns.tolist())
print("Shape:", cricsheet_appearances.shape)

display(cricsheet_appearances.head())

Columns: ['date', 'team', 'player_name', 'source_player_id', 'match_id', 'role', 'source']
Shape: (76618, 7)


,date,team,player_name,source_player_id,match_id,role,source
0,2017-01-13,Australia,B Stanlake,6834d1f2,1000887,batting,Cricsheet
1,2017-01-13,Australia,CA Lynn,45eda7c8,1000887,batting,Cricsheet
2,2017-01-13,Australia,DA Warner,dcce6f09,1000887,batting,Cricsheet
3,2017-01-13,Australia,GJ Maxwell,b681e71e,1000887,batting,Cricsheet
4,2017-01-13,Australia,JP Faulkner,808f425a,1000887,batting,Cricsheet


In [39]:
# 1. Filter all Ireland appearances
ireland_appearances = cricsheet_appearances[
    cricsheet_appearances["team"].eq("Ireland")
].copy()

ireland_appearances["date"] = pd.to_datetime(
    ireland_appearances["date"]
)

# 2. Find Ireland's date range
print("IRELAND ODI COVERAGE IN CRICSHEET")
print("=" * 45)
print("Earliest appearance:", ireland_appearances["date"].min().date())
print("Latest appearance:", ireland_appearances["date"].max().date())
print("Unique matches:", ireland_appearances["match_id"].nunique())

# 3. Create unique player list using stable Cricsheet IDs
ireland_players = (
    ireland_appearances
    .groupby(["source_player_id", "player_name"], as_index=False)
    .agg(
        matches_played=("match_id", "nunique"),
        first_appearance=("date", "min"),
        last_appearance=("date", "max"),
        roles=("role", lambda x: ", ".join(sorted(set(x))))
    )
    .sort_values("player_name")
    .reset_index(drop=True)
)

# 4. Display results
print("\nALL UNIQUE IRELAND PLAYERS")
print("=" * 45)
print("Total unique players:", len(ireland_players))

display(ireland_players)

IRELAND ODI COVERAGE IN CRICSHEET
Earliest appearance: 2006-06-13
Latest appearance: 2026-10-24
Unique matches: 139

ALL UNIQUE IRELAND PLAYERS
Total unique players: 68


,source_player_id,player_name,matches_played,first_appearance,last_appearance,roles
0,6ef13460,A Balbirnie,85,2010-07-07,2026-10-24,"batting, bowling"
1,a159b802,A van der Merwe,3,2010-08-18,2011-09-20,"batting, bowling"
2,d8e79af0,AC Botha,17,2006-06-13,2011-03-11,"batting, bowling"
3,b98cdbf0,AD Poynter,6,2010-07-07,2014-09-12,batting
4,568e5a4a,AR Cusack,29,2007-06-24,2015-03-15,"batting, bowling"
...,...,...,...,...,...,...
63,70f2f922,TF van Woerkom,2,2023-09-26,2023-12-17,bowling
64,c89474d1,TJ Murtagh,31,2013-05-23,2019-07-07,"batting, bowling"
65,29b89ae8,WB Rankin,41,2007-03-15,2020-01-09,"batting, bowling"
66,c942a53b,WK McCallan,15,2006-06-13,2009-08-27,"batting, bowling"


In [40]:
# Get all Cricsheet player IDs already mapped in CricketIQ
mapped_cricsheet_ids = set(
    player_identity_map_updated["cricsheet_player_id"]
    .dropna()
    .astype(str)
)

# Extract only unmapped Ireland players
unresolved_ireland_cricsheet = ireland_players[
    ~ireland_players["source_player_id"].astype(str).isin(
        mapped_cricsheet_ids
    )
].copy()

unresolved_ireland_cricsheet = (
    unresolved_ireland_cricsheet
    .sort_values("first_appearance")
    .reset_index(drop=True)
)

print("UNRESOLVED IRELAND CRICSHEET PLAYERS")
print("=" * 50)
print("Total unresolved players:", len(unresolved_ireland_cricsheet))

display(unresolved_ireland_cricsheet)

UNRESOLVED IRELAND CRICSHEET PLAYERS
Total unresolved players: 67


,source_player_id,player_name,matches_played,first_appearance,last_appearance,roles
0,aae9ce90,JF Mooney,35,2006-06-13,2015-10-11,"batting, bowling"
1,7c63baa7,JP Bray,10,2006-06-13,2007-04-18,batting
2,d8e79af0,AC Botha,17,2006-06-13,2011-03-11,"batting, bowling"
3,243431b5,KJ O'Brien,88,2006-06-13,2021-06-07,"batting, bowling"
4,e01895bb,DT Johnston,33,2006-06-13,2013-09-06,"batting, bowling"
...,...,...,...,...,...,...
62,f3b158d9,C Carmichael,4,2025-05-21,2026-10-24,batting
63,9aedf23b,T Mayes,2,2025-05-21,2025-05-23,"batting, bowling"
64,8bb34d86,J Neill,1,2025-05-25,2025-05-25,bowling
65,3bea7ea9,J Moondra,2,2026-10-22,2026-10-24,"batting, bowling"


In [41]:
print("IDENTITY MAP SUMMARY")
print("=" * 45)

print("Total map rows:", len(player_identity_map_updated))

print("\nRows with Cricsheet IDs:",
      player_identity_map_updated["cricsheet_player_id"].notna().sum())

print("Rows with player_id:",
      player_identity_map_updated["player_id"].notna().sum())

print("Rows with Crickrida names:",
      player_identity_map_updated["crickrida_name"].notna().sum())

print("Rows with Cricsheet names:",
      player_identity_map_updated["cricsheet_name"].notna().sum())

print("\nMapping status distribution:")
display(player_identity_map_updated["match_status"].value_counts(dropna=False))

IDENTITY MAP SUMMARY
Total map rows: 150

Rows with Cricsheet IDs: 1
Rows with player_id: 150
Rows with Crickrida names: 150
Rows with Cricsheet names: 150

Mapping status distribution:


match_status
exact_name_match             98
verified_alias               51
verified_match_and_career     1
Name: count, dtype: int64

In [42]:
# Use player_id as the primary Cricsheet ID column
mapped_cricsheet_ids = set(
    player_identity_map_updated["player_id"]
    .dropna()
    .astype(str)
)

unresolved_ireland_cricsheet = ireland_players[
    ~ireland_players["source_player_id"]
    .astype(str)
    .isin(mapped_cricsheet_ids)
].copy()

unresolved_ireland_cricsheet = (
    unresolved_ireland_cricsheet
    .sort_values("first_appearance")
    .reset_index(drop=True)
)

print("IRELAND IDENTITY AUDIT")
print("=" * 45)
print("Total Ireland players:", len(ireland_players))
print("Already mapped:", len(ireland_players) - len(unresolved_ireland_cricsheet))
print("Genuinely unmapped:", len(unresolved_ireland_cricsheet))

display(unresolved_ireland_cricsheet)

IRELAND IDENTITY AUDIT
Total Ireland players: 68
Already mapped: 29
Genuinely unmapped: 39


,source_player_id,player_name,matches_played,first_appearance,last_appearance,roles
0,7c63baa7,JP Bray,10,2006-06-13,2007-04-18,batting
1,d8e79af0,AC Botha,17,2006-06-13,2011-03-11,"batting, bowling"
2,c942a53b,WK McCallan,15,2006-06-13,2009-08-27,"batting, bowling"
3,d7d744e5,PG Gillespie,2,2006-06-13,2007-04-09,batting
4,fa1aa7e7,D Langford-Smith,10,2006-06-13,2007-04-18,"batting, bowling"
5,fd67a507,DI Joyce,3,2006-06-13,2007-06-24,batting
6,94edecfe,PJK Mooney,2,2006-06-13,2007-04-03,"batting, bowling"
7,84826f48,NJ O'Brien,63,2007-03-15,2018-03-18,batting
8,ec02b798,WTS Porterfield,89,2007-03-15,2022-01-16,batting
9,29b89ae8,WB Rankin,41,2007-03-15,2020-01-09,"batting, bowling"


In [43]:
# Build full-name reference from your supplied list

ireland_full_names = {
    "JP Bray": "Jeremy Bray",
    "AC Botha": "Andre Botha",
    "WK McCallan": "Kyle McCallan",
    "PG Gillespie": "Peter Gillespie",
    "D Langford-Smith": "Dave Langford-Smith",
    "DI Joyce": "Dominick Joyce",
    "PJK Mooney": "Paul Mooney",
    "NJ O'Brien": "Niall O'Brien",
    "WTS Porterfield": "William Porterfield",
    "WB Rankin": "Boyd Rankin",
    "EJG Morgan": "Eoin Morgan",
    "KED Carroll": "Kenny Carroll",
    "RK Whelan": "Roger Whelan",
    "GC Wilson": "Gary Wilson",
    "MJ Fourie": "Thinus Fourie",
    "PC Connell": "Peter Connell",
    "RM West": "Regan West",
    "A van der Merwe": "Albert van der Merwe",
    "JNK Shannon": "James Shannon",
    "MC Sorensen": "Max Sorensen",
    "EJ Richardson": "Eddie Richardson",
    "AR McBrine": "Andy McBrine",
    "SW Poynter": "Stuart Poynter",
    "GJ McCarter": "Graeme McCarter",
    "BJ McCarthy": "Barry McCarthy",
    "PKD Chase": "Peter Chase",
    "SP Terry": "Sean Terry",
    "J Mulder": "Jacob Mulder",
    "SC Getkate": "Shane Getkate",
    "NA Rock": "Neil Rock",
    "GI Hume": "Graham Hume",
    "ST Doheny": "Stephen Doheny",
    "M Commins": "Murray Commins",
    "MJ Humphreys": "Matthew Humphreys",
    "B White": "Ben White",
    "F Hand": "Fionn Hand",
    "T Mayes": "Thomas Mayes",
    "L McCarthy": "Liam McCarthy",
    "J Neill": "Jordan Neill"
}

unresolved_ireland_cricsheet["full_name"] = (
    unresolved_ireland_cricsheet["player_name"]
    .map(ireland_full_names)
)

# Compare full names against Crickrida player history
crickrida_names = set(
    cr_player_history["player_name"].dropna().astype(str)
)

unresolved_ireland_cricsheet["exact_full_name_match"] = (
    unresolved_ireland_cricsheet["full_name"].isin(crickrida_names)
)

display(
    unresolved_ireland_cricsheet[
        [
            "player_name",
            "full_name",
            "source_player_id",
            "exact_full_name_match"
        ]
    ]
)

,player_name,full_name,source_player_id,exact_full_name_match
0,JP Bray,Jeremy Bray,7c63baa7,False
1,AC Botha,Andre Botha,d8e79af0,False
2,WK McCallan,Kyle McCallan,c942a53b,False
3,PG Gillespie,Peter Gillespie,d7d744e5,False
4,D Langford-Smith,Dave Langford-Smith,fa1aa7e7,False
5,DI Joyce,Dominick Joyce,fd67a507,False
6,PJK Mooney,Paul Mooney,94edecfe,False
7,NJ O'Brien,Niall O'Brien,84826f48,True
8,WTS Porterfield,William Porterfield,ec02b798,True
9,WB Rankin,Boyd Rankin,29b89ae8,True


In [44]:
# Inspect existing mappings for the 14 exact full-name candidates

exact_candidates = unresolved_ireland_cricsheet[
    unresolved_ireland_cricsheet["exact_full_name_match"]
].copy()

candidate_ids = set(
    exact_candidates["source_player_id"].astype(str)
)

existing_mappings = player_identity_map_updated[
    player_identity_map_updated["player_id"]
    .astype(str)
    .isin(candidate_ids)
]

print("Already present in identity map:", len(existing_mappings))
display(existing_mappings)

print("\nExact candidates not yet mapped:")
new_candidates = exact_candidates[
    ~exact_candidates["source_player_id"]
    .astype(str)
    .isin(set(player_identity_map_updated["player_id"].astype(str)))
]

display(new_candidates)

Already present in identity map: 0


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,normalized_name,match_status,crickrida_player_name,cricsheet_player_name,cricsheet_player_id,mapping_method,confidence,notes



Exact candidates not yet mapped:


,source_player_id,player_name,matches_played,first_appearance,last_appearance,roles,full_name,exact_full_name_match
7,84826f48,NJ O'Brien,63,2007-03-15,2018-03-18,batting,Niall O'Brien,True
8,ec02b798,WTS Porterfield,89,2007-03-15,2022-01-16,batting,William Porterfield,True
9,29b89ae8,WB Rankin,41,2007-03-15,2020-01-09,"batting, bowling",Boyd Rankin,True
10,d2a6c0e6,EJG Morgan,11,2007-03-15,2009-04-01,batting,Eoin Morgan,True
13,05de1a5a,GC Wilson,57,2007-06-23,2019-05-15,batting,Gary Wilson,True
21,e957b38f,AR McBrine,72,2014-09-08,2026-10-24,"batting, bowling",Andy McBrine,True
22,cdf59953,SW Poynter,7,2014-09-10,2018-01-13,batting,Stuart Poynter,True
24,3766056b,BJ McCarthy,35,2016-06-18,2025-05-25,"batting, bowling",Barry McCarthy,True
25,8c2aa0f9,PKD Chase,13,2016-08-18,2018-01-18,"batting, bowling",Peter Chase,True
26,7f87204d,SP Terry,2,2016-09-25,2016-09-27,batting,Sean Terry,True


In [46]:
print("Columns in cr_player_history:")
print(cr_player_history.columns.tolist())

print("\nSample records:")
display(cr_player_history.head(10))

Columns in cr_player_history:
['team', 'normalized_name', 'player_name', 'crickrida_first_date', 'crickrida_last_date', 'crickrida_matches']

Sample records:


,team,normalized_name,player_name,crickrida_first_date,crickrida_last_date,crickrida_matches
0,Afghanistan,abdulmalik,Abdul Malik,2024-09-22,2024-12-21,3
1,Afghanistan,abdulrahmanafghanistan2023,"Abdul Rahman (Afghanistan, 2023)",2023-07-11,2023-08-24,3
2,Afghanistan,afsarzazai,Afsar Zazai,2015-01-10,2017-06-11,9
3,Afghanistan,aftabalam,Aftab Alam,2014-07-22,2019-06-22,22
4,Afghanistan,amghazanfar,AM Ghazanfar,2024-03-07,2026-10-23,24
5,Afghanistan,amirhamza,Amir Hamza,2012-02-10,2017-06-11,25
6,Afghanistan,asgharafghan,Asghar Afghan,2010-07-03,2021-01-26,77
7,Afghanistan,azmatullahomarzai,Azmatullah Omarzai,2021-01-21,2026-10-23,49
8,Afghanistan,bashirahmad,Bashir Ahmad,2025-10-08,2025-10-11,2
9,Afghanistan,bilalsami,Bilal Sami,2024-12-21,2026-10-23,5


In [47]:
# 1. Select the 14 exact-name candidates
candidate_names = exact_candidates["full_name"].tolist()

# 2. Retrieve their Crickrida career histories
candidate_histories = cr_player_history[
    cr_player_history["player_name"].isin(candidate_names)
].copy()

# 3. Merge with Cricsheet candidate information
candidate_comparison = exact_candidates.merge(
    candidate_histories,
    left_on="full_name",
    right_on="player_name",
    how="left",
    suffixes=("_cricsheet", "_crickrida")
)

# 4. Display relevant columns
comparison_columns = [
    "player_name_cricsheet",
    "full_name",
    "source_player_id",
    "first_appearance",
    "last_appearance",
    "matches_played",
    "team",
    "crickrida_first_date",
    "crickrida_last_date",
    "crickrida_matches"
]

display(
    candidate_comparison[comparison_columns]
    .sort_values("first_appearance")
    .reset_index(drop=True)
)

,player_name_cricsheet,full_name,source_player_id,first_appearance,last_appearance,matches_played,team,crickrida_first_date,crickrida_last_date,crickrida_matches
0,NJ O'Brien,Niall O'Brien,84826f48,2007-03-15,2018-03-18,63,Ireland,2012-07-05,2018-08-31,14
1,WTS Porterfield,William Porterfield,ec02b798,2007-03-15,2022-01-16,89,Ireland,2012-07-05,2019-05-21,25
2,WB Rankin,Boyd Rankin,29b89ae8,2007-03-15,2020-01-09,41,Ireland,2017-12-05,2019-05-21,13
3,EJG Morgan,Eoin Morgan,d2a6c0e6,2007-03-15,2009-04-01,11,England,2019-06-18,2019-06-18,1
4,GC Wilson,Gary Wilson,05de1a5a,2007-06-23,2019-05-15,57,Ireland,2012-07-05,2019-05-21,19
5,AR McBrine,Andy McBrine,e957b38f,2014-09-08,2026-10-24,72,Ireland,2016-07-12,2024-03-12,23
6,SW Poynter,Stuart Poynter,cdf59953,2014-09-10,2018-01-13,7,Ireland,2016-07-12,2019-03-10,11
7,BJ McCarthy,Barry McCarthy,3766056b,2016-06-18,2025-05-25,35,Ireland,2016-07-12,2024-03-12,12
8,PKD Chase,Peter Chase,8c2aa0f9,2016-08-18,2018-01-18,13,Ireland,2016-07-12,2018-08-31,11
9,SP Terry,Sean Terry,7f87204d,2016-09-25,2016-09-27,2,Ireland,2016-07-14,2016-07-19,3


In [48]:
# Inspect detailed Crickrida appearances for the 12 strong candidates

strong_candidate_names = [
    "Niall O'Brien",
    "William Porterfield",
    "Boyd Rankin",
    "Gary Wilson",
    "Andy McBrine",
    "Stuart Poynter",
    "Barry McCarthy",
    "Peter Chase",
    "Sean Terry",
    "Graham Hume",
    "Liam McCarthy",
    "Jordan Neill"
]

strong_candidate_records = crickrida_appearances[
    crickrida_appearances["player_name"].isin(
        strong_candidate_names
    )
].copy()

display(
    strong_candidate_records.sort_values(
        ["player_name", "date"]
    )
)

print("Total appearance records:", len(strong_candidate_records))

,date,team,player_name,source_player_id,match_id,role,source
566,2016-07-12,Ireland,Andy McBrine,CR_andymcbrine,https://crickrida.com/matches/afghanistan-irel...,batting,Crickrida
3097,2016-07-12,Ireland,Andy McBrine,CR_andymcbrine,https://crickrida.com/matches/afghanistan-irel...,bowling,Crickrida
3108,2016-07-14,Ireland,Andy McBrine,CR_andymcbrine,https://crickrida.com/matches/afghanistan-irel...,bowling,Crickrida
604,2016-07-17,Ireland,Andy McBrine,CR_andymcbrine,https://crickrida.com/matches/afghanistan-irel...,batting,Crickrida
3120,2016-07-17,Ireland,Andy McBrine,CR_andymcbrine,https://crickrida.com/matches/afghanistan-irel...,bowling,Crickrida
...,...,...,...,...,...,...,...
1333,2019-03-05,Ireland,William Porterfield,CR_williamporterfield,https://crickrida.com/matches/afghanistan-irel...,batting,Crickrida
1352,2019-03-08,Ireland,William Porterfield,CR_williamporterfield,https://crickrida.com/matches/afghanistan-irel...,batting,Crickrida
1372,2019-03-10,Ireland,William Porterfield,CR_williamporterfield,https://crickrida.com/matches/afghanistan-irel...,batting,Crickrida
1382,2019-05-19,Ireland,William Porterfield,CR_williamporterfield,https://crickrida.com/matches/afghanistan-irel...,batting,Crickrida


Total appearance records: 181


In [49]:
# Summarize Crickrida appearances for the 12 candidates

crickrida_candidate_summary = (
    strong_candidate_records
    .groupby("player_name", as_index=False)
    .agg(
        crickrida_matches=("date", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        teams=("team", lambda x: ", ".join(sorted(set(x)))),
        roles=("role", lambda x: ", ".join(sorted(set(x))))
    )
)

# Attach corresponding Cricsheet information
cricsheet_candidate_info = exact_candidates[
    exact_candidates["exact_full_name_match"]
][[
    "full_name",
    "source_player_id",
    "player_name",
    "matches_played",
    "first_appearance",
    "last_appearance"
]].rename(columns={
    "full_name": "player_name",
    "player_name": "cricsheet_name",
    "matches_played": "cricsheet_matches"
})

final_candidate_evidence = crickrida_candidate_summary.merge(
    cricsheet_candidate_info,
    on="player_name",
    how="left"
)

display(
    final_candidate_evidence.sort_values("player_name")
)

,player_name,crickrida_matches,first_date,last_date,teams,roles,source_player_id,cricsheet_name,cricsheet_matches,first_appearance,last_appearance
0,Andy McBrine,23,2016-07-12,2024-03-12,Ireland,"batting, bowling",e957b38f,AR McBrine,72,2014-09-08,2026-10-24
1,Barry McCarthy,12,2016-07-12,2024-03-12,Ireland,"batting, bowling",3766056b,BJ McCarthy,35,2016-06-18,2025-05-25
2,Boyd Rankin,13,2017-12-05,2019-05-21,Ireland,"batting, bowling",29b89ae8,WB Rankin,41,2007-03-15,2020-01-09
3,Gary Wilson,19,2012-07-05,2019-05-21,Ireland,batting,05de1a5a,GC Wilson,57,2007-06-23,2019-05-15
4,Graham Hume,2,2024-03-07,2024-03-12,Ireland,"batting, bowling",25550d06,GI Hume,19,2022-07-15,2025-02-18
5,Jordan Neill,1,2026-08-15,2026-08-15,Ireland,"batting, bowling",8bb34d86,J Neill,1,2025-05-25,2025-05-25
6,Liam McCarthy,2,2026-08-12,2026-08-15,Ireland,"batting, bowling",8eb2e2e6,L McCarthy,4,2025-05-21,2026-10-22
7,Niall O'Brien,14,2012-07-05,2018-08-31,Ireland,batting,84826f48,NJ O'Brien,63,2007-03-15,2018-03-18
8,Peter Chase,11,2016-07-12,2018-08-31,Ireland,"batting, bowling",8c2aa0f9,PKD Chase,13,2016-08-18,2018-01-18
9,Sean Terry,3,2016-07-14,2016-07-19,Ireland,batting,7f87204d,SP Terry,2,2016-09-25,2016-09-27


In [50]:
# Create candidate mapping table
approved_candidate_rows = exact_candidates[
    exact_candidates["exact_full_name_match"]
].copy()

approved_candidate_rows = approved_candidate_rows[
    [
        "full_name",
        "player_name",
        "source_player_id"
    ]
].rename(columns={
    "full_name": "crickrida_name",
    "player_name": "cricsheet_name",
    "source_player_id": "player_id"
})

# Check against existing identity map
existing_player_ids = set(
    player_identity_map_updated["player_id"].dropna().astype(str)
)

existing_crickrida_names = set(
    player_identity_map_updated["crickrida_name"]
    .dropna()
    .str.casefold()
)

approved_candidate_rows["id_exists"] = (
    approved_candidate_rows["player_id"].astype(str)
    .isin(existing_player_ids)
)

approved_candidate_rows["name_exists"] = (
    approved_candidate_rows["crickrida_name"]
    .str.casefold()
    .isin(existing_crickrida_names)
)

display(approved_candidate_rows)

,crickrida_name,cricsheet_name,player_id,id_exists,name_exists
7,Niall O'Brien,NJ O'Brien,84826f48,False,False
8,William Porterfield,WTS Porterfield,ec02b798,False,False
9,Boyd Rankin,WB Rankin,29b89ae8,False,False
10,Eoin Morgan,EJG Morgan,d2a6c0e6,False,False
13,Gary Wilson,GC Wilson,05de1a5a,False,False
21,Andy McBrine,AR McBrine,e957b38f,False,False
22,Stuart Poynter,SW Poynter,cdf59953,False,False
24,Barry McCarthy,BJ McCarthy,3766056b,False,False
25,Peter Chase,PKD Chase,8c2aa0f9,False,False
26,Sean Terry,SP Terry,7f87204d,False,False


In [51]:
strong_names = [
    "Niall O'Brien",
    "William Porterfield",
    "Boyd Rankin",
    "Gary Wilson",
    "Andy McBrine",
    "Stuart Poynter",
    "Barry McCarthy",
    "Peter Chase",
    "Graham Hume",
    "Liam McCarthy"
]

new_rows = []

for _, row in approved_candidate_rows[
    approved_candidate_rows["crickrida_name"].isin(strong_names)
].iterrows():

    player_id = str(row["player_id"])

    new_rows.append({
        "cricketiq_player_id": f"CRIQ_{player_id}",
        "player_id": player_id,
        "cricsheet_name": row["cricsheet_name"],
        "crickrida_name": row["crickrida_name"],
        "normalized_name": normalize_player_name(row["crickrida_name"]),
        "match_status": "verified_full_name_and_career",
        "crickrida_player_name": row["crickrida_name"],
        "cricsheet_player_name": row["cricsheet_name"],
        "cricsheet_player_id": player_id,
        "mapping_method": "full_name_and_career_overlap",
        "confidence": "high",
        "notes": "Matching full name, Ireland association and overlapping career dates."
    })

new_identity_rows = pd.DataFrame(new_rows)

player_identity_map_updated = pd.concat(
    [player_identity_map_updated, new_identity_rows],
    ignore_index=True
)

print("New mappings added:", len(new_identity_rows))
print("Updated identity map:", player_identity_map_updated.shape)
print(
    "Duplicate player IDs:",
    player_identity_map_updated["player_id"].duplicated().sum()
)

display(new_identity_rows)

New mappings added: 10
Updated identity map: (160, 12)
Duplicate player IDs: 0


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,normalized_name,match_status,crickrida_player_name,cricsheet_player_name,cricsheet_player_id,mapping_method,confidence,notes
0,CRIQ_84826f48,84826f48,NJ O'Brien,Niall O'Brien,niallobrien,verified_full_name_and_career,Niall O'Brien,NJ O'Brien,84826f48,full_name_and_career_overlap,high,"Matching full name, Ireland association and ov..."
1,CRIQ_ec02b798,ec02b798,WTS Porterfield,William Porterfield,williamporterfield,verified_full_name_and_career,William Porterfield,WTS Porterfield,ec02b798,full_name_and_career_overlap,high,"Matching full name, Ireland association and ov..."
2,CRIQ_29b89ae8,29b89ae8,WB Rankin,Boyd Rankin,boydrankin,verified_full_name_and_career,Boyd Rankin,WB Rankin,29b89ae8,full_name_and_career_overlap,high,"Matching full name, Ireland association and ov..."
3,CRIQ_05de1a5a,05de1a5a,GC Wilson,Gary Wilson,garywilson,verified_full_name_and_career,Gary Wilson,GC Wilson,05de1a5a,full_name_and_career_overlap,high,"Matching full name, Ireland association and ov..."
4,CRIQ_e957b38f,e957b38f,AR McBrine,Andy McBrine,andymcbrine,verified_full_name_and_career,Andy McBrine,AR McBrine,e957b38f,full_name_and_career_overlap,high,"Matching full name, Ireland association and ov..."
5,CRIQ_cdf59953,cdf59953,SW Poynter,Stuart Poynter,stuartpoynter,verified_full_name_and_career,Stuart Poynter,SW Poynter,cdf59953,full_name_and_career_overlap,high,"Matching full name, Ireland association and ov..."
6,CRIQ_3766056b,3766056b,BJ McCarthy,Barry McCarthy,barrymccarthy,verified_full_name_and_career,Barry McCarthy,BJ McCarthy,3766056b,full_name_and_career_overlap,high,"Matching full name, Ireland association and ov..."
7,CRIQ_8c2aa0f9,8c2aa0f9,PKD Chase,Peter Chase,peterchase,verified_full_name_and_career,Peter Chase,PKD Chase,8c2aa0f9,full_name_and_career_overlap,high,"Matching full name, Ireland association and ov..."
8,CRIQ_25550d06,25550d06,GI Hume,Graham Hume,grahamhume,verified_full_name_and_career,Graham Hume,GI Hume,25550d06,full_name_and_career_overlap,high,"Matching full name, Ireland association and ov..."
9,CRIQ_8eb2e2e6,8eb2e2e6,L McCarthy,Liam McCarthy,liammccarthy,verified_full_name_and_career,Liam McCarthy,L McCarthy,8eb2e2e6,full_name_and_career_overlap,high,"Matching full name, Ireland association and ov..."


In [52]:
review_names = [
    "Eoin Morgan",
    "Jacob Mulder",
    "Sean Terry",
    "Jordan Neill"
]

review_records = crickrida_appearances[
    crickrida_appearances["player_name"].isin(review_names)
].copy()

display(
    review_records.sort_values(
        ["player_name", "date"]
    )
)

print("Total records:", len(review_records))

,date,team,player_name,source_player_id,match_id,role,source
1491,2019-06-18,England,Eoin Morgan,CR_eoinmorgan,https://crickrida.com/matches/afghanistan-engl...,batting,Crickrida
807,2017-03-15,Ireland,Jacob Mulder,CR_jacobmulder,https://crickrida.com/matches/afghanistan-irel...,batting,Crickrida
3237,2017-03-15,Ireland,Jacob Mulder,CR_jacobmulder,https://crickrida.com/matches/afghanistan-irel...,bowling,Crickrida
3273,2017-03-22,Ireland,Jacob Mulder,CR_jacobmulder,https://crickrida.com/matches/afghanistan-irel...,bowling,Crickrida
877,2017-03-24,Ireland,Jacob Mulder,CR_jacobmulder,https://crickrida.com/matches/afghanistan-irel...,batting,Crickrida
3291,2017-03-24,Ireland,Jacob Mulder,CR_jacobmulder,https://crickrida.com/matches/afghanistan-irel...,bowling,Crickrida
2657,2026-08-15,Ireland,Jordan Neill,CR_jordanneill,https://crickrida.com/matches/afghanistan-irel...,batting,Crickrida
4443,2026-08-15,Ireland,Jordan Neill,CR_jordanneill,https://crickrida.com/matches/afghanistan-irel...,bowling,Crickrida
584,2016-07-14,Ireland,Sean Terry,CR_seanterry,https://crickrida.com/matches/afghanistan-irel...,batting,Crickrida
599,2016-07-17,Ireland,Sean Terry,CR_seanterry,https://crickrida.com/matches/afghanistan-irel...,batting,Crickrida


Total records: 11


In [53]:
# Inspect Ireland Cricsheet matches around Jacob Mulder's career

start_date = pd.Timestamp("2017-03-01")
end_date = pd.Timestamp("2017-03-26")

jacob_period_matches = matches_df[
    (
        (matches_df["team_1"] == "Ireland") |
        (matches_df["team_2"] == "Ireland")
    ) &
    (
        pd.to_datetime(matches_df["date"]).between(
            start_date, end_date
        )
    )
].copy()

display(
    jacob_period_matches[
        ["match_id", "date", "team_1", "team_2", "venue", "city"]
    ].sort_values("date")
)

print("Matches found:", len(jacob_period_matches))

,match_id,date,team_1,team_2,venue,city
96,1080630,2017-03-02,United Arab Emirates,Ireland,ICC Global Cricket Academy,Dubai
97,1080631,2017-03-04,United Arab Emirates,Ireland,ICC Global Cricket Academy,Dubai


Matches found: 2


In [54]:
eoin_morgan_mapping = {
    "cricketiq_player_id": "CRIQ_d2a6c0e6",
    "player_id": "d2a6c0e6",
    "cricsheet_name": "EJG Morgan",
    "crickrida_name": "Eoin Morgan",
    "normalized_name": "eoinmorgan",
    "match_status": "verified_cross_country_identity",
    "crickrida_player_name": "Eoin Morgan",
    "cricsheet_player_name": "EJG Morgan",
    "cricsheet_player_id": "d2a6c0e6",
    "mapping_method": "verified_cross_country_identity",
    "confidence": "high",
    "notes": "Same player represented Ireland and England internationally; identity approved across both international teams."
}

if "d2a6c0e6" not in player_identity_map_updated["player_id"].values:
    player_identity_map_updated = pd.concat(
        [
            player_identity_map_updated,
            pd.DataFrame([eoin_morgan_mapping])
        ],
        ignore_index=True
    )
    print("Eoin Morgan mapping added.")
else:
    print("Eoin Morgan's Cricsheet ID is already in the identity map.")

display(
    player_identity_map_updated[
        player_identity_map_updated["player_id"] == "d2a6c0e6"
    ]
)

print("Identity map shape:", player_identity_map_updated.shape)
print("Duplicate Cricsheet IDs:", player_identity_map_updated["player_id"].duplicated().sum())

Eoin Morgan mapping added.


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,normalized_name,match_status,crickrida_player_name,cricsheet_player_name,cricsheet_player_id,mapping_method,confidence,notes
160,CRIQ_d2a6c0e6,d2a6c0e6,EJG Morgan,Eoin Morgan,eoinmorgan,verified_cross_country_identity,Eoin Morgan,EJG Morgan,d2a6c0e6,verified_cross_country_identity,high,Same player represented Ireland and England in...


Identity map shape: (161, 12)
Duplicate Cricsheet IDs: 0


In [55]:
# Identify Ireland players whose Cricsheet IDs are not yet mapped
mapped_ids = set(
    player_identity_map_updated["player_id"].dropna().astype(str)
)

remaining_ireland = ireland_players[
    ~ireland_players["source_player_id"].astype(str).isin(mapped_ids)
].copy()

# Attach Crickrida history using normalized full names
remaining_ireland["normalized_name"] = (
    remaining_ireland["player_name"].apply(normalize_player_name)
)

ireland_audit = remaining_ireland.merge(
    cr_player_history[
        [
            "normalized_name",
            "player_name",
            "team",
            "crickrida_first_date",
            "crickrida_last_date",
            "crickrida_matches"
        ]
    ],
    on="normalized_name",
    how="left",
    suffixes=("_cricsheet", "_crickrida")
)

display(
    ireland_audit[
        [
            "player_name_cricsheet",
            "source_player_id",
            "matches_played",
            "first_appearance",
            "last_appearance",
            "player_name_crickrida",
            "team",
            "crickrida_first_date",
            "crickrida_last_date",
            "crickrida_matches"
        ]
    ].sort_values("first_appearance")
)

print("Remaining unmapped Ireland players:", len(remaining_ireland))
print("Exact full-name candidates:", ireland_audit["player_name_crickrida"].notna().sum())

,player_name_cricsheet,source_player_id,matches_played,first_appearance,last_appearance,player_name_crickrida,team,crickrida_first_date,crickrida_last_date,crickrida_matches
27,WK McCallan,c942a53b,15,2006-06-13,2009-08-27,NaN,NaN,NaT,NaT,NaN
1,AC Botha,d8e79af0,17,2006-06-13,2011-03-11,NaN,NaN,NaT,NaT,NaN
3,D Langford-Smith,fa1aa7e7,10,2006-06-13,2007-04-18,NaN,NaN,NaT,NaT,NaN
4,DI Joyce,fd67a507,3,2006-06-13,2007-06-24,NaN,NaN,NaT,NaT,NaN
20,PJK Mooney,94edecfe,2,2006-06-13,2007-04-03,NaN,NaN,NaT,NaT,NaN
19,PG Gillespie,d7d744e5,2,2006-06-13,2007-04-09,NaN,NaN,NaT,NaT,NaN
11,JP Bray,7c63baa7,10,2006-06-13,2007-04-18,NaN,NaN,NaT,NaT,NaN
12,KED Carroll,b3e9b7fa,3,2007-04-18,2007-06-24,NaN,NaN,NaT,NaT,NaN
21,RK Whelan,06e649a0,2,2007-06-23,2007-06-24,NaN,NaN,NaT,NaT,NaN
15,MJ Fourie,7a4819e1,2,2007-06-23,2007-06-24,NaN,NaN,NaT,NaT,NaN


Remaining unmapped Ireland players: 28
Exact full-name candidates: 0


In [57]:
# Search all Crickrida player names for likely matches
# using normalized names and fuzzy similarity.

from difflib import SequenceMatcher

crickrida_names = (
    crickrida_appearances[
        ["player_name", "source_player_id", "team"]
    ]
    .drop_duplicates()
    .copy()
)

crickrida_names["normalized_name"] = (
    crickrida_names["player_name"].apply(normalize_player_name)
)

candidate_rows = []

for _, player in remaining_ireland.iterrows():
    cs_name = player["player_name"]
    cs_norm = normalize_player_name(cs_name)

    for _, candidate in crickrida_names.iterrows():
        similarity = SequenceMatcher(
            None, cs_norm, candidate["normalized_name"]
        ).ratio()

        if similarity >= 0.55:
            candidate_rows.append({
                "cricsheet_name": cs_name,
                "cricsheet_id": player["source_player_id"],
                "crickrida_name": candidate["player_name"],
                "crickrida_id": candidate["source_player_id"],
                "crickrida_team": candidate["team"],
                "similarity": round(similarity, 3)
            })

candidate_report = pd.DataFrame(candidate_rows)

display(
    candidate_report.sort_values(
        ["cricsheet_name", "similarity"],
        ascending=[True, False]
    )
)

print("Candidate pairs:", len(candidate_report))

,cricsheet_name,cricsheet_id,crickrida_name,crickrida_id,crickrida_team,similarity
0,B White,1a448be0,Andrew White,CR_andrewwhite,Ireland,0.588
1,B White,1a448be0,Abu Hider,CRIQ_31803fac,Bangladesh,0.571
2,D Langford-Smith,fa1aa7e7,Jason Smith,CR_jasonsmith,South Africa,0.583
3,DI Joyce,fd67a507,Ed Joyce,CR_edjoyce,Ireland,0.857
4,DI Joyce,fd67a507,Dion Myers,CR_dionmyers,Zimbabwe,0.625
5,EJ Richardson,49dfe4a4,Richard Ngarava,CR_richardngarava,Zimbabwe,0.615
6,EJ Richardson,49dfe4a4,Heinrich Klaasen,CR_heinrichklaasen,South Africa,0.593
7,EJ Richardson,49dfe4a4,Rishad Hossain,CRIQ_5935d694,Bangladesh,0.560
8,GJ McCarter,6232e1ed,Jonathan Carter,CR_jonathancarter,West Indies,0.583
10,J Mulder,6b64e7a3,Jacob Mulder,CR_jacobmulder,Ireland,0.778


Candidate pairs: 36


In [58]:
approved_candidates = [
    ("DI Joyce", "Ed Joyce"),
    ("PJK Mooney", "John Mooney"),
    ("SP Terry", "Sean Terry"),
    ("J Mulder", "Jacob Mulder"),
    ("J Neill", "Jordan Neill"),
]

for cs_name, cr_name in approved_candidates:
    cs_row = ireland_players[
        ireland_players["player_name"] == cs_name
    ]

    cr_rows = crickrida_appearances[
        crickrida_appearances["player_name"] == cr_name
    ]

    print(f"\n{'=' * 65}")
    print(f"Cricsheet: {cs_name}")
    print(f"Crickrida: {cr_name}")

    if not cs_row.empty:
        print(
            "Cricsheet ID:",
            cs_row.iloc[0]["source_player_id"]
        )
        print(
            "Cricsheet period:",
            cs_row.iloc[0]["first_appearance"],
            "to",
            cs_row.iloc[0]["last_appearance"]
        )

    display(
        cr_rows[
            ["date", "team", "player_name", "source_player_id", "match_id", "role"]
        ].sort_values("date")
    )


Cricsheet: DI Joyce
Crickrida: Ed Joyce
Cricsheet ID: fd67a507
Cricsheet period: 2006-06-13 00:00:00 to 2007-06-24 00:00:00


,date,team,player_name,source_player_id,match_id,role
39,2012-07-05,Ireland,Ed Joyce,CR_edjoyce,https://crickrida.com/matches/afghanistan-irel...,batting
238,2015-01-10,Ireland,Ed Joyce,CR_edjoyce,https://crickrida.com/matches/afghanistan-irel...,batting
257,2015-01-17,Ireland,Ed Joyce,CR_edjoyce,https://crickrida.com/matches/afghanistan-irel...,batting
561,2016-07-12,Ireland,Ed Joyce,CR_edjoyce,https://crickrida.com/matches/afghanistan-irel...,batting
583,2016-07-14,Ireland,Ed Joyce,CR_edjoyce,https://crickrida.com/matches/afghanistan-irel...,batting
598,2016-07-17,Ireland,Ed Joyce,CR_edjoyce,https://crickrida.com/matches/afghanistan-irel...,batting
609,2016-07-19,Ireland,Ed Joyce,CR_edjoyce,https://crickrida.com/matches/afghanistan-irel...,batting
798,2017-03-15,Ireland,Ed Joyce,CR_edjoyce,https://crickrida.com/matches/afghanistan-irel...,batting
820,2017-03-17,Ireland,Ed Joyce,CR_edjoyce,https://crickrida.com/matches/afghanistan-irel...,batting
841,2017-03-19,Ireland,Ed Joyce,CR_edjoyce,https://crickrida.com/matches/afghanistan-irel...,batting



Cricsheet: PJK Mooney
Crickrida: John Mooney
Cricsheet ID: 94edecfe
Cricsheet period: 2006-06-13 00:00:00 to 2007-04-03 00:00:00


,date,team,player_name,source_player_id,match_id,role
6,2010-07-03,Ireland,John Mooney,CR_johnmooney,https://crickrida.com/matches/afghanistan-irel...,batting
44,2012-07-05,Ireland,John Mooney,CR_johnmooney,https://crickrida.com/matches/afghanistan-irel...,batting
2755,2012-07-05,Ireland,John Mooney,CR_johnmooney,https://crickrida.com/matches/afghanistan-irel...,bowling
243,2015-01-10,Ireland,John Mooney,CR_johnmooney,https://crickrida.com/matches/afghanistan-irel...,batting
2877,2015-01-10,Ireland,John Mooney,CR_johnmooney,https://crickrida.com/matches/afghanistan-irel...,bowling
262,2015-01-17,Ireland,John Mooney,CR_johnmooney,https://crickrida.com/matches/afghanistan-irel...,batting
2891,2015-01-17,Ireland,John Mooney,CR_johnmooney,https://crickrida.com/matches/afghanistan-irel...,bowling



Cricsheet: SP Terry
Crickrida: Sean Terry
Cricsheet ID: 7f87204d
Cricsheet period: 2016-09-25 00:00:00 to 2016-09-27 00:00:00


,date,team,player_name,source_player_id,match_id,role
584,2016-07-14,Ireland,Sean Terry,CR_seanterry,https://crickrida.com/matches/afghanistan-irel...,batting
599,2016-07-17,Ireland,Sean Terry,CR_seanterry,https://crickrida.com/matches/afghanistan-irel...,batting
610,2016-07-19,Ireland,Sean Terry,CR_seanterry,https://crickrida.com/matches/afghanistan-irel...,batting



Cricsheet: J Mulder
Crickrida: Jacob Mulder
Cricsheet ID: 6b64e7a3
Cricsheet period: 2017-03-04 00:00:00 to 2017-03-04 00:00:00


,date,team,player_name,source_player_id,match_id,role
807,2017-03-15,Ireland,Jacob Mulder,CR_jacobmulder,https://crickrida.com/matches/afghanistan-irel...,batting
3237,2017-03-15,Ireland,Jacob Mulder,CR_jacobmulder,https://crickrida.com/matches/afghanistan-irel...,bowling
3273,2017-03-22,Ireland,Jacob Mulder,CR_jacobmulder,https://crickrida.com/matches/afghanistan-irel...,bowling
877,2017-03-24,Ireland,Jacob Mulder,CR_jacobmulder,https://crickrida.com/matches/afghanistan-irel...,batting
3291,2017-03-24,Ireland,Jacob Mulder,CR_jacobmulder,https://crickrida.com/matches/afghanistan-irel...,bowling



Cricsheet: J Neill
Crickrida: Jordan Neill
Cricsheet ID: 8bb34d86
Cricsheet period: 2025-05-25 00:00:00 to 2025-05-25 00:00:00


,date,team,player_name,source_player_id,match_id,role
2657,2026-08-15,Ireland,Jordan Neill,CR_jordanneill,https://crickrida.com/matches/afghanistan-irel...,batting
4443,2026-08-15,Ireland,Jordan Neill,CR_jordanneill,https://crickrida.com/matches/afghanistan-irel...,bowling


In [59]:
approved_ireland_mappings = [
    {
        "cricsheet_name": "DI Joyce",
        "crickrida_name": "Ed Joyce",
        "player_id": "fd67a507",
        "crickrida_id": "CR_edjoyce"
    },
    {
        "cricsheet_name": "PJK Mooney",
        "crickrida_name": "John Mooney",
        "player_id": "94edecfe",
        "crickrida_id": "CR_johnmooney"
    },
    {
        "cricsheet_name": "SP Terry",
        "crickrida_name": "Sean Terry",
        "player_id": "7f87204d",
        "crickrida_id": "CR_seanterry"
    }
]

new_rows = []

for item in approved_ireland_mappings:
    if item["player_id"] in player_identity_map_updated["player_id"].astype(str).values:
        print(f"Already mapped: {item['cricsheet_name']}")
        continue

    new_rows.append({
        "cricketiq_player_id": f"CRIQ_{item['player_id']}",
        "player_id": item["player_id"],
        "cricsheet_name": item["cricsheet_name"],
        "crickrida_name": item["crickrida_name"],
        "normalized_name": normalize_player_name(item["crickrida_name"]),
        "match_status": "verified_ireland_identity",
        "crickrida_player_name": item["crickrida_name"],
        "cricsheet_player_name": item["cricsheet_name"],
        "cricsheet_player_id": item["player_id"],
        "mapping_method": "verified_full_name_and_ireland_career",
        "confidence": "high",
        "notes": "Identity approved using player name, initials and Ireland international career."
    })

if new_rows:
    player_identity_map_updated = pd.concat(
        [player_identity_map_updated, pd.DataFrame(new_rows)],
        ignore_index=True
    )

print("Mappings added:", len(new_rows))
print("Updated identity map:", player_identity_map_updated.shape)
print("Duplicate Cricsheet IDs:", player_identity_map_updated["player_id"].duplicated().sum())

Mappings added: 3
Updated identity map: (164, 12)
Duplicate Cricsheet IDs: 0


In [60]:
# Jacob Mulder and Jordan Neill: compare match context

investigations = [
    {
        "name": "Jacob Mulder",
        "cricsheet_id": "6b64e7a3",
        "crickrida_id": "CR_jacobmulder",
        "start": "2017-03-01",
        "end": "2017-03-26"
    },
    {
        "name": "Jordan Neill",
        "cricsheet_id": "8bb34d86",
        "crickrida_id": "CR_jordanneill",
        "start": "2025-05-20",
        "end": "2026-08-20"
    }
]

for item in investigations:
    print(f"\n{'=' * 70}")
    print("PLAYER:", item["name"])

    start = pd.Timestamp(item["start"])
    end = pd.Timestamp(item["end"])

    ireland_fixtures = matches_df[
        (
            (matches_df["team_1"] == "Ireland") |
            (matches_df["team_2"] == "Ireland")
        ) &
        pd.to_datetime(matches_df["date"]).between(start, end)
    ]

    print("\nCricsheet Ireland fixtures:")
    display(
        ireland_fixtures[
            ["match_id", "date", "team_1", "team_2", "venue", "city"]
        ].sort_values("date")
    )

    print("\nCrickrida player appearances:")
    display(
        crickrida_appearances[
            crickrida_appearances["source_player_id"] == item["crickrida_id"]
        ][
            ["date", "team", "player_name", "match_id", "role"]
        ].sort_values("date")
    )


PLAYER: Jacob Mulder

Cricsheet Ireland fixtures:


,match_id,date,team_1,team_2,venue,city
96,1080630,2017-03-02,United Arab Emirates,Ireland,ICC Global Cricket Academy,Dubai
97,1080631,2017-03-04,United Arab Emirates,Ireland,ICC Global Cricket Academy,Dubai



Crickrida player appearances:


,date,team,player_name,match_id,role
807,2017-03-15,Ireland,Jacob Mulder,https://crickrida.com/matches/afghanistan-irel...,batting
3237,2017-03-15,Ireland,Jacob Mulder,https://crickrida.com/matches/afghanistan-irel...,bowling
3273,2017-03-22,Ireland,Jacob Mulder,https://crickrida.com/matches/afghanistan-irel...,bowling
877,2017-03-24,Ireland,Jacob Mulder,https://crickrida.com/matches/afghanistan-irel...,batting
3291,2017-03-24,Ireland,Jacob Mulder,https://crickrida.com/matches/afghanistan-irel...,bowling



PLAYER: Jordan Neill

Cricsheet Ireland fixtures:


,match_id,date,team_1,team_2,venue,city
948,1472511,2025-05-21,Ireland,West Indies,"Castle Avenue, Dublin",Dublin
949,1472512,2025-05-23,West Indies,Ireland,"Castle Avenue, Dublin",Dublin
950,1472513,2025-05-25,West Indies,Ireland,"Castle Avenue, Dublin",Dublin



Crickrida player appearances:


,date,team,player_name,match_id,role
2657,2026-08-15,Ireland,Jordan Neill,https://crickrida.com/matches/afghanistan-irel...,batting
4443,2026-08-15,Ireland,Jordan Neill,https://crickrida.com/matches/afghanistan-irel...,bowling


In [61]:
jordan_neill_mapping = {
    "cricketiq_player_id": "CRIQ_8bb34d86",
    "player_id": "8bb34d86",
    "cricsheet_name": "J Neill",
    "crickrida_name": "Jordan Neill",
    "normalized_name": "jordanneill",
    "match_status": "verified_ireland_identity",
    "crickrida_player_name": "Jordan Neill",
    "cricsheet_player_name": "J Neill",
    "cricsheet_player_id": "8bb34d86",
    "mapping_method": "verified_full_name_and_ireland_career",
    "confidence": "high",
    "notes": "Same full name and Ireland international career; identity approved despite different recorded appearance dates."
}

if "8bb34d86" not in player_identity_map_updated["player_id"].astype(str).values:
    player_identity_map_updated = pd.concat(
        [player_identity_map_updated, pd.DataFrame([jordan_neill_mapping])],
        ignore_index=True
    )
    print("Jordan Neill mapping added.")
else:
    print("Jordan Neill is already mapped.")

print("Identity map:", player_identity_map_updated.shape)
print("Duplicate Cricsheet IDs:", player_identity_map_updated["player_id"].duplicated().sum())

Jordan Neill mapping added.
Identity map: (165, 12)
Duplicate Cricsheet IDs: 0


In [62]:
jacob_mulder_mapping = {
    "cricketiq_player_id": "CRIQ_6b64e7a3",
    "player_id": "6b64e7a3",
    "cricsheet_name": "J Mulder",
    "crickrida_name": "Jacob Mulder",
    "normalized_name": "jacobmulder",
    "match_status": "verified_external_career_and_match_records",
    "crickrida_player_name": "Jacob Mulder",
    "cricsheet_player_name": "J Mulder",
    "cricsheet_player_id": "6b64e7a3",
    "mapping_method": "verified_full_name_and_odi_career",
    "confidence": "high",
    "notes": "Verified using ODI debut against UAE on 2017-03-04 and Afghanistan ODI appearances on 2017-03-15, 2017-03-22 and 2017-03-24."
}

if "6b64e7a3" not in player_identity_map_updated["player_id"].astype(str).values:
    player_identity_map_updated = pd.concat(
        [player_identity_map_updated, pd.DataFrame([jacob_mulder_mapping])],
        ignore_index=True
    )
    print("Jacob Mulder mapping added.")
else:
    print("Jacob Mulder is already mapped.")

print("Identity map:", player_identity_map_updated.shape)
print("Duplicate Cricsheet IDs:", player_identity_map_updated["player_id"].duplicated().sum())

Jacob Mulder mapping added.
Identity map: (166, 12)
Duplicate Cricsheet IDs: 0


In [63]:
mapped_ids = set(
    player_identity_map_updated["player_id"].dropna().astype(str)
)

remaining_ireland = ireland_players[
    ~ireland_players["source_player_id"].astype(str).isin(mapped_ids)
].copy()

display(
    remaining_ireland[
        [
            "player_name",
            "source_player_id",
            "matches_played",
            "first_appearance",
            "last_appearance"
        ]
    ].sort_values("first_appearance")
)

print("Remaining unmapped Ireland players:", len(remaining_ireland))
print("Total mapped identities:", len(player_identity_map_updated))

,player_name,source_player_id,matches_played,first_appearance,last_appearance
66,WK McCallan,c942a53b,15,2006-06-13,2009-08-27
2,AC Botha,d8e79af0,17,2006-06-13,2011-03-11
13,D Langford-Smith,fa1aa7e7,10,2006-06-13,2007-04-18
49,PG Gillespie,d7d744e5,2,2006-06-13,2007-04-09
35,JP Bray,7c63baa7,10,2006-06-13,2007-04-18
36,KED Carroll,b3e9b7fa,3,2007-04-18,2007-06-24
54,RK Whelan,06e649a0,2,2007-06-23,2007-06-24
42,MJ Fourie,7a4819e1,2,2007-06-23,2007-06-24
48,PC Connell,5500c675,4,2008-07-31,2010-06-17
55,RM West,d1761087,2,2009-04-01,2009-08-27


Remaining unmapped Ireland players: 23
Total mapped identities: 166


In [64]:
from difflib import SequenceMatcher
import re
import pandas as pd

# --------------------------------------------------
# 1. Prepare Crickrida player identity history
# --------------------------------------------------

crickrida_players = (
    crickrida_appearances
    .groupby(
        ["source_player_id", "player_name", "team"],
        dropna=False
    )
    .agg(
        first_date=("date", "min"),
        last_date=("date", "max"),
        appearances=("date", "nunique"),
        roles=("role", lambda x: ", ".join(sorted(set(x.dropna()))))
    )
    .reset_index()
)

crickrida_players["normalized_name"] = (
    crickrida_players["player_name"].apply(normalize_player_name)
)

# --------------------------------------------------
# 2. Name similarity and surname comparison
# --------------------------------------------------

def get_last_name(name):
    parts = re.sub(r"[^a-zA-Z ]", " ", str(name)).split()
    return parts[-1].lower() if parts else ""

candidate_rows = []

for _, cs in remaining_ireland.iterrows():

    cs_name = cs["player_name"]
    cs_norm = normalize_player_name(cs_name)
    cs_surname = get_last_name(cs_name)

    for _, cr in crickrida_players.iterrows():

        cr_name = cr["player_name"]
        cr_norm = cr["normalized_name"]
        cr_surname = get_last_name(cr_name)

        similarity = SequenceMatcher(
            None, cs_norm, cr_norm
        ).ratio()

        same_surname = (
            cs_surname != "" and cs_surname == cr_surname
        )

        # Retain useful name candidates
        if similarity >= 0.45 or same_surname:

            candidate_rows.append({
                "cricsheet_name": cs_name,
                "cricsheet_id": cs["source_player_id"],
                "cs_matches": cs["matches_played"],
                "cs_first_date": cs["first_appearance"],
                "cs_last_date": cs["last_appearance"],
                "crickrida_name": cr_name,
                "crickrida_id": cr["source_player_id"],
                "crickrida_team": cr["team"],
                "cr_first_date": cr["first_date"],
                "cr_last_date": cr["last_date"],
                "cr_appearances": cr["appearances"],
                "cr_roles": cr["roles"],
                "similarity": round(similarity, 3),
                "same_surname": same_surname
            })

candidate_report_23 = pd.DataFrame(candidate_rows)

# --------------------------------------------------
# 3. Prioritize Ireland candidates and strong matches
# --------------------------------------------------

candidate_report_23["priority"] = (
    candidate_report_23["crickrida_team"].eq("Ireland").astype(int)
)

candidate_report_23 = candidate_report_23.sort_values(
    ["cricsheet_name", "priority", "similarity"],
    ascending=[True, False, False]
)

print("REMAINING CRICSHEET IRELAND PLAYERS:", len(remaining_ireland))
print("TOTAL CANDIDATE PAIRS:", len(candidate_report_23))

display(
    candidate_report_23[
        [
            "cricsheet_name",
            "cricsheet_id",
            "cs_first_date",
            "cs_last_date",
            "crickrida_name",
            "crickrida_id",
            "crickrida_team",
            "cr_first_date",
            "cr_last_date",
            "cr_appearances",
            "cr_roles",
            "similarity",
            "same_surname"
        ]
    ]
)

# --------------------------------------------------
# 4. Show players with no candidate
# --------------------------------------------------

players_with_candidates = set(
    candidate_report_23["cricsheet_id"].astype(str)
)

players_without_candidates = remaining_ireland[
    ~remaining_ireland["source_player_id"].astype(str).isin(
        players_with_candidates
    )
]

print("\nPLAYERS WITH NO NAME CANDIDATES:")
display(
    players_without_candidates[
        [
            "player_name",
            "source_player_id",
            "matches_played",
            "first_appearance",
            "last_appearance"
        ]
    ]
)

# --------------------------------------------------
# 5. Save audit report for reference
# --------------------------------------------------

candidate_report_23.to_csv(
    "ireland_remaining_identity_candidates.csv",
    index=False
)

print("\nAudit report saved: ireland_remaining_identity_candidates.csv")

REMAINING CRICSHEET IRELAND PLAYERS: 23
TOTAL CANDIDATE PAIRS: 182


,cricsheet_name,cricsheet_id,cs_first_date,cs_last_date,crickrida_name,crickrida_id,crickrida_team,cr_first_date,cr_last_date,cr_appearances,cr_roles,similarity,same_surname
8,A van der Merwe,a159b802,2010-08-18,2011-09-20,Sean Terry,CR_seanterry,Ireland,2016-07-14,2016-07-19,3,batting,0.476,False
1,A van der Merwe,a159b802,2010-08-18,2011-09-20,Cade Carmichael,CR_cadecarmichael,Ireland,2026-08-07,2026-08-15,4,batting,0.462,False
9,A van der Merwe,a159b802,2010-08-18,2011-09-20,Theo van Woerkom,CR_theovanwoerkom,Ireland,2024-03-07,2024-03-12,2,"batting, bowling",0.462,False
6,A van der Merwe,a159b802,2010-08-18,2011-09-20,Rassie van der Dussen,CR_rassievanderdussen,South Africa,2023-11-10,2025-02-21,2,batting,0.533,False
2,A van der Merwe,a159b802,2010-08-18,2011-09-20,David Warner,CR_davidwarner,Australia,2012-08-25,2023-11-07,4,batting,0.522,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
180,WK McCallan,c942a53b,2006-06-13,2009-08-27,Mitchell Santner,CR_mitchellsantner,New Zealand,2023-10-18,2023-10-18,1,"batting, bowling",0.480,False
175,WK McCallan,c942a53b,2006-06-13,2009-08-27,Mahmudullah,CRIQ_bf1d7d3e,Bangladesh,2015-02-18,2024-11-11,14,"batting, bowling",0.476,False
176,WK McCallan,c942a53b,2006-06-13,2009-08-27,Al Islam,CR_alislam,Bangladesh,2026-10-21,2026-10-21,1,"batting, bowling",0.471,False
178,WK McCallan,c942a53b,2006-06-13,2009-08-27,Ikram Alikhil,CR_ikramalikhil,Afghanistan,2019-03-02,2025-10-14,23,batting,0.455,False



PLAYERS WITH NO NAME CANDIDATES:


,player_name,source_player_id,matches_played,first_appearance,last_appearance



Audit report saved: ireland_remaining_identity_candidates.csv


In [65]:
focused_candidates = candidate_report_23[
    (
        candidate_report_23["crickrida_team"] == "Ireland"
    ) |
    (
        candidate_report_23["similarity"] >= 0.70
    )
].copy()

focused_candidates = focused_candidates.sort_values(
    ["cricsheet_name", "crickrida_team", "similarity"],
    ascending=[True, True, False]
)

display(
    focused_candidates[
        [
            "cricsheet_name",
            "cricsheet_id",
            "cs_first_date",
            "cs_last_date",
            "crickrida_name",
            "crickrida_id",
            "crickrida_team",
            "cr_first_date",
            "cr_last_date",
            "cr_appearances",
            "similarity"
        ]
    ]
)

print("Focused candidates:", len(focused_candidates))
print(
    "Ireland candidates:",
    (focused_candidates["crickrida_team"] == "Ireland").sum()
)

,cricsheet_name,cricsheet_id,cs_first_date,cs_last_date,crickrida_name,crickrida_id,crickrida_team,cr_first_date,cr_last_date,cr_appearances,similarity
8,A van der Merwe,a159b802,2010-08-18,2011-09-20,Sean Terry,CR_seanterry,Ireland,2016-07-14,2016-07-19,3,0.476
1,A van der Merwe,a159b802,2010-08-18,2011-09-20,Cade Carmichael,CR_cadecarmichael,Ireland,2026-08-07,2026-08-15,4,0.462
9,A van der Merwe,a159b802,2010-08-18,2011-09-20,Theo van Woerkom,CR_theovanwoerkom,Ireland,2024-03-07,2024-03-12,2,0.462
20,B White,1a448be0,2023-06-19,2023-06-21,Andrew White,CR_andrewwhite,Ireland,2010-07-03,2010-07-03,1,0.588
22,B White,1a448be0,2023-06-19,2023-06-21,Josh Little,CR_joshlittle,Ireland,2021-01-26,2021-01-26,1,0.500
45,EJ Richardson,49dfe4a4,2013-09-06,2013-09-06,Peter Chase,CR_peterchase,Ireland,2016-07-12,2018-08-31,11,0.545
54,EJ Richardson,49dfe4a4,2013-09-06,2013-09-06,Trent Johnston,CR_trentjohnston,Ireland,2010-07-03,2012-07-05,2,0.480
37,EJ Richardson,49dfe4a4,2013-09-06,2013-09-06,Cade Carmichael,CR_cadecarmichael,Ireland,2026-08-07,2026-08-15,4,0.462
38,EJ Richardson,49dfe4a4,2013-09-06,2013-09-06,Gary Wilson,CR_garywilson,Ireland,2012-07-05,2019-05-21,19,0.455
63,F Hand,e3fe4b49,2024-10-07,2024-10-07,John Anderson,CR_johnanderson,Ireland,2016-07-12,2016-07-12,1,0.471


Focused candidates: 28
Ireland candidates: 28


In [66]:
ireland_manual_review = remaining_ireland[
    [
        "player_name",
        "source_player_id",
        "matches_played",
        "first_appearance",
        "last_appearance"
    ]
].copy()

ireland_manual_review = ireland_manual_review.rename(
    columns={
        "player_name": "cricsheet_name",
        "source_player_id": "cricsheet_player_id"
    }
)

ireland_manual_review["crickrida_name"] = None
ireland_manual_review["crickrida_player_id"] = None
ireland_manual_review["verification_status"] = "unverified"
ireland_manual_review["confidence"] = "not_assessed"
ireland_manual_review["verification_notes"] = None

ireland_manual_review.to_csv(
    "ireland_manual_identity_review.csv",
    index=False
)

display(ireland_manual_review)

print("Manual review records:", len(ireland_manual_review))
print("All statuses:", ireland_manual_review["verification_status"].value_counts())

,cricsheet_name,cricsheet_player_id,matches_played,first_appearance,last_appearance,crickrida_name,crickrida_player_id,verification_status,confidence,verification_notes
1,A van der Merwe,a159b802,3,2010-08-18,2011-09-20,None,None,unverified,not_assessed,None
2,AC Botha,d8e79af0,17,2006-06-13,2011-03-11,None,None,unverified,not_assessed,None
7,B White,1a448be0,2,2023-06-19,2023-06-21,None,None,unverified,not_assessed,None
13,D Langford-Smith,fa1aa7e7,10,2006-06-13,2007-04-18,None,None,unverified,not_assessed,None
17,EJ Richardson,49dfe4a4,1,2013-09-06,2013-09-06,None,None,unverified,not_assessed,None
19,F Hand,e3fe4b49,1,2024-10-07,2024-10-07,None,None,unverified,not_assessed,None
25,GJ McCarter,6232e1ed,1,2014-09-12,2014-09-12,None,None,unverified,not_assessed,None
34,JNK Shannon,6e161a33,1,2013-05-26,2013-05-26,None,None,unverified,not_assessed,None
35,JP Bray,7c63baa7,10,2006-06-13,2007-04-18,None,None,unverified,not_assessed,None
36,KED Carroll,b3e9b7fa,3,2007-04-18,2007-06-24,None,None,unverified,not_assessed,None


Manual review records: 23
All statuses: verification_status
unverified    23
Name: count, dtype: int64


In [67]:
# First batch: older Ireland players
batch_1_ids = [
    "d8e79af0",  # AC Botha
    "fa1aa7e7",  # D Langford-Smith
    "7c63baa7",  # JP Bray
    "d7d744e5",  # PG Gillespie
    "c942a53b"   # WK McCallan
]

batch_1_players = ireland_manual_review[
    ireland_manual_review["cricsheet_player_id"].isin(batch_1_ids)
].copy()

for _, player in batch_1_players.iterrows():

    player_id = player["cricsheet_player_id"]
    player_name = player["cricsheet_name"]

    print("\n" + "=" * 70)
    print("PLAYER:", player_name)
    print("CRICSHEET ID:", player_id)

    appearances = cricsheet_appearances[
        cricsheet_appearances["source_player_id"].astype(str) == player_id
    ].copy()

    display(
        appearances[
            ["date", "team", "player_name", "match_id", "role"]
        ].sort_values("date")
    )

    print("Total appearances:", len(appearances))
    print("Teams:", appearances["team"].dropna().unique())
    print("Roles:", appearances["role"].value_counts().to_dict())

    # Search Crickrida Ireland records for surname-related possibilities
    surname = player_name.split()[-1].lower()

    possible_records = crickrida_appearances[
        (
            crickrida_appearances["team"] == "Ireland"
        ) &
        (
            crickrida_appearances["player_name"]
            .str.lower()
            .str.contains(surname, na=False)
        )
    ]

    print("\nPossible Crickrida Ireland records:")
    display(
        possible_records[
            ["date", "team", "player_name", "source_player_id", "role"]
        ].drop_duplicates().sort_values("date")
    )


PLAYER: AC Botha
CRICSHEET ID: d8e79af0


,date,team,player_name,match_id,role
20298,2006-06-13,Ireland,AC Botha,225171,batting
21681,2007-03-15,Ireland,AC Botha,247461,batting
60151,2007-03-15,Ireland,AC Botha,247461,bowling
21761,2007-03-17,Ireland,AC Botha,247465,batting
60197,2007-03-17,Ireland,AC Botha,247465,bowling
21959,2007-03-23,Ireland,AC Botha,247477,batting
60350,2007-03-23,Ireland,AC Botha,247477,bowling
22086,2007-03-30,Ireland,AC Botha,247484,batting
60431,2007-03-30,Ireland,AC Botha,247484,bowling
22139,2007-04-03,Ireland,AC Botha,247488,batting


Total appearances: 30
Teams: ['Ireland']
Roles: {'batting': 17, 'bowling': 13}

Possible Crickrida Ireland records:


,date,team,player_name,source_player_id,role



PLAYER: D Langford-Smith
CRICSHEET ID: fa1aa7e7


,date,team,player_name,match_id,role
20300,2006-06-13,Ireland,D Langford-Smith,225171,batting
59187,2006-06-13,Ireland,D Langford-Smith,225171,bowling
60153,2007-03-15,Ireland,D Langford-Smith,247461,bowling
21683,2007-03-15,Ireland,D Langford-Smith,247461,batting
60198,2007-03-17,Ireland,D Langford-Smith,247465,bowling
60352,2007-03-23,Ireland,D Langford-Smith,247477,bowling
21961,2007-03-23,Ireland,D Langford-Smith,247477,batting
22088,2007-03-30,Ireland,D Langford-Smith,247484,batting
60433,2007-03-30,Ireland,D Langford-Smith,247484,bowling
22141,2007-04-03,Ireland,D Langford-Smith,247488,batting


... (10 outputs left out)


PLAYER: WK McCallan
CRICSHEET ID: c942a53b


,date,team,player_name,match_id,role
20308,2006-06-13,Ireland,WK McCallan,225171,batting
59192,2006-06-13,Ireland,WK McCallan,225171,bowling
60157,2007-03-15,Ireland,WK McCallan,247461,bowling
21689,2007-03-15,Ireland,WK McCallan,247461,batting
60202,2007-03-17,Ireland,WK McCallan,247465,bowling
21768,2007-03-17,Ireland,WK McCallan,247465,batting
60356,2007-03-23,Ireland,WK McCallan,247477,bowling
21967,2007-03-23,Ireland,WK McCallan,247477,batting
60437,2007-03-30,Ireland,WK McCallan,247484,bowling
22095,2007-03-30,Ireland,WK McCallan,247484,batting


Total appearances: 27
Teams: ['Ireland']
Roles: {'bowling': 14, 'batting': 13}

Possible Crickrida Ireland records:


,date,team,player_name,source_player_id,role


In [68]:
verified_batch_1 = {
    "d8e79af0": {
        "full_name": "Andre Cornelius Botha",
        "crickrida_name": "Andre Botha",
        "crickrida_id": "CR_andrewwhite",  # Replace only if profile ID differs
        "notes": "Confirmed in Ireland vs England ODI, 13 June 2006. Scored 52 off 89 balls."
    },
    "fa1aa7e7": {
        "full_name": "David Langford-Smith",
        "crickrida_name": "Dave Langford-Smith",
        "crickrida_id": None,
        "notes": "Confirmed in Ireland vs England ODI, 13 June 2006. Took 3/63."
    },
    "7c63baa7": {
        "full_name": "Jeremy Paul Bray",
        "crickrida_name": "Jeremy Bray",
        "crickrida_id": None,
        "notes": "Confirmed in Ireland vs England ODI, 13 June 2006. Scored 22 off 29 balls."
    },
    "d7d744e5": {
        "full_name": "Peter Gerard Gillespie",
        "crickrida_name": "Peter Gillespie",
        "crickrida_id": None,
        "notes": "Confirmed in Ireland vs England ODI, 13 June 2006. Scored 0 off 3 balls."
    }
}

for player_id, evidence in verified_batch_1.items():
    mask = ireland_manual_review["cricsheet_player_id"] == player_id

    ireland_manual_review.loc[mask, "crickrida_name"] = evidence["crickrida_name"]
    ireland_manual_review.loc[mask, "crickrida_player_id"] = evidence["crickrida_id"]
    ireland_manual_review.loc[mask, "verification_status"] = "verified"
    ireland_manual_review.loc[mask, "confidence"] = "high"
    ireland_manual_review.loc[mask, "verification_notes"] = (
        evidence["full_name"] + ". " + evidence["notes"]
    )

ireland_manual_review.to_csv(
    "ireland_manual_identity_review.csv",
    index=False
)

display(
    ireland_manual_review[
        ireland_manual_review["cricsheet_player_id"].isin(verified_batch_1)
        | (ireland_manual_review["cricsheet_player_id"] == "c942a53b")
    ]
)

print(ireland_manual_review["verification_status"].value_counts())

,cricsheet_name,cricsheet_player_id,matches_played,first_appearance,last_appearance,crickrida_name,crickrida_player_id,verification_status,confidence,verification_notes
2,AC Botha,d8e79af0,17,2006-06-13,2011-03-11,Andre Botha,CR_andrewwhite,verified,high,Andre Cornelius Botha. Confirmed in Ireland vs...
13,D Langford-Smith,fa1aa7e7,10,2006-06-13,2007-04-18,Dave Langford-Smith,None,verified,high,David Langford-Smith. Confirmed in Ireland vs ...
35,JP Bray,7c63baa7,10,2006-06-13,2007-04-18,Jeremy Bray,None,verified,high,Jeremy Paul Bray. Confirmed in Ireland vs Engl...
49,PG Gillespie,d7d744e5,2,2006-06-13,2007-04-09,Peter Gillespie,None,verified,high,Peter Gerard Gillespie. Confirmed in Ireland v...
66,WK McCallan,c942a53b,15,2006-06-13,2009-08-27,None,None,unverified,not_assessed,None


verification_status
unverified    19
verified       4
Name: count, dtype: int64


In [69]:
batch_1_verified = {
    "d8e79af0": (
        "Andre Cornelius Botha",
        "Andre Botha",
        "Confirmed in Ireland vs England ODI, 13 June 2006; scored 52 off 89."
    ),
    "fa1aa7e7": (
        "David Langford-Smith",
        "Dave Langford-Smith",
        "Confirmed in Ireland vs England ODI, 13 June 2006; took 3/63."
    ),
    "7c63baa7": (
        "Jeremy Paul Bray",
        "Jeremy Bray",
        "Confirmed in Ireland vs England ODI, 13 June 2006; scored 22 off 29."
    ),
    "d7d744e5": (
        "Peter Gerard Gillespie",
        "Peter Gillespie",
        "Confirmed in Ireland vs England ODI, 13 June 2006; scored 0 off 3."
    ),
    "c942a53b": (
        "William Kyle McCallan",
        "Kyle McCallan",
        "Confirmed in 13 June 2006 inaugural Ireland ODI; final ODI 26 August 2009."
    )
}

for player_id, (full_name, cr_name, notes) in batch_1_verified.items():
    mask = ireland_manual_review["cricsheet_player_id"] == player_id

    ireland_manual_review.loc[mask, "crickrida_name"] = cr_name
    ireland_manual_review.loc[mask, "verification_status"] = "verified"
    ireland_manual_review.loc[mask, "confidence"] = "high"
    ireland_manual_review.loc[mask, "verification_notes"] = (
        f"{full_name}. {notes}"
    )

ireland_manual_review.to_csv(
    "ireland_manual_identity_review.csv",
    index=False
)

print(ireland_manual_review["verification_status"].value_counts())

verification_status
unverified    18
verified       5
Name: count, dtype: int64


In [70]:
# All 23 manually verified Ireland identities.
# Crickrida profile IDs remain blank until independently looked up.

ireland_verified_data = {
    "d8e79af0": ("Andre Cornelius Botha", "Andre Botha",
                "Ireland 2001-2011; ODI debut 2006-06-13 vs England; 52 off 89."),
    "fa1aa7e7": ("David Langford-Smith", "Dave Langford-Smith",
                "Ireland 2006-2008; appeared vs England 2006-06-13; took Ireland's first ODI wicket."),
    "7c63baa7": ("Jeremy Paul Bray", "Jeremy Bray",
                "Ireland 2002-2009; appeared vs England 2006-06-13; scored 115 not out vs Zimbabwe in 2007 World Cup."),
    "d7d744e5": ("Peter Gerard Gillespie", "Peter Gillespie",
                "Ireland 1995-2007; appeared vs England 2006-06-13; last ODI 2007-04-09."),
    "c942a53b": ("William Kyle McCallan", "Kyle McCallan",
                "Ireland 1996-2009; appeared in inaugural ODI vs England 2006-06-13; final ODI 2009-08-26."),
    "a159b802": ("Albert van der Merwe", "Albert van der Merwe",
                "Ireland 2010-2011; ODI debut 2010-07-09 vs Netherlands."),
    "b3e9b7fa": ("Kenneth Edward Desmond Carroll", "Kenny Carroll",
                "Ireland 2007; ODI debut 2007-01-31 vs Bermuda."),
    "7a4819e1": ("Marthinus Jacobus Fourie", "Thinus Fourie",
                "Ireland 2007-2008; ODI debut 2007-06-23 vs India."),
    "06e649a0": ("Roger Kyran Whelan", "Roger Whelan",
                "Ireland 2007; ODI debut 2007-06-23 vs India."),
    "5500c675": ("Peter Christopher Connell", "Peter Connell",
                "Ireland 2008-2010; ODI debut 2008-07-01 vs New Zealand."),
    "d1761087": ("Regan Morris West", "Regan West",
                "Ireland 2008-2009; ODI debut 2008-08-25 vs Kenya."),
    "6e161a33": ("James Norman Knight Shannon", "James Shannon",
                "Ireland 2013-2018; sole ODI 2013-05-26 vs Pakistan."),
    "3f18724b": ("Max Christian Sorensen", "Max Sorensen",
                "Ireland 2012-2016; ODI debut 2013-09-03 vs England."),
    "49dfe4a4": ("Edward James Richardson", "Eddie Richardson",
                "Ireland 2013; ODI debut 2013-09-06 vs Scotland."),
    "6232e1ed": ("Graeme John McCarter", "Graeme McCarter",
                "Ireland 2014-2015; sole ODI 2014-09-12 vs Scotland."),
    "c8065f31": ("Shane Charles Getkate", "Shane Getkate",
                "Ireland 2019-2022; ODI debut 2019-07-01 vs Zimbabwe."),
    "43f4087c": ("Neil Alan Rock", "Neil Rock",
                "Ireland 2021-2025; ODI debut 2022-01-08 vs West Indies."),
    "4992d6f6": ("Stephen Thomas Doheny", "Stephen Doheny",
                "Ireland 2023-present; ODI debut 2023-01-18 vs Zimbabwe."),
    "c537cf4d": ("Murray Commins", "Murray Commins",
                "Ireland 2023; ODI debut 2023-01-21 vs Zimbabwe."),
    "2c54c49b": ("Matthew James Humphreys", "Matthew Humphreys",
                "Ireland 2023-present; ODI debut 2023-03-20 vs Bangladesh."),
    "1a448be0": ("Benjamin Charlie White", "Ben White",
                "Ireland 2021-present; ODI debut 2023-06-19 vs Oman."),
    "e3fe4b49": ("Fionn Philip Hand", "Fionn Hand",
                "Ireland 2022-present; ODI appearance 2024-10-07 vs South Africa."),
    "9aedf23b": ("Thomas Mayes", "Thomas Mayes",
                "Ireland 2025-present; ODI debut 2025-05-21 vs West Indies.")
}

# 1. Update manual-review records
for player_id, (full_name, cr_name, notes) in ireland_verified_data.items():
    mask = ireland_manual_review["cricsheet_player_id"].astype(str) == player_id

    ireland_manual_review.loc[mask, "crickrida_name"] = cr_name
    ireland_manual_review.loc[mask, "verification_status"] = "verified"
    ireland_manual_review.loc[mask, "confidence"] = "high"
    ireland_manual_review.loc[mask, "verification_notes"] = (
        f"{full_name}. {notes}"
    )

# 2. Add verified identities to the master identity map
new_identity_rows = []

for player_id, (full_name, cr_name, notes) in ireland_verified_data.items():

    if player_id in player_identity_map_updated["player_id"].astype(str).values:
        print("Already present:", cr_name)
        continue

    new_identity_rows.append({
        "cricketiq_player_id": f"CRIQ_{player_id}",
        "player_id": player_id,
        "cricsheet_name": ireland_players.loc[
            ireland_players["source_player_id"].astype(str) == player_id,
            "player_name"
        ].iloc[0],
        "crickrida_name": cr_name,
        "normalized_name": normalize_player_name(cr_name),
        "match_status": "verified_manual_identity",
        "crickrida_player_name": cr_name,
        "cricsheet_player_name": ireland_players.loc[
            ireland_players["source_player_id"].astype(str) == player_id,
            "player_name"
        ].iloc[0],
        "cricsheet_player_id": player_id,
        "mapping_method": "manual_full_name_career_match_verification",
        "confidence": "high",
        "notes": f"{full_name}. {notes} Crickrida profile ID pending lookup."
    })

if new_identity_rows:
    player_identity_map_updated = pd.concat(
        [player_identity_map_updated, pd.DataFrame(new_identity_rows)],
        ignore_index=True
    )

# 3. Save review and identity map
ireland_manual_review.to_csv(
    "ireland_manual_identity_review.csv", index=False
)

player_identity_map_updated.to_csv(
    "player_identity_map_updated.csv", index=False
)

print("\nNEW IDENTITY MAPPINGS ADDED:", len(new_identity_rows))
print("MASTER IDENTITY MAP:", player_identity_map_updated.shape)
print("DUPLICATE CRICSHEET IDs:",
      player_identity_map_updated["player_id"].duplicated().sum())

print("\nIRELAND MANUAL REVIEW:")
print(ireland_manual_review["verification_status"].value_counts())

display(
    ireland_manual_review[
        ["cricsheet_name", "cricsheet_player_id",
         "crickrida_name", "verification_status", "confidence"]
    ]
)


NEW IDENTITY MAPPINGS ADDED: 23
MASTER IDENTITY MAP: (189, 12)
DUPLICATE CRICSHEET IDs: 0

IRELAND MANUAL REVIEW:
verification_status
verified    23
Name: count, dtype: int64


,cricsheet_name,cricsheet_player_id,crickrida_name,verification_status,confidence
1,A van der Merwe,a159b802,Albert van der Merwe,verified,high
2,AC Botha,d8e79af0,Andre Botha,verified,high
7,B White,1a448be0,Ben White,verified,high
13,D Langford-Smith,fa1aa7e7,Dave Langford-Smith,verified,high
17,EJ Richardson,49dfe4a4,Eddie Richardson,verified,high
19,F Hand,e3fe4b49,Fionn Hand,verified,high
25,GJ McCarter,6232e1ed,Graeme McCarter,verified,high
34,JNK Shannon,6e161a33,James Shannon,verified,high
35,JP Bray,7c63baa7,Jeremy Bray,verified,high
36,KED Carroll,b3e9b7fa,Kenny Carroll,verified,high


In [71]:
# Calculate player identity mapping coverage by international team

all_team_players = (
    cricsheet_appearances[
        ["team", "source_player_id", "player_name"]
    ]
    .drop_duplicates()
)

mapped_ids = set(
    player_identity_map_updated["player_id"]
    .dropna()
    .astype(str)
)

all_team_players["is_mapped"] = (
    all_team_players["source_player_id"].astype(str).isin(mapped_ids)
)

team_mapping_summary = (
    all_team_players
    .groupby("team")
    .agg(
        total_players=("source_player_id", "nunique"),
        mapped_players=("is_mapped", "sum")
    )
    .reset_index()
)

team_mapping_summary["unmapped_players"] = (
    team_mapping_summary["total_players"]
    - team_mapping_summary["mapped_players"]
)

team_mapping_summary["mapping_percentage"] = (
    team_mapping_summary["mapped_players"]
    / team_mapping_summary["total_players"] * 100
).round(2)

display(
    team_mapping_summary.sort_values(
        "unmapped_players", ascending=False
    )
)

print("Total unique players:", all_team_players["source_player_id"].nunique())
print("Mapped players:", all_team_players.loc[
    all_team_players["is_mapped"], "source_player_id"
].nunique())
print("Unmapped players:", all_team_players.loc[
    ~all_team_players["is_mapped"], "source_player_id"
].nunique())

,team,total_players,mapped_players,unmapped_players,mapping_percentage
22,West Indies,142,3,139,2.11
19,Sri Lanka,133,0,133,0.00
4,England,133,8,125,6.02
0,Australia,127,3,124,2.36
6,India,137,14,123,10.22
18,South Africa,122,4,118,3.28
13,New Zealand,118,2,116,1.69
23,Zimbabwe,115,1,114,0.87
15,Pakistan,147,40,107,27.21
20,United Arab Emirates,103,1,102,0.97


Total unique players: 2008
Mapped players: 189
Unmapped players: 1819


In [72]:
# Verify Cricsheet player ID consistency

print("CRICSHEET APPEARANCES")
print("Columns:", cricsheet_appearances.columns.tolist())
print("source_player_id dtype:",
      cricsheet_appearances["source_player_id"].dtype)

print("\nIDENTITY MAP")
print("Columns:", player_identity_map_updated.columns.tolist())
print("player_id dtype:",
      player_identity_map_updated["player_id"].dtype)

appearance_ids = set(
    cricsheet_appearances["source_player_id"].dropna().astype(str)
)

identity_ids = set(
    player_identity_map_updated["player_id"].dropna().astype(str)
)

print("\nID OVERLAP")
print("Unique appearance IDs:", len(appearance_ids))
print("Unique identity-map IDs:", len(identity_ids))
print("Matching IDs:", len(appearance_ids & identity_ids))
print("Identity IDs not found in appearances:",
      len(identity_ids - appearance_ids))

print("\nSAMPLE MATCHED IDs")
print(
    player_identity_map_updated[
        player_identity_map_updated["player_id"].astype(str).isin(
            appearance_ids
        )
    ][["player_id", "cricsheet_name", "crickrida_name"]]
    .head(10).to_string(index=False)
)

print("\nSAMPLE UNMATCHED APPEARANCE IDs")
unmatched_ids = appearance_ids - identity_ids
print(list(sorted(unmatched_ids))[:10])

CRICSHEET APPEARANCES
Columns: ['date', 'team', 'player_name', 'source_player_id', 'match_id', 'role', 'source']
source_player_id dtype: object

IDENTITY MAP
Columns: ['cricketiq_player_id', 'player_id', 'cricsheet_name', 'crickrida_name', 'normalized_name', 'match_status', 'crickrida_player_name', 'cricsheet_player_name', 'cricsheet_player_id', 'mapping_method', 'confidence', 'notes']
player_id dtype: object

ID OVERLAP
Unique appearance IDs: 2008
Unique identity-map IDs: 189
Matching IDs: 189
Identity IDs not found in appearances: 0

SAMPLE MATCHED IDs
player_id  cricsheet_name  crickrida_name
 8a75e999      Babar Azam      Babar Azam
 2911de16       Hasan Ali       Hasan Ali
 9cb8d7a6      Imad Wasim      Imad Wasim
 e174dadd   Mohammad Amir   Mohammad Amir
 9ab63e7b Mohammad Hafeez Mohammad Hafeez
 2f26ac1a Mohammad Rizwan Mohammad Rizwan
 193ef196   Sharjeel Khan   Sharjeel Khan
 fd2bf2a0      Umar Akmal      Umar Akmal
 b3118300      Wahab Riaz      Wahab Riaz
 50250426     Asad 

In [73]:
# Find exact normalized-name matches among unmapped players

unmapped_players = (
    cricsheet_appearances[
        ~cricsheet_appearances["source_player_id"].astype(str).isin(
            identity_ids
        )
    ][["source_player_id", "player_name", "team"]]
    .drop_duplicates(subset=["source_player_id"])
    .copy()
)

unmapped_players["normalized_name"] = (
    unmapped_players["player_name"].apply(normalize_player_name)
)

mapped_names = (
    player_identity_map_updated[
        ["player_id", "cricsheet_name", "normalized_name"]
    ]
    .drop_duplicates(subset=["player_id"])
    .copy()
)

exact_name_candidates = unmapped_players.merge(
    mapped_names,
    on="normalized_name",
    how="inner",
    suffixes=("_unmapped", "_mapped")
)

print("Unmapped unique players:", len(unmapped_players))
print("Exact normalized-name candidate pairs:",
      len(exact_name_candidates))

print("\nCANDIDATES:")
display(
    exact_name_candidates[
        [
            "player_name",
            "team",
            "source_player_id",
            "cricsheet_name",
            "player_id"
        ]
    ].sort_values(["team", "player_name"])
)

Unmapped unique players: 1819
Exact normalized-name candidate pairs: 0

CANDIDATES:


,player_name,team,source_player_id,cricsheet_name,player_id


In [74]:
# Inspect unmapped West Indies players

west_indies_unmapped = (
    cricsheet_appearances[
        (cricsheet_appearances["team"] == "West Indies") &
        (~cricsheet_appearances["source_player_id"].astype(str).isin(
            identity_ids
        ))
    ]
    .groupby(
        ["player_name", "source_player_id"],
        as_index=False
    )
    .agg(
        matches_played=("match_id", "nunique"),
        first_appearance=("date", "min"),
        last_appearance=("date", "max")
    )
    .sort_values(
        ["matches_played", "player_name"],
        ascending=[False, True]
    )
)

print("Unmapped West Indies players:",
      len(west_indies_unmapped))

display(west_indies_unmapped)

Unmapped West Indies players: 139


,player_name,source_player_id,matches_played,first_appearance,last_appearance
19,CH Gayle,db584dad,197,2003-02-09,2019-08-14
82,MN Samuels,f846de6a,148,2003-05-17,2018-12-14
32,DJ Bravo,87e562a9,144,2004-05-02,2014-10-17
117,SD Hope,1fc6ef83,144,2016-11-16,2026-10-03
60,JO Holder,0f721006,127,2013-02-01,2023-07-01
...,...,...,...,...,...
111,RT Crandon,0557a636,1,2009-09-30,2009-09-30
112,Reon King,20fbf68c,1,2005-02-01,2005-02-01
120,SH Lewis,210f4a5d,1,2022-06-04,2022-06-04
123,SO Dowrich,dc2f959a,1,2019-05-07,2019-05-07


In [75]:
# Prepare West Indies identity review table

wi_identity_review = west_indies_unmapped.copy()

wi_identity_review["verification_status"] = "unverified"
wi_identity_review["canonical_player_name"] = None
wi_identity_review["evidence"] = None

wi_identity_review = wi_identity_review[
    [
        "player_name",
        "source_player_id",
        "matches_played",
        "first_appearance",
        "last_appearance",
        "canonical_player_name",
        "verification_status",
        "evidence"
    ]
]

print("West Indies review records:", len(wi_identity_review))
print("\nMost frequently appearing players:")
display(wi_identity_review.head(25))

print("\nSingle-appearance players:",
      (wi_identity_review["matches_played"] == 1).sum())

West Indies review records: 139

Most frequently appearing players:


,player_name,source_player_id,matches_played,first_appearance,last_appearance,canonical_player_name,verification_status,evidence
19,CH Gayle,db584dad,197,2003-02-09,2019-08-14,None,unverified,None
82,MN Samuels,f846de6a,148,2003-05-17,2018-12-14,None,unverified,None
32,DJ Bravo,87e562a9,144,2004-05-02,2014-10-17,None,unverified,None
117,SD Hope,1fc6ef83,144,2016-11-16,2026-10-03,None,unverified,None
60,JO Holder,0f721006,127,2013-02-01,2023-07-01,None,unverified,None
33,DJG Sammy,c03c6200,120,2004-09-15,2015-03-21,None,unverified,None
109,RR Sarwan,36d33dd0,114,2003-02-09,2013-06-11,None,unverified,None
35,DM Bravo,14b14cd8,113,2009-06-26,2022-02-11,None,unverified,None
66,KA Pollard,a757b0d8,112,2007-04-10,2022-02-06,None,unverified,None
113,S Chanderpaul,881a9bdd,108,2003-02-09,2011-03-23,None,unverified,None



Single-appearance players: 12


In [76]:
# Display all 139 unresolved West Indies players

import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

wi_identity_review = wi_identity_review.sort_values(
    ["matches_played", "player_name"],
    ascending=[False, True]
).reset_index(drop=True)

print("TOTAL UNRESOLVED WEST INDIES PLAYERS:",
      len(wi_identity_review))

print("\nCOMPLETE PLAYER LIST:")
display(wi_identity_review)

print("\nAPPEARANCE DISTRIBUTION:")
display(
    wi_identity_review["matches_played"]
    .value_counts()
    .sort_index(ascending=False)
    .rename_axis("matches_played")
    .reset_index(name="player_count")
)

# Save a complete review copy
wi_identity_review.to_csv(
    "west_indies_identity_review_139.csv",
    index=False
)

print("\nSaved: west_indies_identity_review_139.csv")

TOTAL UNRESOLVED WEST INDIES PLAYERS: 139

COMPLETE PLAYER LIST:


,player_name,source_player_id,matches_played,first_appearance,last_appearance,canonical_player_name,verification_status,evidence
0,CH Gayle,db584dad,197,2003-02-09,2019-08-14,None,unverified,None
1,MN Samuels,f846de6a,148,2003-05-17,2018-12-14,None,unverified,None
2,DJ Bravo,87e562a9,144,2004-05-02,2014-10-17,None,unverified,None
3,SD Hope,1fc6ef83,144,2016-11-16,2026-10-03,None,unverified,None
4,JO Holder,0f721006,127,2013-02-01,2023-07-01,None,unverified,None
5,DJG Sammy,c03c6200,120,2004-09-15,2015-03-21,None,unverified,None
6,RR Sarwan,36d33dd0,114,2003-02-09,2013-06-11,None,unverified,None
7,DM Bravo,14b14cd8,113,2009-06-26,2022-02-11,None,unverified,None
8,KA Pollard,a757b0d8,112,2007-04-10,2022-02-06,None,unverified,None
9,S Chanderpaul,881a9bdd,108,2003-02-09,2011-03-23,None,unverified,None



APPEARANCE DISTRIBUTION:


,matches_played,player_count
0,197,1
1,148,1
2,144,2
3,127,1
4,120,1
5,114,1
6,113,1
7,112,1
8,108,1
9,103,1



Saved: west_indies_identity_review_139.csv


In [77]:
# Find West Indies players in the Afghanistan ODI dataset

# West Indies batting appearances
wi_batting = (
    afg_player_batting_df[
        afg_player_batting_df["batting_team"].str.contains(
            "West Indies", case=False, na=False
        )
    ][["player_name", "date", "match_url"]]
    .drop_duplicates()
    .copy()
)

# West Indies bowling appearances
wi_bowling = (
    afg_player_bowling_df[
        afg_player_bowling_df["bowling_team"].str.contains(
            "West Indies", case=False, na=False
        )
    ][["player_name", "date", "match_url"]]
    .drop_duplicates()
    .copy()
)

# Combine both sources
wi_afghanistan_players = pd.concat(
    [wi_batting, wi_bowling],
    ignore_index=True
).drop_duplicates(subset=["player_name"])

print("WEST INDIES PLAYERS IN AFGHANISTAN DATASET")
print("Batting players:", wi_batting["player_name"].nunique())
print("Bowling players:", wi_bowling["player_name"].nunique())
print("Unique players across both:", len(wi_afghanistan_players))

display(
    wi_afghanistan_players.sort_values("player_name")
)

WEST INDIES PLAYERS IN AFGHANISTAN DATASET
Batting players: 24
Bowling players: 18
Unique players across both: 27


,player_name,date,match_url
8,Alzarri Joseph,2017-06-09,https://crickrida.com/matches/afghanistan-west-indies-2017-06-09-1089779/
7,Ashley Nurse,2017-06-09,https://crickrida.com/matches/afghanistan-west-indies-2017-06-09-1089779/
66,Brandon King,2019-11-11,https://crickrida.com/matches/afghanistan-west-indies-2019-11-11-1193499/
26,Carlos Brathwaite,2018-03-15,https://crickrida.com/matches/afghanistan-west-indies-2018-03-15-1133018/
19,Chris Gayle,2018-03-15,https://crickrida.com/matches/afghanistan-west-indies-2018-03-15-1133018/
0,Evin Lewis,2017-06-09,https://crickrida.com/matches/afghanistan-west-indies-2017-06-09-1089779/
47,Fabian Allen,2019-07-04,https://crickrida.com/matches/afghanistan-west-indies-2019-07-04-1144524/
62,Hayden Walsh,2019-11-09,https://crickrida.com/matches/afghanistan-west-indies-2019-11-09-1193498/
6,Jason Holder,2017-06-09,https://crickrida.com/matches/afghanistan-west-indies-2017-06-09-1089779/
3,Jason Mohammed,2017-06-09,https://crickrida.com/matches/afghanistan-west-indies-2017-06-09-1089779/


In [78]:
# Map only the 27 West Indies players appearing in Afghanistan data

wi_afghanistan_aliases = {
    "Alzarri Joseph": "AS Joseph",
    "Ashley Nurse": "AR Nurse",
    "Brandon King": "BA King",
    "Carlos Brathwaite": "CR Brathwaite",
    "Chris Gayle": "CH Gayle",
    "Evin Lewis": "E Lewis",
    "Fabian Allen": "F Allen",
    "Hayden Walsh": "HR Walsh",
    "Jason Holder": "JO Holder",
    "Jason Mohammed": "J Mohammed",
    "Jonathan Carter": "J Carter",
    "Keemo Paul": "KMA Paul",
    "Kemar Roach": "KAJ Roach",
    "Kieran Powell": "KOA Powell",
    "Kieron Pollard": "KA Pollard",
    "Marlon Samuels": "MN Samuels",
    "Miguel Cummins": "M Cummins",
    "Nicholas Pooran": "N Pooran",
    "Nikita Miller": "NO Miller",
    "Oshane Thomas": "O Thomas",
    "Romario Shepherd": "R Shepherd",
    "Roston Chase": "RL Chase",
    "Rovman Powell": "R Powell",
    "Shai Hope": "SD Hope",
    "Shannon Gabriel": "ST Gabriel",
    "Sheldon Cottrell": "SO Cottrell",
    "Shimron Hetmyer": "SO Hetmyer"
}

# Find the corresponding Cricsheet records
wi_mapping_candidates = []

for full_name, cricsheet_name in wi_afghanistan_aliases.items():

    candidate = wi_identity_review[
        wi_identity_review["player_name"].str.strip().str.casefold()
        == cricsheet_name.casefold()
    ]

    if len(candidate) == 1:
        row = candidate.iloc[0]

        wi_mapping_candidates.append({
            "crickrida_name": full_name,
            "cricsheet_name": row["player_name"],
            "player_id": row["source_player_id"],
            "matches_played": row["matches_played"],
            "first_appearance": row["first_appearance"],
            "last_appearance": row["last_appearance"]
        })

    else:
        print(
            f"REVIEW REQUIRED: {full_name} -> {cricsheet_name} "
            f"(matches found: {len(candidate)})"
        )

wi_mapping_candidates_df = pd.DataFrame(wi_mapping_candidates)

print("\nMATCHED CANDIDATES:",
      len(wi_mapping_candidates_df))
display(wi_mapping_candidates_df)

# Stop before modifying the identity map if any name is unresolved
if len(wi_mapping_candidates_df) != 27:
    print("\nNo identity-map changes made. Resolve the missing candidates first.")

else:
    existing_ids = set(
        player_identity_map_updated["player_id"].dropna().astype(str)
    )

    new_rows = []

    for _, row in wi_mapping_candidates_df.iterrows():

        player_id = str(row["player_id"])

        if player_id in existing_ids:
            continue

        new_rows.append({
            "cricketiq_player_id": f"CRIQ_{player_id}",
            "player_id": player_id,
            "cricsheet_name": row["cricsheet_name"],
            "crickrida_name": row["crickrida_name"],
            "normalized_name": normalize_player_name(
                row["crickrida_name"]
            ),
            "match_status": "verified_afghanistan_appearance",
            "crickrida_player_name": row["crickrida_name"],
            "cricsheet_player_name": row["cricsheet_name"],
            "cricsheet_player_id": player_id,
            "mapping_method": "afghanistan_scorecard_and_cricsheet_name_match",
            "confidence": "high",
            "notes": (
                "Full name appears in Afghanistan ODI scorecards; "
                "Cricsheet abbreviated name and player ID matched. "
                "Crickrida profile ID pending lookup."
            )
        })

    if new_rows:
        player_identity_map_updated = pd.concat(
            [
                player_identity_map_updated,
                pd.DataFrame(new_rows)
            ],
            ignore_index=True
        )

    player_identity_map_updated = (
        player_identity_map_updated
        .drop_duplicates(subset=["player_id"], keep="first")
        .reset_index(drop=True)
    )

    player_identity_map_updated.to_csv(
        "player_identity_map_updated.csv",
        index=False
    )

    print("\nNEW WEST INDIES MAPPINGS:", len(new_rows))
    print("ALREADY MAPPED:", 27 - len(new_rows))
    print("UPDATED MASTER MAP:", player_identity_map_updated.shape)
    print(
        "DUPLICATE CRICSHEET IDs:",
        player_identity_map_updated["player_id"].duplicated().sum()
    )

REVIEW REQUIRED: Brandon King -> BA King (matches found: 0)
REVIEW REQUIRED: Carlos Brathwaite -> CR Brathwaite (matches found: 0)
REVIEW REQUIRED: Fabian Allen -> F Allen (matches found: 0)
REVIEW REQUIRED: Jason Mohammed -> J Mohammed (matches found: 0)
REVIEW REQUIRED: Jonathan Carter -> J Carter (matches found: 0)
REVIEW REQUIRED: Kieran Powell -> KOA Powell (matches found: 0)
REVIEW REQUIRED: Miguel Cummins -> M Cummins (matches found: 0)
REVIEW REQUIRED: Sheldon Cottrell -> SO Cottrell (matches found: 0)

MATCHED CANDIDATES: 19


,crickrida_name,cricsheet_name,player_id,matches_played,first_appearance,last_appearance
0,Alzarri Joseph,AS Joseph,b0946605,83,2016-10-02,2026-10-03
1,Ashley Nurse,AR Nurse,4a745f65,45,2016-11-16,2019-06-22
2,Chris Gayle,CH Gayle,db584dad,197,2003-02-09,2019-08-14
3,Evin Lewis,E Lewis,0ebfb1ad,59,2016-10-05,2025-08-12
4,Hayden Walsh,HR Walsh,1ac2a995,21,2019-12-15,2024-10-23
5,Jason Holder,JO Holder,0f721006,127,2013-02-01,2023-07-01
6,Keemo Paul,KMA Paul,75224f22,30,2018-03-19,2026-10-03
7,Kemar Roach,KAJ Roach,78eb4223,88,2009-07-26,2022-02-11
8,Kieron Pollard,KA Pollard,a757b0d8,112,2007-04-10,2022-02-06
9,Marlon Samuels,MN Samuels,f846de6a,148,2003-05-17,2018-12-14



No identity-map changes made. Resolve the missing candidates first.


In [79]:
from difflib import SequenceMatcher

unresolved_wi_names = {
    "Brandon King": "BA King",
    "Carlos Brathwaite": "CR Brathwaite",
    "Fabian Allen": "F Allen",
    "Jason Mohammed": "J Mohammed",
    "Jonathan Carter": "J Carter",
    "Kieran Powell": "KOA Powell",
    "Miguel Cummins": "M Cummins",
    "Sheldon Cottrell": "SO Cottrell"
}

candidate_rows = []

for full_name, assumed_name in unresolved_wi_names.items():

    target = normalize_player_name(full_name)

    scores = []

    for _, row in wi_identity_review.iterrows():
        candidate_name = row["player_name"]
        normalized_candidate = normalize_player_name(candidate_name)

        score = SequenceMatcher(
            None, target, normalized_candidate
        ).ratio()

        scores.append({
            "full_name": full_name,
            "assumed_abbreviation": assumed_name,
            "candidate_cricsheet_name": candidate_name,
            "candidate_player_id": row["source_player_id"],
            "matches_played": row["matches_played"],
            "similarity": round(score, 3)
        })

    top_candidates = sorted(
        scores,
        key=lambda x: x["similarity"],
        reverse=True
    )[:5]

    candidate_rows.extend(top_candidates)

wi_unresolved_candidates = pd.DataFrame(candidate_rows)

display(
    wi_unresolved_candidates.sort_values(
        ["full_name", "similarity"],
        ascending=[True, False]
    )
)

,full_name,assumed_abbreviation,candidate_cricsheet_name,candidate_player_id,matches_played,similarity
0,Brandon King,BA King,Reon King,20fbf68c,1,0.737
1,Brandon King,BA King,RT Crandon,0557a636,1,0.600
2,Brandon King,BA King,N Deonarine,caf1b14c,26,0.571
3,Brandon King,BA King,D Ramdin,d1523761,103,0.556
4,Brandon King,BA King,KJ Harding,714af18b,1,0.500
5,Carlos Brathwaite,CR Brathwaite,KC Brathwaite,7d5af2ea,10,0.786
6,Carlos Brathwaite,CR Brathwaite,AB Barath,d3a3e82d,14,0.500
7,Carlos Brathwaite,CR Brathwaite,CO Browne,a1d8d8ba,8,0.500
8,Carlos Brathwaite,CR Brathwaite,A Athanaze,58c2fac4,17,0.480
9,Carlos Brathwaite,CR Brathwaite,IDR Bradshaw,a7489342,48,0.444


In [80]:
# Inspect candidate identities against Afghanistan ODI appearances

candidate_checks = {
    "Fabian Allen": ("FA Allen", "b52ffbbd"),
    "Jason Mohammed": ("JN Mohammed", "773cec26"),
    "Jonathan Carter": ("JL Carter", "0a9d8b61"),
    "Miguel Cummins": ("ML Cummins", "92aa28fe"),
    "Sheldon Cottrell": ("SS Cottrell", "a1d053dd")
}

for full_name, (cricsheet_name, player_id) in candidate_checks.items():

    print("\n" + "=" * 65)
    print(f"{full_name} | Cricsheet: {cricsheet_name} | ID: {player_id}")

    cricsheet_record = wi_identity_review[
        wi_identity_review["source_player_id"].astype(str) == player_id
    ]

    display(
        cricsheet_record[
            [
                "player_name",
                "source_player_id",
                "matches_played",
                "first_appearance",
                "last_appearance"
            ]
        ]
    )

    afg_record = wi_afghanistan_players[
        wi_afghanistan_players["player_name"].str.casefold()
        == full_name.casefold()
    ]

    print("Afghanistan scorecard evidence:")
    display(afg_record)


Fabian Allen | Cricsheet: FA Allen | ID: b52ffbbd


,player_name,source_player_id,matches_played,first_appearance,last_appearance
61,FA Allen,b52ffbbd,18,2018-10-27,2022-02-11


Afghanistan scorecard evidence:


,player_name,date,match_url
47,Fabian Allen,2019-07-04,https://crickrida.com/matches/afghanistan-west-indies-2019-07-04-1144524/



Jason Mohammed | Cricsheet: JN Mohammed | ID: 773cec26


,player_name,source_player_id,matches_played,first_appearance,last_appearance
43,JN Mohammed,773cec26,31,2011-12-11,2021-07-22


... (10 outputs left out)


Sheldon Cottrell | Cricsheet: SS Cottrell | ID: a1d053dd


,player_name,source_player_id,matches_played,first_appearance,last_appearance
42,SS Cottrell,a1d053dd,33,2015-01-25,2021-07-26


Afghanistan scorecard evidence:


,player_name,date,match_url
94,Sheldon Cottrell,2019-07-04,https://crickrida.com/matches/afghanistan-west-indies-2019-07-04-1144524/


In [81]:
# Inspect the three unresolved West Indies identities

remaining_wi_players = [
    "Brandon King",
    "Carlos Brathwaite",
    "Kieran Powell"
]

for name in remaining_wi_players:

    print("\n" + "=" * 60)
    print(name)

    batting = afg_player_batting_df[
        afg_player_batting_df["player_name"].str.casefold()
        == name.casefold()
    ]

    bowling = afg_player_bowling_df[
        afg_player_bowling_df["player_name"].str.casefold()
        == name.casefold()
    ]

    print("\nBatting records:")
    display(
        batting[
            ["player_name", "date", "batting_team", "match_url"]
        ].drop_duplicates()
    )

    print("\nBowling records:")
    display(
        bowling[
            ["player_name", "date", "bowling_team", "match_url"]
        ].drop_duplicates()
    )


Brandon King

Batting records:


,player_name,date,batting_team,match_url
1633,Brandon King,2019-11-11,West Indies,https://crickrida.com/matches/afghanistan-west-indies-2019-11-11-1193499/



Bowling records:


,player_name,date,bowling_team,match_url



Carlos Brathwaite

Batting records:


,player_name,date,batting_team,match_url
1097,Carlos Brathwaite,2018-03-15,West Indies,https://crickrida.com/matches/afghanistan-west-indies-2018-03-15-1133018/
1132,Carlos Brathwaite,2018-03-25,West Indies,https://crickrida.com/matches/afghanistan-west-indies-2018-03-25-1133031/
1572,Carlos Brathwaite,2019-07-04,West Indies,https://crickrida.com/matches/afghanistan-west-indies-2019-07-04-1144524/


... (2 outputs left out)


Kieran Powell

Batting records:


,player_name,date,batting_team,match_url
892,Kieran Powell,2017-06-09,West Indies,https://crickrida.com/matches/afghanistan-west-indies-2017-06-09-1089779/
914,Kieran Powell,2017-06-11,West Indies,https://crickrida.com/matches/afghanistan-west-indies-2017-06-11-1089780/



Bowling records:


,player_name,date,bowling_team,match_url


In [82]:
# Verify the three players against the complete Cricsheet registry

manual_wi_players = {
    "Brandon King": {
        "expected_cricsheet_name": "BA King",
        "match_dates": ["2019-11-11"]
    },
    "Carlos Brathwaite": {
        "expected_cricsheet_name": "CR Brathwaite",
        "match_dates": [
            "2018-03-15", "2018-03-25", "2019-07-04"
        ]
    },
    "Kieran Powell": {
        "expected_cricsheet_name": "KOA Powell",
        "match_dates": ["2017-06-09", "2017-06-11"]
    }
}

for full_name, details in manual_wi_players.items():

    expected_name = details["expected_cricsheet_name"]
    normalized_expected = normalize_player_name(expected_name)

    print("\n" + "=" * 70)
    print(f"{full_name} | Expected Cricsheet name: {expected_name}")

    # Search the complete player registry
    registry_matches = player_registry_df[
        player_registry_df["player_name"].apply(
            normalize_player_name
        ) == normalized_expected
    ].drop_duplicates()

    print("\nComplete Cricsheet registry matches:")
    display(registry_matches)

    # Search all West Indies appearances on the relevant dates
    for match_date in details["match_dates"]:

        print(f"\nWest Indies players on {match_date}:")

        date_matches = cricsheet_appearances[
            (cricsheet_appearances["team"] == "West Indies") &
            (cricsheet_appearances["date"].astype(str) == match_date)
        ][["player_name", "source_player_id"]].drop_duplicates()

        display(date_matches.sort_values("player_name"))


Brandon King | Expected Cricsheet name: BA King

Complete Cricsheet registry matches:


,match_id,player_name,player_id
14041,1203673,BA King,7fca84b7
14068,1203674,BA King,7fca84b7
14095,1203675,BA King,7fca84b7
19116,1278676,BA King,7fca84b7
19142,1278677,BA King,7fca84b7
19169,1278678,BA King,7fca84b7
20181,1302349,BA King,7fca84b7
20208,1302350,BA King,7fca84b7
20234,1302351,BA King,7fca84b7
21607,1315038,BA King,7fca84b7



West Indies players on 2019-11-11:


,player_name,source_player_id



Carlos Brathwaite | Expected Cricsheet name: CR Brathwaite

Complete Cricsheet registry matches:


,match_id,player_name,player_id
1111,1027315,CR Brathwaite,e342e5fb
1138,1027317,CR Brathwaite,e342e5fb
1167,1027319,CR Brathwaite,e342e5fb
2109,1050223,CR Brathwaite,e342e5fb
2137,1050225,CR Brathwaite,e342e5fb
2423,1059711,CR Brathwaite,e342e5fb
2453,1059712,CR Brathwaite,e342e5fb
2503,1059714,CR Brathwaite,e342e5fb
2533,1059715,CR Brathwaite,e342e5fb
7056,1133007,CR Brathwaite,e342e5fb


... (8 outputs left out)


West Indies players on 2017-06-09:


,player_name,source_player_id



West Indies players on 2017-06-11:


,player_name,source_player_id


In [83]:
# Search the complete Cricsheet registry for Brandon King variants

brandon_registry = player_registry_df[
    player_registry_df["player_name"].str.contains(
        "King", case=False, na=False
    )
].drop_duplicates()

print("CRICSHEET REGISTRY ENTRIES CONTAINING 'KING':")
display(
    brandon_registry.sort_values("player_name")
)

print("\nCRICSHEET APPEARANCE ENTRIES CONTAINING 'KING':")

brandon_appearances = cricsheet_appearances[
    cricsheet_appearances["player_name"].str.contains(
        "King", case=False, na=False
    )
].drop_duplicates(
    subset=["player_name", "source_player_id"]
)

display(brandon_appearances)

CRICSHEET REGISTRY ENTRIES CONTAINING 'KING':


,match_id,player_name,player_id
38728,1490414,A King,83558266
31088,1406079,A King,83558266
39466,1490442,A King,83558266
39357,1490438,A King,83558266
39274,1490435,A King,83558266
39110,1490429,A King,83558266
39000,1490425,A King,83558266
38892,1490421,A King,83558266
23509,1336078,A King,83558266
38591,1488250,A King,83558266



CRICSHEET APPEARANCE ENTRIES CONTAINING 'KING':


,date,team,player_name,source_player_id,match_id,role,source
6529,2020-01-07,West Indies,BA King,7fca84b7,1203673,batting,Cricsheet
7587,2021-05-20,Netherlands,VJ Kingma,3a7ea906,1260085,batting,Cricsheet
39793,2005-02-01,West Indies,Reon King,20fbf68c,65665,batting,Cricsheet


In [84]:
# Check whether the three verified identities already exist

verified_wi_ids = {
    "Brandon King": ("BA King", "7fca84b7"),
    "Carlos Brathwaite": ("CR Brathwaite", "e342e5fb"),
    "Kieran Powell": ("KOA Powell", "e012ad13")
}

for full_name, (cricsheet_name, player_id) in verified_wi_ids.items():

    print("\n", full_name, "|", cricsheet_name, "|", player_id)

    existing = player_identity_map_updated[
        player_identity_map_updated["player_id"].astype(str)
        == player_id
    ]

    if len(existing):
        display(existing)
    else:
        print("Not yet present in identity map.")


 Brandon King | BA King | 7fca84b7


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,normalized_name,match_status,crickrida_player_name,cricsheet_player_name,cricsheet_player_id,mapping_method,confidence,notes
124,CRIQ_7fca84b7,7fca84b7,BA King,Brandon King,brandonking,verified_alias,NaN,NaN,NaN,NaN,NaN,NaN



 Carlos Brathwaite | CR Brathwaite | e342e5fb


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,normalized_name,match_status,crickrida_player_name,cricsheet_player_name,cricsheet_player_id,mapping_method,confidence,notes
117,CRIQ_e342e5fb,e342e5fb,CR Brathwaite,Carlos Brathwaite,carlosbrathwaite,verified_alias,NaN,NaN,NaN,NaN,NaN,NaN



 Kieran Powell | KOA Powell | e012ad13


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,normalized_name,match_status,crickrida_player_name,cricsheet_player_name,cricsheet_player_id,mapping_method,confidence,notes
125,CRIQ_e012ad13,e012ad13,KOA Powell,Kieran Powell,kieranpowell,verified_alias,NaN,NaN,NaN,NaN,NaN,NaN


In [85]:
# Enrich the three existing West Indies identity records

verified_wi_evidence = {
    "7fca84b7": {
        "method": "manual_full_name_career_and_afghanistan_match_verification",
        "notes": (
            "Brandon Alexander King. ESPNcricinfo ID 670035. "
            "West Indies ODI debut against Afghanistan on 2019-11-11. "
            "Cricsheet BA King identity confirmed; distinct from Reon King."
        )
    },
    "e342e5fb": {
        "method": "manual_full_name_career_and_afghanistan_match_verification",
        "notes": (
            "Carlos Ricardo Brathwaite. ESPNcricinfo ID 457249. "
            "Afghanistan ODI appearances on 2018-03-15, 2018-03-25 "
            "and 2019-07-04. Distinct from Kraigg Brathwaite."
        )
    },
    "e012ad13": {
        "method": "manual_full_name_career_and_afghanistan_match_verification",
        "notes": (
            "Kieran Omar Akeem Powell. ESPNcricinfo ID 252932. "
            "Afghanistan ODI appearances on 2017-06-09 and 2017-06-11. "
            "Distinct from Rovman Powell."
        )
    }
}

for player_id, evidence in verified_wi_evidence.items():

    mask = (
        player_identity_map_updated["player_id"].astype(str)
        == player_id
    )

    if mask.sum() != 1:
        raise ValueError(
            f"Expected exactly one record for {player_id}; "
            f"found {mask.sum()}"
        )

    player_identity_map_updated.loc[
        mask, "mapping_method"
    ] = evidence["method"]

    player_identity_map_updated.loc[
        mask, "confidence"
    ] = "high"

    player_identity_map_updated.loc[
        mask, "notes"
    ] = evidence["notes"]

player_identity_map_updated.to_csv(
    "player_identity_map_updated.csv",
    index=False
)

print("Existing records enriched:", len(verified_wi_evidence))
print("Master identity map:", player_identity_map_updated.shape)
print("Duplicate Cricsheet IDs:",
      player_identity_map_updated["player_id"].duplicated().sum())

Existing records enriched: 3
Master identity map: (189, 12)
Duplicate Cricsheet IDs: 0


In [86]:
# Audit all 27 Afghanistan West Indies players against the master map

wi_names = sorted(wi_afghanistan_players["player_name"].unique())

mapped_names = set(
    player_identity_map_updated["crickrida_name"]
    .dropna()
    .str.strip()
)

wi_mapped = [name for name in wi_names if name in mapped_names]
wi_unmapped = [name for name in wi_names if name not in mapped_names]

print("Total Afghanistan West Indies players:", len(wi_names))
print("Already mapped:", len(wi_mapped))
print("Still unmapped:", len(wi_unmapped))

print("\nALREADY MAPPED")
for name in wi_mapped:
    print("-", name)

print("\nSTILL UNMAPPED")
for name in wi_unmapped:
    print("-", name)

Total Afghanistan West Indies players: 27
Already mapped: 3
Still unmapped: 24

ALREADY MAPPED
- Brandon King
- Carlos Brathwaite
- Kieran Powell

STILL UNMAPPED
- Alzarri Joseph
- Ashley Nurse
- Chris Gayle
- Evin Lewis
- Fabian Allen
- Hayden Walsh
- Jason Holder
- Jason Mohammed
- Jonathan Carter
- Keemo Paul
- Kemar Roach
- Kieron Pollard
- Marlon Samuels
- Miguel Cummins
- Nicholas Pooran
- Nikita Miller
- Oshane Thomas
- Romario Shepherd
- Roston Chase
- Rovman Powell
- Shai Hope
- Shannon Gabriel
- Sheldon Cottrell
- Shimron Hetmyer


In [87]:
# Find Cricsheet registry entries for the 24 unmapped West Indies players

wi_candidate_names = {
    "Alzarri Joseph": "AS Joseph",
    "Ashley Nurse": "AR Nurse",
    "Chris Gayle": "CH Gayle",
    "Evin Lewis": "E Lewis",
    "Fabian Allen": "FA Allen",
    "Hayden Walsh": "HR Walsh",
    "Jason Holder": "JO Holder",
    "Jason Mohammed": "JN Mohammed",
    "Jonathan Carter": "JL Carter",
    "Keemo Paul": "KMA Paul",
    "Kemar Roach": "KAJ Roach",
    "Kieron Pollard": "KA Pollard",
    "Marlon Samuels": "MN Samuels",
    "Miguel Cummins": "ML Cummins",
    "Nicholas Pooran": "N Pooran",
    "Nikita Miller": "NO Miller",
    "Oshane Thomas": "O Thomas",
    "Romario Shepherd": "R Shepherd",
    "Roston Chase": "RL Chase",
    "Rovman Powell": "R Powell",
    "Shai Hope": "SD Hope",
    "Shannon Gabriel": "ST Gabriel",
    "Sheldon Cottrell": "SS Cottrell",
    "Shimron Hetmyer": "SO Hetmyer"
}

for full_name, expected_name in wi_candidate_names.items():

    matches = player_registry_df[
        player_registry_df["player_name"].apply(normalize_player_name)
        == normalize_player_name(expected_name)
    ][["player_name", "player_id"]].drop_duplicates()

    print(f"\n{full_name} -> {expected_name}")
    if len(matches):
        display(matches)
    else:
        print("No exact registry entry found.")


Alzarri Joseph -> AS Joseph


,player_name,player_id
1163,AS Joseph,b0946605



Ashley Nurse -> AR Nurse


,player_name,player_id
1107,AR Nurse,4a745f65



Chris Gayle -> CH Gayle


,player_name,player_id
1546,CH Gayle,db584dad


... (38 outputs left out)


Sheldon Cottrell -> SS Cottrell


,player_name,player_id
5214,SS Cottrell,a1d053dd



Shimron Hetmyer -> SO Hetmyer


,player_name,player_id
5186,SO Hetmyer,48a1d7b7


In [88]:
# Build verified candidate mapping from the registry output

wi_verified_mapping = {
    "Alzarri Joseph": ("AS Joseph", "b0946605"),
    "Ashley Nurse": ("AR Nurse", "4a745f65"),
    "Chris Gayle": ("CH Gayle", "db584dad"),
    "Evin Lewis": ("E Lewis", "0ebfb1ad"),
    "Fabian Allen": ("FA Allen", "b52ffbbd"),
    "Hayden Walsh": ("HR Walsh", "1ac2a995"),
    "Jason Holder": ("JO Holder", "0f721006"),
    "Jason Mohammed": ("JN Mohammed", "773cec26"),
    "Jonathan Carter": ("JL Carter", "0a9d8b61"),
    "Keemo Paul": ("KMA Paul", "75224f22"),
    "Kemar Roach": ("KAJ Roach", "78eb4223"),
    "Kieron Pollard": ("KA Pollard", "a757b0d8"),
    "Marlon Samuels": ("MN Samuels", "f846de6a"),
    "Miguel Cummins": ("ML Cummins", "92aa28fe"),
    "Nicholas Pooran": ("N Pooran", "3241e3fd"),
    "Nikita Miller": ("NO Miller", "ea3ffd1b"),
    "Oshane Thomas": ("O Thomas", "c6097d68"),
    "Romario Shepherd": ("R Shepherd", "c5aef772"),
    "Roston Chase": ("RL Chase", "3feda4fa"),
    "Rovman Powell": ("R Powell", "650d5e49"),
    "Shai Hope": ("SD Hope", "1fc6ef83"),
    "Shannon Gabriel": ("ST Gabriel", "d9609443"),
    "Sheldon Cottrell": ("SS Cottrell", "a1d053dd"),
    "Shimron Hetmyer": ("SO Hetmyer", "48a1d7b7")
}

existing_ids = set(
    player_identity_map_updated["player_id"].astype(str)
)

for full_name, (cricsheet_name, player_id) in wi_verified_mapping.items():
    status = "ALREADY EXISTS" if player_id in existing_ids else "NEW"
    print(f"{full_name:20} | {cricsheet_name:12} | {player_id} | {status}")

print("\nTotal candidates:", len(wi_verified_mapping))
print("Already in master map:", sum(
    player_id in existing_ids
    for _, player_id in wi_verified_mapping.values()
))
print("New candidates:", sum(
    player_id not in existing_ids
    for _, player_id in wi_verified_mapping.values()
))

Alzarri Joseph       | AS Joseph    | b0946605 | NEW
Ashley Nurse         | AR Nurse     | 4a745f65 | NEW
Chris Gayle          | CH Gayle     | db584dad | NEW
Evin Lewis           | E Lewis      | 0ebfb1ad | NEW
Fabian Allen         | FA Allen     | b52ffbbd | NEW
Hayden Walsh         | HR Walsh     | 1ac2a995 | NEW
Jason Holder         | JO Holder    | 0f721006 | NEW
Jason Mohammed       | JN Mohammed  | 773cec26 | NEW
Jonathan Carter      | JL Carter    | 0a9d8b61 | NEW
Keemo Paul           | KMA Paul     | 75224f22 | NEW
Kemar Roach          | KAJ Roach    | 78eb4223 | NEW
Kieron Pollard       | KA Pollard   | a757b0d8 | NEW
Marlon Samuels       | MN Samuels   | f846de6a | NEW
Miguel Cummins       | ML Cummins   | 92aa28fe | NEW
Nicholas Pooran      | N Pooran     | 3241e3fd | NEW
Nikita Miller        | NO Miller    | ea3ffd1b | NEW
Oshane Thomas        | O Thomas     | c6097d68 | NEW
Romario Shepherd     | R Shepherd   | c5aef772 | NEW
Roston Chase         | RL Chase     | 3feda4fa

In [89]:
# Add the 24 verified West Indies identities

new_wi_rows = []

for full_name, (cricsheet_name, player_id) in wi_verified_mapping.items():

    new_wi_rows.append({
        "cricketiq_player_id": f"CRIQ_{player_id}",
        "player_id": player_id,
        "cricsheet_name": cricsheet_name,
        "crickrida_name": full_name,
        "normalized_name": normalize_player_name(full_name),
        "match_status": "verified_alias",
        "crickrida_player_name": full_name,
        "cricsheet_player_name": cricsheet_name,
        "cricsheet_player_id": player_id,
        "mapping_method": "verified_registry_and_scorecard_identity",
        "confidence": "high",
        "notes": (
            "West Indies player identity matched using the "
            "Cricsheet registry player ID and Afghanistan ODI "
            "scorecard name."
        )
    })

new_wi_df = pd.DataFrame(new_wi_rows)

# Safety checks before adding
assert len(new_wi_df) == 24
assert not new_wi_df["player_id"].duplicated().any()
assert not new_wi_df["player_id"].astype(str).isin(
    player_identity_map_updated["player_id"].astype(str)
).any()

# Add to master map
player_identity_map_updated = pd.concat(
    [player_identity_map_updated, new_wi_df],
    ignore_index=True
)

# Save updated identity map
player_identity_map_updated.to_csv(
    "player_identity_map_updated.csv",
    index=False
)

print("New identities added:", len(new_wi_df))
print("Updated master map:", player_identity_map_updated.shape)
print(
    "Duplicate Cricsheet IDs:",
    player_identity_map_updated["player_id"].duplicated().sum()
)

New identities added: 24
Updated master map: (213, 12)
Duplicate Cricsheet IDs: 0


In [90]:
# Validate Afghanistan West Indies player identity coverage

wi_validation = wi_afghanistan_players[
    ["player_name"]
].drop_duplicates().copy()

wi_validation = wi_validation.merge(
    player_identity_map_updated[
        ["crickrida_name", "cricsheet_name", "player_id",
         "confidence", "mapping_method"]
    ],
    left_on="player_name",
    right_on="crickrida_name",
    how="left"
)

wi_validation["mapping_status"] = wi_validation["player_id"].notna().map({
    True: "Mapped",
    False: "Unmapped"
})

display(
    wi_validation.sort_values(
        ["mapping_status", "player_name"]
    )
)

print("\nTotal Afghanistan WI players:", len(wi_validation))
print("Mapped:", (wi_validation["mapping_status"] == "Mapped").sum())
print("Unmapped:", (wi_validation["mapping_status"] == "Unmapped").sum())
print("Duplicate player IDs in these mappings:",
      wi_validation["player_id"].dropna().duplicated().sum())

,player_name,crickrida_name,cricsheet_name,player_id,confidence,mapping_method,mapping_status
8,Alzarri Joseph,Alzarri Joseph,AS Joseph,b0946605,high,verified_registry_and_scorecard_identity,Mapped
7,Ashley Nurse,Ashley Nurse,AR Nurse,4a745f65,high,verified_registry_and_scorecard_identity,Mapped
23,Brandon King,Brandon King,BA King,7fca84b7,high,manual_full_name_career_and_afghanistan_match_verification,Mapped
15,Carlos Brathwaite,Carlos Brathwaite,CR Brathwaite,e342e5fb,high,manual_full_name_career_and_afghanistan_match_verification,Mapped
12,Chris Gayle,Chris Gayle,CH Gayle,db584dad,high,verified_registry_and_scorecard_identity,Mapped
0,Evin Lewis,Evin Lewis,E Lewis,0ebfb1ad,high,verified_registry_and_scorecard_identity,Mapped
19,Fabian Allen,Fabian Allen,FA Allen,b52ffbbd,high,verified_registry_and_scorecard_identity,Mapped
22,Hayden Walsh,Hayden Walsh,HR Walsh,1ac2a995,high,verified_registry_and_scorecard_identity,Mapped
6,Jason Holder,Jason Holder,JO Holder,0f721006,high,verified_registry_and_scorecard_identity,Mapped
3,Jason Mohammed,Jason Mohammed,JN Mohammed,773cec26,high,verified_registry_and_scorecard_identity,Mapped



Total Afghanistan WI players: 27
Mapped: 27
Unmapped: 0
Duplicate player IDs in these mappings: 0


## 5. Extended Country-Wise Identity Reconciliation

Continue identity resolution through country-specific source history investigations. Review and correct identity collisions, stage proposed mappings, validate candidates, and retain confirmed source-only identities where no counterpart is established.

In [91]:

# ============================================================
# CRICKETIQ - NOTEBOOK 06 FULL RUNTIME CHECKPOINT
# Saves the current Python session without rerunning old cells
# ============================================================

import cloudpickle
import gzip
import types
from pathlib import Path
from datetime import datetime

# Create timestamped checkpoint filename
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

checkpoint_dir = (
    Path(r"C:\CricIQ")
    / "backups"
    / "notebook_checkpoints"
)
checkpoint_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = (
    checkpoint_dir / f"notebook06_checkpoint_{timestamp}.pkl.gz"
)

# Exclude temporary Jupyter and module objects
excluded_names = {
    "In", "Out", "get_ipython",
    "exit", "quit", "display", "print"
}

# Capture all useful notebook variables and functions
checkpoint_state = {
    name: value
    for name, value in globals().copy().items()
    if not name.startswith("_")
    and name not in excluded_names
    and not isinstance(value, types.ModuleType)
}

# Save metadata
checkpoint_state["__checkpoint_metadata__"] = {
    "project": "CricketIQ",
    "notebook": "06_player_data_processing.ipynb",
    "stage": "Player identity processing",
    "last_completed_cell": 288,
    "saved_at": timestamp,
    "identity_map_records": len(player_identity_map_updated)
}

# Write safely using a temporary file
temp_path = checkpoint_path.with_name(
    checkpoint_path.name + ".tmp"
)

try:
    with gzip.open(temp_path, "wb") as f:
        cloudpickle.dump(
            checkpoint_state,
            f,
            protocol=5
        )

    temp_path.replace(checkpoint_path)

    print("=" * 55)
    print("CRICKETIQ CHECKPOINT SAVED SUCCESSFULLY")
    print("=" * 55)

    print("File:", checkpoint_path)
    print("Variables saved:", len(checkpoint_state))
    print("Identity map:", player_identity_map_updated.shape)
    print("West Indies validation:", wi_validation.shape)
    print("Checkpoint size:",
          f"{checkpoint_path.stat().st_size / (1024**2):.2f} MB")

except Exception as e:
    if temp_path.exists():
        temp_path.unlink()

    print("CHECKPOINT FAILED:", repr(e))
    raise


CRICKETIQ CHECKPOINT SAVED SUCCESSFULLY
File: c:\CricIQ\notebooks\notebook06_checkpoint_20261116_024853.pkl.gz
Variables saved: 630
Identity map: (213, 12)
West Indies validation: (27, 7)
Checkpoint size: 62.62 MB


In [1]:

# ============================================================
# CRICKETIQ - RESTORE NOTEBOOK 06 CHECKPOINT
# ============================================================

import cloudpickle
import gzip
import pandas as pd
import numpy as np

from pathlib import Path

# Find the latest checkpoint in the current directory
checkpoint_dir = (
    Path(r"C:\CricIQ")
    / "backups"
    / "notebook_checkpoints"
)

checkpoint_files = list(
    checkpoint_dir.glob("notebook06_checkpoint_*.pkl.gz")
)

if not checkpoint_files:
    raise FileNotFoundError(
        "No Notebook 06 checkpoint found!"
    )

checkpoint_path = max(
    checkpoint_files,
    key=lambda p: p.stat().st_mtime
)

print("Loading:", checkpoint_path.name)

# Load saved notebook state
with gzip.open(checkpoint_path, "rb") as f:
    restored_state = cloudpickle.load(f)

# Extract metadata
checkpoint_metadata = restored_state.pop(
    "__checkpoint_metadata__", {}
)

# Restore variables and functions
globals().update(restored_state)

print("\n" + "=" * 55)
print("CRICKETIQ CHECKPOINT RESTORED")
print("=" * 55)

print("Saved at:", checkpoint_metadata.get("saved_at"))
print("Last completed cell:",
      checkpoint_metadata.get("last_completed_cell"))

print("\nRestored identity map:",
      player_identity_map_updated.shape)

print("Restored registry:",
      player_registry_final.shape)

print("Restored West Indies validation:",
      wi_validation.shape)

print("\nNotebook state is ready to continue.")


Loading: notebook06_checkpoint_20261116_024853.pkl.gz

CRICKETIQ CHECKPOINT RESTORED
Saved at: 20261116_024853
Last completed cell: 288

Restored identity map: (213, 12)
Restored registry: (3635, 9)
Restored West Indies validation: (27, 7)

Notebook state is ready to continue.


In [4]:

# ============================================================
# CRICKETIQ - PLAYER REGISTRY VALIDATION (READ ONLY)
# ============================================================

import pandas as pd

registry = player_registry_final.copy()
identity_map = player_identity_map_updated.copy()

print("=" * 60)
print("CRICKETIQ PLAYER REGISTRY VALIDATION")
print("=" * 60)

# 1. Dataset dimensions
print("\n1. DATASET DIMENSIONS")
print("Registry:", registry.shape)
print("Identity map:", identity_map.shape)

# 2. CricketIQ identity uniqueness
print("\n2. CRICKETIQ ID VALIDATION")

print("Registry duplicate CricketIQ IDs:",
      registry["cricketiq_player_id"].duplicated().sum())

print("Identity map duplicate CricketIQ IDs:",
      identity_map["cricketiq_player_id"].duplicated().sum())

print("Registry missing CricketIQ IDs:",
      registry["cricketiq_player_id"].isna().sum())

# 3. Source player ID validation
print("\n3. SOURCE PLAYER ID VALIDATION")

print("Registry duplicate player IDs within source:")
print(
    registry.duplicated(
        subset=["source", "player_id"]
    ).sum()
)

# 4. Check list-valued columns
print("\n4. LIST COLUMN INSPECTION")

if "player_names" in registry.columns:
    list_count = registry["player_names"].apply(
        lambda x: isinstance(x, list)
    ).sum()

    print("Rows containing player_names lists:", list_count)
    print("Example:", registry["player_names"].iloc[0])

# 5. Identity map versus registry
print("\n5. IDENTITY MAP COVERAGE")

map_ids = set(
    identity_map["cricketiq_player_id"].dropna()
)

registry_ids = set(
    registry["cricketiq_player_id"].dropna()
)

missing_from_registry = map_ids - registry_ids
extra_in_registry = registry_ids - map_ids

print("Identity map unique IDs:", len(map_ids))
print("Registry unique IDs:", len(registry_ids))
print("Mapped IDs missing from registry:",
      len(missing_from_registry))
print("Additional registry IDs:",
      len(extra_in_registry))

# 6. Display missing identities
if missing_from_registry:
    print("\n6. MISSING IDENTITY RECORDS")

    display(
        identity_map[
            identity_map["cricketiq_player_id"].isin(
                missing_from_registry
            )
        ][[
            "cricketiq_player_id",
            "cricsheet_name",
            "crickrida_name",
            "match_status"
        ]]
    )
else:
    print("\nAll identity-map IDs exist in registry.")

# 7. Source distribution
print("\n7. REGISTRY SOURCE DISTRIBUTION")

print(registry["source"].value_counts(dropna=False))

print("\nVALIDATION COMPLETED - NO DATA MODIFIED")


CRICKETIQ PLAYER REGISTRY VALIDATION

1. DATASET DIMENSIONS
Registry: (3635, 9)
Identity map: (213, 12)

2. CRICKETIQ ID VALIDATION
Registry duplicate CricketIQ IDs: 0
Identity map duplicate CricketIQ IDs: 0
Registry missing CricketIQ IDs: 0

3. SOURCE PLAYER ID VALIDATION
Registry duplicate player IDs within source:
347

4. LIST COLUMN INSPECTION
Rows containing player_names lists: 3181
Example: nan

5. IDENTITY MAP COVERAGE
Identity map unique IDs: 213
Registry unique IDs: 3635
Mapped IDs missing from registry: 107
Additional registry IDs: 3529

6. MISSING IDENTITY RECORDS


,cricketiq_player_id,cricsheet_name,crickrida_name,match_status
106,CRIQ_e1b9f3a9,BJ Dwarshuis,Ben Dwarshuis,verified_alias
107,CRIQ_c58b0108,Mehedi Hasan Miraz,Mehidy Hasan Miraz,verified_alias
108,CRIQ_d12143bf,JM Anderson,James Anderson,verified_alias
109,CRIQ_50c6bc2b,LS Livingstone,Liam Livingstone,verified_alias
110,CRIQ_8d92a2c3,MA Wood,Mark Wood,verified_alias
...,...,...,...,...
208,CRIQ_650d5e49,R Powell,Rovman Powell,verified_alias
209,CRIQ_1fc6ef83,SD Hope,Shai Hope,verified_alias
210,CRIQ_d9609443,ST Gabriel,Shannon Gabriel,verified_alias
211,CRIQ_a1d053dd,SS Cottrell,Sheldon Cottrell,verified_alias



7. REGISTRY SOURCE DISTRIBUTION
source
Cricsheet    3181
Crickrida     348
Both          106
Name: count, dtype: int64

VALIDATION COMPLETED - NO DATA MODIFIED


In [5]:

# ============================================================
# CRICKETIQ - INVESTIGATE MISSING REGISTRY IDENTITIES
# READ ONLY - NO DATA MODIFICATION
# ============================================================

# Identify updated mappings absent from the registry
missing_map = player_identity_map_updated[
    ~player_identity_map_updated["cricketiq_player_id"].isin(
        player_registry_final["cricketiq_player_id"]
    )
].copy()

print("=" * 60)
print("MISSING IDENTITY INVESTIGATION")
print("=" * 60)

print("\nMissing identity records:", len(missing_map))

# Display complete missing mapping information
display(
    missing_map[[
        "cricketiq_player_id",
        "player_id",
        "cricsheet_name",
        "crickrida_name",
        "cricsheet_player_id",
        "match_status",
        "mapping_method",
        "confidence",
        "notes"
    ]].to_string(index=False)
)

# Check whether the source player IDs already exist
print("\nSOURCE PLAYER ID CHECK")

existing_player_ids = set(
    player_registry_final["player_id"].dropna().astype(str)
)

missing_map["player_id_exists_in_registry"] = (
    missing_map["player_id"].astype(str).isin(
        existing_player_ids
    )
)

print(
    missing_map["player_id_exists_in_registry"]
    .value_counts(dropna=False)
)

# Investigate duplicate source/player ID combinations
print("\nDUPLICATE SOURCE / PLAYER ID INVESTIGATION")

registry_duplicates = player_registry_final[
    player_registry_final.duplicated(
        subset=["source", "player_id"],
        keep=False
    )
].sort_values(["source", "player_id"])

print("Rows involved:", len(registry_duplicates))

display(registry_duplicates.head(30))

# Check null player IDs separately
print("\nNULL SOURCE PLAYER IDs")

print(
    player_registry_final.groupby("source")["player_id"]
    .apply(lambda s: s.isna().sum())
)

print("\nINVESTIGATION COMPLETED - ORIGINAL DATA UNCHANGED")


MISSING IDENTITY INVESTIGATION

Missing identity records: 107


"cricketiq_player_id player_id     cricsheet_name       crickrida_name cricsheet_player_id                               match_status                                             mapping_method confidence                                                                                                                                                                   notes\n      CRIQ_e1b9f3a9  e1b9f3a9       BJ Dwarshuis        Ben Dwarshuis                 NaN                             verified_alias                                                        NaN        NaN                                                                                                                                                                     NaN\n      CRIQ_c58b0108  c58b0108 Mehedi Hasan Miraz   Mehidy Hasan Miraz                 NaN                             verified_alias                                                        NaN        NaN                                                     


SOURCE PLAYER ID CHECK
player_id_exists_in_registry
True    107
Name: count, dtype: int64

DUPLICATE SOURCE / PLAYER ID INVESTIGATION
Rows involved: 348


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,match_status,source,player_names,player_name,canonical_player_name
3288,CR_paulstirling,NaN,NaN,NaN,source_only,Crickrida,NaN,Paul Stirling,Paul Stirling
3289,CR_jameshall,NaN,NaN,NaN,source_only,Crickrida,NaN,James Hall,James Hall
3290,CR_alexcusack,NaN,NaN,NaN,source_only,Crickrida,NaN,Alex Cusack,Alex Cusack
3291,CR_kevinobrien,NaN,NaN,NaN,source_only,Crickrida,NaN,Kevin O'Brien,Kevin O'Brien
3292,CR_andrewpoynter,NaN,NaN,NaN,source_only,Crickrida,NaN,Andrew Poynter,Andrew Poynter
3293,CR_andrewwhite,NaN,NaN,NaN,source_only,Crickrida,NaN,Andrew White,Andrew White
3294,CR_johnmooney,NaN,NaN,NaN,source_only,Crickrida,NaN,John Mooney,John Mooney
3295,CR_trentjohnston,NaN,NaN,NaN,source_only,Crickrida,NaN,Trent Johnston,Trent Johnston
3296,CR_nigeljones,NaN,NaN,NaN,source_only,Crickrida,NaN,Nigel Jones,Nigel Jones
3297,CR_rorymccann,NaN,NaN,NaN,source_only,Crickrida,NaN,Rory McCann,Rory McCann



NULL SOURCE PLAYER IDs
source
Both           0
Crickrida    348
Cricsheet      0
Name: player_id, dtype: int64

INVESTIGATION COMPLETED - ORIGINAL DATA UNCHANGED


In [6]:
# ============================================================
# STEP 1: COMPARE THE 107 MISSING IDENTITY RECORDS
# READ-ONLY — DOES NOT MODIFY ORIGINAL DATAFRAMES
# ============================================================

missing_check = missing_map.copy()
registry_check = player_registry_final.copy()

# Normalize IDs only in temporary copies
missing_check["player_id_check"] = (
    missing_check["player_id"].astype("string").str.strip()
)

registry_check["player_id_check"] = (
    registry_check["player_id"].astype("string").str.strip()
)

# Match using the underlying player_id
comparison = missing_check.merge(
    registry_check[
        [
            "player_id_check",
            "cricketiq_player_id",
            "source",
            "player_name",
            "canonical_player_name"
        ]
    ],
    on="player_id_check",
    how="left",
    suffixes=("_identity", "_registry"),
    indicator=True
)

print("IDENTITY-TO-REGISTRY COMPARISON")
print("Total identity records:", len(missing_check))
print("Matched registry rows:", (comparison["_merge"] == "both").sum())
print("Unmatched identity rows:", (comparison["_merge"] == "left_only").sum())

display(
    comparison[
        [
            "cricketiq_player_id_identity",
            "player_id_check",
            "cricsheet_name",
            "crickrida_name",
            "cricketiq_player_id_registry",
            "source",
            "player_name",
            "canonical_player_name",
            "_merge"
        ]
    ].head(30)
)

# Check whether any player_id maps to multiple registry rows
registry_id_counts = (
    registry_check.dropna(subset=["player_id_check"])
    .groupby("player_id_check")
    .size()
)

print("\nPlayer IDs mapping to multiple registry rows:",
      (registry_id_counts > 1).sum())

# Check duplicate non-null source/player_id combinations
non_null_registry = registry_check.dropna(
    subset=["player_id_check"]
)

print("\nNon-null duplicate (source, player_id) rows:",
      non_null_registry.duplicated(
          subset=["source", "player_id_check"],
          keep=False
      ).sum())

IDENTITY-TO-REGISTRY COMPARISON
Total identity records: 107
Matched registry rows: 107
Unmatched identity rows: 0


,cricketiq_player_id_identity,player_id_check,cricsheet_name,crickrida_name,cricketiq_player_id_registry,source,player_name,canonical_player_name,_merge
0,CRIQ_e1b9f3a9,e1b9f3a9,BJ Dwarshuis,Ben Dwarshuis,CS_e1b9f3a9,Cricsheet,NaN,BJ Dwarshuis,both
1,CRIQ_c58b0108,c58b0108,Mehedi Hasan Miraz,Mehidy Hasan Miraz,CS_c58b0108,Cricsheet,NaN,Mehedi Hasan Miraz,both
2,CRIQ_d12143bf,d12143bf,JM Anderson,James Anderson,CS_d12143bf,Cricsheet,NaN,JM Anderson,both
3,CRIQ_50c6bc2b,50c6bc2b,LS Livingstone,Liam Livingstone,CS_50c6bc2b,Cricsheet,NaN,LS Livingstone,both
4,CRIQ_8d92a2c3,8d92a2c3,MA Wood,Mark Wood,CS_8d92a2c3,Cricsheet,NaN,MA Wood,both
5,CRIQ_fc55ec67,fc55ec67,MR Adair,Mark Adair,CS_fc55ec67,Cricsheet,NaN,MR Adair,both
6,CRIQ_6ef13460,6ef13460,A Balbirnie,Andy Balbirnie,CS_6ef13460,Cricsheet,NaN,A Balbirnie,both
7,CRIQ_c7105dab,c7105dab,JA McCollum,James McCollum,CS_c7105dab,Cricsheet,NaN,JA McCollum,both
8,CRIQ_e824e6ee,e824e6ee,TWM Latham,Tom Latham,CS_e824e6ee,Cricsheet,NaN,TWM Latham,both
9,CRIQ_94d7f855,94d7f855,C de Grandhomme,Colin de Grandhomme,CS_94d7f855,Cricsheet,NaN,C de Grandhomme,both



Player IDs mapping to multiple registry rows: 0

Non-null duplicate (source, player_id) rows: 0


In [8]:
# Find all currently available DataFrames related to player data

import pandas as pd

for name, obj in globals().copy().items():
    if isinstance(obj, pd.DataFrame):
        if any(word in name.lower() for word in [
            "player", "batting", "bowling", "dismissal", "wicket"
        ]):
            print(f"{name:45} {obj.shape}")

afg_batting_df                                (2719, 13)
afg_bowling_df                                (1765, 15)
player_registry_df                            (86666, 3)
player_master_df                              (3287, 5)
player_lookup                                 (3294, 2)
match_player_lookup                           (86666, 3)
player_participation_df                       (55172, 4)
batting_deliveries                            (1379686, 22)
player_batting_df                             (45432, 12)
calculated_batting_totals                     (5143, 3)
batting_validation_df                         (5143, 11)
player_runs_validation                        (5143, 3)
dismissals_df                                 (37772, 12)
player_out_lookup                             (86666, 3)
valid_dismissals                              (37711, 12)
player_dismissal_df                           (37711, 8)
bowling_deliveries                            (1379686, 24)
bowler_dismissals         

In [9]:

import pandas as pd

dataframes_to_check = {
    "Identity Map": player_identity_map_updated,
    "Updated Registry": player_registry_final_updated,
    "Updated Batting": player_batting_updated,
    "Updated Bowling": player_bowling_updated,
    "Updated Dismissals": player_dismissal_updated,
    "Updated Bowler Dismissals": dismissal_bowler_updated,
    "Updated Afghanistan Batting": afg_player_batting_updated,
    "Updated Afghanistan Bowling": afg_player_bowling_updated
}

for name, df in dataframes_to_check.items():

    print("\n" + "=" * 70)
    print(name, "| Shape:", df.shape)

    id_columns = [
        col for col in df.columns
        if "cricketiq_player_id" in col.lower()
        or col.lower() == "player_id"
    ]

    if not id_columns:
        print("No CricketIQ/player ID column found.")
        print("Available columns:", df.columns.tolist())
        continue

    for col in id_columns:

        values = df[col].dropna().astype(str).str.strip()

        prefixes = (
            values.str.extract(r"^([A-Za-z]+)_", expand=False)
            .value_counts()
            .to_dict()
        )

        print(f"\nColumn: {col}")
        print("Non-null IDs:", len(values))
        print("Unique IDs:", values.nunique())
        print("Prefix distribution:", prefixes)
        print("Sample IDs:", values.drop_duplicates().head(5).tolist())



Identity Map | Shape: (213, 12)

Column: cricketiq_player_id
Non-null IDs: 213
Unique IDs: 213
Prefix distribution: {'CRIQ': 213}
Sample IDs: ['CRIQ_8a75e999', 'CRIQ_2911de16', 'CRIQ_9cb8d7a6', 'CRIQ_e174dadd', 'CRIQ_9ab63e7b']

Column: player_id
Non-null IDs: 213
Unique IDs: 213
Prefix distribution: {}
Sample IDs: ['8a75e999', '2911de16', '9cb8d7a6', 'e174dadd', '9ab63e7b']

Updated Registry | Shape: (3620, 10)

Column: cricketiq_player_id
Non-null IDs: 3620
Unique IDs: 3620
Prefix distribution: {'CS': 3166, 'CR': 333, 'CRIQ': 121}
Sample IDs: ['CRIQ_8a75e999', 'CRIQ_2911de16', 'CRIQ_9cb8d7a6', 'CRIQ_e174dadd', 'CRIQ_9ab63e7b']

Column: player_id
Non-null IDs: 3287
Unique IDs: 3287
Prefix distribution: {}
Sample IDs: ['8a75e999', '2911de16', '9cb8d7a6', 'e174dadd', '9ab63e7b']

Updated Batting | Shape: (45432, 12)

Column: cricketiq_player_id
Non-null IDs: 45432
Unique IDs: 1934
Prefix distribution: {'CS': 39263, 'CRIQ': 6169}
Sample IDs: ['CS_6834d1f2', 'CS_45eda7c8', 'CS_dcce6f09',

In [10]:

# ============================================================
# STEP 4: TRACE IDENTITY MAP COVERAGE
# READ-ONLY — NO DATAFRAME MODIFICATIONS
# ============================================================

import pandas as pd

identity_ids = set(
    player_identity_map_updated["cricketiq_player_id"]
    .dropna().astype(str)
)

registry_updated_ids = set(
    player_registry_updated["cricketiq_player_id"]
    .dropna().astype(str)
)

registry_final_ids = set(
    player_registry_final_updated["cricketiq_player_id"]
    .dropna().astype(str)
)

print("=" * 65)
print("IDENTITY MAP vs REGISTRY")
print("=" * 65)

print("Identity map IDs:", len(identity_ids))
print("Updated registry IDs:", len(registry_updated_ids))
print("Final updated registry IDs:", len(registry_final_ids))

print("\nIdentity IDs in player_registry_updated:",
      len(identity_ids & registry_updated_ids))

print("Identity IDs in player_registry_final_updated:",
      len(identity_ids & registry_final_ids))

missing_from_final = identity_ids - registry_final_ids

print("\nIdentity IDs missing from final registry:",
      len(missing_from_final))

# Inspect missing IDs and their names
missing_identity_details = player_identity_map_updated[
    player_identity_map_updated["cricketiq_player_id"].isin(
        missing_from_final
    )
].copy()

display(
    missing_identity_details[
        [
            "cricketiq_player_id",
            "player_id",
            "cricsheet_name",
            "crickrida_name",
            "match_status",
            "mapping_method"
        ]
    ].sort_values("cricsheet_name")
)

# Check whether these IDs appear in performance datasets
performance_datasets = {
    "Batting": player_batting_updated,
    "Bowling": player_bowling_updated,
    "Dismissals": player_dismissal_updated,
    "Bowler Dismissals": dismissal_bowler_updated,
    "Afghanistan Batting": afg_player_batting_updated,
    "Afghanistan Bowling": afg_player_bowling_updated
}

print("\n" + "=" * 65)
print("USAGE OF MISSING IDs IN PERFORMANCE DATA")
print("=" * 65)

for name, df in performance_datasets.items():

    if "cricketiq_player_id" not in df.columns:
        print(name, ": No CricketIQ ID column")
        continue

    used_ids = set(
        df["cricketiq_player_id"].dropna().astype(str)
    )

    matched = missing_from_final & used_ids

    print(
        f"{name:25} "
        f"Missing IDs used: {len(matched)}"
    )

    if matched:
        print("  Examples:", sorted(matched)[:5])

# Compare the two registry versions
print("\n" + "=" * 65)
print("REGISTRY VERSION DIFFERENCES")
print("=" * 65)

print(
    "IDs in updated registry but not final:",
    len(registry_updated_ids - registry_final_ids)
)

print(
    "IDs in final but not updated registry:",
    len(registry_final_ids - registry_updated_ids)
)


IDENTITY MAP vs REGISTRY
Identity map IDs: 213
Updated registry IDs: 3635
Final updated registry IDs: 3620

Identity IDs in player_registry_updated: 121
Identity IDs in player_registry_final_updated: 121

Identity IDs missing from final registry: 92


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,match_status,mapping_method
171,CRIQ_a159b802,a159b802,A van der Merwe,Albert van der Merwe,verified_manual_identity,manual_full_name_career_match_verification
166,CRIQ_d8e79af0,d8e79af0,AC Botha,Andre Botha,verified_manual_identity,manual_full_name_career_match_verification
131,CRIQ_b98cdbf0,b98cdbf0,AD Poynter,Andrew Poynter,verified_alias,NaN
129,CRIQ_568e5a4a,568e5a4a,AR Cusack,Alex Cusack,verified_alias,NaN
154,CRIQ_e957b38f,e957b38f,AR McBrine,Andy McBrine,verified_full_name_and_career,full_name_and_career_overlap
...,...,...,...,...,...,...
143,CRIQ_70f2f922,70f2f922,TF van Woerkom,Theo van Woerkom,verified_alias,NaN
144,CRIQ_c89474d1,c89474d1,TJ Murtagh,Tim Murtagh,verified_alias,NaN
152,CRIQ_29b89ae8,29b89ae8,WB Rankin,Boyd Rankin,verified_full_name_and_career,full_name_and_career_overlap
170,CRIQ_c942a53b,c942a53b,WK McCallan,Kyle McCallan,verified_manual_identity,manual_full_name_career_match_verification



USAGE OF MISSING IDs IN PERFORMANCE DATA
Batting                   Missing IDs used: 0
Bowling                   Missing IDs used: 0
Dismissals                Missing IDs used: 0
Bowler Dismissals         Missing IDs used: 0
Afghanistan Batting       Missing IDs used: 0
Afghanistan Bowling       Missing IDs used: 0

REGISTRY VERSION DIFFERENCES
IDs in updated registry but not final: 15
IDs in final but not updated registry: 0


In [11]:

# ============================================================
# STEP 5: VERIFY IDENTITY COVERAGE AND REGISTRY CHANGES
# READ-ONLY
# ============================================================

# 1. Compare all identity mappings using the underlying player_id

identity_check = player_identity_map_updated.copy()
registry_check = player_registry_final_updated.copy()

identity_check["player_id_check"] = (
    identity_check["player_id"].astype("string").str.strip()
)

registry_check["player_id_check"] = (
    registry_check["player_id"].astype("string").str.strip()
)

player_id_comparison = identity_check.merge(
    registry_check[
        [
            "player_id_check",
            "cricketiq_player_id",
            "source",
            "player_name",
            "canonical_player_name"
        ]
    ],
    on="player_id_check",
    how="left",
    suffixes=("_identity", "_registry"),
    indicator=True
)

print("=" * 65)
print("IDENTITY COVERAGE BY PLAYER ID")
print("=" * 65)

print("Total identity mappings:", len(identity_check))
print(
    "Matched by player_id:",
    (player_id_comparison["_merge"] == "both").sum()
)
print(
    "Unmatched by player_id:",
    (player_id_comparison["_merge"] == "left_only").sum()
)

print("\nMatched records with different CricketIQ IDs:")

different_ids = player_id_comparison[
    (player_id_comparison["_merge"] == "both") &
    (
        player_id_comparison["cricketiq_player_id_identity"] !=
        player_id_comparison["cricketiq_player_id_registry"]
    )
]

print("Count:", len(different_ids))

display(
    different_ids[
        [
            "cricketiq_player_id_identity",
            "cricketiq_player_id_registry",
            "player_id_check",
            "cricsheet_name",
            "crickrida_name",
            "source"
        ]
    ].head(30)
)

# 2. Investigate the 15 IDs removed from the final registry

updated_ids = set(
    player_registry_updated["cricketiq_player_id"].dropna().astype(str)
)

final_ids = set(
    player_registry_final_updated["cricketiq_player_id"].dropna().astype(str)
)

removed_ids = updated_ids - final_ids

removed_rows = player_registry_updated[
    player_registry_updated["cricketiq_player_id"].isin(removed_ids)
].copy()

print("\n" + "=" * 65)
print("REGISTRY RECORDS REMOVED")
print("=" * 65)

print("Removed IDs:", len(removed_ids))
print("Removed rows:", len(removed_rows))

display(removed_rows)

# 3. Compare source distributions

print("\nSOURCE DISTRIBUTION — UPDATED REGISTRY")
print(player_registry_updated["source"].value_counts(dropna=False))

print("\nSOURCE DISTRIBUTION — FINAL UPDATED REGISTRY")
print(player_registry_final_updated["source"].value_counts(dropna=False))

print("\nINVESTIGATION COMPLETE — ORIGINAL DATA UNCHANGED")


IDENTITY COVERAGE BY PLAYER ID
Total identity mappings: 213
Matched by player_id: 213
Unmatched by player_id: 0

Matched records with different CricketIQ IDs:
Count: 92


,cricketiq_player_id_identity,cricketiq_player_id_registry,player_id_check,cricsheet_name,crickrida_name,source
121,CRIQ_dcce6f09,CS_dcce6f09,dcce6f09,DA Warner,David Warner,Cricsheet
122,CRIQ_23eeb873,CS_23eeb873,23eeb873,DL Chahar,Deepak Chahar,Cricsheet
123,CRIQ_243431b5,CS_243431b5,243431b5,KJ O'Brien,Kevin O'Brien,Cricsheet
124,CRIQ_7fca84b7,CS_7fca84b7,7fca84b7,BA King,Brandon King,Cricsheet
125,CRIQ_e012ad13,CS_e012ad13,e012ad13,KOA Powell,Kieran Powell,Cricsheet
126,CRIQ_a90e53ec,CS_a90e53ec,a90e53ec,MW Short,Matthew Short,Cricsheet
127,CRIQ_e01895bb,CS_e01895bb,e01895bb,DT Johnston,Trent Johnston,Cricsheet
128,CRIQ_cf494ffe,CS_cf494ffe,cf494ffe,PR Stirling,Paul Stirling,Cricsheet
129,CRIQ_568e5a4a,CS_568e5a4a,568e5a4a,AR Cusack,Alex Cusack,Cricsheet
130,CRIQ_f27f89e8,CS_f27f89e8,f27f89e8,AR White,Andrew White,Cricsheet



REGISTRY RECORDS REMOVED
Removed IDs: 15
Removed rows: 15


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,match_status,source,player_names,player_name,canonical_player_name,normalized_name
3312,CR_edjoyce,NaN,NaN,NaN,source_only,Crickrida,NaN,Ed Joyce,Ed Joyce,NaN
3363,CR_andybalbirnie,NaN,NaN,NaN,source_only,Crickrida,NaN,Andy Balbirnie,Andy Balbirnie,NaN
3434,CR_carlosbrathwaite,NaN,NaN,NaN,source_only,Crickrida,NaN,Carlos Brathwaite,Carlos Brathwaite,NaN
3446,CR_mehidyhasanmiraz,NaN,NaN,NaN,source_only,Crickrida,NaN,Mehidy Hasan Miraz,Mehidy Hasan Miraz,NaN
3454,CR_jamesmccollum,NaN,NaN,NaN,source_only,Crickrida,NaN,James McCollum,James McCollum,NaN
3457,CR_markadair,NaN,NaN,NaN,source_only,Crickrida,NaN,Mark Adair,Mark Adair,NaN
3463,CR_tomlatham,NaN,NaN,NaN,source_only,Crickrida,NaN,Tom Latham,Tom Latham,NaN
3466,CR_andilephehlukwayo,NaN,NaN,NaN,source_only,Crickrida,NaN,Andile Phehlukwayo,Andile Phehlukwayo,NaN
3520,CR_liamlivingstone,NaN,NaN,NaN,source_only,Crickrida,NaN,Liam Livingstone,Liam Livingstone,NaN
3522,CR_markwood,NaN,NaN,NaN,source_only,Crickrida,NaN,Mark Wood,Mark Wood,NaN



SOURCE DISTRIBUTION — UPDATED REGISTRY
source
Cricsheet    3166
Crickrida     348
Both          121
Name: count, dtype: int64

SOURCE DISTRIBUTION — FINAL UPDATED REGISTRY
source
Cricsheet    3166
Crickrida     333
Both          121
Name: count, dtype: int64

INVESTIGATION COMPLETE — ORIGINAL DATA UNCHANGED


In [12]:

# ============================================================
# STEP 6: VERIFY THE 15 REMOVED CRICKRIDA RECORDS
# READ-ONLY
# ============================================================

# Match removed Crickrida names against verified identity mappings
removed_name_check = removed_rows[
    ["cricketiq_player_id", "player_name", "canonical_player_name"]
].copy()

removed_name_check = removed_name_check.merge(
    player_identity_map_updated[
        [
            "player_id",
            "cricsheet_name",
            "crickrida_name",
            "match_status"
        ]
    ],
    left_on="player_name",
    right_on="crickrida_name",
    how="left"
)

# Find corresponding records in the final registry
removed_name_check = removed_name_check.merge(
    player_registry_final_updated[
        [
            "cricketiq_player_id",
            "player_id",
            "source",
            "canonical_player_name",
            "player_names"
        ]
    ],
    on="player_id",
    how="left",
    suffixes=("_removed", "_final")
)

print("=" * 70)
print("REMOVED CRICKRIDA PLAYERS AND THEIR FINAL REGISTRY RECORDS")
print("=" * 70)

display(removed_name_check)

print("\nSummary:")
print("Removed source-only records:", len(removed_rows))
print("Matched to identity mappings:",
      removed_name_check["crickrida_name"].notna().sum())
print("Matched to final registry:",
      removed_name_check["source"].notna().sum())
print("Final records marked Both:",
      (removed_name_check["source"] == "Both").sum())

print("\nPotentially ambiguous name mappings:")
print(
    removed_name_check.groupby("player_name")["player_id"]
    .nunique()
    .loc[lambda x: x > 1]
)


REMOVED CRICKRIDA PLAYERS AND THEIR FINAL REGISTRY RECORDS


,cricketiq_player_id_removed,player_name,canonical_player_name_removed,player_id,cricsheet_name,crickrida_name,match_status,cricketiq_player_id_final,source,canonical_player_name_final,player_names
0,CR_edjoyce,Ed Joyce,Ed Joyce,a48fc79d,EC Joyce,Ed Joyce,verified_alias,CRIQ_a48fc79d,Both,Ed Joyce,None
1,CR_edjoyce,Ed Joyce,Ed Joyce,fd67a507,DI Joyce,Ed Joyce,verified_ireland_identity,CS_fd67a507,Cricsheet,DI Joyce,[DI Joyce]
2,CR_andybalbirnie,Andy Balbirnie,Andy Balbirnie,6ef13460,A Balbirnie,Andy Balbirnie,verified_alias,CRIQ_6ef13460,Both,Andy Balbirnie,None
3,CR_carlosbrathwaite,Carlos Brathwaite,Carlos Brathwaite,e342e5fb,CR Brathwaite,Carlos Brathwaite,verified_alias,CRIQ_e342e5fb,Both,Carlos Brathwaite,None
4,CR_mehidyhasanmiraz,Mehidy Hasan Miraz,Mehidy Hasan Miraz,c58b0108,Mehedi Hasan Miraz,Mehidy Hasan Miraz,verified_alias,CRIQ_c58b0108,Both,Mehidy Hasan Miraz,None
5,CR_jamesmccollum,James McCollum,James McCollum,c7105dab,JA McCollum,James McCollum,verified_alias,CRIQ_c7105dab,Both,James McCollum,None
6,CR_markadair,Mark Adair,Mark Adair,fc55ec67,MR Adair,Mark Adair,verified_alias,CRIQ_fc55ec67,Both,Mark Adair,None
7,CR_tomlatham,Tom Latham,Tom Latham,e824e6ee,TWM Latham,Tom Latham,verified_alias,CRIQ_e824e6ee,Both,Tom Latham,None
8,CR_andilephehlukwayo,Andile Phehlukwayo,Andile Phehlukwayo,ab6b1c45,AL Phehlukwayo,Andile Phehlukwayo,verified_alias,CRIQ_ab6b1c45,Both,Andile Phehlukwayo,None
9,CR_liamlivingstone,Liam Livingstone,Liam Livingstone,50c6bc2b,LS Livingstone,Liam Livingstone,verified_alias,CRIQ_50c6bc2b,Both,Liam Livingstone,None



Summary:
Removed source-only records: 15
Matched to identity mappings: 16
Matched to final registry: 16
Final records marked Both: 15

Potentially ambiguous name mappings:
player_name
Ed Joyce    2
Name: player_id, dtype: int64


In [13]:
# ============================================================
# STEP 7: INVESTIGATE ED JOYCE IDENTITY COLLISION
# READ-ONLY
# ============================================================

print("=" * 65)
print("ED JOYCE — IDENTITY MAP")
print("=" * 65)

display(
    player_identity_map_updated[
        player_identity_map_updated["crickrida_name"]
        .str.contains("Ed Joyce", case=False, na=False)
    ].to_string(index=False)
)

print("\n" + "=" * 65)
print("ED JOYCE — FINAL REGISTRY")
print("=" * 65)

display(
    player_registry_final_updated[
        player_registry_final_updated["canonical_player_name"]
        .str.contains("Joyce", case=False, na=False)
    ].to_string(index=False)
)

print("\n" + "=" * 65)
print("ED JOYCE — ORIGINAL REGISTRY")
print("=" * 65)

display(
    player_registry_updated[
        player_registry_updated["player_name"]
        .str.contains("Ed Joyce", case=False, na=False)
    ].to_string(index=False)
)

ED JOYCE — IDENTITY MAP


'cricketiq_player_id player_id cricsheet_name crickrida_name normalized_name              match_status crickrida_player_name cricsheet_player_name cricsheet_player_id                        mapping_method confidence                                                                           notes\n      CRIQ_a48fc79d  a48fc79d       EC Joyce       Ed Joyce         edjoyce            verified_alias                   NaN                   NaN                 NaN                                   NaN        NaN                                                                             NaN\n      CRIQ_fd67a507  fd67a507       DI Joyce       Ed Joyce         edjoyce verified_ireland_identity              Ed Joyce              DI Joyce            fd67a507 verified_full_name_and_ireland_career       high Identity approved using player name, initials and Ireland international career.'


ED JOYCE — FINAL REGISTRY


'cricketiq_player_id player_id cricsheet_name crickrida_name normalized_name   match_status    source player_names player_name canonical_player_name\n      CRIQ_a48fc79d  a48fc79d       EC Joyce       Ed Joyce         edjoyce verified_alias      Both         None        None              Ed Joyce\n        CS_56418eae  56418eae           None           None             NaN    source_only Cricsheet [CNIM Joyce]        None            CNIM Joyce\n        CS_d3b4b8ed  d3b4b8ed           None           None             NaN    source_only Cricsheet [IMHC Joyce]        None            IMHC Joyce\n        CS_fd67a507  fd67a507           None           None             NaN    source_only Cricsheet   [DI Joyce]        None              DI Joyce'


ED JOYCE — ORIGINAL REGISTRY


'cricketiq_player_id player_id cricsheet_name crickrida_name match_status    source player_names player_name canonical_player_name normalized_name\n         CR_edjoyce       NaN            NaN            NaN  source_only Crickrida          NaN    Ed Joyce              Ed Joyce             NaN'

In [14]:

# ============================================================
# STEP 8: ED JOYCE SOURCE RECORD INVESTIGATION
# READ-ONLY
# ============================================================

print("=" * 70)
print("CRICSHEET PLAYER RECORDS")
print("=" * 70)

display(
    cricsheet_players_all[
        cricsheet_players_all.astype(str)
        .apply(
            lambda col: col.str.contains(
                "Joyce", case=False, na=False
            )
        ).any(axis=1)
    ]
)

print("\n" + "=" * 70)
print("CRICKRIDA PLAYER RECORDS")
print("=" * 70)

display(
    crickrida_players_all[
        crickrida_players_all.astype(str)
        .apply(
            lambda col: col.str.contains(
                "Joyce", case=False, na=False
            )
        ).any(axis=1)
    ]
)

print("\n" + "=" * 70)
print("CRICSHEET PLAYER HISTORY")
print("=" * 70)

display(
    cs_player_history[
        cs_player_history.astype(str)
        .apply(
            lambda col: col.str.contains(
                "Joyce", case=False, na=False
            )
        ).any(axis=1)
    ]
)

print("\n" + "=" * 70)
print("CRICKRIDA PLAYER HISTORY")
print("=" * 70)

display(
    cr_player_history[
        cr_player_history.astype(str)
        .apply(
            lambda col: col.str.contains(
                "Joyce", case=False, na=False
            )
        ).any(axis=1)
    ]
)


CRICSHEET PLAYER RECORDS


,team,player_name,source_player_id,normalized_name
758,Ireland,EC Joyce,a48fc79d,ecjoyce
20290,England,EC Joyce,a48fc79d,ecjoyce
20301,Ireland,DI Joyce,fd67a507,dijoyce



CRICKRIDA PLAYER RECORDS


,team,player_name,source_player_id,normalized_name
39,Ireland,Ed Joyce,CR_edjoyce,edjoyce



CRICSHEET PLAYER HISTORY


,team,normalized_name,player_name,source_player_id,cricsheet_first_date,cricsheet_last_date,cricsheet_matches
390,England,ecjoyce,EC Joyce,a48fc79d,2006-06-13,2007-04-04,17
674,Ireland,dijoyce,DI Joyce,fd67a507,2006-06-13,2007-06-24,3
677,Ireland,ecjoyce,EC Joyce,a48fc79d,2011-02-25,2018-03-16,43



CRICKRIDA PLAYER HISTORY


,team,normalized_name,player_name,crickrida_first_date,crickrida_last_date,crickrida_matches
214,Ireland,edjoyce,Ed Joyce,2012-07-05,2017-03-24,12


In [15]:
# ============================================================
# STEP 9: CHECK DI JOYCE PERFORMANCE DATA REFERENCES
# READ-ONLY
# ============================================================

target_id = "fd67a507"

datasets = {
    "Batting": player_batting_updated,
    "Bowling": player_bowling_updated,
    "Dismissals": player_dismissal_updated,
    "Bowler Dismissals": dismissal_bowler_updated,
    "Afghanistan Batting": afg_player_batting_updated,
    "Afghanistan Bowling": afg_player_bowling_updated
}

for name, df in datasets.items():

    print("\n" + "=" * 60)
    print(name)

    if "cricketiq_player_id" not in df.columns:
        print("No CricketIQ ID column")
        continue

    matches = df[
        df["cricketiq_player_id"].astype(str).str.endswith(target_id)
    ]

    print("Rows:", len(matches))

    if len(matches):
        display(matches.head(10))


Batting
Rows: 3


,match_id,innings_number,batting_team,batter,batter_id,runs,balls_faced,fours,sixes,strike_rate,cricketiq_player_id,canonical_player_name
20301,225171,2,Ireland,DI Joyce,fd67a507,0,3,0,0,0.000000,CS_fd67a507,DI Joyce
24542,293071,1,Ireland,DI Joyce,fd67a507,18,34,2,0,52.941176,CS_fd67a507,DI Joyce
24563,293072,2,Ireland,DI Joyce,fd67a507,11,19,2,0,57.894737,CS_fd67a507,DI Joyce



Bowling
Rows: 0

Dismissals
Rows: 3


,match_id,innings_number,batting_team,player_out,player_out_id,dismissals,cricketiq_player_id,canonical_player_name
16915,225171,2,Ireland,DI Joyce,fd67a507,1,CS_fd67a507,DI Joyce
20422,293071,1,Ireland,DI Joyce,fd67a507,1,CS_fd67a507,DI Joyce
20437,293072,2,Ireland,DI Joyce,fd67a507,1,CS_fd67a507,DI Joyce



Bowler Dismissals
Rows: 0

Afghanistan Batting
Rows: 0

Afghanistan Bowling
Rows: 0


In [16]:
# ============================================================
# STEP 10: PREPARE ED JOYCE IDENTITY CORRECTION
# READ-ONLY
# ============================================================

print("CURRENT ED JOYCE IDENTITY RECORDS")

display(
    player_identity_map_updated[
        player_identity_map_updated["player_id"].isin(
            ["a48fc79d", "fd67a507"]
        )
    ].to_string(index=False)
)

print("\nCURRENT ED JOYCE REGISTRY RECORDS")

display(
    player_registry_final_updated[
        player_registry_final_updated["player_id"].isin(
            ["a48fc79d", "fd67a507"]
        )
    ].to_string(index=False)
)

print("\nORIGINAL IDENTITY MAP")

display(
    player_identity_map[
        player_identity_map.astype(str)
        .apply(
            lambda col: col.str.contains(
                "Ed Joyce", case=False, na=False
            )
        ).any(axis=1)
    ].to_string(index=False)
)

CURRENT ED JOYCE IDENTITY RECORDS


'cricketiq_player_id player_id cricsheet_name crickrida_name normalized_name              match_status crickrida_player_name cricsheet_player_name cricsheet_player_id                        mapping_method confidence                                                                           notes\n      CRIQ_a48fc79d  a48fc79d       EC Joyce       Ed Joyce         edjoyce            verified_alias                   NaN                   NaN                 NaN                                   NaN        NaN                                                                             NaN\n      CRIQ_fd67a507  fd67a507       DI Joyce       Ed Joyce         edjoyce verified_ireland_identity              Ed Joyce              DI Joyce            fd67a507 verified_full_name_and_ireland_career       high Identity approved using player name, initials and Ireland international career.'


CURRENT ED JOYCE REGISTRY RECORDS


'cricketiq_player_id player_id cricsheet_name crickrida_name normalized_name   match_status    source player_names player_name canonical_player_name\n      CRIQ_a48fc79d  a48fc79d       EC Joyce       Ed Joyce         edjoyce verified_alias      Both         None        None              Ed Joyce\n        CS_fd67a507  fd67a507           None           None             NaN    source_only Cricsheet   [DI Joyce]        None              DI Joyce'


ORIGINAL IDENTITY MAP


'Empty DataFrame\nColumns: [cricketiq_player_id, player_id, cricsheet_name, crickrida_name, normalized_name, match_status]\nIndex: []'

In [17]:

# ============================================================
# STEP 12: CORRECT APPROVED IRELAND MAPPINGS
# ============================================================

approved_ireland_mappings = [
    {
        "cricsheet_name": "PJK Mooney",
        "crickrida_name": "John Mooney",
        "player_id": "94edecfe",
        "crickrida_id": "CR_johnmooney"
    },
    {
        "cricsheet_name": "SP Terry",
        "crickrida_name": "Sean Terry",
        "player_id": "7f87204d",
        "crickrida_id": "CR_seanterry"
    }
]

# DI Joyce is deliberately excluded.
# His Cricsheet identity must remain independent.


In [18]:

# ============================================================
# STEP 13: REMOVE INCORRECT DI JOYCE ASSOCIATION
# ============================================================

# Preserve the current version before correction
player_identity_map_updated_before_joyce_fix = (
    player_identity_map_updated.copy(deep=True)
)

# Remove only the incorrect DI Joyce -> Ed Joyce association
incorrect_mapping = (
    (player_identity_map_updated["player_id"].astype(str) == "fd67a507") &
    (player_identity_map_updated["crickrida_name"] == "Ed Joyce")
)

print("Incorrect association rows found:", incorrect_mapping.sum())

player_identity_map_updated = (
    player_identity_map_updated.loc[~incorrect_mapping].copy()
)

print("Corrected identity map:", player_identity_map_updated.shape)

# Verify EC Joyce remains mapped
display(
    player_identity_map_updated[
        player_identity_map_updated["player_id"].astype(str)
        .isin(["a48fc79d", "fd67a507"])
    ]
)


Incorrect association rows found: 1
Corrected identity map: (212, 12)


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,normalized_name,match_status,crickrida_player_name,cricsheet_player_name,cricsheet_player_id,mapping_method,confidence,notes
118,CRIQ_a48fc79d,a48fc79d,EC Joyce,Ed Joyce,edjoyce,verified_alias,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:

# ============================================================
# STEP 14: VALIDATE ED JOYCE IDENTITY CORRECTION
# ============================================================

assert not (
    (player_identity_map_updated["player_id"].astype(str) == "fd67a507") &
    (player_identity_map_updated["crickrida_name"] == "Ed Joyce")
).any(), "Incorrect DI Joyce mapping still exists!"

assert (
    (player_identity_map_updated["player_id"].astype(str) == "a48fc79d") &
    (player_identity_map_updated["crickrida_name"] == "Ed Joyce")
).any(), "EC Joyce mapping was lost!"

assert (
    player_registry_final_updated["cricketiq_player_id"] == "CS_fd67a507"
).any(), "DI Joyce registry identity was lost!"

print("ED JOYCE IDENTITY VALIDATION PASSED")
print("EC Joyce mapping: preserved")
print("DI Joyce mapping: removed from shared identity map")
print("DI Joyce registry identity: preserved")


ED JOYCE IDENTITY VALIDATION PASSED
EC Joyce mapping: preserved
DI Joyce mapping: removed from shared identity map
DI Joyce registry identity: preserved


In [20]:

# ============================================================
# STEP 15: VERIFY SOURCE-ONLY INPUTS BEFORE REGISTRY REBUILD
# READ-ONLY
# ============================================================

print("=" * 65)
print("SOURCE-ONLY INPUT DATASETS")
print("=" * 65)

print("Cricsheet-only master:", cricsheet_only_master.shape)
print("Crickrida-only master:", crickrida_only_master.shape)
print("Corrected identity map:", player_identity_map_updated.shape)

print("\nCricsheet-only columns:")
print(cricsheet_only_master.columns.tolist())

print("\nCrickrida-only columns:")
print(crickrida_only_master.columns.tolist())

# Check DI Joyce in original Cricsheet-only records
print("\nDI JOYCE IN CRICSHEET-ONLY MASTER")

display(
    cricsheet_only_master[
        cricsheet_only_master["player_id"].astype(str) == "fd67a507"
    ]
)

# Check Ed Joyce in original Crickrida-only records
print("\nED JOYCE IN CRICKRIDA-ONLY MASTER")

display(
    crickrida_only_master[
        crickrida_only_master["player_name"].astype(str).str.lower()
        == "ed joyce"
    ]
)

# Check all approved identity mappings
print("\nIDENTITY MAP COUNTS")

print("Total mappings:", len(player_identity_map_updated))
print(
    "Unique player IDs:",
    player_identity_map_updated["player_id"].nunique()
)
print(
    "Duplicate player IDs:",
    player_identity_map_updated["player_id"].duplicated().sum()
)
print(
    "Unique Crickrida names:",
    player_identity_map_updated["crickrida_name"].nunique()
)


SOURCE-ONLY INPUT DATASETS
Cricsheet-only master: (3182, 6)
Crickrida-only master: (348, 5)
Corrected identity map: (212, 12)

Cricsheet-only columns:
['player_id', 'player_names', 'normalized_names', 'cricketiq_player_id', 'source', 'match_status']

Crickrida-only columns:
['player_name', 'normalized_name', 'cricketiq_player_id', 'source', 'match_status']

DI JOYCE IN CRICSHEET-ONLY MASTER


,player_id,player_names,normalized_names,cricketiq_player_id,source,match_status
3141,fd67a507,[DI Joyce],[dijoyce],CS_fd67a507,Cricsheet,source_only



ED JOYCE IN CRICKRIDA-ONLY MASTER


,player_name,normalized_name,cricketiq_player_id,source,match_status
39,Ed Joyce,edjoyce,CR_edjoyce,Crickrida,source_only



IDENTITY MAP COUNTS
Total mappings: 212
Unique player IDs: 212
Duplicate player IDs: 0
Unique Crickrida names: 211


In [21]:

# ============================================================
# STEP 16: FINAL PRE-REBUILD IDENTITY AUDIT
# READ-ONLY
# ============================================================

# 1. Identify duplicate Crickrida names
duplicate_crickrida_names = (
    player_identity_map_updated[
        player_identity_map_updated["crickrida_name"].notna()
    ]
    .groupby("crickrida_name")
    .filter(lambda x: len(x) > 1)
    .sort_values("crickrida_name")
)

print("=" * 65)
print("DUPLICATE CRICKRIDA NAMES")
print("=" * 65)

print("Rows involved:", len(duplicate_crickrida_names))

display(
    duplicate_crickrida_names[
        [
            "cricketiq_player_id",
            "player_id",
            "cricsheet_name",
            "crickrida_name",
            "match_status",
            "mapping_method",
            "confidence",
            "notes"
        ]
    ]
)

# 2. Check how many identity IDs occur in Cricsheet-only master
identity_player_ids = set(
    player_identity_map_updated["player_id"].dropna().astype(str)
)

cricsheet_master_ids = set(
    cricsheet_only_master["player_id"].dropna().astype(str)
)

print("\n" + "=" * 65)
print("CRICSHEET SOURCE COVERAGE")
print("=" * 65)

print("Identity player IDs:", len(identity_player_ids))
print("Present in Cricsheet-only master:",
      len(identity_player_ids & cricsheet_master_ids))
print("Not present in Cricsheet-only master:",
      len(identity_player_ids - cricsheet_master_ids))

# 3. Check Crickrida names in source-only master
identity_crickrida_names = set(
    player_identity_map_updated["crickrida_name"]
    .dropna().astype(str).str.strip()
)

crickrida_master_names = set(
    crickrida_only_master["player_name"]
    .dropna().astype(str).str.strip()
)

print("\n" + "=" * 65)
print("CRICKRIDA SOURCE COVERAGE")
print("=" * 65)

print("Identity Crickrida names:", len(identity_crickrida_names))
print("Present in Crickrida-only master:",
      len(identity_crickrida_names & crickrida_master_names))
print("Not present in Crickrida-only master:",
      len(identity_crickrida_names - crickrida_master_names))

# 4. Check duplicate source identifiers
print("\n" + "=" * 65)
print("SOURCE MASTER DUPLICATE CHECKS")
print("=" * 65)

print(
    "Duplicate Cricsheet player IDs:",
    cricsheet_only_master["player_id"].duplicated().sum()
)

print(
    "Duplicate Crickrida player names:",
    crickrida_only_master["player_name"].duplicated().sum()
)

print("\nAUDIT COMPLETE — NO ORIGINAL DATA MODIFIED")


DUPLICATE CRICKRIDA NAMES
Rows involved: 2


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,match_status,mapping_method,confidence,notes
135,CRIQ_aae9ce90,aae9ce90,JF Mooney,John Mooney,verified_alias,NaN,NaN,NaN
162,CRIQ_94edecfe,94edecfe,PJK Mooney,John Mooney,verified_ireland_identity,verified_full_name_and_ireland_career,high,"Identity approved using player name, initials ..."



CRICSHEET SOURCE COVERAGE
Identity player IDs: 212
Present in Cricsheet-only master: 107
Not present in Cricsheet-only master: 105

CRICKRIDA SOURCE COVERAGE
Identity Crickrida names: 211
Present in Crickrida-only master: 82
Not present in Crickrida-only master: 129

SOURCE MASTER DUPLICATE CHECKS
Duplicate Cricsheet player IDs: 0
Duplicate Crickrida player names: 0

AUDIT COMPLETE — NO ORIGINAL DATA MODIFIED


In [22]:
duplicate_crickrida_names = (
    player_identity_map_updated[
        player_identity_map_updated["crickrida_name"].notna()
    ]
    .groupby("crickrida_name")
    .filter(lambda x: len(x) > 1)
    .sort_values("crickrida_name")
)

display(
    duplicate_crickrida_names[
        [
            "cricketiq_player_id",
            "player_id",
            "cricsheet_name",
            "crickrida_name",
            "match_status",
            "mapping_method",
            "confidence",
            "notes"
        ]
    ]
)

,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,match_status,mapping_method,confidence,notes
135,CRIQ_aae9ce90,aae9ce90,JF Mooney,John Mooney,verified_alias,NaN,NaN,NaN
162,CRIQ_94edecfe,94edecfe,PJK Mooney,John Mooney,verified_ireland_identity,verified_full_name_and_ireland_career,high,"Identity approved using player name, initials ..."


In [24]:
# ============================================================
# STEP 17A: INSPECT HISTORY DATAFRAME SCHEMAS
# READ-ONLY
# ============================================================

print("=" * 70)
print("CRICSHEET PLAYER HISTORY")
print("=" * 70)

print("Shape:", cs_player_history.shape)
print("Columns:", cs_player_history.columns.tolist())

display(cs_player_history.head(5))


print("\n" + "=" * 70)
print("CRICKRIDA PLAYER HISTORY")
print("=" * 70)

print("Shape:", cr_player_history.shape)
print("Columns:", cr_player_history.columns.tolist())

display(cr_player_history.head(5))


print("\n" + "=" * 70)
print("RELATED PLAYER DATAFRAME SCHEMAS")
print("=" * 70)

for name in [
    "cricsheet_players_all",
    "crickrida_players_all",
    "player_identity_map_updated"
]:
    df = globals()[name]
    print(f"\n{name}: {df.shape}")
    print(df.columns.tolist())

CRICSHEET PLAYER HISTORY
Shape: (2025, 7)
Columns: ['team', 'normalized_name', 'player_name', 'source_player_id', 'cricsheet_first_date', 'cricsheet_last_date', 'cricsheet_matches']


,team,normalized_name,player_name,source_player_id,cricsheet_first_date,cricsheet_last_date,cricsheet_matches
0,Australia,aanoffke,AA Noffke,b69e69ed,2008-02-03,2008-02-03,1
1,Australia,acagar,AC Agar,a2421394,2015-09-08,2023-09-07,22
2,Australia,acgilchrist,AC Gilchrist,2b6e6dec,2003-01-09,2008-03-04,116
3,Australia,acvoges,AC Voges,864c199e,2007-02-20,2013-11-02,29
4,Australia,ajbichel,AJ Bichel,3cf83487,2003-01-09,2004-02-01,27



CRICKRIDA PLAYER HISTORY
Shape: (454, 6)
Columns: ['team', 'normalized_name', 'player_name', 'crickrida_first_date', 'crickrida_last_date', 'crickrida_matches']


,team,normalized_name,player_name,crickrida_first_date,crickrida_last_date,crickrida_matches
0,Afghanistan,abdulmalik,Abdul Malik,2024-09-22,2024-12-21,3
1,Afghanistan,abdulrahmanafghanistan2023,"Abdul Rahman (Afghanistan, 2023)",2023-07-11,2023-08-24,3
2,Afghanistan,afsarzazai,Afsar Zazai,2015-01-10,2017-06-11,9
3,Afghanistan,aftabalam,Aftab Alam,2014-07-22,2019-06-22,22
4,Afghanistan,amghazanfar,AM Ghazanfar,2024-03-07,2026-10-23,24



RELATED PLAYER DATAFRAME SCHEMAS

cricsheet_players_all: (1354, 4)
['team', 'player_name', 'source_player_id', 'normalized_name']

crickrida_players_all: (396, 4)
['team', 'player_name', 'source_player_id', 'normalized_name']

player_identity_map_updated: (212, 12)
['cricketiq_player_id', 'player_id', 'cricsheet_name', 'crickrida_name', 'normalized_name', 'match_status', 'crickrida_player_name', 'cricsheet_player_name', 'cricsheet_player_id', 'mapping_method', 'confidence', 'notes']


In [25]:

# ============================================================
# STEP 17B: JOHN MOONEY IDENTITY COLLISION INVESTIGATION
# READ-ONLY
# ============================================================

mooney_ids = ["aae9ce90", "94edecfe"]

# 1. Cricsheet history for both identities
print("=" * 70)
print("CRICSHEET HISTORY: BOTH MOONEY IDENTITIES")
print("=" * 70)

mooney_cricsheet_history = cs_player_history[
    cs_player_history["source_player_id"]
    .astype(str).isin(mooney_ids)
].copy()

display(
    mooney_cricsheet_history.sort_values(
        ["source_player_id", "cricsheet_first_date"]
    )
)

# 2. Crickrida history for John Mooney
print("\n" + "=" * 70)
print("CRICKRIDA HISTORY: JOHN MOONEY")
print("=" * 70)

mooney_crickrida_history = cr_player_history[
    cr_player_history["normalized_name"]
    .astype(str).str.strip().str.lower().eq("johnmooney")
].copy()

display(mooney_crickrida_history)

# 3. Exact identity map entries
print("\n" + "=" * 70)
print("CURRENT IDENTITY MAP ENTRIES")
print("=" * 70)

display(
    player_identity_map_updated[
        player_identity_map_updated["player_id"]
        .astype(str).isin(mooney_ids)
    ][
        [
            "cricketiq_player_id",
            "player_id",
            "cricsheet_name",
            "crickrida_name",
            "match_status",
            "mapping_method",
            "confidence",
            "notes"
        ]
    ]
)

# 4. Compare career date ranges
print("\n" + "=" * 70)
print("CAREER DATE COMPARISON")
print("=" * 70)

for _, cs_row in mooney_cricsheet_history.iterrows():
    for _, cr_row in mooney_crickrida_history.iterrows():

        cs_start = pd.to_datetime(cs_row["cricsheet_first_date"])
        cs_end = pd.to_datetime(cs_row["cricsheet_last_date"])

        cr_start = pd.to_datetime(cr_row["crickrida_first_date"])
        cr_end = pd.to_datetime(cr_row["crickrida_last_date"])

        overlaps = max(cs_start, cr_start) <= min(cs_end, cr_end)

        print(
            f"\n{cs_row['player_name']} ({cs_row['source_player_id']})"
        )
        print(
            f"Cricsheet: {cs_start.date()} to {cs_end.date()}"
        )
        print(
            f"Crickrida: {cr_start.date()} to {cr_end.date()}"
        )
        print("Date ranges overlap:", overlaps)

print("\nINVESTIGATION COMPLETE — NO DATA MODIFIED")


CRICSHEET HISTORY: BOTH MOONEY IDENTITIES


,team,normalized_name,player_name,source_player_id,cricsheet_first_date,cricsheet_last_date,cricsheet_matches
711,Ireland,pjkmooney,PJK Mooney,94edecfe,2006-06-13,2007-04-03,2
690,Ireland,jfmooney,JF Mooney,aae9ce90,2006-06-13,2015-10-11,35



CRICKRIDA HISTORY: JOHN MOONEY


,team,normalized_name,player_name,crickrida_first_date,crickrida_last_date,crickrida_matches
227,Ireland,johnmooney,John Mooney,2010-07-03,2015-01-17,4



CURRENT IDENTITY MAP ENTRIES


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,match_status,mapping_method,confidence,notes
135,CRIQ_aae9ce90,aae9ce90,JF Mooney,John Mooney,verified_alias,NaN,NaN,NaN
162,CRIQ_94edecfe,94edecfe,PJK Mooney,John Mooney,verified_ireland_identity,verified_full_name_and_ireland_career,high,"Identity approved using player name, initials ..."



CAREER DATE COMPARISON

JF Mooney (aae9ce90)
Cricsheet: 2006-06-13 to 2015-10-11
Crickrida: 2010-07-03 to 2015-01-17
Date ranges overlap: True

PJK Mooney (94edecfe)
Cricsheet: 2006-06-13 to 2007-04-03
Crickrida: 2010-07-03 to 2015-01-17
Date ranges overlap: False

INVESTIGATION COMPLETE — NO DATA MODIFIED


In [26]:
# ============================================================
# STEP 18: CORRECT JOHN MOONEY IDENTITY COLLISION
# ============================================================

# Preserve a backup before modification
player_identity_map_before_mooney_fix = (
    player_identity_map_updated.copy(deep=True)
)

# Identify the incorrect PJK Mooney association
incorrect_mooney_mapping = (
    (player_identity_map_updated["player_id"].astype(str) == "94edecfe") &
    (player_identity_map_updated["crickrida_name"] == "John Mooney")
)

print(
    "Incorrect mapping rows found:",
    incorrect_mooney_mapping.sum()
)

# Remove only the incorrect shared association
player_identity_map_updated = (
    player_identity_map_updated.loc[
        ~incorrect_mooney_mapping
    ].copy()
)

# Validate the corrected identities
print("\nCorrected identity map:", player_identity_map_updated.shape)

print("\nJF Mooney mapping:")
display(
    player_identity_map_updated[
        player_identity_map_updated["player_id"].astype(str) == "aae9ce90"
    ]
)

print("\nPJK Mooney mapping:")
display(
    player_identity_map_updated[
        player_identity_map_updated["player_id"].astype(str) == "94edecfe"
    ]
)

print("\nUnique player IDs:",
      player_identity_map_updated["player_id"].nunique())

print("Duplicate player IDs:",
      player_identity_map_updated["player_id"].duplicated().sum())

print(
    "\nJohn Mooney mappings:",
    player_identity_map_updated[
        player_identity_map_updated["crickrida_name"] == "John Mooney"
    ].shape[0]
)

Incorrect mapping rows found: 1

Corrected identity map: (211, 12)

JF Mooney mapping:


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,normalized_name,match_status,crickrida_player_name,cricsheet_player_name,cricsheet_player_id,mapping_method,confidence,notes
135,CRIQ_aae9ce90,aae9ce90,JF Mooney,John Mooney,johnmooney,verified_alias,NaN,NaN,NaN,NaN,NaN,NaN



PJK Mooney mapping:


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,normalized_name,match_status,crickrida_player_name,cricsheet_player_name,cricsheet_player_id,mapping_method,confidence,notes



Unique player IDs: 211
Duplicate player IDs: 0

John Mooney mappings: 1


In [27]:

# ============================================================
# STEP 19: FINAL IDENTITY MAP AUDIT
# READ-ONLY
# ============================================================

print("=" * 70)
print("IDENTITY MAP VALIDATION")
print("=" * 70)

print("Shape:", player_identity_map_updated.shape)

print(
    "Unique player IDs:",
    player_identity_map_updated["player_id"].nunique()
)

print(
    "Duplicate player IDs:",
    player_identity_map_updated["player_id"].duplicated().sum()
)

print(
    "Missing player IDs:",
    player_identity_map_updated["player_id"].isna().sum()
)

# Duplicate Crickrida names
duplicate_names = (
    player_identity_map_updated[
        player_identity_map_updated["crickrida_name"].notna()
    ]
    .groupby("crickrida_name")
    .filter(lambda x: len(x) > 1)
    .sort_values("crickrida_name")
)

print("\nDuplicate Crickrida names:", len(duplicate_names))

if len(duplicate_names) > 0:
    display(
        duplicate_names[
            [
                "player_id",
                "cricsheet_name",
                "crickrida_name",
                "match_status",
                "mapping_method",
                "confidence"
            ]
        ]
    )

# Confirm known corrections
print("\n" + "=" * 70)
print("KNOWN IDENTITY CORRECTIONS")
print("=" * 70)

for player_id in ["a48fc79d", "fd67a507", "aae9ce90", "94edecfe"]:

    result = player_identity_map_updated[
        player_identity_map_updated["player_id"].astype(str) == player_id
    ]

    print(f"\nPlayer ID: {player_id}")
    print("Mapping rows:", len(result))

    if not result.empty:
        display(result[["cricsheet_name", "crickrida_name", "match_status"]])

print("\nFINAL AUDIT COMPLETE")


IDENTITY MAP VALIDATION
Shape: (211, 12)
Unique player IDs: 211
Duplicate player IDs: 0
Missing player IDs: 0

Duplicate Crickrida names: 0

KNOWN IDENTITY CORRECTIONS

Player ID: a48fc79d
Mapping rows: 1


,cricsheet_name,crickrida_name,match_status
118,EC Joyce,Ed Joyce,verified_alias



Player ID: fd67a507
Mapping rows: 0

Player ID: aae9ce90
Mapping rows: 1


,cricsheet_name,crickrida_name,match_status
135,JF Mooney,John Mooney,verified_alias



Player ID: 94edecfe
Mapping rows: 0

FINAL AUDIT COMPLETE


In [28]:

# ============================================================
# STEP 20: BUILD CORRECTED PLAYER REGISTRY CANDIDATE
# Does NOT overwrite existing registry DataFrames
# Does NOT save any files
# ============================================================

# 1. Define the expected registry schema
registry_columns = [
    "cricketiq_player_id",
    "player_id",
    "cricsheet_name",
    "crickrida_name",
    "match_status",
    "source",
    "player_names",
    "player_name",
    "canonical_player_name",
    "normalized_name"
]

# 2. Prepare matching identifiers
identity_player_ids = set(
    player_identity_map_updated["player_id"]
    .dropna().astype(str)
)

identity_crickrida_names = set(
    player_identity_map_updated["crickrida_name"]
    .dropna().astype(str).str.strip()
)

# 3. Remove mapped identities from source-only masters
cricsheet_only_candidate = cricsheet_only_master[
    ~cricsheet_only_master["player_id"].astype(str).isin(
        identity_player_ids
    )
].copy()

crickrida_only_candidate = crickrida_only_master[
    ~crickrida_only_master["player_name"].astype(str).str.strip().isin(
        identity_crickrida_names
    )
].copy()

# 4. Prepare the 211 shared identities
both_candidate = pd.DataFrame({
    "cricketiq_player_id":
        player_identity_map_updated["cricketiq_player_id"].values,

    "player_id":
        player_identity_map_updated["player_id"].values,

    "cricsheet_name":
        player_identity_map_updated["cricsheet_name"].values,

    "crickrida_name":
        player_identity_map_updated["crickrida_name"].values,

    "match_status":
        player_identity_map_updated["match_status"].values,

    "source": "Both",

    "player_names": None,

    "player_name": None,

    "canonical_player_name":
        player_identity_map_updated["crickrida_name"].values,

    "normalized_name":
        player_identity_map_updated["normalized_name"].values
})

# 5. Prepare Cricsheet-only records
cs_candidate = pd.DataFrame({
    "cricketiq_player_id":
        cricsheet_only_candidate["cricketiq_player_id"].values,

    "player_id":
        cricsheet_only_candidate["player_id"].values,

    "cricsheet_name": None,
    "crickrida_name": None,
    "match_status": "source_only",
    "source": "Cricsheet",

    "player_names":
        cricsheet_only_candidate["player_names"].values,

    "player_name": None,

    "canonical_player_name":
        cricsheet_only_candidate["player_names"].apply(
            lambda x: x[0]
            if isinstance(x, (list, tuple)) and len(x) > 0
            else None
        ).values,

    "normalized_name":
        cricsheet_only_candidate["normalized_names"].apply(
            lambda x: x[0]
            if isinstance(x, (list, tuple)) and len(x) > 0
            else None
        ).values
})

# 6. Prepare Crickrida-only records
cr_candidate = pd.DataFrame({
    "cricketiq_player_id":
        crickrida_only_candidate["cricketiq_player_id"].values,

    "player_id": None,
    "cricsheet_name": None,

    "crickrida_name":
        crickrida_only_candidate["player_name"].values,

    "match_status": "source_only",
    "source": "Crickrida",
    "player_names": None,

    "player_name":
        crickrida_only_candidate["player_name"].values,

    "canonical_player_name":
        crickrida_only_candidate["player_name"].values,

    "normalized_name":
        crickrida_only_candidate["normalized_name"].values
})

# 7. Combine all three groups
player_registry_candidate = pd.concat(
    [both_candidate, cs_candidate, cr_candidate],
    ignore_index=True
)

# Enforce exact column order
player_registry_candidate = (
    player_registry_candidate[registry_columns].copy()
)

# ============================================================
# STEP 20A: REGISTRY BUILD SUMMARY
# ============================================================

print("=" * 70)
print("CORRECTED REGISTRY CANDIDATE")
print("=" * 70)

print("Shared identities:", len(both_candidate))
print("Cricsheet-only identities:", len(cs_candidate))
print("Crickrida-only identities:", len(cr_candidate))

print("\nFinal candidate shape:", player_registry_candidate.shape)

print("\nSource distribution:")
print(player_registry_candidate["source"].value_counts())

# ============================================================
# STEP 20B: VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("REGISTRY VALIDATION")
print("=" * 70)

print(
    "Expected row count:",
    len(both_candidate) + len(cs_candidate) + len(cr_candidate)
)

print(
    "Actual row count:",
    len(player_registry_candidate)
)

print(
    "Duplicate CricketIQ IDs:",
    player_registry_candidate["cricketiq_player_id"].duplicated().sum()
)

print(
    "Missing CricketIQ IDs:",
    player_registry_candidate["cricketiq_player_id"].isna().sum()
)

print(
    "Registry columns correct:",
    player_registry_candidate.columns.tolist() == registry_columns
)

print(
    "Shared identity count correct:",
    (player_registry_candidate["source"] == "Both").sum()
    == len(player_identity_map_updated)
)

# ============================================================
# STEP 20C: CHECK KNOWN PLAYER IDENTITIES
# ============================================================

print("\n" + "=" * 70)
print("KNOWN PLAYER VALIDATION")
print("=" * 70)

known_ids = ["a48fc79d", "fd67a507", "aae9ce90", "94edecfe"]

display(
    player_registry_candidate[
        player_registry_candidate["player_id"]
        .astype(str).isin(known_ids)
    ][registry_columns]
)

print("\nCandidate built. No original registry modified.")


CORRECTED REGISTRY CANDIDATE
Shared identities: 211
Cricsheet-only identities: 3076
Crickrida-only identities: 266

Final candidate shape: (3553, 10)

Source distribution:
source
Cricsheet    3076
Crickrida     266
Both          211
Name: count, dtype: int64

REGISTRY VALIDATION
Expected row count: 3553
Actual row count: 3553
Duplicate CricketIQ IDs: 0
Missing CricketIQ IDs: 0
Registry columns correct: True
Shared identity count correct: True

KNOWN PLAYER VALIDATION


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,match_status,source,player_names,player_name,canonical_player_name,normalized_name
118,CRIQ_a48fc79d,a48fc79d,EC Joyce,Ed Joyce,verified_alias,Both,None,None,Ed Joyce,edjoyce
135,CRIQ_aae9ce90,aae9ce90,JF Mooney,John Mooney,verified_alias,Both,None,None,John Mooney,johnmooney
1994,CS_94edecfe,94edecfe,None,None,source_only,Cricsheet,[PJK Mooney],None,PJK Mooney,pjkmooney
3246,CS_fd67a507,fd67a507,None,None,source_only,Cricsheet,[DI Joyce],None,DI Joyce,dijoyce



Candidate built. No original registry modified.


In [29]:

# ============================================================
# STEP 21: OLD VS NEW REGISTRY COMPARISON
# READ-ONLY
# ============================================================

old_registry = player_registry_final_updated.copy()
new_registry = player_registry_candidate.copy()

print("=" * 70)
print("REGISTRY SIZE COMPARISON")
print("=" * 70)

print("Old registry:", old_registry.shape)
print("New registry:", new_registry.shape)

# Compare CricketIQ IDs
old_ids = set(old_registry["cricketiq_player_id"].dropna())
new_ids = set(new_registry["cricketiq_player_id"].dropna())

removed_ids = old_ids - new_ids
added_ids = new_ids - old_ids
common_ids = old_ids & new_ids

print("\nOld unique IDs:", len(old_ids))
print("New unique IDs:", len(new_ids))
print("Common IDs:", len(common_ids))
print("Removed IDs:", len(removed_ids))
print("Added IDs:", len(added_ids))

# Inspect removed records
removed_records = old_registry[
    old_registry["cricketiq_player_id"].isin(removed_ids)
].copy()

print("\n" + "=" * 70)
print("RECORDS REMOVED FROM OLD REGISTRY")
print("=" * 70)

print("Removed record count:", len(removed_records))

print("\nRemoved records by source:")
print(removed_records["source"].value_counts(dropna=False))

display(
    removed_records[
        [
            "cricketiq_player_id",
            "player_id",
            "cricsheet_name",
            "crickrida_name",
            "source",
            "canonical_player_name"
        ]
    ].sort_values(["source", "canonical_player_name"])
)

# Inspect added records
added_records = new_registry[
    new_registry["cricketiq_player_id"].isin(added_ids)
].copy()

print("\n" + "=" * 70)
print("RECORDS ADDED TO NEW REGISTRY")
print("=" * 70)

print("Added record count:", len(added_records))

print("\nAdded records by source:")
print(added_records["source"].value_counts(dropna=False))

display(
    added_records[
        [
            "cricketiq_player_id",
            "player_id",
            "cricsheet_name",
            "crickrida_name",
            "source",
            "canonical_player_name"
        ]
    ].sort_values(["source", "canonical_player_name"])
)

print("\nCOMPARISON COMPLETE — NO DATA MODIFIED")


REGISTRY SIZE COMPARISON
Old registry: (3620, 10)
New registry: (3553, 10)

Old unique IDs: 3620
New unique IDs: 3553
Common IDs: 3463
Removed IDs: 157
Added IDs: 90

RECORDS REMOVED FROM OLD REGISTRY
Removed record count: 157

Removed records by source:
source
Cricsheet    90
Crickrida    67
Name: count, dtype: int64


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,source,canonical_player_name
3289,CR_alexcusack,None,None,Alex Cusack,Crickrida,Alex Cusack
3419,CR_alzarrijoseph,None,None,Alzarri Joseph,Crickrida,Alzarri Joseph
3291,CR_andrewpoynter,None,None,Andrew Poynter,Crickrida,Andrew Poynter
3292,CR_andrewwhite,None,None,Andrew White,Crickrida,Andrew White
3395,CR_andymcbrine,None,None,Andy McBrine,Crickrida,Andy McBrine
...,...,...,...,...,...,...
1493,CS_70f2f922,70f2f922,None,None,Cricsheet,TF van Woerkom
2602,CS_c89474d1,c89474d1,None,None,Cricsheet,TJ Murtagh
597,CS_29b89ae8,29b89ae8,None,None,Cricsheet,WB Rankin
2611,CS_c942a53b,c942a53b,None,None,Cricsheet,WK McCallan



RECORDS ADDED TO NEW REGISTRY
Added record count: 90

Added records by source:
source
Both    90
Name: count, dtype: int64


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,source,canonical_player_name
169,CRIQ_a159b802,a159b802,A van der Merwe,Albert van der Merwe,Both,Albert van der Merwe
129,CRIQ_568e5a4a,568e5a4a,AR Cusack,Alex Cusack,Both,Alex Cusack
187,CRIQ_b0946605,b0946605,AS Joseph,Alzarri Joseph,Both,Alzarri Joseph
164,CRIQ_d8e79af0,d8e79af0,AC Botha,Andre Botha,Both,Andre Botha
131,CRIQ_b98cdbf0,b98cdbf0,AD Poynter,Andrew Poynter,Both,Andrew Poynter
...,...,...,...,...,...,...
171,CRIQ_7a4819e1,7a4819e1,MJ Fourie,Thinus Fourie,Both,Thinus Fourie
186,CRIQ_9aedf23b,9aedf23b,T Mayes,Thomas Mayes,Both,Thomas Mayes
144,CRIQ_c89474d1,c89474d1,TJ Murtagh,Tim Murtagh,Both,Tim Murtagh
127,CRIQ_e01895bb,e01895bb,DT Johnston,Trent Johnston,Both,Trent Johnston



COMPARISON COMPLETE — NO DATA MODIFIED


In [30]:

# ============================================================
# STEP 22: RECONCILE OLD AND NEW REGISTRY IDENTITIES
# READ-ONLY
# ============================================================

# Identify records from the previous ID comparison
old_removed_records = old_registry[
    ~old_registry["cricketiq_player_id"].isin(new_ids)
].copy()

new_added_records = new_registry[
    ~new_registry["cricketiq_player_id"].isin(old_ids)
].copy()

new_shared_records = new_registry[
    new_registry["source"] == "Both"
].copy()

# Normalize names for reliable comparison
def normalize_registry_name(series):
    return (
        series.astype("string")
        .str.strip()
        .str.casefold()
    )

# ------------------------------------------------------------
# 1. Reconcile removed Cricsheet records using player_id
# ------------------------------------------------------------

old_removed_cs = old_removed_records[
    old_removed_records["source"] == "Cricsheet"
].copy()

old_removed_cs["comparison_id"] = (
    old_removed_cs["player_id"].astype("string")
)

new_shared_records["comparison_id"] = (
    new_shared_records["player_id"].astype("string")
)

cs_reconciliation = old_removed_cs.merge(
    new_shared_records[
        ["player_id", "cricsheet_name", "crickrida_name",
         "cricketiq_player_id", "comparison_id"]
    ],
    on="comparison_id",
    how="left",
    suffixes=("_old", "_new"),
    indicator=True
)

print("=" * 70)
print("CRICSHEET RECORD RECONCILIATION")
print("=" * 70)

print("Old Cricsheet records removed:", len(old_removed_cs))
print("Matched to new shared identities:",
      (cs_reconciliation["_merge"] == "both").sum())
print("Unmatched:",
      (cs_reconciliation["_merge"] == "left_only").sum())

if (cs_reconciliation["_merge"] == "left_only").any():
    display(
        cs_reconciliation[
            cs_reconciliation["_merge"] == "left_only"
        ]
    )

# ------------------------------------------------------------
# 2. Reconcile removed Crickrida records using player names
# ------------------------------------------------------------

old_removed_cr = old_removed_records[
    old_removed_records["source"] == "Crickrida"
].copy()

old_removed_cr["comparison_name"] = normalize_registry_name(
    old_removed_cr["crickrida_name"]
)

new_shared_records["comparison_name"] = normalize_registry_name(
    new_shared_records["crickrida_name"]
)

cr_reconciliation = old_removed_cr.merge(
    new_shared_records[
        ["cricketiq_player_id", "crickrida_name",
         "player_id", "comparison_name"]
    ],
    on="comparison_name",
    how="left",
    suffixes=("_old", "_new"),
    indicator=True
)

print("\n" + "=" * 70)
print("CRICKRIDA RECORD RECONCILIATION")
print("=" * 70)

print("Old Crickrida records removed:", len(old_removed_cr))
print("Matched to new shared identities:",
      (cr_reconciliation["_merge"] == "both").sum())
print("Unmatched:",
      (cr_reconciliation["_merge"] == "left_only").sum())

if (cr_reconciliation["_merge"] == "left_only").any():
    display(
        cr_reconciliation[
            cr_reconciliation["_merge"] == "left_only"
        ]
    )

# ------------------------------------------------------------
# 3. Summary of the new shared identities
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("NEW SHARED IDENTITY SUMMARY")
print("=" * 70)

print("New shared identities:", len(new_shared_records))
print("Unique underlying player IDs:",
      new_shared_records["player_id"].nunique())
print("Unique Crickrida names:",
      new_shared_records["comparison_name"].nunique())

print("\nRECONCILIATION COMPLETE — NO DATA MODIFIED")


CRICSHEET RECORD RECONCILIATION
Old Cricsheet records removed: 90
Matched to new shared identities: 90
Unmatched: 0

CRICKRIDA RECORD RECONCILIATION
Old Crickrida records removed: 67
Matched to new shared identities: 67
Unmatched: 0

NEW SHARED IDENTITY SUMMARY
New shared identities: 211
Unique underlying player IDs: 211
Unique Crickrida names: 211

RECONCILIATION COMPLETE — NO DATA MODIFIED


In [31]:

# ============================================================
# STEP 23: PERFORMANCE DATA ID COVERAGE AUDIT
# READ-ONLY
# ============================================================

performance_datasets = {
    "Batting": player_batting_updated,
    "Bowling": player_bowling_updated,
    "Dismissals": player_dismissal_updated,
    "Bowler dismissals": dismissal_bowler_updated,
    "Afghanistan batting": afg_player_batting_updated,
    "Afghanistan bowling": afg_player_bowling_updated
}

registry_ids = set(
    player_registry_candidate["cricketiq_player_id"]
    .dropna().astype(str)
)

print("=" * 75)
print("PERFORMANCE DATA ID COVERAGE")
print("=" * 75)

for dataset_name, df in performance_datasets.items():

    if "cricketiq_player_id" not in df.columns:
        print(f"\n{dataset_name}: cricketiq_player_id column missing")
        continue

    performance_ids = set(
        df["cricketiq_player_id"]
        .dropna().astype(str)
    )

    matched_ids = performance_ids & registry_ids
    unmatched_ids = performance_ids - registry_ids

    print(f"\n{dataset_name}")
    print("-" * 50)
    print("Rows:", len(df))
    print("Unique player IDs:", len(performance_ids))
    print("Matched IDs:", len(matched_ids))
    print("Unmatched IDs:", len(unmatched_ids))

    if unmatched_ids:
        print("Unmatched ID examples:", list(unmatched_ids)[:10])

print("\nPERFORMANCE AUDIT COMPLETE — NO DATA MODIFIED")


PERFORMANCE DATA ID COVERAGE

Batting
--------------------------------------------------
Rows: 45432
Unique player IDs: 1934
Matched IDs: 1848
Unmatched IDs: 86
Unmatched ID examples: ['CS_0a9d8b61', 'CS_3979b5e0', 'CS_e012ad13', 'CS_d2a6c0e6', 'CS_773cec26', 'CS_4a745f65', 'CS_3f18724b', 'CS_3241e3fd', 'CS_b0946605', 'CS_29b89ae8']

Bowling
--------------------------------------------------
Rows: 31186
Unique player IDs: 1502
Matched IDs: 1433
Unmatched IDs: 69
Unmatched ID examples: ['CS_b3e9b7fa', 'CS_d2c250c8', 'CS_0a9d8b61', 'CS_d8e79af0', 'CS_99cddc5e', 'CS_c5326dd0', 'CS_e01895bb', 'CS_33b3d2df', 'CS_c5aef772', 'CS_e3fe4b49']

Dismissals
--------------------------------------------------
Rows: 37711
Unique player IDs: 1841
Matched IDs: 1759
Unmatched IDs: 82
Unmatched ID examples: ['CS_0a9d8b61', 'CS_3979b5e0', 'CS_e012ad13', 'CS_d2a6c0e6', 'CS_773cec26', 'CS_4a745f65', 'CS_3f18724b', 'CS_3241e3fd', 'CS_b0946605', 'CS_29b89ae8']

Bowler dismissals
-------------------------------

In [32]:

# ============================================================
# STEP 24: CORRECTED PERFORMANCE IDENTITY VALIDATION
# READ-ONLY
# ============================================================

# 1. Prepare candidate identity keys

candidate_cs_ids = set(
    player_registry_candidate["player_id"]
    .dropna()
    .astype(str)
)

def normalize_name(series):
    return (
        series.astype("string")
        .str.lower()
        .str.replace(r"[^a-z0-9]", "", regex=True)
    )

candidate_cr_names = set(
    normalize_name(
        player_registry_candidate["crickrida_name"].dropna()
    )
)

print("=" * 75)
print("CANDIDATE IDENTITY KEYS")
print("=" * 75)

print("Cricsheet player IDs:", len(candidate_cs_ids))
print("Crickrida player names:", len(candidate_cr_names))


# 2. Validate Cricsheet performance datasets

cricsheet_datasets = {
    "Batting": player_batting_updated,
    "Bowling": player_bowling_updated,
    "Dismissals": player_dismissal_updated,
    "Bowler dismissals": dismissal_bowler_updated
}

print("\n" + "=" * 75)
print("CRICSHEET PERFORMANCE VALIDATION")
print("=" * 75)

for name, df in cricsheet_datasets.items():

    performance_ids = (
        df["cricketiq_player_id"]
        .dropna()
        .astype(str)
    )

    # CS_hash and CRIQ_hash both resolve to hash
    stable_ids = set(
        performance_ids.str.split("_", n=1).str[-1]
    )

    matched = stable_ids & candidate_cs_ids
    unmatched = stable_ids - candidate_cs_ids

    print(f"\n{name}")
    print("Rows:", len(df))
    print("Unique underlying IDs:", len(stable_ids))
    print("Matched:", len(matched))
    print("Unmatched:", len(unmatched))

    if unmatched:
        print("Unmatched examples:", list(unmatched)[:10])


# 3. Validate Afghanistan / Crickrida performance datasets

afghanistan_datasets = {
    "Afghanistan batting": afg_player_batting_updated,
    "Afghanistan bowling": afg_player_bowling_updated
}

print("\n" + "=" * 75)
print("CRICKRIDA PERFORMANCE VALIDATION")
print("=" * 75)

for name, df in afghanistan_datasets.items():

    performance_names = normalize_name(df["player_name"])
    unique_names = set(performance_names.dropna())

    matched = unique_names & candidate_cr_names
    unmatched = unique_names - candidate_cr_names

    print(f"\n{name}")
    print("Rows:", len(df))
    print("Unique player names:", len(unique_names))
    print("Matched:", len(matched))
    print("Unmatched:", len(unmatched))

    if unmatched:
        print("Unmatched examples:", list(unmatched)[:15])


print("\nCORRECTED PERFORMANCE AUDIT COMPLETE")
print("No original DataFrames or files modified.")


CANDIDATE IDENTITY KEYS
Cricsheet player IDs: 3287
Crickrida player names: 477

CRICSHEET PERFORMANCE VALIDATION

Batting
Rows: 45432
Unique underlying IDs: 1934
Matched: 1934
Unmatched: 0

Bowling
Rows: 31186
Unique underlying IDs: 1502
Matched: 1502
Unmatched: 0

Dismissals
Rows: 37711
Unique underlying IDs: 1841
Matched: 1841
Unmatched: 0

Bowler dismissals
Rows: 34975
Unique underlying IDs: 1281
Matched: 1281
Unmatched: 0

CRICKRIDA PERFORMANCE VALIDATION

Afghanistan batting
Rows: 2719
Unique player names: 403
Matched: 403
Unmatched: 0

Afghanistan bowling
Rows: 1765
Unique player names: 303
Matched: 303
Unmatched: 0

CORRECTED PERFORMANCE AUDIT COMPLETE
No original DataFrames or files modified.


In [33]:
print("=" * 75)
print("CANDIDATE REGISTRY VALIDATION")
print("=" * 75)

print("\nShape:", player_registry_candidate.shape)
print("\nColumns:")
print(player_registry_candidate.columns.tolist())

print("\nSource distribution:")
print(player_registry_candidate["source"].value_counts(dropna=False))

print("\nDuplicate CricketIQ IDs:",
      player_registry_candidate["cricketiq_player_id"].duplicated().sum())

print("\nDuplicate non-null player IDs:",
      player_registry_candidate["player_id"].dropna().duplicated().sum())

print("\nMissing CricketIQ IDs:",
      player_registry_candidate["cricketiq_player_id"].isna().sum())

print("\nMissing canonical names:",
      player_registry_candidate["canonical_player_name"].isna().sum())

print("\n" + "=" * 75)
print("JOYCE IDENTITY VALIDATION")
print("=" * 75)

joyce_records = player_registry_candidate[
    player_registry_candidate["cricsheet_name"]
    .astype("string").str.contains("Joyce", case=False, na=False)
    |
    player_registry_candidate["crickrida_name"]
    .astype("string").str.contains("Joyce", case=False, na=False)
]

display(joyce_records)

CANDIDATE REGISTRY VALIDATION

Shape: (3553, 10)

Columns:
['cricketiq_player_id', 'player_id', 'cricsheet_name', 'crickrida_name', 'match_status', 'source', 'player_names', 'player_name', 'canonical_player_name', 'normalized_name']

Source distribution:
source
Cricsheet    3076
Crickrida     266
Both          211
Name: count, dtype: int64

Duplicate CricketIQ IDs: 0

Duplicate non-null player IDs: 0

Missing CricketIQ IDs: 0

Missing canonical names: 0

JOYCE IDENTITY VALIDATION


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,match_status,source,player_names,player_name,canonical_player_name,normalized_name
118,CRIQ_a48fc79d,a48fc79d,EC Joyce,Ed Joyce,verified_alias,Both,None,None,Ed Joyce,edjoyce


In [34]:
# Check DI Joyce is preserved as a separate Cricsheet identity
di_joyce = player_registry_candidate[
    player_registry_candidate["player_id"].astype("string") == "fd67a507"
]

print("DI Joyce records:")
display(di_joyce)

assert len(di_joyce) == 1, "DI Joyce should have exactly one registry record"
assert di_joyce.iloc[0]["source"] == "Cricsheet", (
    "DI Joyce should remain Cricsheet-only"
)
assert di_joyce.iloc[0]["crickrida_name"] is None or pd.isna(
    di_joyce.iloc[0]["crickrida_name"]
), "DI Joyce must not be associated with Crickrida Ed Joyce"

print("DI Joyce identity preserved successfully.")

DI Joyce records:


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,match_status,source,player_names,player_name,canonical_player_name,normalized_name
3246,CS_fd67a507,fd67a507,None,None,source_only,Cricsheet,[DI Joyce],None,DI Joyce,dijoyce


DI Joyce identity preserved successfully.


In [35]:
import pandas as pd

def normalize_player_name(value):
    if pd.isna(value):
        return ""
    return (
        str(value)
        .lower()
        .replace(".", "")
        .replace("'", "")
        .replace("-", "")
        .replace(" ", "")
    )

# Current corrected identity map
identity_map = player_identity_map_updated.copy(deep=True)

# --------------------------------------------------
# 1. CRICSHEET IRELAND PLAYERS
# --------------------------------------------------

ireland_cs = cricsheet_appearances[
    cricsheet_appearances["team"].astype(str).str.strip().str.casefold()
    == "ireland"
].copy()

ireland_cs["date"] = pd.to_datetime(
    ireland_cs["date"], errors="coerce"
)

ireland_cs_summary = (
    ireland_cs.groupby(
        ["player_name", "source_player_id"], as_index=False
    )
    .agg(
        matches_played=("match_id", "nunique"),
        first_appearance=("date", "min"),
        last_appearance=("date", "max")
    )
)

mapped_cs_ids = set(
    identity_map["player_id"].dropna().astype(str)
)

unmapped_ireland_cs = ireland_cs_summary[
    ~ireland_cs_summary["source_player_id"].astype(str).isin(mapped_cs_ids)
].copy()

# --------------------------------------------------
# 2. CRICKRIDA IRELAND PLAYERS
# --------------------------------------------------

ireland_cr = cr_player_history[
    cr_player_history["team"].astype(str).str.strip().str.casefold()
    == "ireland"
].copy()

mapped_cr_names = set(
    identity_map["crickrida_name"]
    .dropna()
    .apply(normalize_player_name)
)

unmapped_ireland_cr = ireland_cr[
    ~ireland_cr["player_name"]
    .apply(normalize_player_name)
    .isin(mapped_cr_names)
].copy()

# --------------------------------------------------
# 3. REPORT
# --------------------------------------------------

print("=" * 70)
print("CURRENT IRELAND IDENTITY STATUS")
print("=" * 70)

print("\nCricsheet Ireland:")
print("Total players:", len(ireland_cs_summary))
print("Mapped:", len(ireland_cs_summary) - len(unmapped_ireland_cs))
print("Unmapped:", len(unmapped_ireland_cs))

print("\nCrickrida Ireland:")
print("Total players:", len(ireland_cr))
print("Mapped:", len(ireland_cr) - len(unmapped_ireland_cr))
print("Unmapped:", len(unmapped_ireland_cr))

print("\n" + "=" * 70)
print("UNMAPPED CRICSHEET IRELAND PLAYERS")
print("=" * 70)

display(
    unmapped_ireland_cs.sort_values("first_appearance")
)

print("\n" + "=" * 70)
print("UNMAPPED CRICKRIDA IRELAND PLAYERS")
print("=" * 70)

display(
    unmapped_ireland_cr.sort_values("crickrida_first_date")
)

CURRENT IRELAND IDENTITY STATUS

Cricsheet Ireland:
Total players: 68
Mapped: 66
Unmapped: 2

Crickrida Ireland:
Total players: 45
Mapped: 42
Unmapped: 3

UNMAPPED CRICSHEET IRELAND PLAYERS


,player_name,source_player_id,matches_played,first_appearance,last_appearance
14,DI Joyce,fd67a507,3,2006-06-13,2007-06-24
50,PJK Mooney,94edecfe,2,2006-06-13,2007-04-03



UNMAPPED CRICKRIDA IRELAND PLAYERS


,team,normalized_name,player_name,crickrida_first_date,crickrida_last_date,crickrida_matches
224,Ireland,jameshall,James Hall,2010-07-03,2010-07-03,1
223,Ireland,jamescamerondow,James Cameron-Dow,2019-03-02,2019-03-10,4
208,Ireland,bencalitz,Ben Calitz,2026-08-07,2026-08-12,3


In [36]:
# READ-ONLY FINAL IRELAND VALIDATION

expected_cs_only = {
    "fd67a507": "DI Joyce",
    "94edecfe": "PJK Mooney"
}

expected_cr_only = {
    "James Hall",
    "James Cameron-Dow",
    "Ben Calitz"
}

# Check Cricsheet-only players
print("CRICSHEET SOURCE-ONLY CHECK")

for player_id, expected_name in expected_cs_only.items():
    map_rows = player_identity_map_updated[
        player_identity_map_updated["player_id"].astype(str)
        == player_id
    ]

    source_rows = cricsheet_only_master[
        cricsheet_only_master["player_id"].astype(str)
        == player_id
    ]

    print(
        expected_name,
        "| In identity map:", len(map_rows),
        "| In Cricsheet-only source:", len(source_rows)
    )

# Check Crickrida-only players
print("\nCRICKRIDA SOURCE-ONLY CHECK")

for player_name in expected_cr_only:
    map_rows = player_identity_map_updated[
        player_identity_map_updated["crickrida_name"]
        .astype("string").str.casefold()
        == player_name.casefold()
    ]

    source_rows = crickrida_only_master[
        crickrida_only_master["player_name"]
        .astype("string").str.casefold()
        == player_name.casefold()
    ]

    print(
        player_name,
        "| In identity map:", len(map_rows),
        "| In Crickrida-only source:", len(source_rows)
    )

CRICSHEET SOURCE-ONLY CHECK
DI Joyce | In identity map: 0 | In Cricsheet-only source: 1
PJK Mooney | In identity map: 0 | In Cricsheet-only source: 1

CRICKRIDA SOURCE-ONLY CHECK
Ben Calitz | In identity map: 0 | In Crickrida-only source: 1
James Cameron-Dow | In identity map: 0 | In Crickrida-only source: 1
James Hall | In identity map: 0 | In Crickrida-only source: 1


In [37]:

# ============================================================
# CRICKETIQ — COMPLETE CRICKRIDA COUNTRY MAPPING AUDIT
# READ-ONLY: DOES NOT MODIFY ORIGINAL DATA
# ============================================================

import pandas as pd
import re

# 1. Validate required DataFrames
assert "cr_player_history" in globals(), "cr_player_history is missing"
assert "player_identity_map_updated" in globals(), "Updated identity map is missing"

history = cr_player_history.copy()
identity_map = player_identity_map_updated.copy()

# 2. Normalize names consistently
def normalize_player_name(name):
    if pd.isna(name):
        return ""
    return re.sub(r"[^a-z0-9]", "", str(name).casefold())

# 3. Get all Crickrida names already represented in the identity map
mapped_names = set()

for column in ["crickrida_name", "crickrida_player_name"]:
    if column in identity_map.columns:
        names = identity_map[column].dropna().apply(normalize_player_name)
        mapped_names.update(name for name in names if name)

# 4. Prepare Crickrida history
history["audit_name"] = history["player_name"].apply(
    normalize_player_name
)

history["is_mapped"] = history["audit_name"].isin(mapped_names)

# 5. Country-level summary
country_audit = (
    history.groupby("team")
    .agg(
        total_players=("audit_name", "nunique"),
        mapped_players=("is_mapped", "sum")
    )
    .reset_index()
)

country_audit["unmapped_players"] = (
    country_audit["total_players"] -
    country_audit["mapped_players"]
)

country_audit["mapping_percentage"] = (
    country_audit["mapped_players"] /
    country_audit["total_players"] * 100
).round(2)

country_audit["status"] = country_audit["unmapped_players"].apply(
    lambda x: "COMPLETE" if x == 0 else "NEEDS MAPPING"
)

country_audit = country_audit.sort_values(
    ["unmapped_players", "team"],
    ascending=[False, True]
).reset_index(drop=True)

# 6. Print complete country summary
print("=" * 75)
print("CRICKRIDA COUNTRY MAPPING STATUS")
print("=" * 75)

display(country_audit)

# 7. Print all remaining unmapped players, country by country
unmapped_players = (
    history[~history["is_mapped"]]
    [["team", "player_name", "crickrida_first_date",
      "crickrida_last_date", "crickrida_matches"]]
    .drop_duplicates(subset=["team", "player_name"])
    .sort_values(["team", "player_name"])
    .reset_index(drop=True)
)

print("\n" + "=" * 75)
print("ALL UNMAPPED CRICKRIDA PLAYERS")
print("=" * 75)

for country in country_audit.loc[
    country_audit["unmapped_players"] > 0, "team"
]:
    country_players = unmapped_players[
        unmapped_players["team"] == country
    ]

    print(f"\n{country} — {len(country_players)} unmapped players")
    display(country_players)

# 8. Overall totals
print("\n" + "=" * 75)
print("OVERALL AUDIT")
print("=" * 75)

print("Countries:", history["team"].nunique())
print("Total country-player records:", len(history))
print("Total distinct country-player names:",
      history[["team", "audit_name"]].drop_duplicates().shape[0])
print("Countries complete:",
      (country_audit["status"] == "COMPLETE").sum())
print("Countries needing mapping:",
      (country_audit["status"] == "NEEDS MAPPING").sum())
print("Total unmapped country-player records:",
      country_audit["unmapped_players"].sum())

print("\nAUDIT COMPLETE — ORIGINAL DATA UNCHANGED")


CRICKRIDA COUNTRY MAPPING STATUS


,team,total_players,mapped_players,unmapped_players,mapping_percentage,status
0,Afghanistan,58,2,56,3.45,NEEDS MAPPING
1,Zimbabwe,52,1,51,1.92,NEEDS MAPPING
2,Sri Lanka,40,0,40,0.00,NEEDS MAPPING
3,Australia,29,3,26,10.34,NEEDS MAPPING
4,England,28,6,22,21.43,NEEDS MAPPING
5,India,36,14,22,38.89,NEEDS MAPPING
6,South Africa,26,4,22,15.38,NEEDS MAPPING
7,New Zealand,23,2,21,8.70,NEEDS MAPPING
8,Ireland,45,42,3,93.33,NEEDS MAPPING
9,Bangladesh,51,49,2,96.08,NEEDS MAPPING



ALL UNMAPPED CRICKRIDA PLAYERS

Afghanistan — 56 unmapped players


,team,player_name,crickrida_first_date,crickrida_last_date,crickrida_matches
0,Afghanistan,AM Ghazanfar,2024-03-07,2026-10-23,24
1,Afghanistan,Abdul Malik,2024-09-22,2024-12-21,3
2,Afghanistan,"Abdul Rahman (Afghanistan, 2023)",2023-07-11,2023-08-24,3
3,Afghanistan,Afsar Zazai,2015-01-10,2017-06-11,9
4,Afghanistan,Aftab Alam,2014-07-22,2019-06-22,22
5,Afghanistan,Amir Hamza,2012-02-10,2017-06-11,25
6,Afghanistan,Asghar Afghan,2010-07-03,2021-01-26,77
7,Afghanistan,Azmatullah Omarzai,2021-01-21,2026-10-23,49
8,Afghanistan,Bashir Ahmad,2025-10-08,2025-10-11,2
9,Afghanistan,Bilal Sami,2024-12-21,2026-10-23,5



Zimbabwe — 51 unmapped players


,team,player_name,crickrida_first_date,crickrida_last_date,crickrida_matches
215,Zimbabwe,Ben Curran,2024-12-17,2026-10-19,4
216,Zimbabwe,Blessing Muzarabani,2018-02-11,2026-10-23,10
217,Zimbabwe,Brad Evans,2026-10-19,2026-10-23,2
218,Zimbabwe,Brendan Taylor,2014-07-18,2026-10-23,12
219,Zimbabwe,Brian Bennett,2024-12-17,2026-10-23,5
220,Zimbabwe,Brian Vitori,2018-02-09,2018-03-06,4
221,Zimbabwe,Cephas Zhuwao,2018-03-06,2018-03-06,1
222,Zimbabwe,Chamu Chibhabha,2015-10-16,2016-01-06,9
223,Zimbabwe,Chris Mpofu,2017-02-16,2017-02-26,5
224,Zimbabwe,Craig Ervine,2015-10-16,2024-12-21,24


... (15 outputs left out)

,team,player_name,crickrida_first_date,crickrida_last_date,crickrida_matches
82,Bangladesh,Al Islam,2026-10-21,2026-10-21,1
83,Bangladesh,"Yasir Ali (Bangladesh, 2021 to 2026)",2022-02-23,2022-02-28,2



Pakistan — 1 unmapped players


,team,player_name,crickrida_first_date,crickrida_last_date,crickrida_matches
152,Pakistan,Salman Agha,2023-08-22,2023-08-26,3



OVERALL AUDIT
Countries: 12
Total country-player records: 454
Total distinct country-player names: 454
Countries complete: 1
Countries needing mapping: 11
Total unmapped country-player records: 266

AUDIT COMPLETE — ORIGINAL DATA UNCHANGED


In [38]:

# Inspect unresolved Bangladesh and Pakistan players

countries_to_inspect = ["Bangladesh", "Pakistan"]

for country in countries_to_inspect:
    print(f"\n{'=' * 60}")
    print(country.upper())
    print("=" * 60)

    result = unmapped_players[
        unmapped_players["team"] == country
    ].copy()

    display(result)



BANGLADESH


,team,player_name,crickrida_first_date,crickrida_last_date,crickrida_matches
82,Bangladesh,Al Islam,2026-10-21,2026-10-21,1
83,Bangladesh,"Yasir Ali (Bangladesh, 2021 to 2026)",2022-02-23,2022-02-28,2



PAKISTAN


,team,player_name,crickrida_first_date,crickrida_last_date,crickrida_matches
152,Pakistan,Salman Agha,2023-08-22,2023-08-26,3


In [39]:

# ============================================================
# BANGLADESH + PAKISTAN: CRICSHEET IDENTITY INVESTIGATION
# READ-ONLY — NO MAPPINGS WILL BE CREATED
# ============================================================

import pandas as pd
import re
from difflib import SequenceMatcher

targets = [
    {
        "country": "Bangladesh",
        "crickrida_name": "Al Islam",
        "first_date": "2026-10-21",
        "last_date": "2026-10-21"
    },
    {
        "country": "Bangladesh",
        "crickrida_name": "Yasir Ali (Bangladesh, 2021 to 2026)",
        "first_date": "2022-02-23",
        "last_date": "2022-02-28"
    },
    {
        "country": "Pakistan",
        "crickrida_name": "Salman Agha",
        "first_date": "2023-08-22",
        "last_date": "2023-08-26"
    }
]

cs_history = cs_player_history.copy()

def normalize_name(name):
    name = str(name).lower()
    name = re.sub(r"\([^)]*\)", "", name)
    return re.sub(r"[^a-z0-9]", "", name)

cs_history["normalized_check"] = (
    cs_history["player_name"].apply(normalize_name)
)

cs_history["cricsheet_first_date"] = pd.to_datetime(
    cs_history["cricsheet_first_date"]
)
cs_history["cricsheet_last_date"] = pd.to_datetime(
    cs_history["cricsheet_last_date"]
)

for target in targets:

    print("\n" + "=" * 75)
    print("CRICKRIDA:", target["crickrida_name"])
    print("COUNTRY:", target["country"])
    print("=" * 75)

    country_history = cs_history[
        cs_history["team"].str.casefold()
        == target["country"].casefold()
    ].copy()

    target_name = normalize_name(target["crickrida_name"])
    target_first = pd.Timestamp(target["first_date"])
    target_last = pd.Timestamp(target["last_date"])

    country_history["name_similarity"] = (
        country_history["normalized_check"].apply(
            lambda x: SequenceMatcher(
                None, target_name, x
            ).ratio()
        )
    )

    # Career date distance in days
    def date_distance(row):
        if row["cricsheet_last_date"] < target_first:
            return (target_first - row["cricsheet_last_date"]).days
        elif row["cricsheet_first_date"] > target_last:
            return (row["cricsheet_first_date"] - target_last).days
        return 0

    country_history["date_distance_days"] = (
        country_history.apply(date_distance, axis=1)
    )

    # Prioritize name similarity, then date proximity
    candidates = country_history.sort_values(
        ["name_similarity", "date_distance_days"],
        ascending=[False, True]
    )

    display(
        candidates[
            [
                "player_name",
                "source_player_id",
                "cricsheet_first_date",
                "cricsheet_last_date",
                "cricsheet_matches",
                "name_similarity",
                "date_distance_days"
            ]
        ].head(15)
    )



CRICKRIDA: Al Islam
COUNTRY: Bangladesh


,player_name,source_player_id,cricsheet_first_date,cricsheet_last_date,cricsheet_matches,name_similarity,date_distance_days
221,Taijul Islam,9d1d34bd,2014-12-01,2024-03-15,17,0.777778,950
194,Nazmul Islam,6575e3bc,2018-09-28,2018-10-26,4,0.777778,2917
206,Shafiul Islam,1a8eebbe,2010-01-04,2020-03-03,58,0.736842,2423
154,Jahurul Islam,6752c57b,2010-06-21,2013-05-08,11,0.736842,4914
186,Naeem Islam,02e83ac5,2008-11-07,2014-02-26,50,0.705882,4620
219,Sunzamul Islam,0f348f58,2017-05-19,2018-01-23,3,0.700000,3193
168,Manjural Islam,2cfa6534,2003-02-11,2003-03-01,5,0.700000,8635
225,Tanvir Islam,aa467af0,2025-07-02,2026-07-11,11,0.666667,102
200,Robiul Islam,d876610c,2013-05-03,2013-05-08,3,0.666667,4914
213,Shoriful Islam,bb34fd31,2021-05-25,2026-10-17,38,0.600000,4



CRICKRIDA: Yasir Ali (Bangladesh, 2021 to 2026)
COUNTRY: Bangladesh


,player_name,source_player_id,cricsheet_first_date,cricsheet_last_date,cricsheet_matches,name_similarity,date_distance_days
233,Yasir Ali Chowdhury,5791b840,2022-03-18,2023-03-20,5,0.640000,18
155,Jaker Ali,cd56a813,2024-12-08,2025-07-08,8,0.625000,1014
189,Nasir Hossain,6eea0b32,2011-08-14,2018-01-25,63,0.600000,1490
187,Nafees Iqbal,c5b87d8c,2003-11-07,2005-06-18,9,0.526316,6094
216,Soumya Sarkar,4d9f9686,2014-12-01,2026-10-17,76,0.500000,0
158,Jubair Hossain,902d916b,2014-11-28,2015-07-10,3,0.476190,2420
146,Faisal Hossain,89ffd337,2004-05-19,2010-07-16,4,0.476190,4240
168,Manjural Islam,2cfa6534,2003-02-11,2003-03-01,5,0.476190,6934
188,Nahid Rana,5318ee68,2024-12-08,2026-07-09,13,0.470588,1014
220,Syed Rasel,b743f01b,2005-08-31,2010-06-18,40,0.470588,4268



CRICKRIDA: Salman Agha
COUNTRY: Pakistan


,player_name,source_player_id,cricsheet_first_date,cricsheet_last_date,cricsheet_matches,name_similarity,date_distance_days
1162,Kamran Ghulam,35f4cf7e,2024-11-04,2025-02-08,7,0.636364,436
1253,Zaman Khan,28392e11,2023-09-14,2023-09-14,1,0.631579,19
1238,Usman Khan,bafb9238,2025-03-29,2025-03-29,1,0.631579,581
1239,Usman Khan (2),bafb9238,2025-04-05,2025-04-05,1,0.631579,588
1116,Agha Salman,89c16049,2022-08-16,2026-10-25,49,0.600000,0
1210,Salman Butt,4d6d6280,2004-09-22,2010-06-19,63,0.600000,4812
1161,Kamran Akmal,ff077124,2005-01-19,2017-04-11,118,0.571429,2324
1155,Imran Farhat,480ed517,2003-11-29,2013-06-10,44,0.571429,3725
1192,Naumanullah,a6c84e0e,2008-04-19,2008-04-19,1,0.571429,5603
1209,Saleem Elahi,1e7b1aa7,2003-02-11,2004-01-03,7,0.571429,7171


In [40]:

# ============================================================
# CRICKETIQ — APPROVE BANGLADESH + PAKISTAN IDENTITIES
# Adds Yasir Ali Chowdhury and Agha Salman
# Aliss Islam remains source-only
# ============================================================

import pandas as pd
import re

new_mappings = [
    {
        "player_id": "5791b840",
        "cricsheet_name": "Yasir Ali Chowdhury",
        "crickrida_name": "Yasir Ali (Bangladesh, 2021 to 2026)",
        "mapping_method": "verified_full_name_and_career_review",
        "confidence": "high",
        "notes": (
            "Full name identified as Yasir Ali Chowdhury. "
            "Crickrida records: 2022-02-23 to 2022-02-28. "
            "Cricsheet career begins 2022-03-18; no direct "
            "date overlap established."
        )
    },
    {
        "player_id": "89c16049",
        "cricsheet_name": "Agha Salman",
        "crickrida_name": "Salman Agha",
        "mapping_method": "verified_name_order_and_career_review",
        "confidence": "high",
        "notes": (
            "Salman Agha and Agha Salman are the same player. "
            "Crickrida records: 2023-08-22 to 2023-08-26, "
            "three matches. England scorecard supplied for "
            "2023-11-11."
        )
    }
]

# Work on a copy first
updated_map = player_identity_map_updated.copy()

# Validate required Cricsheet identities
for item in new_mappings:
    cs_match = cs_player_history[
        cs_player_history["source_player_id"].astype(str)
        == item["player_id"]
    ]

    assert len(cs_match) == 1, (
        f"Expected exactly one Cricsheet identity: "
        f"{item['cricsheet_name']}"
    )

    assert (
        cs_match.iloc[0]["player_name"] == item["cricsheet_name"]
    ), "Cricsheet name verification failed"

# Prevent duplicate player IDs or names
for item in new_mappings:
    assert not (
        updated_map["player_id"].astype(str) == item["player_id"]
    ).any(), f"Player ID already mapped: {item['player_id']}"

    assert not (
        updated_map["crickrida_name"].astype(str)
        == item["crickrida_name"]
    ).any(), f"Crickrida name already mapped: {item['crickrida_name']}"

    assert not (
        updated_map["cricsheet_name"].astype(str)
        == item["cricsheet_name"]
    ).any(), f"Cricsheet name already mapped: {item['cricsheet_name']}"

# Build records using the existing map schema
rows = []

for item in new_mappings:
    row = {column: None for column in updated_map.columns}

    row.update({
        "cricketiq_player_id": "CRIQ_" + item["player_id"],
        "player_id": item["player_id"],
        "cricsheet_name": item["cricsheet_name"],
        "crickrida_name": item["crickrida_name"],
        "normalized_name": re.sub(
            r"[^a-z0-9]",
            "",
            item["crickrida_name"].lower()
        ),
        "match_status": "verified_alias",
        "mapping_method": item["mapping_method"],
        "confidence": item["confidence"],
        "notes": item["notes"]
    })

    rows.append(row)

# Append only after all validations pass
updated_map = pd.concat(
    [updated_map, pd.DataFrame(rows)],
    ignore_index=True
)

# Final validation
assert updated_map["player_id"].dropna().is_unique
assert updated_map["cricketiq_player_id"].is_unique
assert updated_map["crickrida_name"].dropna().is_unique
assert updated_map["cricsheet_name"].dropna().is_unique

player_identity_map_updated = updated_map

print("MAPPINGS ADDED:", len(new_mappings))
print("UPDATED IDENTITY MAP:", player_identity_map_updated.shape)

display(
    player_identity_map_updated[
        player_identity_map_updated["player_id"].isin(
            ["5791b840", "89c16049"]
        )
    ]
)

print("\nAliss Islam has NOT been added.")
print("No registry or performance datasets were rebuilt.")


MAPPINGS ADDED: 2
UPDATED IDENTITY MAP: (213, 12)


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,normalized_name,match_status,crickrida_player_name,cricsheet_player_name,cricsheet_player_id,mapping_method,confidence,notes
211,CRIQ_5791b840,5791b840,Yasir Ali Chowdhury,"Yasir Ali (Bangladesh, 2021 to 2026)",yasiralibangladesh2021to2026,verified_alias,None,None,None,verified_full_name_and_career_review,high,Full name identified as Yasir Ali Chowdhury. C...
212,CRIQ_89c16049,89c16049,Agha Salman,Salman Agha,salmanagha,verified_alias,None,None,None,verified_name_order_and_career_review,high,Salman Agha and Agha Salman are the same playe...



Aliss Islam has NOT been added.
No registry or performance datasets were rebuilt.


In [41]:

# ============================================================
# CRICKETIQ — REGISTER ALISS ISLAM AS CRICKRIDA-ONLY
# No Cricsheet identity is assigned
# ============================================================

import pandas as pd
import re

player_name = "Al Islam"
canonical_name = "Aliss Islam"

# Find the original Crickrida history record
aliss_record = cr_player_history[
    (cr_player_history["team"] == "Bangladesh") &
    (cr_player_history["player_name"] == player_name)
]

assert len(aliss_record) == 1, (
    "Expected exactly one Al Islam Crickrida record."
)

# Check that the player is not already registered
existing = player_identity_map_updated[
    player_identity_map_updated["crickrida_name"]
    .astype("string").str.casefold() == player_name.casefold()
]

assert len(existing) == 0, "Player already exists in identity map."

# Create a Crickrida-only identity
new_row = {
    column: None
    for column in player_identity_map_updated.columns
}

new_row.update({
    "cricketiq_player_id": "CR_aliss_islam",
    "player_id": None,
    "cricsheet_name": None,
    "crickrida_name": player_name,
    "normalized_name": re.sub(
        r"[^a-z0-9]", "", canonical_name.casefold()
    ),
    "match_status": "crickrida_only",
    "crickrida_player_name": canonical_name,
    "mapping_method": "verified_crickrida_full_name",
    "confidence": "high",
    "notes": (
        "Full name identified as Aliss Islam. "
        "Crickrida Bangladesh player; reported to have "
        "played only against Afghanistan. No Cricsheet "
        "counterpart established. Retain Crickrida data only."
    )
})

player_identity_map_updated = pd.concat(
    [
        player_identity_map_updated,
        pd.DataFrame([new_row])
    ],
    ignore_index=True
)

print("Aliss Islam registered as Crickrida-only.")
print("Updated identity map:", player_identity_map_updated.shape)

display(
    player_identity_map_updated[
        player_identity_map_updated["crickrida_player_name"]
        .astype("string").str.casefold() == canonical_name.casefold()
    ]
)


Aliss Islam registered as Crickrida-only.
Updated identity map: (214, 12)


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,normalized_name,match_status,crickrida_player_name,cricsheet_player_name,cricsheet_player_id,mapping_method,confidence,notes
213,CR_aliss_islam,None,None,Al Islam,alissislam,crickrida_only,Aliss Islam,None,None,verified_crickrida_full_name,high,Full name identified as Aliss Islam. Crickrida...


In [42]:

# ============================================================
# IRELAND — SOURCE-ONLY IDENTITY INSPECTION
# READ-ONLY
# ============================================================

irish_crickrida_only = [
    "James Hall",
    "James Cameron-Dow",
    "Ben Calitz"
]

irish_cricsheet_only = {
    "fd67a507": "DI Joyce",
    "94edecfe": "PJK Mooney"
}

print("CRICKRIDA-ONLY MASTER RECORDS")
display(
    crickrida_only_master[
        crickrida_only_master["player_name"].isin(
            irish_crickrida_only
        )
    ]
)

print("\nCRICSHEET-ONLY MASTER RECORDS")
display(
    cricsheet_only_master[
        cricsheet_only_master["player_id"].astype(str).isin(
            irish_cricsheet_only.keys()
        )
    ]
)

print("\nCURRENT IDENTITY MAP ENTRIES")
display(
    player_identity_map_updated[
        player_identity_map_updated["crickrida_name"].isin(
            irish_crickrida_only
        )
        |
        player_identity_map_updated["player_id"].astype(str).isin(
            irish_cricsheet_only.keys()
        )
    ]
)


CRICKRIDA-ONLY MASTER RECORDS


,player_name,normalized_name,cricketiq_player_id,source,match_status
1,James Hall,jameshall,CR_jameshall,Crickrida,source_only
1360,James Cameron-Dow,jamescamerondow,CR_jamescamerondow,Crickrida,source_only
2604,Ben Calitz,bencalitz,CR_bencalitz,Crickrida,source_only



CRICSHEET-ONLY MASTER RECORDS


,player_id,player_names,normalized_names,cricketiq_player_id,source,match_status
1838,94edecfe,[PJK Mooney],[pjkmooney],CS_94edecfe,Cricsheet,source_only
3141,fd67a507,[DI Joyce],[dijoyce],CS_fd67a507,Cricsheet,source_only



CURRENT IDENTITY MAP ENTRIES


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,normalized_name,match_status,crickrida_player_name,cricsheet_player_name,cricsheet_player_id,mapping_method,confidence,notes


In [43]:

# ============================================================
# CRICKETIQ — RESOLVED IDENTITY AUDIT
# SHARED + CRICKRIDA-ONLY
# ============================================================

import pandas as pd
import re

def normalize_identity(name):
    if pd.isna(name):
        return ""
    return re.sub(r"[^a-z0-9]", "", str(name).casefold())

history = cr_player_history.copy()

# Shared Crickrida identities
shared_names = set(
    player_identity_map_updated["crickrida_name"]
    .dropna()
    .apply(normalize_identity)
)

# Crickrida-only identities
source_only_names = set(
    crickrida_only_master["player_name"]
    .dropna()
    .apply(normalize_identity)
)

# All resolved Crickrida identities
resolved_names = shared_names | source_only_names

history["audit_name"] = history["player_name"].apply(
    normalize_identity
)

history["identity_status"] = history["audit_name"].apply(
    lambda name: (
        "Shared"
        if name in shared_names
        else "Crickrida-only"
        if name in source_only_names
        else "Unresolved"
    )
)

# Country summary
country_audit_corrected = (
    history.groupby("team")
    .agg(
        total_players=("audit_name", "nunique"),
        shared_players=(
            "identity_status",
            lambda x: (x == "Shared").sum()
        ),
        crickrida_only_players=(
            "identity_status",
            lambda x: (x == "Crickrida-only").sum()
        ),
        unresolved_players=(
            "identity_status",
            lambda x: (x == "Unresolved").sum()
        )
    )
    .reset_index()
)

country_audit_corrected["resolved_players"] = (
    country_audit_corrected["shared_players"] +
    country_audit_corrected["crickrida_only_players"]
)

country_audit_corrected["resolution_percentage"] = (
    country_audit_corrected["resolved_players"] /
    country_audit_corrected["total_players"] * 100
).round(2)

display(
    country_audit_corrected.sort_values(
        "unresolved_players",
        ascending=False
    ).reset_index(drop=True)
)

# Ireland detailed validation
print("\nIRELAND IDENTITY STATUS")

display(
    history[history["team"] == "Ireland"][
        ["team", "player_name", "identity_status"]
    ].drop_duplicates().sort_values("player_name")
)

print("\nIRELAND TOTALS")
print(
    history.loc[
        history["team"] == "Ireland", "player_name"
    ].nunique(),
    "Crickrida players"
)


,team,total_players,shared_players,crickrida_only_players,unresolved_players,resolved_players,resolution_percentage
0,Afghanistan,58,2,56,0,58,100.0
1,Australia,29,3,26,0,29,100.0
2,Bangladesh,51,51,0,0,51,100.0
3,England,28,6,22,0,28,100.0
4,India,36,14,22,0,36,100.0
5,Ireland,45,42,3,0,45,100.0
6,New Zealand,23,2,21,0,23,100.0
7,Pakistan,39,39,0,0,39,100.0
8,South Africa,26,4,22,0,26,100.0
9,Sri Lanka,40,0,40,0,40,100.0



IRELAND IDENTITY STATUS


,team,player_name,identity_status
202,Ireland,Alex Cusack,Shared
203,Ireland,Andrew Poynter,Shared
204,Ireland,Andrew White,Shared
205,Ireland,Andy Balbirnie,Shared
206,Ireland,Andy McBrine,Shared
207,Ireland,Barry McCarthy,Shared
208,Ireland,Ben Calitz,Crickrida-only
209,Ireland,Boyd Rankin,Shared
210,Ireland,Byron McDonough,Shared
211,Ireland,Cade Carmichael,Shared



IRELAND TOTALS
45 Crickrida players


In [44]:

# ============================================================
# CRICKETIQ — STRICT IDENTITY STATUS AUDIT
# Only verified shared and explicitly confirmed source-only
# identities are classified as resolved.
# ============================================================

import pandas as pd
import re

def norm_name(name):
    if pd.isna(name):
        return ""
    return re.sub(r"[^a-z0-9]", "", str(name).casefold())

history = cr_player_history.copy()
identity_map = player_identity_map_updated.copy()

# 1. Shared identities must have an actual Cricsheet identity
shared_rows = identity_map[
    identity_map["player_id"].notna() &
    identity_map["cricsheet_name"].notna() &
    identity_map["crickrida_name"].notna() &
    (identity_map["match_status"] != "crickrida_only")
]

shared_names = set(
    shared_rows["crickrida_name"].apply(norm_name)
)

# 2. Explicitly verified Crickrida-only identities
verified_crickrida_only = {
    "Al Islam",          # Aliss Islam
    "James Hall",
    "James Cameron-Dow",
    "Ben Calitz"
}

source_only_names = {
    norm_name(name)
    for name in verified_crickrida_only
}

# 3. Classify every Crickrida player
history["audit_name"] = history["player_name"].apply(norm_name)

def classify_identity(name):
    if name in shared_names:
        return "Shared"
    elif name in source_only_names:
        return "Crickrida-only"
    return "Unresolved"

history["identity_status"] = history["audit_name"].apply(
    classify_identity
)

# 4. Country-level summary
corrected_audit = (
    history.groupby("team")
    .agg(
        total_players=("audit_name", "nunique"),
        shared_players=(
            "identity_status",
            lambda x: (x == "Shared").sum()
        ),
        crickrida_only_players=(
            "identity_status",
            lambda x: (x == "Crickrida-only").sum()
        ),
        unresolved_players=(
            "identity_status",
            lambda x: (x == "Unresolved").sum()
        )
    )
    .reset_index()
)

corrected_audit["resolved_players"] = (
    corrected_audit["shared_players"] +
    corrected_audit["crickrida_only_players"]
)

corrected_audit["resolution_percentage"] = (
    corrected_audit["resolved_players"] /
    corrected_audit["total_players"] * 100
).round(2)

display(
    corrected_audit.sort_values(
        "unresolved_players",
        ascending=False
    ).reset_index(drop=True)
)

# 5. List unresolved players for the next investigation
unresolved = (
    history[history["identity_status"] == "Unresolved"]
    [[
        "team",
        "player_name",
        "crickrida_first_date",
        "crickrida_last_date",
        "crickrida_matches"
    ]]
    .drop_duplicates(["team", "player_name"])
    .sort_values(["team", "player_name"])
    .reset_index(drop=True)
)

print("\nUNRESOLVED CRICKRIDA PLAYERS")
print("Total:", len(unresolved))
display(unresolved)

print("\nSTRICT AUDIT COMPLETE — NO DATA MODIFIED")


,team,total_players,shared_players,crickrida_only_players,unresolved_players,resolved_players,resolution_percentage
0,Afghanistan,58,2,0,56,2,3.45
1,Zimbabwe,52,1,0,51,1,1.92
2,Sri Lanka,40,0,0,40,0,0.00
3,Australia,29,3,0,26,3,10.34
4,India,36,14,0,22,14,38.89
5,England,28,6,0,22,6,21.43
6,South Africa,26,4,0,22,4,15.38
7,New Zealand,23,2,0,21,2,8.70
8,Bangladesh,51,50,1,0,51,100.00
9,Ireland,45,42,3,0,45,100.00



UNRESOLVED CRICKRIDA PLAYERS
Total: 260


,team,player_name,crickrida_first_date,crickrida_last_date,crickrida_matches
0,Afghanistan,AM Ghazanfar,2024-03-07,2026-10-23,24
1,Afghanistan,Abdul Malik,2024-09-22,2024-12-21,3
2,Afghanistan,"Abdul Rahman (Afghanistan, 2023)",2023-07-11,2023-08-24,3
3,Afghanistan,Afsar Zazai,2015-01-10,2017-06-11,9
4,Afghanistan,Aftab Alam,2014-07-22,2019-06-22,22
...,...,...,...,...,...
255,Zimbabwe,Tinotenda Mutombodzi,2015-10-16,2015-10-24,5
256,Zimbabwe,Trevor Gwandu,2024-12-19,2024-12-21,2
257,Zimbabwe,Vusi Sibanda,2014-07-18,2014-07-18,1
258,Zimbabwe,Wellington Masakadza,2015-10-16,2026-10-23,10



STRICT AUDIT COMPLETE — NO DATA MODIFIED


In [45]:
player_identity_map_updated[
    player_identity_map_updated["crickrida_name"]
    .astype(str)
    .str.contains("Salman Agha", case=False, na=False)
][[
    "cricketiq_player_id",
    "player_id",
    "cricsheet_name",
    "crickrida_name",
    "match_status",
    "mapping_method",
    "confidence"
]]

,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,match_status,mapping_method,confidence
212,CRIQ_89c16049,89c16049,Agha Salman,Salman Agha,verified_alias,verified_name_order_and_career_review,high


In [46]:
row = player_identity_map_updated[
    player_identity_map_updated["player_id"].astype(str) == "89c16049"
]

print(row.T.to_string())

                                                                                                                                                                           212
cricketiq_player_id                                                                                                                                              CRIQ_89c16049
player_id                                                                                                                                                             89c16049
cricsheet_name                                                                                                                                                     Agha Salman
crickrida_name                                                                                                                                                     Salman Agha
normalized_name                                                                                                              

In [47]:
# Check which teams are represented in Cricsheet player history

print("CRICSHEET PLAYER HISTORY COLUMNS:")
print(cs_player_history.columns.tolist())

print("\nZIMBABWE RECORDS IN CRICSHEET:")
display(
    cs_player_history[
        cs_player_history.astype(str)
        .apply(
            lambda col: col.str.contains(
                "Zimbabwe", case=False, na=False
            )
        )
        .any(axis=1)
    ]
)

CRICSHEET PLAYER HISTORY COLUMNS:
['team', 'normalized_name', 'player_name', 'source_player_id', 'cricsheet_first_date', 'cricsheet_last_date', 'cricsheet_matches']

ZIMBABWE RECORDS IN CRICSHEET:


,team,normalized_name,player_name,source_player_id,cricsheet_first_date,cricsheet_last_date,cricsheet_matches
1910,Zimbabwe,adrcampbell,ADR Campbell,134ba8ae,2003-03-12,2003-03-12,1
1911,Zimbabwe,aflower,A Flower,35aa26af,2003-02-10,2003-03-15,7
1912,Zimbabwe,agcremer,AG Cremer,9f0270e0,2009-01-31,2026-09-20,68
1913,Zimbabwe,ajireland,AJ Ireland,a7c7802a,2005-08-29,2007-03-19,13
1914,Zimbabwe,amaregwede,A Maregwede,340db614,2003-11-30,2005-02-27,8
...,...,...,...,...,...,...,...
2020,Zimbabwe,vmnyauchi,VM Nyauchi,1bd16445,2022-08-05,2023-01-21,10
2021,Zimbabwe,vsibanda,V Sibanda,0fdabe05,2003-11-23,2016-06-15,96
2022,Zimbabwe,wmadhevere,W Madhevere,a879b92c,2020-03-01,2026-10-17,42
2023,Zimbabwe,wmwayenga,W Mwayenga,96dced3e,2004-05-27,2004-05-29,2


In [49]:
# Recreate unresolved Crickrida players from player history

identity_map = player_identity_map_updated.copy()

# Only genuinely shared mappings
shared_rows = identity_map[
    identity_map["player_id"].notna() &
    identity_map["cricsheet_name"].notna() &
    identity_map["crickrida_name"].notna() &
    (identity_map["match_status"] != "crickrida_only")
]

shared_names = set(
    shared_rows["crickrida_name"]
    .dropna()
    .apply(normalize_identity)
)

# Verified Crickrida-only identities
source_only_rows = identity_map[
    identity_map["match_status"] == "crickrida_only"
]

source_only_names = set(
    source_only_rows["crickrida_name"]
    .dropna()
    .apply(normalize_identity)
)

# Include previously verified Ireland-only players
source_only_names.update(
    normalize_identity(name)
    for name in [
        "James Hall",
        "James Cameron-Dow",
        "Ben Calitz"
    ]
)

# Classify Crickrida history
cr_history = cr_player_history.copy()

cr_history["normalized_check"] = (
    cr_history["player_name"].apply(normalize_identity)
)

cr_history["identity_status"] = cr_history["normalized_check"].apply(
    lambda name:
        "Shared" if name in shared_names
        else "Crickrida-only" if name in source_only_names
        else "Unresolved"
)

unresolved_crickrida_players = cr_history[
    cr_history["identity_status"] == "Unresolved"
].copy()

print("Unresolved players:", len(unresolved_crickrida_players))

Unresolved players: 260


In [50]:
z_cr = unresolved_crickrida_players[
    unresolved_crickrida_players["team"].str.casefold() == "zimbabwe"
].copy()

z_cs = cs_player_history[
    cs_player_history["team"].str.casefold() == "zimbabwe"
].copy()

print("Zimbabwe unresolved:", len(z_cr))
print("Zimbabwe Cricsheet players:", len(z_cs))

display(z_cr)

Zimbabwe unresolved: 51
Zimbabwe Cricsheet players: 115


,team,normalized_name,player_name,crickrida_first_date,crickrida_last_date,crickrida_matches,normalized_check,identity_status
402,Zimbabwe,bencurran,Ben Curran,2024-12-17,2026-10-19,4,bencurran,Unresolved
403,Zimbabwe,blessingmuzarabani,Blessing Muzarabani,2018-02-11,2026-10-23,10,blessingmuzarabani,Unresolved
404,Zimbabwe,bradevans,Brad Evans,2026-10-19,2026-10-23,2,bradevans,Unresolved
405,Zimbabwe,brendantaylor,Brendan Taylor,2014-07-18,2026-10-23,12,brendantaylor,Unresolved
406,Zimbabwe,brianbennett,Brian Bennett,2024-12-17,2026-10-23,5,brianbennett,Unresolved
407,Zimbabwe,brianvitori,Brian Vitori,2018-02-09,2018-03-06,4,brianvitori,Unresolved
408,Zimbabwe,cephaszhuwao,Cephas Zhuwao,2018-03-06,2018-03-06,1,cephaszhuwao,Unresolved
409,Zimbabwe,chamuchibhabha,Chamu Chibhabha,2015-10-16,2016-01-06,9,chamuchibhabha,Unresolved
410,Zimbabwe,chrismpofu,Chris Mpofu,2017-02-16,2017-02-26,5,chrismpofu,Unresolved
411,Zimbabwe,craigervine,Craig Ervine,2015-10-16,2024-12-21,24,craigervine,Unresolved


In [51]:
import re
import pandas as pd
from difflib import SequenceMatcher

def clean_name(name):
    return re.sub(r"[^a-z]", "", str(name).lower())

def get_surname(name):
    parts = str(name).lower().split()
    return clean_name(parts[-1]) if parts else ""

def get_first_initial(name):
    match = re.search(r"[a-z]", str(name).lower())
    return match.group(0) if match else ""

candidate_rows = []

for _, cr in z_cr.iterrows():

    cr_name = cr["player_name"]
    cr_surname = get_surname(cr_name)
    cr_initial = get_first_initial(cr_name)

    cr_start = pd.to_datetime(cr["crickrida_first_date"])
    cr_end = pd.to_datetime(cr["crickrida_last_date"])

    candidates = []

    for _, cs in z_cs.iterrows():

        cs_name = cs["player_name"]
        cs_surname = get_surname(cs_name)
        cs_initial = get_first_initial(cs_name)

        full_similarity = SequenceMatcher(
            None, clean_name(cr_name), clean_name(cs_name)
        ).ratio()

        surname_similarity = SequenceMatcher(
            None, cr_surname, cs_surname
        ).ratio()

        surname_exact = cr_surname == cs_surname
        initial_match = cr_initial == cs_initial

        cs_start = pd.to_datetime(cs["cricsheet_first_date"])
        cs_end = pd.to_datetime(cs["cricsheet_last_date"])

        date_overlap = (
            cr_start <= cs_end and cs_start <= cr_end
        )

        candidates.append({
            "crickrida_name": cr_name,
            "crickrida_matches": cr["crickrida_matches"],
            "cricsheet_name": cs_name,
            "source_player_id": cs["source_player_id"],
            "surname_exact": surname_exact,
            "first_initial_match": initial_match,
            "full_similarity": round(full_similarity * 100, 1),
            "surname_similarity": round(surname_similarity * 100, 1),
            "career_date_overlap": date_overlap,
            "crickrida_dates": (
                f"{cr_start.date()} to {cr_end.date()}"
            ),
            "cricsheet_dates": (
                f"{cs_start.date()} to {cs_end.date()}"
            )
        })

    candidates.sort(
        key=lambda x: (
            x["surname_exact"],
            x["first_initial_match"],
            x["surname_similarity"],
            x["full_similarity"],
            x["career_date_overlap"]
        ),
        reverse=True
    )

    # Retain credible candidates only
    credible = [
        c for c in candidates
        if c["surname_similarity"] >= 60
        or c["full_similarity"] >= 50
    ]

    for candidate in credible[:3]:
        candidate_rows.append(candidate)

zimbabwe_candidates = pd.DataFrame(candidate_rows)

display(
    zimbabwe_candidates.sort_values(
        ["crickrida_name", "surname_exact", "full_similarity"],
        ascending=[True, False, False]
    ).reset_index(drop=True)
)

print("Candidate rows:", len(zimbabwe_candidates))
print("Unresolved Zimbabwe players:", len(z_cr))

,crickrida_name,crickrida_matches,cricsheet_name,source_player_id,surname_exact,first_initial_match,full_similarity,surname_similarity,career_date_overlap,crickrida_dates,cricsheet_dates
0,Ben Curran,4,BJ Curran,1786a21b,True,True,82.4,100.0,True,2024-12-17 to 2026-10-19,2025-02-14 to 2026-10-17
1,Ben Curran,4,B Evans,e7e86505,False,True,53.3,36.4,True,2024-12-17 to 2026-10-19,2022-08-07 to 2026-10-17
2,Ben Curran,4,B Muzarabani,449f4e3c,False,True,50.0,50.0,True,2024-12-17 to 2026-10-19,2018-01-15 to 2026-10-17
3,Blessing Muzarabani,10,B Muzarabani,449f4e3c,True,True,75.9,100.0,True,2018-02-11 to 2026-10-23,2018-01-15 to 2026-10-17
4,Blessing Muzarabani,10,T Muzarabani,a945bb28,True,False,69.0,100.0,False,2018-02-11 to 2026-10-23,2015-10-13 to 2016-06-13
...,...,...,...,...,...,...,...,...,...,...,...
110,Wellington Masakadza,10,WP Masakadza,4bc0a378,True,True,66.7,100.0,True,2015-10-16 to 2026-10-23,2015-10-09 to 2026-10-17
111,Wellington Masakadza,10,SW Masakadza,2182013e,True,False,66.7,100.0,False,2015-10-16 to 2026-10-23,2010-03-04 to 2014-08-17
112,Wellington Masakadza,10,H Masakadza,31fbf891,True,False,62.1,100.0,True,2015-10-16 to 2026-10-23,2004-12-01 to 2019-07-07
113,Wessly Madhevere,5,W Madhevere,a879b92c,True,True,80.0,100.0,True,2022-06-04 to 2026-10-23,2020-03-01 to 2026-10-17


Candidate rows: 115
Unresolved Zimbabwe players: 51


In [52]:
strong_zimbabwe_candidates = zimbabwe_candidates[
    (zimbabwe_candidates["surname_exact"] == True) &
    (zimbabwe_candidates["first_initial_match"] == True) &
    (zimbabwe_candidates["career_date_overlap"] == True)
].copy()

# Count candidates for each Crickrida player
strong_zimbabwe_candidates["candidate_count"] = (
    strong_zimbabwe_candidates
    .groupby("crickrida_name")["cricsheet_name"]
    .transform("count")
)

display(
    strong_zimbabwe_candidates[
        [
            "crickrida_name",
            "cricsheet_name",
            "source_player_id",
            "full_similarity",
            "career_date_overlap",
            "candidate_count",
            "crickrida_dates",
            "cricsheet_dates"
        ]
    ].sort_values(
        ["candidate_count", "crickrida_name"],
        ascending=[False, True]
    ).reset_index(drop=True)
)

print("Strong candidate pairs:", len(strong_zimbabwe_candidates))
print("Players with candidates:",
      strong_zimbabwe_candidates["crickrida_name"].nunique())

,crickrida_name,cricsheet_name,source_player_id,full_similarity,career_date_overlap,candidate_count,crickrida_dates,cricsheet_dates
0,Ben Curran,BJ Curran,1786a21b,82.4,True,1,2024-12-17 to 2026-10-19,2025-02-14 to 2026-10-17
1,Blessing Muzarabani,B Muzarabani,449f4e3c,75.9,True,1,2018-02-11 to 2026-10-23,2018-01-15 to 2026-10-17
2,Brendan Taylor,BRM Taylor,53597be1,72.7,True,1,2014-07-18 to 2026-10-23,2004-04-20 to 2026-10-17
3,Brian Bennett,BJ Bennett,eaa36b2a,76.2,True,1,2024-12-17 to 2026-10-23,2024-11-24 to 2026-10-17
4,Cephas Zhuwao,C Zhuwao,3e51db81,73.7,True,1,2018-03-06 to 2018-03-06,2008-10-17 to 2018-10-26
5,Chamu Chibhabha,CJ Chibhabha,f32dbc84,80.0,True,1,2015-10-16 to 2016-01-06,2005-08-31 to 2023-01-23
6,Chris Mpofu,CB Mpofu,49691d7a,70.6,True,1,2017-02-16 to 2017-02-26,2004-11-28 to 2020-03-01
7,Craig Ervine,CR Ervine,c96311d4,84.2,True,1,2015-10-16 to 2024-12-21,2010-05-28 to 2026-10-17
8,Dion Myers,D Myers,d96165b5,80.0,True,1,2022-06-09 to 2024-12-19,2021-07-16 to 2024-11-28
9,Donald Tiripano,DT Tiripano,dbfb8218,75.0,True,1,2014-07-18 to 2022-06-06,2014-08-31 to 2021-07-20


Strong candidate pairs: 39
Players with candidates: 39


In [53]:
# Players with at least one strong candidate
strong_candidate_names = set(
    strong_zimbabwe_candidates["crickrida_name"]
)

# Remaining Zimbabwe players
zimbabwe_no_candidate = z_cr[
    ~z_cr["player_name"].isin(strong_candidate_names)
].copy()

print("Players requiring further investigation:",
      len(zimbabwe_no_candidate))

display(
    zimbabwe_no_candidate[
        [
            "player_name",
            "crickrida_first_date",
            "crickrida_last_date",
            "crickrida_matches"
        ]
    ].reset_index(drop=True)
)

Players requiring further investigation: 12


,player_name,crickrida_first_date,crickrida_last_date,crickrida_matches
0,Brad Evans,2026-10-19,2026-10-23,2
1,Brian Vitori,2018-02-09,2018-03-06,4
2,Ernest Masuku,2026-10-19,2026-10-19,1
3,Graeme Cremer,2015-12-25,2018-03-06,16
4,Innocent Kaia,2022-06-04,2022-06-09,3
5,John Nyumbu,2015-10-16,2015-10-18,2
6,Joylord Gumbie,2024-12-21,2024-12-21,1
7,Natsai M'shangwe,2014-07-22,2014-07-22,1
8,Tafadzwa Tsiga,2026-10-19,2026-10-23,2
9,Tanaka Chivanga,2022-06-04,2022-06-09,3


In [54]:
zimbabwe_mapping_review = strong_zimbabwe_candidates[
    [
        "crickrida_name",
        "cricsheet_name",
        "source_player_id",
        "full_similarity",
        "career_date_overlap",
        "crickrida_dates",
        "cricsheet_dates"
    ]
].copy()

zimbabwe_mapping_review["review_status"] = "candidate_pending_verification"

display(zimbabwe_mapping_review)

,crickrida_name,cricsheet_name,source_player_id,full_similarity,career_date_overlap,crickrida_dates,cricsheet_dates,review_status
0,Ben Curran,BJ Curran,1786a21b,82.4,True,2024-12-17 to 2026-10-19,2025-02-14 to 2026-10-17,candidate_pending_verification
3,Blessing Muzarabani,B Muzarabani,449f4e3c,75.9,True,2018-02-11 to 2026-10-23,2018-01-15 to 2026-10-17,candidate_pending_verification
9,Brendan Taylor,BRM Taylor,53597be1,72.7,True,2014-07-18 to 2026-10-23,2004-04-20 to 2026-10-17,candidate_pending_verification
11,Brian Bennett,BJ Bennett,eaa36b2a,76.2,True,2024-12-17 to 2026-10-23,2024-11-24 to 2026-10-17,candidate_pending_verification
16,Cephas Zhuwao,C Zhuwao,3e51db81,73.7,True,2018-03-06 to 2018-03-06,2008-10-17 to 2018-10-26,candidate_pending_verification
18,Chamu Chibhabha,CJ Chibhabha,f32dbc84,80.0,True,2015-10-16 to 2016-01-06,2005-08-31 to 2023-01-23,candidate_pending_verification
20,Chris Mpofu,CB Mpofu,49691d7a,70.6,True,2017-02-16 to 2017-02-26,2004-11-28 to 2020-03-01,candidate_pending_verification
23,Craig Ervine,CR Ervine,c96311d4,84.2,True,2015-10-16 to 2024-12-21,2010-05-28 to 2026-10-17,candidate_pending_verification
26,Dion Myers,D Myers,d96165b5,80.0,True,2022-06-09 to 2024-12-19,2021-07-16 to 2024-11-28,candidate_pending_verification
27,Donald Tiripano,DT Tiripano,dbfb8218,75.0,True,2014-07-18 to 2022-06-06,2014-08-31 to 2021-07-20,candidate_pending_verification


In [55]:
# Inspect all generated candidates for the 12 remaining players

remaining_names = set(zimbabwe_no_candidate["player_name"])

remaining_candidates = zimbabwe_candidates[
    zimbabwe_candidates["crickrida_name"].isin(remaining_names)
].copy()

display(
    remaining_candidates[
        [
            "crickrida_name",
            "cricsheet_name",
            "source_player_id",
            "surname_exact",
            "first_initial_match",
            "full_similarity",
            "surname_similarity",
            "career_date_overlap",
            "crickrida_dates",
            "cricsheet_dates"
        ]
    ].sort_values(
        ["crickrida_name", "full_similarity"],
        ascending=[True, False]
    ).reset_index(drop=True)
)

# Identify players with no generated candidates
players_with_candidates = set(
    remaining_candidates["crickrida_name"]
)

no_candidate_names = remaining_names - players_with_candidates

print("\nPlayers with candidates:", len(players_with_candidates))
print("Players with no candidates:", len(no_candidate_names))
print(sorted(no_candidate_names))

,crickrida_name,cricsheet_name,source_player_id,surname_exact,first_initial_match,full_similarity,surname_similarity,career_date_overlap,crickrida_dates,cricsheet_dates
0,Brad Evans,B Evans,e7e86505,True,True,80.0,100.0,False,2026-10-19 to 2026-10-23,2022-08-07 to 2026-10-17
1,Brad Evans,B Mavuta,dadb9703,False,True,50.0,36.4,False,2026-10-19 to 2026-10-23,2018-09-30 to 2024-11-26
2,Brad Evans,B Muzarabani,449f4e3c,False,True,50.0,26.7,False,2026-10-19 to 2026-10-23,2018-01-15 to 2026-10-17
3,Brian Vitori,BV Vitori,37137c8b,True,True,73.7,100.0,False,2018-02-09 to 2018-03-06,2011-08-12 to 2016-11-27
4,Brian Vitori,BRM Taylor,53597be1,False,True,50.0,50.0,True,2018-02-09 to 2018-03-06,2004-04-20 to 2026-10-17
5,Ernest Masuku,E Masuku,7e6e7274,True,True,73.7,100.0,False,2026-10-19 to 2026-10-19,2025-08-31 to 2026-09-20
6,Graeme Cremer,AG Cremer,9f0270e0,True,False,70.0,100.0,True,2015-12-25 to 2018-03-06,2009-01-31 to 2026-09-20
7,Innocent Kaia,I Kaia,5ad8c974,True,True,58.8,100.0,False,2022-06-04 to 2022-06-09,2022-08-05 to 2026-09-20
8,Innocent Kaia,T Kaitano,df7ed601,False,False,50.0,72.7,True,2022-06-04 to 2022-06-09,2022-01-16 to 2024-01-11
9,Innocent Kaia,ML Nkala,be8eec9c,False,False,42.1,66.7,False,2022-06-04 to 2022-06-09,2004-04-20 to 2006-12-09



Players with candidates: 11
Players with no candidates: 1
['Tinotenda Maposa']


In [56]:
# Collect the 11 candidate mappings
zimbabwe_candidate_ids = (
    remaining_candidates
    .sort_values("full_similarity", ascending=False)
    .drop_duplicates("crickrida_name")
    [["crickrida_name", "cricsheet_name", "source_player_id"]]
    .copy()
)

# Check whether source IDs already exist in the identity map
existing_id_check = player_identity_map_updated[
    player_identity_map_updated["player_id"].astype(str).isin(
        zimbabwe_candidate_ids["source_player_id"].astype(str)
    )
][[
    "cricketiq_player_id",
    "player_id",
    "cricsheet_name",
    "crickrida_name",
    "match_status"
]]

print("CANDIDATE MAPPINGS")
display(zimbabwe_candidate_ids.sort_values("crickrida_name"))

print("\nEXISTING IDENTITY-MAP RECORDS USING THESE IDS")
display(existing_id_check)

print("\nCANDIDATE SOURCE IDs:", len(zimbabwe_candidate_ids))
print("EXISTING ID MATCHES:", len(existing_id_check))

CANDIDATE MAPPINGS


,crickrida_name,cricsheet_name,source_player_id
6,Brad Evans,B Evans,e7e86505
14,Brian Vitori,BV Vitori,37137c8b
30,Ernest Masuku,E Masuku,7e6e7274
31,Graeme Cremer,AG Cremer,9f0270e0
35,Innocent Kaia,I Kaia,5ad8c974
38,John Nyumbu,J Nyumbu,024a173d
40,Joylord Gumbie,J Gumbie,c8d74c21
53,Natsai M'shangwe,N M'shangwe,dd468277
85,Tafadzwa Tsiga,TE Tsiga,a1dcb5b9
86,Tanaka Chivanga,TL Chivanga,61fb296e



EXISTING IDENTITY-MAP RECORDS USING THESE IDS


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,match_status



CANDIDATE SOURCE IDs: 11
EXISTING ID MATCHES: 0


In [57]:
# Combine the two candidate batches
zimbabwe_shared_candidates = pd.concat(
    [
        zimbabwe_mapping_review[
            ["crickrida_name", "cricsheet_name", "source_player_id"]
        ],
        zimbabwe_candidate_ids[
            ["crickrida_name", "cricsheet_name", "source_player_id"]
        ]
    ],
    ignore_index=True
).drop_duplicates()

# Duplicate checks
print("TOTAL CANDIDATES:", len(zimbabwe_shared_candidates))
print(
    "Duplicate Crickrida names:",
    zimbabwe_shared_candidates["crickrida_name"].duplicated().sum()
)
print(
    "Duplicate Cricsheet IDs:",
    zimbabwe_shared_candidates["source_player_id"].duplicated().sum()
)

# Check existing player IDs
existing_ids = set(
    player_identity_map_updated["player_id"].dropna().astype(str)
)

candidate_ids = set(
    zimbabwe_shared_candidates["source_player_id"].astype(str)
)

print("Existing ID conflicts:", len(existing_ids & candidate_ids))

# Check existing Crickrida names
existing_names = set(
    player_identity_map_updated["crickrida_name"]
    .dropna()
    .apply(normalize_identity)
)

candidate_names = set(
    zimbabwe_shared_candidates["crickrida_name"]
    .apply(normalize_identity)
)

print("Existing Crickrida-name conflicts:",
      len(existing_names & candidate_names))

display(
    zimbabwe_shared_candidates.sort_values(
        "crickrida_name"
    ).reset_index(drop=True)
)

TOTAL CANDIDATES: 50
Duplicate Crickrida names: 0
Duplicate Cricsheet IDs: 0
Existing ID conflicts: 0
Existing Crickrida-name conflicts: 0


,crickrida_name,cricsheet_name,source_player_id
0,Ben Curran,BJ Curran,1786a21b
1,Blessing Muzarabani,B Muzarabani,449f4e3c
2,Brad Evans,B Evans,e7e86505
3,Brendan Taylor,BRM Taylor,53597be1
4,Brian Bennett,BJ Bennett,eaa36b2a
5,Brian Vitori,BV Vitori,37137c8b
6,Cephas Zhuwao,C Zhuwao,3e51db81
7,Chamu Chibhabha,CJ Chibhabha,f32dbc84
8,Chris Mpofu,CB Mpofu,49691d7a
9,Craig Ervine,CR Ervine,c96311d4


In [58]:
maposa_master = crickrida_only_master[
    crickrida_only_master["player_name"]
    .apply(normalize_identity)
    == normalize_identity("Tinotenda Maposa")
].copy()

print("Maposa records:", len(maposa_master))
print("Available columns:", maposa_master.columns.tolist())

display(maposa_master)

Maposa records: 1
Available columns: ['player_name', 'normalized_name', 'cricketiq_player_id', 'source', 'match_status']


,player_name,normalized_name,cricketiq_player_id,source,match_status
2405,Tinotenda Maposa,tinotendamaposa,CR_tinotendamaposa,Crickrida,source_only


In [59]:
display(
    player_identity_map_updated[
        player_identity_map_updated["crickrida_name"]
        .astype(str)
        .apply(normalize_identity)
        == normalize_identity("Tinotenda Maposa")
    ]
)

,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,normalized_name,match_status,crickrida_player_name,cricsheet_player_name,cricsheet_player_id,mapping_method,confidence,notes


In [60]:
# Backup current identity map
player_identity_map_updated_backup = (
    player_identity_map_updated.copy(deep=True)
)

# Prepare verified Crickrida-only record
maposa_record = {
    "cricketiq_player_id": "CR_tinotendamaposa",
    "player_id": None,
    "cricsheet_name": None,
    "crickrida_name": "Tinotenda Maposa",
    "normalized_name": "tinotendamaposa",
    "match_status": "crickrida_only",
    "crickrida_player_name": "Tinotenda Maposa",
    "cricsheet_player_name": None,
    "cricsheet_player_id": None,
    "mapping_method": "verified_source_coverage",
    "confidence": "high",
    "notes": (
        "Zimbabwe player; one Crickrida match on 2024-12-19 "
        "against Afghanistan. Afghanistan is excluded from "
        "the Cricsheet dataset."
    )
}

# Prevent duplicate insertion
if not (
    player_identity_map_updated["cricketiq_player_id"]
    .astype(str)
    .eq("CR_tinotendamaposa")
    .any()
):
    player_identity_map_updated = pd.concat(
        [
            player_identity_map_updated,
            pd.DataFrame([maposa_record])
        ],
        ignore_index=True
    )

print("Updated identity map:",
      player_identity_map_updated.shape)

display(
    player_identity_map_updated[
        player_identity_map_updated["cricketiq_player_id"]
        == "CR_tinotendamaposa"
    ].T
)

Updated identity map: (215, 12)


,214
cricketiq_player_id,CR_tinotendamaposa
player_id,None
cricsheet_name,None
crickrida_name,Tinotenda Maposa
normalized_name,tinotendamaposa
match_status,crickrida_only
crickrida_player_name,Tinotenda Maposa
cricsheet_player_name,None
cricsheet_player_id,None
mapping_method,verified_source_coverage


In [61]:
zimbabwe_shared_review = zimbabwe_shared_candidates.copy()

zimbabwe_shared_review["normalized_name"] = (
    zimbabwe_shared_review["crickrida_name"]
    .apply(normalize_identity)
)

zimbabwe_shared_review["review_status"] = (
    "pending_identity_confirmation"
)

zimbabwe_shared_review["proposed_cricketiq_player_id"] = (
    "CRIQ_" + zimbabwe_shared_review["source_player_id"].astype(str)
)

display(
    zimbabwe_shared_review.sort_values("crickrida_name")
)

print("Candidates awaiting confirmation:",
      len(zimbabwe_shared_review))

,crickrida_name,cricsheet_name,source_player_id,normalized_name,review_status,proposed_cricketiq_player_id
0,Ben Curran,BJ Curran,1786a21b,bencurran,pending_identity_confirmation,CRIQ_1786a21b
1,Blessing Muzarabani,B Muzarabani,449f4e3c,blessingmuzarabani,pending_identity_confirmation,CRIQ_449f4e3c
40,Brad Evans,B Evans,e7e86505,bradevans,pending_identity_confirmation,CRIQ_e7e86505
2,Brendan Taylor,BRM Taylor,53597be1,brendantaylor,pending_identity_confirmation,CRIQ_53597be1
3,Brian Bennett,BJ Bennett,eaa36b2a,brianbennett,pending_identity_confirmation,CRIQ_eaa36b2a
44,Brian Vitori,BV Vitori,37137c8b,brianvitori,pending_identity_confirmation,CRIQ_37137c8b
4,Cephas Zhuwao,C Zhuwao,3e51db81,cephaszhuwao,pending_identity_confirmation,CRIQ_3e51db81
5,Chamu Chibhabha,CJ Chibhabha,f32dbc84,chamuchibhabha,pending_identity_confirmation,CRIQ_f32dbc84
6,Chris Mpofu,CB Mpofu,49691d7a,chrismpofu,pending_identity_confirmation,CRIQ_49691d7a
7,Craig Ervine,CR Ervine,c96311d4,craigervine,pending_identity_confirmation,CRIQ_c96311d4


Candidates awaiting confirmation: 50


In [62]:
# ==========================================
# ZIMBABWE: COMMIT 50 SHARED IDENTITIES
# ==========================================

# 1. Create a backup immediately before insertion
player_identity_map_updated_backup_before_zimbabwe_commit = (
    player_identity_map_updated.copy(deep=True)
)

# 2. Prepare candidate table
zimbabwe_to_add = zimbabwe_shared_candidates.copy()

zimbabwe_to_add["normalized_name"] = (
    zimbabwe_to_add["crickrida_name"].apply(normalize_identity)
)

zimbabwe_to_add["cricketiq_player_id"] = (
    "CRIQ_" + zimbabwe_to_add["source_player_id"].astype(str)
)

# 3. Pre-insertion validation
assert len(zimbabwe_to_add) == 50, "Expected exactly 50 candidates."

assert zimbabwe_to_add["crickrida_name"].is_unique, (
    "Duplicate Crickrida names found."
)

assert zimbabwe_to_add["source_player_id"].is_unique, (
    "Duplicate Cricsheet IDs found."
)

existing_ids = set(
    player_identity_map_updated["player_id"].dropna().astype(str)
)

existing_cricketiq_ids = set(
    player_identity_map_updated["cricketiq_player_id"].dropna().astype(str)
)

existing_names = set(
    player_identity_map_updated["crickrida_name"]
    .dropna()
    .apply(normalize_identity)
)

assert not (
    set(zimbabwe_to_add["source_player_id"].astype(str)) & existing_ids
), "Existing player ID conflict."

assert not (
    set(zimbabwe_to_add["cricketiq_player_id"]) & existing_cricketiq_ids
), "Existing CricketIQ ID conflict."

assert not (
    set(zimbabwe_to_add["normalized_name"]) & existing_names
), "Existing Crickrida name conflict."

# 4. Build records using the existing identity-map schema
new_records = []

for _, row in zimbabwe_to_add.iterrows():

    new_records.append({
        "cricketiq_player_id": row["cricketiq_player_id"],
        "player_id": row["source_player_id"],
        "cricsheet_name": row["cricsheet_name"],
        "crickrida_name": row["crickrida_name"],
        "normalized_name": row["normalized_name"],
        "match_status": "verified_alias",
        "crickrida_player_name": None,
        "cricsheet_player_name": None,
        "cricsheet_player_id": None,
        "mapping_method": "verified_zimbabwe_name_and_career_review",
        "confidence": "high",
        "notes": (
            "Zimbabwe identity matched across Crickrida and Cricsheet. "
            "Reviewed using name/initial variations, surname and "
            "available career-history evidence."
        )
    })

new_records_df = pd.DataFrame(
    new_records,
    columns=player_identity_map_updated.columns
)

# 5. Append without replacing existing records
player_identity_map_updated = pd.concat(
    [player_identity_map_updated, new_records_df],
    ignore_index=True
)

# 6. Post-insertion validation
print("New records added:", len(new_records_df))
print("Updated identity map:", player_identity_map_updated.shape)

print(
    "Duplicate CricketIQ IDs:",
    player_identity_map_updated["cricketiq_player_id"].duplicated().sum()
)

print(
    "Duplicate non-null player IDs:",
    player_identity_map_updated.loc[
        player_identity_map_updated["player_id"].notna(),
        "player_id"
    ].duplicated().sum()
)

# Confirm Maposa is still present
print(
    "\nMaposa preserved:",
    (
        player_identity_map_updated["cricketiq_player_id"]
        == "CR_tinotendamaposa"
    ).sum()
)

display(
    player_identity_map_updated.tail(50)[
        [
            "cricketiq_player_id",
            "player_id",
            "cricsheet_name",
            "crickrida_name",
            "match_status",
            "confidence"
        ]
    ]
)

New records added: 50
Updated identity map: (265, 12)
Duplicate CricketIQ IDs: 0
Duplicate non-null player IDs: 0

Maposa preserved: 1


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,match_status,confidence
215,CRIQ_1786a21b,1786a21b,BJ Curran,Ben Curran,verified_alias,high
216,CRIQ_449f4e3c,449f4e3c,B Muzarabani,Blessing Muzarabani,verified_alias,high
217,CRIQ_53597be1,53597be1,BRM Taylor,Brendan Taylor,verified_alias,high
218,CRIQ_eaa36b2a,eaa36b2a,BJ Bennett,Brian Bennett,verified_alias,high
219,CRIQ_3e51db81,3e51db81,C Zhuwao,Cephas Zhuwao,verified_alias,high
220,CRIQ_f32dbc84,f32dbc84,CJ Chibhabha,Chamu Chibhabha,verified_alias,high
221,CRIQ_49691d7a,49691d7a,CB Mpofu,Chris Mpofu,verified_alias,high
222,CRIQ_c96311d4,c96311d4,CR Ervine,Craig Ervine,verified_alias,high
223,CRIQ_d96165b5,d96165b5,D Myers,Dion Myers,verified_alias,high
224,CRIQ_dbfb8218,dbfb8218,DT Tiripano,Donald Tiripano,verified_alias,high


In [63]:
# Rebuild shared identities from the current map
identity_map = player_identity_map_updated.copy()

shared_rows = identity_map[
    identity_map["player_id"].notna() &
    identity_map["cricsheet_name"].notna() &
    identity_map["crickrida_name"].notna() &
    (identity_map["match_status"] != "crickrida_only")
]

shared_names = set(
    shared_rows["crickrida_name"]
    .dropna()
    .apply(normalize_identity)
)

source_only_rows = identity_map[
    identity_map["match_status"] == "crickrida_only"
]

source_only_names = set(
    source_only_rows["crickrida_name"]
    .dropna()
    .apply(normalize_identity)
)

source_only_names.update(
    normalize_identity(name)
    for name in [
        "James Hall",
        "James Cameron-Dow",
        "Ben Calitz"
    ]
)

# Reclassify all Crickrida history
cr_history = cr_player_history.copy()

cr_history["normalized_check"] = (
    cr_history["player_name"].apply(normalize_identity)
)

cr_history["identity_status"] = cr_history["normalized_check"].apply(
    lambda name:
        "Shared" if name in shared_names
        else "Crickrida-only" if name in source_only_names
        else "Unresolved"
)

# Zimbabwe audit
zimbabwe_final = cr_history[
    cr_history["team"].str.casefold() == "zimbabwe"
].copy()

print("Zimbabwe total:", len(zimbabwe_final))
print("\nIdentity distribution:")
print(zimbabwe_final["identity_status"].value_counts())

print("\nUnresolved Zimbabwe players:")
display(
    zimbabwe_final[
        zimbabwe_final["identity_status"] == "Unresolved"
    ]
)

Zimbabwe total: 52

Identity distribution:
identity_status
Shared            51
Crickrida-only     1
Name: count, dtype: int64

Unresolved Zimbabwe players:


,team,normalized_name,player_name,crickrida_first_date,crickrida_last_date,crickrida_matches,normalized_check,identity_status


In [64]:
# Check Sri Lanka coverage in Cricsheet

sl_cs = cs_player_history[
    cs_player_history["team"].str.casefold() == "sri lanka"
].copy()

sl_cr = unresolved_crickrida_players[
    unresolved_crickrida_players["team"].str.casefold() == "sri lanka"
].copy()

print("Cricsheet Sri Lankan players:", len(sl_cs))
print("Unresolved Crickrida Sri Lankan players:", len(sl_cr))

print("\nCRICSHEET HISTORY:")
display(sl_cs)

print("\nCRICKRIDA UNRESOLVED HISTORY:")
display(sl_cr)

Cricsheet Sri Lankan players: 133
Unresolved Crickrida Sri Lankan players: 40

CRICSHEET HISTORY:


,team,normalized_name,player_name,source_player_id,cricsheet_first_date,cricsheet_last_date,cricsheet_matches
1475,Sri Lanka,adananjaya,A Dananjaya,5b16a806,2017-06-30,2024-08-04,38
1476,Sri Lanka,admathews,AD Mathews,896d78ad,2008-11-30,2023-11-09,210
1477,Sri Lanka,akperera,AK Perera,c8ad0749,2013-07-26,2019-10-02,5
1478,Sri Lanka,amfernando,AM Fernando,de3d549a,2017-07-08,2026-10-22,33
1479,Sri Lanka,bawmendis,BAW Mendis,ea0cdc12,2008-04-10,2015-12-26,79
...,...,...,...,...,...,...,...
1603,Sri Lanka,wcaganegama,WCA Ganegama,42d1b920,2006-01-26,2006-01-29,2
1604,Sri Lanka,wiafernando,WIA Fernando,9550470d,2016-08-31,2026-10-22,49
1605,Sri Lanka,wpujcvaas,WPUJC Vaas,bcce309e,2002-06-27,2008-08-27,95
1606,Sri Lanka,wsjayantha,WS Jayantha,858b6611,2004-02-27,2004-10-16,9



CRICKRIDA UNRESOLVED HISTORY:


,team,normalized_name,player_name,crickrida_first_date,crickrida_last_date,crickrida_matches,normalized_check,identity_status
335,Sri Lanka,ajanthamendis,Ajantha Mendis,2014-03-03,2014-03-03,1,ajanthamendis,Unresolved
336,Sri Lanka,akiladananjaya,Akila Dananjaya,2018-09-17,2024-02-14,2,akiladananjaya,Unresolved
337,Sri Lanka,angelomathews,Angelo Mathews,2014-03-03,2023-10-30,6,angelomathews,Unresolved
338,Sri Lanka,asithafernando,Asitha Fernando,2022-11-30,2024-02-14,3,asithafernando,Unresolved
339,Sri Lanka,avishkafernando,Avishka Fernando,2024-02-09,2024-02-14,3,avishkafernando,Unresolved
340,Sri Lanka,charithasalanka,Charith Asalanka,2022-11-25,2024-02-14,9,charithasalanka,Unresolved
341,Sri Lanka,chaturangadesilva,Chaturanga de Silva,2014-03-03,2014-03-03,1,chaturangadesilva,Unresolved
342,Sri Lanka,dasunshanaka,Dasun Shanaka,2018-09-17,2023-09-05,6,dasunshanaka,Unresolved
343,Sri Lanka,dhananjayadesilva,Dhananjaya de Silva,2018-09-17,2023-10-30,9,dhananjayadesilva,Unresolved
344,Sri Lanka,dhananjayalakshan,Dhananjaya Lakshan,2022-11-25,2022-11-27,2,dhananjayalakshan,Unresolved


In [65]:
import pandas as pd

# Work on copies
cr = sl_cr.copy()
cs = sl_cs.copy()

# Convert dates
for col in ["crickrida_first_date", "crickrida_last_date"]:
    cr[col] = pd.to_datetime(cr[col], errors="coerce")

for col in ["cricsheet_first_date", "cricsheet_last_date"]:
    cs[col] = pd.to_datetime(cs[col], errors="coerce")


def get_surname(name):
    parts = str(name).strip().split()
    return parts[-1].casefold() if parts else ""


def get_first_initial(name):
    parts = str(name).strip().split()
    return parts[0][0].casefold() if parts else ""


def get_cricsheet_initials(name):
    parts = str(name).strip().split()
    if len(parts) < 2:
        return ""
    return "".join(
        word[0].casefold()
        for word in parts[:-1]
        if word
    )


candidates = []

for _, cr_row in cr.iterrows():

    cr_surname = get_surname(cr_row["player_name"])
    cr_initial = get_first_initial(cr_row["player_name"])

    for _, cs_row in cs.iterrows():

        cs_surname = get_surname(cs_row["player_name"])

        if cr_surname != cs_surname:
            continue

        cs_initials = get_cricsheet_initials(cs_row["player_name"])

        initial_match = cr_initial in cs_initials

        date_overlap = (
            pd.notna(cr_row["crickrida_first_date"]) and
            pd.notna(cr_row["crickrida_last_date"]) and
            pd.notna(cs_row["cricsheet_first_date"]) and
            pd.notna(cs_row["cricsheet_last_date"]) and
            cr_row["crickrida_first_date"] <= cs_row["cricsheet_last_date"] and
            cs_row["cricsheet_first_date"] <= cr_row["crickrida_last_date"]
        )

        candidates.append({
            "crickrida_name": cr_row["player_name"],
            "crickrida_matches": cr_row["crickrida_matches"],
            "crickrida_first_date": cr_row["crickrida_first_date"],
            "crickrida_last_date": cr_row["crickrida_last_date"],
            "cricsheet_name": cs_row["player_name"],
            "cricsheet_id": cs_row["source_player_id"],
            "cricsheet_matches": cs_row["cricsheet_matches"],
            "initial_match": initial_match,
            "date_overlap": date_overlap
        })

sl_candidates = pd.DataFrame(candidates)

if not sl_candidates.empty:
    sl_candidates["review_score"] = (
        sl_candidates["initial_match"].astype(int) * 2 +
        sl_candidates["date_overlap"].astype(int)
    )

    sl_candidates = sl_candidates.sort_values(
        ["review_score", "crickrida_name"],
        ascending=[False, True]
    ).reset_index(drop=True)

    display(sl_candidates)
else:
    print("No same-surname candidates found.")

# Players with no same-surname candidate
matched_candidate_names = (
    set(sl_candidates["crickrida_name"])
    if not sl_candidates.empty else set()
)

sl_no_candidates = cr[
    ~cr["player_name"].isin(matched_candidate_names)
].copy()

print("Total candidate pairs:", len(sl_candidates))
print("Crickrida players with candidates:", len(matched_candidate_names))
print("Crickrida players with no same-surname candidates:", len(sl_no_candidates))

display(sl_no_candidates[["player_name", "crickrida_first_date",
                          "crickrida_last_date", "crickrida_matches"]])

,crickrida_name,crickrida_matches,crickrida_first_date,crickrida_last_date,cricsheet_name,cricsheet_id,cricsheet_matches,initial_match,date_overlap,review_score
0,Akila Dananjaya,2,2018-09-17,2024-02-14,A Dananjaya,5b16a806,38,True,True,3
1,Angelo Mathews,6,2014-03-03,2023-10-30,AD Mathews,896d78ad,210,True,True,3
2,Asitha Fernando,3,2022-11-30,2024-02-14,AM Fernando,de3d549a,33,True,True,3
3,Avishka Fernando,3,2024-02-09,2024-02-14,AM Fernando,de3d549a,33,True,True,3
4,Dhananjaya de Silva,9,2018-09-17,2023-10-30,DM de Silva,7d608e12,81,True,True,3
...,...,...,...,...,...,...,...,...,...,...
92,Mahela Jayawardene,2,2014-03-03,2015-02-22,HAPW Jayawardene,4a25c894,2,False,False,0
93,Matheesha Pathirana,2,2023-06-02,2023-09-05,SS Pathirana,753c95b9,18,False,False,0
94,Pathum Nissanka,11,2022-11-25,2024-02-14,RAP Nissanka,6cc006ef,9,False,False,0
95,Shehan Jayasuriya,1,2018-09-17,2018-09-17,NGRP Jayasuriya,127bc5ca,2,False,False,0


Total candidate pairs: 97
Crickrida players with candidates: 38
Crickrida players with no same-surname candidates: 2


,player_name,crickrida_first_date,crickrida_last_date,crickrida_matches
366,Pramod Madushan,2024-02-09,2024-02-14,3
374,Wanindu Hasaranga,2022-11-25,2024-02-11,7


In [66]:
# Group candidates by Crickrida player

sl_candidate_summary = (
    sl_candidates
    .sort_values(
        ["crickrida_name", "review_score", "cricsheet_name"],
        ascending=[True, False, True]
    )
    .groupby("crickrida_name")
    .agg(
        candidate_count=("cricsheet_name", "count"),
        candidates=("cricsheet_name", lambda x: list(x)),
        candidate_ids=("cricsheet_id", lambda x: list(x)),
        initial_matches=("initial_match", "sum"),
        date_overlaps=("date_overlap", "sum")
    )
    .reset_index()
)

display(sl_candidate_summary.to_string(index=False))

'        crickrida_name  candidate_count                                                                                                                                                candidates                                                                                                  candidate_ids  initial_matches  date_overlaps\n        Ajantha Mendis                5                                                                                            [BAW Mendis, BMAJ Mendis, BKG Mendis, PHKD Mendis, RTM Mendis]                                                             [ea0cdc12, 9868bc75, 5d1e7582, 08548b13, 1be67d37]                0              2\n       Akila Dananjaya                1                                                                                                                                             [A Dananjaya]                                                                                                     [5b16a806]                1  

In [67]:
# Find Cricsheet IDs proposed for multiple Crickrida players

sl_id_conflicts = (
    sl_candidates
    .groupby("cricsheet_id")
    .agg(
        cricsheet_name=("cricsheet_name", "first"),
        crickrida_players=("crickrida_name", lambda x: sorted(set(x))),
        crickrida_count=("crickrida_name", "nunique")
    )
    .reset_index()
)

sl_id_conflicts = (
    sl_id_conflicts[
        sl_id_conflicts["crickrida_count"] > 1
    ]
    .sort_values("crickrida_count", ascending=False)
)

print("Conflicting Cricsheet IDs:", len(sl_id_conflicts))
display(sl_id_conflicts.to_string(index=False))

Conflicting Cricsheet IDs: 27


'cricsheet_id cricsheet_name                             crickrida_players  crickrida_count\n    08548b13    PHKD Mendis [Ajantha Mendis, Jeevan Mendis, Kusal Mendis]                3\n    1be67d37     RTM Mendis [Ajantha Mendis, Jeevan Mendis, Kusal Mendis]                3\n    5d1e7582     BKG Mendis [Ajantha Mendis, Jeevan Mendis, Kusal Mendis]                3\n    ea0cdc12     BAW Mendis [Ajantha Mendis, Jeevan Mendis, Kusal Mendis]                3\n    9868bc75    BMAJ Mendis [Ajantha Mendis, Jeevan Mendis, Kusal Mendis]                3\n    0f12f9df    NLTC Perera                [Kusal Perera, Thisara Perera]                2\n    1be70c88      LPC Silva    [Chaturanga de Silva, Dhananjaya de Silva]                2\n    34ff9e07    PA de Silva    [Chaturanga de Silva, Dhananjaya de Silva]                2\n    4cd3960e  KADM Fernando           [Asitha Fernando, Avishka Fernando]                2\n    342d8ade   CRD Fernando           [Asitha Fernando, Avishka Fernando]      

In [68]:
# Get players having exactly one candidate
candidate_counts = (
    sl_candidates.groupby("crickrida_name")["cricsheet_id"]
    .nunique()
)

single_candidate_names = candidate_counts[
    candidate_counts == 1
].index

sl_single_candidates = sl_candidates[
    sl_candidates["crickrida_name"].isin(single_candidate_names)
].copy()

# Add Cricsheet career dates
sl_single_candidates = sl_single_candidates.merge(
    cs[
        ["source_player_id", "cricsheet_first_date",
         "cricsheet_last_date"]
    ],
    left_on="cricsheet_id",
    right_on="source_player_id",
    how="left"
)

display(
    sl_single_candidates[
        [
            "crickrida_name",
            "crickrida_first_date",
            "crickrida_last_date",
            "cricsheet_name",
            "cricsheet_id",
            "cricsheet_first_date",
            "cricsheet_last_date",
            "initial_match",
            "date_overlap"
        ]
    ].sort_values("crickrida_name").to_string(index=False)
)

'        crickrida_name crickrida_first_date crickrida_last_date   cricsheet_name cricsheet_id cricsheet_first_date cricsheet_last_date  initial_match  date_overlap\n       Akila Dananjaya           2018-09-17          2024-02-14      A Dananjaya     5b16a806           2017-06-30          2024-08-04           True          True\n        Angelo Mathews           2014-03-03          2023-10-30       AD Mathews     896d78ad           2008-11-30          2023-11-09           True          True\n      Charith Asalanka           2022-11-25          2024-02-14     KIC Asalanka     732c038e           2021-06-29          2026-09-27          False          True\n         Dasun Shanaka           2018-09-17          2023-09-05       MD Shanaka     3ff033bb           2016-06-18          2026-10-22          False          True\n    Dhananjaya Lakshan           2022-11-25          2022-11-27        D Lakshan     0a2184d9           2021-06-29          2021-06-29           True         False\n      Din

In [69]:
# Corrected Sri Lanka candidate matching

def get_cricsheet_initial_token(name):
    parts = str(name).strip().split()
    return parts[0].casefold() if parts else ""


corrected_candidates = []

for _, cr_row in cr.iterrows():

    cr_surname = get_surname(cr_row["player_name"])
    cr_initial = get_first_initial(cr_row["player_name"])

    for _, cs_row in cs.iterrows():

        if cr_surname != get_surname(cs_row["player_name"]):
            continue

        cs_initials = get_cricsheet_initial_token(
            cs_row["player_name"]
        )

        initial_match = cr_initial in cs_initials

        date_overlap = (
            pd.notna(cr_row["crickrida_first_date"]) and
            pd.notna(cr_row["crickrida_last_date"]) and
            pd.notna(cs_row["cricsheet_first_date"]) and
            pd.notna(cs_row["cricsheet_last_date"]) and
            cr_row["crickrida_first_date"] <= cs_row["cricsheet_last_date"] and
            cs_row["cricsheet_first_date"] <= cr_row["crickrida_last_date"]
        )

        corrected_candidates.append({
            "crickrida_name": cr_row["player_name"],
            "crickrida_first_date": cr_row["crickrida_first_date"],
            "crickrida_last_date": cr_row["crickrida_last_date"],
            "cricsheet_name": cs_row["player_name"],
            "cricsheet_id": cs_row["source_player_id"],
            "cricsheet_first_date": cs_row["cricsheet_first_date"],
            "cricsheet_last_date": cs_row["cricsheet_last_date"],
            "initial_match": initial_match,
            "date_overlap": date_overlap
        })

sl_candidates_corrected = pd.DataFrame(corrected_candidates)

sl_candidates_corrected["review_score"] = (
    sl_candidates_corrected["initial_match"].astype(int) * 2 +
    sl_candidates_corrected["date_overlap"].astype(int)
)

sl_candidates_corrected = sl_candidates_corrected.sort_values(
    ["crickrida_name", "review_score"],
    ascending=[True, False]
).reset_index(drop=True)

print("Total candidate pairs:", len(sl_candidates_corrected))

display(sl_candidates_corrected.to_string(index=False))

Total candidate pairs: 97


'        crickrida_name crickrida_first_date crickrida_last_date   cricsheet_name cricsheet_id cricsheet_first_date cricsheet_last_date  initial_match  date_overlap  review_score\n        Ajantha Mendis           2014-03-03          2014-03-03       BAW Mendis     ea0cdc12           2008-04-10          2015-12-26           True          True             3\n        Ajantha Mendis           2014-03-03          2014-03-03      BMAJ Mendis     9868bc75           2010-06-01          2019-06-28           True          True             3\n        Ajantha Mendis           2014-03-03          2014-03-03       BKG Mendis     5d1e7582           2016-06-18          2026-10-18          False         False             0\n        Ajantha Mendis           2014-03-03          2014-03-03      PHKD Mendis     08548b13           2019-03-10          2026-10-22          False         False             0\n        Ajantha Mendis           2014-03-03          2014-03-03       RTM Mendis     1be67d37           

In [70]:
sl_strong_candidates = sl_candidates_corrected[
    (sl_candidates_corrected["initial_match"] == True) &
    (sl_candidates_corrected["date_overlap"] == True)
].copy()

print("Candidates with matching initials and overlapping dates:",
      len(sl_strong_candidates))

display(
    sl_strong_candidates[
        ["crickrida_name", "cricsheet_name", "cricsheet_id",
         "crickrida_first_date", "crickrida_last_date",
         "cricsheet_first_date", "cricsheet_last_date"]
    ].to_string(index=False)
)

Candidates with matching initials and overlapping dates: 42


'        crickrida_name   cricsheet_name cricsheet_id crickrida_first_date crickrida_last_date cricsheet_first_date cricsheet_last_date\n        Ajantha Mendis       BAW Mendis     ea0cdc12           2014-03-03          2014-03-03           2008-04-10          2015-12-26\n        Ajantha Mendis      BMAJ Mendis     9868bc75           2014-03-03          2014-03-03           2010-06-01          2019-06-28\n       Akila Dananjaya      A Dananjaya     5b16a806           2018-09-17          2024-02-14           2017-06-30          2024-08-04\n        Angelo Mathews       AD Mathews     896d78ad           2014-03-03          2023-10-30           2008-11-30          2023-11-09\n       Asitha Fernando      AM Fernando     de3d549a           2022-11-30          2024-02-14           2017-07-08          2026-10-22\n       Asitha Fernando     WIA Fernando     9550470d           2022-11-30          2024-02-14           2016-08-31          2026-10-22\n      Avishka Fernando      AM Fernando     de3

In [71]:
# Start with initial-compatible candidates with overlapping dates
strong = sl_candidates_corrected[
    sl_candidates_corrected["initial_match"] &
    sl_candidates_corrected["date_overlap"]
].copy()

# Count candidates per Crickrida player
cr_counts = strong.groupby("crickrida_name")["cricsheet_id"].nunique()

# Count Crickrida players per Cricsheet ID
cs_counts = strong.groupby("cricsheet_id")["crickrida_name"].nunique()

# Keep only mutually unique candidate pairs
sl_unique_candidates = strong[
    (strong["crickrida_name"].map(cr_counts) == 1) &
    (strong["cricsheet_id"].map(cs_counts) == 1)
].copy()

print("Initial-compatible pairs:", len(strong))
print("Mutually unique candidate pairs:", len(sl_unique_candidates))

display(
    sl_unique_candidates[
        [
            "crickrida_name",
            "cricsheet_name",
            "cricsheet_id",
            "crickrida_first_date",
            "crickrida_last_date",
            "cricsheet_first_date",
            "cricsheet_last_date"
        ]
    ].sort_values("crickrida_name").to_string(index=False)
)

Initial-compatible pairs: 42
Mutually unique candidate pairs: 30


'        crickrida_name   cricsheet_name cricsheet_id crickrida_first_date crickrida_last_date cricsheet_first_date cricsheet_last_date\n       Akila Dananjaya      A Dananjaya     5b16a806           2018-09-17          2024-02-14           2017-06-30          2024-08-04\n        Angelo Mathews       AD Mathews     896d78ad           2014-03-03          2023-10-30           2008-11-30          2023-11-09\n      Charith Asalanka     KIC Asalanka     732c038e           2022-11-25          2024-02-14           2021-06-29          2026-09-27\n   Chaturanga de Silva      PC de Silva     b40ee8a8           2014-03-03          2014-03-03           2014-02-25          2017-02-01\n         Dasun Shanaka       MD Shanaka     3ff033bb           2018-09-17          2023-09-05           2016-06-18          2026-10-22\n   Dhananjaya de Silva      DM de Silva     7d608e12           2018-09-17          2023-10-30           2016-08-21          2026-01-27\n    Dilshan Madushanka     D Madushanka     de7

In [72]:
for name in ["Pramod Madushan", "Wanindu Hasaranga"]:
    print(f"\n{name}")

    display(
        cs[
            cs["player_name"].str.contains(
                name.split()[-1], case=False, regex=False, na=False
            )
        ][
            [
                "player_name",
                "source_player_id",
                "cricsheet_first_date",
                "cricsheet_last_date",
                "cricsheet_matches"
            ]
        ]
    )


Pramod Madushan


,player_name,source_player_id,cricsheet_first_date,cricsheet_last_date,cricsheet_matches
1496,D Madushanka,de7d833e,2023-01-10,2026-10-22,29
1501,DSK Madushanka,248698f4,2018-01-27,2018-01-27,1
1531,LD Madushanka,de063a39,2017-02-04,2017-06-30,4



Wanindu Hasaranga


,player_name,source_player_id,cricsheet_first_date,cricsheet_last_date,cricsheet_matches


In [73]:
# Search the entire Sri Lanka Cricsheet history
# using parts of the players' names.

search_terms = [
    "Pramod",
    "Madushan",
    "Madushanka",
    "Wanindu",
    "Hasaranga"
]

for term in search_terms:
    print(f"\n--- Search: {term} ---")

    results = cs[
        cs["player_name"].str.contains(
            term, case=False, regex=False, na=False
        )
    ]

    display(
        results[
            [
                "player_name",
                "source_player_id",
                "cricsheet_first_date",
                "cricsheet_last_date",
                "cricsheet_matches"
            ]
        ]
    )


--- Search: Pramod ---


,player_name,source_player_id,cricsheet_first_date,cricsheet_last_date,cricsheet_matches



--- Search: Madushan ---


,player_name,source_player_id,cricsheet_first_date,cricsheet_last_date,cricsheet_matches
1496,D Madushanka,de7d833e,2023-01-10,2026-10-22,29
1501,DSK Madushanka,248698f4,2018-01-27,2018-01-27,1
1531,LD Madushanka,de063a39,2017-02-04,2017-06-30,4



--- Search: Madushanka ---


,player_name,source_player_id,cricsheet_first_date,cricsheet_last_date,cricsheet_matches
1496,D Madushanka,de7d833e,2023-01-10,2026-10-22,29
1501,DSK Madushanka,248698f4,2018-01-27,2018-01-27,1
1531,LD Madushanka,de063a39,2017-02-04,2017-06-30,4



--- Search: Wanindu ---


,player_name,source_player_id,cricsheet_first_date,cricsheet_last_date,cricsheet_matches



--- Search: Hasaranga ---


,player_name,source_player_id,cricsheet_first_date,cricsheet_last_date,cricsheet_matches


In [74]:
resolved_candidate_names = set(
    sl_unique_candidates["crickrida_name"]
)

sl_remaining_review = cr[
    ~cr["player_name"].isin(resolved_candidate_names)
].copy()

print("Remaining players:", len(sl_remaining_review))

display(
    sl_remaining_review[
        [
            "player_name",
            "crickrida_first_date",
            "crickrida_last_date",
            "crickrida_matches"
        ]
    ].sort_values("player_name")
)

Remaining players: 10


,player_name,crickrida_first_date,crickrida_last_date,crickrida_matches
335,Ajantha Mendis,2014-03-03,2014-03-03,1
338,Asitha Fernando,2022-11-30,2024-02-14,3
339,Avishka Fernando,2024-02-09,2024-02-14,3
344,Dhananjaya Lakshan,2022-11-25,2022-11-27,2
349,Dushan Hemantha,2023-06-02,2023-06-02,1
353,Jeevan Mendis,2015-02-22,2015-02-22,1
356,Kusal Mendis,2018-09-17,2024-02-14,13
357,Kusal Perera,2014-03-03,2019-06-04,3
366,Pramod Madushan,2024-02-09,2024-02-14,3
374,Wanindu Hasaranga,2022-11-25,2024-02-11,7


In [75]:
ambiguous_sl_players = [
    "Ajantha Mendis",
    "Asitha Fernando",
    "Avishka Fernando",
    "Dhananjaya Lakshan",
    "Dushan Hemantha",
    "Jeevan Mendis",
    "Kusal Mendis",
    "Kusal Perera"
]

sl_ambiguous_details = (
    sl_candidates_corrected[
        sl_candidates_corrected["crickrida_name"].isin(
            ambiguous_sl_players
        )
    ]
    .sort_values(
        ["crickrida_name", "review_score"],
        ascending=[True, False]
    )
)

display(
    sl_ambiguous_details[
        [
            "crickrida_name",
            "cricsheet_name",
            "cricsheet_id",
            "crickrida_first_date",
            "crickrida_last_date",
            "cricsheet_first_date",
            "cricsheet_last_date",
            "initial_match",
            "date_overlap",
            "review_score"
        ]
    ].to_string(index=False)
)

'    crickrida_name cricsheet_name cricsheet_id crickrida_first_date crickrida_last_date cricsheet_first_date cricsheet_last_date  initial_match  date_overlap  review_score\n    Ajantha Mendis     BAW Mendis     ea0cdc12           2014-03-03          2014-03-03           2008-04-10          2015-12-26           True          True             3\n    Ajantha Mendis    BMAJ Mendis     9868bc75           2014-03-03          2014-03-03           2010-06-01          2019-06-28           True          True             3\n    Ajantha Mendis     BKG Mendis     5d1e7582           2014-03-03          2014-03-03           2016-06-18          2026-10-18          False         False             0\n    Ajantha Mendis    PHKD Mendis     08548b13           2014-03-03          2014-03-03           2019-03-10          2026-10-22          False         False             0\n    Ajantha Mendis     RTM Mendis     1be67d37           2014-03-03          2014-03-03           2021-05-28          2022-01-21      

In [77]:
# Inspect all five relevant Cricsheet player records

review_ids = [
    "ea0cdc12",  # BAW Mendis
    "9868bc75",  # BMAJ Mendis
    "5d1e7582",  # BKG Mendis
    "de3d549a",  # AM Fernando
    "9550470d"   # WIA Fernando
]

display(
    cs_player_history[
        cs_player_history["source_player_id"].isin(review_ids)
    ].sort_values("player_name")
)

,team,normalized_name,player_name,source_player_id,cricsheet_first_date,cricsheet_last_date,cricsheet_matches
1478,Sri Lanka,amfernando,AM Fernando,de3d549a,2017-07-08,2026-10-22,33
1479,Sri Lanka,bawmendis,BAW Mendis,ea0cdc12,2008-04-10,2015-12-26,79
1481,Sri Lanka,bkgmendis,BKG Mendis,5d1e7582,2016-06-18,2026-10-18,141
1482,Sri Lanka,bmajmendis,BMAJ Mendis,9868bc75,2010-06-01,2019-06-28,52
1604,Sri Lanka,wiafernando,WIA Fernando,9550470d,2016-08-31,2026-10-22,49


In [78]:
print("BATTING TABLE COLUMNS:")
print(player_batting_updated.columns.tolist())

print("\nBATTING SAMPLE:")
display(player_batting_updated.head(3))

print("\nBOWLING TABLE COLUMNS:")
print(player_bowling_updated.columns.tolist())

print("\nBOWLING SAMPLE:")
display(player_bowling_updated.head(3))

BATTING TABLE COLUMNS:
['match_id', 'innings_number', 'batting_team', 'batter', 'batter_id', 'runs', 'balls_faced', 'fours', 'sixes', 'strike_rate', 'cricketiq_player_id', 'canonical_player_name']

BATTING SAMPLE:


,match_id,innings_number,batting_team,batter,batter_id,runs,balls_faced,fours,sixes,strike_rate,cricketiq_player_id,canonical_player_name
0,1000887,1,Australia,B Stanlake,6834d1f2,1,6,0,0,16.666667,CS_6834d1f2,B Stanlake
1,1000887,1,Australia,CA Lynn,45eda7c8,16,12,1,1,133.333333,CS_45eda7c8,CA Lynn
2,1000887,1,Australia,DA Warner,dcce6f09,7,18,1,0,38.888889,CS_dcce6f09,DA Warner



BOWLING TABLE COLUMNS:
['match_id', 'innings_number', 'bowler', 'bowler_id', 'runs_conceded', 'wides', 'no_balls', 'wickets', 'legal_balls', 'overs', 'economy', 'cricketiq_player_id', 'canonical_player_name']

BOWLING SAMPLE:


,match_id,innings_number,bowler,bowler_id,runs_conceded,wides,no_balls,wickets,legal_balls,overs,economy,cricketiq_player_id,canonical_player_name
0,1000887,1,Hasan Ali,2911de16,65,2,0,3,54,9.0,7.222222,CRIQ_2911de16,Hasan Ali
1,1000887,1,Imad Wasim,9cb8d7a6,35,0,0,2,60,10.0,3.500000,CRIQ_9cb8d7a6,Imad Wasim
2,1000887,1,Mohammad Amir,e174dadd,54,3,0,2,60,10.0,5.400000,CRIQ_e174dadd,Mohammad Amir


In [79]:
# Five Sri Lankan identity candidates
review_players = {
    "Ajantha Mendis": "ea0cdc12",
    "Jeevan Mendis": "9868bc75",
    "Kusal Mendis": "5d1e7582",
    "Asitha Fernando": "de3d549a",
    "Avishka Fernando": "9550470d"
}

for expected_name, player_id in review_players.items():

    print(f"\n{'='*65}")
    print(f"EXPECTED CRICKRIDA PLAYER: {expected_name}")
    print(f"CRICSHEET ID: {player_id}")
    print("="*65)

    batting = player_batting_updated[
        player_batting_updated["batter_id"] == player_id
    ]

    bowling = player_bowling_updated[
        player_bowling_updated["bowler_id"] == player_id
    ]

    batting_matches = batting["match_id"].nunique()
    bowling_matches = bowling["match_id"].nunique()

    all_matches = set(batting["match_id"]) | set(bowling["match_id"])

    print("Cricsheet batting records:", len(batting))
    print("Cricsheet bowling records:", len(bowling))
    print("Unique batting matches:", batting_matches)
    print("Unique bowling matches:", bowling_matches)
    print("Unique matches across both:", len(all_matches))

    print("\nBatting names:")
    display(batting["batter"].value_counts().to_frame("records"))

    print("\nBowling names:")
    display(bowling["bowler"].value_counts().to_frame("records"))


EXPECTED CRICKRIDA PLAYER: Ajantha Mendis
CRICSHEET ID: ea0cdc12
Cricsheet batting records: 39
Cricsheet bowling records: 79
Unique batting matches: 39
Unique bowling matches: 79
Unique matches across both: 79

Batting names:


,records
batter,
BAW Mendis,39



Bowling names:


,records
bowler,
BAW Mendis,79



EXPECTED CRICKRIDA PLAYER: Jeevan Mendis
CRICSHEET ID: 9868bc75
Cricsheet batting records: 42
Cricsheet bowling records: 47
Unique batting matches: 42
Unique bowling matches: 47
Unique matches across both: 52

Batting names:


,records
batter,
BMAJ Mendis,42


... (10 outputs left out)


EXPECTED CRICKRIDA PLAYER: Avishka Fernando
CRICSHEET ID: 9550470d
Cricsheet batting records: 49
Cricsheet bowling records: 0
Unique batting matches: 49
Unique bowling matches: 0
Unique matches across both: 49

Batting names:


,records
batter,
WIA Fernando,49



Bowling names:


,records
bowler,


In [80]:
review_ids = [
    "ea0cdc12",
    "9868bc75",
    "5d1e7582",
    "de3d549a",
    "9550470d"
]

display(
    player_identity_map_updated[
        player_identity_map_updated["player_id"].isin(review_ids)
    ][
        [
            "cricketiq_player_id",
            "player_id",
            "cricsheet_name",
            "crickrida_name",
            "match_status",
            "mapping_method",
            "confidence",
            "notes"
        ]
    ].to_string(index=False)
)

'Empty DataFrame\nColumns: [cricketiq_player_id, player_id, cricsheet_name, crickrida_name, match_status, mapping_method, confidence, notes]\nIndex: []'

In [81]:
review_names = [
    "Ajantha Mendis",
    "Jeevan Mendis",
    "Kusal Mendis",
    "Asitha Fernando",
    "Avishka Fernando"
]

display(
    player_identity_map_updated[
        player_identity_map_updated["crickrida_name"].isin(review_names)
    ].to_string(index=False)
)

'Empty DataFrame\nColumns: [cricketiq_player_id, player_id, cricsheet_name, crickrida_name, normalized_name, match_status, crickrida_player_name, cricsheet_player_name, cricsheet_player_id, mapping_method, confidence, notes]\nIndex: []'

In [82]:
sl_validated_candidates = [
    {
        "crickrida_name": "Ajantha Mendis",
        "cricsheet_name": "BAW Mendis",
        "player_id": "ea0cdc12"
    },
    {
        "crickrida_name": "Jeevan Mendis",
        "cricsheet_name": "BMAJ Mendis",
        "player_id": "9868bc75"
    },
    {
        "crickrida_name": "Kusal Mendis",
        "cricsheet_name": "BKG Mendis",
        "player_id": "5d1e7582"
    },
    {
        "crickrida_name": "Asitha Fernando",
        "cricsheet_name": "AM Fernando",
        "player_id": "de3d549a"
    },
    {
        "crickrida_name": "Avishka Fernando",
        "cricsheet_name": "WIA Fernando",
        "player_id": "9550470d"
    }
]

sl_validated_candidates_df = pd.DataFrame(sl_validated_candidates)

sl_validated_candidates_df["cricketiq_player_id"] = (
    "CRIQ_" + sl_validated_candidates_df["player_id"]
)

display(sl_validated_candidates_df)

,crickrida_name,cricsheet_name,player_id,cricketiq_player_id
0,Ajantha Mendis,BAW Mendis,ea0cdc12,CRIQ_ea0cdc12
1,Jeevan Mendis,BMAJ Mendis,9868bc75,CRIQ_9868bc75
2,Kusal Mendis,BKG Mendis,5d1e7582,CRIQ_5d1e7582
3,Asitha Fernando,AM Fernando,de3d549a,CRIQ_de3d549a
4,Avishka Fernando,WIA Fernando,9550470d,CRIQ_9550470d


In [83]:
# Check duplicate Crickrida names in the proposed batch
duplicate_crickrida = sl_validated_candidates_df[
    sl_validated_candidates_df["crickrida_name"].duplicated(keep=False)
]

# Check duplicate Cricsheet IDs in the proposed batch
duplicate_cricsheet_ids = sl_validated_candidates_df[
    sl_validated_candidates_df["player_id"].duplicated(keep=False)
]

# Check duplicate CricketIQ IDs in the proposed batch
duplicate_cricketiq_ids = sl_validated_candidates_df[
    sl_validated_candidates_df["cricketiq_player_id"].duplicated(keep=False)
]

# Check conflicts against the existing identity map
existing_id_conflicts = player_identity_map_updated[
    player_identity_map_updated["player_id"].isin(
        sl_validated_candidates_df["player_id"]
    )
]

existing_name_conflicts = player_identity_map_updated[
    player_identity_map_updated["crickrida_name"].isin(
        sl_validated_candidates_df["crickrida_name"]
    )
]

print("Proposed mappings:", len(sl_validated_candidates_df))
print("Duplicate Crickrida names:", len(duplicate_crickrida))
print("Duplicate Cricsheet IDs:", len(duplicate_cricsheet_ids))
print("Duplicate CricketIQ IDs:", len(duplicate_cricketiq_ids))
print("Existing ID conflicts:", len(existing_id_conflicts))
print("Existing name conflicts:", len(existing_name_conflicts))

Proposed mappings: 5
Duplicate Crickrida names: 0
Duplicate Cricsheet IDs: 0
Duplicate CricketIQ IDs: 0
Existing ID conflicts: 0
Existing name conflicts: 0


In [84]:
new_sl_rows = []

for _, candidate in sl_validated_candidates_df.iterrows():

    crickrida_name = candidate["crickrida_name"]
    cricsheet_name = candidate["cricsheet_name"]
    player_id = candidate["player_id"]

    new_sl_rows.append({
        "cricketiq_player_id": f"CRIQ_{player_id}",
        "player_id": player_id,
        "cricsheet_name": cricsheet_name,
        "crickrida_name": crickrida_name,
        "normalized_name": normalize_identity(crickrida_name),
        "match_status": "verified_alias",
        "crickrida_player_name": crickrida_name,
        "cricsheet_player_name": cricsheet_name,
        "cricsheet_player_id": player_id,
        "mapping_method": "verified_sri_lanka_name_and_performance_review",
        "confidence": "high",
        "notes": (
            "Matched using player-name/initial evidence, "
            "career-period review, and Cricsheet performance-table inspection."
        )
    })

new_sl_rows_df = pd.DataFrame(
    new_sl_rows,
    columns=player_identity_map_updated.columns
)

player_identity_map_updated = pd.concat(
    [player_identity_map_updated, new_sl_rows_df],
    ignore_index=True
)

print("New mappings added:", len(new_sl_rows_df))
print("Updated identity map shape:", player_identity_map_updated.shape)

display(new_sl_rows_df)

New mappings added: 5
Updated identity map shape: (270, 12)


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,normalized_name,match_status,crickrida_player_name,cricsheet_player_name,cricsheet_player_id,mapping_method,confidence,notes
0,CRIQ_ea0cdc12,ea0cdc12,BAW Mendis,Ajantha Mendis,ajanthamendis,verified_alias,Ajantha Mendis,BAW Mendis,ea0cdc12,verified_sri_lanka_name_and_performance_review,high,"Matched using player-name/initial evidence, ca..."
1,CRIQ_9868bc75,9868bc75,BMAJ Mendis,Jeevan Mendis,jeevanmendis,verified_alias,Jeevan Mendis,BMAJ Mendis,9868bc75,verified_sri_lanka_name_and_performance_review,high,"Matched using player-name/initial evidence, ca..."
2,CRIQ_5d1e7582,5d1e7582,BKG Mendis,Kusal Mendis,kusalmendis,verified_alias,Kusal Mendis,BKG Mendis,5d1e7582,verified_sri_lanka_name_and_performance_review,high,"Matched using player-name/initial evidence, ca..."
3,CRIQ_de3d549a,de3d549a,AM Fernando,Asitha Fernando,asithafernando,verified_alias,Asitha Fernando,AM Fernando,de3d549a,verified_sri_lanka_name_and_performance_review,high,"Matched using player-name/initial evidence, ca..."
4,CRIQ_9550470d,9550470d,WIA Fernando,Avishka Fernando,avishkafernando,verified_alias,Avishka Fernando,WIA Fernando,9550470d,verified_sri_lanka_name_and_performance_review,high,"Matched using player-name/initial evidence, ca..."


In [85]:
# Search the complete Sri Lankan Cricsheet player history
sl_search_terms = [
    "Lakshan",
    "Hemantha",
    "Perera",
    "Madushan",
    "Liyanagamage",
    "Hasaranga",
    "de Silva",
    "Silva",
    "Wanindu",
    "Pramod"
]

sl_cricsheet_search = cs_player_history[
    cs_player_history["team"].str.contains(
        "Sri Lanka", case=False, na=False
    )
].copy()

# Find matching names
sl_name_matches = sl_cricsheet_search[
    sl_cricsheet_search["player_name"].apply(
        lambda name: any(
            term.casefold() in str(name).casefold()
            for term in sl_search_terms
        )
    )
]

display(
    sl_name_matches[
        [
            "player_name",
            "source_player_id",
            "cricsheet_first_date",
            "cricsheet_last_date",
            "cricsheet_matches"
        ]
    ].sort_values("player_name").to_string(index=False)
)

'    player_name source_player_id cricsheet_first_date cricsheet_last_date  cricsheet_matches\n      AK Perera         c8ad0749           2013-07-26          2019-10-02                  5\n      D Lakshan         0a2184d9           2021-06-29          2021-06-29                  1\n   D Madushanka         de7d833e           2023-01-10          2026-10-22                 29\n    DM de Silva         7d608e12           2016-08-21          2026-01-27                 81\n DSK Madushanka         248698f4           2018-01-27          2018-01-27                  1\n  LD Madushanka         de063a39           2017-02-04          2017-06-30                  4\n      LPC Silva         1be70c88           2007-01-02          2011-11-23                 50\n  MADI Hemantha         58431ca0           2023-07-07          2023-11-02                  4\n     MDK Perera         5531248e           2007-10-13          2018-09-15                 13\n    MDKJ Perera         f21043a5           2013-01-13      

In [86]:
review_ids = [
    "0a2184d9",  # D Lakshan
    "58431ca0",  # MADI Hemantha
    "f21043a5",  # MDKJ Perera
    "5531248e",  # MDK Perera
]

display(
    sl_cricsheet_search[
        sl_cricsheet_search["source_player_id"].isin(review_ids)
    ][
        [
            "player_name",
            "source_player_id",
            "cricsheet_first_date",
            "cricsheet_last_date",
            "cricsheet_matches"
        ]
    ].to_string(index=False)
)

'  player_name source_player_id cricsheet_first_date cricsheet_last_date  cricsheet_matches\n    D Lakshan         0a2184d9           2021-06-29          2021-06-29                  1\nMADI Hemantha         58431ca0           2023-07-07          2023-11-02                  4\n  MDKJ Perera         f21043a5           2013-01-13          2023-11-09                107\n   MDK Perera         5531248e           2007-10-13          2018-09-15                 13'

In [87]:
sl_review_ids = {
    "Dhananjaya Lakshan": "0a2184d9",
    "Dushan Hemantha": "58431ca0",
    "Kusal Perera": "f21043a5",
    "Pramod Madushan": "3ee0050f",
    "Wanindu Hasaranga": "a97c8ec2"
}

for player_name, player_id in sl_review_ids.items():

    batting = player_batting_updated[
        player_batting_updated["batter_id"].astype(str) == player_id
    ]

    bowling = player_bowling_updated[
        player_bowling_updated["bowler_id"].astype(str) == player_id
    ]

    print(f"\n{'='*55}")
    print(player_name)
    print(f"Cricsheet ID: {player_id}")
    print("Batting records:", len(batting))
    print("Bowling records:", len(bowling))
    print("Unique batting matches:", batting["match_id"].nunique())
    print("Unique bowling matches:", bowling["match_id"].nunique())
    print("Batting names:", batting["batter"].dropna().unique())
    print("Bowling names:", bowling["bowler"].dropna().unique())


Dhananjaya Lakshan
Cricsheet ID: 0a2184d9
Batting records: 1
Bowling records: 1
Unique batting matches: 1
Unique bowling matches: 1
Batting names: ['D Lakshan']
Bowling names: ['D Lakshan']

Dushan Hemantha
Cricsheet ID: 58431ca0
Batting records: 3
Bowling records: 3
Unique batting matches: 3
Unique bowling matches: 3
Batting names: ['MADI Hemantha']
Bowling names: ['MADI Hemantha']

Kusal Perera
Cricsheet ID: f21043a5
Batting records: 107
Bowling records: 0
Unique batting matches: 107
Unique bowling matches: 0
Batting names: ['MDKJ Perera']
Bowling names: []

Pramod Madushan
Cricsheet ID: 3ee0050f
Batting records: 9
Bowling records: 10
Unique batting matches: 9
Unique bowling matches: 10
Batting names: ['PM Liyanagamage']
Bowling names: ['PM Liyanagamage']

Wanindu Hasaranga
Cricsheet ID: a97c8ec2
Batting records: 54
Bowling records: 62
Unique batting matches: 54
Unique bowling matches: 62
Batting names: ['PWH de Silva']
Bowling names: ['PWH de Silva']


In [88]:
review_ids = list(sl_review_ids.values())

review_names = list(sl_review_ids.keys())

print("EXISTING CRICSHEET ID CONFLICTS")

display(
    player_identity_map_updated[
        player_identity_map_updated["player_id"].isin(review_ids)
    ].to_string(index=False)
)

print("\nEXISTING CRICKRIDA NAME CONFLICTS")

display(
    player_identity_map_updated[
        player_identity_map_updated["crickrida_name"].isin(review_names)
    ].to_string(index=False)
)

EXISTING CRICSHEET ID CONFLICTS


'Empty DataFrame\nColumns: [cricketiq_player_id, player_id, cricsheet_name, crickrida_name, normalized_name, match_status, crickrida_player_name, cricsheet_player_name, cricsheet_player_id, mapping_method, confidence, notes]\nIndex: []'


EXISTING CRICKRIDA NAME CONFLICTS


'Empty DataFrame\nColumns: [cricketiq_player_id, player_id, cricsheet_name, crickrida_name, normalized_name, match_status, crickrida_player_name, cricsheet_player_name, cricsheet_player_id, mapping_method, confidence, notes]\nIndex: []'

In [89]:
sl_next_candidates = [
    {
        "crickrida_name": "Dhananjaya Lakshan",
        "cricsheet_name": "D Lakshan",
        "player_id": "0a2184d9",
        "notes": (
            "Full-name initial and surname match. "
            "Cricsheet contains one match dated 2021-06-29. "
            "Crickrida history dates do not overlap; "
            "limited Cricsheet coverage noted."
        )
    },
    {
        "crickrida_name": "Dushan Hemantha",
        "cricsheet_name": "MADI Hemantha",
        "player_id": "58431ca0",
        "notes": (
            "Full-name initials MADI and surname Hemantha match. "
            "Consistent name in batting and bowling performance tables."
        )
    },
    {
        "crickrida_name": "Kusal Perera",
        "cricsheet_name": "MDKJ Perera",
        "player_id": "f21043a5",
        "notes": (
            "Full-name initials MDKJ and surname Perera match. "
            "107 consistent batting records. "
            "Distinguished from the separate MDK Perera record."
        )
    },
    {
        "crickrida_name": "Pramod Madushan",
        "cricsheet_name": "PM Liyanagamage",
        "player_id": "3ee0050f",
        "notes": (
            "Full-name initials PML and surname Liyanagamage match. "
            "Consistent name in batting and bowling performance tables."
        )
    },
    {
        "crickrida_name": "Wanindu Hasaranga",
        "cricsheet_name": "PWH de Silva",
        "player_id": "a97c8ec2",
        "notes": (
            "Full-name initials PWH and surname de Silva match. "
            "Consistent name in batting and bowling performance tables."
        )
    }
]

sl_next_candidates_df = pd.DataFrame(sl_next_candidates)

sl_next_candidates_df["cricketiq_player_id"] = (
    "CRIQ_" + sl_next_candidates_df["player_id"]
)

display(sl_next_candidates_df)

,crickrida_name,cricsheet_name,player_id,notes,cricketiq_player_id
0,Dhananjaya Lakshan,D Lakshan,0a2184d9,Full-name initial and surname match. Cricsheet...,CRIQ_0a2184d9
1,Dushan Hemantha,MADI Hemantha,58431ca0,Full-name initials MADI and surname Hemantha m...,CRIQ_58431ca0
2,Kusal Perera,MDKJ Perera,f21043a5,Full-name initials MDKJ and surname Perera mat...,CRIQ_f21043a5
3,Pramod Madushan,PM Liyanagamage,3ee0050f,Full-name initials PML and surname Liyanagamag...,CRIQ_3ee0050f
4,Wanindu Hasaranga,PWH de Silva,a97c8ec2,Full-name initials PWH and surname de Silva ma...,CRIQ_a97c8ec2


In [90]:
print("Proposed mappings:", len(sl_next_candidates_df))

print(
    "Duplicate Crickrida names:",
    sl_next_candidates_df["crickrida_name"].duplicated().sum()
)

print(
    "Duplicate Cricsheet IDs:",
    sl_next_candidates_df["player_id"].duplicated().sum()
)

print(
    "Duplicate CricketIQ IDs:",
    sl_next_candidates_df["cricketiq_player_id"].duplicated().sum()
)

print(
    "Existing ID conflicts:",
    player_identity_map_updated["player_id"].isin(
        sl_next_candidates_df["player_id"]
    ).sum()
)

print(
    "Existing name conflicts:",
    player_identity_map_updated["crickrida_name"].isin(
        sl_next_candidates_df["crickrida_name"]
    ).sum()
)

Proposed mappings: 5
Duplicate Crickrida names: 0
Duplicate Cricsheet IDs: 0
Duplicate CricketIQ IDs: 0
Existing ID conflicts: 0
Existing name conflicts: 0


In [91]:
new_sl_rows = []

for _, candidate in sl_next_candidates_df.iterrows():

    player_id = candidate["player_id"]
    crickrida_name = candidate["crickrida_name"]
    cricsheet_name = candidate["cricsheet_name"]

    new_sl_rows.append({
        "cricketiq_player_id": f"CRIQ_{player_id}",
        "player_id": player_id,
        "cricsheet_name": cricsheet_name,
        "crickrida_name": crickrida_name,
        "normalized_name": normalize_identity(crickrida_name),
        "match_status": "verified_alias",
        "crickrida_player_name": crickrida_name,
        "cricsheet_player_name": cricsheet_name,
        "cricsheet_player_id": player_id,
        "mapping_method": "verified_sri_lanka_full_name_review",
        "confidence": "high",
        "notes": candidate["notes"]
    })

new_sl_rows_df = pd.DataFrame(
    new_sl_rows,
    columns=player_identity_map_updated.columns
)

player_identity_map_updated = pd.concat(
    [player_identity_map_updated, new_sl_rows_df],
    ignore_index=True
)

print("New mappings added:", len(new_sl_rows_df))
print("Updated identity map shape:", player_identity_map_updated.shape)

New mappings added: 5
Updated identity map shape: (275, 12)


In [92]:
sl_inserted_ids = [
    "0a2184d9",
    "58431ca0",
    "f21043a5",
    "3ee0050f",
    "a97c8ec2"
]

display(
    player_identity_map_updated[
        player_identity_map_updated["player_id"].isin(sl_inserted_ids)
    ][[
        "cricketiq_player_id",
        "crickrida_name",
        "cricsheet_name",
        "match_status",
        "mapping_method",
        "confidence",
        "notes"
    ]].to_string(index=False)
)

'cricketiq_player_id     crickrida_name  cricsheet_name   match_status                      mapping_method confidence                                                                                                                                                         notes\n      CRIQ_0a2184d9 Dhananjaya Lakshan       D Lakshan verified_alias verified_sri_lanka_full_name_review       high Full-name initial and surname match. Cricsheet contains one match dated 2021-06-29. Crickrida history dates do not overlap; limited Cricsheet coverage noted.\n      CRIQ_58431ca0    Dushan Hemantha   MADI Hemantha verified_alias verified_sri_lanka_full_name_review       high                                                Full-name initials MADI and surname Hemantha match. Consistent name in batting and bowling performance tables.\n      CRIQ_f21043a5       Kusal Perera     MDKJ Perera verified_alias verified_sri_lanka_full_name_review       high                          Full-name initials MDKJ and s

In [93]:
sl_batch_1_ids = [
    "5b16a806", "896d78ad", "732c038e", "b40ee8a8",
    "3ff033bb", "7d608e12", "de7d833e", "400e2d4d",
    "8dd02a98", "736123bb", "327b58d3", "ac5ae4af",
    "84212ffb", "8012d0b8", "6b71e6cf", "4751caa3",
    "c5928dec", "a12e1d51", "f24c6701", "d18f9182"
]

sl_batch_1 = sl_cricsheet_search[
    sl_cricsheet_search["source_player_id"].isin(sl_batch_1_ids)
].copy()

display(
    sl_batch_1[
        [
            "player_name",
            "source_player_id",
            "cricsheet_first_date",
            "cricsheet_last_date",
            "cricsheet_matches"
        ]
    ].sort_values("player_name").to_string(index=False)
)

'     player_name source_player_id cricsheet_first_date cricsheet_last_date  cricsheet_matches\n     A Dananjaya         5b16a806           2017-06-30          2024-08-04                 38\n      AD Mathews         896d78ad           2008-11-30          2023-11-09                210\n     CAK Rajitha         8012d0b8           2018-08-01          2023-11-06                 27\n    CBRLS Kumara         4751caa3           2017-02-04          2025-01-05                 28\n    D Madushanka         de7d833e           2023-01-10          2026-10-22                 29\n     DM de Silva         7d608e12           2016-08-21          2026-01-27                 81\n    DN Wellalage         736123bb           2022-06-14          2026-10-22                 35\nDPMD Jayawardene         d18f9182           2002-06-27          2015-03-18                260\n FDM Karunaratne         400e2d4d           2011-07-09          2023-11-02                 38\n HDRL Thirimanne         c5928dec           2010-

In [94]:
sl_batch_1_review = {
    "Akila Dananjaya": "5b16a806",
    "Angelo Mathews": "896d78ad",
    "Charith Asalanka": "732c038e",
    "Chaturanga de Silva": "b40ee8a8",
    "Dasun Shanaka": "3ff033bb",
    "Dhananjaya de Silva": "7d608e12",
    "Dilshan Madushanka": "de7d833e",
    "Dimuth Karunaratne": "400e2d4d",
    "Dinesh Chandimal": "8dd02a98",
    "Dunith Wellalage": "736123bb",
    "Dushmantha Chameera": "327b58d3",
    "Isuru Udana": "ac5ae4af",
    "Janith Liyanage": "84212ffb",
    "Kasun Rajitha": "8012d0b8",
    "Kumar Sangakkara": "6b71e6cf",
    "Lahiru Kumara": "4751caa3",
    "Lahiru Thirimanne": "c5928dec",
    "Lasith Malinga": "a12e1d51",
    "Maheesh Theekshana": "f24c6701",
    "Mahela Jayawardene": "d18f9182"
}

sl_batch_1_performance = []

for player_name, player_id in sl_batch_1_review.items():

    batting = player_batting_updated[
        player_batting_updated["batter_id"].astype(str) == player_id
    ]

    bowling = player_bowling_updated[
        player_bowling_updated["bowler_id"].astype(str) == player_id
    ]

    sl_batch_1_performance.append({
        "crickrida_name": player_name,
        "cricsheet_id": player_id,
        "batting_records": len(batting),
        "bowling_records": len(bowling),
        "batting_matches": batting["match_id"].nunique(),
        "bowling_matches": bowling["match_id"].nunique(),
        "batting_names": ", ".join(
            batting["batter"].dropna().unique()
        ),
        "bowling_names": ", ".join(
            bowling["bowler"].dropna().unique()
        )
    })

sl_batch_1_performance_df = pd.DataFrame(sl_batch_1_performance)

display(sl_batch_1_performance_df.to_string(index=False))

'     crickrida_name cricsheet_id  batting_records  bowling_records  batting_matches  bowling_matches    batting_names    bowling_names\n    Akila Dananjaya     5b16a806               29               38               29               38      A Dananjaya      A Dananjaya\n     Angelo Mathews     896d78ad              184              158              184              158       AD Mathews       AD Mathews\n   Charith Asalanka     732c038e               70               34               70               34     KIC Asalanka     KIC Asalanka\nChaturanga de Silva     b40ee8a8                6                6                6                6      PC de Silva      PC de Silva\n      Dasun Shanaka     3ff033bb               58               42               58               42       MD Shanaka       MD Shanaka\nDhananjaya de Silva     7d608e12               76               68               76               68      DM de Silva      DM de Silva\n Dilshan Madushanka     de7d833e               

In [95]:
sl_batch_1_candidate_ids = list(sl_batch_1_review.values())
sl_batch_1_candidate_names = list(sl_batch_1_review.keys())

print("EXISTING CRICSHEET ID CONFLICTS")

display(
    player_identity_map_updated[
        player_identity_map_updated["player_id"].isin(
            sl_batch_1_candidate_ids
        )
    ].to_string(index=False)
)

print("\nEXISTING CRICKRIDA NAME CONFLICTS")

display(
    player_identity_map_updated[
        player_identity_map_updated["crickrida_name"].isin(
            sl_batch_1_candidate_names
        )
    ].to_string(index=False)
)

EXISTING CRICSHEET ID CONFLICTS


'Empty DataFrame\nColumns: [cricketiq_player_id, player_id, cricsheet_name, crickrida_name, normalized_name, match_status, crickrida_player_name, cricsheet_player_name, cricsheet_player_id, mapping_method, confidence, notes]\nIndex: []'


EXISTING CRICKRIDA NAME CONFLICTS


'Empty DataFrame\nColumns: [cricketiq_player_id, player_id, cricsheet_name, crickrida_name, normalized_name, match_status, crickrida_player_name, cricsheet_player_name, cricsheet_player_id, mapping_method, confidence, notes]\nIndex: []'

In [96]:
sl_batch_1_full_names = {
    "Akila Dananjaya": "Mahamarakkala Kurukulasooriya Patabendige Akila Dananjaya Perera",
    "Angelo Mathews": "Angelo Davis Mathews",
    "Charith Asalanka": "Kariyawasam Indipalage Charith Asalanka",
    "Chaturanga de Silva": "Pinnaduwage Chaturanga de Silva",
    "Dasun Shanaka": "Madagamagamage Dasun Shanaka",
    "Dhananjaya de Silva": "Dhananjaya Maduranga de Silva",
    "Dilshan Madushanka": "Lokumarakkalage Dilshan Madushanka",
    "Dimuth Karunaratne": "Frank Dimuth Madushanka Karunaratne",
    "Dinesh Chandimal": "Lokuge Dinesh Chandimal",
    "Dunith Wellalage": "Dunith Nethmika Wellalage",
    "Dushmantha Chameera": "Pathira Vasan Dushmantha Chameera",
    "Isuru Udana": "Kande Arachchige Isuru Udana Tillakaratna",
    "Janith Liyanage": "Kongala Liyanage Janith Vimukthi",
    "Kasun Rajitha": "Chalana Anuradha Kasun Rajitha",
    "Kumar Sangakkara": "Kumar Chokshanada Sangakkara",
    "Lahiru Kumara": "Charith Bandara Rajapaksha Lahiru Sampath Kumara",
    "Lahiru Thirimanne": "Hettiarachchige Don Rumesh Lahiru Thirimanne",
    "Lasith Malinga": "Separamadu Lasith Malinga",
    "Maheesh Theekshana": "Morawakage Maheesh Theekshana",
    "Mahela Jayawardene": "Denagamage Proboth Mahela de Silva Jayawardene"
}

sl_batch_1_candidates = sl_batch_1[
    ["player_name", "source_player_id"]
].copy()

sl_batch_1_candidates = sl_batch_1_candidates.rename(
    columns={
        "player_name": "cricsheet_name",
        "source_player_id": "player_id"
    }
)

sl_batch_1_candidates["crickrida_name"] = (
    sl_batch_1_candidates["cricsheet_name"].map({
        "A Dananjaya": "Akila Dananjaya",
        "AD Mathews": "Angelo Mathews",
        "KIC Asalanka": "Charith Asalanka",
        "PC de Silva": "Chaturanga de Silva",
        "MD Shanaka": "Dasun Shanaka",
        "DM de Silva": "Dhananjaya de Silva",
        "D Madushanka": "Dilshan Madushanka",
        "FDM Karunaratne": "Dimuth Karunaratne",
        "LD Chandimal": "Dinesh Chandimal",
        "DN Wellalage": "Dunith Wellalage",
        "PVD Chameera": "Dushmantha Chameera",
        "I Udana": "Isuru Udana",
        "J Liyanage": "Janith Liyanage",
        "CAK Rajitha": "Kasun Rajitha",
        "KC Sangakkara": "Kumar Sangakkara",
        "CBRLS Kumara": "Lahiru Kumara",
        "HDRL Thirimanne": "Lahiru Thirimanne",
        "SL Malinga": "Lasith Malinga",
        "M Theekshana": "Maheesh Theekshana",
        "DPMD Jayawardene": "Mahela Jayawardene"
    })
)

sl_batch_1_candidates["full_name"] = (
    sl_batch_1_candidates["crickrida_name"].map(
        sl_batch_1_full_names
    )
)

sl_batch_1_candidates["cricketiq_player_id"] = (
    "CRIQ_" + sl_batch_1_candidates["player_id"]
)

display(sl_batch_1_candidates)

,cricsheet_name,player_id,crickrida_name,full_name,cricketiq_player_id
1475,A Dananjaya,5b16a806,Akila Dananjaya,Mahamarakkala Kurukulasooriya Patabendige Akil...,CRIQ_5b16a806
1476,AD Mathews,896d78ad,Angelo Mathews,Angelo Davis Mathews,CRIQ_896d78ad
1484,CAK Rajitha,8012d0b8,Kasun Rajitha,Chalana Anuradha Kasun Rajitha,CRIQ_8012d0b8
1485,CBRLS Kumara,4751caa3,Lahiru Kumara,Charith Bandara Rajapaksha Lahiru Sampath Kumara,CRIQ_4751caa3
1496,D Madushanka,de7d833e,Dilshan Madushanka,Lokumarakkalage Dilshan Madushanka,CRIQ_de7d833e
1497,DM de Silva,7d608e12,Dhananjaya de Silva,Dhananjaya Maduranga de Silva,CRIQ_7d608e12
1499,DN Wellalage,736123bb,Dunith Wellalage,Dunith Nethmika Wellalage,CRIQ_736123bb
1500,DPMD Jayawardene,d18f9182,Mahela Jayawardene,Denagamage Proboth Mahela de Silva Jayawardene,CRIQ_d18f9182
1505,FDM Karunaratne,400e2d4d,Dimuth Karunaratne,Frank Dimuth Madushanka Karunaratne,CRIQ_400e2d4d
1509,HDRL Thirimanne,c5928dec,Lahiru Thirimanne,Hettiarachchige Don Rumesh Lahiru Thirimanne,CRIQ_c5928dec


In [97]:
print("Proposed mappings:", len(sl_batch_1_candidates))

print(
    "Missing Crickrida names:",
    sl_batch_1_candidates["crickrida_name"].isna().sum()
)

print(
    "Missing full names:",
    sl_batch_1_candidates["full_name"].isna().sum()
)

print(
    "Duplicate Crickrida names:",
    sl_batch_1_candidates["crickrida_name"].duplicated().sum()
)

print(
    "Duplicate Cricsheet IDs:",
    sl_batch_1_candidates["player_id"].duplicated().sum()
)

print(
    "Duplicate CricketIQ IDs:",
    sl_batch_1_candidates["cricketiq_player_id"].duplicated().sum()
)

print(
    "Existing ID conflicts:",
    player_identity_map_updated["player_id"].isin(
        sl_batch_1_candidates["player_id"]
    ).sum()
)

print(
    "Existing name conflicts:",
    player_identity_map_updated["crickrida_name"].isin(
        sl_batch_1_candidates["crickrida_name"]
    ).sum()
)

Proposed mappings: 20
Missing Crickrida names: 0
Missing full names: 0
Duplicate Crickrida names: 0
Duplicate Cricsheet IDs: 0
Duplicate CricketIQ IDs: 0
Existing ID conflicts: 0
Existing name conflicts: 0


In [98]:
new_sl_rows = []

for _, candidate in sl_batch_1_candidates.iterrows():

    player_id = candidate["player_id"]
    crickrida_name = candidate["crickrida_name"]
    cricsheet_name = candidate["cricsheet_name"]
    full_name = candidate["full_name"]

    new_sl_rows.append({
        "cricketiq_player_id": f"CRIQ_{player_id}",
        "player_id": player_id,
        "cricsheet_name": cricsheet_name,
        "crickrida_name": crickrida_name,
        "normalized_name": normalize_identity(crickrida_name),
        "match_status": "verified_alias",
        "crickrida_player_name": crickrida_name,
        "cricsheet_player_name": cricsheet_name,
        "cricsheet_player_id": player_id,
        "mapping_method": "verified_sri_lanka_full_name_and_performance_review",
        "confidence": "high",
        "notes": (
            f"Full name: {full_name}. "
            "Cricsheet initials and surname match the full-name components. "
            "Batting and bowling performance records inspected "
            "where available; player names are consistent."
        )
    })

new_sl_rows_df = pd.DataFrame(
    new_sl_rows,
    columns=player_identity_map_updated.columns
)

player_identity_map_updated = pd.concat(
    [player_identity_map_updated, new_sl_rows_df],
    ignore_index=True
)

print("New mappings added:", len(new_sl_rows_df))
print("Updated identity map shape:", player_identity_map_updated.shape)

New mappings added: 20
Updated identity map shape: (295, 12)


In [100]:
print("cr_player_history columns:")
print(cr_player_history.columns.tolist())

print("\nSample records:")
display(cr_player_history.head(10))

cr_player_history columns:
['team', 'normalized_name', 'player_name', 'crickrida_first_date', 'crickrida_last_date', 'crickrida_matches']

Sample records:


,team,normalized_name,player_name,crickrida_first_date,crickrida_last_date,crickrida_matches
0,Afghanistan,abdulmalik,Abdul Malik,2024-09-22,2024-12-21,3
1,Afghanistan,abdulrahmanafghanistan2023,"Abdul Rahman (Afghanistan, 2023)",2023-07-11,2023-08-24,3
2,Afghanistan,afsarzazai,Afsar Zazai,2015-01-10,2017-06-11,9
3,Afghanistan,aftabalam,Aftab Alam,2014-07-22,2019-06-22,22
4,Afghanistan,amghazanfar,AM Ghazanfar,2024-03-07,2026-10-23,24
5,Afghanistan,amirhamza,Amir Hamza,2012-02-10,2017-06-11,25
6,Afghanistan,asgharafghan,Asghar Afghan,2010-07-03,2021-01-26,77
7,Afghanistan,azmatullahomarzai,Azmatullah Omarzai,2021-01-21,2026-10-23,49
8,Afghanistan,bashirahmad,Bashir Ahmad,2025-10-08,2025-10-11,2
9,Afghanistan,bilalsami,Bilal Sami,2024-12-21,2026-10-23,5


In [101]:
sl_final_candidates = [
    {
        "crickrida_name": "Ajantha Mendis",
        "cricsheet_name": "BAW Mendis",
        "player_id": "ea0cdc12",
        "full_name": "Balapuwaduge Ajantha Winslow Mendis"
    },
    {
        "crickrida_name": "Asitha Fernando",
        "cricsheet_name": "AM Fernando",
        "player_id": "de3d549a",
        "full_name": "Asitha Madusanka Fernando"
    },
    {
        "crickrida_name": "Avishka Fernando",
        "cricsheet_name": "WIA Fernando",
        "player_id": "9550470d",
        "full_name": "Weerahandige Inol Avishka Fernando"
    },
    {
        "crickrida_name": "Jeevan Mendis",
        "cricsheet_name": "BMAJ Mendis",
        "player_id": "9868bc75",
        "full_name": "Balapuwaduge Manukulasuriya Amith Jeevan Mendis"
    },
    {
        "crickrida_name": "Kusal Mendis",
        "cricsheet_name": "BKG Mendis",
        "player_id": "5d1e7582",
        "full_name": "Balapuwaduge Kusal Gimhan Mendis"
    },
    {
        "crickrida_name": "Kusal Perera",
        "cricsheet_name": "MDKJ Perera",
        "player_id": "f21043a5",
        "full_name": "Mathurage Don Kusal Janith Perera"
    },
    {
        "crickrida_name": "Matheesha Pathirana",
        "cricsheet_name": "HPMAV Pathirana",
        "player_id": "64839cb3",
        "full_name": "Heenatigala Pathiranage Matheesha Anushal Vihanga Pathirana"
    },
    {
        "crickrida_name": "Nuwan Pradeep",
        "cricsheet_name": "ANPR Fernando",
        "player_id": "2af838ee",
        "full_name": "Aththachchi Nuwan Pradeep Roshan Fernando"
    }
]

sl_final_candidates_df = pd.DataFrame(sl_final_candidates)

sl_final_candidates_df["cricketiq_player_id"] = (
    "CRIQ_" + sl_final_candidates_df["player_id"]
)

display(sl_final_candidates_df)

,crickrida_name,cricsheet_name,player_id,full_name,cricketiq_player_id
0,Ajantha Mendis,BAW Mendis,ea0cdc12,Balapuwaduge Ajantha Winslow Mendis,CRIQ_ea0cdc12
1,Asitha Fernando,AM Fernando,de3d549a,Asitha Madusanka Fernando,CRIQ_de3d549a
2,Avishka Fernando,WIA Fernando,9550470d,Weerahandige Inol Avishka Fernando,CRIQ_9550470d
3,Jeevan Mendis,BMAJ Mendis,9868bc75,Balapuwaduge Manukulasuriya Amith Jeevan Mendis,CRIQ_9868bc75
4,Kusal Mendis,BKG Mendis,5d1e7582,Balapuwaduge Kusal Gimhan Mendis,CRIQ_5d1e7582
5,Kusal Perera,MDKJ Perera,f21043a5,Mathurage Don Kusal Janith Perera,CRIQ_f21043a5
6,Matheesha Pathirana,HPMAV Pathirana,64839cb3,Heenatigala Pathiranage Matheesha Anushal Viha...,CRIQ_64839cb3
7,Nuwan Pradeep,ANPR Fernando,2af838ee,Aththachchi Nuwan Pradeep Roshan Fernando,CRIQ_2af838ee


In [102]:
final_candidate_ids = sl_final_candidates_df["player_id"].tolist()

final_candidate_names = sl_final_candidates_df["crickrida_name"].tolist()

print("EXISTING IDENTITY MAP RECORDS")

display(
    player_identity_map_updated[
        player_identity_map_updated["player_id"].isin(final_candidate_ids)
        |
        player_identity_map_updated["crickrida_name"].isin(final_candidate_names)
    ][[
        "cricketiq_player_id",
        "player_id",
        "crickrida_name",
        "cricsheet_name",
        "match_status",
        "mapping_method",
        "confidence"
    ]].to_string(index=False)
)

EXISTING IDENTITY MAP RECORDS


'cricketiq_player_id player_id   crickrida_name cricsheet_name   match_status                                 mapping_method confidence\n      CRIQ_ea0cdc12  ea0cdc12   Ajantha Mendis     BAW Mendis verified_alias verified_sri_lanka_name_and_performance_review       high\n      CRIQ_9868bc75  9868bc75    Jeevan Mendis    BMAJ Mendis verified_alias verified_sri_lanka_name_and_performance_review       high\n      CRIQ_5d1e7582  5d1e7582     Kusal Mendis     BKG Mendis verified_alias verified_sri_lanka_name_and_performance_review       high\n      CRIQ_de3d549a  de3d549a  Asitha Fernando    AM Fernando verified_alias verified_sri_lanka_name_and_performance_review       high\n      CRIQ_9550470d  9550470d Avishka Fernando   WIA Fernando verified_alias verified_sri_lanka_name_and_performance_review       high\n      CRIQ_f21043a5  f21043a5     Kusal Perera    MDKJ Perera verified_alias            verified_sri_lanka_full_name_review       high'

In [103]:
sl_last_two = {
    "Matheesha Pathirana": {
        "player_id": "64839cb3",
        "cricsheet_name": "HPMAV Pathirana"
    },
    "Nuwan Pradeep": {
        "player_id": "2af838ee",
        "cricsheet_name": "ANPR Fernando"
    }
}

for player_name, details in sl_last_two.items():

    player_id = details["player_id"]

    batting = player_batting_updated[
        player_batting_updated["batter_id"].astype(str) == player_id
    ]

    bowling = player_bowling_updated[
        player_bowling_updated["bowler_id"].astype(str) == player_id
    ]

    print(f"\n{'=' * 55}")
    print("Player:", player_name)
    print("Cricsheet name:", details["cricsheet_name"])
    print("Cricsheet ID:", player_id)

    print("\nBatting records:", len(batting))
    print("Unique batting matches:", batting["match_id"].nunique())
    print("Batting names:", batting["batter"].dropna().unique())

    print("\nBowling records:", len(bowling))
    print("Unique bowling matches:", bowling["match_id"].nunique())
    print("Bowling names:", bowling["bowler"].dropna().unique())


Player: Matheesha Pathirana
Cricsheet name: HPMAV Pathirana
Cricsheet ID: 64839cb3

Batting records: 5
Unique batting matches: 5
Batting names: ['M Pathirana']

Bowling records: 10
Unique bowling matches: 10
Bowling names: ['M Pathirana']

Player: Nuwan Pradeep
Cricsheet name: ANPR Fernando
Cricsheet ID: 2af838ee

Batting records: 19
Unique batting matches: 19
Batting names: ['N Pradeep']

Bowling records: 45
Unique bowling matches: 45
Bowling names: ['N Pradeep']


In [104]:
last_two_ids = ["64839cb3", "2af838ee"]

print(
    player_identity_map_updated[
        player_identity_map_updated["player_id"].isin(last_two_ids)
    ].to_string(index=False)
)

print(
    "Duplicate CricketIQ IDs in current map:",
    player_identity_map_updated["cricketiq_player_id"].duplicated().sum()
)

Empty DataFrame
Columns: [cricketiq_player_id, player_id, cricsheet_name, crickrida_name, normalized_name, match_status, crickrida_player_name, cricsheet_player_name, cricsheet_player_id, mapping_method, confidence, notes]
Index: []
Duplicate CricketIQ IDs in current map: 0


In [105]:
sl_last_two_rows = [
    {
        "cricketiq_player_id": "CRIQ_64839cb3",
        "player_id": "64839cb3",
        "cricsheet_name": "HPMAV Pathirana",
        "crickrida_name": "Matheesha Pathirana",
        "normalized_name": normalize_identity("Matheesha Pathirana"),
        "match_status": "verified_alias",
        "crickrida_player_name": "Matheesha Pathirana",
        "cricsheet_player_name": "HPMAV Pathirana",
        "cricsheet_player_id": "64839cb3",
        "mapping_method": "verified_sri_lanka_full_name_and_performance_review",
        "confidence": "high",
        "notes": (
            "Full name: Heenatigala Pathiranage Matheesha Anushal Vihanga Pathirana. "
            "Full-name initials and surname match. "
            "Consistent Cricsheet identity in batting and bowling records."
        )
    },
    {
        "cricketiq_player_id": "CRIQ_2af838ee",
        "player_id": "2af838ee",
        "cricsheet_name": "ANPR Fernando",
        "crickrida_name": "Nuwan Pradeep",
        "normalized_name": normalize_identity("Nuwan Pradeep"),
        "match_status": "verified_alias",
        "crickrida_player_name": "Nuwan Pradeep",
        "cricsheet_player_name": "ANPR Fernando",
        "cricsheet_player_id": "2af838ee",
        "mapping_method": "verified_sri_lanka_full_name_and_performance_review",
        "confidence": "high",
        "notes": (
            "Full name: Aththachchi Nuwan Pradeep Roshan Fernando. "
            "Full-name initials and final surname match. "
            "Consistent Cricsheet identity in batting and bowling records."
        )
    }
]

sl_last_two_rows_df = pd.DataFrame(
    sl_last_two_rows,
    columns=player_identity_map_updated.columns
)

player_identity_map_updated = pd.concat(
    [player_identity_map_updated, sl_last_two_rows_df],
    ignore_index=True
)

print("New mappings added:", len(sl_last_two_rows_df))
print("Updated identity map shape:", player_identity_map_updated.shape)

New mappings added: 2
Updated identity map shape: (297, 12)


In [106]:
display(
    player_identity_map_updated[
        player_identity_map_updated["player_id"].isin(
            ["64839cb3", "2af838ee"]
        )
    ].to_string(index=False)
)

'cricketiq_player_id player_id  cricsheet_name      crickrida_name    normalized_name   match_status crickrida_player_name cricsheet_player_name cricsheet_player_id                                      mapping_method confidence                                                                                                                                                                       notes\n      CRIQ_64839cb3  64839cb3 HPMAV Pathirana Matheesha Pathirana matheeshapathirana verified_alias   Matheesha Pathirana       HPMAV Pathirana            64839cb3 verified_sri_lanka_full_name_and_performance_review       high Full name: Heenatigala Pathiranage Matheesha Anushal Vihanga Pathirana. Full-name initials and surname match. Consistent Cricsheet identity in batting and bowling records.\n      CRIQ_2af838ee  2af838ee   ANPR Fernando       Nuwan Pradeep       nuwanpradeep verified_alias         Nuwan Pradeep         ANPR Fernando            2af838ee verified_sri_lanka_full_name_and_pe

In [107]:
player_identity_map_updated.to_csv(
    "player_identity_map_after_sri_lanka.csv",
    index=False
)

print("Sri Lanka identity map checkpoint saved.")
print("Total records:", len(player_identity_map_updated))

Sri Lanka identity map checkpoint saved.
Total records: 297


In [108]:
print("Crickrida history columns:")
print(cr_player_history.columns.tolist())

display(cr_player_history.head())

Crickrida history columns:
['team', 'normalized_name', 'player_name', 'crickrida_first_date', 'crickrida_last_date', 'crickrida_matches']


,team,normalized_name,player_name,crickrida_first_date,crickrida_last_date,crickrida_matches
0,Afghanistan,abdulmalik,Abdul Malik,2024-09-22,2024-12-21,3
1,Afghanistan,abdulrahmanafghanistan2023,"Abdul Rahman (Afghanistan, 2023)",2023-07-11,2023-08-24,3
2,Afghanistan,afsarzazai,Afsar Zazai,2015-01-10,2017-06-11,9
3,Afghanistan,aftabalam,Aftab Alam,2014-07-22,2019-06-22,22
4,Afghanistan,amghazanfar,AM Ghazanfar,2024-03-07,2026-10-23,24


In [109]:
# Extract Australia player history
aus_crickrida_history = cr_player_history[
    cr_player_history["team"].str.strip().str.casefold() == "australia"
].copy()

print("Total Australian Crickrida players:", len(aus_crickrida_history))

display(
    aus_crickrida_history[
        [
            "player_name",
            "normalized_name",
            "crickrida_first_date",
            "crickrida_last_date",
            "crickrida_matches"
        ]
    ].sort_values("player_name").to_string(index=False)
)

Total Australian Crickrida players: 29


'        player_name   normalized_name crickrida_first_date crickrida_last_date  crickrida_matches\n        Aaron Finch        aaronfinch           2015-03-04          2019-06-01                  2\n         Adam Zampa         adamzampa           2019-06-01          2025-02-28                  3\n      Ben Dwarshuis      bendwarshuis           2025-02-28          2025-02-28                  1\n        Brad Haddin        bradhaddin           2015-03-04          2015-03-04                  1\n       David Hussey       davidhussey           2012-08-25          2012-08-25                  1\n       David Warner       davidwarner           2012-08-25          2023-11-07                  4\n      George Bailey      georgebailey           2012-08-25          2012-08-25                  1\n      Glenn Maxwell      glennmaxwell           2012-08-25          2025-02-28                  5\n     James Faulkner     jamesfaulkner           2015-03-04          2015-03-04                  1\n    James

In [110]:
# Get all mapped Crickrida names, excluding source-only records
aus_mapped_names = set(
    player_identity_map_updated.loc[
        player_identity_map_updated["crickrida_name"].notna()
        & (player_identity_map_updated["match_status"] != "crickrida_only"),
        "crickrida_name"
    ].apply(normalize_identity)
)

# Normalize Australian history names
aus_crickrida_history["normalized_check"] = (
    aus_crickrida_history["player_name"].apply(normalize_identity)
)

# Separate mapped and unresolved players
aus_already_mapped = aus_crickrida_history[
    aus_crickrida_history["normalized_check"].isin(aus_mapped_names)
].copy()

aus_unresolved = aus_crickrida_history[
    ~aus_crickrida_history["normalized_check"].isin(aus_mapped_names)
].copy()

print("Australia total:", len(aus_crickrida_history))
print("Already mapped:", len(aus_already_mapped))
print("Unresolved:", len(aus_unresolved))

print("\nALREADY MAPPED PLAYERS")
display(aus_already_mapped[["player_name"]].to_string(index=False))

print("\nUNRESOLVED PLAYERS")
display(
    aus_unresolved[
        [
            "player_name",
            "crickrida_first_date",
            "crickrida_last_date",
            "crickrida_matches"
        ]
    ].sort_values("player_name").to_string(index=False)
)

Australia total: 29
Already mapped: 3
Unresolved: 26

ALREADY MAPPED PLAYERS


'  player_name\nBen Dwarshuis\n David Warner\nMatthew Short'


UNRESOLVED PLAYERS


'        player_name crickrida_first_date crickrida_last_date  crickrida_matches\n        Aaron Finch           2015-03-04          2019-06-01                  2\n         Adam Zampa           2019-06-01          2025-02-28                  3\n        Brad Haddin           2015-03-04          2015-03-04                  1\n       David Hussey           2012-08-25          2012-08-25                  1\n      George Bailey           2012-08-25          2012-08-25                  1\n      Glenn Maxwell           2012-08-25          2025-02-28                  5\n     James Faulkner           2015-03-04          2015-03-04                  1\n    James Pattinson           2012-08-25          2012-08-25                  1\n     Josh Hazlewood           2015-03-04          2023-11-07                  2\n        Josh Inglis           2023-11-07          2023-11-07                  1\n     Marcus Stoinis           2019-06-01          2023-11-07                  2\n Marnus Labuschagne        

In [112]:
# 1. Extract Australian Cricsheet players
aus_cricsheet_history = cs_player_history[
    cs_player_history["team"].str.strip().str.casefold() == "australia"
].copy()

# 2. Helper functions
def get_last_name(name):
    parts = str(name).strip().split()
    return parts[-1].casefold() if parts else ""

def get_first_initial(name):
    parts = str(name).strip().split()
    return parts[0][0].casefold() if parts else ""

def get_cricsheet_initial_token(name):
    parts = str(name).strip().split()
    return parts[0].casefold() if parts else ""

# 3. Convert the correct date columns
cr_date_columns = [
    "crickrida_first_date",
    "crickrida_last_date"
]

cs_date_columns = [
    "cricsheet_first_date",
    "cricsheet_last_date"
]

for col in cr_date_columns:
    aus_unresolved[col] = pd.to_datetime(
        aus_unresolved[col], errors="coerce"
    )

for col in cs_date_columns:
    aus_cricsheet_history[col] = pd.to_datetime(
        aus_cricsheet_history[col], errors="coerce"
    )

# 4. Generate same-surname candidates
aus_candidate_rows = []

for _, cr in aus_unresolved.iterrows():

    cr_name = cr["player_name"]
    cr_surname = get_last_name(cr_name)
    cr_initial = get_first_initial(cr_name)

    for _, cs in aus_cricsheet_history.iterrows():

        cs_name = cs["player_name"]
        cs_surname = get_last_name(cs_name)
        cs_initial_token = get_cricsheet_initial_token(cs_name)

        if cr_surname != cs_surname:
            continue

        initial_match = cr_initial in cs_initial_token

        cr_start = cr["crickrida_first_date"]
        cr_end = cr["crickrida_last_date"]

        cs_start = cs["cricsheet_first_date"]
        cs_end = cs["cricsheet_last_date"]

        date_overlap = (
            pd.notna(cr_start)
            and pd.notna(cr_end)
            and pd.notna(cs_start)
            and pd.notna(cs_end)
            and cr_start <= cs_end
            and cs_start <= cr_end
        )

        aus_candidate_rows.append({
            "crickrida_name": cr_name,
            "cricsheet_name": cs_name,
            "cricsheet_player_id": cs["source_player_id"],
            "initial_match": initial_match,
            "date_overlap": date_overlap,
            "crickrida_first_date": cr_start,
            "crickrida_last_date": cr_end,
            "cricsheet_first_date": cs_start,
            "cricsheet_last_date": cs_end,
            "cricsheet_matches": cs["cricsheet_matches"]
        })

aus_candidates_df = pd.DataFrame(aus_candidate_rows)

print("Australian Cricsheet players:", len(aus_cricsheet_history))
print("Same-surname candidate pairs:", len(aus_candidates_df))

if not aus_candidates_df.empty:
    display(
        aus_candidates_df.sort_values(
            ["crickrida_name", "initial_match", "date_overlap"],
            ascending=[True, False, False]
        ).to_string(index=False)
    )
else:
    print("No same-surname candidate pairs found.")

Australian Cricsheet players: 127
Same-surname candidate pairs: 31


'     crickrida_name  cricsheet_name cricsheet_player_id  initial_match  date_overlap crickrida_first_date crickrida_last_date cricsheet_first_date cricsheet_last_date  cricsheet_matches\n        Aaron Finch        AJ Finch            b8d490fd           True          True           2015-03-04          2019-06-01           2013-01-11          2022-09-11                139\n         Adam Zampa         A Zampa            14f96089           True          True           2019-06-01          2025-02-28           2016-02-06          2026-09-30                124\n        Brad Haddin       BJ Haddin            b0c772ee           True          True           2015-03-04          2015-03-04           2004-01-29          2015-03-26                107\n       David Hussey       DJ Hussey            fd835ab3           True          True           2012-08-25          2012-08-25           2008-07-04          2013-01-23                 62\n       David Hussey      MEK Hussey            48fd7349         

In [113]:
aus_strong_candidates = aus_candidates_df[
    aus_candidates_df["initial_match"]
    & aus_candidates_df["date_overlap"]
].copy()

# Keep only players with exactly one matching candidate
candidate_counts = aus_strong_candidates.groupby(
    "crickrida_name"
)["cricsheet_player_id"].nunique()

aus_unique_candidates = aus_strong_candidates[
    aus_strong_candidates["crickrida_name"].isin(
        candidate_counts[candidate_counts == 1].index
    )
].copy()

print("Unique candidate players:", aus_unique_candidates["crickrida_name"].nunique())

aus_performance_rows = []

for _, candidate in aus_unique_candidates.iterrows():

    player_id = candidate["cricsheet_player_id"]

    batting = player_batting_updated[
        player_batting_updated["batter_id"].astype(str) == str(player_id)
    ]

    bowling = player_bowling_updated[
        player_bowling_updated["bowler_id"].astype(str) == str(player_id)
    ]

    aus_performance_rows.append({
        "crickrida_name": candidate["crickrida_name"],
        "cricsheet_name": candidate["cricsheet_name"],
        "player_id": player_id,
        "batting_records": len(batting),
        "bowling_records": len(bowling),
        "batting_names": ", ".join(batting["batter"].dropna().unique()),
        "bowling_names": ", ".join(bowling["bowler"].dropna().unique())
    })

aus_performance_df = pd.DataFrame(aus_performance_rows)

display(aus_performance_df.to_string(index=False))

Unique candidate players: 26


'     crickrida_name  cricsheet_name player_id  batting_records  bowling_records   batting_names   bowling_names\n        Aaron Finch        AJ Finch  b8d490fd              139               21        AJ Finch        AJ Finch\n         Adam Zampa         A Zampa  14f96089               67              124         A Zampa         A Zampa\n        Brad Haddin       BJ Haddin  b0c772ee              107                0       BJ Haddin                \n       David Hussey       DJ Hussey  fd835ab3               59               37       DJ Hussey       DJ Hussey\n      George Bailey       GJ Bailey  5056011d               83                0       GJ Bailey                \n      Glenn Maxwell      GJ Maxwell  b681e71e              132              115      GJ Maxwell      GJ Maxwell\n     James Faulkner     JP Faulkner  808f425a               51               65     JP Faulkner     JP Faulkner\n    James Pattinson    JL Pattinson  f752db61                7               14    JL Pattinson

In [114]:
aus_candidate_ids = (
    aus_unique_candidates["cricsheet_player_id"]
    .astype(str)
    .tolist()
)

aus_candidate_names = (
    aus_unique_candidates["crickrida_name"]
    .tolist()
)

print("EXISTING CRICSHEET ID CONFLICTS")

display(
    player_identity_map_updated[
        player_identity_map_updated["player_id"].astype(str).isin(
            aus_candidate_ids
        )
    ].to_string(index=False)
)

print("\nEXISTING CRICKRIDA NAME CONFLICTS")

display(
    player_identity_map_updated[
        player_identity_map_updated["crickrida_name"].isin(
            aus_candidate_names
        )
    ].to_string(index=False)
)

EXISTING CRICSHEET ID CONFLICTS


'Empty DataFrame\nColumns: [cricketiq_player_id, player_id, cricsheet_name, crickrida_name, normalized_name, match_status, crickrida_player_name, cricsheet_player_name, cricsheet_player_id, mapping_method, confidence, notes]\nIndex: []'


EXISTING CRICKRIDA NAME CONFLICTS


'Empty DataFrame\nColumns: [cricketiq_player_id, player_id, cricsheet_name, crickrida_name, normalized_name, match_status, crickrida_player_name, cricsheet_player_name, cricsheet_player_id, mapping_method, confidence, notes]\nIndex: []'

In [115]:
aus_full_names = {
    "Aaron Finch": "Aaron James Finch",
    "Adam Zampa": "Adam Zampa",
    "Brad Haddin": "Bradley James Haddin",
    "David Hussey": "David John Hussey",
    "George Bailey": "George John Bailey",
    "Glenn Maxwell": "Glenn James Maxwell",
    "James Faulkner": "James Peter Faulkner",
    "James Pattinson": "James Lee Pattinson",
    "Josh Hazlewood": "Josh Reginald Hazlewood",
    "Josh Inglis": "Joshua Patrick Inglis",
    "Marcus Stoinis": "Marcus Peter Stoinis",
    "Marnus Labuschagne": "Marnus Labuschagne",
    "Matthew Wade": "Matthew Scott Wade",
    "Michael Clarke": "Michael John Clarke",
    "Michael Hussey": "Michael Edward Killeen Hussey",
    "Mitchell Johnson": "Mitchell Guy Johnson",
    "Mitchell Marsh": "Mitchell Ross Marsh",
    "Mitchell Starc": "Mitchell Aaron Starc",
    "Nathan Coulter-Nile": "Nathan Mitchell Coulter-Nile",
    "Nathan Ellis": "Nathan Trevor Ellis",
    "Pat Cummins": "Patrick James Cummins",
    "Spencer Johnson": "Spencer Henry Johnson",
    "Steve Smith": "Steven Peter Devereux Smith",
    "Travis Head": "Travis Michael Head",
    "Usman Khawaja": "Usman Tariq Khawaja",
    "Xavier Doherty": "Xavier John Doherty"
}

In [116]:
import pandas as pd

# Identify the correct Cricsheet ID column
if "cricsheet_player_id" in aus_unique_candidates.columns:
    id_column = "cricsheet_player_id"
elif "player_id" in aus_unique_candidates.columns:
    id_column = "player_id"
else:
    raise ValueError("Cricsheet player ID column not found.")

# Verify all unresolved players have full names
candidate_names = set(aus_unique_candidates["crickrida_name"])
provided_names = set(aus_full_names)

assert candidate_names == provided_names, (
    f"Missing: {candidate_names - provided_names}, "
    f"Extra: {provided_names - candidate_names}"
)

# Build staged records
aus_verified_mapping_rows = []

for _, row in aus_unique_candidates.iterrows():

    common_name = row["crickrida_name"]
    cricsheet_name = row["cricsheet_name"]
    player_id = str(row[id_column])
    full_name = aus_full_names[common_name]

    aus_verified_mapping_rows.append({
        "cricketiq_player_id": f"CRIQ_{player_id}",
        "player_id": player_id,
        "cricsheet_name": cricsheet_name,
        "crickrida_name": common_name,
        "normalized_name": normalize_identity(common_name),
        "match_status": "verified_alias",
        "crickrida_player_name": common_name,
        "cricsheet_player_name": cricsheet_name,
        "cricsheet_player_id": player_id,
        "mapping_method": "verified_full_name_and_career_review",
        "confidence": "high",
        "notes": (
            f"Full name: {full_name}. "
            f"Matched using full-name initials, career overlap, "
            f"and consistent performance records."
        )
    })

aus_verified_mapping_df = pd.DataFrame(
    aus_verified_mapping_rows,
    columns=player_identity_map_updated.columns
)

print("Staged records:", len(aus_verified_mapping_df))
print("Unique CricketIQ IDs:",
      aus_verified_mapping_df["cricketiq_player_id"].nunique())
print("Unique Cricsheet IDs:",
      aus_verified_mapping_df["player_id"].nunique())

display(aus_verified_mapping_df.to_string(index=False))

Staged records: 26
Unique CricketIQ IDs: 26
Unique Cricsheet IDs: 26


'cricketiq_player_id player_id  cricsheet_name      crickrida_name   normalized_name   match_status crickrida_player_name cricsheet_player_name cricsheet_player_id                       mapping_method confidence                                                                                                                           notes\n      CRIQ_b8d490fd  b8d490fd        AJ Finch         Aaron Finch        aaronfinch verified_alias           Aaron Finch              AJ Finch            b8d490fd verified_full_name_and_career_review       high             Full name: Aaron James Finch. Matched using full-name initials, career overlap, and consistent performance records.\n      CRIQ_14f96089  14f96089         A Zampa          Adam Zampa         adamzampa verified_alias            Adam Zampa               A Zampa            14f96089 verified_full_name_and_career_review       high                    Full name: Adam Zampa. Matched using full-name initials, career overlap, and consistent p

In [117]:
# Preserve the current map before modification
player_identity_map_before_australia = (
    player_identity_map_updated.copy(deep=True)
)

before_count = len(player_identity_map_updated)

# Append the 26 verified Australian mappings
player_identity_map_updated = pd.concat(
    [
        player_identity_map_updated,
        aus_verified_mapping_df
    ],
    ignore_index=True
)

after_count = len(player_identity_map_updated)

print("Before insertion:", before_count)
print("Records inserted:", len(aus_verified_mapping_df))
print("After insertion:", after_count)

assert after_count == before_count + 26

print("\nAustralian mappings inserted successfully.")

Before insertion: 297
Records inserted: 26
After insertion: 323

Australian mappings inserted successfully.


In [118]:
# Check global ID uniqueness
assert player_identity_map_updated["cricketiq_player_id"].is_unique

# Check all 26 Australian records exist
inserted_ids = set(aus_verified_mapping_df["cricketiq_player_id"])

updated_ids = set(player_identity_map_updated["cricketiq_player_id"])

assert inserted_ids.issubset(updated_ids)

# Check statuses
aus_inserted = player_identity_map_updated[
    player_identity_map_updated["cricketiq_player_id"].isin(
        inserted_ids
    )
]

assert len(aus_inserted) == 26
assert (aus_inserted["match_status"] == "verified_alias").all()
assert (aus_inserted["confidence"] == "high").all()

print("All 26 Australian mappings verified.")
print("Updated identity map shape:", player_identity_map_updated.shape)

All 26 Australian mappings verified.
Updated identity map shape: (323, 12)


In [119]:
# All Australian Crickrida source-history names
aus_source_names = set(
    cr_player_history.loc[
        cr_player_history["team"].str.lower() == "australia",
        "normalized_name"
    ].dropna()
)

# Australian identities now represented in the identity map
aus_map = player_identity_map_updated[
    player_identity_map_updated["crickrida_name"].isin(
        aus_crickrida_history["player_name"]
    )
].copy()

aus_shared = aus_map[
    aus_map["player_id"].notna() &
    aus_map["cricsheet_name"].notna()
]

aus_crickrida_only = aus_map[
    aus_map["match_status"] == "crickrida_only"
]

print("AUSTRALIA AUDIT")
print("----------------------------")
print("Crickrida source players:", len(aus_crickrida_history))
print("Shared/mapped players:", len(aus_shared))
print("Crickrida-only players:", len(aus_crickrida_only))
print("Unresolved players:",
      len(aus_unresolved) - len(aus_verified_mapping_df))

display(
    aus_shared[
        ["crickrida_name", "cricsheet_name",
         "player_id", "match_status", "confidence"]
    ].sort_values("crickrida_name").to_string(index=False)
)

AUSTRALIA AUDIT
----------------------------
Crickrida source players: 29
Shared/mapped players: 29
Crickrida-only players: 0
Unresolved players: 0


'     crickrida_name  cricsheet_name player_id   match_status confidence\n        Aaron Finch        AJ Finch  b8d490fd verified_alias       high\n         Adam Zampa         A Zampa  14f96089 verified_alias       high\n      Ben Dwarshuis    BJ Dwarshuis  e1b9f3a9 verified_alias        NaN\n        Brad Haddin       BJ Haddin  b0c772ee verified_alias       high\n       David Hussey       DJ Hussey  fd835ab3 verified_alias       high\n       David Warner       DA Warner  dcce6f09 verified_alias        NaN\n      George Bailey       GJ Bailey  5056011d verified_alias       high\n      Glenn Maxwell      GJ Maxwell  b681e71e verified_alias       high\n     James Faulkner     JP Faulkner  808f425a verified_alias       high\n    James Pattinson    JL Pattinson  f752db61 verified_alias       high\n     Josh Hazlewood    JR Hazlewood  03806cf8 verified_alias       high\n        Josh Inglis       JP Inglis  989889ff verified_alias       high\n     Marcus Stoinis      MP Stoinis  d9273ee7 veri

In [120]:
# Build a normalized lookup of all Crickrida identities
mapped_crickrida_names = set(
    player_identity_map_updated["crickrida_name"]
    .dropna()
    .apply(normalize_identity)
)

# Generate country-level coverage
country_audit_rows = []

for country, group in cr_player_history.groupby("team"):

    country_players = (
        group[["normalized_name", "player_name"]]
        .drop_duplicates(subset="normalized_name")
    )

    total = len(country_players)

    mapped_mask = country_players["normalized_name"].isin(
        mapped_crickrida_names
    )

    mapped = int(mapped_mask.sum())
    unresolved = total - mapped

    country_audit_rows.append({
        "country": country,
        "total_crickrida_players": total,
        "mapped": mapped,
        "unresolved": unresolved,
        "coverage_percent": round(mapped / total * 100, 2)
        if total else 0
    })

country_audit_summary = pd.DataFrame(country_audit_rows)

country_audit_summary = country_audit_summary.sort_values(
    ["unresolved", "country"],
    ascending=[False, True]
).reset_index(drop=True)

display(country_audit_summary.to_string(index=False))

'     country  total_crickrida_players  mapped  unresolved  coverage_percent\n Afghanistan                       58       2          56              3.45\n     England                       28       6          22             21.43\n       India                       36      14          22             38.89\nSouth Africa                       26       4          22             15.38\n New Zealand                       23       2          21              8.70\n   Sri Lanka                       40      32           8             80.00\n     Ireland                       45      42           3             93.33\n   Australia                       29      29           0            100.00\n  Bangladesh                       51      51           0            100.00\n    Pakistan                       39      39           0            100.00\n West Indies                       27      27           0            100.00\n    Zimbabwe                       52      52           0            100.00

In [121]:
# England: Crickrida source history
eng_crickrida_history = (
    cr_player_history[
        cr_player_history["team"].str.lower() == "england"
    ]
    .drop_duplicates(subset="normalized_name")
    .copy()
)

# England: Cricsheet source history
eng_cricsheet_history = (
    cs_player_history[
        cs_player_history["team"].str.lower() == "england"
    ]
    .drop_duplicates(subset="normalized_name")
    .copy()
)

# Current England identities represented in the map
eng_crickrida_names = set(
    eng_crickrida_history["normalized_name"].dropna()
)

eng_map = player_identity_map_updated[
    player_identity_map_updated["normalized_name"].isin(
        eng_crickrida_names
    )
].copy()

# Names not currently represented by exact normalized-name matching
eng_mapped_names = set(
    eng_map["normalized_name"].dropna()
)

eng_unresolved = eng_crickrida_history[
    ~eng_crickrida_history["normalized_name"].isin(
        eng_mapped_names
    )
].copy()

print("ENGLAND SOURCE SUMMARY")
print("----------------------------")
print("Crickrida players:", len(eng_crickrida_history))
print("Cricsheet players:", len(eng_cricsheet_history))
print("Currently represented:", len(eng_map))
print("Names requiring investigation:", len(eng_unresolved))

print("\nCRICKRIDA ENGLAND PLAYERS")
display(
    eng_crickrida_history[
        ["player_name", "normalized_name",
         "crickrida_first_date", "crickrida_last_date",
         "crickrida_matches"]
    ].sort_values("player_name").to_string(index=False)
)

print("\nCRICSHEET ENGLAND PLAYERS")
display(
    eng_cricsheet_history[
        ["player_name", "source_player_id",
         "cricsheet_first_date", "cricsheet_last_date",
         "cricsheet_matches"]
    ].sort_values("player_name").to_string(index=False)
)

print("\nCURRENTLY UNREPRESENTED CRICKRIDA PLAYERS")
display(
    eng_unresolved[
        ["player_name", "normalized_name",
         "crickrida_first_date", "crickrida_last_date",
         "crickrida_matches"]
    ].sort_values("player_name").to_string(index=False)
)

ENGLAND SOURCE SUMMARY
----------------------------
Crickrida players: 28
Cricsheet players: 133
Currently represented: 6
Names requiring investigation: 22

CRICKRIDA ENGLAND PLAYERS


'     player_name normalized_name crickrida_first_date crickrida_last_date  crickrida_matches\n     Adil Rashid      adilrashid           2019-06-18          2025-02-26                  3\n      Alex Hales       alexhales           2015-03-13          2015-03-13                  1\n     Ben Duckett      benduckett           2025-02-26          2025-02-26                  1\n      Ben Stokes       benstokes           2019-06-18          2019-06-18                  1\n    Chris Jordan     chrisjordan           2015-03-13          2015-03-13                  1\n    Chris Woakes     chriswoakes           2019-06-18          2023-10-15                  2\n     Dawid Malan      dawidmalan           2023-10-15          2023-10-15                  1\n     Eoin Morgan      eoinmorgan           2019-06-18          2019-06-18                  1\n     Harry Brook      harrybrook           2023-10-15          2025-02-26                  2\n        Ian Bell         ianbell           2015-03-13      


CRICSHEET ENGLAND PLAYERS


'    player_name source_player_id cricsheet_first_date cricsheet_last_date  cricsheet_matches\n     A Flintoff         ddc0828d           2002-06-27          2009-04-03                 80\n      A McGrath         e8b5ea8d           2003-06-17          2004-09-05                 13\n      A Shahzad         a73321d3           2010-03-05          2011-03-11                 10\n   AAP Atkinson         70d57519           2023-09-08          2026-10-25                 19\n       AD Hales         92aeac25           2014-08-27          2019-03-02                 65\n AD Mascarenhas         3edb58fc           2007-07-01          2009-09-17                 20\n       AF Giles         a4af4849           2002-06-27          2005-07-12                 37\n       AG Wharf         2e0fcc58           2004-09-01          2005-02-13                 12\n     AGR Loudon         ee5aa838           2006-06-24          2006-06-24                  1\n     AJ Stewart         a8ce3491           2002-06-27      


CURRENTLY UNREPRESENTED CRICKRIDA PLAYERS


'   player_name normalized_name crickrida_first_date crickrida_last_date  crickrida_matches\n   Adil Rashid      adilrashid           2019-06-18          2025-02-26                  3\n    Alex Hales       alexhales           2015-03-13          2015-03-13                  1\n   Ben Duckett      benduckett           2025-02-26          2025-02-26                  1\n    Ben Stokes       benstokes           2019-06-18          2019-06-18                  1\n  Chris Jordan     chrisjordan           2015-03-13          2015-03-13                  1\n  Chris Woakes     chriswoakes           2019-06-18          2023-10-15                  2\n   Dawid Malan      dawidmalan           2023-10-15          2023-10-15                  1\n   Harry Brook      harrybrook           2023-10-15          2025-02-26                  2\n      Ian Bell         ianbell           2015-03-13          2015-03-13                  1\n  James Taylor     jamestaylor           2015-03-13          2015-03-13        

In [122]:
# Prepare date columns
eng_crickrida_history["crickrida_first_date"] = pd.to_datetime(
    eng_crickrida_history["crickrida_first_date"]
)
eng_crickrida_history["crickrida_last_date"] = pd.to_datetime(
    eng_crickrida_history["crickrida_last_date"]
)

eng_cricsheet_history["cricsheet_first_date"] = pd.to_datetime(
    eng_cricsheet_history["cricsheet_first_date"]
)
eng_cricsheet_history["cricsheet_last_date"] = pd.to_datetime(
    eng_cricsheet_history["cricsheet_last_date"]
)

candidate_rows = []

for _, cr in eng_unresolved.iterrows():

    cr_name = cr["player_name"]
    cr_tokens = cr_name.split()

    cr_first_initial = cr_tokens[0][0].lower()
    cr_surname = normalize_identity(cr_tokens[-1])

    for _, cs in eng_cricsheet_history.iterrows():

        cs_name = cs["player_name"]
        cs_tokens = cs_name.split()

        cs_initials = cs_tokens[0].lower()
        cs_surname = normalize_identity(cs_tokens[-1])

        # Same surname
        if cr_surname != cs_surname:
            continue

        # First initial comparison
        initial_match = (
            cr_first_initial == cs_initials[0]
        )

        # Career date overlap
        date_overlap = (
            cr["crickrida_first_date"]
            <= cs["cricsheet_last_date"]
            and
            cs["cricsheet_first_date"]
            <= cr["crickrida_last_date"]
        )

        candidate_rows.append({
            "crickrida_name": cr_name,
            "cricsheet_name": cs_name,
            "cricsheet_player_id": cs["source_player_id"],
            "initial_match": initial_match,
            "date_overlap": date_overlap,
            "crickrida_first_date": cr["crickrida_first_date"],
            "crickrida_last_date": cr["crickrida_last_date"],
            "cricsheet_first_date": cs["cricsheet_first_date"],
            "cricsheet_last_date": cs["cricsheet_last_date"],
            "cricsheet_matches": cs["cricsheet_matches"]
        })

eng_candidates_df = pd.DataFrame(candidate_rows)

if not eng_candidates_df.empty:
    eng_candidates_df = eng_candidates_df.sort_values(
        ["crickrida_name", "initial_match", "date_overlap"],
        ascending=[True, False, False]
    )

print("Total same-surname candidate pairs:", len(eng_candidates_df))

print("\nINITIAL + DATE OVERLAP CANDIDATES")

display(
    eng_candidates_df[
        eng_candidates_df["initial_match"] &
        eng_candidates_df["date_overlap"]
    ].to_string(index=False)
)

print("\nALL SAME-SURNAME CANDIDATES")

display(eng_candidates_df.to_string(index=False))

Total same-surname candidate pairs: 24

INITIAL + DATE OVERLAP CANDIDATES


'crickrida_name cricsheet_name cricsheet_player_id  initial_match  date_overlap crickrida_first_date crickrida_last_date cricsheet_first_date cricsheet_last_date  cricsheet_matches\n   Adil Rashid      AU Rashid            249d60c9           True          True           2019-06-18          2025-02-26           2009-08-27          2026-10-20                157\n    Alex Hales       AD Hales            92aeac25           True          True           2015-03-13          2015-03-13           2014-08-27          2019-03-02                 65\n   Ben Duckett     BM Duckett            5f26f677           True          True           2025-02-26          2025-02-26           2016-10-07          2026-10-25                 42\n    Ben Stokes      BA Stokes            e087956b           True          True           2019-06-18          2019-06-18           2011-08-25          2023-11-11                108\n  Chris Jordan      CJ Jordan            ffe699c0           True          True           2015-


ALL SAME-SURNAME CANDIDATES


'crickrida_name cricsheet_name cricsheet_player_id  initial_match  date_overlap crickrida_first_date crickrida_last_date cricsheet_first_date cricsheet_last_date  cricsheet_matches\n   Adil Rashid      AU Rashid            249d60c9           True          True           2019-06-18          2025-02-26           2009-08-27          2026-10-20                157\n    Alex Hales       AD Hales            92aeac25           True          True           2015-03-13          2015-03-13           2014-08-27          2019-03-02                 65\n   Ben Duckett     BM Duckett            5f26f677           True          True           2025-02-26          2025-02-26           2016-10-07          2026-10-25                 42\n    Ben Stokes      BA Stokes            e087956b           True          True           2019-06-18          2019-06-18           2011-08-25          2023-11-11                108\n  Chris Jordan      CJ Jordan            ffe699c0           True          True           2015-

In [123]:
# Keep only candidates passing both initial and date checks
eng_strong_candidates = eng_candidates_df[
    eng_candidates_df["initial_match"] &
    eng_candidates_df["date_overlap"]
].copy()

# Confirm candidate uniqueness
eng_candidate_counts = (
    eng_strong_candidates.groupby("crickrida_name")[
        "cricsheet_player_id"
    ].nunique()
)

print("Strong candidate rows:", len(eng_strong_candidates))
print("Unique Crickrida players:", eng_candidate_counts.size)
print("Players with multiple strong candidates:",
      eng_candidate_counts[eng_candidate_counts > 1])

assert len(eng_strong_candidates) == 19
assert eng_candidate_counts.eq(1).all()

# Inspect performance records
eng_performance_rows = []

for _, row in eng_strong_candidates.iterrows():

    pid = str(row["cricsheet_player_id"])

    batting = player_batting_updated[
        player_batting_updated["batter_id"].astype(str) == pid
    ]

    bowling = player_bowling_updated[
        player_bowling_updated["bowler_id"].astype(str) == pid
    ]

    eng_performance_rows.append({
        "crickrida_name": row["crickrida_name"],
        "cricsheet_name": row["cricsheet_name"],
        "player_id": pid,
        "batting_records": len(batting),
        "bowling_records": len(bowling),
        "batting_names": ", ".join(
            sorted(batting["batter"].dropna().unique())
        ),
        "bowling_names": ", ".join(
            sorted(bowling["bowler"].dropna().unique())
        )
    })

eng_performance_check = pd.DataFrame(eng_performance_rows)

display(
    eng_performance_check.sort_values(
        "crickrida_name"
    ).to_string(index=False)
)

Strong candidate rows: 19
Unique Crickrida players: 19
Players with multiple strong candidates: Series([], Name: cricsheet_player_id, dtype: int64)


'crickrida_name cricsheet_name player_id  batting_records  bowling_records batting_names bowling_names\n   Adil Rashid      AU Rashid  249d60c9               83              155     AU Rashid     AU Rashid\n    Alex Hales       AD Hales  92aeac25               65                0      AD Hales              \n   Ben Duckett     BM Duckett  5f26f677               42                0    BM Duckett              \n    Ben Stokes      BA Stokes  e087956b               99               87     BA Stokes     BA Stokes\n  Chris Jordan      CJ Jordan  ffe699c0               23               34     CJ Jordan     CJ Jordan\n  Chris Woakes      CR Woakes  4c5d73db               84              116     CR Woakes     CR Woakes\n   Dawid Malan       DJ Malan  ad9c32a2               29                2      DJ Malan      DJ Malan\n   Harry Brook       HC Brook  4ae1755b               45                0      HC Brook              \n  James Taylor     JWA Taylor  c374f37f               25                

In [124]:
eng_candidate_ids = set(
    eng_strong_candidates["cricsheet_player_id"].astype(str)
)

eng_candidate_names = set(
    eng_strong_candidates["crickrida_name"]
)

print("EXISTING CRICSHEET ID CONFLICTS")

display(
    player_identity_map_updated[
        player_identity_map_updated["player_id"].astype(str).isin(
            eng_candidate_ids
        )
    ].to_string(index=False)
)

print("\nEXISTING CRICKRIDA NAME CONFLICTS")

display(
    player_identity_map_updated[
        player_identity_map_updated["crickrida_name"].isin(
            eng_candidate_names
        )
    ].to_string(index=False)
)

EXISTING CRICSHEET ID CONFLICTS


'Empty DataFrame\nColumns: [cricketiq_player_id, player_id, cricsheet_name, crickrida_name, normalized_name, match_status, crickrida_player_name, cricsheet_player_name, cricsheet_player_id, mapping_method, confidence, notes]\nIndex: []'


EXISTING CRICKRIDA NAME CONFLICTS


'Empty DataFrame\nColumns: [cricketiq_player_id, player_id, cricsheet_name, crickrida_name, normalized_name, match_status, crickrida_player_name, cricsheet_player_name, cricsheet_player_id, mapping_method, confidence, notes]\nIndex: []'

In [125]:
eng_review_names = [
    "Ian Bell",
    "James Tredwell",
    "Ravi Bopara"
]

eng_review_candidates = eng_candidates_df[
    eng_candidates_df["crickrida_name"].isin(
        eng_review_names
    )
].copy()

review_rows = []

for _, row in eng_review_candidates.iterrows():

    pid = str(row["cricsheet_player_id"])

    batting = player_batting_updated[
        player_batting_updated["batter_id"].astype(str) == pid
    ]

    bowling = player_bowling_updated[
        player_bowling_updated["bowler_id"].astype(str) == pid
    ]

    review_rows.append({
        "crickrida_name": row["crickrida_name"],
        "cricsheet_name": row["cricsheet_name"],
        "player_id": pid,
        "crickrida_matches": int(
            eng_crickrida_history.loc[
                eng_crickrida_history["player_name"]
                == row["crickrida_name"],
                "crickrida_matches"
            ].iloc[0]
        ),
        "batting_records": len(batting),
        "bowling_records": len(bowling),
        "batting_names": sorted(
            batting["batter"].dropna().unique().tolist()
        ),
        "bowling_names": sorted(
            bowling["bowler"].dropna().unique().tolist()
        )
    })

eng_date_review = pd.DataFrame(review_rows)

display(eng_date_review.to_string(index=False))

'crickrida_name cricsheet_name player_id  crickrida_matches  batting_records  bowling_records batting_names bowling_names\n      Ian Bell        IR Bell  33155d1b                  1              152                6     [IR Bell]     [IR Bell]\nJames Tredwell    JC Tredwell  49b6c09f                  1               25               43 [JC Tredwell] [JC Tredwell]\n   Ravi Bopara      RS Bopara  7ca5e05d                  1              108               73   [RS Bopara]   [RS Bopara]'

In [127]:
# Recreate all 22 England candidate mappings
eng_all_verified_candidates = eng_candidates_df[
    eng_candidates_df["crickrida_name"].isin(
        set(eng_unresolved["player_name"])
    )
].copy()

# Keep candidates with matching first initials
eng_all_verified_candidates = (
    eng_all_verified_candidates[
        eng_all_verified_candidates["initial_match"]
    ].copy()
)

print("Candidate rows:",
      len(eng_all_verified_candidates))

print("Unique Crickrida names:",
      eng_all_verified_candidates["crickrida_name"].nunique())

print("Unique Cricsheet IDs:",
      eng_all_verified_candidates["cricsheet_player_id"].nunique())

display(
    eng_all_verified_candidates[
        ["crickrida_name", "cricsheet_name",
         "cricsheet_player_id", "initial_match",
         "date_overlap"]
    ].sort_values("crickrida_name").to_string(index=False)
)

Candidate rows: 22
Unique Crickrida names: 22
Unique Cricsheet IDs: 22


'crickrida_name cricsheet_name cricsheet_player_id  initial_match  date_overlap\n   Adil Rashid      AU Rashid            249d60c9           True          True\n    Alex Hales       AD Hales            92aeac25           True          True\n   Ben Duckett     BM Duckett            5f26f677           True          True\n    Ben Stokes      BA Stokes            e087956b           True          True\n  Chris Jordan      CJ Jordan            ffe699c0           True          True\n  Chris Woakes      CR Woakes            4c5d73db           True          True\n   Dawid Malan       DJ Malan            ad9c32a2           True          True\n   Harry Brook       HC Brook            4ae1755b           True          True\n      Ian Bell        IR Bell            33155d1b           True         False\n  James Taylor     JWA Taylor            c374f37f           True          True\nJames Tredwell    JC Tredwell            49b6c09f           True         False\n   James Vince       JM Vince          

In [128]:
eng_all_candidate_ids = set(
    eng_all_verified_candidates["cricsheet_player_id"].astype(str)
)

eng_all_candidate_names = set(
    eng_all_verified_candidates["crickrida_name"]
)

id_conflicts = player_identity_map_updated[
    player_identity_map_updated["player_id"].astype(str).isin(
        eng_all_candidate_ids
    )
]

name_conflicts = player_identity_map_updated[
    player_identity_map_updated["crickrida_name"].isin(
        eng_all_candidate_names
    )
]

print("ID conflicts:", len(id_conflicts))
display(id_conflicts.to_string(index=False))

print("\nName conflicts:", len(name_conflicts))
display(name_conflicts.to_string(index=False))

ID conflicts: 0


'Empty DataFrame\nColumns: [cricketiq_player_id, player_id, cricsheet_name, crickrida_name, normalized_name, match_status, crickrida_player_name, cricsheet_player_name, cricsheet_player_id, mapping_method, confidence, notes]\nIndex: []'


Name conflicts: 0


'Empty DataFrame\nColumns: [cricketiq_player_id, player_id, cricsheet_name, crickrida_name, normalized_name, match_status, crickrida_player_name, cricsheet_player_name, cricsheet_player_id, mapping_method, confidence, notes]\nIndex: []'

In [129]:
eng_full_names = {
    "Adil Rashid": "Adil Usman Rashid",
    "Alex Hales": "Alexander Daniel Hales",
    "Ben Duckett": "Benjamin Matthew Duckett",
    "Ben Stokes": "Benjamin Andrew Stokes",
    "Chris Jordan": "Christopher James Jordan",
    "Chris Woakes": "Christopher Roger Woakes",
    "Dawid Malan": "Dawid Johannes Malan",
    "Harry Brook": "Harry Cherrington Brook",
    "Ian Bell": "Ian Ronald Bell",
    "James Taylor": "James William Arthur Taylor",
    "James Tredwell": "James Charles Tredwell",
    "James Vince": "James Michael Vince",
    "Jamie Overton": "Jamie Overton",
    "Jamie Smith": "Jamie Luke Smith",
    "Jofra Archer": "Jofra Chioke Archer",
    "Jonny Bairstow": "Jonathan Marc Bairstow",
    "Jos Buttler": "Joseph Charles Buttler",
    "Moeen Ali": "Moeen Munir Ali",
    "Phil Salt": "Philip Dean Salt",
    "Ravi Bopara": "Ravinder Singh Bopara",
    "Reece Topley": "Reece James William Topley",
    "Stuart Broad": "Stuart Christopher John Broad"
}

In [130]:
eng_verified_mapping_rows = []

date_discrepancy_notes = {
    "Ian Bell": (
        "Career-date discrepancy: Crickrida record is 2015-03-13; "
        "Cricsheet history ends 2015-03-09. Identity supported by "
        "full name, initials and consistent performance records."
    ),
    "James Tredwell": (
        "Career-date discrepancy: Crickrida record is 2015-03-13; "
        "Cricsheet history ends 2014-12-16. Identity supported by "
        "full name, initials and consistent performance records."
    ),
    "Ravi Bopara": (
        "Career-date discrepancy: Crickrida record is 2015-03-13; "
        "Cricsheet history ends 2015-02-01. Identity supported by "
        "full name, initials and consistent performance records."
    )
}

assert set(eng_full_names) == set(
    eng_all_verified_candidates["crickrida_name"]
)

for _, row in eng_all_verified_candidates.iterrows():

    common_name = row["crickrida_name"]
    cricsheet_name = row["cricsheet_name"]
    player_id = str(row["cricsheet_player_id"])

    notes = (
        f"Full name: {eng_full_names[common_name]}. "
        "Matched using full-name initials and consistent "
        "batting/bowling performance records."
    )

    if common_name in date_discrepancy_notes:
        notes += " " + date_discrepancy_notes[common_name]

    eng_verified_mapping_rows.append({
        "cricketiq_player_id": f"CRIQ_{player_id}",
        "player_id": player_id,
        "cricsheet_name": cricsheet_name,
        "crickrida_name": common_name,
        "normalized_name": normalize_identity(common_name),
        "match_status": "verified_alias",
        "crickrida_player_name": common_name,
        "cricsheet_player_name": cricsheet_name,
        "cricsheet_player_id": player_id,
        "mapping_method": "verified_full_name_and_performance_review",
        "confidence": "high",
        "notes": notes
    })

eng_verified_mapping_df = pd.DataFrame(
    eng_verified_mapping_rows,
    columns=player_identity_map_updated.columns
)

print("Staged England records:", len(eng_verified_mapping_df))
print("Unique CricketIQ IDs:",
      eng_verified_mapping_df["cricketiq_player_id"].nunique())
print("Unique Cricsheet IDs:",
      eng_verified_mapping_df["player_id"].nunique())

display(eng_verified_mapping_df.to_string(index=False))

Staged England records: 22
Unique CricketIQ IDs: 22
Unique Cricsheet IDs: 22


'cricketiq_player_id player_id cricsheet_name crickrida_name normalized_name   match_status crickrida_player_name cricsheet_player_name cricsheet_player_id                            mapping_method confidence                                                                                                                                                                                                                                                                                             notes\n      CRIQ_249d60c9  249d60c9      AU Rashid    Adil Rashid      adilrashid verified_alias           Adil Rashid             AU Rashid            249d60c9 verified_full_name_and_performance_review       high                                                                                                                                                                                Full name: Adil Usman Rashid. Matched using full-name initials and consistent batting/bowling performance records.\n 

In [131]:
assert len(eng_verified_mapping_df) == 22
assert eng_verified_mapping_df["player_id"].is_unique
assert eng_verified_mapping_df["crickrida_name"].is_unique
assert eng_verified_mapping_df["cricketiq_player_id"].is_unique

assert eng_verified_mapping_df["confidence"].eq("high").all()
assert eng_verified_mapping_df["match_status"].eq("verified_alias").all()

assert not eng_verified_mapping_df["player_id"].isin(
    player_identity_map_updated["player_id"].dropna().astype(str)
).any()

assert not eng_verified_mapping_df["crickrida_name"].isin(
    player_identity_map_updated["crickrida_name"].dropna()
).any()

assert eng_verified_mapping_df.isnull().sum().sum() == 0

print("All 22 England records passed staging validation.")

All 22 England records passed staging validation.


In [132]:
# Preserve the current identity map
player_identity_map_before_england = (
    player_identity_map_updated.copy(deep=True)
)

before_count = len(player_identity_map_updated)

# Safety checks before insertion
assert len(eng_verified_mapping_df) == 22

assert eng_verified_mapping_df["cricketiq_player_id"].is_unique
assert eng_verified_mapping_df["player_id"].is_unique
assert eng_verified_mapping_df["crickrida_name"].is_unique

existing_ids = set(
    player_identity_map_updated["player_id"].dropna().astype(str)
)

existing_names = set(
    player_identity_map_updated["crickrida_name"].dropna()
)

assert not set(
    eng_verified_mapping_df["player_id"].astype(str)
).intersection(existing_ids)

assert not set(
    eng_verified_mapping_df["crickrida_name"]
).intersection(existing_names)

# Append verified mappings
player_identity_map_updated = pd.concat(
    [
        player_identity_map_updated,
        eng_verified_mapping_df
    ],
    ignore_index=True
)

after_count = len(player_identity_map_updated)

print("Before insertion:", before_count)
print("Records inserted:", len(eng_verified_mapping_df))
print("After insertion:", after_count)

assert after_count == before_count + 22

print("\nEngland mappings inserted successfully.")

Before insertion: 323
Records inserted: 22
After insertion: 345

England mappings inserted successfully.


In [133]:
# Check global CricketIQ ID uniqueness
assert player_identity_map_updated[
    "cricketiq_player_id"
].is_unique

# Confirm all 22 inserted records exist
eng_inserted_ids = set(
    eng_verified_mapping_df["cricketiq_player_id"]
)

eng_inserted = player_identity_map_updated[
    player_identity_map_updated["cricketiq_player_id"].isin(
        eng_inserted_ids
    )
]

assert len(eng_inserted) == 22
assert eng_inserted["player_id"].is_unique

assert (eng_inserted["match_status"] == "verified_alias").all()
assert (eng_inserted["confidence"] == "high").all()

print("All 22 England mappings verified.")
print("Identity map shape:", player_identity_map_updated.shape)

All 22 England mappings verified.
Identity map shape: (345, 12)


In [134]:
eng_source_names = set(
    eng_crickrida_history["normalized_name"].dropna()
)

eng_map_names = set(
    player_identity_map_updated["normalized_name"].dropna()
)

eng_resolved_names = eng_source_names.intersection(
    eng_map_names
)

eng_unresolved_final = eng_source_names - eng_map_names

print("ENGLAND FINAL AUDIT")
print("----------------------------")
print("Crickrida players:", len(eng_source_names))
print("Resolved:", len(eng_resolved_names))
print("Unresolved:", len(eng_unresolved_final))

print("\nUnresolved names:", sorted(eng_unresolved_final))

assert len(eng_resolved_names) == 28
assert len(eng_unresolved_final) == 0

ENGLAND FINAL AUDIT
----------------------------
Crickrida players: 28
Resolved: 28
Unresolved: 0

Unresolved names: []


In [135]:
# India: Crickrida source history
ind_crickrida_history = (
    cr_player_history[
        cr_player_history["team"].str.lower() == "india"
    ]
    .drop_duplicates(subset="normalized_name")
    .copy()
)

# India: Cricsheet source history
ind_cricsheet_history = (
    cs_player_history[
        cs_player_history["team"].str.lower() == "india"
    ]
    .drop_duplicates(subset="normalized_name")
    .copy()
)

# Identify already represented Crickrida identities
ind_mapped_names = set(
    player_identity_map_updated["crickrida_name"]
    .dropna()
    .apply(normalize_identity)
)

ind_crickrida_history["identity_key"] = (
    ind_crickrida_history["player_name"]
    .apply(normalize_identity)
)

ind_unresolved = ind_crickrida_history[
    ~ind_crickrida_history["identity_key"].isin(
        ind_mapped_names
    )
].copy()

ind_already_mapped = ind_crickrida_history[
    ind_crickrida_history["identity_key"].isin(
        ind_mapped_names
    )
].copy()

print("INDIA SOURCE SUMMARY")
print("----------------------------")
print("Crickrida players:", len(ind_crickrida_history))
print("Cricsheet players:", len(ind_cricsheet_history))
print("Already represented:", len(ind_already_mapped))
print("Names requiring investigation:", len(ind_unresolved))

print("\nALREADY REPRESENTED")
display(
    ind_already_mapped[
        ["player_name", "crickrida_first_date",
         "crickrida_last_date", "crickrida_matches"]
    ].sort_values("player_name").to_string(index=False)
)

print("\nUNRESOLVED CRICKRIDA PLAYERS")
display(
    ind_unresolved[
        ["player_name", "normalized_name",
         "crickrida_first_date", "crickrida_last_date",
         "crickrida_matches"]
    ].sort_values("player_name").to_string(index=False)
)

print("\nCRICSHEET INDIA PLAYERS")
display(
    ind_cricsheet_history[
        ["player_name", "source_player_id",
         "cricsheet_first_date", "cricsheet_last_date",
         "cricsheet_matches"]
    ].sort_values("player_name").to_string(index=False)
)

INDIA SOURCE SUMMARY
----------------------------
Crickrida players: 36
Cricsheet players: 137
Already represented: 14
Names requiring investigation: 22

ALREADY REPRESENTED


'       player_name crickrida_first_date crickrida_last_date  crickrida_matches\n    Arshdeep Singh           2026-06-13          2026-06-17                  2\n     Deepak Chahar           2018-09-25          2018-09-25                  1\n      Gurnoor Brar           2026-06-13          2026-06-20                  3\n      Ishan Kishan           2023-10-11          2026-06-17                  3\n          KL Rahul           2018-09-25          2026-06-17                  4\n     Kuldeep Yadav           2018-09-25          2026-06-17                  4\n          MS Dhoni           2018-09-25          2019-06-22                  2\n    Mohammed Shami           2014-03-05          2019-06-22                  2\n    Mohammed Siraj           2023-10-11          2023-10-11                  1\nNitish Kumar Reddy           2026-06-13          2026-06-20                  2\n   Prasidh Krishna           2026-06-13          2026-06-20                  2\n      Prince Yadav           2026-06-17


UNRESOLVED CRICKRIDA PLAYERS


'        player_name    normalized_name crickrida_first_date crickrida_last_date  crickrida_matches\n     Ajinkya Rahane      ajinkyarahane           2014-03-05          2014-03-05                  1\n      Ambati Rayudu       ambatirayudu           2018-09-25          2018-09-25                  1\n        Amit Mishra         amitmishra           2014-03-05          2014-03-05                  1\n  Bhuvneshwar Kumar   bhuvneshwarkumar           2014-03-05          2014-03-05                  1\n     Dinesh Karthik      dineshkarthik           2014-03-05          2018-09-25                  2\n      Hardik Pandya       hardikpandya           2019-06-22          2023-10-11                  2\n        Harsh Dubey         harshdubey           2026-06-13          2026-06-20                  2\n     Jasprit Bumrah      jaspritbumrah           2019-06-22          2023-10-11                  2\n       Kedar Jadhav        kedarjadhav           2018-09-25          2019-06-22                  2\


CRICSHEET INDIA PLAYERS


'        player_name source_player_id cricsheet_first_date cricsheet_last_date  cricsheet_matches\n           A Kamboj         fcc21ace           2026-10-03          2026-10-03                  1\n           A Kumble         0c2730df           2002-06-29          2007-03-19                 27\n           A Mishra         6b19d823           2009-09-28          2016-10-29                 32\n           A Mithun         66cf56a5           2010-02-27          2011-12-11                  5\n            A Nehra         96fd40ae           2002-12-29          2011-03-30                 77\n         AB Agarkar         fa463154           2002-06-29          2007-09-05                 65\n           AB Dinda         66b30f71           2010-05-28          2013-01-11                 13\n           AM Nayar         a45a5e8d           2009-09-30          2009-09-30                  1\n          AM Rahane         29e95537           2011-09-03          2018-02-16                 86\n           AR Patel

In [136]:
# Convert source dates
ind_crickrida_history["crickrida_first_date"] = pd.to_datetime(
    ind_crickrida_history["crickrida_first_date"]
)

ind_crickrida_history["crickrida_last_date"] = pd.to_datetime(
    ind_crickrida_history["crickrida_last_date"]
)

ind_cricsheet_history["cricsheet_first_date"] = pd.to_datetime(
    ind_cricsheet_history["cricsheet_first_date"]
)

ind_cricsheet_history["cricsheet_last_date"] = pd.to_datetime(
    ind_cricsheet_history["cricsheet_last_date"]
)

ind_candidate_rows = []

for _, cr in ind_unresolved.iterrows():

    cr_name = cr["player_name"]
    cr_tokens = cr_name.split()

    cr_initial = cr_tokens[0][0].lower()
    cr_surname = normalize_identity(cr_tokens[-1])

    for _, cs in ind_cricsheet_history.iterrows():

        cs_name = cs["player_name"]
        cs_tokens = cs_name.split()

        cs_initial = cs_tokens[0][0].lower()
        cs_surname = normalize_identity(cs_tokens[-1])

        if cr_surname != cs_surname:
            continue

        initial_match = cr_initial == cs_initial

        date_overlap = (
            cr["crickrida_first_date"] <= cs["cricsheet_last_date"]
            and
            cs["cricsheet_first_date"] <= cr["crickrida_last_date"]
        )

        ind_candidate_rows.append({
            "crickrida_name": cr_name,
            "cricsheet_name": cs_name,
            "cricsheet_player_id": cs["source_player_id"],
            "initial_match": initial_match,
            "date_overlap": date_overlap,
            "crickrida_first_date": cr["crickrida_first_date"],
            "crickrida_last_date": cr["crickrida_last_date"],
            "cricsheet_first_date": cs["cricsheet_first_date"],
            "cricsheet_last_date": cs["cricsheet_last_date"],
            "cricsheet_matches": cs["cricsheet_matches"]
        })

ind_candidates_df = pd.DataFrame(ind_candidate_rows)

if not ind_candidates_df.empty:
    ind_candidates_df = ind_candidates_df.sort_values(
        ["crickrida_name", "initial_match", "date_overlap"],
        ascending=[True, False, False]
    )

print("Total same-surname candidate pairs:",
      len(ind_candidates_df))

print("\nINITIAL + DATE OVERLAP CANDIDATES")

display(
    ind_candidates_df[
        ind_candidates_df["initial_match"] &
        ind_candidates_df["date_overlap"]
    ].to_string(index=False)
)

print("\nALL SAME-SURNAME CANDIDATES")

display(ind_candidates_df.to_string(index=False))

Total same-surname candidate pairs: 33

INITIAL + DATE OVERLAP CANDIDATES


'     crickrida_name cricsheet_name cricsheet_player_id  initial_match  date_overlap crickrida_first_date crickrida_last_date cricsheet_first_date cricsheet_last_date  cricsheet_matches\n     Ajinkya Rahane      AM Rahane            29e95537           True          True           2014-03-05          2014-03-05           2011-09-03          2018-02-16                 86\n      Ambati Rayudu      AT Rayudu            70d205c9           True          True           2018-09-25          2018-09-25           2013-07-24          2019-03-08                 49\n        Amit Mishra       A Mishra            6b19d823           True          True           2014-03-05          2014-03-05           2009-09-28          2016-10-29                 32\n  Bhuvneshwar Kumar        B Kumar            2e81a32d           True          True           2014-03-05          2014-03-05           2012-12-30          2022-01-21                119\n      Hardik Pandya      HH Pandya            dbe50b21           True


ALL SAME-SURNAME CANDIDATES


'     crickrida_name  cricsheet_name cricsheet_player_id  initial_match  date_overlap crickrida_first_date crickrida_last_date cricsheet_first_date cricsheet_last_date  cricsheet_matches\n     Ajinkya Rahane       AM Rahane            29e95537           True          True           2014-03-05          2014-03-05           2011-09-03          2018-02-16                 86\n      Ambati Rayudu       AT Rayudu            70d205c9           True          True           2018-09-25          2018-09-25           2013-07-24          2019-03-08                 49\n        Amit Mishra        A Mishra            6b19d823           True          True           2014-03-05          2014-03-05           2009-09-28          2016-10-29                 32\n  Bhuvneshwar Kumar         B Kumar            2e81a32d           True          True           2014-03-05          2014-03-05           2012-12-30          2022-01-21                119\n  Bhuvneshwar Kumar    Mukesh Kumar            2cffab74         

In [137]:
ind_players_with_candidates = set(
    ind_candidates_df["crickrida_name"]
) if not ind_candidates_df.empty else set()

ind_no_candidate = ind_unresolved[
    ~ind_unresolved["player_name"].isin(
        ind_players_with_candidates
    )
]

print("Players without same-surname candidates:",
      len(ind_no_candidate))

display(
    ind_no_candidate[
        ["player_name", "crickrida_first_date",
         "crickrida_last_date", "crickrida_matches"]
    ].to_string(index=False)
)

Players without same-surname candidates: 1


'player_name crickrida_first_date crickrida_last_date  crickrida_matches\nHarsh Dubey           2026-06-13          2026-06-20                  2'

In [138]:
# All same-surname candidates, including weaker alternatives
ind_review_ids = (
    ind_candidates_df["cricsheet_player_id"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

# Include the additional Rohit Sharma alternative
ind_review_ids.append("5d9a1a73")

ind_review_ids = list(dict.fromkeys(ind_review_ids))

print("Candidate IDs being reviewed:", len(ind_review_ids))


# -----------------------------------
# BATTING PERFORMANCE
# -----------------------------------

ind_batting_review = player_batting_updated[
    player_batting_updated["batter_id"]
    .astype(str)
    .isin(ind_review_ids)
].copy()

print("\nBATTING PERFORMANCE SUMMARY")

display(
    ind_batting_review.groupby(
        ["batter_id", "batter"],
        dropna=False
    ).agg(
        innings=("match_id", "nunique"),
        total_runs=("runs", "sum"),
        total_balls=("balls_faced", "sum")
    ).reset_index().sort_values("batter")
)


# -----------------------------------
# BOWLING PERFORMANCE
# -----------------------------------

ind_bowling_review = player_bowling_updated[
    player_bowling_updated["bowler_id"]
    .astype(str)
    .isin(ind_review_ids)
].copy()

print("\nBOWLING PERFORMANCE SUMMARY")

display(
    ind_bowling_review.groupby(
        ["bowler_id", "bowler"],
        dropna=False
    ).agg(
        innings=("match_id", "nunique"),
        wickets=("wickets", "sum"),
        legal_balls=("legal_balls", "sum")
    ).reset_index().sort_values("bowler")
)


# -----------------------------------
# EXISTING IDENTITY MAP CONFLICT CHECK
# -----------------------------------

print("\nEXISTING IDENTITY MAP RECORDS")

display(
    player_identity_map_updated[
        player_identity_map_updated["player_id"]
        .astype(str)
        .isin(ind_review_ids)
    ][
        ["cricketiq_player_id", "player_id",
         "cricsheet_name", "crickrida_name",
         "match_status", "confidence", "notes"]
    ].to_string(index=False)
)

Candidate IDs being reviewed: 33

BATTING PERFORMANCE SUMMARY


,batter_id,batter,innings,total_runs,total_balls
14,6b19d823,A Mishra,10,43,82
4,29e95537,AM Rahane,86,2906,3701
16,70d205c9,AT Rayudu,49,1637,2094
6,2e81a32d,B Kumar,53,552,747
27,dbe50b21,HH Pandya,67,1897,1708
12,5bb1a1c4,I Sharma,26,72,203
8,462411b3,JJ Bumrah,22,110,171
22,9f961c14,Joginder Sharma,1,5,2
26,c03f1114,KD Karthik,76,1624,2212
11,5b8c830e,KH Pandya,4,130,128



BOWLING PERFORMANCE SUMMARY


,bowler_id,bowler,innings,wickets,legal_balls
13,6b19d823,A Mishra,31,61,1773
14,70d205c9,AT Rayudu,9,3,121
5,2e81a32d,B Kumar,119,141,5799
23,dbe50b21,HH Pandya,86,87,3359
11,5bb1a1c4,I Sharma,77,114,3709
7,462411b3,JJ Bumrah,88,145,4574
19,9f961c14,Joginder Sharma,2,1,66
10,5b8c830e,KH Pandya,5,2,228
21,a2f46292,KK Ahmed,10,14,420
18,99d63244,KM Jadhav,41,26,1145



EXISTING IDENTITY MAP RECORDS


'Empty DataFrame\nColumns: [cricketiq_player_id, player_id, cricsheet_name, crickrida_name, match_status, confidence, notes]\nIndex: []'

In [139]:
# -----------------------------------
# BOWLING PERFORMANCE SUMMARY
# -----------------------------------

print("BOWLING PERFORMANCE SUMMARY")

display(
    ind_bowling_review.groupby(
        ["bowler_id", "bowler"],
        dropna=False
    ).agg(
        innings=("match_id", "nunique"),
        wickets=("wickets", "sum"),
        legal_balls=("legal_balls", "sum")
    ).reset_index().sort_values("bowler")
)


# -----------------------------------
# EXISTING IDENTITY MAP CONFLICT CHECK
# -----------------------------------

print("\nEXISTING IDENTITY MAP RECORDS")

display(
    player_identity_map_updated[
        player_identity_map_updated["player_id"]
        .astype(str)
        .isin(ind_review_ids)
    ][
        ["cricketiq_player_id", "player_id",
         "cricsheet_name", "crickrida_name",
         "match_status", "confidence", "notes"]
    ].to_string(index=False)
)

BOWLING PERFORMANCE SUMMARY


,bowler_id,bowler,innings,wickets,legal_balls
13,6b19d823,A Mishra,31,61,1773
14,70d205c9,AT Rayudu,9,3,121
5,2e81a32d,B Kumar,119,141,5799
23,dbe50b21,HH Pandya,86,87,3359
11,5bb1a1c4,I Sharma,77,114,3709
7,462411b3,JJ Bumrah,88,145,4574
19,9f961c14,Joginder Sharma,2,1,66
10,5b8c830e,KH Pandya,5,2,228
21,a2f46292,KK Ahmed,10,14,420
18,99d63244,KM Jadhav,41,26,1145



EXISTING IDENTITY MAP RECORDS


'Empty DataFrame\nColumns: [cricketiq_player_id, player_id, cricsheet_name, crickrida_name, match_status, confidence, notes]\nIndex: []'

In [140]:
# Rahul Sharma — Cricsheet ID
r_sharma_id = "5d9a1a73"

# 1. Batting record
r_sharma_batting = player_batting_updated[
    player_batting_updated["batter_id"].astype(str)
    == r_sharma_id
].copy()

# 2. Bowling records
r_sharma_bowling = player_bowling_updated[
    player_bowling_updated["bowler_id"].astype(str)
    == r_sharma_id
].copy()

print("RAHUL SHARMA — BATTING RECORD")
display(
    r_sharma_batting[
        ["match_id", "innings_number", "batting_team",
         "batter", "runs", "balls_faced"]
    ].to_string(index=False)
)

print("\nRAHUL SHARMA — BOWLING RECORDS")
display(
    r_sharma_bowling[
        ["match_id", "innings_number", "bowler",
         "runs_conceded", "wickets", "legal_balls",
         "overs", "economy"]
    ].sort_values("match_id").to_string(index=False)
)

print("\nMATCH IDS")
display(
    sorted(
        set(r_sharma_batting["match_id"].dropna())
        | set(r_sharma_bowling["match_id"].dropna())
    )
)

RAHUL SHARMA — BATTING RECORD


'match_id  innings_number batting_team   batter  runs  balls_faced\n  518956               2        India R Sharma     1            2'


RAHUL SHARMA — BOWLING RECORDS


'match_id  innings_number   bowler  runs_conceded  wickets  legal_balls  overs  economy\n  518956               1 R Sharma             44        1           38    6.2 6.947368\n  536932               2 R Sharma             43        3           60   10.0 4.300000\n  536933               2 R Sharma             45        1           60   10.0 4.500000\n  564783               1 R Sharma             45        1           48    8.0 5.625000'


MATCH IDS


['518956', '536932', '536933', '564783']

In [141]:

# ============================================================
# INDIA IDENTITY MAPPING — STAGING ONLY
# 21 shared + 1 Crickrida-only + 1 Cricsheet-only
# ============================================================

india_verified_pairs = [
    ("Ajinkya Rahane", "AM Rahane", "29e95537"),
    ("Ambati Rayudu", "AT Rayudu", "70d205c9"),
    ("Amit Mishra", "A Mishra", "6b19d823"),
    ("Bhuvneshwar Kumar", "B Kumar", "2e81a32d"),
    ("Hardik Pandya", "HH Pandya", "dbe50b21"),
    ("Jasprit Bumrah", "JJ Bumrah", "462411b3"),
    ("Kedar Jadhav", "KM Jadhav", "99d63244"),
    ("Khaleel Ahmed", "KK Ahmed", "a2f46292"),
    ("Manish Pandey", "MK Pandey", "93b4fc78"),
    ("Ravichandran Ashwin", "R Ashwin", "495d42a5"),
    ("Ravindra Jadeja", "RA Jadeja", "fe93fd9d"),
    ("Rohit Sharma", "RG Sharma", "740742ef"),
    ("Shardul Thakur", "SN Thakur", "1abb78f8"),
    ("Shikhar Dhawan", "S Dhawan", "0a476045"),
    ("Shreyas Iyer", "SS Iyer", "85ec8e33"),
    ("Vijay Shankar", "V Shankar", "0994d0ae"),
    ("Virat Kohli", "V Kohli", "ba607b88"),

    # Additional identities verified using full names
    ("Dinesh Karthik", "KD Karthik", "c03f1114"),
    ("Siddarth Kaul", "S Kaul", "dcf81436"),
    ("Yashasvi Jaiswal", "YBK Jaiswal", "6c19c6e5"),
]

india_staged_rows = []

# ------------------------------------------------------------
# A. BUILD VERIFIED SHARED MAPPINGS
# ------------------------------------------------------------

for cr_name, cs_name, cs_id in india_verified_pairs:

    cr_match = ind_crickrida_history[
        ind_crickrida_history["player_name"] == cr_name
    ]

    cs_match = ind_cricsheet_history[
        ind_cricsheet_history["source_player_id"].astype(str) == cs_id
    ]

    # Source validation
    assert len(cr_match) == 1, f"Crickrida source issue: {cr_name}"
    assert len(cs_match) == 1, f"Cricsheet source issue: {cs_id}"
    assert cs_match.iloc[0]["player_name"] == cs_name, (
        f"Cricsheet name mismatch: {cs_id}"
    )

    # Special evidence notes
    if cr_name == "Dinesh Karthik":
        method = "verified_full_name_and_performance_review"
        notes = (
            "Verified using full name Krishnakumar Dinesh Karthik "
            "and consistent performance records. Initials are reversed."
        )

    elif cr_name == "Siddarth Kaul":
        method = "verified_full_name_and_scorecard_review"
        notes = (
            "Verified using full name and 25 September 2018 "
            "India scorecard. Source career-history dates do not overlap."
        )

    elif cr_name == "Yashasvi Jaiswal":
        method = "verified_full_name_and_scorecard_review"
        notes = (
            "Verified using full name and India scorecards dated "
            "17 and 20 June 2026. Source career-history dates do not overlap."
        )

    else:
        method = "verified_india_name_and_career_review"
        notes = (
            "Verified using player name, initials, career dates "
            "and consistent batting/bowling performance records."
        )

    india_staged_rows.append({
        "cricketiq_player_id": f"CRIQ_{cs_id}",
        "player_id": cs_id,
        "cricsheet_name": cs_name,
        "crickrida_name": cr_name,
        "normalized_name": normalize_identity(cr_name),
        "match_status": "verified_alias",
        "crickrida_player_name": cr_name,
        "cricsheet_player_name": cs_name,
        "cricsheet_player_id": cs_id,
        "mapping_method": method,
        "confidence": "high",
        "notes": notes
    })


# ------------------------------------------------------------
# B. HARSH DUBEY — CRICKRIDA ONLY
# ------------------------------------------------------------

harsh_match = ind_crickrida_history[
    ind_crickrida_history["player_name"] == "Harsh Dubey"
]

assert len(harsh_match) == 1, "Harsh Dubey source record not unique"

india_staged_rows.append({
    "cricketiq_player_id": "CR_harshdubey",
    "player_id": None,
    "cricsheet_name": None,
    "crickrida_name": "Harsh Dubey",
    "normalized_name": normalize_identity("Harsh Dubey"),
    "match_status": "crickrida_only",
    "crickrida_player_name": "Harsh Dubey",
    "cricsheet_player_name": None,
    "cricsheet_player_id": None,
    "mapping_method": "verified_source_coverage",
    "confidence": "high",
    "notes": (
        "Two Crickrida appearances against Afghanistan. "
        "Afghanistan is excluded from the Cricsheet dataset."
    )
})


# ------------------------------------------------------------
# C. RAHUL SHARMA — CRICSHEET ONLY
# ------------------------------------------------------------

rahul_match = ind_cricsheet_history[
    ind_cricsheet_history["source_player_id"].astype(str)
    == "5d9a1a73"
]

assert len(rahul_match) == 1, "Rahul Sharma Cricsheet record not unique"
assert rahul_match.iloc[0]["player_name"] == "R Sharma"

india_staged_rows.append({
    "cricketiq_player_id": "CS_5d9a1a73",
    "player_id": None,
    "cricsheet_name": "R Sharma",
    "crickrida_name": None,
    "normalized_name": normalize_identity("R Sharma"),
    "match_status": "cricsheet_only",
    "crickrida_player_name": None,
    "cricsheet_player_name": "R Sharma",
    "cricsheet_player_id": "5d9a1a73",
    "mapping_method": "verified_source_coverage",
    "confidence": "high",
    "notes": (
        "Identified as Rahul Sharma, not Rohit Sharma. "
        "Four ODI match records: 518956, 536932, 536933 and 564783. "
        "One batting innings, 1 run; six wickets in four bowling records. "
        "No corresponding Crickrida India history record."
    )
})


# ------------------------------------------------------------
# D. CREATE STAGING DATAFRAME
# ------------------------------------------------------------

india_staged_map = pd.DataFrame(
    india_staged_rows,
    columns=player_identity_map_updated.columns
)

print("INDIA STAGING SUMMARY")
print("----------------------------")
print("Total staged rows:", len(india_staged_map))
print(
    "Shared mappings:",
    (india_staged_map["match_status"] == "verified_alias").sum()
)
print(
    "Crickrida-only:",
    (india_staged_map["match_status"] == "crickrida_only").sum()
)
print(
    "Cricsheet-only:",
    (india_staged_map["match_status"] == "cricsheet_only").sum()
)

display(
    india_staged_map[
        ["cricketiq_player_id", "player_id",
         "cricsheet_name", "crickrida_name",
         "match_status", "mapping_method", "confidence"]
    ].to_string(index=False)
)


INDIA STAGING SUMMARY
----------------------------
Total staged rows: 22
Shared mappings: 20
Crickrida-only: 1
Cricsheet-only: 1


'cricketiq_player_id player_id cricsheet_name      crickrida_name   match_status                            mapping_method confidence\n      CRIQ_29e95537  29e95537      AM Rahane      Ajinkya Rahane verified_alias     verified_india_name_and_career_review       high\n      CRIQ_70d205c9  70d205c9      AT Rayudu       Ambati Rayudu verified_alias     verified_india_name_and_career_review       high\n      CRIQ_6b19d823  6b19d823       A Mishra         Amit Mishra verified_alias     verified_india_name_and_career_review       high\n      CRIQ_2e81a32d  2e81a32d        B Kumar   Bhuvneshwar Kumar verified_alias     verified_india_name_and_career_review       high\n      CRIQ_dbe50b21  dbe50b21      HH Pandya       Hardik Pandya verified_alias     verified_india_name_and_career_review       high\n      CRIQ_462411b3  462411b3      JJ Bumrah      Jasprit Bumrah verified_alias     verified_india_name_and_career_review       high\n      CRIQ_99d63244  99d63244      KM Jadhav        Kedar Jad

In [142]:

# ============================================================
# INDIA STAGING VALIDATION
# ============================================================

existing_map = player_identity_map_updated.copy()

# 1. Duplicate Cricsheet IDs within staged records
staged_player_duplicates = india_staged_map[
    india_staged_map["player_id"].notna()
    & india_staged_map["player_id"].duplicated(keep=False)
]

# 2. Duplicate CricketIQ IDs within staged records
staged_cricketiq_duplicates = india_staged_map[
    india_staged_map["cricketiq_player_id"].duplicated(keep=False)
]

# 3. Conflicts with existing Cricsheet IDs
existing_player_ids = set(
    existing_map["player_id"].dropna().astype(str)
)

staged_existing_player_conflicts = india_staged_map[
    india_staged_map["player_id"].notna()
    & india_staged_map["player_id"].astype(str).isin(existing_player_ids)
]

# 4. Conflicts with existing CricketIQ IDs
existing_cricketiq_ids = set(
    existing_map["cricketiq_player_id"].dropna().astype(str)
)

staged_existing_cricketiq_conflicts = india_staged_map[
    india_staged_map["cricketiq_player_id"].astype(str)
    .isin(existing_cricketiq_ids)
]

# 5. Check duplicate normalized Crickrida names
staged_crickrida_names = india_staged_map[
    india_staged_map["crickrida_name"].notna()
]["normalized_name"]

duplicate_staged_names = staged_crickrida_names[
    staged_crickrida_names.duplicated(keep=False)
]

# 6. Verify expected status counts
status_counts = india_staged_map["match_status"].value_counts()

print("VALIDATION RESULTS")
print("----------------------------")

print("Duplicate staged player IDs:",
      len(staged_player_duplicates))

print("Duplicate staged CricketIQ IDs:",
      len(staged_cricketiq_duplicates))

print("Existing player ID conflicts:",
      len(staged_existing_player_conflicts))

print("Existing CricketIQ ID conflicts:",
      len(staged_existing_cricketiq_conflicts))

print("Duplicate staged Crickrida names:",
      len(duplicate_staged_names))

print("\nSTATUS COUNTS")
print(status_counts)

# 7. Show any problems
if not staged_player_duplicates.empty:
    display(staged_player_duplicates)

if not staged_cricketiq_duplicates.empty:
    display(staged_cricketiq_duplicates)

if not staged_existing_player_conflicts.empty:
    display(staged_existing_player_conflicts)

if not staged_existing_cricketiq_conflicts.empty:
    display(staged_existing_cricketiq_conflicts)

if len(duplicate_staged_names) > 0:
    print("Duplicate normalized Crickrida names:")
    display(duplicate_staged_names)

# 8. Final decision — validation only, no insertion
india_staging_valid = (
    len(india_staged_map) == 23
    and len(staged_player_duplicates) == 0
    and len(staged_cricketiq_duplicates) == 0
    and len(staged_existing_player_conflicts) == 0
    and len(staged_existing_cricketiq_conflicts) == 0
    and len(duplicate_staged_names) == 0
    and status_counts.get("verified_alias", 0) == 21
    and status_counts.get("crickrida_only", 0) == 1
    and status_counts.get("cricsheet_only", 0) == 1
)

print("\nFINAL STAGING STATUS:",
      "READY FOR INSERTION" if india_staging_valid else "REQUIRES REVIEW")


VALIDATION RESULTS
----------------------------
Duplicate staged player IDs: 0
Duplicate staged CricketIQ IDs: 0
Existing player ID conflicts: 0
Existing CricketIQ ID conflicts: 0
Duplicate staged Crickrida names: 0

STATUS COUNTS
match_status
verified_alias    20
crickrida_only     1
cricsheet_only     1
Name: count, dtype: int64

FINAL STAGING STATUS: REQUIRES REVIEW


In [143]:
india_staged_map.to_csv(
    "india_identity_mapping_staged_20261116.csv",
    index=False
)

print("India staging CSV saved.")

India staging CSV saved.


In [144]:

# Add the missing Yuzvendra Chahal mapping

cr_name = "Yuzvendra Chahal"
cs_name = "YS Chahal"
cs_id = "57ee1fde"

# Verify source records
cr_match = ind_crickrida_history[
    ind_crickrida_history["player_name"] == cr_name
]

cs_match = ind_cricsheet_history[
    ind_cricsheet_history["source_player_id"].astype(str) == cs_id
]

assert len(cr_match) == 1
assert len(cs_match) == 1
assert cs_match.iloc[0]["player_name"] == cs_name

# Check it is not already staged
assert not (
    india_staged_map["player_id"].astype(str) == cs_id
).any()

# Build missing row
yuzvendra_row = {
    "cricketiq_player_id": f"CRIQ_{cs_id}",
    "player_id": cs_id,
    "cricsheet_name": cs_name,
    "crickrida_name": cr_name,
    "normalized_name": normalize_identity(cr_name),
    "match_status": "verified_alias",
    "crickrida_player_name": cr_name,
    "cricsheet_player_name": cs_name,
    "cricsheet_player_id": cs_id,
    "mapping_method": "verified_india_name_and_career_review",
    "confidence": "high",
    "notes": (
        "Verified using matching initials, overlapping career dates "
        "and consistent batting/bowling performance records."
    )
}

india_staged_map = pd.concat(
    [india_staged_map, pd.DataFrame([yuzvendra_row])],
    ignore_index=True
)

print("Added:", cr_name, "->", cs_name)
print("Updated staged rows:", len(india_staged_map))


Added: Yuzvendra Chahal -> YS Chahal
Updated staged rows: 23


In [149]:
# Recalculate validation using the CURRENT staging DataFrame

status_counts = india_staged_map["match_status"].value_counts()

staged_player_duplicates = india_staged_map[
    india_staged_map["player_id"].notna()
    & india_staged_map["player_id"].duplicated(keep=False)
]

staged_cricketiq_duplicates = india_staged_map[
    india_staged_map["cricketiq_player_id"].duplicated(keep=False)
]

existing_player_ids = set(
    player_identity_map_updated["player_id"].dropna().astype(str)
)

existing_cricketiq_ids = set(
    player_identity_map_updated["cricketiq_player_id"].dropna().astype(str)
)

existing_player_conflicts = india_staged_map[
    india_staged_map["player_id"].notna()
    & india_staged_map["player_id"].astype(str).isin(existing_player_ids)
]

existing_cricketiq_conflicts = india_staged_map[
    india_staged_map["cricketiq_player_id"].astype(str)
    .isin(existing_cricketiq_ids)
]

duplicate_names = india_staged_map[
    india_staged_map["crickrida_name"].notna()
]["normalized_name"]

duplicate_names = duplicate_names[
    duplicate_names.duplicated(keep=False)
]

# Print every condition independently
checks = {
    "Total rows == 23": len(india_staged_map) == 23,
    "Shared mappings == 21": status_counts.get("verified_alias", 0) == 21,
    "Crickrida-only == 1": status_counts.get("crickrida_only", 0) == 1,
    "Cricsheet-only == 1": status_counts.get("cricsheet_only", 0) == 1,
    "No duplicate player IDs": len(staged_player_duplicates) == 0,
    "No duplicate CricketIQ IDs": len(staged_cricketiq_duplicates) == 0,
    "No existing player ID conflicts": len(existing_player_conflicts) == 0,
    "No existing CricketIQ ID conflicts": len(existing_cricketiq_conflicts) == 0,
    "No duplicate Crickrida names": len(duplicate_names) == 0
}

print("CURRENT INDIA VALIDATION")
print("------------------------------")

for description, passed in checks.items():
    print(("PASS" if passed else "FAIL"), "-", description)

print("\nSTATUS COUNTS:")
print(status_counts)

print("\nFAILED CHECK DETAILS:")

if len(staged_player_duplicates):
    display(staged_player_duplicates)

if len(staged_cricketiq_duplicates):
    display(staged_cricketiq_duplicates)

if len(existing_player_conflicts):
    display(existing_player_conflicts)

if len(existing_cricketiq_conflicts):
    display(existing_cricketiq_conflicts)

if len(duplicate_names):
    display(duplicate_names)

# Refresh the validation flag
india_staging_valid = all(checks.values())

print("\nFINAL STATUS:",
      "READY FOR INSERTION" if india_staging_valid
      else "REQUIRES REVIEW")

CURRENT INDIA VALIDATION
------------------------------
PASS - Total rows == 23
PASS - Shared mappings == 21
PASS - Crickrida-only == 1
PASS - Cricsheet-only == 1
PASS - No duplicate player IDs
PASS - No duplicate CricketIQ IDs
PASS - No existing player ID conflicts
PASS - No existing CricketIQ ID conflicts
PASS - No duplicate Crickrida names

STATUS COUNTS:
match_status
verified_alias    21
crickrida_only     1
cricsheet_only     1
Name: count, dtype: int64

FAILED CHECK DETAILS:

FINAL STATUS: READY FOR INSERTION


In [150]:
assert india_staging_valid

india_staged_map.to_csv(
    "india_identity_mapping_staged_20261116.csv",
    index=False
)

print("India staging CSV saved successfully.")
print("Total rows:", len(india_staged_map))

India staging CSV saved successfully.
Total rows: 23


In [151]:
# ============================================================
# INDIA IDENTITY MAP INSERTION
# ============================================================

# Final safety checks
assert india_staging_valid
assert len(india_staged_map) == 23

# Backup authoritative map
player_identity_map_backup_before_india = (
    player_identity_map_updated.copy(deep=True)
)

# Insert staged rows
player_identity_map_updated = pd.concat(
    [
        player_identity_map_updated,
        india_staged_map
    ],
    ignore_index=True
)

# Verify insertion
print("INDIA INSERTION COMPLETE")
print("----------------------------")
print("Rows inserted:", len(india_staged_map))
print("Previous map size:",
      len(player_identity_map_backup_before_india))
print("Current map size:",
      len(player_identity_map_updated))

print("\nIndia status counts:")
display(
    player_identity_map_updated[
        player_identity_map_updated["crickrida_name"].isin(
            ind_crickrida_history["player_name"]
        )
    ]["match_status"].value_counts()
)

INDIA INSERTION COMPLETE
----------------------------
Rows inserted: 23
Previous map size: 345
Current map size: 368

India status counts:


match_status
verified_alias      24
exact_name_match    11
crickrida_only       1
Name: count, dtype: int64

In [152]:
print("FINAL MAP INTEGRITY")
print("----------------------------")

print(
    "Duplicate CricketIQ IDs:",
    player_identity_map_updated[
        "cricketiq_player_id"
    ].duplicated().sum()
)

print(
    "Duplicate Cricsheet player IDs:",
    player_identity_map_updated[
        "player_id"
    ].dropna().duplicated().sum()
)

print(
    "Map dimensions:",
    player_identity_map_updated.shape
)

FINAL MAP INTEGRITY
----------------------------
Duplicate CricketIQ IDs: 0
Duplicate Cricsheet player IDs: 0
Map dimensions: (368, 12)


In [153]:

# ============================================================
# SOUTH AFRICA — SOURCE EXTRACTION
# ============================================================

# 1. Crickrida South Africa players
sa_crickrida_history = (
    cr_player_history[
        cr_player_history["team"].str.lower() == "south africa"
    ]
    .drop_duplicates(subset="normalized_name")
    .copy()
)

# 2. Cricsheet South Africa players
sa_cricsheet_history = (
    cs_player_history[
        cs_player_history["team"].str.lower() == "south africa"
    ]
    .drop_duplicates(subset="normalized_name")
    .copy()
)

# 3. Identify already represented Crickrida identities
sa_mapped_names = set(
    player_identity_map_updated["crickrida_name"]
    .dropna()
    .apply(normalize_identity)
)

sa_crickrida_history["identity_key"] = (
    sa_crickrida_history["player_name"]
    .apply(normalize_identity)
)

sa_already_mapped = sa_crickrida_history[
    sa_crickrida_history["identity_key"].isin(sa_mapped_names)
].copy()

sa_unresolved = sa_crickrida_history[
    ~sa_crickrida_history["identity_key"].isin(sa_mapped_names)
].copy()


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("SOUTH AFRICA SOURCE SUMMARY")
print("----------------------------")
print("Crickrida players:", len(sa_crickrida_history))
print("Cricsheet players:", len(sa_cricsheet_history))
print("Already represented:", len(sa_already_mapped))
print("Names requiring investigation:", len(sa_unresolved))


print("\nALREADY REPRESENTED")

display(
    sa_already_mapped[
        ["player_name", "crickrida_first_date",
         "crickrida_last_date", "crickrida_matches"]
    ].sort_values("player_name").to_string(index=False)
)


print("\nUNRESOLVED CRICKRIDA PLAYERS")

display(
    sa_unresolved[
        ["player_name", "normalized_name",
         "crickrida_first_date", "crickrida_last_date",
         "crickrida_matches"]
    ].sort_values("player_name").to_string(index=False)
)


print("\nCRICSHEET SOUTH AFRICA PLAYERS")

display(
    sa_cricsheet_history[
        ["player_name", "source_player_id",
         "cricsheet_first_date", "cricsheet_last_date",
         "cricsheet_matches"]
    ].sort_values("player_name").to_string(index=False)
)


SOUTH AFRICA SOURCE SUMMARY
----------------------------
Crickrida players: 26
Cricsheet players: 122
Already represented: 4
Names requiring investigation: 22

ALREADY REPRESENTED


'       player_name crickrida_first_date crickrida_last_date  crickrida_matches\nAndile Phehlukwayo           2019-06-15          2024-09-22                  4\n  Beuran Hendricks           2019-06-15          2019-06-15                  1\n       Imran Tahir           2019-06-15          2019-06-15                  1\n   Reeza Hendricks           2024-09-18          2024-09-22                  3'


UNRESOLVED CRICKRIDA PLAYERS


'          player_name    normalized_name crickrida_first_date crickrida_last_date  crickrida_matches\n        Aiden Markram       aidenmarkram           2023-11-10          2025-02-21                  5\n        Bjorn Fortuin       bjornfortuin           2024-09-18          2024-09-22                  3\n         Chris Morris        chrismorris           2019-06-15          2019-06-15                  1\n         David Miller        davidmiller           2023-11-10          2025-02-21                  2\n       Gerald Coetzee      geraldcoetzee           2023-11-10          2023-11-10                  1\n          Hashim Amla         hashimamla           2019-06-15          2019-06-15                  1\n     Heinrich Klaasen    heinrichklaasen           2023-11-10          2023-11-10                  1\n          Jason Smith         jasonsmith           2024-09-18          2024-09-18                  1\n        Kagiso Rabada       kagisorabada           2019-06-15          2025-02-21


CRICSHEET SOUTH AFRICA PLAYERS


'      player_name source_player_id cricsheet_first_date cricsheet_last_date  cricsheet_matches\n            A Nel         2503e881           2003-07-10          2008-08-31                 58\n         A Nortje         acdc62f5           2019-03-03          2023-09-09                 21\n        AA Donald         4c931d56           2003-02-09          2003-02-27                  3\n   AB de Villiers         c4487b84           2005-02-02          2018-02-16                209\n        AC Dawson         62decd8e           2003-04-21          2003-10-12                  6\n        AG Prince         035cb925           2005-01-30          2007-04-25                 31\n       AG Puttick         637b6fbd           2005-10-28          2005-10-28                  1\n          AJ Hall         6b0c3a20           2003-02-22          2007-07-01                 54\n       AK Markram         6a26221c           2017-10-22          2026-09-30                 78\n   AL Phehlukwayo         ab6b1c45     

In [154]:

# ============================================================
# SOUTH AFRICA — CANDIDATE GENERATION
# ============================================================

# Convert date columns
sa_crickrida_history["crickrida_first_date"] = pd.to_datetime(
    sa_crickrida_history["crickrida_first_date"]
)

sa_crickrida_history["crickrida_last_date"] = pd.to_datetime(
    sa_crickrida_history["crickrida_last_date"]
)

sa_cricsheet_history["cricsheet_first_date"] = pd.to_datetime(
    sa_cricsheet_history["cricsheet_first_date"]
)

sa_cricsheet_history["cricsheet_last_date"] = pd.to_datetime(
    sa_cricsheet_history["cricsheet_last_date"]
)

sa_candidate_rows = []

for _, cr in sa_unresolved.iterrows():

    cr_name = cr["player_name"]
    cr_tokens = cr_name.split()

    cr_initial = cr_tokens[0][0].lower()
    cr_surname = normalize_identity(cr_tokens[-1])

    for _, cs in sa_cricsheet_history.iterrows():

        cs_name = cs["player_name"]
        cs_tokens = cs_name.split()

        cs_initial = cs_tokens[0][0].lower()
        cs_surname = normalize_identity(cs_tokens[-1])

        if cr_surname != cs_surname:
            continue

        initial_match = cr_initial == cs_initial

        date_overlap = (
            cr["crickrida_first_date"] <= cs["cricsheet_last_date"]
            and
            cs["cricsheet_first_date"] <= cr["crickrida_last_date"]
        )

        sa_candidate_rows.append({
            "crickrida_name": cr_name,
            "cricsheet_name": cs_name,
            "cricsheet_player_id": cs["source_player_id"],
            "initial_match": initial_match,
            "date_overlap": date_overlap,
            "crickrida_first_date": cr["crickrida_first_date"],
            "crickrida_last_date": cr["crickrida_last_date"],
            "cricsheet_first_date": cs["cricsheet_first_date"],
            "cricsheet_last_date": cs["cricsheet_last_date"],
            "cricsheet_matches": cs["cricsheet_matches"]
        })

sa_candidates_df = pd.DataFrame(sa_candidate_rows)

if not sa_candidates_df.empty:
    sa_candidates_df = sa_candidates_df.sort_values(
        ["crickrida_name", "initial_match", "date_overlap"],
        ascending=[True, False, False]
    )

print("Total same-surname candidate pairs:",
      len(sa_candidates_df))

print("\nINITIAL + DATE OVERLAP CANDIDATES")

if not sa_candidates_df.empty:
    display(
        sa_candidates_df[
            sa_candidates_df["initial_match"]
            & sa_candidates_df["date_overlap"]
        ].to_string(index=False)
    )

print("\nALL SAME-SURNAME CANDIDATES")

if not sa_candidates_df.empty:
    display(sa_candidates_df.to_string(index=False))

# Find unresolved players without same-surname candidates
sa_players_with_candidates = (
    set(sa_candidates_df["crickrida_name"])
    if not sa_candidates_df.empty else set()
)

sa_no_candidate = sa_unresolved[
    ~sa_unresolved["player_name"].isin(
        sa_players_with_candidates
    )
]

print("\nPLAYERS WITHOUT SAME-SURNAME CANDIDATES")

display(
    sa_no_candidate[
        ["player_name", "crickrida_first_date",
         "crickrida_last_date", "crickrida_matches"]
    ].to_string(index=False)
)


Total same-surname candidate pairs: 24

INITIAL + DATE OVERLAP CANDIDATES


'  crickrida_name cricsheet_name cricsheet_player_id  initial_match  date_overlap crickrida_first_date crickrida_last_date cricsheet_first_date cricsheet_last_date  cricsheet_matches\n   Aiden Markram     AK Markram            6a26221c           True          True           2023-11-10          2025-02-21           2017-10-22          2026-09-30                 78\n   Bjorn Fortuin     BC Fortuin            9a2fc964           True          True           2024-09-18          2024-09-22           2022-10-09          2026-09-30                 17\n    Chris Morris      CH Morris            fb66ce1f           True          True           2019-06-15          2019-06-15           2013-06-10          2019-07-06                 39\n    David Miller      DA Miller            d67d5f00           True          True           2023-11-10          2025-02-21           2010-05-22          2026-09-27                152\n  Gerald Coetzee      G Coetzee            3204c99f           True          True    


ALL SAME-SURNAME CANDIDATES


'       crickrida_name    cricsheet_name cricsheet_player_id  initial_match  date_overlap crickrida_first_date crickrida_last_date cricsheet_first_date cricsheet_last_date  cricsheet_matches\n        Aiden Markram        AK Markram            6a26221c           True          True           2023-11-10          2025-02-21           2017-10-22          2026-09-30                 78\n        Bjorn Fortuin        BC Fortuin            9a2fc964           True          True           2024-09-18          2024-09-22           2022-10-09          2026-09-30                 17\n         Chris Morris         CH Morris            fb66ce1f           True          True           2019-06-15          2019-06-15           2013-06-10          2019-07-06                 39\n         David Miller         DA Miller            d67d5f00           True          True           2023-11-10          2025-02-21           2010-05-22          2026-09-27                152\n       Gerald Coetzee         G Coetzee     


PLAYERS WITHOUT SAME-SURNAME CANDIDATES


'Empty DataFrame\nColumns: [player_name, crickrida_first_date, crickrida_last_date, crickrida_matches]\nIndex: []'

In [155]:

# ============================================================
# SOUTH AFRICA — PERFORMANCE VALIDATION
# ============================================================

sa_review_ids = (
    sa_candidates_df["cricsheet_player_id"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

print("Candidate IDs being reviewed:", len(sa_review_ids))


# ------------------------------------------------------------
# BATTING PERFORMANCE
# ------------------------------------------------------------

sa_batting_review = player_batting_updated[
    player_batting_updated["batter_id"]
    .astype(str).isin(sa_review_ids)
].copy()

print("\nBATTING PERFORMANCE SUMMARY")

display(
    sa_batting_review.groupby(
        ["batter_id", "batter"],
        dropna=False
    ).agg(
        innings=("match_id", "nunique"),
        total_runs=("runs", "sum"),
        total_balls=("balls_faced", "sum")
    ).reset_index().sort_values("batter")
)


# ------------------------------------------------------------
# BOWLING PERFORMANCE
# ------------------------------------------------------------

sa_bowling_review = player_bowling_updated[
    player_bowling_updated["bowler_id"]
    .astype(str).isin(sa_review_ids)
].copy()

print("\nBOWLING PERFORMANCE SUMMARY")

display(
    sa_bowling_review.groupby(
        ["bowler_id", "bowler"],
        dropna=False
    ).agg(
        innings=("match_id", "nunique"),
        wickets=("wickets", "sum"),
        legal_balls=("legal_balls", "sum")
    ).reset_index().sort_values("bowler")
)


# ------------------------------------------------------------
# EXISTING IDENTITY MAP CONFLICT CHECK
# ------------------------------------------------------------

print("\nEXISTING IDENTITY MAP RECORDS")

display(
    player_identity_map_updated[
        player_identity_map_updated["player_id"]
        .astype(str).isin(sa_review_ids)
    ][
        ["cricketiq_player_id", "player_id",
         "cricsheet_name", "crickrida_name",
         "match_status", "confidence", "notes"]
    ].to_string(index=False)
)


Candidate IDs being reviewed: 24

BATTING PERFORMANCE SUMMARY


,batter_id,batter,innings,total_runs,total_balls
6,6a26221c,AK Markram,78,2554,2613
13,9a2fc964,BC Fortuin,11,67,102
23,fb66ce1f,CH Morris,27,467,465
10,8dc152d1,D Jansen,3,30,21
18,d67d5f00,DA Miller,152,4724,4514
3,3204c99f,G Coetzee,8,63,94
7,72861603,GC Smith,151,5280,6462
2,235c2bb6,H Klaasen,55,2131,1816
12,9948e262,HE van der Dussen,63,2529,2912
21,e798611a,HM Amla,174,7834,8861



BOWLING PERFORMANCE SUMMARY


,bowler_id,bowler,innings,wickets,legal_balls
5,6a26221c,AK Markram,41,19,1250
12,9a2fc964,BC Fortuin,17,24,843
17,fb66ce1f,CH Morris,39,45,1858
9,8dc152d1,D Jansen,3,5,120
3,3204c99f,G Coetzee,15,30,708
6,72861603,GC Smith,33,15,808
2,235c2bb6,H Klaasen,2,0,30
11,9948e262,HE van der Dussen,1,1,6
10,91cd48e0,JF Smith,2,1,54
15,e62dd25d,K Rabada,101,164,5285



EXISTING IDENTITY MAP RECORDS


'Empty DataFrame\nColumns: [cricketiq_player_id, player_id, cricsheet_name, crickrida_name, match_status, confidence, notes]\nIndex: []'

In [156]:

# ============================================================
# SOUTH AFRICA — STAGE 22 VERIFIED IDENTITY MAPPINGS
# ============================================================

sa_verified_mappings = [
    ("Aiden Markram", "AK Markram", "6a26221c"),
    ("Bjorn Fortuin", "BC Fortuin", "9a2fc964"),
    ("Chris Morris", "CH Morris", "fb66ce1f"),
    ("David Miller", "DA Miller", "d67d5f00"),
    ("Gerald Coetzee", "G Coetzee", "3204c99f"),
    ("Hashim Amla", "HM Amla", "e798611a"),
    ("Heinrich Klaasen", "H Klaasen", "235c2bb6"),
    ("Jason Smith", "JF Smith", "91cd48e0"),
    ("Kagiso Rabada", "K Rabada", "e62dd25d"),
    ("Keshav Maharaj", "KA Maharaj", "0b60eb09"),
    ("Kyle Verreynne", "K Verreynne", "bf814547"),
    ("Lungi Ngidi", "L Ngidi", "f834dcfc"),
    ("Marco Jansen", "M Jansen", "81c36ee9"),
    ("Nandre Burger", "N Burger", "465aa633"),
    ("Nqabayomzi Peter", "N Peter", "212aeedd"),
    ("Quinton de Kock", "Q de Kock", "372455c4"),
    ("Rassie van der Dussen", "HE van der Dussen", "9948e262"),
    ("Ryan Rickelton", "RD Rickelton", "e66732f8"),
    ("Temba Bavuma", "T Bavuma", "9ffd1ac1"),
    ("Tony de Zorzi", "T de Zorzi", "d3a1c63d"),
    ("Tristan Stubbs", "T Stubbs", "85b3fab2"),
    ("Wiaan Mulder", "PWA Mulder", "c96f6ac5")
]


# ------------------------------------------------------------
# SPECIAL NOTES
# ------------------------------------------------------------

sa_special_notes = {
    "Jason Smith":
        "Verified by full name Jason Franswyn Smith. "
        "Crickrida and Cricsheet career dates do not overlap.",

    "Nqabayomzi Peter":
        "Verified by full name Nqabayomzi Xolela Peter. "
        "Known as Nqaba Peter. Source career dates do not overlap.",

    "Rassie van der Dussen":
        "Verified full identity Hendrik Erasmus van der Dussen. "
        "Rassie is his nickname; Cricsheet uses legal initials HE.",

    "Wiaan Mulder":
        "Verified full identity Pieter Willem Adriaan Mulder. "
        "Wiaan is his nickname; Cricsheet uses initials PWA."
}


# ------------------------------------------------------------
# BUILD STAGING DATAFRAME
# ------------------------------------------------------------

sa_staged_rows = []

for cr_name, cs_name, player_id in sa_verified_mappings:

    note = sa_special_notes.get(
        cr_name,
        "Verified using matching name, career history "
        "and Cricsheet batting/bowling performance."
    )

    sa_staged_rows.append({
        "cricketiq_player_id": f"CRIQ_{player_id}",
        "player_id": player_id,
        "cricsheet_name": cs_name,
        "crickrida_name": cr_name,
        "normalized_name": normalize_identity(cr_name),
        "match_status": "verified_alias",
        "crickrida_player_name": cr_name,
        "cricsheet_player_name": cs_name,
        "cricsheet_player_id": player_id,
        "mapping_method": "verified_south_africa_name_and_performance_review",
        "confidence": "high",
        "notes": note
    })


sa_identity_staging = pd.DataFrame(sa_staged_rows)

print("Staged South Africa mappings:", len(sa_identity_staging))

display(sa_identity_staging)


Staged South Africa mappings: 22


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,normalized_name,match_status,crickrida_player_name,cricsheet_player_name,cricsheet_player_id,mapping_method,confidence,notes
0,CRIQ_6a26221c,6a26221c,AK Markram,Aiden Markram,aidenmarkram,verified_alias,Aiden Markram,AK Markram,6a26221c,verified_south_africa_name_and_performance_review,high,"Verified using matching name, career history a..."
1,CRIQ_9a2fc964,9a2fc964,BC Fortuin,Bjorn Fortuin,bjornfortuin,verified_alias,Bjorn Fortuin,BC Fortuin,9a2fc964,verified_south_africa_name_and_performance_review,high,"Verified using matching name, career history a..."
2,CRIQ_fb66ce1f,fb66ce1f,CH Morris,Chris Morris,chrismorris,verified_alias,Chris Morris,CH Morris,fb66ce1f,verified_south_africa_name_and_performance_review,high,"Verified using matching name, career history a..."
3,CRIQ_d67d5f00,d67d5f00,DA Miller,David Miller,davidmiller,verified_alias,David Miller,DA Miller,d67d5f00,verified_south_africa_name_and_performance_review,high,"Verified using matching name, career history a..."
4,CRIQ_3204c99f,3204c99f,G Coetzee,Gerald Coetzee,geraldcoetzee,verified_alias,Gerald Coetzee,G Coetzee,3204c99f,verified_south_africa_name_and_performance_review,high,"Verified using matching name, career history a..."
5,CRIQ_e798611a,e798611a,HM Amla,Hashim Amla,hashimamla,verified_alias,Hashim Amla,HM Amla,e798611a,verified_south_africa_name_and_performance_review,high,"Verified using matching name, career history a..."
6,CRIQ_235c2bb6,235c2bb6,H Klaasen,Heinrich Klaasen,heinrichklaasen,verified_alias,Heinrich Klaasen,H Klaasen,235c2bb6,verified_south_africa_name_and_performance_review,high,"Verified using matching name, career history a..."
7,CRIQ_91cd48e0,91cd48e0,JF Smith,Jason Smith,jasonsmith,verified_alias,Jason Smith,JF Smith,91cd48e0,verified_south_africa_name_and_performance_review,high,Verified by full name Jason Franswyn Smith. Cr...
8,CRIQ_e62dd25d,e62dd25d,K Rabada,Kagiso Rabada,kagisorabada,verified_alias,Kagiso Rabada,K Rabada,e62dd25d,verified_south_africa_name_and_performance_review,high,"Verified using matching name, career history a..."
9,CRIQ_0b60eb09,0b60eb09,KA Maharaj,Keshav Maharaj,keshavmaharaj,verified_alias,Keshav Maharaj,KA Maharaj,0b60eb09,verified_south_africa_name_and_performance_review,high,"Verified using matching name, career history a..."


In [157]:

# ============================================================
# SOUTH AFRICA — STAGING VALIDATION
# ============================================================

existing_player_ids = set(
    player_identity_map_updated["player_id"]
    .dropna().astype(str)
)

existing_cricketiq_ids = set(
    player_identity_map_updated["cricketiq_player_id"]
    .dropna().astype(str)
)

existing_crickrida_names = set(
    player_identity_map_updated["crickrida_name"]
    .dropna().apply(normalize_identity)
)

checks = {
    "Exactly 22 staged rows":
        len(sa_identity_staging) == 22,

    "No duplicate Crickrida names":
        not sa_identity_staging["normalized_name"].duplicated().any(),

    "No duplicate Cricsheet player IDs":
        not sa_identity_staging["player_id"].duplicated().any(),

    "No duplicate CricketIQ IDs":
        not sa_identity_staging["cricketiq_player_id"].duplicated().any(),

    "No existing Cricsheet player ID conflicts":
        not sa_identity_staging["player_id"].astype(str)
        .isin(existing_player_ids).any(),

    "No existing CricketIQ ID conflicts":
        not sa_identity_staging["cricketiq_player_id"].astype(str)
        .isin(existing_cricketiq_ids).any(),

    "No existing Crickrida name conflicts":
        not sa_identity_staging["normalized_name"]
        .isin(existing_crickrida_names).any(),

    "All mappings marked verified_alias":
        sa_identity_staging["match_status"].eq("verified_alias").all(),

    "All mappings have high confidence":
        sa_identity_staging["confidence"].eq("high").all()
}

for check, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {check}")

sa_staging_valid = all(checks.values())

print("\nFINAL VALIDATION:", sa_staging_valid)


PASS: Exactly 22 staged rows
PASS: No duplicate Crickrida names
PASS: No duplicate Cricsheet player IDs
PASS: No duplicate CricketIQ IDs
PASS: No existing Cricsheet player ID conflicts
PASS: No existing CricketIQ ID conflicts
PASS: No existing Crickrida name conflicts
PASS: All mappings marked verified_alias
PASS: All mappings have high confidence

FINAL VALIDATION: True


In [158]:

# Save staging file
sa_identity_staging.to_csv(
    "south_africa_identity_mapping_staged_20261116.csv",
    index=False
)

# Preserve the current identity map
player_identity_map_backup_before_sa = (
    player_identity_map_updated.copy(deep=True)
)

print("Backup rows:", len(player_identity_map_backup_before_sa))
print("Staged rows:", len(sa_identity_staging))


Backup rows: 368
Staged rows: 22


In [159]:

# ============================================================
# INSERT VERIFIED SOUTH AFRICA MAPPINGS
# ============================================================

if not sa_staging_valid:
    raise ValueError("South Africa staging validation failed.")

player_identity_map_updated = pd.concat(
    [
        player_identity_map_updated,
        sa_identity_staging
    ],
    ignore_index=True
)

print("Updated identity map shape:",
      player_identity_map_updated.shape)

print(
    "Duplicate CricketIQ IDs:",
    player_identity_map_updated[
        "cricketiq_player_id"
    ].duplicated().sum()
)

print(
    "Duplicate Cricsheet player IDs:",
    player_identity_map_updated[
        "player_id"
    ].dropna().duplicated().sum()
)


Updated identity map shape: (390, 12)
Duplicate CricketIQ IDs: 0
Duplicate Cricsheet player IDs: 0


In [160]:

sa_inserted_names = set(
    player_identity_map_updated["crickrida_name"]
    .dropna()
    .apply(normalize_identity)
)

sa_expected_names = set(
    sa_identity_staging["normalized_name"]
)

sa_final_check = {
    "All 22 South Africa mappings inserted":
        sa_expected_names.issubset(sa_inserted_names),

    "Exactly 22 new rows added":
        len(player_identity_map_updated)
        == len(player_identity_map_backup_before_sa) + 22,

    "No duplicate CricketIQ IDs":
        not player_identity_map_updated[
            "cricketiq_player_id"
        ].duplicated().any(),

    "No duplicate Cricsheet player IDs":
        not player_identity_map_updated[
            "player_id"
        ].dropna().duplicated().any()
}

for check, passed in sa_final_check.items():
    print(f"{'PASS' if passed else 'FAIL'}: {check}")

print("\nSouth Africa insertion successful:",
      all(sa_final_check.values()))


PASS: All 22 South Africa mappings inserted
PASS: Exactly 22 new rows added
PASS: No duplicate CricketIQ IDs
PASS: No duplicate Cricsheet player IDs

South Africa insertion successful: True


In [161]:
# Countries available in Crickrida history
print(
    cr_player_history["team"]
    .value_counts()
    .to_string()
)

team
Afghanistan     58
Zimbabwe        52
Bangladesh      51
Ireland         45
Sri Lanka       40
Pakistan        39
India           36
Australia       29
England         28
West Indies     27
South Africa    26
New Zealand     23


In [162]:

# ============================================================
# WEST INDIES — PLAYER IDENTITY AUDIT
# STEP 1: EXTRACT COUNTRY HISTORIES
# ============================================================

wi_crickrida_history = (
    cr_player_history[
        cr_player_history["team"].str.lower() == "west indies"
    ]
    .drop_duplicates(subset="normalized_name")
    .copy()
)

wi_cricsheet_history = (
    cs_player_history[
        cs_player_history["team"].str.lower() == "west indies"
    ]
    .drop_duplicates(subset="normalized_name")
    .copy()
)

wi_mapped_names = set(
    player_identity_map_updated["crickrida_name"]
    .dropna()
    .apply(normalize_identity)
)

wi_crickrida_history["identity_key"] = (
    wi_crickrida_history["player_name"]
    .apply(normalize_identity)
)

wi_already_mapped = wi_crickrida_history[
    wi_crickrida_history["identity_key"].isin(wi_mapped_names)
].copy()

wi_unresolved = wi_crickrida_history[
    ~wi_crickrida_history["identity_key"].isin(wi_mapped_names)
].copy()

print("WEST INDIES AUDIT")
print("=" * 50)

print("Crickrida players:", len(wi_crickrida_history))
print("Cricsheet players:", len(wi_cricsheet_history))
print("Already represented:", len(wi_already_mapped))
print("Names requiring investigation:", len(wi_unresolved))

print("\nALREADY REPRESENTED")
display(
    wi_already_mapped[
        ["player_name", "crickrida_first_date",
         "crickrida_last_date", "crickrida_matches"]
    ].sort_values("player_name")
)

print("\nUNRESOLVED PLAYERS")
display(
    wi_unresolved[
        ["player_name", "crickrida_first_date",
         "crickrida_last_date", "crickrida_matches"]
    ].sort_values("player_name")
)


WEST INDIES AUDIT
Crickrida players: 27
Cricsheet players: 142
Already represented: 27
Names requiring investigation: 0

ALREADY REPRESENTED


,player_name,crickrida_first_date,crickrida_last_date,crickrida_matches
375,Alzarri Joseph,2017-06-09,2019-11-11,5
376,Ashley Nurse,2017-06-09,2018-03-25,4
377,Brandon King,2019-11-11,2019-11-11,1
378,Carlos Brathwaite,2018-03-15,2019-07-04,3
379,Chris Gayle,2018-03-15,2019-07-04,3
380,Evin Lewis,2017-06-09,2019-11-11,8
381,Fabian Allen,2019-07-04,2019-07-04,1
382,Hayden Walsh,2019-11-06,2019-11-11,3
383,Jason Holder,2017-06-09,2019-11-09,7
384,Jason Mohammed,2017-06-09,2017-06-11,2



UNRESOLVED PLAYERS


,player_name,crickrida_first_date,crickrida_last_date,crickrida_matches


In [163]:

# ============================================================
# NEW ZEALAND — PLAYER IDENTITY AUDIT
# STEP 1: EXTRACT COUNTRY HISTORIES
# ============================================================

nz_crickrida_history = (
    cr_player_history[
        cr_player_history["team"].str.lower() == "new zealand"
    ]
    .drop_duplicates(subset="normalized_name")
    .copy()
)

nz_cricsheet_history = (
    cs_player_history[
        cs_player_history["team"].str.lower() == "new zealand"
    ]
    .drop_duplicates(subset="normalized_name")
    .copy()
)

# Names already represented in the identity map
nz_mapped_names = set(
    player_identity_map_updated["crickrida_name"]
    .dropna()
    .apply(normalize_identity)
)

nz_crickrida_history["identity_key"] = (
    nz_crickrida_history["player_name"]
    .apply(normalize_identity)
)

nz_already_mapped = nz_crickrida_history[
    nz_crickrida_history["identity_key"].isin(nz_mapped_names)
].copy()

nz_unresolved = nz_crickrida_history[
    ~nz_crickrida_history["identity_key"].isin(nz_mapped_names)
].copy()


# ------------------------------------------------------------
# AUDIT SUMMARY
# ------------------------------------------------------------

print("NEW ZEALAND AUDIT")
print("=" * 50)

print("Crickrida players:", len(nz_crickrida_history))
print("Cricsheet players:", len(nz_cricsheet_history))
print("Already represented:", len(nz_already_mapped))
print("Names requiring investigation:", len(nz_unresolved))


# ------------------------------------------------------------
# ALREADY REPRESENTED
# ------------------------------------------------------------

print("\nALREADY REPRESENTED")

display(
    nz_already_mapped[
        [
            "player_name",
            "crickrida_first_date",
            "crickrida_last_date",
            "crickrida_matches"
        ]
    ].sort_values("player_name")
)


# ------------------------------------------------------------
# UNRESOLVED PLAYERS
# ------------------------------------------------------------

print("\nUNRESOLVED PLAYERS")

display(
    nz_unresolved[
        [
            "player_name",
            "crickrida_first_date",
            "crickrida_last_date",
            "crickrida_matches"
        ]
    ].sort_values("player_name")
)


NEW ZEALAND AUDIT
Crickrida players: 23
Cricsheet players: 118
Already represented: 2
Names requiring investigation: 21

ALREADY REPRESENTED


,player_name,crickrida_first_date,crickrida_last_date,crickrida_matches
249,Colin de Grandhomme,2019-06-08,2019-06-08,1
267,Tom Latham,2019-06-08,2023-10-18,2



UNRESOLVED PLAYERS


,player_name,crickrida_first_date,crickrida_last_date,crickrida_matches
247,Adam Milne,2015-03-08,2015-03-08,1
248,Brendon McCullum,2015-03-08,2015-03-08,1
250,Colin Munro,2019-06-08,2019-06-08,1
251,Corey Anderson,2015-03-08,2015-03-08,1
252,Daniel Vettori,2015-03-08,2015-03-08,1
253,Daryl Mitchell,2023-10-18,2023-10-18,1
254,Devon Conway,2023-10-18,2023-10-18,1
255,Glenn Phillips,2023-10-18,2023-10-18,1
256,Grant Elliott,2015-03-08,2015-03-08,1
257,James Neesham,2019-06-08,2019-06-08,1


In [164]:

# ============================================================
# NEW ZEALAND — STEP 2: CANDIDATE GENERATION
# ============================================================

nz_candidate_rows = []

for _, cr in nz_unresolved.iterrows():

    cr_name = cr["player_name"]
    cr_tokens = cr_name.split()

    cr_initial = cr_tokens[0][0].lower()
    cr_surname = normalize_identity(cr_tokens[-1])

    for _, cs in nz_cricsheet_history.iterrows():

        cs_name = cs["player_name"]
        cs_tokens = cs_name.split()

        cs_initial = cs_tokens[0][0].lower()
        cs_surname = normalize_identity(cs_tokens[-1])

        if cr_surname != cs_surname:
            continue

        initial_match = cr_initial == cs_initial

        date_overlap = (
            cr["crickrida_first_date"] <= cs["cricsheet_last_date"]
            and cs["cricsheet_first_date"] <= cr["crickrida_last_date"]
        )

        nz_candidate_rows.append({
            "crickrida_name": cr_name,
            "cricsheet_name": cs_name,
            "cricsheet_player_id": cs["source_player_id"],
            "initial_match": initial_match,
            "date_overlap": date_overlap,
            "crickrida_first_date": cr["crickrida_first_date"],
            "crickrida_last_date": cr["crickrida_last_date"],
            "cricsheet_first_date": cs["cricsheet_first_date"],
            "cricsheet_last_date": cs["cricsheet_last_date"],
            "cricsheet_matches": cs["cricsheet_matches"]
        })

nz_candidates_df = pd.DataFrame(nz_candidate_rows)

if nz_candidates_df.empty:
    print("No same-surname candidates found.")
else:
    print("Total same-surname candidate pairs:", len(nz_candidates_df))

    print("\nINITIAL + DATE OVERLAP CANDIDATES")

    display(
        nz_candidates_df[
            nz_candidates_df["initial_match"]
            & nz_candidates_df["date_overlap"]
        ].sort_values("crickrida_name")
    )

    print("\nALL SAME-SURNAME CANDIDATES")

    display(
        nz_candidates_df.sort_values(
            ["crickrida_name", "initial_match", "date_overlap"],
            ascending=[True, False, False]
        )
    )

    # Players without any same-surname candidate
    nz_candidate_names = set(
        nz_candidates_df["crickrida_name"]
    )

    nz_no_candidate = nz_unresolved[
        ~nz_unresolved["player_name"].isin(nz_candidate_names)
    ].copy()

    print("\nPLAYERS WITHOUT SAME-SURNAME CANDIDATES")

    display(
        nz_no_candidate[
            [
                "player_name",
                "crickrida_first_date",
                "crickrida_last_date",
                "crickrida_matches"
            ]
        ]
    )


Total same-surname candidate pairs: 22

INITIAL + DATE OVERLAP CANDIDATES


,crickrida_name,cricsheet_name,cricsheet_player_id,initial_match,date_overlap,crickrida_first_date,crickrida_last_date,cricsheet_first_date,cricsheet_last_date,cricsheet_matches
0,Adam Milne,AF Milne,350bb1b1,True,True,2015-03-08,2015-03-08,2012-11-10,2023-12-23,47
1,Brendon McCullum,BB McCullum,b8a55852,True,True,2015-03-08,2015-03-08,2002-12-29,2016-02-08,186
3,Colin Munro,C Munro,af2c687b,True,True,2019-06-08,2019-06-08,2013-01-22,2019-06-26,53
4,Corey Anderson,CJ Anderson,8abdf100,True,True,2015-03-08,2015-03-08,2013-06-16,2017-06-09,47
5,Daniel Vettori,DL Vettori,d7c6af50,True,True,2015-03-08,2015-03-08,2002-12-29,2015-03-29,151
6,Daryl Mitchell,DJ Mitchell,eade4650,True,True,2023-10-18,2023-10-18,2021-03-23,2026-07-16,57
7,Devon Conway,DP Conway,df5a6881,True,True,2023-10-18,2023-10-18,2021-03-20,2026-01-18,41
8,Glenn Phillips,GD Phillips,9a46c4e5,True,True,2023-10-18,2023-10-18,2022-07-10,2026-01-18,40
9,Grant Elliott,GD Elliott,c03449e0,True,True,2015-03-08,2015-03-08,2008-06-18,2016-02-08,71
10,James Neesham,JDS Neesham,9219eff0,True,True,2019-06-08,2019-06-08,2013-01-19,2023-11-01,75



ALL SAME-SURNAME CANDIDATES


,crickrida_name,cricsheet_name,cricsheet_player_id,initial_match,date_overlap,crickrida_first_date,crickrida_last_date,cricsheet_first_date,cricsheet_last_date,cricsheet_matches
0,Adam Milne,AF Milne,350bb1b1,True,True,2015-03-08,2015-03-08,2012-11-10,2023-12-23,47
1,Brendon McCullum,BB McCullum,b8a55852,True,True,2015-03-08,2015-03-08,2002-12-29,2016-02-08,186
2,Brendon McCullum,NL McCullum,5673a3fc,False,True,2015-03-08,2015-03-08,2009-09-08,2015-08-19,82
3,Colin Munro,C Munro,af2c687b,True,True,2019-06-08,2019-06-08,2013-01-22,2019-06-26,53
4,Corey Anderson,CJ Anderson,8abdf100,True,True,2015-03-08,2015-03-08,2013-06-16,2017-06-09,47
5,Daniel Vettori,DL Vettori,d7c6af50,True,True,2015-03-08,2015-03-08,2002-12-29,2015-03-29,151
6,Daryl Mitchell,DJ Mitchell,eade4650,True,True,2023-10-18,2023-10-18,2021-03-23,2026-07-16,57
7,Devon Conway,DP Conway,df5a6881,True,True,2023-10-18,2023-10-18,2021-03-20,2026-01-18,41
8,Glenn Phillips,GD Phillips,9a46c4e5,True,True,2023-10-18,2023-10-18,2022-07-10,2026-01-18,40
9,Grant Elliott,GD Elliott,c03449e0,True,True,2015-03-08,2015-03-08,2008-06-18,2016-02-08,71



PLAYERS WITHOUT SAME-SURNAME CANDIDATES


,player_name,crickrida_first_date,crickrida_last_date,crickrida_matches


In [165]:

# ============================================================
# NEW ZEALAND — PERFORMANCE VALIDATION
# ============================================================

nz_review_ids = (
    nz_candidates_df["cricsheet_player_id"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

print("Candidate IDs being reviewed:", len(nz_review_ids))


# ------------------------------------------------------------
# BATTING PERFORMANCE
# ------------------------------------------------------------

nz_batting_review = player_batting_updated[
    player_batting_updated["batter_id"]
    .astype(str).isin(nz_review_ids)
].copy()

print("\nBATTING PERFORMANCE SUMMARY")

display(
    nz_batting_review.groupby(
        ["batter_id", "batter"],
        dropna=False
    ).agg(
        innings=("match_id", "nunique"),
        total_runs=("runs", "sum"),
        total_balls=("balls_faced", "sum")
    ).reset_index().sort_values("batter")
)


# ------------------------------------------------------------
# BOWLING PERFORMANCE
# ------------------------------------------------------------

nz_bowling_review = player_bowling_updated[
    player_bowling_updated["bowler_id"]
    .astype(str).isin(nz_review_ids)
].copy()

print("\nBOWLING PERFORMANCE SUMMARY")

display(
    nz_bowling_review.groupby(
        ["bowler_id", "bowler"],
        dropna=False
    ).agg(
        innings=("match_id", "nunique"),
        wickets=("wickets", "sum"),
        legal_balls=("legal_balls", "sum")
    ).reset_index().sort_values("bowler")
)


# ------------------------------------------------------------
# EXISTING IDENTITY MAP CONFLICT CHECK
# ------------------------------------------------------------

print("\nEXISTING IDENTITY MAP RECORDS")

display(
    player_identity_map_updated[
        player_identity_map_updated["player_id"]
        .astype(str).isin(nz_review_ids)
    ][
        [
            "cricketiq_player_id",
            "player_id",
            "cricsheet_name",
            "crickrida_name",
            "match_status",
            "confidence",
            "notes"
        ]
    ].to_string(index=False)
)


Candidate IDs being reviewed: 22

BATTING PERFORMANCE SUMMARY


,batter_id,batter,innings,total_runs,total_balls
3,350bb1b1,AF Milne,22,180,205
13,b8a55852,BB McCullum,186,5157,5347
11,af2c687b,C Munro,52,1249,1190
6,8abdf100,CJ Anderson,43,1102,1012
21,eade4650,DJ Mitchell,56,2810,2968
17,d7c6af50,DL Vettori,94,1440,1690
18,df5a6881,DP Conway,41,1672,1908
15,c03449e0,GD Elliott,66,1913,2344
9,9a46c4e5,GD Phillips,38,1191,1143
8,9219eff0,JDS Neesham,65,1508,1501



BOWLING PERFORMANCE SUMMARY


,bowler_id,bowler,innings,wickets,legal_balls
3,350bb1b1,AF Milne,47,56,2182
9,af2c687b,C Munro,25,7,552
5,8abdf100,CJ Anderson,42,58,1445
17,eade4650,DJ Mitchell,18,14,339
14,d7c6af50,DL Vettori,151,183,7805
12,c03449e0,GD Elliott,52,35,1200
7,9a46c4e5,GD Phillips,31,16,798
6,9219eff0,JDS Neesham,71,66,2388
13,d027ba9f,KS Williamson,65,37,1467
2,2f9d0389,LH Ferguson,62,92,3203



EXISTING IDENTITY MAP RECORDS


'Empty DataFrame\nColumns: [cricketiq_player_id, player_id, cricsheet_name, crickrida_name, match_status, confidence, notes]\nIndex: []'

In [166]:

# ============================================================
# NEW ZEALAND — STAGE 21 VERIFIED IDENTITY MAPPINGS
# ============================================================

nz_verified_mappings = [
    ("Adam Milne", "AF Milne", "350bb1b1"),
    ("Brendon McCullum", "BB McCullum", "b8a55852"),
    ("Colin Munro", "C Munro", "af2c687b"),
    ("Corey Anderson", "CJ Anderson", "8abdf100"),
    ("Daniel Vettori", "DL Vettori", "d7c6af50"),
    ("Daryl Mitchell", "DJ Mitchell", "eade4650"),
    ("Devon Conway", "DP Conway", "df5a6881"),
    ("Glenn Phillips", "GD Phillips", "9a46c4e5"),
    ("Grant Elliott", "GD Elliott", "c03449e0"),
    ("James Neesham", "JDS Neesham", "9219eff0"),
    ("Kane Williamson", "KS Williamson", "d027ba9f"),
    ("Lockie Ferguson", "LH Ferguson", "2f9d0389"),
    ("Mark Chapman", "MS Chapman", "7fa12533"),
    ("Martin Guptill", "MJ Guptill", "2be41edb"),
    ("Matt Henry", "MJ Henry", "e84ac20c"),
    ("Mitchell Santner", "MJ Santner", "e4a0deae"),
    ("Rachin Ravindra", "R Ravindra", "ba5e1069"),
    ("Ross Taylor", "LRPL Taylor", "b61a3e1a"),
    ("Tim Southee", "TG Southee", "13c35c9e"),
    ("Trent Boult", "TA Boult", "a818c1be"),
    ("Will Young", "WA Young", "8afe73e2")
]


nz_staged_rows = []

for cr_name, cs_name, player_id in nz_verified_mappings:

    if cr_name == "Ross Taylor":
        note = (
            "Verified using full name Luteru Ross Poutoa Lote Taylor. "
            "LRPL initials correspond to his full name. "
            "Batting and bowling records are consistent."
        )

    elif cr_name == "Brendon McCullum":
        note = (
            "Verified as Brendon McCullum using BB McCullum. "
            "NL McCullum is Nathan McCullum, a different player. "
            "Batting performance supports the BB McCullum identity."
        )

    else:
        note = (
            "Verified using matching name initials, career-date "
            "overlap and Cricsheet batting/bowling performance."
        )

    nz_staged_rows.append({
        "cricketiq_player_id": f"CRIQ_{player_id}",
        "player_id": player_id,
        "cricsheet_name": cs_name,
        "crickrida_name": cr_name,
        "normalized_name": normalize_identity(cr_name),
        "match_status": "verified_alias",
        "crickrida_player_name": cr_name,
        "cricsheet_player_name": cs_name,
        "cricsheet_player_id": player_id,
        "mapping_method": "verified_new_zealand_name_and_performance_review",
        "confidence": "high",
        "notes": note
    })


nz_identity_staging = pd.DataFrame(nz_staged_rows)

print("Staged New Zealand mappings:", len(nz_identity_staging))

display(nz_identity_staging)


Staged New Zealand mappings: 21


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,normalized_name,match_status,crickrida_player_name,cricsheet_player_name,cricsheet_player_id,mapping_method,confidence,notes
0,CRIQ_350bb1b1,350bb1b1,AF Milne,Adam Milne,adammilne,verified_alias,Adam Milne,AF Milne,350bb1b1,verified_new_zealand_name_and_performance_review,high,"Verified using matching name initials, career-..."
1,CRIQ_b8a55852,b8a55852,BB McCullum,Brendon McCullum,brendonmccullum,verified_alias,Brendon McCullum,BB McCullum,b8a55852,verified_new_zealand_name_and_performance_review,high,Verified as Brendon McCullum using BB McCullum...
2,CRIQ_af2c687b,af2c687b,C Munro,Colin Munro,colinmunro,verified_alias,Colin Munro,C Munro,af2c687b,verified_new_zealand_name_and_performance_review,high,"Verified using matching name initials, career-..."
3,CRIQ_8abdf100,8abdf100,CJ Anderson,Corey Anderson,coreyanderson,verified_alias,Corey Anderson,CJ Anderson,8abdf100,verified_new_zealand_name_and_performance_review,high,"Verified using matching name initials, career-..."
4,CRIQ_d7c6af50,d7c6af50,DL Vettori,Daniel Vettori,danielvettori,verified_alias,Daniel Vettori,DL Vettori,d7c6af50,verified_new_zealand_name_and_performance_review,high,"Verified using matching name initials, career-..."
5,CRIQ_eade4650,eade4650,DJ Mitchell,Daryl Mitchell,darylmitchell,verified_alias,Daryl Mitchell,DJ Mitchell,eade4650,verified_new_zealand_name_and_performance_review,high,"Verified using matching name initials, career-..."
6,CRIQ_df5a6881,df5a6881,DP Conway,Devon Conway,devonconway,verified_alias,Devon Conway,DP Conway,df5a6881,verified_new_zealand_name_and_performance_review,high,"Verified using matching name initials, career-..."
7,CRIQ_9a46c4e5,9a46c4e5,GD Phillips,Glenn Phillips,glennphillips,verified_alias,Glenn Phillips,GD Phillips,9a46c4e5,verified_new_zealand_name_and_performance_review,high,"Verified using matching name initials, career-..."
8,CRIQ_c03449e0,c03449e0,GD Elliott,Grant Elliott,grantelliott,verified_alias,Grant Elliott,GD Elliott,c03449e0,verified_new_zealand_name_and_performance_review,high,"Verified using matching name initials, career-..."
9,CRIQ_9219eff0,9219eff0,JDS Neesham,James Neesham,jamesneesham,verified_alias,James Neesham,JDS Neesham,9219eff0,verified_new_zealand_name_and_performance_review,high,"Verified using matching name initials, career-..."


In [167]:

# ============================================================
# NEW ZEALAND — STAGING VALIDATION
# ============================================================

existing_player_ids = set(
    player_identity_map_updated["player_id"]
    .dropna().astype(str)
)

existing_cricketiq_ids = set(
    player_identity_map_updated["cricketiq_player_id"]
    .dropna().astype(str)
)

existing_crickrida_names = set(
    player_identity_map_updated["crickrida_name"]
    .dropna().apply(normalize_identity)
)

checks = {
    "Exactly 21 staged rows":
        len(nz_identity_staging) == 21,

    "No duplicate Crickrida names":
        not nz_identity_staging["normalized_name"].duplicated().any(),

    "No duplicate Cricsheet player IDs":
        not nz_identity_staging["player_id"].duplicated().any(),

    "No duplicate CricketIQ IDs":
        not nz_identity_staging["cricketiq_player_id"].duplicated().any(),

    "No existing Cricsheet player ID conflicts":
        not nz_identity_staging["player_id"].astype(str)
        .isin(existing_player_ids).any(),

    "No existing CricketIQ ID conflicts":
        not nz_identity_staging["cricketiq_player_id"].astype(str)
        .isin(existing_cricketiq_ids).any(),

    "No existing Crickrida name conflicts":
        not nz_identity_staging["normalized_name"]
        .isin(existing_crickrida_names).any(),

    "All mappings marked verified_alias":
        nz_identity_staging["match_status"].eq("verified_alias").all(),

    "All mappings have high confidence":
        nz_identity_staging["confidence"].eq("high").all()
}

for check, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {check}")

nz_staging_valid = all(checks.values())

print("\nFINAL VALIDATION:", nz_staging_valid)


PASS: Exactly 21 staged rows
PASS: No duplicate Crickrida names
PASS: No duplicate Cricsheet player IDs
PASS: No duplicate CricketIQ IDs
PASS: No existing Cricsheet player ID conflicts
PASS: No existing CricketIQ ID conflicts
PASS: No existing Crickrida name conflicts
PASS: All mappings marked verified_alias
PASS: All mappings have high confidence

FINAL VALIDATION: True


In [168]:

# Save New Zealand staging CSV
nz_identity_staging.to_csv(
    "new_zealand_identity_mapping_staged_20261116.csv",
    index=False
)

# Preserve current identity map before insertion
player_identity_map_backup_before_nz = (
    player_identity_map_updated.copy(deep=True)
)

print("Backup rows:", len(player_identity_map_backup_before_nz))
print("Staged rows:", len(nz_identity_staging))
print("Staging validation:", nz_staging_valid)


Backup rows: 390
Staged rows: 21
Staging validation: True


In [169]:

# ============================================================
# INSERT VERIFIED NEW ZEALAND MAPPINGS
# ============================================================

if not nz_staging_valid:
    raise ValueError("New Zealand staging validation failed.")

player_identity_map_updated = pd.concat(
    [
        player_identity_map_updated,
        nz_identity_staging
    ],
    ignore_index=True
)

print("Updated identity map shape:",
      player_identity_map_updated.shape)

print(
    "Duplicate CricketIQ IDs:",
    player_identity_map_updated[
        "cricketiq_player_id"
    ].duplicated().sum()
)

print(
    "Duplicate Cricsheet player IDs:",
    player_identity_map_updated[
        "player_id"
    ].dropna().duplicated().sum()
)


Updated identity map shape: (411, 12)
Duplicate CricketIQ IDs: 0
Duplicate Cricsheet player IDs: 0


In [170]:

# ============================================================
# NEW ZEALAND — FINAL INSERTION CHECK
# ============================================================

nz_inserted_names = set(
    player_identity_map_updated["crickrida_name"]
    .dropna()
    .apply(normalize_identity)
)

nz_expected_names = set(
    nz_identity_staging["normalized_name"]
)

nz_final_check = {
    "All 21 New Zealand mappings inserted":
        nz_expected_names.issubset(nz_inserted_names),

    "Exactly 21 new rows added":
        len(player_identity_map_updated)
        == len(player_identity_map_backup_before_nz) + 21,

    "No duplicate CricketIQ IDs":
        not player_identity_map_updated[
            "cricketiq_player_id"
        ].duplicated().any(),

    "No duplicate Cricsheet player IDs":
        not player_identity_map_updated[
            "player_id"
        ].dropna().duplicated().any()
}

for check, passed in nz_final_check.items():
    print(f"{'PASS' if passed else 'FAIL'}: {check}")

print(
    "\nNew Zealand insertion successful:",
    all(nz_final_check.values())
)


PASS: All 21 New Zealand mappings inserted
PASS: Exactly 21 new rows added
PASS: No duplicate CricketIQ IDs
PASS: No duplicate Cricsheet player IDs

New Zealand insertion successful: True


In [171]:

# ============================================================
# CRICKETIQ — GLOBAL IDENTITY MAP INTEGRITY CHECK
# ============================================================

identity_map = player_identity_map_updated.copy()

print("IDENTITY MAP SHAPE:", identity_map.shape)
print("\nMAPPING STATUS COUNTS:")
display(identity_map["match_status"].value_counts(dropna=False))

# Duplicate ID checks
print("\nDUPLICATE CHECKS")

print(
    "Duplicate CricketIQ IDs:",
    identity_map["cricketiq_player_id"].duplicated().sum()
)

print(
    "Duplicate Cricsheet player IDs:",
    identity_map["player_id"].dropna().duplicated().sum()
)

# Duplicate normalized identity names
cr_names = identity_map["crickrida_name"].dropna().apply(normalize_identity)
cs_names = identity_map["cricsheet_name"].dropna().apply(normalize_identity)

print(
    "Duplicate normalized Crickrida names:",
    cr_names.duplicated().sum()
)

print(
    "Duplicate normalized Cricsheet names:",
    cs_names.duplicated().sum()
)

# Missing essential fields
print("\nMISSING VALUE CHECKS")

display(
    identity_map.isna().sum().to_frame("missing_values")
)

# Mapping status versus source identifiers
print("\nSTATUS / IDENTIFIER CONSISTENCY")

display(
    identity_map.groupby("match_status", dropna=False).agg(
        records=("cricketiq_player_id", "size"),
        missing_player_id=("player_id", lambda x: x.isna().sum()),
        missing_crickrida_name=("crickrida_name", lambda x: x.isna().sum()),
        missing_cricsheet_name=("cricsheet_name", lambda x: x.isna().sum())
    )
)


IDENTITY MAP SHAPE: (411, 12)

MAPPING STATUS COUNTS:


match_status
verified_alias                                271
exact_name_match                               98
verified_manual_identity                       23
verified_full_name_and_career                  10
crickrida_only                                  3
verified_ireland_identity                       2
verified_cross_country_identity                 1
verified_match_and_career                       1
verified_external_career_and_match_records      1
cricsheet_only                                  1
Name: count, dtype: int64


DUPLICATE CHECKS
Duplicate CricketIQ IDs: 0
Duplicate Cricsheet player IDs: 0
Duplicate normalized Crickrida names: 0
Duplicate normalized Cricsheet names: 0

MISSING VALUE CHECKS


,missing_values
cricketiq_player_id,0
player_id,4
cricsheet_name,3
crickrida_name,1
normalized_name,0
match_status,0
crickrida_player_name,202
cricsheet_player_name,204
cricsheet_player_id,204
mapping_method,146



STATUS / IDENTIFIER CONSISTENCY


,records,missing_player_id,missing_crickrida_name,missing_cricsheet_name
match_status,,,,
crickrida_only,3,3,0,3
cricsheet_only,1,1,1,0
exact_name_match,98,0,0,0
verified_alias,271,0,0,0
verified_cross_country_identity,1,0,0,0
verified_external_career_and_match_records,1,0,0,0
verified_full_name_and_career,10,0,0,0
verified_ireland_identity,2,0,0,0
verified_manual_identity,23,0,0,0


## 6. Global Source Reconciliation and Final Identity Map

Reconcile the identity map against Crickrida and Cricsheet source coverage. Validate source IDs and identity status, account for source-only records, and establish the final authoritative player identity map.

In [172]:

# ============================================================
# CRICKETIQ — GLOBAL SOURCE RECONCILIATION
# ============================================================

# ------------------------------------------------------------
# 1. CRICKRIDA SOURCE IDENTITIES
# ------------------------------------------------------------

cr_source = cr_player_history.copy()

cr_source["identity_key"] = (
    cr_source["player_name"].apply(normalize_identity)
)

# Known verified alias
cr_alias_overrides = {
    "alissislam": "alislam"
}

cr_source["identity_key"] = (
    cr_source["identity_key"]
    .replace(cr_alias_overrides)
)

cr_source_unique = cr_source.drop_duplicates(
    subset="identity_key"
).copy()

mapped_cr_names = set(
    identity_map["crickrida_name"]
    .dropna()
    .apply(normalize_identity)
)

source_cr_names = set(cr_source_unique["identity_key"])

unmapped_cr_names = source_cr_names - mapped_cr_names
extra_mapped_cr_names = mapped_cr_names - source_cr_names


# ------------------------------------------------------------
# 2. CRICSHEET SOURCE IDENTITIES
# ------------------------------------------------------------

cs_source = cs_player_history.copy()

cs_source["source_player_id"] = (
    cs_source["source_player_id"].astype(str).str.strip()
)

cs_source_unique = cs_source.drop_duplicates(
    subset="source_player_id"
).copy()

source_cs_ids = set(
    cs_source_unique["source_player_id"]
)

mapped_cs_ids = set(
    identity_map["player_id"]
    .dropna()
    .astype(str)
    .str.strip()
)

missing_cs_ids = source_cs_ids - mapped_cs_ids
extra_cs_ids = mapped_cs_ids - source_cs_ids


# ------------------------------------------------------------
# 3. SUMMARY
# ------------------------------------------------------------

print("GLOBAL SOURCE RECONCILIATION")
print("=" * 55)

print("\nCRICKRIDA")
print("Unique source identities:", len(source_cr_names))
print("Mapped identities:", len(source_cr_names & mapped_cr_names))
print("Unmapped identities:", len(unmapped_cr_names))
print("Extra map names:", len(extra_mapped_cr_names))

print("\nCRICSHEET")
print("Unique source player IDs:", len(source_cs_ids))
print("Mapped player IDs:", len(source_cs_ids & mapped_cs_ids))
print("Unmapped player IDs:", len(missing_cs_ids))
print("Extra map player IDs:", len(extra_cs_ids))


# ------------------------------------------------------------
# 4. DISPLAY UNMATCHED IDENTITIES
# ------------------------------------------------------------

print("\nCRICKRIDA IDENTITIES NOT FOUND IN MAP")

display(
    cr_source_unique[
        cr_source_unique["identity_key"].isin(unmapped_cr_names)
    ][
        ["team", "player_name", "crickrida_first_date",
         "crickrida_last_date", "crickrida_matches"]
    ].sort_values(["team", "player_name"])
)

print("\nCRICSHEET PLAYER IDs NOT FOUND IN MAP")

display(
    cs_source_unique[
        cs_source_unique["source_player_id"].isin(missing_cs_ids)
    ][
        ["team", "player_name", "source_player_id",
         "cricsheet_first_date", "cricsheet_last_date",
         "cricsheet_matches"]
    ].sort_values(["team", "player_name"])
)

print("\nEXTRA MAP CRICKRIDA NAMES")

display(
    identity_map[
        identity_map["crickrida_name"]
        .apply(lambda x: normalize_identity(x) if pd.notna(x) else None)
        .isin(extra_mapped_cr_names)
    ][["cricketiq_player_id", "crickrida_name", "match_status", "notes"]]
)

print("\nEXTRA MAP CRICSHEET IDs")

display(
    identity_map[
        identity_map["player_id"].astype(str).isin(extra_cs_ids)
    ][["cricketiq_player_id", "player_id", "cricsheet_name", "notes"]]
)


GLOBAL SOURCE RECONCILIATION

CRICKRIDA
Unique source identities: 454
Mapped identities: 387
Unmapped identities: 67
Extra map names: 23

CRICSHEET
Unique source player IDs: 2008
Mapped player IDs: 407
Unmapped player IDs: 1601
Extra map player IDs: 0

CRICKRIDA IDENTITIES NOT FOUND IN MAP


,team,player_name,crickrida_first_date,crickrida_last_date,crickrida_matches
4,Afghanistan,AM Ghazanfar,2024-03-07,2026-10-23,24
0,Afghanistan,Abdul Malik,2024-09-22,2024-12-21,3
1,Afghanistan,"Abdul Rahman (Afghanistan, 2023)",2023-07-11,2023-08-24,3
2,Afghanistan,Afsar Zazai,2015-01-10,2017-06-11,9
3,Afghanistan,Aftab Alam,2014-07-22,2019-06-22,22
...,...,...,...,...,...
369,Sri Lanka,Shehan Jayasuriya,2018-09-17,2018-09-17,1
370,Sri Lanka,Suranga Lakmal,2014-03-03,2019-06-04,3
371,Sri Lanka,Thisara Perera,2014-03-03,2019-06-04,4
372,Sri Lanka,Tillakaratne Dilshan,2015-02-22,2015-02-22,1



CRICSHEET PLAYER IDs NOT FOUND IN MAP


,team,player_name,source_player_id,cricsheet_first_date,cricsheet_last_date,cricsheet_matches
9,Australia,A Symonds,bd77eb62,2003-01-09,2009-05-03,116
0,Australia,AA Noffke,b69e69ed,2008-02-03,2008-02-03,1
1,Australia,AC Agar,a2421394,2015-09-08,2023-09-07,22
2,Australia,AC Gilchrist,2b6e6dec,2003-01-09,2008-03-04,116
3,Australia,AC Voges,864c199e,2007-02-20,2013-11-02,29
...,...,...,...,...,...,...
2015,Zimbabwe,TR Gripper,dd121907,2003-11-23,2003-11-26,2
2017,Zimbabwe,TS Kamunhukamwe,4e1b4323,2018-07-20,2024-01-08,12
2019,Zimbabwe,TV Mufambisi,11196470,2006-08-06,2006-10-10,4
2020,Zimbabwe,VM Nyauchi,1bd16445,2022-08-05,2023-01-21,10



EXTRA MAP CRICKRIDA NAMES


,cricketiq_player_id,crickrida_name,match_status,notes
164,CRIQ_d8e79af0,Andre Botha,verified_manual_identity,Andre Cornelius Botha. Ireland 2001-2011; ODI ...
165,CRIQ_fa1aa7e7,Dave Langford-Smith,verified_manual_identity,David Langford-Smith. Ireland 2006-2008; appea...
166,CRIQ_7c63baa7,Jeremy Bray,verified_manual_identity,Jeremy Paul Bray. Ireland 2002-2009; appeared ...
167,CRIQ_d7d744e5,Peter Gillespie,verified_manual_identity,Peter Gerard Gillespie. Ireland 1995-2007; app...
168,CRIQ_c942a53b,Kyle McCallan,verified_manual_identity,William Kyle McCallan. Ireland 1996-2009; appe...
169,CRIQ_a159b802,Albert van der Merwe,verified_manual_identity,Albert van der Merwe. Ireland 2010-2011; ODI d...
170,CRIQ_b3e9b7fa,Kenny Carroll,verified_manual_identity,Kenneth Edward Desmond Carroll. Ireland 2007; ...
171,CRIQ_7a4819e1,Thinus Fourie,verified_manual_identity,Marthinus Jacobus Fourie. Ireland 2007-2008; O...
172,CRIQ_06e649a0,Roger Whelan,verified_manual_identity,Roger Kyran Whelan. Ireland 2007; ODI debut 20...
173,CRIQ_5500c675,Peter Connell,verified_manual_identity,Peter Christopher Connell. Ireland 2008-2010; ...



EXTRA MAP CRICSHEET IDs


,cricketiq_player_id,player_id,cricsheet_name,notes


In [173]:

# ============================================================
# COUNTRY-LEVEL CRICKRIDA COVERAGE
# ============================================================

country_rows = []

for country, group in cr_source.groupby("team"):

    country_names = set(group["identity_key"])

    matched = country_names & mapped_cr_names
    missing = country_names - mapped_cr_names

    country_rows.append({
        "country": country,
        "source_players": len(country_names),
        "mapped_players": len(matched),
        "unmapped_players": len(missing)
    })

country_coverage = pd.DataFrame(country_rows)

country_coverage = country_coverage.sort_values(
    "unmapped_players",
    ascending=False
)

display(country_coverage)

print("\nTOTAL CRICKRIDA SOURCE PLAYERS:",
      country_coverage["source_players"].sum())

print("TOTAL UNMAPPED COUNTRY ENTRIES:",
      country_coverage["unmapped_players"].sum())


,country,source_players,mapped_players,unmapped_players
0,Afghanistan,58,2,56
9,Sri Lanka,40,32,8
5,Ireland,45,42,3
1,Australia,29,29,0
3,England,28,28,0
2,Bangladesh,51,51,0
6,New Zealand,23,23,0
4,India,36,36,0
7,Pakistan,39,39,0
8,South Africa,26,26,0



TOTAL CRICKRIDA SOURCE PLAYERS: 454
TOTAL UNMAPPED COUNTRY ENTRIES: 67


In [174]:

# ============================================================
# CORRECTED CRICKRIDA RECONCILIATION
# USE THE DEDICATED normalized_name COLUMN
# ============================================================

cr_source_check = cr_player_history.copy()

cr_source_check["identity_key"] = (
    cr_source_check["player_name"].apply(normalize_identity)
)

# Use the authoritative normalized identity key from the map
mapped_cr_names_corrected = set(
    identity_map["normalized_name"]
    .dropna()
    .apply(normalize_identity)
)

source_cr_names_corrected = set(
    cr_source_check["identity_key"]
)

unmapped_cr_names_corrected = (
    source_cr_names_corrected - mapped_cr_names_corrected
)

print("CORRECTED CRICKRIDA RECONCILIATION")
print("=" * 55)

print("Unique source identities:", len(source_cr_names_corrected))
print(
    "Mapped identities:",
    len(source_cr_names_corrected & mapped_cr_names_corrected)
)
print("Unmapped identities:", len(unmapped_cr_names_corrected))


# ------------------------------------------------------------
# COUNTRY-LEVEL COVERAGE
# ------------------------------------------------------------

country_rows_corrected = []

for country, group in cr_source_check.groupby("team"):

    country_names = set(group["identity_key"])

    matched = country_names & mapped_cr_names_corrected
    missing = country_names - mapped_cr_names_corrected

    country_rows_corrected.append({
        "country": country,
        "source_players": len(country_names),
        "mapped_players": len(matched),
        "unmapped_players": len(missing)
    })

country_coverage_corrected = pd.DataFrame(
    country_rows_corrected
).sort_values("unmapped_players", ascending=False)

print("\nCOUNTRY COVERAGE")

display(country_coverage_corrected)


# ------------------------------------------------------------
# DISPLAY REMAINING UNMAPPED PLAYERS
# ------------------------------------------------------------

print("\nREMAINING UNMAPPED CRICKRIDA IDENTITIES")

unmapped_cr_details = cr_source_check[
    cr_source_check["identity_key"].isin(
        unmapped_cr_names_corrected
    )
].drop_duplicates(subset="identity_key")

display(
    unmapped_cr_details[
        [
            "team",
            "player_name",
            "identity_key",
            "crickrida_first_date",
            "crickrida_last_date",
            "crickrida_matches"
        ]
    ].sort_values(["team", "player_name"])
)


CORRECTED CRICKRIDA RECONCILIATION
Unique source identities: 454
Mapped identities: 386
Unmapped identities: 68

COUNTRY COVERAGE


,country,source_players,mapped_players,unmapped_players
0,Afghanistan,58,2,56
9,Sri Lanka,40,32,8
5,Ireland,45,42,3
2,Bangladesh,51,50,1
3,England,28,28,0
1,Australia,29,29,0
6,New Zealand,23,23,0
4,India,36,36,0
7,Pakistan,39,39,0
8,South Africa,26,26,0



REMAINING UNMAPPED CRICKRIDA IDENTITIES


,team,player_name,identity_key,crickrida_first_date,crickrida_last_date,crickrida_matches
4,Afghanistan,AM Ghazanfar,amghazanfar,2024-03-07,2026-10-23,24
0,Afghanistan,Abdul Malik,abdulmalik,2024-09-22,2024-12-21,3
1,Afghanistan,"Abdul Rahman (Afghanistan, 2023)",abdulrahmanafghanistan2023,2023-07-11,2023-08-24,3
2,Afghanistan,Afsar Zazai,afsarzazai,2015-01-10,2017-06-11,9
3,Afghanistan,Aftab Alam,aftabalam,2014-07-22,2019-06-22,22
...,...,...,...,...,...,...
369,Sri Lanka,Shehan Jayasuriya,shehanjayasuriya,2018-09-17,2018-09-17,1
370,Sri Lanka,Suranga Lakmal,surangalakmal,2014-03-03,2019-06-04,3
371,Sri Lanka,Thisara Perera,thisaraperera,2014-03-03,2019-06-04,4
372,Sri Lanka,Tillakaratne Dilshan,tillakaratnedilshan,2015-02-22,2015-02-22,1


In [175]:

# ============================================================
# AFGHANISTAN — STAGE ALL 56 CRICKRIDA-ONLY PLAYERS
# ============================================================

# Extract Afghanistan source identities that are currently unmapped
afg_unmapped_source = (
    cr_source_check[
        (cr_source_check["team"].str.lower() == "afghanistan")
        & (
            cr_source_check["identity_key"].isin(
                unmapped_cr_names_corrected
            )
        )
    ]
    .drop_duplicates(subset="identity_key")
    .copy()
)

afg_staged_rows = []

for _, player in afg_unmapped_source.iterrows():

    player_name = player["player_name"]
    identity_key = player["identity_key"]

    afg_staged_rows.append({
        "cricketiq_player_id": f"CR_{identity_key}",
        "player_id": None,
        "cricsheet_name": None,
        "crickrida_name": player_name,
        "normalized_name": identity_key,
        "match_status": "crickrida_only",
        "crickrida_player_name": player_name,
        "cricsheet_player_name": None,
        "cricsheet_player_id": None,
        "mapping_method": "verified_source_coverage",
        "confidence": "high",
        "notes": (
            "Afghanistan is excluded from the Cricsheet dataset. "
            "Retained as a Crickrida-only identity. "
            "No cross-country matching performed."
        )
    })

afg_identity_staging = pd.DataFrame(afg_staged_rows)

print("Afghanistan players staged:", len(afg_identity_staging))

display(
    afg_identity_staging[
        [
            "cricketiq_player_id",
            "crickrida_name",
            "normalized_name",
            "match_status",
            "mapping_method",
            "confidence"
        ]
    ].sort_values("crickrida_name")
)


Afghanistan players staged: 56


,cricketiq_player_id,crickrida_name,normalized_name,match_status,mapping_method,confidence
4,CR_amghazanfar,AM Ghazanfar,amghazanfar,crickrida_only,verified_source_coverage,high
0,CR_abdulmalik,Abdul Malik,abdulmalik,crickrida_only,verified_source_coverage,high
1,CR_abdulrahmanafghanistan2023,"Abdul Rahman (Afghanistan, 2023)",abdulrahmanafghanistan2023,crickrida_only,verified_source_coverage,high
2,CR_afsarzazai,Afsar Zazai,afsarzazai,crickrida_only,verified_source_coverage,high
3,CR_aftabalam,Aftab Alam,aftabalam,crickrida_only,verified_source_coverage,high
5,CR_amirhamza,Amir Hamza,amirhamza,crickrida_only,verified_source_coverage,high
6,CR_asgharafghan,Asghar Afghan,asgharafghan,crickrida_only,verified_source_coverage,high
7,CR_azmatullahomarzai,Azmatullah Omarzai,azmatullahomarzai,crickrida_only,verified_source_coverage,high
8,CR_bashirahmad,Bashir Ahmad,bashirahmad,crickrida_only,verified_source_coverage,high
9,CR_bilalsami,Bilal Sami,bilalsami,crickrida_only,verified_source_coverage,high


In [176]:

# ============================================================
# AFGHANISTAN — STAGING VALIDATION
# ============================================================

existing_criq_ids = set(
    player_identity_map_updated["cricketiq_player_id"]
    .dropna().astype(str)
)

existing_normalized_names = set(
    player_identity_map_updated["normalized_name"]
    .dropna().apply(normalize_identity)
)

checks = {
    "Exactly 56 Afghan players staged":
        len(afg_identity_staging) == 56,

    "No duplicate normalized names":
        not afg_identity_staging["normalized_name"].duplicated().any(),

    "No duplicate CricketIQ IDs":
        not afg_identity_staging["cricketiq_player_id"].duplicated().any(),

    "No existing CricketIQ ID conflicts":
        not afg_identity_staging["cricketiq_player_id"]
        .isin(existing_criq_ids).any(),

    "No existing normalized name conflicts":
        not afg_identity_staging["normalized_name"]
        .isin(existing_normalized_names).any(),

    "All records have crickrida_only status":
        afg_identity_staging["match_status"]
        .eq("crickrida_only").all(),

    "All Cricsheet IDs are empty":
        afg_identity_staging["player_id"].isna().all(),

    "All Cricsheet names are empty":
        afg_identity_staging["cricsheet_name"].isna().all(),

    "All records have high confidence":
        afg_identity_staging["confidence"].eq("high").all()
}

for check, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {check}")

afg_staging_valid = all(checks.values())

print("\nFINAL VALIDATION:", afg_staging_valid)


PASS: Exactly 56 Afghan players staged
PASS: No duplicate normalized names
PASS: No duplicate CricketIQ IDs
PASS: No existing CricketIQ ID conflicts
PASS: No existing normalized name conflicts
PASS: All records have crickrida_only status
PASS: All Cricsheet IDs are empty
PASS: All Cricsheet names are empty
PASS: All records have high confidence

FINAL VALIDATION: True


In [177]:

# ============================================================
# AFGHANISTAN — INSERT 56 VALIDATED IDENTITIES
# ============================================================

# Safety check: do not insert unless staging validation passed
assert afg_staging_valid, "Staging validation failed. Insertion cancelled."

# Preserve the original map before modification
player_identity_map_backup_before_afghanistan = (
    player_identity_map_updated.copy(deep=True)
)

rows_before = len(player_identity_map_updated)

# Insert the 56 Afghanistan-only identities
player_identity_map_updated = pd.concat(
    [
        player_identity_map_updated,
        afg_identity_staging
    ],
    ignore_index=True
)

rows_after = len(player_identity_map_updated)

print("Rows before insertion:", rows_before)
print("Rows inserted:", rows_after - rows_before)
print("Rows after insertion:", rows_after)

# Verify uniqueness
print(
    "\nDuplicate CricketIQ IDs:",
    player_identity_map_updated["cricketiq_player_id"].duplicated().sum()
)

print(
    "Duplicate normalized names:",
    player_identity_map_updated["normalized_name"]
    .dropna()
    .apply(normalize_identity)
    .duplicated().sum()
)

print(
    "Afghanistan-only records:",
    (
        player_identity_map_updated["match_status"]
        .eq("crickrida_only")
        & player_identity_map_updated["notes"]
        .fillna("")
        .str.contains("Afghanistan is excluded", case=False)
    ).sum()
)


Rows before insertion: 411
Rows inserted: 56
Rows after insertion: 467

Duplicate CricketIQ IDs: 0
Duplicate normalized names: 0
Afghanistan-only records: 58


In [178]:

# ============================================================
# NEW ZEALAND — PERFORMANCE VALIDATION
# ============================================================

nz_review_ids = (
    nz_candidates_df["cricsheet_player_id"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

print("Candidate IDs being reviewed:", len(nz_review_ids))


# ------------------------------------------------------------
# BATTING PERFORMANCE
# ------------------------------------------------------------

nz_batting_review = player_batting_updated[
    player_batting_updated["batter_id"]
    .astype(str).isin(nz_review_ids)
].copy()

print("\nBATTING PERFORMANCE SUMMARY")

display(
    nz_batting_review.groupby(
        ["batter_id", "batter"],
        dropna=False
    ).agg(
        innings=("match_id", "nunique"),
        total_runs=("runs", "sum"),
        total_balls=("balls_faced", "sum")
    ).reset_index().sort_values("batter")
)


# ------------------------------------------------------------
# BOWLING PERFORMANCE
# ------------------------------------------------------------

nz_bowling_review = player_bowling_updated[
    player_bowling_updated["bowler_id"]
    .astype(str).isin(nz_review_ids)
].copy()

print("\nBOWLING PERFORMANCE SUMMARY")

display(
    nz_bowling_review.groupby(
        ["bowler_id", "bowler"],
        dropna=False
    ).agg(
        innings=("match_id", "nunique"),
        wickets=("wickets", "sum"),
        legal_balls=("legal_balls", "sum")
    ).reset_index().sort_values("bowler")
)


# ------------------------------------------------------------
# EXISTING IDENTITY MAP CONFLICT CHECK
# ------------------------------------------------------------

print("\nEXISTING IDENTITY MAP RECORDS")

display(
    player_identity_map_updated[
        player_identity_map_updated["player_id"]
        .astype(str).isin(nz_review_ids)
    ][
        [
            "cricketiq_player_id",
            "player_id",
            "cricsheet_name",
            "crickrida_name",
            "match_status",
            "confidence",
            "notes"
        ]
    ].to_string(index=False)
)


Candidate IDs being reviewed: 22

BATTING PERFORMANCE SUMMARY


,batter_id,batter,innings,total_runs,total_balls
3,350bb1b1,AF Milne,22,180,205
13,b8a55852,BB McCullum,186,5157,5347
11,af2c687b,C Munro,52,1249,1190
6,8abdf100,CJ Anderson,43,1102,1012
21,eade4650,DJ Mitchell,56,2810,2968
17,d7c6af50,DL Vettori,94,1440,1690
18,df5a6881,DP Conway,41,1672,1908
15,c03449e0,GD Elliott,66,1913,2344
9,9a46c4e5,GD Phillips,38,1191,1143
8,9219eff0,JDS Neesham,65,1508,1501



BOWLING PERFORMANCE SUMMARY


,bowler_id,bowler,innings,wickets,legal_balls
3,350bb1b1,AF Milne,47,56,2182
9,af2c687b,C Munro,25,7,552
5,8abdf100,CJ Anderson,42,58,1445
17,eade4650,DJ Mitchell,18,14,339
14,d7c6af50,DL Vettori,151,183,7805
12,c03449e0,GD Elliott,52,35,1200
7,9a46c4e5,GD Phillips,31,16,798
6,9219eff0,JDS Neesham,71,66,2388
13,d027ba9f,KS Williamson,65,37,1467
2,2f9d0389,LH Ferguson,62,92,3203



EXISTING IDENTITY MAP RECORDS


'cricketiq_player_id player_id cricsheet_name   crickrida_name   match_status confidence                                                                                                                                                      notes\n      CRIQ_350bb1b1  350bb1b1       AF Milne       Adam Milne verified_alias       high                                                      Verified using matching name initials, career-date overlap and Cricsheet batting/bowling performance.\n      CRIQ_b8a55852  b8a55852    BB McCullum Brendon McCullum verified_alias       high Verified as Brendon McCullum using BB McCullum. NL McCullum is Nathan McCullum, a different player. Batting performance supports the BB McCullum identity.\n      CRIQ_af2c687b  af2c687b        C Munro      Colin Munro verified_alias       high                                                      Verified using matching name initials, career-date overlap and Cricsheet batting/bowling performance.\n      CRIQ_8abdf100  8a

In [179]:
afg_source_names = set(
    cr_player_history.loc[
        cr_player_history["team"].str.lower() == "afghanistan",
        "normalized_name"
    ].apply(normalize_identity)
)

afg_map_names = set(
    player_identity_map_updated["normalized_name"]
    .dropna()
    .apply(normalize_identity)
)

afg_unmapped = afg_source_names - afg_map_names

print("Afghanistan source players:", len(afg_source_names))
print("Afghanistan mapped players:", len(afg_source_names & afg_map_names))
print("Afghanistan unmapped players:", len(afg_unmapped))
print("Unmapped names:", sorted(afg_unmapped))

assert len(afg_source_names) == 58
assert len(afg_unmapped) == 0

print("\nAFGHANISTAN AUDIT: PASS")

Afghanistan source players: 58
Afghanistan mapped players: 58
Afghanistan unmapped players: 0
Unmapped names: []

AFGHANISTAN AUDIT: PASS


In [180]:

# ============================================================
# GLOBAL RECONCILIATION — REMAINING CRICKRIDA IDENTITIES
# Uses the latest identity map, including Afghanistan additions
# ============================================================

# Build normalized source identities
cr_source_audit = cr_player_history.copy()

cr_source_audit["identity_key"] = (
    cr_source_audit["player_name"]
    .apply(normalize_identity)
)

# Build normalized identity-map keys
mapped_identity_keys = set(
    player_identity_map_updated["normalized_name"]
    .dropna()
    .apply(normalize_identity)
)

# Find source identities not represented in the map
unmapped_cr_details = (
    cr_source_audit[
        ~cr_source_audit["identity_key"].isin(mapped_identity_keys)
    ]
    .drop_duplicates(subset=["team", "identity_key"])
    .copy()
)

# Focus on the three countries requiring investigation
countries_to_review = ["Sri Lanka", "Ireland", "Bangladesh"]

remaining_12 = (
    unmapped_cr_details[
        unmapped_cr_details["team"].isin(countries_to_review)
    ]
    [
        [
            "team",
            "player_name",
            "identity_key",
            "crickrida_first_date",
            "crickrida_last_date",
            "crickrida_matches"
        ]
    ]
    .sort_values(["team", "player_name"])
    .reset_index(drop=True)
)

print("TOTAL UNIQUE CRICKRIDA SOURCE IDENTITIES:",
      cr_source_audit["identity_key"].nunique())

print("TOTAL UNMAPPED IDENTITIES:",
      len(unmapped_cr_details))

print("\nUNMAPPED IDENTITIES BY COUNTRY:")
display(unmapped_cr_details["team"].value_counts())

print("\nIDENTITIES REQUIRING INVESTIGATION:")
display(remaining_12.to_string(index=False))

print("\nCOUNTRY COUNTS:")
display(remaining_12["team"].value_counts())


TOTAL UNIQUE CRICKRIDA SOURCE IDENTITIES: 454
TOTAL UNMAPPED IDENTITIES: 12

UNMAPPED IDENTITIES BY COUNTRY:


team
Sri Lanka     8
Ireland       3
Bangladesh    1
Name: count, dtype: int64


IDENTITIES REQUIRING INVESTIGATION:


'      team            player_name          identity_key crickrida_first_date crickrida_last_date  crickrida_matches\nBangladesh               Al Islam               alislam           2026-10-21          2026-10-21                  1\n   Ireland             Ben Calitz             bencalitz           2026-08-07          2026-08-12                  3\n   Ireland      James Cameron-Dow       jamescamerondow           2019-03-02          2019-03-10                  4\n   Ireland             James Hall             jameshall           2010-07-03          2010-07-03                  1\n Sri Lanka        Pathum Nissanka        pathumnissanka           2022-11-25          2024-02-14                 11\n Sri Lanka         Rangana Herath         ranganaherath           2015-02-22          2015-02-22                  1\n Sri Lanka Sadeera Samarawickrama sadeerasamarawickrama           2023-06-04          2024-02-14                  6\n Sri Lanka      Shehan Jayasuriya      shehanjayasuriya        


COUNTRY COUNTS:


team
Sri Lanka     8
Ireland       3
Bangladesh    1
Name: count, dtype: int64

In [181]:

# ============================================================
# INVESTIGATE THE 12 UNMAPPED CRICKRIDA IDENTITIES
# READ-ONLY: NO INSERTIONS OR MODIFICATIONS
# ============================================================

review_players = [
    ("Bangladesh", "Al Islam"),
    ("Ireland", "Ben Calitz"),
    ("Ireland", "James Cameron-Dow"),
    ("Ireland", "James Hall"),
    ("Sri Lanka", "Pathum Nissanka"),
    ("Sri Lanka", "Rangana Herath"),
    ("Sri Lanka", "Sadeera Samarawickrama"),
    ("Sri Lanka", "Shehan Jayasuriya"),
    ("Sri Lanka", "Suranga Lakmal"),
    ("Sri Lanka", "Thisara Perera"),
    ("Sri Lanka", "Tillakaratne Dilshan"),
    ("Sri Lanka", "Upul Tharanga"),
]

# Prepare searchable versions of the identity map
map_search = player_identity_map_updated.copy()

for col in ["normalized_name", "crickrida_name", "cricsheet_name"]:
    map_search[f"{col}_key"] = (
        map_search[col].fillna("").apply(normalize_identity)
    )

# Prepare Cricsheet history
cs_search = cs_player_history.copy()

cs_search["name_key"] = (
    cs_search["player_name"].fillna("").apply(normalize_identity)
)

for country, player_name in review_players:

    key = normalize_identity(player_name)
    surname = normalize_identity(player_name.split()[-1])

    print("\n" + "=" * 85)
    print(f"{country.upper()} — {player_name}")
    print("Source identity key:", key)

    # 1. Exact identity or display-name match in current map
    exact_matches = map_search[
        (map_search["normalized_name_key"] == key)
        | (map_search["crickrida_name_key"] == key)
    ]

    print("\nEXACT EXISTING MAP MATCHES:")
    if len(exact_matches):
        display(
            exact_matches[
                [
                    "cricketiq_player_id",
                    "player_id",
                    "cricsheet_name",
                    "crickrida_name",
                    "normalized_name",
                    "match_status",
                    "confidence"
                ]
            ]
        )
    else:
        print("None")

    # 2. Existing map entries with the same surname
    surname_matches = map_search[
        map_search["crickrida_name_key"].str.endswith(surname)
        | map_search["cricsheet_name_key"].str.endswith(surname)
    ]

    print("\nEXISTING MAP SURNAME CANDIDATES:")
    if len(surname_matches):
        display(
            surname_matches[
                [
                    "cricketiq_player_id",
                    "player_id",
                    "cricsheet_name",
                    "crickrida_name",
                    "match_status"
                ]
            ].drop_duplicates()
        )
    else:
        print("None")

    # 3. Cricsheet history candidates from the same surname
    cs_candidates = cs_search[
        (cs_search["team"].str.lower() == country.lower())
        & cs_search["name_key"].str.endswith(surname)
    ]

    print("\nCRICSHEET SOURCE CANDIDATES:")
    if len(cs_candidates):
        display(
            cs_candidates[
                [
                    "team",
                    "player_name",
                    "source_player_id",
                    "cricsheet_first_date",
                    "cricsheet_last_date",
                    "cricsheet_matches"
                ]
            ].drop_duplicates()
        )
    else:
        print("None")



BANGLADESH — Al Islam
Source identity key: alislam

EXACT EXISTING MAP MATCHES:


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,normalized_name,match_status,confidence
213,CR_aliss_islam,None,None,Al Islam,alissislam,crickrida_only,high



EXISTING MAP SURNAME CANDIDATES:


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,match_status
34,CRIQ_1a8eebbe,1a8eebbe,Shafiul Islam,Shafiul Islam,exact_name_match
50,CRIQ_6575e3bc,6575e3bc,Nazmul Islam,Nazmul Islam,exact_name_match
53,CRIQ_9d1d34bd,9d1d34bd,Taijul Islam,Taijul Islam,exact_name_match
61,CRIQ_bb34fd31,bb34fd31,Shoriful Islam,Shoriful Islam,exact_name_match
80,CRIQ_aa467af0,aa467af0,Tanvir Islam,Tanvir Islam,exact_name_match
91,CRIQ_02e83ac5,02e83ac5,Naeem Islam,Naeem Islam,exact_name_match
213,CR_aliss_islam,None,None,Al Islam,crickrida_only



CRICSHEET SOURCE CANDIDATES:


,team,player_name,source_player_id,cricsheet_first_date,cricsheet_last_date,cricsheet_matches
154,Bangladesh,Jahurul Islam,6752c57b,2010-06-21,2013-05-08,11
168,Bangladesh,Manjural Islam,2cfa6534,2003-02-11,2003-03-01,5
186,Bangladesh,Naeem Islam,02e83ac5,2008-11-07,2014-02-26,50
194,Bangladesh,Nazmul Islam,6575e3bc,2018-09-28,2018-10-26,4
200,Bangladesh,Robiul Islam,d876610c,2013-05-03,2013-05-08,3
206,Bangladesh,Shafiul Islam,1a8eebbe,2010-01-04,2020-03-03,58
213,Bangladesh,Shoriful Islam,bb34fd31,2021-05-25,2026-10-17,38
219,Bangladesh,Sunzamul Islam,0f348f58,2017-05-19,2018-01-23,3
221,Bangladesh,Taijul Islam,9d1d34bd,2014-12-01,2024-03-15,17
225,Bangladesh,Tanvir Islam,aa467af0,2025-07-02,2026-07-11,11


... (14 outputs left out)


SRI LANKA — Tillakaratne Dilshan
Source identity key: tillakaratnedilshan

EXACT EXISTING MAP MATCHES:
None

EXISTING MAP SURNAME CANDIDATES:
None

CRICSHEET SOURCE CANDIDATES:


,team,player_name,source_player_id,cricsheet_first_date,cricsheet_last_date,cricsheet_matches
1598,Sri Lanka,TM Dilshan,5bdcdb72,2003-05-19,2016-08-28,267



SRI LANKA — Upul Tharanga
Source identity key: upultharanga

EXACT EXISTING MAP MATCHES:
None

EXISTING MAP SURNAME CANDIDATES:
None

CRICSHEET SOURCE CANDIDATES:


,team,player_name,source_player_id,cricsheet_first_date,cricsheet_last_date,cricsheet_matches
1607,Sri Lanka,WU Tharanga,7ed9fd56,2005-08-02,2019-03-16,205


In [182]:

# ============================================================
# SRI LANKA — CANDIDATE PERFORMANCE VALIDATION
# READ-ONLY: DOES NOT MODIFY THE IDENTITY MAP
# ============================================================

candidate_ids = {
    "Pathum Nissanka": ["8ee36b18"],
    "Rangana Herath": ["d2fcce2f"],
    "Sadeera Samarawickrama": ["9ab2914d"],
    "Shehan Jayasuriya": [
        "b58b7db6",  # GSNFG Jayasuriya
        "127bc5ca",  # NGRP Jayasuriya
        "f233bbb4"   # ST Jayasuriya
    ],
    "Suranga Lakmal": ["2f28dc94"],
    "Thisara Perera": ["0f12f9df"],
    "Tillakaratne Dilshan": ["5bdcdb72"],
    "Upul Tharanga": ["7ed9fd56"]
}

all_candidate_ids = [
    player_id
    for ids in candidate_ids.values()
    for player_id in ids
]

# Source history details
candidate_history = (
    cs_player_history[
        cs_player_history["source_player_id"].astype(str).isin(
            all_candidate_ids
        )
    ][
        [
            "team",
            "player_name",
            "source_player_id",
            "cricsheet_first_date",
            "cricsheet_last_date",
            "cricsheet_matches"
        ]
    ]
    .drop_duplicates()
    .copy()
)

# Batting performance
batting_summary = (
    player_batting_updated[
        player_batting_updated["batter_id"].astype(str).isin(
            all_candidate_ids
        )
    ]
    .groupby("batter_id")
    .agg(
        batting_innings=("match_id", "nunique"),
        total_runs=("runs", "sum"),
        total_balls=("balls_faced", "sum")
    )
    .reset_index()
    .rename(columns={"batter_id": "source_player_id"})
)

# Bowling performance
bowling_summary = (
    player_bowling_updated[
        player_bowling_updated["bowler_id"].astype(str).isin(
            all_candidate_ids
        )
    ]
    .groupby("bowler_id")
    .agg(
        bowling_innings=("match_id", "nunique"),
        wickets=("wickets", "sum"),
        legal_balls=("legal_balls", "sum")
    )
    .reset_index()
    .rename(columns={"bowler_id": "source_player_id"})
)

# Combine source history and performance
candidate_report = (
    candidate_history
    .merge(batting_summary, on="source_player_id", how="left")
    .merge(bowling_summary, on="source_player_id", how="left")
)

print("SRI LANKA CANDIDATE HISTORY AND PERFORMANCE")
display(
    candidate_report.sort_values(
        ["player_name", "source_player_id"]
    ).fillna(0).to_string(index=False)
)

print("\nSHEHAN JAYASURIYA CANDIDATES ONLY")
display(
    candidate_report[
        candidate_report["source_player_id"].isin(
            ["b58b7db6", "127bc5ca", "f233bbb4"]
        )
    ].fillna(0).to_string(index=False)
)


SRI LANKA CANDIDATE HISTORY AND PERFORMANCE


'                    team      player_name source_player_id cricsheet_first_date cricsheet_last_date  cricsheet_matches  batting_innings  total_runs  total_balls  bowling_innings  wickets  legal_balls\n               Sri Lanka GSNFG Jayasuriya         b58b7db6           2015-11-01          2019-10-02                 11               20         682          827             15.0      6.0        402.0\nUnited States of America GSNFG Jayasuriya         b58b7db6           2026-05-14          2026-10-13                 11               20         682          827             15.0      6.0        402.0\n               Sri Lanka     HMRKB Herath         d2fcce2f           2004-04-27          2015-03-01                 64               29         140          217             63.0     68.0       3026.0\n               Sri Lanka  NGRP Jayasuriya         127bc5ca           2018-08-01          2018-08-05                  2                2          11           13              2.0      0.0         


SHEHAN JAYASURIYA CANDIDATES ONLY


'                    team      player_name source_player_id cricsheet_first_date cricsheet_last_date  cricsheet_matches  batting_innings  total_runs  total_balls  bowling_innings  wickets  legal_balls\n               Sri Lanka GSNFG Jayasuriya         b58b7db6           2015-11-01          2019-10-02                 11               20         682          827             15.0      6.0        402.0\n               Sri Lanka  NGRP Jayasuriya         127bc5ca           2018-08-01          2018-08-05                  2                2          11           13              2.0      0.0         96.0\n               Sri Lanka    ST Jayasuriya         f233bbb4           2002-06-27          2011-06-28                131              130        4376         4573            103.0     86.0       3883.0\nUnited States of America GSNFG Jayasuriya         b58b7db6           2026-05-14          2026-10-13                 11               20         682          827             15.0      6.0        4

In [183]:

# ============================================================
# STAGE REMAINING COUNTRY IDENTITIES
# 8 Sri Lanka shared + 3 Ireland source-only
# Bangladesh key correction staged separately
# ============================================================

verified_sl_mappings = [
    ("Pathum Nissanka", "P Nissanka", "8ee36b18"),
    ("Rangana Herath", "HMRKB Herath", "d2fcce2f"),
    ("Sadeera Samarawickrama", "S Samarawickrama", "9ab2914d"),
    ("Shehan Jayasuriya", "GSNFG Jayasuriya", "b58b7db6"),
    ("Suranga Lakmal", "RAS Lakmal", "2f28dc94"),
    ("Thisara Perera", "NLTC Perera", "0f12f9df"),
    ("Tillakaratne Dilshan", "TM Dilshan", "5bdcdb72"),
    ("Upul Tharanga", "WU Tharanga", "7ed9fd56")
]

verified_ireland_source_only = [
    "Ben Calitz",
    "James Cameron-Dow",
    "James Hall"
]

staged_rows = []

# ------------------------------------------------------------
# A. Sri Lanka shared identities
# ------------------------------------------------------------

for crickrida_name, cricsheet_name, player_id in verified_sl_mappings:

    identity_key = normalize_identity(crickrida_name)

    staged_rows.append({
        "cricketiq_player_id": f"CRIQ_{player_id}",
        "player_id": player_id,
        "cricsheet_name": cricsheet_name,
        "crickrida_name": crickrida_name,
        "normalized_name": identity_key,
        "match_status": "verified_alias",
        "crickrida_player_name": crickrida_name,
        "cricsheet_player_name": cricsheet_name,
        "cricsheet_player_id": player_id,
        "mapping_method": "verified_full_name_career_and_performance_review",
        "confidence": "high",
        "notes": (
            "Verified using player identity, matching name initials, "
            "career-date overlap and Cricsheet performance records."
        )
    })

# ------------------------------------------------------------
# B. Ireland source-only identities
# ------------------------------------------------------------

for player_name in verified_ireland_source_only:

    identity_key = normalize_identity(player_name)

    staged_rows.append({
        "cricketiq_player_id": f"CR_{identity_key}",
        "player_id": None,
        "cricsheet_name": None,
        "crickrida_name": player_name,
        "normalized_name": identity_key,
        "match_status": "crickrida_only",
        "crickrida_player_name": player_name,
        "cricsheet_player_name": None,
        "cricsheet_player_id": None,
        "mapping_method": "verified_source_coverage",
        "confidence": "high",
        "notes": (
            "Ireland identity exists in Crickrida source history "
            "but has no corresponding player in the Cricsheet "
            "Ireland source history. Retained as Crickrida-only."
        )
    })

remaining_identity_staging = pd.DataFrame(staged_rows)

# ------------------------------------------------------------
# C. Stage Bangladesh normalized-key correction
# ------------------------------------------------------------

al_islam_rows = player_identity_map_updated[
    player_identity_map_updated["cricketiq_player_id"]
    == "CR_aliss_islam"
]

assert len(al_islam_rows) == 1, (
    "Expected exactly one existing Al Islam identity."
)

al_islam_old_key = al_islam_rows.iloc[0]["normalized_name"]
al_islam_new_key = normalize_identity(
    al_islam_rows.iloc[0]["crickrida_name"]
)

print("NEW RECORDS STAGED:", len(remaining_identity_staging))
print("Sri Lanka shared:", len(verified_sl_mappings))
print("Ireland source-only:", len(verified_ireland_source_only))

print("\nBANGLADESH KEY CORRECTION")
print("CricketIQ ID: CR_aliss_islam")
print("Old normalized key:", al_islam_old_key)
print("Proposed normalized key:", al_islam_new_key)


NEW RECORDS STAGED: 11
Sri Lanka shared: 8
Ireland source-only: 3

BANGLADESH KEY CORRECTION
CricketIQ ID: CR_aliss_islam
Old normalized key: alissislam
Proposed normalized key: alislam


In [184]:

# ============================================================
# VALIDATE REMAINING IDENTITY STAGING
# ============================================================

existing_criq_ids = set(
    player_identity_map_updated["cricketiq_player_id"]
    .dropna().astype(str)
)

existing_player_ids = set(
    player_identity_map_updated["player_id"]
    .dropna().astype(str)
)

existing_normalized_keys = set(
    player_identity_map_updated["normalized_name"]
    .dropna().apply(normalize_identity)
)

staged_shared = remaining_identity_staging[
    remaining_identity_staging["match_status"] == "verified_alias"
]

staged_source_only = remaining_identity_staging[
    remaining_identity_staging["match_status"] == "crickrida_only"
]

checks = {
    "Exactly 11 new rows staged":
        len(remaining_identity_staging) == 11,

    "Exactly 8 Sri Lankan shared mappings":
        len(staged_shared) == 8,

    "Exactly 3 Irish source-only identities":
        len(staged_source_only) == 3,

    "No duplicate staged CricketIQ IDs":
        not remaining_identity_staging[
            "cricketiq_player_id"
        ].duplicated().any(),

    "No duplicate staged normalized names":
        not remaining_identity_staging[
            "normalized_name"
        ].duplicated().any(),

    "No existing CricketIQ ID conflicts":
        not remaining_identity_staging[
            "cricketiq_player_id"
        ].isin(existing_criq_ids).any(),

    "No existing player ID conflicts":
        not staged_shared["player_id"].isin(
            existing_player_ids
        ).any(),

    "No existing normalized-name conflicts":
        not remaining_identity_staging[
            "normalized_name"
        ].isin(existing_normalized_keys).any(),

    "All Ireland source-only Cricsheet IDs empty":
        staged_source_only["player_id"].isna().all()
        and staged_source_only["cricsheet_player_id"].isna().all(),

    "Al Islam correction has no existing key conflict":
        al_islam_new_key not in existing_normalized_keys
}

for description, passed in checks.items():
    print(("PASS" if passed else "FAIL") + ": " + description)

remaining_staging_valid = all(checks.values())

print("\nFINAL VALIDATION:", remaining_staging_valid)

display(
    remaining_identity_staging[
        [
            "cricketiq_player_id",
            "player_id",
            "cricsheet_name",
            "crickrida_name",
            "match_status",
            "confidence"
        ]
    ]
)


PASS: Exactly 11 new rows staged
PASS: Exactly 8 Sri Lankan shared mappings
PASS: Exactly 3 Irish source-only identities
PASS: No duplicate staged CricketIQ IDs
PASS: No duplicate staged normalized names
PASS: No existing CricketIQ ID conflicts
PASS: No existing player ID conflicts
PASS: No existing normalized-name conflicts
PASS: All Ireland source-only Cricsheet IDs empty
PASS: Al Islam correction has no existing key conflict

FINAL VALIDATION: True


,cricketiq_player_id,player_id,cricsheet_name,crickrida_name,match_status,confidence
0,CRIQ_8ee36b18,8ee36b18,P Nissanka,Pathum Nissanka,verified_alias,high
1,CRIQ_d2fcce2f,d2fcce2f,HMRKB Herath,Rangana Herath,verified_alias,high
2,CRIQ_9ab2914d,9ab2914d,S Samarawickrama,Sadeera Samarawickrama,verified_alias,high
3,CRIQ_b58b7db6,b58b7db6,GSNFG Jayasuriya,Shehan Jayasuriya,verified_alias,high
4,CRIQ_2f28dc94,2f28dc94,RAS Lakmal,Suranga Lakmal,verified_alias,high
5,CRIQ_0f12f9df,0f12f9df,NLTC Perera,Thisara Perera,verified_alias,high
6,CRIQ_5bdcdb72,5bdcdb72,TM Dilshan,Tillakaratne Dilshan,verified_alias,high
7,CRIQ_7ed9fd56,7ed9fd56,WU Tharanga,Upul Tharanga,verified_alias,high
8,CR_bencalitz,None,None,Ben Calitz,crickrida_only,high
9,CR_jamescamerondow,None,None,James Cameron-Dow,crickrida_only,high


In [185]:

# ============================================================
# FINAL INSERTION — SRI LANKA + IRELAND + BANGLADESH CORRECTION
# ============================================================

assert remaining_staging_valid, (
    "Validation failed. No changes made."
)

# Preserve backup
player_identity_map_backup_before_final_12 = (
    player_identity_map_updated.copy(deep=True)
)

rows_before = len(player_identity_map_updated)

# Insert 11 validated records
player_identity_map_updated = pd.concat(
    [
        player_identity_map_updated,
        remaining_identity_staging
    ],
    ignore_index=True
)

# Correct the existing Al Islam normalized key
al_islam_index = player_identity_map_updated.index[
    player_identity_map_updated["cricketiq_player_id"]
    == "CR_aliss_islam"
]

assert len(al_islam_index) == 1

player_identity_map_updated.loc[
    al_islam_index[0], "normalized_name"
] = "alislam"

rows_after = len(player_identity_map_updated)

# ------------------------------------------------------------
# POST-INSERTION VALIDATION
# ------------------------------------------------------------

duplicate_criq_ids = (
    player_identity_map_updated[
        "cricketiq_player_id"
    ].dropna().duplicated().sum()
)

duplicate_player_ids = (
    player_identity_map_updated[
        "player_id"
    ].dropna().duplicated().sum()
)

duplicate_normalized_names = (
    player_identity_map_updated[
        "normalized_name"
    ].dropna().apply(normalize_identity).duplicated().sum()
)

print("ROWS BEFORE:", rows_before)
print("ROWS INSERTED:", rows_after - rows_before)
print("ROWS AFTER:", rows_after)

print("\nDUPLICATE CHECKS")
print("Duplicate CricketIQ IDs:", duplicate_criq_ids)
print("Duplicate Cricsheet player IDs:", duplicate_player_ids)
print("Duplicate normalized names:", duplicate_normalized_names)

print("\nAL ISLAM CORRECTION")
display(
    player_identity_map_updated[
        player_identity_map_updated["cricketiq_player_id"]
        == "CR_aliss_islam"
    ][
        [
            "cricketiq_player_id",
            "crickrida_name",
            "normalized_name",
            "match_status"
        ]
    ]
)

assert rows_before == 467
assert rows_after == 478
assert duplicate_criq_ids == 0
assert duplicate_player_ids == 0
assert duplicate_normalized_names == 0

print("\nFINAL INSERTION: PASS")


ROWS BEFORE: 467
ROWS INSERTED: 11
ROWS AFTER: 478

DUPLICATE CHECKS
Duplicate CricketIQ IDs: 0
Duplicate Cricsheet player IDs: 0
Duplicate normalized names: 0

AL ISLAM CORRECTION


,cricketiq_player_id,crickrida_name,normalized_name,match_status
213,CR_aliss_islam,Al Islam,alislam,crickrida_only



FINAL INSERTION: PASS


In [186]:

# ============================================================
# GLOBAL RECONCILIATION — REMAINING CRICKRIDA IDENTITIES
# Uses the latest identity map, including Afghanistan additions
# ============================================================

# Build normalized source identities
cr_source_audit = cr_player_history.copy()

cr_source_audit["identity_key"] = (
    cr_source_audit["player_name"]
    .apply(normalize_identity)
)

# Build normalized identity-map keys
mapped_identity_keys = set(
    player_identity_map_updated["normalized_name"]
    .dropna()
    .apply(normalize_identity)
)

# Find source identities not represented in the map
unmapped_cr_details = (
    cr_source_audit[
        ~cr_source_audit["identity_key"].isin(mapped_identity_keys)
    ]
    .drop_duplicates(subset=["team", "identity_key"])
    .copy()
)

# Focus on the three countries requiring investigation
countries_to_review = ["Sri Lanka", "Ireland", "Bangladesh"]

remaining_12 = (
    unmapped_cr_details[
        unmapped_cr_details["team"].isin(countries_to_review)
    ]
    [
        [
            "team",
            "player_name",
            "identity_key",
            "crickrida_first_date",
            "crickrida_last_date",
            "crickrida_matches"
        ]
    ]
    .sort_values(["team", "player_name"])
    .reset_index(drop=True)
)

print("TOTAL UNIQUE CRICKRIDA SOURCE IDENTITIES:",
      cr_source_audit["identity_key"].nunique())

print("TOTAL UNMAPPED IDENTITIES:",
      len(unmapped_cr_details))

print("\nUNMAPPED IDENTITIES BY COUNTRY:")
display(unmapped_cr_details["team"].value_counts())

print("\nIDENTITIES REQUIRING INVESTIGATION:")
display(remaining_12.to_string(index=False))

print("\nCOUNTRY COUNTS:")
display(remaining_12["team"].value_counts())


TOTAL UNIQUE CRICKRIDA SOURCE IDENTITIES: 454
TOTAL UNMAPPED IDENTITIES: 0

UNMAPPED IDENTITIES BY COUNTRY:


Series([], Name: count, dtype: int64)


IDENTITIES REQUIRING INVESTIGATION:


'Empty DataFrame\nColumns: [team, player_name, identity_key, crickrida_first_date, crickrida_last_date, crickrida_matches]\nIndex: []'


COUNTRY COUNTS:


Series([], Name: count, dtype: int64)

In [187]:

# ============================================================
# CRICSHEET-SIDE RECONCILIATION
# READ-ONLY — DO NOT MODIFY THE IDENTITY MAP
# ============================================================

cs_source_audit = cs_player_history.copy()

# Normalize source IDs
cs_source_audit["source_player_id"] = (
    cs_source_audit["source_player_id"].astype(str).str.strip()
)

# Unique Cricsheet player IDs
cs_unique_players = (
    cs_source_audit[
        ["source_player_id", "player_name", "team",
         "cricsheet_first_date", "cricsheet_last_date",
         "cricsheet_matches"]
    ]
    .sort_values("cricsheet_matches", ascending=False)
    .drop_duplicates(subset="source_player_id")
    .copy()
)

# Every Cricsheet ID represented in the identity map
mapped_cs_ids = set(
    player_identity_map_updated["player_id"]
    .dropna().astype(str)
)

mapped_cs_ids.update(
    player_identity_map_updated["cricsheet_player_id"]
    .dropna().astype(str)
)

source_cs_ids = set(cs_unique_players["source_player_id"])

unmapped_cs_ids = source_cs_ids - mapped_cs_ids

unmapped_cs_details = cs_unique_players[
    cs_unique_players["source_player_id"].isin(unmapped_cs_ids)
].copy()

print("CRICSHEET SOURCE RECONCILIATION")
print("=" * 50)
print("Unique Cricsheet player IDs:", len(source_cs_ids))
print("Mapped Cricsheet player IDs:", len(source_cs_ids & mapped_cs_ids))
print("Unmapped Cricsheet player IDs:", len(unmapped_cs_ids))

print("\nUNMAPPED PLAYER COUNTS BY COUNTRY")
display(
    unmapped_cs_details.groupby("team")
    .agg(
        unique_players=("source_player_id", "nunique"),
        total_recorded_matches=("cricsheet_matches", "sum")
    )
    .sort_values("unique_players", ascending=False)
)

print("\nUNMAPPED PLAYER COUNTS BY CAREER PERIOD")
unmapped_cs_details["first_year"] = pd.to_datetime(
    unmapped_cs_details["cricsheet_first_date"],
    errors="coerce"
).dt.year

display(
    unmapped_cs_details.groupby(
        pd.cut(
            unmapped_cs_details["first_year"],
            bins=[1900, 1999, 2009, 2019, 2029],
            labels=["Before 2000", "2000–2009", "2010–2019", "2020–2029"]
        ),
        observed=False
    )
    .agg(
        unique_players=("source_player_id", "nunique"),
        recorded_matches=("cricsheet_matches", "sum")
    )
)

print("\nTOP 30 UNMAPPED CRICSHEET PLAYERS BY MATCH COUNT")
display(
    unmapped_cs_details[
        [
            "team", "player_name", "source_player_id",
            "cricsheet_first_date", "cricsheet_last_date",
            "cricsheet_matches"
        ]
    ]
    .sort_values("cricsheet_matches", ascending=False)
    .head(30)
    .to_string(index=False)
)


CRICSHEET SOURCE RECONCILIATION
Unique Cricsheet player IDs: 2008
Mapped Cricsheet player IDs: 416
Unmapped Cricsheet player IDs: 1592

UNMAPPED PLAYER COUNTS BY COUNTRY


,unique_players,total_recorded_matches
team,,
West Indies,115,2540
Pakistan,106,1557
England,103,2443
United Arab Emirates,102,1318
India,101,2826
Australia,97,2603
Canada,96,773
New Zealand,95,1995
South Africa,93,2741



UNMAPPED PLAYER COUNTS BY CAREER PERIOD


,unique_players,recorded_matches
first_year,,
Before 2000,0,0
2000–2009,673,16964
2010–2019,449,8073
2020–2029,470,4819



TOP 30 UNMAPPED CRICSHEET PLAYERS BY MATCH COUNT


'        team     player_name source_player_id cricsheet_first_date cricsheet_last_date  cricsheet_matches\n       India    Yuvraj Singh         1c914163           2002-06-29          2017-06-30                212\nSouth Africa  AB de Villiers         c4487b84           2005-02-02          2018-02-16                209\n       India        SK Raina         1dc12ab9           2005-07-31          2018-07-17                201\n   Australia      RT Ponting         7d415ea5           2003-01-09          2012-02-19                187\nSouth Africa       JP Duminy         2e8994e7           2004-08-28          2019-07-06                186\n   Sri Lanka KMDN Kulasekara         469ea22b           2003-11-18          2017-07-10                177\n   Australia       SR Watson         4329fbb5           2003-01-09          2015-09-05                160\n       India        V Sehwag         8ba8195d           2002-06-29          2013-01-03                159\n     England  PD Collingwood        

In [188]:

# ============================================================
# IDENTITY MAP STATUS AND SOURCE COVERAGE
# READ-ONLY
# ============================================================

print("IDENTITY MAP STATUS COUNTS")
display(
    player_identity_map_updated["match_status"]
    .value_counts(dropna=False)
)

print("\nIDENTITY MAP TOTALS")
print("Total rows:", len(player_identity_map_updated))
print(
    "Rows with Crickrida identity:",
    player_identity_map_updated["crickrida_name"].notna().sum()
)
print(
    "Rows with Cricsheet player ID:",
    player_identity_map_updated["player_id"].notna().sum()
)
print(
    "Rows without Cricsheet player ID:",
    player_identity_map_updated["player_id"].isna().sum()
)

print("\nCRICSHEET-ONLY IDENTITY RECORDS")
display(
    player_identity_map_updated[
        player_identity_map_updated["match_status"] == "cricsheet_only"
    ][
        [
            "cricketiq_player_id",
            "player_id",
            "cricsheet_name",
            "match_status",
            "notes"
        ]
    ].to_string(index=False)
)

print("\nCRICKRIDA-ONLY IDENTITY RECORDS")
display(
    player_identity_map_updated[
        player_identity_map_updated["match_status"] == "crickrida_only"
    ][
        [
            "cricketiq_player_id",
            "crickrida_name",
            "normalized_name",
            "notes"
        ]
    ].to_string(index=False)
)


IDENTITY MAP STATUS COUNTS


match_status
verified_alias                                279
exact_name_match                               98
crickrida_only                                 62
verified_manual_identity                       23
verified_full_name_and_career                  10
verified_ireland_identity                       2
verified_cross_country_identity                 1
verified_match_and_career                       1
verified_external_career_and_match_records      1
cricsheet_only                                  1
Name: count, dtype: int64


IDENTITY MAP TOTALS
Total rows: 478
Rows with Crickrida identity: 477
Rows with Cricsheet player ID: 415
Rows without Cricsheet player ID: 63

CRICSHEET-ONLY IDENTITY RECORDS


'cricketiq_player_id player_id cricsheet_name   match_status                                                                                                                                                                                                                      notes\n        CS_5d9a1a73      None       R Sharma cricsheet_only Identified as Rahul Sharma, not Rohit Sharma. Four ODI match records: 518956, 536932, 536933 and 564783. One batting innings, 1 run; six wickets in four bowling records. No corresponding Crickrida India history record.'


CRICKRIDA-ONLY IDENTITY RECORDS


'                cricketiq_player_id                           crickrida_name                  normalized_name                                                                                                                                                                                 notes\n                     CR_aliss_islam                                 Al Islam                          alislam Full name identified as Aliss Islam. Crickrida Bangladesh player; reported to have played only against Afghanistan. No Cricsheet counterpart established. Retain Crickrida data only.\n                 CR_tinotendamaposa                         Tinotenda Maposa                  tinotendamaposa                                                           Zimbabwe player; one Crickrida match on 2024-12-19 against Afghanistan. Afghanistan is excluded from the Cricsheet dataset.\n                      CR_harshdubey                              Harsh Dubey                       harshdubey           

In [189]:

# ============================================================
# CRICSHEET-ONLY IDENTITY QUALITY AUDIT
# READ-ONLY — NO INSERTIONS
# ============================================================

# Get all Cricsheet history rows for currently unmapped IDs
cs_unmapped_history = cs_player_history[
    cs_player_history["source_player_id"]
    .astype(str)
    .isin(unmapped_cs_ids)
].copy()

# One summary per Cricsheet player ID
cs_id_audit = (
    cs_unmapped_history
    .groupby("source_player_id")
    .agg(
        name_count=("player_name", "nunique"),
        team_count=("team", "nunique"),
        player_names=(
            "player_name",
            lambda x: sorted(set(x.dropna()))
        ),
        teams=(
            "team",
            lambda x: sorted(set(x.dropna()))
        ),
        total_matches=("cricsheet_matches", "sum")
    )
    .reset_index()
)

# IDs with multiple names
multiple_name_ids = cs_id_audit[
    cs_id_audit["name_count"] > 1
].copy()

# IDs appearing under multiple teams
multiple_team_ids = cs_id_audit[
    cs_id_audit["team_count"] > 1
].copy()

# Check normalized names shared by different Cricsheet IDs
cs_unmapped_history["normalized_key"] = (
    cs_unmapped_history["player_name"]
    .apply(normalize_identity)
)

name_id_counts = (
    cs_unmapped_history.groupby("normalized_key")[
        "source_player_id"
    ].nunique()
)

duplicate_name_keys = name_id_counts[
    name_id_counts > 1
]

print("CRICSHEET-ONLY QUALITY AUDIT")
print("=" * 50)

print("Unmapped unique IDs:", cs_id_audit["source_player_id"].nunique())

print("\nIDs with multiple player names:", len(multiple_name_ids))
print("IDs associated with multiple teams:", len(multiple_team_ids))
print("Normalized names shared by multiple IDs:", len(duplicate_name_keys))

print("\nMULTIPLE-NAME ID DETAILS")
display(multiple_name_ids.to_string(index=False))

print("\nMULTIPLE-TEAM ID DETAILS")
display(multiple_team_ids.to_string(index=False))

print("\nDUPLICATE NORMALIZED NAME DETAILS")
if len(duplicate_name_keys):
    display(
        cs_unmapped_history[
            cs_unmapped_history["normalized_key"].isin(
                duplicate_name_keys.index
            )
        ][
            ["normalized_key", "player_name", "source_player_id", "team"]
        ]
        .drop_duplicates()
        .sort_values("normalized_key")
        .to_string(index=False)
    )
else:
    print("None")

print("\nTOTAL AUDIT ROWS:", len(cs_id_audit))


CRICSHEET-ONLY QUALITY AUDIT
Unmapped unique IDs: 1592

IDs with multiple player names: 3
IDs associated with multiple teams: 7
Normalized names shared by multiple IDs: 4

MULTIPLE-NAME ID DETAILS


'source_player_id  name_count  team_count                      player_names                      teams  total_matches\n        a9231c3f           2           1   [SR Taylor, Steven Ryan Taylor] [United States of America]             46\n        bafb9238           2           1      [Usman Khan, Usman Khan (2)]                 [Pakistan]              2\n        e237b28c           2           1 [Mohammad Yousuf, Yousuf Youhana]                 [Pakistan]            125'


MULTIPLE-TEAM ID DETAILS


'source_player_id  name_count  team_count       player_names                                    teams  total_matches\n        05c2ca46           1           2 [RE van der Merwe]              [Netherlands, South Africa]             44\n        0fa5042b           1           2         [L Ronchi]                 [Australia, New Zealand]             67\n        210b05cb           1           2       [Ehsan Adil]     [Pakistan, United States of America]             12\n        755a77c6           1           2      [GS Ballance]                      [England, Zimbabwe]             20\n        aa8d28ae           1           2          [D Wiese]                  [Namibia, South Africa]             14\n        dec8e038           1           2         [J Theron] [South Africa, United States of America]             18\n        ffb504b1           1           2      [XM Marshall]  [United States of America, West Indies]             26'


DUPLICATE NORMALIZED NAME DETAILS


'normalized_key     player_name source_player_id                     team\n       alikhan        Ali Khan         692c7070                   Canada\n       alikhan        Ali Khan         b0eaaac6 United States of America\n     haiderali      Haider Ali         c13e34f4                 Pakistan\n     haiderali      Haider Ali         dd572696     United Arab Emirates\njunaidsiddique Junaid Siddique         62e203d4               Bangladesh\njunaidsiddique Junaid Siddique         84dc72db     United Arab Emirates\n    shoaibkhan     Shoaib Khan         21afb939                     Oman\n    shoaibkhan     Shoaib Khan         fdee4a01     United Arab Emirates'


TOTAL AUDIT ROWS: 1592


In [191]:
# ============================================================
# STAGE CRICSHEET-ONLY IDENTITIES
# READ-ONLY AGAINST AUTHORITATIVE MAP
# NO INSERTIONS
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. Confirm the source data and expected count
# ------------------------------------------------------------

assert len(cs_id_audit) == 1592, (
    f"Expected 1592 unmapped IDs, found {len(cs_id_audit)}"
)

assert cs_id_audit["source_player_id"].nunique() == 1592

# ------------------------------------------------------------
# 2. Prepare one staged record per Cricsheet source ID
# ------------------------------------------------------------

staged_records = []

for source_id, group in cs_unmapped_history.groupby(
    "source_player_id",
    sort=True
):

    source_id = str(source_id)

    # Collect every name and team associated with this source ID
    names = sorted(
        group["player_name"].dropna().astype(str).unique().tolist()
    )

    teams = sorted(
        group["team"].dropna().astype(str).unique().tolist()
    )

    # Choose the name with the greatest recorded match count.
    # If tied, prefer the longer name, then alphabetical order.
    name_weights = (
        group.groupby("player_name")["cricsheet_matches"]
        .sum()
        .reset_index()
    )

    name_weights["player_name"] = (
        name_weights["player_name"].astype(str)
    )

    name_weights["name_length"] = (
        name_weights["player_name"].str.len()
    )

    name_weights = name_weights.sort_values(
        ["cricsheet_matches", "name_length", "player_name"],
        ascending=[False, False, True]
    )

    representative_name = name_weights.iloc[0]["player_name"]

    # Preserve all observed names and teams in the audit notes
    notes = (
        "Cricsheet source identity with no verified Crickrida "
        "counterpart. Retained as a Cricsheet-only identity; "
        "no cross-source name matching performed. "
        f"Observed names: {', '.join(names)}. "
        f"Observed teams: {', '.join(teams)}."
    )

    staged_records.append({
        "cricketiq_player_id": f"CS_{source_id}",
        "player_id": source_id,
        "cricsheet_name": representative_name,
        "crickrida_name": None,
        "normalized_name": None,
        "match_status": "cricsheet_only",
        "crickrida_player_name": None,
        "cricsheet_player_name": representative_name,
        "cricsheet_player_id": source_id,
        "mapping_method": "verified_source_coverage",
        "confidence": "high",
        "notes": notes
    })


# Build staged DataFrame using the authoritative map's column order
cricsheet_only_staged = pd.DataFrame(
    staged_records,
    columns=player_identity_map_updated.columns
)

# ------------------------------------------------------------
# 3. Check for ID collisions with the existing map
# ------------------------------------------------------------

existing_cricketiq_ids = set(
    player_identity_map_updated["cricketiq_player_id"]
    .dropna().astype(str)
)

existing_player_ids = set(
    player_identity_map_updated["player_id"]
    .dropna().astype(str)
)

existing_cricsheet_ids = set(
    player_identity_map_updated["cricsheet_player_id"]
    .dropna().astype(str)
)

staged_cricketiq_ids = set(
    cricsheet_only_staged["cricketiq_player_id"].astype(str)
)

staged_player_ids = set(
    cricsheet_only_staged["player_id"].astype(str)
)

staged_cricsheet_ids = set(
    cricsheet_only_staged["cricsheet_player_id"].astype(str)
)

cricketiq_id_conflicts = (
    staged_cricketiq_ids & existing_cricketiq_ids
)

player_id_conflicts = (
    staged_player_ids & existing_player_ids
)

cricsheet_id_conflicts = (
    staged_cricsheet_ids & existing_cricsheet_ids
)

# ------------------------------------------------------------
# 4. Validate staged records
# ------------------------------------------------------------

checks = {
    "Expected staged rows": len(cricsheet_only_staged) == 1592,

    "Unique CricketIQ IDs": (
        cricsheet_only_staged["cricketiq_player_id"].is_unique
    ),

    "Unique player IDs": (
        cricsheet_only_staged["player_id"].is_unique
    ),

    "Unique Cricsheet IDs": (
        cricsheet_only_staged["cricsheet_player_id"].is_unique
    ),

    "No CricketIQ ID conflicts": len(cricketiq_id_conflicts) == 0,

    "No player ID conflicts": len(player_id_conflicts) == 0,

    "No Cricsheet ID conflicts": len(cricsheet_id_conflicts) == 0,

    "All rows are cricsheet_only": (
        cricsheet_only_staged["match_status"]
        .eq("cricsheet_only").all()
    ),

    "Crickrida fields are empty": (
        cricsheet_only_staged[
            [
                "crickrida_name",
                "crickrida_player_name",
                "normalized_name"
            ]
        ].isna().all().all()
    ),

    "All source IDs are present": (
        cricsheet_only_staged[
            ["player_id", "cricsheet_player_id"]
        ].notna().all().all()
    )
}

print("CRICSHEET-ONLY STAGING VALIDATION")
print("=" * 55)

for check, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} | {check}")

print("\nSTAGING SUMMARY")
print("-" * 55)
print("Staged records:", len(cricsheet_only_staged))
print("Existing map rows:", len(player_identity_map_updated))
print(
    "Expected map rows after future insertion:",
    len(player_identity_map_updated) + len(cricsheet_only_staged)
)

print("\nID CONFLICTS")
print("CricketIQ ID conflicts:", len(cricketiq_id_conflicts))
print("player_id conflicts:", len(player_id_conflicts))
print("Cricsheet ID conflicts:", len(cricsheet_id_conflicts))

print("\nSAMPLE STAGED RECORDS")
display(
    cricsheet_only_staged[
        [
            "cricketiq_player_id",
            "cricsheet_name",
            "player_id",
            "match_status",
            "notes"
        ]
    ].head(10)
)

# ------------------------------------------------------------
# 5. Final staging decision
# ------------------------------------------------------------

staging_valid = all(checks.values())

print("\nFINAL STAGING STATUS:")
print("READY FOR REVIEW" if staging_valid else "STOP — INVESTIGATE FAILURES")

CRICSHEET-ONLY STAGING VALIDATION
PASS | Expected staged rows
PASS | Unique CricketIQ IDs
PASS | Unique player IDs
PASS | Unique Cricsheet IDs
PASS | No CricketIQ ID conflicts
PASS | No player ID conflicts
PASS | No Cricsheet ID conflicts
PASS | All rows are cricsheet_only
PASS | Crickrida fields are empty
PASS | All source IDs are present

STAGING SUMMARY
-------------------------------------------------------
Staged records: 1592
Existing map rows: 478
Expected map rows after future insertion: 2070

ID CONFLICTS
CricketIQ ID conflicts: 0
player_id conflicts: 0
Cricsheet ID conflicts: 0

SAMPLE STAGED RECORDS


,cricketiq_player_id,cricsheet_name,player_id,match_status,notes
0,CS_004c9e85,PJ Hughes,004c9e85,cricsheet_only,Cricsheet source identity with no verified Cri...
1,CS_008589a4,CM Wright,008589a4,cricsheet_only,Cricsheet source identity with no verified Cri...
2,CS_00d69fa4,SO Tikolo,00d69fa4,cricsheet_only,Cricsheet source identity with no verified Cri...
3,CS_00ea847a,MA Agarwal,00ea847a,cricsheet_only,Cricsheet source identity with no verified Cri...
4,CS_00ea9791,HP Tillakaratne,00ea9791,cricsheet_only,Cricsheet source identity with no verified Cri...
5,CS_012829ff,JW Hastings,012829ff,cricsheet_only,Cricsheet source identity with no verified Cri...
6,CS_0164b064,MG Neser,0164b064,cricsheet_only,Cricsheet source identity with no verified Cri...
7,CS_016e5101,M Shezi,016e5101,cricsheet_only,Cricsheet source identity with no verified Cri...
8,CS_017faef2,Z Groenewald,017faef2,cricsheet_only,Cricsheet source identity with no verified Cri...
9,CS_0184dc35,R Dravid,0184dc35,cricsheet_only,Cricsheet source identity with no verified Cri...



FINAL STAGING STATUS:
READY FOR REVIEW


In [192]:
# ============================================================
# SAVE IDENTITY MAP BACKUP + CRICSHEET-ONLY STAGING
# NO INSERTION INTO AUTHORITATIVE MAP
# ============================================================

from pathlib import Path
from datetime import datetime
import pandas as pd

# ------------------------------------------------------------
# 1. Final pre-save safety checks
# ------------------------------------------------------------

assert staging_valid, "Staging validation failed. Do not save for insertion."

assert len(player_identity_map_updated) == 478
assert len(cricsheet_only_staged) == 1592

assert list(player_identity_map_updated.columns) == list(
    cricsheet_only_staged.columns
)

# ------------------------------------------------------------
# 2. Create a timestamped backup folder
# ------------------------------------------------------------

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

backup_dir = (
    Path(r"C:\CricIQ")
    / "backups"
    / "player_identity_reconciliation"
)
backup_dir.mkdir(parents=True, exist_ok=True)

map_backup_path = backup_dir / f"identity_map_backup_{timestamp}.csv"
staged_path = backup_dir / f"cricsheet_only_staged_{timestamp}.csv"
combined_path = backup_dir / f"identity_map_combined_review_{timestamp}.csv"

# ------------------------------------------------------------
# 3. Create a separate combined review DataFrame
# ------------------------------------------------------------

combined_identity_review = pd.concat(
    [
        player_identity_map_updated.copy(),
        cricsheet_only_staged.copy()
    ],
    ignore_index=True
)

# ------------------------------------------------------------
# 4. Validate the combined review BEFORE saving
# ------------------------------------------------------------

assert len(combined_identity_review) == 2070

assert combined_identity_review["cricketiq_player_id"].is_unique

assert combined_identity_review["player_id"].dropna().is_unique

assert (
    combined_identity_review["cricsheet_player_id"]
    .dropna()
    .is_unique
)

# Confirm existing authoritative map remains unchanged
assert len(player_identity_map_updated) == 478

# ------------------------------------------------------------
# 5. Save all three CSV files
# ------------------------------------------------------------

player_identity_map_updated.to_csv(
    map_backup_path,
    index=False
)

cricsheet_only_staged.to_csv(
    staged_path,
    index=False
)

combined_identity_review.to_csv(
    combined_path,
    index=False
)

# ------------------------------------------------------------
# 6. Verify saved files by reading them back
# ------------------------------------------------------------

saved_map = pd.read_csv(map_backup_path)
saved_staged = pd.read_csv(staged_path)
saved_combined = pd.read_csv(combined_path)

assert len(saved_map) == 478
assert len(saved_staged) == 1592
assert len(saved_combined) == 2070

assert saved_map["cricketiq_player_id"].is_unique
assert saved_staged["cricketiq_player_id"].is_unique
assert saved_combined["cricketiq_player_id"].is_unique

print("BACKUP AND EXPORT VALIDATION")
print("=" * 55)

print("PASS | Existing map backup:", len(saved_map))
print("PASS | Cricsheet-only staged file:", len(saved_staged))
print("PASS | Combined review file:", len(saved_combined))

print("\nSAVED FILES")
print("-" * 55)

print("Map backup:       ", map_backup_path.resolve())
print("Staged identities:", staged_path.resolve())
print("Combined review:  ", combined_path.resolve())

print("\nAUTHORITATIVE MAP")
print("-" * 55)
print("Current map rows:", len(player_identity_map_updated))
print("Current map unchanged: YES")

print("\nSTATUS: BACKUPS SAVED AND VERIFIED")
print("NO IDENTITIES INSERTED")

BACKUP AND EXPORT VALIDATION
PASS | Existing map backup: 478
PASS | Cricsheet-only staged file: 1592
PASS | Combined review file: 2070

SAVED FILES
-------------------------------------------------------
Map backup:        C:\CricIQ\notebooks\identity_reconciliation_backups\identity_map_backup_20261116_231047.csv
Staged identities: C:\CricIQ\notebooks\identity_reconciliation_backups\cricsheet_only_staged_20261116_231047.csv
Combined review:   C:\CricIQ\notebooks\identity_reconciliation_backups\identity_map_combined_review_20261116_231047.csv

AUTHORITATIVE MAP
-------------------------------------------------------
Current map rows: 478
Current map unchanged: YES

STATUS: BACKUPS SAVED AND VERIFIED
NO IDENTITIES INSERTED


In [194]:
# ============================================================
# CORRECTED CANDIDATE MAP VALIDATION
# READ-ONLY — NO AUTHORITATIVE MAP OVERWRITE
# ============================================================

# 1. Count existing Cricsheet-only records in the original map
original_cs_only_count = (
    player_identity_map_before_cricsheet_insert["match_status"]
    .eq("cricsheet_only")
    .sum()
)

# 2. Count new staged records
new_cs_only_count = len(cricsheet_only_staged)

# 3. Calculate the correct expected total
expected_cs_only_count = (
    original_cs_only_count + new_cs_only_count
)

# 4. Count Cricsheet-only records in candidate map
candidate_status_counts = (
    player_identity_map_cricsheet_complete["match_status"]
    .value_counts(dropna=False)
)

actual_cs_only_count = candidate_status_counts.get(
    "cricsheet_only", 0
)

# 5. Verify the original map is fully preserved
pd.testing.assert_frame_equal(
    player_identity_map_updated,
    player_identity_map_before_cricsheet_insert
)

# 6. Verify staged IDs are all present in candidate map
candidate_cs_ids = set(
    player_identity_map_cricsheet_complete[
        "cricsheet_player_id"
    ].dropna().astype(str)
)

staged_cs_ids = set(
    cricsheet_only_staged[
        "cricsheet_player_id"
    ].dropna().astype(str)
)

all_staged_ids_present = staged_cs_ids.issubset(candidate_cs_ids)

# 7. Verify original map identities are preserved
original_cricketiq_ids = set(
    player_identity_map_before_cricsheet_insert[
        "cricketiq_player_id"
    ].dropna().astype(str)
)

candidate_cricketiq_ids = set(
    player_identity_map_cricsheet_complete[
        "cricketiq_player_id"
    ].dropna().astype(str)
)

all_original_ids_preserved = (
    original_cricketiq_ids.issubset(candidate_cricketiq_ids)
)

# 8. Final checks
checks = {
    "Candidate has 2070 rows":
        len(player_identity_map_cricsheet_complete) == 2070,

    "Correct Cricsheet-only count":
        actual_cs_only_count == expected_cs_only_count,

    "All staged Cricsheet IDs present":
        all_staged_ids_present,

    "All original CricketIQ IDs preserved":
        all_original_ids_preserved,

    "Unique CricketIQ IDs":
        player_identity_map_cricsheet_complete[
            "cricketiq_player_id"
        ].is_unique,

    "Unique Cricsheet IDs":
        player_identity_map_cricsheet_complete[
            "cricsheet_player_id"
        ].dropna().is_unique,

    "Unique player IDs":
        player_identity_map_cricsheet_complete[
            "player_id"
        ].dropna().is_unique,

    "Original authoritative map unchanged":
        len(player_identity_map_updated) == 478
}

print("CORRECTED CANDIDATE MAP VALIDATION")
print("=" * 55)

print("Original Cricsheet-only records:", original_cs_only_count)
print("New staged Cricsheet-only records:", new_cs_only_count)
print("Expected total Cricsheet-only records:", expected_cs_only_count)
print("Actual candidate Cricsheet-only records:", actual_cs_only_count)

print("\nVALIDATION CHECKS")
for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} | {name}")

print("\nCANDIDATE STATUS COUNTS")
display(candidate_status_counts.to_frame("records"))

print("\nFINAL RESULT:")
print(
    "READY FOR GLOBAL RECONCILIATION"
    if all(checks.values())
    else "STOP — INVESTIGATE FAILED CHECKS"
)

CORRECTED CANDIDATE MAP VALIDATION
Original Cricsheet-only records: 1
New staged Cricsheet-only records: 1592
Expected total Cricsheet-only records: 1593
Actual candidate Cricsheet-only records: 1593

VALIDATION CHECKS
PASS | Candidate has 2070 rows
PASS | Correct Cricsheet-only count
PASS | All staged Cricsheet IDs present
PASS | All original CricketIQ IDs preserved
PASS | Unique CricketIQ IDs
PASS | Unique Cricsheet IDs
PASS | Unique player IDs
PASS | Original authoritative map unchanged

CANDIDATE STATUS COUNTS


,records
match_status,
cricsheet_only,1593
verified_alias,279
exact_name_match,98
crickrida_only,62
verified_manual_identity,23
verified_full_name_and_career,10
verified_ireland_identity,2
verified_match_and_career,1
verified_cross_country_identity,1



FINAL RESULT:
READY FOR GLOBAL RECONCILIATION


In [195]:
# ============================================================
# GLOBAL IDENTITY SOURCE RECONCILIATION
# READ-ONLY — NO INSERTIONS OR REGISTRY REBUILD
# ============================================================

# ------------------------------------------------------------
# 1. Extract unique source identities
# ------------------------------------------------------------

source_cs_ids = set(
    cs_player_history["source_player_id"]
    .dropna().astype(str).unique()
)

source_cr_ids = set(
    cr_player_history["normalized_name"]
    .dropna().astype(str).unique()
)

mapped_cs_ids = set(
    player_identity_map_cricsheet_complete[
        "cricsheet_player_id"
    ].dropna().astype(str)
)

mapped_cr_keys = set(
    player_identity_map_cricsheet_complete[
        "normalized_name"
    ].dropna().astype(str)
)

# ------------------------------------------------------------
# 2. Compare source and mapped identities
# ------------------------------------------------------------

unmapped_cs_ids_final = source_cs_ids - mapped_cs_ids
extra_cs_ids_final = mapped_cs_ids - source_cs_ids

unmapped_cr_keys_final = source_cr_ids - mapped_cr_keys
extra_cr_keys_final = mapped_cr_keys - source_cr_ids

# ------------------------------------------------------------
# 3. Check source-ID multiplicity in the candidate map
# ------------------------------------------------------------

cs_mapping_counts = (
    player_identity_map_cricsheet_complete[
        "cricsheet_player_id"
    ]
    .dropna()
    .astype(str)
    .value_counts()
)

cr_mapping_counts = (
    player_identity_map_cricsheet_complete[
        "normalized_name"
    ]
    .dropna()
    .astype(str)
    .value_counts()
)

duplicate_cs_mappings = cs_mapping_counts[
    cs_mapping_counts > 1
]

duplicate_cr_mappings = cr_mapping_counts[
    cr_mapping_counts > 1
]

# ------------------------------------------------------------
# 4. Source coverage summary
# ------------------------------------------------------------

print("GLOBAL IDENTITY SOURCE RECONCILIATION")
print("=" * 60)

print("\nCRICSHEET")
print("-" * 60)
print("Unique source IDs:", len(source_cs_ids))
print("Mapped source IDs:", len(source_cs_ids & mapped_cs_ids))
print("Unmapped source IDs:", len(unmapped_cs_ids_final))
print("Extra mapped IDs:", len(extra_cs_ids_final))
print("Duplicate source mappings:", len(duplicate_cs_mappings))

print("\nCRICKRIDA")
print("-" * 60)
print("Unique source identities:", len(source_cr_ids))
print("Mapped normalized identities:", len(source_cr_ids & mapped_cr_keys))
print("Unmapped source identities:", len(unmapped_cr_keys_final))
print("Extra normalized keys:", len(extra_cr_keys_final))
print("Duplicate normalized-key mappings:", len(duplicate_cr_mappings))

# ------------------------------------------------------------
# 5. Show any remaining exceptions
# ------------------------------------------------------------

print("\nREMAINING CRICSHEET IDs")
display(sorted(unmapped_cs_ids_final)[:30])

print("\nREMAINING CRICKRIDA NORMALIZED KEYS")
display(sorted(unmapped_cr_keys_final)[:30])

print("\nEXTRA CRICSHEET IDs")
display(sorted(extra_cs_ids_final)[:30])

print("\nEXTRA CRICKRIDA NORMALIZED KEYS")
display(sorted(extra_cr_keys_final)[:30])

# ------------------------------------------------------------
# 6. Final result
# ------------------------------------------------------------

global_reconciliation_passed = (
    len(unmapped_cs_ids_final) == 0
    and len(extra_cs_ids_final) == 0
    and len(unmapped_cr_keys_final) == 0
    and len(extra_cr_keys_final) == 0
    and len(duplicate_cs_mappings) == 0
    and len(duplicate_cr_mappings) == 0
)

print("\nGLOBAL RECONCILIATION STATUS:")
print(
    "PASS — ALL SOURCE IDENTITIES REPRESENTED"
    if global_reconciliation_passed
    else "REVIEW REQUIRED — SOURCE DIFFERENCES FOUND"
)

GLOBAL IDENTITY SOURCE RECONCILIATION

CRICSHEET
------------------------------------------------------------
Unique source IDs: 2008
Mapped source IDs: 1807
Unmapped source IDs: 201
Extra mapped IDs: 0
Duplicate source mappings: 0

CRICKRIDA
------------------------------------------------------------
Unique source identities: 454
Mapped normalized identities: 454
Unmapped source identities: 0
Extra normalized keys: 24
Duplicate normalized-key mappings: 0

REMAINING CRICSHEET IDs


['024a173d',
 '02789d88',
 '02e83ac5',
 '0404d43c',
 '05d74535',
 '086f5984',
 '0a4736eb',
 '0a8fce53',
 '0b0cc297',
 '0dc00542',
 '0fdabe05',
 '1274d7ab',
 '16dfcc19',
 '1777c020',
 '1786a21b',
 '193ef196',
 '19d0a172',
 '1a8eebbe',
 '1f7adc8a',
 '1fe81cc9',
 '2182013e',
 '2254ab79',
 '23eeb873',
 '243431b5',
 '244048f6',
 '24bb1c2f',
 '26d041c4',
 '26f12506',
 '2911de16',
 '29e253dd']


REMAINING CRICKRIDA NORMALIZED KEYS


[]


EXTRA CRICSHEET IDs


[]


EXTRA CRICKRIDA NORMALIZED KEYS


['albertvandermerwe',
 'andrebotha',
 'benwhite',
 'davelangfordsmith',
 'eddierichardson',
 'fionnhand',
 'graememccarter',
 'jamesshannon',
 'jeremybray',
 'kennycarroll',
 'kylemccallan',
 'matthewhumphreys',
 'maxsorensen',
 'murraycommins',
 'neilrock',
 'peterconnell',
 'petergillespie',
 'reganwest',
 'rogerwhelan',
 'rsharma',
 'shanegetkate',
 'stephendoheny',
 'thinusfourie',
 'thomasmayes']


GLOBAL RECONCILIATION STATUS:
REVIEW REQUIRED — SOURCE DIFFERENCES FOUND


In [196]:
# ============================================================
# CRICSHEET ID STORAGE DIAGNOSTIC
# READ-ONLY — NO DATA MODIFICATIONS
# ============================================================

# 1. All unique Cricsheet source IDs
source_cs_ids = set(
    cs_player_history["source_player_id"]
    .dropna().astype(str).str.strip()
)

# 2. IDs stored in each candidate-map column
candidate_player_ids = set(
    player_identity_map_cricsheet_complete["player_id"]
    .dropna().astype(str).str.strip()
)

candidate_cricsheet_ids = set(
    player_identity_map_cricsheet_complete["cricsheet_player_id"]
    .dropna().astype(str).str.strip()
)

# 3. Combined coverage using either column
combined_candidate_ids = (
    candidate_player_ids | candidate_cricsheet_ids
)

# 4. Calculate coverage and missing IDs
covered_by_player_id = source_cs_ids & candidate_player_ids
covered_by_cricsheet_id = source_cs_ids & candidate_cricsheet_ids
covered_by_either_column = source_cs_ids & combined_candidate_ids

missing_from_both = source_cs_ids - combined_candidate_ids

# 5. Check staged records against current source history
staged_ids = set(
    cricsheet_only_staged["cricsheet_player_id"]
    .dropna().astype(str).str.strip()
)

staged_not_in_source = staged_ids - source_cs_ids
source_not_in_staged = source_cs_ids - staged_ids

# 6. Inspect candidate rows corresponding to missing IDs
missing_rows = player_identity_map_cricsheet_complete[
    player_identity_map_cricsheet_complete[
        "player_id"
    ].astype(str).isin(missing_from_both)
    |
    player_identity_map_cricsheet_complete[
        "cricsheet_player_id"
    ].astype(str).isin(missing_from_both)
].copy()

print("CRICSHEET ID STORAGE DIAGNOSTIC")
print("=" * 60)

print("\nSOURCE COVERAGE")
print("-" * 60)
print("Unique source IDs:", len(source_cs_ids))
print("Found in player_id:", len(covered_by_player_id))
print("Found in cricsheet_player_id:", len(covered_by_cricsheet_id))
print("Found in either column:", len(covered_by_either_column))
print("Missing from both columns:", len(missing_from_both))

print("\nSTAGED ID CHECK")
print("-" * 60)
print("Staged IDs:", len(staged_ids))
print("Staged IDs not in current source:", len(staged_not_in_source))
print("Source IDs not in staged dataset:", len(source_not_in_staged))

print("\nMISSING SOURCE ID SAMPLE")
display(sorted(missing_from_both)[:40])

print("\nSTAGED IDs NOT IN SOURCE SAMPLE")
display(sorted(staged_not_in_source)[:40])

print("\nSOURCE IDs NOT IN STAGED SAMPLE")
display(sorted(source_not_in_staged)[:40])

print("\nCANDIDATE MAP COLUMN NULL COUNTS")
print("-" * 60)
print(
    player_identity_map_cricsheet_complete[
        ["player_id", "cricsheet_player_id"]
    ].isna().sum()
)

print("\nCANDIDATE MAP COLUMN SAMPLE")
display(
    player_identity_map_cricsheet_complete[
        [
            "cricketiq_player_id",
            "cricsheet_name",
            "player_id",
            "cricsheet_player_id",
            "match_status"
        ]
    ].head(20)
)

print("\nSTATUS: DIAGNOSTIC COMPLETE — NO CHANGES MADE")

CRICSHEET ID STORAGE DIAGNOSTIC

SOURCE COVERAGE
------------------------------------------------------------
Unique source IDs: 2008
Found in player_id: 2007
Found in cricsheet_player_id: 1807
Found in either column: 2008
Missing from both columns: 0

STAGED ID CHECK
------------------------------------------------------------
Staged IDs: 1592
Staged IDs not in current source: 0
Source IDs not in staged dataset: 416

MISSING SOURCE ID SAMPLE


[]


STAGED IDs NOT IN SOURCE SAMPLE


[]


SOURCE IDs NOT IN STAGED SAMPLE


['024a173d',
 '02789d88',
 '02e83ac5',
 '03806cf8',
 '0404d43c',
 '05d74535',
 '05de1a5a',
 '06e649a0',
 '086f5984',
 '0994d0ae',
 '0a2184d9',
 '0a4736eb',
 '0a476045',
 '0a8fce53',
 '0a9d8b61',
 '0b0cc297',
 '0b60eb09',
 '0dc00542',
 '0ebfb1ad',
 '0f12f9df',
 '0f721006',
 '0fdabe05',
 '1274d7ab',
 '12b610c2',
 '13c35c9e',
 '14f96089',
 '16dfcc19',
 '1777c020',
 '1786a21b',
 '193ef196',
 '19d0a172',
 '1a448be0',
 '1a8eebbe',
 '1abb78f8',
 '1ac2a995',
 '1f7adc8a',
 '1fc6ef83',
 '1fe81cc9',
 '212aeedd',
 '2182013e']


CANDIDATE MAP COLUMN NULL COUNTS
------------------------------------------------------------
player_id               63
cricsheet_player_id    263
dtype: int64

CANDIDATE MAP COLUMN SAMPLE


,cricketiq_player_id,cricsheet_name,player_id,cricsheet_player_id,match_status
0,CRIQ_8a75e999,Babar Azam,8a75e999,NaN,exact_name_match
1,CRIQ_2911de16,Hasan Ali,2911de16,NaN,exact_name_match
2,CRIQ_9cb8d7a6,Imad Wasim,9cb8d7a6,NaN,exact_name_match
3,CRIQ_e174dadd,Mohammad Amir,e174dadd,NaN,exact_name_match
4,CRIQ_9ab63e7b,Mohammad Hafeez,9ab63e7b,NaN,exact_name_match
5,CRIQ_2f26ac1a,Mohammad Rizwan,2f26ac1a,NaN,exact_name_match
6,CRIQ_193ef196,Sharjeel Khan,193ef196,NaN,exact_name_match
7,CRIQ_fd2bf2a0,Umar Akmal,fd2bf2a0,NaN,exact_name_match
8,CRIQ_b3118300,Wahab Riaz,b3118300,NaN,exact_name_match
9,CRIQ_50250426,Asad Shafiq,50250426,NaN,exact_name_match



STATUS: DIAGNOSTIC COMPLETE — NO CHANGES MADE


In [197]:
# ============================================================
# DETAILED CRICSHEET ID COLUMN CONSISTENCY AUDIT
# READ-ONLY — NO MODIFICATIONS
# ============================================================

candidate = player_identity_map_cricsheet_complete.copy()

# Normalize both columns for comparison
player_ids = candidate["player_id"].astype("string").str.strip()
cricsheet_ids = (
    candidate["cricsheet_player_id"].astype("string").str.strip()
)

source_ids = set(
    cs_player_history["source_player_id"]
    .dropna().astype(str).str.strip()
)

# ------------------------------------------------------------
# 1. Column population
# ------------------------------------------------------------

has_player_id = player_ids.notna() & player_ids.ne("")
has_cricsheet_id = cricsheet_ids.notna() & cricsheet_ids.ne("")

both_present = has_player_id & has_cricsheet_id
player_id_only = has_player_id & ~has_cricsheet_id
cricsheet_id_only = ~has_player_id & has_cricsheet_id
neither_present = ~has_player_id & ~has_cricsheet_id

# ------------------------------------------------------------
# 2. Compare rows where both columns are populated
# ------------------------------------------------------------

matching_both = both_present & player_ids.eq(cricsheet_ids)
mismatching_both = both_present & player_ids.ne(cricsheet_ids)

# ------------------------------------------------------------
# 3. Compare each column with the source IDs
# ------------------------------------------------------------

player_id_values = set(player_ids[has_player_id].astype(str))
cricsheet_id_values = set(cricsheet_ids[has_cricsheet_id].astype(str))

player_ids_not_in_source = player_id_values - source_ids
cricsheet_ids_not_in_source = cricsheet_id_values - source_ids

player_only_source_ids = (
    set(player_ids[player_id_only].astype(str)) & source_ids
)

cricsheet_only_source_ids = (
    set(cricsheet_ids[cricsheet_id_only].astype(str)) & source_ids
)

# ------------------------------------------------------------
# 4. Display summary
# ------------------------------------------------------------

print("CRICSHEET ID COLUMN CONSISTENCY AUDIT")
print("=" * 60)

print("\nCOLUMN POPULATION")
print("-" * 60)
print("Both columns populated:", both_present.sum())
print("Only player_id populated:", player_id_only.sum())
print("Only cricsheet_player_id populated:", cricsheet_id_only.sum())
print("Neither column populated:", neither_present.sum())

print("\nID AGREEMENT")
print("-" * 60)
print("Both columns agree:", matching_both.sum())
print("Both columns disagree:", mismatching_both.sum())

print("\nSOURCE ID VALIDATION")
print("-" * 60)
print("player_id values not in Cricsheet source:", len(player_ids_not_in_source))
print("cricsheet_player_id values not in source:", len(cricsheet_ids_not_in_source))
print("Source IDs stored only in player_id:", len(player_only_source_ids))
print("Source IDs stored only in cricsheet_player_id:", len(cricsheet_only_source_ids))

# ------------------------------------------------------------
# 5. Inspect mismatched rows
# ------------------------------------------------------------

print("\nROWS WHERE BOTH COLUMNS DISAGREE")
display(
    candidate.loc[
        mismatching_both,
        [
            "cricketiq_player_id",
            "cricsheet_name",
            "player_id",
            "cricsheet_player_id",
            "match_status",
            "notes"
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# 6. Inspect source IDs stored in only one column
# ------------------------------------------------------------

print("\nSOURCE IDs STORED ONLY IN player_id — SAMPLE")
display(
    candidate.loc[
        player_id_only & player_ids.isin(player_only_source_ids),
        [
            "cricketiq_player_id",
            "cricsheet_name",
            "player_id",
            "cricsheet_player_id",
            "match_status"
        ]
    ].head(25).to_string(index=False)
)

print("\nSOURCE IDs STORED ONLY IN cricsheet_player_id")
display(
    candidate.loc[
        cricsheet_id_only & cricsheet_ids.isin(cricsheet_only_source_ids),
        [
            "cricketiq_player_id",
            "cricsheet_name",
            "player_id",
            "cricsheet_player_id",
            "match_status"
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# 7. Show values outside Cricsheet source history
# ------------------------------------------------------------

print("\nplayer_id VALUES NOT IN SOURCE")
display(sorted(player_ids_not_in_source)[:30])

print("\ncricsheet_player_id VALUES NOT IN SOURCE")
display(sorted(cricsheet_ids_not_in_source)[:30])

print("\nSTATUS: AUDIT COMPLETE — NO CHANGES MADE")

CRICSHEET ID COLUMN CONSISTENCY AUDIT

COLUMN POPULATION
------------------------------------------------------------
Both columns populated: 1806
Only player_id populated: 201
Only cricsheet_player_id populated: 1
Neither column populated: 62

ID AGREEMENT
------------------------------------------------------------
Both columns agree: 1806
Both columns disagree: 0

SOURCE ID VALIDATION
------------------------------------------------------------
player_id values not in Cricsheet source: 0
cricsheet_player_id values not in source: 0
Source IDs stored only in player_id: 201
Source IDs stored only in cricsheet_player_id: 1

ROWS WHERE BOTH COLUMNS DISAGREE


'Empty DataFrame\nColumns: [cricketiq_player_id, cricsheet_name, player_id, cricsheet_player_id, match_status, notes]\nIndex: []'


SOURCE IDs STORED ONLY IN player_id — SAMPLE


'cricketiq_player_id    cricsheet_name player_id cricsheet_player_id     match_status\n      CRIQ_8a75e999        Babar Azam  8a75e999                 NaN exact_name_match\n      CRIQ_2911de16         Hasan Ali  2911de16                 NaN exact_name_match\n      CRIQ_9cb8d7a6        Imad Wasim  9cb8d7a6                 NaN exact_name_match\n      CRIQ_e174dadd     Mohammad Amir  e174dadd                 NaN exact_name_match\n      CRIQ_9ab63e7b   Mohammad Hafeez  9ab63e7b                 NaN exact_name_match\n      CRIQ_2f26ac1a   Mohammad Rizwan  2f26ac1a                 NaN exact_name_match\n      CRIQ_193ef196     Sharjeel Khan  193ef196                 NaN exact_name_match\n      CRIQ_fd2bf2a0        Umar Akmal  fd2bf2a0                 NaN exact_name_match\n      CRIQ_b3118300        Wahab Riaz  b3118300                 NaN exact_name_match\n      CRIQ_50250426       Asad Shafiq  50250426                 NaN exact_name_match\n      CRIQ_086f5984       Junaid Khan  086f5984      


SOURCE IDs STORED ONLY IN cricsheet_player_id


'cricketiq_player_id cricsheet_name player_id cricsheet_player_id   match_status\n        CS_5d9a1a73       R Sharma      None            5d9a1a73 cricsheet_only'


player_id VALUES NOT IN SOURCE


[]


cricsheet_player_id VALUES NOT IN SOURCE


[]


STATUS: AUDIT COMPLETE — NO CHANGES MADE


In [198]:
# ============================================================
# FINAL GLOBAL SOURCE RECONCILIATION
# CORRECTED FOR BOTH CRICSHEET ID COLUMNS
# READ-ONLY
# ============================================================

candidate = player_identity_map_cricsheet_complete.copy()

# ------------------------------------------------------------
# 1. Source identities
# ------------------------------------------------------------

source_cs_ids = set(
    cs_player_history["source_player_id"]
    .dropna().astype(str).str.strip()
)

source_cr_keys = set(
    cr_player_history["normalized_name"]
    .dropna().astype(str).str.strip()
)

# ------------------------------------------------------------
# 2. Cricsheet IDs from BOTH map columns
# ------------------------------------------------------------

mapped_player_ids = set(
    candidate["player_id"]
    .dropna().astype(str).str.strip()
)

mapped_cricsheet_ids = set(
    candidate["cricsheet_player_id"]
    .dropna().astype(str).str.strip()
)

mapped_cs_ids = mapped_player_ids | mapped_cricsheet_ids

# ------------------------------------------------------------
# 3. Crickrida normalized identities
# ------------------------------------------------------------

mapped_cr_keys = set(
    candidate["normalized_name"]
    .dropna().astype(str).str.strip()
)

# ------------------------------------------------------------
# 4. Reconciliation differences
# ------------------------------------------------------------

unmapped_cs = source_cs_ids - mapped_cs_ids
extra_cs = mapped_cs_ids - source_cs_ids

unmapped_cr = source_cr_keys - mapped_cr_keys
extra_cr = mapped_cr_keys - source_cr_keys

# ------------------------------------------------------------
# 5. Duplicate checks
# ------------------------------------------------------------

duplicate_cricketiq_ids = (
    candidate["cricketiq_player_id"].duplicated().sum()
)

duplicate_player_ids = (
    candidate["player_id"].dropna().duplicated().sum()
)

duplicate_cricsheet_ids = (
    candidate["cricsheet_player_id"].dropna().duplicated().sum()
)

duplicate_cr_keys = (
    candidate["normalized_name"].dropna().duplicated().sum()
)

# ------------------------------------------------------------
# 6. Verify original and staged identities
# ------------------------------------------------------------

original_ids = set(
    player_identity_map_before_cricsheet_insert[
        "cricketiq_player_id"
    ].dropna().astype(str)
)

staged_ids = set(
    cricsheet_only_staged[
        "cricketiq_player_id"
    ].dropna().astype(str)
)

candidate_ids = set(
    candidate["cricketiq_player_id"].dropna().astype(str)
)

original_preserved = original_ids.issubset(candidate_ids)
staged_preserved = staged_ids.issubset(candidate_ids)

# ------------------------------------------------------------
# 7. Final checks
# ------------------------------------------------------------

final_checks = {
    "All 2008 Cricsheet IDs represented":
        len(source_cs_ids) == 2008 and len(unmapped_cs) == 0,

    "No extra Cricsheet IDs":
        len(extra_cs) == 0,

    "All 454 Crickrida identities represented":
        len(source_cr_keys) == 454 and len(unmapped_cr) == 0,

    "No duplicate CricketIQ IDs":
        duplicate_cricketiq_ids == 0,

    "No duplicate player IDs":
        duplicate_player_ids == 0,

    "No duplicate Cricsheet IDs":
        duplicate_cricsheet_ids == 0,

    "No duplicate Crickrida normalized keys":
        duplicate_cr_keys == 0,

    "All original identities preserved":
        original_preserved,

    "All staged identities preserved":
        staged_preserved,

    "Candidate map contains 2070 rows":
        len(candidate) == 2070
}

print("FINAL GLOBAL IDENTITY RECONCILIATION")
print("=" * 60)

print("\nSOURCE COVERAGE")
print("Cricsheet:", len(source_cs_ids), "| Missing:", len(unmapped_cs))
print("Crickrida:", len(source_cr_keys), "| Missing:", len(unmapped_cr))

print("\nEXTRA IDENTITIES")
print("Extra Cricsheet IDs:", len(extra_cs))
print("Extra Crickrida normalized keys:", len(extra_cr))

print("\nDUPLICATE CHECKS")
print("CricketIQ IDs:", duplicate_cricketiq_ids)
print("player_id:", duplicate_player_ids)
print("Cricsheet IDs:", duplicate_cricsheet_ids)
print("Crickrida normalized keys:", duplicate_cr_keys)

print("\nVALIDATION")
for check, passed in final_checks.items():
    print(f"{'PASS' if passed else 'FAIL'} | {check}")

print("\nEXTRA CRICKRIDA KEYS")
display(sorted(extra_cr))

final_global_reconciliation_passed = all(final_checks.values())

print("\nFINAL RESULT:")
print(
    "PASS — GLOBAL SOURCE RECONCILIATION COMPLETE"
    if final_global_reconciliation_passed
    else "STOP — REVIEW FAILED CHECKS"
)

FINAL GLOBAL IDENTITY RECONCILIATION

SOURCE COVERAGE
Cricsheet: 2008 | Missing: 0
Crickrida: 454 | Missing: 0

EXTRA IDENTITIES
Extra Cricsheet IDs: 0
Extra Crickrida normalized keys: 24

DUPLICATE CHECKS
CricketIQ IDs: 0
player_id: 0
Cricsheet IDs: 0
Crickrida normalized keys: 0

VALIDATION
PASS | All 2008 Cricsheet IDs represented
PASS | No extra Cricsheet IDs
PASS | All 454 Crickrida identities represented
PASS | No duplicate CricketIQ IDs
PASS | No duplicate player IDs
PASS | No duplicate Cricsheet IDs
PASS | No duplicate Crickrida normalized keys
PASS | All original identities preserved
PASS | All staged identities preserved
PASS | Candidate map contains 2070 rows

EXTRA CRICKRIDA KEYS


['albertvandermerwe',
 'andrebotha',
 'benwhite',
 'davelangfordsmith',
 'eddierichardson',
 'fionnhand',
 'graememccarter',
 'jamesshannon',
 'jeremybray',
 'kennycarroll',
 'kylemccallan',
 'matthewhumphreys',
 'maxsorensen',
 'murraycommins',
 'neilrock',
 'peterconnell',
 'petergillespie',
 'reganwest',
 'rogerwhelan',
 'rsharma',
 'shanegetkate',
 'stephendoheny',
 'thinusfourie',
 'thomasmayes']


FINAL RESULT:
PASS — GLOBAL SOURCE RECONCILIATION COMPLETE


In [199]:
# ============================================================
# PROMOTE FINAL PLAYER IDENTITY MAP
# PRESERVE ORIGINAL MAP AND SAVE FINAL VERSION
# ============================================================

from pathlib import Path
from datetime import datetime
import pandas as pd

# ------------------------------------------------------------
# 1. Safety checks before promotion
# ------------------------------------------------------------

assert final_global_reconciliation_passed, (
    "Global reconciliation has not passed. Promotion stopped."
)

assert len(player_identity_map_cricsheet_complete) == 2070

assert len(player_identity_map_before_cricsheet_insert) == 478

assert player_identity_map_cricsheet_complete[
    "cricketiq_player_id"
].is_unique

assert (
    player_identity_map_cricsheet_complete[
        "player_id"
    ].dropna().is_unique
)

assert (
    player_identity_map_cricsheet_complete[
        "cricsheet_player_id"
    ].dropna().is_unique
)

# ------------------------------------------------------------
# 2. Prepare final map separately
# ------------------------------------------------------------

player_identity_map_promoted = (
    player_identity_map_cricsheet_complete.copy(deep=True)
)

# ------------------------------------------------------------
# 3. Save the final map to a new file
# ------------------------------------------------------------

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

backup_dir = (
    Path(r"C:\CricIQ")
    / "backups"
    / "player_identity_reconciliation"
)
backup_dir.mkdir(parents=True, exist_ok=True)

final_map_path = (
    backup_dir / f"player_identity_map_final_{timestamp}.csv"
)

player_identity_map_promoted.to_csv(
    final_map_path,
    index=False
)

# ------------------------------------------------------------
# 4. Read back and validate the saved file
# ------------------------------------------------------------

saved_final_map = pd.read_csv(final_map_path)

assert len(saved_final_map) == 2070

assert saved_final_map[
    "cricketiq_player_id"
].is_unique

assert saved_final_map["player_id"].dropna().is_unique

assert saved_final_map[
    "cricsheet_player_id"
].dropna().is_unique

assert saved_final_map[
    "normalized_name"
].dropna().is_unique

# ------------------------------------------------------------
# 5. Promote to authoritative variable
# ------------------------------------------------------------

player_identity_map_updated = (
    player_identity_map_promoted.copy(deep=True)
)

# ------------------------------------------------------------
# 6. Final verification
# ------------------------------------------------------------

assert len(player_identity_map_updated) == 2070

pd.testing.assert_frame_equal(
    player_identity_map_updated,
    player_identity_map_promoted
)

assert len(player_identity_map_before_cricsheet_insert) == 478

print("PLAYER IDENTITY MAP PROMOTION")
print("=" * 60)

print("PASS | Final map saved and read back")
print("PASS | 2070 records promoted")
print("PASS | CricketIQ IDs unique")
print("PASS | player_id values unique")
print("PASS | Cricsheet IDs unique")
print("PASS | Crickrida normalized keys unique")
print("PASS | Original 478-row map preserved separately")

print("\nFINAL MAP")
print("Rows:", len(player_identity_map_updated))
print("Columns:", len(player_identity_map_updated.columns))

print("\nSAVED FILE:")
print(final_map_path.resolve())

print("\nSTATUS: FINAL IDENTITY MAP PROMOTED")

PLAYER IDENTITY MAP PROMOTION
PASS | Final map saved and read back
PASS | 2070 records promoted
PASS | CricketIQ IDs unique
PASS | player_id values unique
PASS | Cricsheet IDs unique
PASS | Crickrida normalized keys unique
PASS | Original 478-row map preserved separately

FINAL MAP
Rows: 2070
Columns: 12

SAVED FILE:
C:\CricIQ\notebooks\identity_reconciliation_backups\player_identity_map_final_20261116_231942.csv

STATUS: FINAL IDENTITY MAP PROMOTED


## 7. Performance Table ID Migration and Canonical Names

Audit player references in the performance tables, translate legacy IDs to current CricketIQ IDs using staged copies, and validate canonical player names. Preserve the original tables and verify that non-identity performance values remain unchanged.

In [200]:
# ============================================================
# PERFORMANCE TABLE IDENTITY INTEGRITY AUDIT
# READ-ONLY — NO MODIFICATIONS
# ============================================================

performance_tables = {
    "batting": player_batting_updated,
    "bowling": player_bowling_updated,
    "dismissal": player_dismissal_updated,
    "dismissal_bowler": dismissal_bowler_updated
}

# Final identity map lookup
identity_lookup = (
    player_identity_map_updated[
        ["cricketiq_player_id", "cricsheet_name"]
    ]
    .drop_duplicates("cricketiq_player_id")
    .set_index("cricketiq_player_id")["cricsheet_name"]
)

print("PERFORMANCE TABLE IDENTITY AUDIT")
print("=" * 65)

audit_results = {}

for table_name, df in performance_tables.items():

    ids = df["cricketiq_player_id"]

    valid_ids = ids.notna() & ids.isin(identity_lookup.index)

    missing_ids = ids.notna() & ~ids.isin(identity_lookup.index)

    null_ids = ids.isna()

    # Compare canonical names with identity-map names
    mapped_names = ids.map(identity_lookup)

    canonical_names = df["canonical_player_name"]

    name_comparison = (
        canonical_names.fillna("").astype(str).str.strip()
        ==
        mapped_names.fillna("").astype(str).str.strip()
    )

    name_mismatches = valid_ids & ~name_comparison

    # Unique IDs and names
    unique_ids = ids.dropna().nunique()

    multiple_names = (
        df.dropna(subset=["cricketiq_player_id"])
        .groupby("cricketiq_player_id")["canonical_player_name"]
        .nunique()
    )

    conflicting_names = multiple_names[multiple_names > 1]

    audit_results[table_name] = {
        "rows": len(df),
        "unique_player_ids": unique_ids,
        "null_ids": int(null_ids.sum()),
        "unmapped_ids": int(missing_ids.sum()),
        "name_mismatches": int(name_mismatches.sum()),
        "players_with_multiple_names": len(conflicting_names)
    }

    print(f"\n{table_name.upper()}")
    print("-" * 50)
    print("Total rows:", len(df))
    print("Unique player IDs:", unique_ids)
    print("Null CricketIQ IDs:", null_ids.sum())
    print("IDs missing from final map:", missing_ids.sum())
    print("Canonical-name mismatches:", name_mismatches.sum())
    print("Players with multiple canonical names:", len(conflicting_names))

print("\nAUDIT COMPLETE — NO DATA MODIFIED")

PERFORMANCE TABLE IDENTITY AUDIT

BATTING
--------------------------------------------------
Total rows: 45432
Unique player IDs: 1934
Null CricketIQ IDs: 0
IDs missing from final map: 14938
Canonical-name mismatches: 1182
Players with multiple canonical names: 0

BOWLING
--------------------------------------------------
Total rows: 31186
Unique player IDs: 1502
Null CricketIQ IDs: 0
IDs missing from final map: 9151
Canonical-name mismatches: 773
Players with multiple canonical names: 0

DISMISSAL
--------------------------------------------------
Total rows: 37711
Unique player IDs: 1841
Null CricketIQ IDs: 0
IDs missing from final map: 12504
Canonical-name mismatches: 978
Players with multiple canonical names: 0

DISMISSAL_BOWLER
--------------------------------------------------
Total rows: 34975
Unique player IDs: 1281
Null CricketIQ IDs: 0
IDs missing from final map: 10085
Canonical-name mismatches: 851
Players with multiple canonical names: 0

AUDIT COMPLETE — NO DATA MODIFIED


In [201]:
# ============================================================
# PERFORMANCE IDENTITY MISMATCH DIAGNOSTICS
# READ-ONLY — NO MODIFICATIONS
# ============================================================

import pandas as pd

performance_tables = {
    "batting": player_batting_updated,
    "bowling": player_bowling_updated,
    "dismissal": player_dismissal_updated,
    "dismissal_bowler": dismissal_bowler_updated
}

identity_map = player_identity_map_updated.copy()

# Normalize ID representations for comparison
identity_map["_audit_id"] = (
    identity_map["cricketiq_player_id"]
    .astype("string").str.strip()
)

identity_ids = set(identity_map["_audit_id"].dropna())

print("PERFORMANCE IDENTITY DIAGNOSTICS")
print("=" * 70)

for table_name, df in performance_tables.items():

    temp = df.copy()

    temp["_audit_id"] = (
        temp["cricketiq_player_id"]
        .astype("string").str.strip()
    )

    # --------------------------------------------------------
    # 1. Identify missing IDs and their frequencies
    # --------------------------------------------------------

    missing_rows = temp[
        temp["_audit_id"].notna()
        & ~temp["_audit_id"].isin(identity_ids)
    ]

    missing_id_counts = (
        missing_rows["_audit_id"]
        .value_counts()
        .rename_axis("cricketiq_player_id")
        .reset_index(name="row_count")
    )

    # --------------------------------------------------------
    # 2. Identify valid IDs with name mismatches
    # --------------------------------------------------------

    valid_rows = temp[
        temp["_audit_id"].isin(identity_ids)
    ].copy()

    lookup = identity_map.set_index("_audit_id")

    valid_rows["_map_cricsheet_name"] = (
        valid_rows["_audit_id"].map(
            lookup["cricsheet_name"]
        )
    )

    valid_rows["_map_crickrida_name"] = (
        valid_rows["_audit_id"].map(
            lookup["crickrida_name"]
        )
    )

    valid_rows["_map_status"] = (
        valid_rows["_audit_id"].map(
            lookup["match_status"]
        )
    )

    valid_rows["_canonical_clean"] = (
        valid_rows["canonical_player_name"]
        .astype("string").str.strip()
    )

    valid_rows["_map_name_clean"] = (
        valid_rows["_map_cricsheet_name"]
        .astype("string").str.strip()
    )

    name_mismatch_rows = valid_rows[
        valid_rows["_canonical_clean"]
        .ne(valid_rows["_map_name_clean"])
    ]

    # --------------------------------------------------------
    # 3. Summary
    # --------------------------------------------------------

    print(f"\n{'=' * 70}")
    print(table_name.upper())
    print("=" * 70)

    print("Total rows:", len(temp))
    print("Unique performance IDs:", temp["_audit_id"].nunique())
    print("Unique missing IDs:", missing_id_counts.shape[0])
    print("Rows with missing IDs:", len(missing_rows))
    print("Unique valid IDs:", valid_rows["_audit_id"].nunique())
    print("Rows with name mismatches:", len(name_mismatch_rows))

    # --------------------------------------------------------
    # 4. Show missing ID examples
    # --------------------------------------------------------

    print("\nTOP 25 MISSING IDs BY ROW COUNT")

    display(missing_id_counts.head(25))

    # --------------------------------------------------------
    # 5. Show actual performance records for missing IDs
    # --------------------------------------------------------

    print("\nSAMPLE MISSING PERFORMANCE RECORDS")

    display(
        missing_rows[
            [
                "cricketiq_player_id",
                "canonical_player_name"
            ]
        ].drop_duplicates().head(25)
    )

    # --------------------------------------------------------
    # 6. Show name mismatches with map context
    # --------------------------------------------------------

    print("\nSAMPLE VALID-ID NAME MISMATCHES")

    display(
        name_mismatch_rows[
            [
                "cricketiq_player_id",
                "canonical_player_name",
                "_map_cricsheet_name",
                "_map_crickrida_name",
                "_map_status"
            ]
        ].drop_duplicates().head(25)
    )

# ------------------------------------------------------------
# 7. Inspect ID formats in the final map
# ------------------------------------------------------------

print("\nFINAL IDENTITY MAP — ID FORMAT SAMPLE")
print("=" * 70)

display(
    identity_map[
        [
            "cricketiq_player_id",
            "player_id",
            "cricsheet_player_id",
            "cricsheet_name",
            "crickrida_name",
            "match_status"
        ]
    ].head(30)
)

print("\nIDENTITY MAP ID PREFIX COUNTS")

display(
    identity_map["cricketiq_player_id"]
    .astype("string")
    .str.extract(r"^([A-Za-z]+)")[0]
    .value_counts(dropna=False)
    .to_frame("records")
)

print("\nSTATUS: DIAGNOSTICS COMPLETE — NO DATA MODIFIED")

PERFORMANCE IDENTITY DIAGNOSTICS

BATTING
Total rows: 45432
Unique performance IDs: 1934
Unique missing IDs: 290
Rows with missing IDs: 14938
Unique valid IDs: 1644
Rows with name mismatches: 1182

TOP 25 MISSING IDs BY ROW COUNT


,cricketiq_player_id,row_count
0,CS_ba607b88,302
1,CS_6b71e6cf,280
2,CS_740742ef,276
3,CS_d18f9182,260
4,CS_5bdcdb72,259
5,CS_d2a6c0e6,215
6,CS_b61a3e1a,209
7,CS_7ed9fd56,205
8,CS_db584dad,196
9,CS_f842c2cf,196



SAMPLE MISSING PERFORMANCE RECORDS


,cricketiq_player_id,canonical_player_name
2,CS_dcce6f09,DA Warner
3,CS_b681e71e,GJ Maxwell
4,CS_808f425a,JP Faulkner
5,CS_3fb19989,MA Starc
6,CS_3d8feaf8,MR Marsh
7,CS_afa7e784,MS Wade
8,CS_ded9240e,PJ Cummins
9,CS_30a45b23,SPD Smith
10,CS_12b610c2,TM Head
25,CS_03806cf8,JR Hazlewood



SAMPLE VALID-ID NAME MISMATCHES


,cricketiq_player_id,canonical_player_name,_map_cricsheet_name,_map_crickrida_name,_map_status
101,CRIQ_94d7f855,Colin de Grandhomme,C de Grandhomme,Colin de Grandhomme,verified_alias
109,CRIQ_e824e6ee,Tom Latham,TWM Latham,Tom Latham,verified_alias
332,CRIQ_ab6b1c45,Andile Phehlukwayo,AL Phehlukwayo,Andile Phehlukwayo,verified_alias
416,CRIQ_a343262c,Joe Root,JE Root,Joe Root,verified_alias
463,CRIQ_2254ab79,Sarfaraz Ahmed,Sarfraz Ahmed,Sarfaraz Ahmed,verified_alias
469,CRIQ_c58b0108,Mehidy Hasan Miraz,Mehedi Hasan Miraz,Mehidy Hasan Miraz,verified_alias
489,CRIQ_8d92a2c3,Mark Wood,MA Wood,Mark Wood,verified_alias
706,CRIQ_e342e5fb,Carlos Brathwaite,CR Brathwaite,Carlos Brathwaite,verified_alias
757,CRIQ_6ef13460,Andy Balbirnie,A Balbirnie,Andy Balbirnie,verified_alias
758,CRIQ_a48fc79d,Ed Joyce,EC Joyce,Ed Joyce,verified_alias


... (19 outputs left out)

,cricketiq_player_id,player_id,cricsheet_player_id,cricsheet_name,crickrida_name,match_status
0,CRIQ_8a75e999,8a75e999,NaN,Babar Azam,Babar Azam,exact_name_match
1,CRIQ_2911de16,2911de16,NaN,Hasan Ali,Hasan Ali,exact_name_match
2,CRIQ_9cb8d7a6,9cb8d7a6,NaN,Imad Wasim,Imad Wasim,exact_name_match
3,CRIQ_e174dadd,e174dadd,NaN,Mohammad Amir,Mohammad Amir,exact_name_match
4,CRIQ_9ab63e7b,9ab63e7b,NaN,Mohammad Hafeez,Mohammad Hafeez,exact_name_match
5,CRIQ_2f26ac1a,2f26ac1a,NaN,Mohammad Rizwan,Mohammad Rizwan,exact_name_match
6,CRIQ_193ef196,193ef196,NaN,Sharjeel Khan,Sharjeel Khan,exact_name_match
7,CRIQ_fd2bf2a0,fd2bf2a0,NaN,Umar Akmal,Umar Akmal,exact_name_match
8,CRIQ_b3118300,b3118300,NaN,Wahab Riaz,Wahab Riaz,exact_name_match
9,CRIQ_50250426,50250426,NaN,Asad Shafiq,Asad Shafiq,exact_name_match



IDENTITY MAP ID PREFIX COUNTS


,records
0,
CS,1593
CRIQ,415
CR,62



STATUS: DIAGNOSTICS COMPLETE — NO DATA MODIFIED


In [202]:

# ============================================================
# OLD-TO-NEW CRICKETIQ ID MIGRATION AUDIT
# READ-ONLY — NO PERFORMANCE TABLES MODIFIED
# ============================================================

import pandas as pd

performance_tables = {
    "batting": player_batting_updated,
    "bowling": player_bowling_updated,
    "dismissal": player_dismissal_updated,
    "dismissal_bowler": dismissal_bowler_updated
}

# ------------------------------------------------------------
# 1. Build a source-ID-to-new-CricketIQ-ID lookup
# ------------------------------------------------------------

map_source_parts = []

for column in ["player_id", "cricsheet_player_id"]:

    part = player_identity_map_updated[
        ["cricketiq_player_id", column]
    ].copy()

    part.columns = ["new_cricketiq_id", "source_player_id"]

    part = part.dropna(subset=["source_player_id"])

    part["source_player_id"] = (
        part["source_player_id"].astype(str).str.strip()
    )

    map_source_parts.append(part)

source_identity_lookup = pd.concat(
    map_source_parts,
    ignore_index=True
).drop_duplicates()

# Check whether one source ID points to multiple new identities
source_id_target_counts = (
    source_identity_lookup
    .groupby("source_player_id")["new_cricketiq_id"]
    .nunique()
)

ambiguous_source_ids = source_id_target_counts[
    source_id_target_counts > 1
]

# Keep only unambiguous source IDs for the audit lookup
safe_source_lookup = (
    source_identity_lookup
    .drop_duplicates("source_player_id")
    .set_index("source_player_id")["new_cricketiq_id"]
)

# ------------------------------------------------------------
# 2. Collect all distinct performance IDs
# ------------------------------------------------------------

performance_id_records = []

for table_name, df in performance_tables.items():

    counts = (
        df["cricketiq_player_id"]
        .dropna()
        .astype(str)
        .str.strip()
        .value_counts()
    )

    for old_id, row_count in counts.items():

        performance_id_records.append({
            "table": table_name,
            "old_cricketiq_id": old_id,
            "row_count": row_count
        })

performance_id_inventory = pd.DataFrame(
    performance_id_records
)

# ------------------------------------------------------------
# 3. Extract source IDs from old CS_ identifiers
# ------------------------------------------------------------

performance_id_inventory["id_prefix_valid"] = (
    performance_id_inventory["old_cricketiq_id"]
    .str.startswith("CS_")
)

performance_id_inventory["source_player_id"] = (
    performance_id_inventory["old_cricketiq_id"]
    .where(
        performance_id_inventory["id_prefix_valid"],
        pd.NA
    )
    .str.replace(r"^CS_", "", regex=True)
)

# ------------------------------------------------------------
# 4. Resolve old IDs to new CricketIQ IDs
# ------------------------------------------------------------

performance_id_inventory["new_cricketiq_id"] = (
    performance_id_inventory["source_player_id"]
    .map(safe_source_lookup)
)

performance_id_inventory["mapping_status"] = "unresolved"

performance_id_inventory.loc[
    performance_id_inventory["new_cricketiq_id"].notna(),
    "mapping_status"
] = "verified_source_id_mapping"

performance_id_inventory.loc[
    ~performance_id_inventory["id_prefix_valid"],
    "mapping_status"
] = "non_CS_old_id"

# ------------------------------------------------------------
# 5. Summary by performance table
# ------------------------------------------------------------

print("OLD-TO-NEW ID MIGRATION AUDIT")
print("=" * 65)

for table_name in performance_tables:

    subset = performance_id_inventory[
        performance_id_inventory["table"] == table_name
    ]

    print(f"\n{table_name.upper()}")
    print("-" * 45)

    print("Unique performance IDs:", len(subset))

    print(
        "IDs resolved:",
        subset["new_cricketiq_id"].notna().sum()
    )

    print(
        "IDs unresolved:",
        subset["new_cricketiq_id"].isna().sum()
    )

    print(
        "Performance rows covered:",
        subset.loc[
            subset["new_cricketiq_id"].notna(),
            "row_count"
        ].sum()
    )

# ------------------------------------------------------------
# 6. Global safety checks
# ------------------------------------------------------------

print("\nGLOBAL MAPPING SUMMARY")
print("=" * 65)

print(
    "Ambiguous source IDs:",
    len(ambiguous_source_ids)
)

print(
    "Old IDs with CS_ prefix:",
    performance_id_inventory["id_prefix_valid"].sum()
)

print(
    "Old IDs successfully resolved:",
    performance_id_inventory["new_cricketiq_id"].notna().sum()
)

print(
    "Unresolved old IDs:",
    performance_id_inventory["new_cricketiq_id"].isna().sum()
)

print(
    "Performance rows covered:",
    performance_id_inventory.loc[
        performance_id_inventory["new_cricketiq_id"].notna(),
        "row_count"
    ].sum()
)

# ------------------------------------------------------------
# 7. Inspect unresolved IDs
# ------------------------------------------------------------

print("\nUNRESOLVED PERFORMANCE IDs")
display(
    performance_id_inventory[
        performance_id_inventory["new_cricketiq_id"].isna()
    ].sort_values(["table", "row_count"], ascending=[True, False])
)

print("\nAMBIGUOUS SOURCE IDs")
display(ambiguous_source_ids.to_frame("target_identity_count"))

# ------------------------------------------------------------
# 8. Preview verified translations
# ------------------------------------------------------------

print("\nSAMPLE VERIFIED ID TRANSLATIONS")

display(
    performance_id_inventory[
        performance_id_inventory["new_cricketiq_id"].notna()
    ][
        [
            "table",
            "old_cricketiq_id",
            "source_player_id",
            "new_cricketiq_id",
            "row_count"
        ]
    ].drop_duplicates("old_cricketiq_id").head(30)
)

print("\nSTATUS: AUDIT ONLY — NO DATA MODIFIED")


OLD-TO-NEW ID MIGRATION AUDIT

BATTING
---------------------------------------------
Unique performance IDs: 1934
IDs resolved: 1816
IDs unresolved: 118
Performance rows covered: 39263

BOWLING
---------------------------------------------
Unique performance IDs: 1502
IDs resolved: 1402
IDs unresolved: 100
Performance rows covered: 26885

DISMISSAL
---------------------------------------------
Unique performance IDs: 1841
IDs resolved: 1724
IDs unresolved: 117
Performance rows covered: 32647

DISMISSAL_BOWLER
---------------------------------------------
Unique performance IDs: 1281
IDs resolved: 1191
IDs unresolved: 90
Performance rows covered: 29813

GLOBAL MAPPING SUMMARY
Ambiguous source IDs: 0
Old IDs with CS_ prefix: 6133
Old IDs successfully resolved: 6133
Unresolved old IDs: 425
Performance rows covered: 128608

UNRESOLVED PERFORMANCE IDs


,table,old_cricketiq_id,row_count,id_prefix_valid,source_player_id,new_cricketiq_id,mapping_status
2,batting,CRIQ_4a8a2e3b,276,False,<NA>,NaN,non_CS_old_id
6,batting,CRIQ_a94e08ea,227,False,<NA>,NaN,non_CS_old_id
7,batting,CRIQ_3b041a12,215,False,<NA>,NaN,non_CS_old_id
12,batting,CRIQ_64c34cd0,206,False,<NA>,NaN,non_CS_old_id
14,batting,CRIQ_9ab63e7b,202,False,<NA>,NaN,non_CS_old_id
...,...,...,...,...,...,...,...
6505,dismissal_bowler,CRIQ_8eb611d0,1,False,<NA>,NaN,non_CS_old_id
6508,dismissal_bowler,CRIQ_aad0c365,1,False,<NA>,NaN,non_CS_old_id
6514,dismissal_bowler,CRIQ_80b2fb19,1,False,<NA>,NaN,non_CS_old_id
6523,dismissal_bowler,CRIQ_d07c1b2f,1,False,<NA>,NaN,non_CS_old_id



AMBIGUOUS SOURCE IDs


,target_identity_count
source_player_id,



SAMPLE VERIFIED ID TRANSLATIONS


,table,old_cricketiq_id,source_player_id,new_cricketiq_id,row_count
0,batting,CS_ba607b88,ba607b88,CRIQ_ba607b88,302
1,batting,CS_6b71e6cf,6b71e6cf,CRIQ_6b71e6cf,280
3,batting,CS_740742ef,740742ef,CRIQ_740742ef,276
4,batting,CS_d18f9182,d18f9182,CRIQ_d18f9182,260
5,batting,CS_5bdcdb72,5bdcdb72,CRIQ_5bdcdb72,259
8,batting,CS_d2a6c0e6,d2a6c0e6,CRIQ_d2a6c0e6,215
9,batting,CS_b61a3e1a,b61a3e1a,CRIQ_b61a3e1a,209
10,batting,CS_c4487b84,c4487b84,CS_c4487b84,209
11,batting,CS_1c914163,1c914163,CS_1c914163,206
13,batting,CS_7ed9fd56,7ed9fd56,CRIQ_7ed9fd56,205



STATUS: AUDIT ONLY — NO DATA MODIFIED


In [203]:

# ============================================================
# CORRECTED OLD-TO-NEW ID MIGRATION AUDIT
# DIRECT ID + SOURCE ID RESOLUTION
# READ-ONLY — NO MODIFICATIONS
# ============================================================

import pandas as pd

performance_tables = {
    "batting": player_batting_updated,
    "bowling": player_bowling_updated,
    "dismissal": player_dismissal_updated,
    "dismissal_bowler": dismissal_bowler_updated
}

# ------------------------------------------------------------
# 1. Build the final CricketIQ ID set
# ------------------------------------------------------------

current_ids = set(
    player_identity_map_updated["cricketiq_player_id"]
    .dropna().astype(str).str.strip()
)

# ------------------------------------------------------------
# 2. Build source ID -> CricketIQ ID lookup
# ------------------------------------------------------------

source_parts = []

for column in ["player_id", "cricsheet_player_id"]:

    part = player_identity_map_updated[
        ["cricketiq_player_id", column]
    ].copy()

    part.columns = ["target_id", "source_id"]

    part = part.dropna(subset=["source_id"])

    part["source_id"] = (
        part["source_id"].astype(str).str.strip()
    )

    source_parts.append(part)

source_lookup_df = pd.concat(
    source_parts,
    ignore_index=True
).drop_duplicates()

# Verify each source ID has exactly one target
source_target_counts = (
    source_lookup_df.groupby("source_id")["target_id"]
    .nunique()
)

ambiguous_source_ids = source_target_counts[
    source_target_counts > 1
]

assert len(ambiguous_source_ids) == 0, (
    "Ambiguous source IDs found. Stop migration."
)

source_to_cricketiq = (
    source_lookup_df
    .drop_duplicates("source_id")
    .set_index("source_id")["target_id"]
)

# ------------------------------------------------------------
# 3. Create a fresh performance ID inventory
# ------------------------------------------------------------

inventory_records = []

for table_name, df in performance_tables.items():

    counts = (
        df["cricketiq_player_id"]
        .dropna()
        .astype(str)
        .str.strip()
        .value_counts()
    )

    for old_id, count in counts.items():

        inventory_records.append({
            "table": table_name,
            "old_id": old_id,
            "row_count": int(count)
        })

migration_inventory = pd.DataFrame(inventory_records)

# ------------------------------------------------------------
# 4. Resolution path 1: ID already exists
# ------------------------------------------------------------

migration_inventory["new_id"] = pd.NA
migration_inventory["resolution"] = "unresolved"

direct_mask = migration_inventory["old_id"].isin(current_ids)

migration_inventory.loc[direct_mask, "new_id"] = (
    migration_inventory.loc[direct_mask, "old_id"]
)

migration_inventory.loc[direct_mask, "resolution"] = (
    "already_current"
)

# ------------------------------------------------------------
# 5. Resolution path 2: old CS_ ID -> source ID -> new ID
# ------------------------------------------------------------

needs_translation = (
    ~direct_mask
    & migration_inventory["old_id"].str.startswith("CS_")
)

migration_inventory["source_id"] = pd.NA

migration_inventory.loc[needs_translation, "source_id"] = (
    migration_inventory.loc[needs_translation, "old_id"]
    .str.replace(r"^CS_", "", regex=True)
)

translated_ids = (
    migration_inventory.loc[needs_translation, "source_id"]
    .map(source_to_cricketiq)
)

migration_inventory.loc[needs_translation, "new_id"] = (
    translated_ids
)

translated_mask = (
    needs_translation
    & migration_inventory["new_id"].notna()
)

migration_inventory.loc[translated_mask, "resolution"] = (
    "translated_by_source_id"
)

# ------------------------------------------------------------
# 6. Validate every proposed target against final map
# ------------------------------------------------------------

resolved_mask = migration_inventory["new_id"].notna()

targets_in_map = migration_inventory.loc[
    resolved_mask, "new_id"
].isin(current_ids)

assert targets_in_map.all(), (
    "A proposed target ID does not exist in the final map."
)

# ------------------------------------------------------------
# 7. Summary by performance table
# ------------------------------------------------------------

print("CORRECTED ID MIGRATION AUDIT")
print("=" * 65)

for table_name in performance_tables:

    subset = migration_inventory[
        migration_inventory["table"] == table_name
    ]

    print(f"\n{table_name.upper()}")
    print("-" * 45)

    print("Unique performance IDs:", len(subset))

    print(
        "Already current:",
        (subset["resolution"] == "already_current").sum()
    )

    print(
        "Translated by source ID:",
        (subset["resolution"] == "translated_by_source_id").sum()
    )

    print(
        "Unresolved:",
        (subset["resolution"] == "unresolved").sum()
    )

    print(
        "Rows safely resolvable:",
        subset.loc[resolved_mask.loc[subset.index], "row_count"].sum()
    )

# ------------------------------------------------------------
# 8. Global summary
# ------------------------------------------------------------

print("\nGLOBAL SUMMARY")
print("=" * 65)

print("Total table-specific unique IDs:", len(migration_inventory))

print(
    "Already current:",
    (migration_inventory["resolution"] == "already_current").sum()
)

print(
    "Translated by source ID:",
    (migration_inventory["resolution"] == "translated_by_source_id").sum()
)

print(
    "Unresolved:",
    (migration_inventory["resolution"] == "unresolved").sum()
)

print(
    "Total performance rows:",
    migration_inventory["row_count"].sum()
)

print(
    "Rows safely resolvable:",
    migration_inventory.loc[
        resolved_mask, "row_count"
    ].sum()
)

print(
    "Rows unresolved:",
    migration_inventory.loc[
        ~resolved_mask, "row_count"
    ].sum()
)

# ------------------------------------------------------------
# 9. Show genuine unresolved IDs only
# ------------------------------------------------------------

print("\nGENUINELY UNRESOLVED IDs")

display(
    migration_inventory[
        migration_inventory["resolution"] == "unresolved"
    ].sort_values(
        ["table", "row_count"],
        ascending=[True, False]
    )
)

# ------------------------------------------------------------
# 10. Preview both types of resolution
# ------------------------------------------------------------

print("\nSAMPLE DIRECTLY VALID IDs")

display(
    migration_inventory[
        migration_inventory["resolution"] == "already_current"
    ].head(15)
)

print("\nSAMPLE SOURCE-ID TRANSLATIONS")

display(
    migration_inventory[
        migration_inventory["resolution"] == "translated_by_source_id"
    ].head(25)
)

print("\nSTATUS: CORRECTED AUDIT COMPLETE — NO DATA MODIFIED")


CORRECTED ID MIGRATION AUDIT

BATTING
---------------------------------------------
Unique performance IDs: 1934
Already current: 1644
Translated by source ID: 290
Unresolved: 0
Rows safely resolvable: 45432

BOWLING
---------------------------------------------
Unique performance IDs: 1502
Already current: 1275
Translated by source ID: 227
Unresolved: 0
Rows safely resolvable: 31186

DISMISSAL
---------------------------------------------
Unique performance IDs: 1841
Already current: 1555
Translated by source ID: 286
Unresolved: 0
Rows safely resolvable: 37711

DISMISSAL_BOWLER
---------------------------------------------
Unique performance IDs: 1281
Already current: 1073
Translated by source ID: 208
Unresolved: 0
Rows safely resolvable: 34975

GLOBAL SUMMARY
Total table-specific unique IDs: 6558
Already current: 5547
Translated by source ID: 1011
Unresolved: 0
Total performance rows: 149304
Rows safely resolvable: 149304
Rows unresolved: 0

GENUINELY UNRESOLVED IDs


,table,old_id,row_count,new_id,resolution,source_id



SAMPLE DIRECTLY VALID IDs


,table,old_id,row_count,new_id,resolution,source_id
2,batting,CRIQ_4a8a2e3b,276,CRIQ_4a8a2e3b,already_current,<NA>
6,batting,CRIQ_a94e08ea,227,CRIQ_a94e08ea,already_current,<NA>
7,batting,CRIQ_3b041a12,215,CRIQ_3b041a12,already_current,<NA>
10,batting,CS_c4487b84,209,CS_c4487b84,already_current,<NA>
11,batting,CS_1c914163,206,CS_1c914163,already_current,<NA>
12,batting,CRIQ_64c34cd0,206,CRIQ_64c34cd0,already_current,<NA>
14,batting,CRIQ_9ab63e7b,202,CRIQ_9ab63e7b,already_current,<NA>
15,batting,CRIQ_7dc35884,200,CRIQ_7dc35884,already_current,<NA>
19,batting,CS_1dc12ab9,190,CS_1dc12ab9,already_current,<NA>
20,batting,CRIQ_bf1d7d3e,189,CRIQ_bf1d7d3e,already_current,<NA>



SAMPLE SOURCE-ID TRANSLATIONS


,table,old_id,row_count,new_id,resolution,source_id
0,batting,CS_ba607b88,302,CRIQ_ba607b88,translated_by_source_id,ba607b88
1,batting,CS_6b71e6cf,280,CRIQ_6b71e6cf,translated_by_source_id,6b71e6cf
3,batting,CS_740742ef,276,CRIQ_740742ef,translated_by_source_id,740742ef
4,batting,CS_d18f9182,260,CRIQ_d18f9182,translated_by_source_id,d18f9182
5,batting,CS_5bdcdb72,259,CRIQ_5bdcdb72,translated_by_source_id,5bdcdb72
8,batting,CS_d2a6c0e6,215,CRIQ_d2a6c0e6,translated_by_source_id,d2a6c0e6
9,batting,CS_b61a3e1a,209,CRIQ_b61a3e1a,translated_by_source_id,b61a3e1a
13,batting,CS_7ed9fd56,205,CRIQ_7ed9fd56,translated_by_source_id,7ed9fd56
16,batting,CS_f842c2cf,196,CRIQ_f842c2cf,translated_by_source_id,f842c2cf
17,batting,CS_db584dad,196,CRIQ_db584dad,translated_by_source_id,db584dad



STATUS: CORRECTED AUDIT COMPLETE — NO DATA MODIFIED


In [204]:

# ============================================================
# PERFORMANCE TABLE ID MIGRATION
# CREATE NEW COPIES — ORIGINALS PRESERVED
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. Safety checks
# ------------------------------------------------------------

assert len(migration_inventory) == 6558

assert migration_inventory["new_id"].notna().all()

assert migration_inventory["resolution"].isin(
    ["already_current", "translated_by_source_id"]
).all()

assert len(ambiguous_source_ids) == 0

# Preserve original performance tables
player_batting_before_id_migration = player_batting_updated.copy(deep=True)
player_bowling_before_id_migration = player_bowling_updated.copy(deep=True)
player_dismissal_before_id_migration = player_dismissal_updated.copy(deep=True)
dismissal_bowler_before_id_migration = dismissal_bowler_updated.copy(deep=True)

# ------------------------------------------------------------
# 2. Build table-specific old-to-new ID mappings
# ------------------------------------------------------------

table_id_mappings = {}

for table_name in performance_tables:

    subset = migration_inventory[
        migration_inventory["table"] == table_name
    ]

    mapping = dict(
        zip(subset["old_id"], subset["new_id"])
    )

    assert len(mapping) == subset["old_id"].nunique()

    table_id_mappings[table_name] = mapping

# ------------------------------------------------------------
# 3. Create migrated copies
# ------------------------------------------------------------

player_batting_migrated = player_batting_before_id_migration.copy(deep=True)
player_bowling_migrated = player_bowling_before_id_migration.copy(deep=True)
player_dismissal_migrated = player_dismissal_before_id_migration.copy(deep=True)
dismissal_bowler_migrated = dismissal_bowler_before_id_migration.copy(deep=True)

migrated_tables = {
    "batting": player_batting_migrated,
    "bowling": player_bowling_migrated,
    "dismissal": player_dismissal_migrated,
    "dismissal_bowler": dismissal_bowler_migrated
}

# ------------------------------------------------------------
# 4. Replace IDs using exact verified mappings
# ------------------------------------------------------------

for table_name, df in migrated_tables.items():

    old_ids = df["cricketiq_player_id"].astype(str).str.strip()

    new_ids = old_ids.map(table_id_mappings[table_name])

    assert new_ids.notna().all(), (
        f"Unresolved ID encountered in {table_name}"
    )

    df["cricketiq_player_id"] = new_ids

# ------------------------------------------------------------
# 5. Validate migrated IDs against final identity map
# ------------------------------------------------------------

final_identity_ids = set(
    player_identity_map_updated["cricketiq_player_id"]
    .dropna().astype(str)
)

print("PERFORMANCE ID MIGRATION VALIDATION")
print("=" * 65)

validation_results = {}

original_tables = {
    "batting": player_batting_before_id_migration,
    "bowling": player_bowling_before_id_migration,
    "dismissal": player_dismissal_before_id_migration,
    "dismissal_bowler": dismissal_bowler_before_id_migration
}

for table_name, migrated_df in migrated_tables.items():

    original_df = original_tables[table_name]

    missing_ids = set(
        migrated_df["cricketiq_player_id"].dropna().astype(str)
    ) - final_identity_ids

    # Check row count
    row_count_ok = len(migrated_df) == len(original_df)

    # Check column preservation
    columns_ok = list(migrated_df.columns) == list(original_df.columns)

    # Check all IDs exist in final map
    ids_ok = len(missing_ids) == 0

    # Check canonical names remain unchanged
    names_ok = (
        migrated_df["canonical_player_name"]
        .reset_index(drop=True)
        .equals(
            original_df["canonical_player_name"]
            .reset_index(drop=True)
        )
    )

    # Check all non-ID columns remain unchanged
    non_id_columns = [
        col for col in original_df.columns
        if col != "cricketiq_player_id"
    ]

    values_ok = migrated_df[non_id_columns].equals(
        original_df[non_id_columns]
    )

    validation_results[table_name] = {
        "rows_preserved": row_count_ok,
        "columns_preserved": columns_ok,
        "all_ids_in_final_map": ids_ok,
        "canonical_names_unchanged": names_ok,
        "all_other_values_unchanged": values_ok
    }

    print(f"\n{table_name.upper()}")
    print("-" * 45)
    print("Rows:", len(migrated_df))
    print("Unique migrated IDs:", migrated_df["cricketiq_player_id"].nunique())
    print("IDs missing from final map:", len(missing_ids))
    print("Rows preserved:", row_count_ok)
    print("Columns preserved:", columns_ok)
    print("Canonical names unchanged:", names_ok)
    print("Other values unchanged:", values_ok)

# ------------------------------------------------------------
# 6. Final result
# ------------------------------------------------------------

migration_validation_passed = all(
    all(result.values())
    for result in validation_results.values()
)

print("\nFINAL MIGRATION VALIDATION")
print("=" * 65)

print(
    "PASS — MIGRATED COPIES VALIDATED"
    if migration_validation_passed
    else "STOP — INVESTIGATE FAILED CHECKS"
)

print("\nOriginal performance tables overwritten: NO")
print("Canonical names modified: NO")
print("Performance statistics modified: NO")


PERFORMANCE ID MIGRATION VALIDATION

BATTING
---------------------------------------------
Rows: 45432
Unique migrated IDs: 1934
IDs missing from final map: 0
Rows preserved: True
Columns preserved: True
Canonical names unchanged: True
Other values unchanged: True

BOWLING
---------------------------------------------
Rows: 31186
Unique migrated IDs: 1502
IDs missing from final map: 0
Rows preserved: True
Columns preserved: True
Canonical names unchanged: True
Other values unchanged: True

DISMISSAL
---------------------------------------------
Rows: 37711
Unique migrated IDs: 1841
IDs missing from final map: 0
Rows preserved: True
Columns preserved: True
Canonical names unchanged: True
Other values unchanged: True

DISMISSAL_BOWLER
---------------------------------------------
Rows: 34975
Unique migrated IDs: 1281
IDs missing from final map: 0
Rows preserved: True
Columns preserved: True
Canonical names unchanged: True
Other values unchanged: True

FINAL MIGRATION VALIDATION
PASS — MI

In [205]:

# ============================================================
# CANONICAL NAME AUDIT — READ ONLY
# No performance tables will be modified
# ============================================================

import pandas as pd

# Final identity map lookup
identity_lookup = (
    player_identity_map_updated[
        [
            "cricketiq_player_id",
            "crickrida_name",
            "cricsheet_name",
            "match_status"
        ]
    ]
    .drop_duplicates("cricketiq_player_id")
    .set_index("cricketiq_player_id")
)

tables = {
    "batting": player_batting_migrated,
    "bowling": player_bowling_migrated,
    "dismissal": player_dismissal_migrated,
    "dismissal_bowler": dismissal_bowler_migrated
}

canonical_name_audit = []

for table_name, df in tables.items():

    audit = df[
        ["cricketiq_player_id", "canonical_player_name"]
    ].copy()

    audit["table"] = table_name

    audit = audit.join(
        identity_lookup,
        on="cricketiq_player_id"
    )

    # Preferred name:
    # 1. Verified Crickrida full name
    # 2. Existing canonical name
    # 3. Cricsheet source name

    audit["proposed_canonical_name"] = (
        audit["crickrida_name"]
        .combine_first(audit["canonical_player_name"])
        .combine_first(audit["cricsheet_name"])
    )

    audit["name_changed"] = (
        audit["canonical_player_name"].astype("string")
        != audit["proposed_canonical_name"].astype("string")
    )

    canonical_name_audit.append(audit)

canonical_name_audit = pd.concat(
    canonical_name_audit,
    ignore_index=True
)

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("CANONICAL NAME AUDIT")
print("=" * 65)

print("Total performance records:", len(canonical_name_audit))

print(
    "Missing proposed names:",
    canonical_name_audit["proposed_canonical_name"].isna().sum()
)

print(
    "Records retaining existing names:",
    (~canonical_name_audit["name_changed"]).sum()
)

print(
    "Records with proposed name changes:",
    canonical_name_audit["name_changed"].sum()
)

print("\nNAME CHANGES BY TABLE")

print(
    canonical_name_audit.groupby("table")["name_changed"]
    .agg(["count", "sum"])
)

# ------------------------------------------------------------
# UNIQUE PLAYER-LEVEL NAME CHANGES
# ------------------------------------------------------------

changed_names = canonical_name_audit[
    canonical_name_audit["name_changed"]
][
    [
        "cricketiq_player_id",
        "canonical_player_name",
        "proposed_canonical_name",
        "crickrida_name",
        "cricsheet_name",
        "match_status",
        "table"
    ]
].drop_duplicates(
    ["cricketiq_player_id", "canonical_player_name",
     "proposed_canonical_name"]
)

print("\nUNIQUE PLAYER NAME CHANGES")
print("=" * 65)

print("Unique changes:", len(changed_names))

display(
    changed_names.head(50)
)

print("\nSTATUS: READ-ONLY AUDIT COMPLETE")
print("Performance tables modified: NO")


CANONICAL NAME AUDIT
Total performance records: 149304
Missing proposed names: 0
Records retaining existing names: 102626
Records with proposed name changes: 46678

NAME CHANGES BY TABLE
                  count    sum
table                         
batting           45432  14938
bowling           31186   9151
dismissal         37711  12504
dismissal_bowler  34975  10085

UNIQUE PLAYER NAME CHANGES
Unique changes: 294


,cricketiq_player_id,canonical_player_name,proposed_canonical_name,crickrida_name,cricsheet_name,match_status,table
2,CRIQ_dcce6f09,DA Warner,David Warner,David Warner,DA Warner,verified_alias,batting
3,CRIQ_b681e71e,GJ Maxwell,Glenn Maxwell,Glenn Maxwell,GJ Maxwell,verified_alias,batting
4,CRIQ_808f425a,JP Faulkner,James Faulkner,James Faulkner,JP Faulkner,verified_alias,batting
5,CRIQ_3fb19989,MA Starc,Mitchell Starc,Mitchell Starc,MA Starc,verified_alias,batting
6,CRIQ_3d8feaf8,MR Marsh,Mitchell Marsh,Mitchell Marsh,MR Marsh,verified_alias,batting
7,CRIQ_afa7e784,MS Wade,Matthew Wade,Matthew Wade,MS Wade,verified_alias,batting
8,CRIQ_ded9240e,PJ Cummins,Pat Cummins,Pat Cummins,PJ Cummins,verified_alias,batting
9,CRIQ_30a45b23,SPD Smith,Steve Smith,Steve Smith,SPD Smith,verified_alias,batting
10,CRIQ_12b610c2,TM Head,Travis Head,Travis Head,TM Head,verified_alias,batting
25,CRIQ_03806cf8,JR Hazlewood,Josh Hazlewood,Josh Hazlewood,JR Hazlewood,verified_alias,batting



STATUS: READ-ONLY AUDIT COMPLETE
Performance tables modified: NO


In [206]:

# ============================================================
# PLAYER-LEVEL CANONICAL NAME CONSISTENCY AUDIT
# READ ONLY — NO DATA MODIFIED
# ============================================================

# 1. Group all performance records by CricketIQ ID
player_name_review = (
    canonical_name_audit
    .groupby("cricketiq_player_id")
    .agg(
        current_name_count=("canonical_player_name", "nunique"),
        proposed_name_count=("proposed_canonical_name", "nunique"),
        identity_status_count=("match_status", "nunique"),
        current_names=("canonical_player_name",
                       lambda x: sorted(x.dropna().unique())),
        proposed_names=("proposed_canonical_name",
                        lambda x: sorted(x.dropna().unique())),
        identity_status=("match_status", "first"),
        tables=("table", lambda x: sorted(x.unique())),
        record_count=("table", "size")
    )
    .reset_index()
)

# 2. Identify any conflicting identities
conflicts = player_name_review[
    (player_name_review["current_name_count"] != 1) |
    (player_name_review["proposed_name_count"] != 1) |
    (player_name_review["identity_status_count"] != 1)
].copy()

# 3. Keep only players whose names would change
player_name_changes = player_name_review[
    player_name_review.apply(
        lambda row: row["current_names"] != row["proposed_names"],
        axis=1
    )
].copy()

# 4. Summary by identity status
print("PLAYER-LEVEL NAME CONSISTENCY AUDIT")
print("=" * 65)

print("Unique performance players:", len(player_name_review))
print("Players with proposed name changes:", len(player_name_changes))
print("Players with identity conflicts:", len(conflicts))

print("\nCHANGES BY IDENTITY STATUS")
display(
    player_name_changes.groupby("identity_status")
    .agg(
        players=("cricketiq_player_id", "nunique"),
        performance_records=("record_count", "sum")
    )
    .sort_values("players", ascending=False)
)

# 5. Display every changed player, not just the first 50
print("\nALL PROPOSED PLAYER NAME CHANGES")
display(
    player_name_changes[
        [
            "cricketiq_player_id",
            "current_names",
            "proposed_names",
            "identity_status",
            "tables",
            "record_count"
        ]
    ].sort_values("proposed_names")
)

# 6. Final consistency result
print("\nFINAL RESULT")
print("=" * 65)

if len(conflicts) == 0:
    print("PASS — Every player has one consistent current name.")
    print("PASS — Every player has one consistent proposed name.")
    print("PASS — Every player has one consistent identity status.")
else:
    print("REVIEW REQUIRED — Conflicting player identities found.")
    display(conflicts)

print("\nPerformance tables modified: NO")
print("Identity map modified: NO")


PLAYER-LEVEL NAME CONSISTENCY AUDIT
Unique performance players: 2008
Players with proposed name changes: 294
Players with identity conflicts: 0

CHANGES BY IDENTITY STATUS


,players,performance_records
identity_status,,
verified_alias,256,44924
verified_manual_identity,23,311
verified_full_name_and_career,10,1023
verified_ireland_identity,2,5
verified_cross_country_identity,1,398
verified_external_career_and_match_records,1,3
verified_match_and_career,1,14



ALL PROPOSED PLAYER NAME CHANGES


,cricketiq_player_id,current_names,proposed_names,identity_status,tables,record_count
297,CRIQ_b8d490fd,[AJ Finch],[Aaron Finch],verified_alias,"[batting, bowling, dismissal, dismissal_bowler]",300
75,CRIQ_350bb1b1,[AF Milne],[Adam Milne],verified_alias,"[batting, bowling, dismissal, dismissal_bowler]",139
25,CRIQ_14f96089,[A Zampa],[Adam Zampa],verified_alias,"[batting, bowling, dismissal, dismissal_bowler]",435
45,CRIQ_249d60c9,[AU Rashid],[Adil Rashid],verified_alias,"[batting, bowling, dismissal, dismissal_bowler]",530
154,CRIQ_6a26221c,[AK Markram],[Aiden Markram],verified_alias,"[batting, bowling, dismissal, dismissal_bowler]",210
...,...,...,...,...,...,...
388,CRIQ_ec02b798,[WTS Porterfield],[William Porterfield],verified_full_name_and_career,"[batting, dismissal]",178
122,CRIQ_502b2c81,[XJ Doherty],[Xavier Doherty],verified_alias,"[batting, bowling, dismissal, dismissal_bowler]",138
158,CRIQ_6c19c6e5,[YBK Jaiswal],[Yashasvi Jaiswal],verified_alias,"[batting, dismissal]",7
131,CRIQ_5791b840,[Yasir Ali Chowdhury],"[Yasir Ali (Bangladesh, 2021 to 2026)]",verified_alias,"[batting, dismissal]",10



FINAL RESULT
PASS — Every player has one consistent current name.
PASS — Every player has one consistent proposed name.
PASS — Every player has one consistent identity status.

Performance tables modified: NO
Identity map modified: NO


In [207]:

# ============================================================
# CANONICAL NAME UPDATE
# CREATE NEW COPIES — PRESERVE ALL PREVIOUS VERSIONS
# ============================================================

# ------------------------------------------------------------
# 1. Build one verified name per CricketIQ ID
# ------------------------------------------------------------

canonical_name_mapping = (
    canonical_name_audit[
        ["cricketiq_player_id", "proposed_canonical_name"]
    ]
    .drop_duplicates()
)

# Safety checks
assert canonical_name_mapping["cricketiq_player_id"].is_unique

assert canonical_name_mapping["proposed_canonical_name"].notna().all()

assert len(canonical_name_mapping) == 2008

canonical_name_lookup = canonical_name_mapping.set_index(
    "cricketiq_player_id"
)["proposed_canonical_name"]

# ------------------------------------------------------------
# 2. Preserve ID-migrated versions before name updates
# ------------------------------------------------------------

player_batting_before_canonical_update = (
    player_batting_migrated.copy(deep=True)
)

player_bowling_before_canonical_update = (
    player_bowling_migrated.copy(deep=True)
)

player_dismissal_before_canonical_update = (
    player_dismissal_migrated.copy(deep=True)
)

dismissal_bowler_before_canonical_update = (
    dismissal_bowler_migrated.copy(deep=True)
)

# ------------------------------------------------------------
# 3. Create new canonical-name-updated copies
# ------------------------------------------------------------

player_batting_canonical = player_batting_migrated.copy(deep=True)
player_bowling_canonical = player_bowling_migrated.copy(deep=True)
player_dismissal_canonical = player_dismissal_migrated.copy(deep=True)
dismissal_bowler_canonical = dismissal_bowler_migrated.copy(deep=True)

canonical_tables = {
    "batting": player_batting_canonical,
    "bowling": player_bowling_canonical,
    "dismissal": player_dismissal_canonical,
    "dismissal_bowler": dismissal_bowler_canonical
}

pre_update_tables = {
    "batting": player_batting_before_canonical_update,
    "bowling": player_bowling_before_canonical_update,
    "dismissal": player_dismissal_before_canonical_update,
    "dismissal_bowler": dismissal_bowler_before_canonical_update
}

# ------------------------------------------------------------
# 4. Apply verified canonical names
# ------------------------------------------------------------

for table_name, df in canonical_tables.items():

    proposed_names = df["cricketiq_player_id"].map(
        canonical_name_lookup
    )

    assert proposed_names.notna().all(), (
        f"Missing canonical name in {table_name}"
    )

    df["canonical_player_name"] = proposed_names

# ------------------------------------------------------------
# 5. Validate every table
# ------------------------------------------------------------

print("CANONICAL NAME UPDATE VALIDATION")
print("=" * 65)

canonical_validation = {}

for table_name, updated_df in canonical_tables.items():

    before_df = pre_update_tables[table_name]

    # All non-name columns must remain unchanged
    other_columns = [
        col for col in before_df.columns
        if col != "canonical_player_name"
    ]

    other_values_unchanged = updated_df[other_columns].equals(
        before_df[other_columns]
    )

    row_count_unchanged = len(updated_df) == len(before_df)

    no_missing_names = (
        updated_df["canonical_player_name"].notna().all()
    )

    # Check each player's name against the verified lookup
    expected_names = updated_df["cricketiq_player_id"].map(
        canonical_name_lookup
    )

    names_correct = updated_df[
        "canonical_player_name"
    ].equals(expected_names)

    canonical_validation[table_name] = {
        "row_count_unchanged": row_count_unchanged,
        "other_values_unchanged": other_values_unchanged,
        "no_missing_names": no_missing_names,
        "names_match_verified_lookup": names_correct
    }

    print(f"\n{table_name.upper()}")
    print("-" * 45)
    print("Rows:", len(updated_df))
    print("Unique players:", updated_df["cricketiq_player_id"].nunique())
    print("Missing canonical names:",
          updated_df["canonical_player_name"].isna().sum())
    print("Rows preserved:", row_count_unchanged)
    print("All other values unchanged:", other_values_unchanged)
    print("Names match verified lookup:", names_correct)

# ------------------------------------------------------------
# 6. Final result
# ------------------------------------------------------------

canonical_update_passed = all(
    all(result.values())
    for result in canonical_validation.values()
)

print("\nFINAL RESULT")
print("=" * 65)

if canonical_update_passed:
    print("PASS — CANONICAL NAME UPDATE VALIDATED")
else:
    print("STOP — INVESTIGATE FAILED VALIDATION")

print("\nOriginal performance tables overwritten: NO")
print("ID-migrated copies overwritten: NO")
print("Identity map modified: NO")
print("Player registry modified: NO")


CANONICAL NAME UPDATE VALIDATION

BATTING
---------------------------------------------
Rows: 45432
Unique players: 1934
Missing canonical names: 0
Rows preserved: True
All other values unchanged: True
Names match verified lookup: True

BOWLING
---------------------------------------------
Rows: 31186
Unique players: 1502
Missing canonical names: 0
Rows preserved: True
All other values unchanged: True
Names match verified lookup: True

DISMISSAL
---------------------------------------------
Rows: 37711
Unique players: 1841
Missing canonical names: 0
Rows preserved: True
All other values unchanged: True
Names match verified lookup: True

DISMISSAL_BOWLER
---------------------------------------------
Rows: 34975
Unique players: 1281
Missing canonical names: 0
Rows preserved: True
All other values unchanged: True
Names match verified lookup: True

FINAL RESULT
PASS — CANONICAL NAME UPDATE VALIDATED

Original performance tables overwritten: NO
ID-migrated copies overwritten: NO
Identity ma

## 8. Legacy Registry Reconciliation and Historical Aliases

Compare historical player registries with current source history and the final identity map. Investigate unmatched legacy records and preserve verified historical name variations in a separate alias table.

In [208]:

# ============================================================
# PLAYER REGISTRY — PRE-REBUILD INSPECTION
# READ ONLY — NO DATA MODIFICATIONS
# ============================================================

print("IDENTITY MAP")
print("=" * 60)
print("Shape:", player_identity_map_updated.shape)
print("Columns:")
print(player_identity_map_updated.columns.tolist())

print("\nIDENTITY STATUS COUNTS")
print(
    player_identity_map_updated["match_status"]
    .value_counts(dropna=False)
)

print("\nEXISTING REGISTRY VARIABLES")
print("=" * 60)

for name in [
    "player_registry",
    "player_registry_updated",
    "player_registry_final_updated",
    "player_registry_final",
    "player_registry_backup"
]:
    if name in globals():
        df = globals()[name]
        print(f"\n{name}: {df.shape}")
        print("Columns:", df.columns.tolist())
        print(df.head(3).to_string(index=False))
    else:
        print(f"\n{name}: Not found")

print("\nVALIDATED PERFORMANCE DATASETS")
print("=" * 60)

for name in [
    "player_batting_canonical",
    "player_bowling_canonical",
    "player_dismissal_canonical",
    "dismissal_bowler_canonical"
]:
    df = globals()[name]
    print(f"{name}: {df.shape}")
    print("Columns:", df.columns.tolist())

print("\nINSPECTION COMPLETE — NO DATA MODIFIED")


IDENTITY MAP
Shape: (2070, 12)
Columns:
['cricketiq_player_id', 'player_id', 'cricsheet_name', 'crickrida_name', 'normalized_name', 'match_status', 'crickrida_player_name', 'cricsheet_player_name', 'cricsheet_player_id', 'mapping_method', 'confidence', 'notes']

IDENTITY STATUS COUNTS
match_status
cricsheet_only                                1593
verified_alias                                 279
exact_name_match                                98
crickrida_only                                  62
verified_manual_identity                        23
verified_full_name_and_career                   10
verified_ireland_identity                        2
verified_match_and_career                        1
verified_cross_country_identity                  1
verified_external_career_and_match_records       1
Name: count, dtype: int64

EXISTING REGISTRY VARIABLES

player_registry: Not found

player_registry_updated: (3635, 10)
Columns: ['cricketiq_player_id', 'player_id', 'cricsheet_name', 'crickr

In [210]:

# ============================================================
# OLD REGISTRY AUDIT — CORRECTED
# READ ONLY — NO DATA MODIFICATIONS
# ============================================================

import pandas as pd

old_registries = {
    "player_registry_updated": player_registry_updated,
    "player_registry_final_updated": player_registry_final_updated,
    "player_registry_final": player_registry_final
}

map_ids = set(
    player_identity_map_updated["cricketiq_player_id"]
    .dropna()
    .astype(str)
)

def safe_unique_count(series):
    """Count unique values, including list/dict values."""
    values = series.dropna().apply(
        lambda x: repr(x) if isinstance(x, (list, dict, set)) else str(x)
    )
    return values.nunique()

for registry_name, registry in old_registries.items():

    print("\n" + "=" * 75)
    print(f"REGISTRY AUDIT: {registry_name}")
    print("=" * 75)

    print("Shape:", registry.shape)
    print("\nMissing values:")
    print(registry.isna().sum().to_string())

    print("\nDuplicate CricketIQ IDs:")
    print(registry["cricketiq_player_id"].duplicated().sum())

    old_ids = set(
        registry["cricketiq_player_id"].dropna().astype(str)
    )

    print("\nIdentity ID comparison:")
    print("Old registry IDs:", len(old_ids))
    print("IDs also in final map:", len(old_ids & map_ids))
    print("IDs absent from final map:", len(old_ids - map_ids))

    if "source" in registry.columns:
        print("\nSource distribution:")
        print(registry["source"].value_counts(dropna=False).to_string())

    for column in [
        "player_names",
        "player_name",
        "canonical_player_name",
        "cricsheet_name",
        "crickrida_name",
        "normalized_name"
    ]:
        if column in registry.columns:
            print(f"\n{column}:")
            print("Unique non-null values:", safe_unique_count(registry[column]))
            print("Non-null values:", registry[column].notna().sum())
            print("Examples:")
            print(
                registry[column]
                .dropna()
                .head(5)
                .tolist()
            )

# ============================================================
# INVESTIGATE IDs ABSENT FROM FINAL MAP
# ============================================================

print("\n" + "=" * 75)
print("INVESTIGATING OLD REGISTRY IDs ABSENT FROM FINAL MAP")
print("=" * 75)

old_registry = player_registry_updated.copy()

old_registry_ids = set(
    old_registry["cricketiq_player_id"].dropna().astype(str)
)

absent_ids = old_registry_ids - map_ids

absent_records = old_registry[
    old_registry["cricketiq_player_id"].astype(str).isin(absent_ids)
].copy()

print("Absent records:", len(absent_records))

print("\nID prefix distribution:")
print(
    absent_records["cricketiq_player_id"]
    .astype(str)
    .str.split("_")
    .str[0]
    .value_counts(dropna=False)
)

print("\nSample absent records:")
print(
    absent_records[
        [
            "cricketiq_player_id",
            "player_id",
            "cricsheet_name",
            "crickrida_name",
            "match_status",
            "source",
            "player_name",
            "canonical_player_name",
            "normalized_name"
        ]
    ].head(25).to_string(index=False)
)

print("\nAUDIT COMPLETE — NO DATA MODIFIED")



REGISTRY AUDIT: player_registry_updated
Shape: (3635, 10)

Missing values:
cricketiq_player_id         0
player_id                 348
cricsheet_name           3514
crickrida_name           3514
match_status                0
source                      0
player_names              469
player_name              3287
canonical_player_name       0
normalized_name          3620

Duplicate CricketIQ IDs:
0

Identity ID comparison:
Old registry IDs: 3635
IDs also in final map: 1775
IDs absent from final map: 1860

Source distribution:
source
Cricsheet    3166
Crickrida     348
Both          121

player_names:
Unique non-null values: 3155
Non-null values: 3166
Examples:
[['PJ Hughes'], ['Panna Ghosh'], ['Zulfiqar Jan'], ['Glen Walklin'], ['Lui Els']]

player_name:
Unique non-null values: 348
Non-null values: 348
Examples:
['Paul Stirling', 'James Hall', 'Alex Cusack', "Kevin O'Brien", 'Andrew Poynter']

canonical_player_name:
Unique non-null values: 3608
Non-null values: 3635
Examples:
['Babar

In [211]:
# ============================================================
# PERFORMANCE IDENTITY MISMATCH DIAGNOSTICS
# READ-ONLY — NO MODIFICATIONS
# ============================================================

import pandas as pd

performance_tables = {
    "batting": player_batting_updated,
    "bowling": player_bowling_updated,
    "dismissal": player_dismissal_updated,
    "dismissal_bowler": dismissal_bowler_updated
}

identity_map = player_identity_map_updated.copy()

# Normalize ID representations for comparison
identity_map["_audit_id"] = (
    identity_map["cricketiq_player_id"]
    .astype("string").str.strip()
)

identity_ids = set(identity_map["_audit_id"].dropna())

print("PERFORMANCE IDENTITY DIAGNOSTICS")
print("=" * 70)

for table_name, df in performance_tables.items():

    temp = df.copy()

    temp["_audit_id"] = (
        temp["cricketiq_player_id"]
        .astype("string").str.strip()
    )

    # --------------------------------------------------------
    # 1. Identify missing IDs and their frequencies
    # --------------------------------------------------------

    missing_rows = temp[
        temp["_audit_id"].notna()
        & ~temp["_audit_id"].isin(identity_ids)
    ]

    missing_id_counts = (
        missing_rows["_audit_id"]
        .value_counts()
        .rename_axis("cricketiq_player_id")
        .reset_index(name="row_count")
    )

    # --------------------------------------------------------
    # 2. Identify valid IDs with name mismatches
    # --------------------------------------------------------

    valid_rows = temp[
        temp["_audit_id"].isin(identity_ids)
    ].copy()

    lookup = identity_map.set_index("_audit_id")

    valid_rows["_map_cricsheet_name"] = (
        valid_rows["_audit_id"].map(
            lookup["cricsheet_name"]
        )
    )

    valid_rows["_map_crickrida_name"] = (
        valid_rows["_audit_id"].map(
            lookup["crickrida_name"]
        )
    )

    valid_rows["_map_status"] = (
        valid_rows["_audit_id"].map(
            lookup["match_status"]
        )
    )

    valid_rows["_canonical_clean"] = (
        valid_rows["canonical_player_name"]
        .astype("string").str.strip()
    )

    valid_rows["_map_name_clean"] = (
        valid_rows["_map_cricsheet_name"]
        .astype("string").str.strip()
    )

    name_mismatch_rows = valid_rows[
        valid_rows["_canonical_clean"]
        .ne(valid_rows["_map_name_clean"])
    ]

    # --------------------------------------------------------
    # 3. Summary
    # --------------------------------------------------------

    print(f"\n{'=' * 70}")
    print(table_name.upper())
    print("=" * 70)

    print("Total rows:", len(temp))
    print("Unique performance IDs:", temp["_audit_id"].nunique())
    print("Unique missing IDs:", missing_id_counts.shape[0])
    print("Rows with missing IDs:", len(missing_rows))
    print("Unique valid IDs:", valid_rows["_audit_id"].nunique())
    print("Rows with name mismatches:", len(name_mismatch_rows))

    # --------------------------------------------------------
    # 4. Show missing ID examples
    # --------------------------------------------------------

    print("\nTOP 25 MISSING IDs BY ROW COUNT")

    display(missing_id_counts.head(25))

    # --------------------------------------------------------
    # 5. Show actual performance records for missing IDs
    # --------------------------------------------------------

    print("\nSAMPLE MISSING PERFORMANCE RECORDS")

    display(
        missing_rows[
            [
                "cricketiq_player_id",
                "canonical_player_name"
            ]
        ].drop_duplicates().head(25)
    )

    # --------------------------------------------------------
    # 6. Show name mismatches with map context
    # --------------------------------------------------------

    print("\nSAMPLE VALID-ID NAME MISMATCHES")

    display(
        name_mismatch_rows[
            [
                "cricketiq_player_id",
                "canonical_player_name",
                "_map_cricsheet_name",
                "_map_crickrida_name",
                "_map_status"
            ]
        ].drop_duplicates().head(25)
    )

# ------------------------------------------------------------
# 7. Inspect ID formats in the final map
# ------------------------------------------------------------

print("\nFINAL IDENTITY MAP — ID FORMAT SAMPLE")
print("=" * 70)

display(
    identity_map[
        [
            "cricketiq_player_id",
            "player_id",
            "cricsheet_player_id",
            "cricsheet_name",
            "crickrida_name",
            "match_status"
        ]
    ].head(30)
)

print("\nIDENTITY MAP ID PREFIX COUNTS")

display(
    identity_map["cricketiq_player_id"]
    .astype("string")
    .str.extract(r"^([A-Za-z]+)")[0]
    .value_counts(dropna=False)
    .to_frame("records")
)

print("\nSTATUS: DIAGNOSTICS COMPLETE — NO DATA MODIFIED")

PERFORMANCE IDENTITY DIAGNOSTICS

BATTING
Total rows: 45432
Unique performance IDs: 1934
Unique missing IDs: 290
Rows with missing IDs: 14938
Unique valid IDs: 1644
Rows with name mismatches: 1182

TOP 25 MISSING IDs BY ROW COUNT


,cricketiq_player_id,row_count
0,CS_ba607b88,302
1,CS_6b71e6cf,280
2,CS_740742ef,276
3,CS_d18f9182,260
4,CS_5bdcdb72,259
5,CS_d2a6c0e6,215
6,CS_b61a3e1a,209
7,CS_7ed9fd56,205
8,CS_db584dad,196
9,CS_f842c2cf,196



SAMPLE MISSING PERFORMANCE RECORDS


,cricketiq_player_id,canonical_player_name
2,CS_dcce6f09,DA Warner
3,CS_b681e71e,GJ Maxwell
4,CS_808f425a,JP Faulkner
5,CS_3fb19989,MA Starc
6,CS_3d8feaf8,MR Marsh
7,CS_afa7e784,MS Wade
8,CS_ded9240e,PJ Cummins
9,CS_30a45b23,SPD Smith
10,CS_12b610c2,TM Head
25,CS_03806cf8,JR Hazlewood



SAMPLE VALID-ID NAME MISMATCHES


,cricketiq_player_id,canonical_player_name,_map_cricsheet_name,_map_crickrida_name,_map_status
101,CRIQ_94d7f855,Colin de Grandhomme,C de Grandhomme,Colin de Grandhomme,verified_alias
109,CRIQ_e824e6ee,Tom Latham,TWM Latham,Tom Latham,verified_alias
332,CRIQ_ab6b1c45,Andile Phehlukwayo,AL Phehlukwayo,Andile Phehlukwayo,verified_alias
416,CRIQ_a343262c,Joe Root,JE Root,Joe Root,verified_alias
463,CRIQ_2254ab79,Sarfaraz Ahmed,Sarfraz Ahmed,Sarfaraz Ahmed,verified_alias
469,CRIQ_c58b0108,Mehidy Hasan Miraz,Mehedi Hasan Miraz,Mehidy Hasan Miraz,verified_alias
489,CRIQ_8d92a2c3,Mark Wood,MA Wood,Mark Wood,verified_alias
706,CRIQ_e342e5fb,Carlos Brathwaite,CR Brathwaite,Carlos Brathwaite,verified_alias
757,CRIQ_6ef13460,Andy Balbirnie,A Balbirnie,Andy Balbirnie,verified_alias
758,CRIQ_a48fc79d,Ed Joyce,EC Joyce,Ed Joyce,verified_alias


... (19 outputs left out)

,cricketiq_player_id,player_id,cricsheet_player_id,cricsheet_name,crickrida_name,match_status
0,CRIQ_8a75e999,8a75e999,NaN,Babar Azam,Babar Azam,exact_name_match
1,CRIQ_2911de16,2911de16,NaN,Hasan Ali,Hasan Ali,exact_name_match
2,CRIQ_9cb8d7a6,9cb8d7a6,NaN,Imad Wasim,Imad Wasim,exact_name_match
3,CRIQ_e174dadd,e174dadd,NaN,Mohammad Amir,Mohammad Amir,exact_name_match
4,CRIQ_9ab63e7b,9ab63e7b,NaN,Mohammad Hafeez,Mohammad Hafeez,exact_name_match
5,CRIQ_2f26ac1a,2f26ac1a,NaN,Mohammad Rizwan,Mohammad Rizwan,exact_name_match
6,CRIQ_193ef196,193ef196,NaN,Sharjeel Khan,Sharjeel Khan,exact_name_match
7,CRIQ_fd2bf2a0,fd2bf2a0,NaN,Umar Akmal,Umar Akmal,exact_name_match
8,CRIQ_b3118300,b3118300,NaN,Wahab Riaz,Wahab Riaz,exact_name_match
9,CRIQ_50250426,50250426,NaN,Asad Shafiq,Asad Shafiq,exact_name_match



IDENTITY MAP ID PREFIX COUNTS


,records
0,
CS,1593
CRIQ,415
CR,62



STATUS: DIAGNOSTICS COMPLETE — NO DATA MODIFIED


In [212]:

# ============================================================
# OLD REGISTRY vs CURRENT SOURCE HISTORY RECONCILIATION
# READ ONLY — NO IDENTITY CHANGES
# ============================================================

import pandas as pd

old = player_registry_updated.copy()
old_final = player_registry_final_updated.copy()

# ------------------------------------------------------------
# 1. Prepare old records absent from the final identity map
# ------------------------------------------------------------

final_ids = set(
    player_identity_map_updated["cricketiq_player_id"]
    .dropna()
    .astype(str)
)

old["cricketiq_player_id"] = (
    old["cricketiq_player_id"].astype(str)
)

old_absent = old[
    ~old["cricketiq_player_id"].isin(final_ids)
].copy()

# Bring in normalized names from the older registry version
old_norm_lookup = (
    old_final[
        ["cricketiq_player_id", "normalized_name"]
    ]
    .dropna(subset=["normalized_name"])
    .drop_duplicates("cricketiq_player_id")
)

old_absent = old_absent.merge(
    old_norm_lookup,
    on="cricketiq_player_id",
    how="left",
    suffixes=("", "_from_final")
)

# ------------------------------------------------------------
# 2. Cricsheet source ID reconciliation
# ------------------------------------------------------------

cs_source_ids = set(
    cs_player_history["source_player_id"]
    .dropna()
    .astype(str)
)

old_absent["player_id"] = (
    old_absent["player_id"].astype("string")
)

old_absent["cs_source_id_exists"] = (
    old_absent["player_id"].isin(cs_source_ids)
)

old_cs = old_absent[
    old_absent["cricketiq_player_id"].str.startswith("CS_")
].copy()

print("=" * 75)
print("CRICSHEET OLD ID RECONCILIATION")
print("=" * 75)

print("Old absent CS records:", len(old_cs))
print(
    "Player IDs found in current Cricsheet history:",
    int(old_cs["cs_source_id_exists"].sum())
)
print(
    "Player IDs NOT found:",
    int((~old_cs["cs_source_id_exists"]).sum())
)

print("\nExamples found in current history:")
print(
    old_cs.loc[
        old_cs["cs_source_id_exists"],
        [
            "cricketiq_player_id",
            "player_id",
            "canonical_player_name"
        ]
    ].head(15).to_string(index=False)
)

print("\nExamples NOT found in current history:")
print(
    old_cs.loc[
        ~old_cs["cs_source_id_exists"],
        [
            "cricketiq_player_id",
            "player_id",
            "canonical_player_name"
        ]
    ].head(15).to_string(index=False)
)

# ------------------------------------------------------------
# 3. Crickrida normalized-name reconciliation
# ------------------------------------------------------------

cr_source_names = set(
    cr_player_history["normalized_name"]
    .dropna()
    .astype(str)
)

old_cr = old_absent[
    old_absent["cricketiq_player_id"].str.startswith("CR_")
].copy()

old_cr["normalized_name"] = (
    old_cr["normalized_name"]
    .fillna(old_cr.get("normalized_name_from_final"))
)

old_cr["cr_normalized_name_exists"] = (
    old_cr["normalized_name"]
    .astype("string")
    .isin(cr_source_names)
)

print("\n" + "=" * 75)
print("CRICKRIDA OLD ID RECONCILIATION")
print("=" * 75)

print("Old absent CR records:", len(old_cr))
print(
    "Normalized names found in current Crickrida history:",
    int(old_cr["cr_normalized_name_exists"].sum())
)
print(
    "Normalized names NOT found:",
    int((~old_cr["cr_normalized_name_exists"]).sum())
)
print(
    "Records with no normalized name:",
    int(old_cr["normalized_name"].isna().sum())
)

print("\nExamples found in current history:")
print(
    old_cr.loc[
        old_cr["cr_normalized_name_exists"],
        [
            "cricketiq_player_id",
            "player_name",
            "canonical_player_name",
            "normalized_name"
        ]
    ].head(15).to_string(index=False)
)

print("\nExamples NOT found:")
print(
    old_cr.loc[
        ~old_cr["cr_normalized_name_exists"],
        [
            "cricketiq_player_id",
            "player_name",
            "canonical_player_name",
            "normalized_name"
        ]
    ].head(20).to_string(index=False)
)

# ------------------------------------------------------------
# 4. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("RECONCILIATION SUMMARY")
print("=" * 75)

print("Old absent records:", len(old_absent))
print("Old CS records:", len(old_cs))
print("Old CR records:", len(old_cr))

print(
    "CS IDs present in current source:",
    int(old_cs["cs_source_id_exists"].sum())
)

print(
    "CR normalized names present in current source:",
    int(old_cr["cr_normalized_name_exists"].sum())
)

print("\nREAD-ONLY RECONCILIATION COMPLETE")
print("No identities merged, deleted, or modified.")


CRICSHEET OLD ID RECONCILIATION
Old absent CS records: 1573
Player IDs found in current Cricsheet history: 294
Player IDs NOT found: 1279

Examples found in current history:
cricketiq_player_id player_id canonical_player_name
        CS_024a173d  024a173d              J Nyumbu
        CS_03806cf8  03806cf8          JR Hazlewood
        CS_05de1a5a  05de1a5a             GC Wilson
        CS_06e649a0  06e649a0             RK Whelan
        CS_0994d0ae  0994d0ae             V Shankar
        CS_0a2184d9  0a2184d9             D Lakshan
        CS_0a476045  0a476045              S Dhawan
        CS_0a9d8b61  0a9d8b61             JL Carter
        CS_0b60eb09  0b60eb09            KA Maharaj
        CS_0ebfb1ad  0ebfb1ad               E Lewis
        CS_0f12f9df  0f12f9df           NLTC Perera
        CS_0f721006  0f721006             JO Holder
        CS_0fdabe05  0fdabe05             V Sibanda
        CS_12b610c2  12b610c2               TM Head
        CS_13c35c9e  13c35c9e            TG So

C:\Users\ASUS\AppData\Local\Temp\ipykernel_14868\3199112145.py:121: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(old_cr.get("normalized_name_from_final"))


In [215]:

# ============================================================
# 294 OLD CRICSHEET IDS — CORRECTED RECONCILIATION
# READ ONLY — NO IDENTITY CHANGES
# ============================================================

# 1. Extract old records whose source IDs exist in current history
old_cs_found = old_cs.loc[
    old_cs["cs_source_id_exists"]
].copy()

old_cs_found["source_player_id"] = (
    old_cs_found["player_id"].astype("string")
)

# 2. Build a clean source-ID lookup from final identity map
map_player_id = player_identity_map_updated[
    [
        "cricketiq_player_id",
        "player_id",
        "cricsheet_name",
        "crickrida_name",
        "match_status"
    ]
].copy()

map_player_id = map_player_id.rename(
    columns={"player_id": "source_player_id"}
)

map_cricsheet_id = player_identity_map_updated[
    [
        "cricketiq_player_id",
        "cricsheet_player_id",
        "cricsheet_name",
        "crickrida_name",
        "match_status"
    ]
].copy()

map_cricsheet_id = map_cricsheet_id.rename(
    columns={"cricsheet_player_id": "source_player_id"}
)

# 3. Combine the two source-ID columns
current_source_lookup = pd.concat(
    [map_player_id, map_cricsheet_id],
    ignore_index=True
)

current_source_lookup["source_player_id"] = (
    current_source_lookup["source_player_id"].astype("string")
)

current_source_lookup = (
    current_source_lookup
    .dropna(subset=["source_player_id"])
    .drop_duplicates(
        subset=["source_player_id", "cricketiq_player_id"]
    )
)

# 4. Merge old records against current identities
old_cs_comparison = old_cs_found.merge(
    current_source_lookup,
    on="source_player_id",
    how="left",
    suffixes=("_old", "_current"),
    validate="many_to_many"
)

# 5. Classify
old_cs_comparison["reconciliation_result"] = (
    old_cs_comparison["cricketiq_player_id_current"]
    .notna()
    .map({
        True: "SOURCE_ID_ALREADY_IN_FINAL_MAP",
        False: "SOURCE_ID_NOT_IN_FINAL_MAP"
    })
)

print("=" * 75)
print("294 OLD CRICSHEET ID RECONCILIATION")
print("=" * 75)

print(
    old_cs_comparison["reconciliation_result"]
    .value_counts(dropna=False)
)

print("\nSAMPLE SOURCE IDs ALREADY IN FINAL MAP")

print(
    old_cs_comparison[
        old_cs_comparison["reconciliation_result"]
        == "SOURCE_ID_ALREADY_IN_FINAL_MAP"
    ][
        [
            "cricketiq_player_id",
            "canonical_player_name",
            "source_player_id",
            "cricketiq_player_id_current",
            "cricsheet_name",
            "crickrida_name",
            "match_status"
        ]
    ].head(30).to_string(index=False)
)

print("\nSAMPLE SOURCE IDs NOT IN FINAL MAP")

print(
    old_cs_comparison[
        old_cs_comparison["reconciliation_result"]
        == "SOURCE_ID_NOT_IN_FINAL_MAP"
    ][
        [
            "cricketiq_player_id",
            "canonical_player_name",
            "source_player_id"
        ]
    ].head(20).to_string(index=False)
)

print("\nSource IDs mapping to multiple CricketIQ identities:")

mapping_counts = (
    current_source_lookup.groupby("source_player_id")
    ["cricketiq_player_id"]
    .nunique()
)

print((mapping_counts > 1).sum())

print("\nAUDIT COMPLETE — NO DATA MODIFIED")


294 OLD CRICSHEET ID RECONCILIATION
reconciliation_result
SOURCE_ID_ALREADY_IN_FINAL_MAP    294
Name: count, dtype: int64

SAMPLE SOURCE IDs ALREADY IN FINAL MAP


KeyError: "['cricketiq_player_id', 'cricsheet_name', 'crickrida_name', 'match_status'] not in index"

In [216]:

# ============================================================
# OLD vs CURRENT CRICSHEET IDENTITY COMPARISON
# READ ONLY
# ============================================================

# Build a clean lookup from the final identity map
lookup_player = player_identity_map_updated[
    [
        "cricketiq_player_id",
        "player_id",
        "cricsheet_name",
        "crickrida_name",
        "match_status"
    ]
].rename(columns={
    "cricketiq_player_id": "current_cricketiq_id",
    "player_id": "source_player_id",
    "cricsheet_name": "current_cricsheet_name",
    "crickrida_name": "current_crickrida_name",
    "match_status": "current_match_status"
})

lookup_cricsheet = player_identity_map_updated[
    [
        "cricketiq_player_id",
        "cricsheet_player_id",
        "cricsheet_name",
        "crickrida_name",
        "match_status"
    ]
].rename(columns={
    "cricketiq_player_id": "current_cricketiq_id",
    "cricsheet_player_id": "source_player_id",
    "cricsheet_name": "current_cricsheet_name",
    "crickrida_name": "current_crickrida_name",
    "match_status": "current_match_status"
})

current_lookup = pd.concat(
    [lookup_player, lookup_cricsheet],
    ignore_index=True
).dropna(subset=["source_player_id"])

current_lookup["source_player_id"] = (
    current_lookup["source_player_id"].astype(str)
)

current_lookup = current_lookup.drop_duplicates()

# Check whether any source ID maps to multiple current identities
source_mapping_counts = (
    current_lookup.groupby("source_player_id")
    ["current_cricketiq_id"].nunique()
)

ambiguous_source_ids = source_mapping_counts[
    source_mapping_counts > 1
]

print("Source IDs mapping to multiple current identities:",
      len(ambiguous_source_ids))

# Prepare old records
old_display = old_cs_found[
    [
        "cricketiq_player_id",
        "player_id",
        "canonical_player_name"
    ]
].copy()

old_display = old_display.rename(columns={
    "cricketiq_player_id": "old_cricketiq_id",
    "player_id": "source_player_id",
    "canonical_player_name": "old_canonical_name"
})

old_display["source_player_id"] = (
    old_display["source_player_id"].astype(str)
)

# Merge using explicit column names
comparison = old_display.merge(
    current_lookup,
    on="source_player_id",
    how="left"
)

# Display results
print("\nRECONCILIATION COUNTS")
print("=" * 65)

print("Old records:", len(old_display))
print("Matched current identities:",
      comparison["current_cricketiq_id"].notna().sum())
print("Unmatched:", comparison["current_cricketiq_id"].isna().sum())

print("\nOLD ID vs CURRENT ID — SAMPLE")
print("=" * 65)

print(
    comparison[
        [
            "old_cricketiq_id",
            "old_canonical_name",
            "source_player_id",
            "current_cricketiq_id",
            "current_cricsheet_name",
            "current_crickrida_name",
            "current_match_status"
        ]
    ].head(30).to_string(index=False)
)

print("\nOLD ID AND CURRENT ID DIFFERENCES")
print("=" * 65)

comparison["same_cricketiq_id"] = (
    comparison["old_cricketiq_id"]
    == comparison["current_cricketiq_id"]
)

print(
    comparison["same_cricketiq_id"]
    .value_counts(dropna=False)
)

print("\nAUDIT COMPLETE — NO DATA MODIFIED")


Source IDs mapping to multiple current identities: 0

RECONCILIATION COUNTS
Old records: 294
Matched current identities: 294
Unmatched: 0

OLD ID vs CURRENT ID — SAMPLE
old_cricketiq_id old_canonical_name source_player_id current_cricketiq_id current_cricsheet_name current_crickrida_name          current_match_status
     CS_024a173d           J Nyumbu         024a173d        CRIQ_024a173d               J Nyumbu            John Nyumbu                verified_alias
     CS_03806cf8       JR Hazlewood         03806cf8        CRIQ_03806cf8           JR Hazlewood         Josh Hazlewood                verified_alias
     CS_05de1a5a          GC Wilson         05de1a5a        CRIQ_05de1a5a              GC Wilson            Gary Wilson verified_full_name_and_career
     CS_06e649a0          RK Whelan         06e649a0        CRIQ_06e649a0              RK Whelan           Roger Whelan      verified_manual_identity
     CS_0994d0ae          V Shankar         0994d0ae        CRIQ_0994d0ae        

In [218]:

# ============================================================
# OLD CRICKRIDA NAME RECONCILIATION — FIXED
# READ ONLY — NO IDENTITY CHANGES
# ============================================================

old_cr = old_absent[
    old_absent["cricketiq_player_id"].str.startswith("CR_")
].copy()

# 1. Generate normalized names from old registry names
old_cr["old_normalized_name"] = (
    old_cr["player_name"]
    .fillna(old_cr["canonical_player_name"])
    .apply(
        lambda x: normalize_identity(x)
        if pd.notna(x) else None
    )
)

# 2. Prepare current Crickrida history with explicit column names
current_cr_lookup = cr_player_history[
    [
        "normalized_name",
        "player_name",
        "team",
        "crickrida_first_date",
        "crickrida_last_date",
        "crickrida_matches"
    ]
].copy()

current_cr_lookup = current_cr_lookup.rename(
    columns={
        "normalized_name": "current_normalized_name",
        "player_name": "current_player_name",
        "team": "current_team",
        "crickrida_first_date": "current_first_date",
        "crickrida_last_date": "current_last_date",
        "crickrida_matches": "current_matches"
    }
)

current_cr_lookup = current_cr_lookup.drop_duplicates(
    subset=["current_normalized_name"]
)

# 3. Exact normalized-name comparison
old_cr_comparison = old_cr.merge(
    current_cr_lookup,
    left_on="old_normalized_name",
    right_on="current_normalized_name",
    how="left"
)

# 4. Classify matches
old_cr_comparison["reconciliation_result"] = (
    old_cr_comparison["current_normalized_name"]
    .notna()
    .map({
        True: "NORMALIZED_NAME_MATCH",
        False: "NO_EXACT_NORMALIZED_MATCH"
    })
)

# 5. Results
print("=" * 75)
print("OLD CRICKRIDA NAME RECONCILIATION")
print("=" * 75)

print("Old records:", len(old_cr_comparison))

print("\nReconciliation results:")
print(
    old_cr_comparison["reconciliation_result"]
    .value_counts(dropna=False)
)

print("\nEXACT NORMALIZED NAME MATCHES")
print("-" * 75)

print(
    old_cr_comparison.loc[
        old_cr_comparison["reconciliation_result"]
        == "NORMALIZED_NAME_MATCH",
        [
            "cricketiq_player_id",
            "player_name",
            "old_normalized_name",
            "current_player_name",
            "current_team",
            "current_first_date",
            "current_last_date",
            "current_matches"
        ]
    ].head(40).to_string(index=False)
)

print("\nNO EXACT MATCHES — SAMPLE")
print("-" * 75)

print(
    old_cr_comparison.loc[
        old_cr_comparison["reconciliation_result"]
        == "NO_EXACT_NORMALIZED_MATCH",
        [
            "cricketiq_player_id",
            "player_name",
            "canonical_player_name",
            "old_normalized_name"
        ]
    ].head(30).to_string(index=False)
)

print("\nMissing normalized old names:",
      old_cr_comparison["old_normalized_name"].isna().sum())

print("\nAUDIT COMPLETE — NO DATA MODIFIED")


OLD CRICKRIDA NAME RECONCILIATION
Old records: 287

Reconciliation results:
reconciliation_result
NORMALIZED_NAME_MATCH    287
Name: count, dtype: int64

EXACT NORMALIZED NAME MATCHES
---------------------------------------------------------------------------
  cricketiq_player_id         player_name old_normalized_name current_player_name current_team current_first_date current_last_date  current_matches
      CR_paulstirling       Paul Stirling        paulstirling       Paul Stirling      Ireland         2010-07-03        2026-08-15               35
        CR_alexcusack         Alex Cusack          alexcusack         Alex Cusack      Ireland         2010-07-03        2015-01-17                4
       CR_kevinobrien       Kevin O'Brien         kevinobrien       Kevin O'Brien      Ireland         2010-07-03        2021-01-24               25
     CR_andrewpoynter      Andrew Poynter       andrewpoynter      Andrew Poynter      Ireland         2010-07-03        2010-07-03             

In [219]:

# ============================================================
# OLD CRICKRIDA IDs vs FINAL IDENTITY MAP
# READ ONLY — NO IDENTITY CHANGES
# ============================================================

# 1. Prepare old Crickrida records
old_cr_map_check = old_cr_comparison[
    [
        "cricketiq_player_id",
        "player_name",
        "old_normalized_name"
    ]
].copy()

old_cr_map_check = old_cr_map_check.rename(
    columns={
        "cricketiq_player_id": "old_cricketiq_id",
        "player_name": "old_player_name"
    }
)

# 2. Prepare final identity map using actual Crickrida names
current_map_cr = player_identity_map_updated[
    [
        "cricketiq_player_id",
        "crickrida_name",
        "cricsheet_name",
        "normalized_name",
        "match_status",
        "confidence"
    ]
].copy()

current_map_cr["map_crickrida_normalized"] = (
    current_map_cr["crickrida_name"]
    .apply(
        lambda x: normalize_identity(x)
        if pd.notna(x) else None
    )
)

current_map_cr = current_map_cr.rename(
    columns={
        "cricketiq_player_id": "current_cricketiq_id",
        "crickrida_name": "current_crickrida_name",
        "cricsheet_name": "current_cricsheet_name",
        "normalized_name": "map_stored_normalized_name",
        "match_status": "current_match_status",
        "confidence": "current_confidence"
    }
)

# 3. Check normalized-name uniqueness in final map
map_name_counts = (
    current_map_cr.dropna(subset=["map_crickrida_normalized"])
    .groupby("map_crickrida_normalized")
    ["current_cricketiq_id"]
    .nunique()
)

duplicate_map_names = map_name_counts[map_name_counts > 1]

print("=" * 75)
print("FINAL MAP NORMALIZED-NAME UNIQUENESS")
print("=" * 75)

print("Rows with Crickrida names:",
      current_map_cr["map_crickrida_normalized"].notna().sum())

print("Duplicate normalized names mapping to multiple IDs:",
      len(duplicate_map_names))

if len(duplicate_map_names):
    print("\nDuplicate normalized keys:")
    print(duplicate_map_names.to_string())

# 4. Compare old records against final map
old_cr_map_comparison = old_cr_map_check.merge(
    current_map_cr,
    left_on="old_normalized_name",
    right_on="map_crickrida_normalized",
    how="left"
)

old_cr_map_comparison["map_match_result"] = (
    old_cr_map_comparison["current_cricketiq_id"]
    .notna()
    .map({
        True: "FOUND_IN_FINAL_MAP",
        False: "NOT_FOUND_IN_FINAL_MAP"
    })
)

print("\n" + "=" * 75)
print("OLD CRICKRIDA vs FINAL MAP RESULTS")
print("=" * 75)

print(
    old_cr_map_comparison["map_match_result"]
    .value_counts(dropna=False)
)

print("\nOld records:", len(old_cr_map_check))
print(
    "Matched old records:",
    old_cr_map_comparison["old_cricketiq_id"].nunique()
)
print(
    "Unique current identities matched:",
    old_cr_map_comparison["current_cricketiq_id"].nunique()
)

# 5. Check whether old IDs already equal current IDs
matched = old_cr_map_comparison[
    old_cr_map_comparison["current_cricketiq_id"].notna()
].copy()

matched["old_id_equals_current_id"] = (
    matched["old_cricketiq_id"]
    == matched["current_cricketiq_id"]
)

print("\nOld ID vs current ID:")
print(
    matched["old_id_equals_current_id"]
    .value_counts(dropna=False)
)

# 6. Show matched examples
print("\nMATCHED RECORDS — SAMPLE")
print("-" * 75)

print(
    matched[
        [
            "old_cricketiq_id",
            "old_player_name",
            "current_cricketiq_id",
            "current_crickrida_name",
            "current_cricsheet_name",
            "current_match_status",
            "current_confidence"
        ]
    ].head(40).to_string(index=False)
)

# 7. Show records not found
print("\nNOT FOUND IN FINAL MAP — SAMPLE")
print("-" * 75)

print(
    old_cr_map_comparison.loc[
        old_cr_map_comparison["current_cricketiq_id"].isna(),
        [
            "old_cricketiq_id",
            "old_player_name",
            "old_normalized_name"
        ]
    ].head(30).to_string(index=False)
)

print("\nAUDIT COMPLETE — NO DATA MODIFIED")


FINAL MAP NORMALIZED-NAME UNIQUENESS
Rows with Crickrida names: 477
Duplicate normalized names mapping to multiple IDs: 0

OLD CRICKRIDA vs FINAL MAP RESULTS
map_match_result
FOUND_IN_FINAL_MAP    287
Name: count, dtype: int64

Old records: 287
Matched old records: 287
Unique current identities matched: 287

Old ID vs current ID:
old_id_equals_current_id
False    287
Name: count, dtype: int64

MATCHED RECORDS — SAMPLE
---------------------------------------------------------------------------
     old_cricketiq_id     old_player_name current_cricketiq_id current_crickrida_name current_cricsheet_name          current_match_status current_confidence
      CR_paulstirling       Paul Stirling        CRIQ_cf494ffe          Paul Stirling            PR Stirling                verified_alias                NaN
        CR_alexcusack         Alex Cusack        CRIQ_568e5a4a            Alex Cusack              AR Cusack                verified_alias                NaN
       CR_kevinobrien       

In [220]:

# ============================================================
# INVESTIGATE REMAINING 1,279 OLD CRICSHEET IDENTITIES
# READ ONLY — NO DATA MODIFICATIONS
# ============================================================

# 1. Extract old Cricsheet records not found in current history
old_cs_unmatched = old_cs.loc[
    ~old_cs["cs_source_id_exists"]
].copy()

old_cs_unmatched["source_player_id"] = (
    old_cs_unmatched["player_id"].astype("string")
)

print("=" * 75)
print("REMAINING OLD CRICSHEET IDENTITIES")
print("=" * 75)

print("Unmatched records:", len(old_cs_unmatched))

# 2. Inspect available source dataset
print("\nCRICSHEET PLAYERS ALL")
print("-" * 75)

print("Shape:", cricsheet_players_all.shape)
print("Columns:", cricsheet_players_all.columns.tolist())
print(cricsheet_players_all.head(5).to_string(index=False))

# 3. Identify source ID column automatically
possible_id_columns = [
    "source_player_id",
    "player_id",
    "cricsheet_player_id"
]

source_id_column = next(
    (c for c in possible_id_columns
     if c in cricsheet_players_all.columns),
    None
)

if source_id_column is None:
    print("\nNo recognized source ID column found.")
else:
    current_all_ids = set(
        cricsheet_players_all[source_id_column]
        .dropna()
        .astype(str)
    )

    unmatched_ids = set(
        old_cs_unmatched["source_player_id"]
        .dropna()
        .astype(str)
    )

    overlap = unmatched_ids & current_all_ids

    print("\nSOURCE ID OVERLAP")
    print("-" * 75)
    print("Old unmatched source IDs:", len(unmatched_ids))
    print("Found in cricsheet_players_all:", len(overlap))
    print("Not found:", len(unmatched_ids - current_all_ids))

# 4. Inspect old names and aliases
print("\nOLD REGISTRY NAME INFORMATION")
print("-" * 75)

for column in [
    "canonical_player_name",
    "player_names",
    "player_id"
]:
    if column in old_cs_unmatched.columns:
        print(f"\n{column} — examples:")
        print(
            old_cs_unmatched[column]
            .dropna()
            .head(15)
            .tolist()
        )

# 5. Compare old IDs with current final-map source IDs
final_source_ids = set(
    player_identity_map_updated["player_id"]
    .dropna()
    .astype(str)
) | set(
    player_identity_map_updated["cricsheet_player_id"]
    .dropna()
    .astype(str)
)

print("\nFINAL IDENTITY MAP SOURCE-ID CHECK")
print("-" * 75)
print(
    "Unmatched old IDs found in final map source columns:",
    len(unmatched_ids & final_source_ids)
    if source_id_column is not None else "Not checked"
)

print("\nAUDIT COMPLETE — NO DATA MODIFIED")


REMAINING OLD CRICSHEET IDENTITIES
Unmatched records: 1279

CRICSHEET PLAYERS ALL
---------------------------------------------------------------------------
Shape: (1354, 4)
Columns: ['team', 'player_name', 'source_player_id', 'normalized_name']
     team player_name source_player_id normalized_name
Australia  B Stanlake         6834d1f2       bstanlake
Australia     CA Lynn         45eda7c8          calynn
Australia   DA Warner         dcce6f09        dawarner
Australia  GJ Maxwell         b681e71e       gjmaxwell
Australia JP Faulkner         808f425a      jpfaulkner

SOURCE ID OVERLAP
---------------------------------------------------------------------------
Old unmatched source IDs: 1279
Found in cricsheet_players_all: 0
Not found: 1279

OLD REGISTRY NAME INFORMATION
---------------------------------------------------------------------------

canonical_player_name — examples:
['Panna Ghosh', 'Zulfiqar Jan', 'Glen Walklin', 'Lui Els', 'CK Nandan', 'D Lohia', 'EL Lamb', 'D Shawane'

In [221]:

# ============================================================
# CHECK WHETHER LEGACY CRICSHEET IDS ARE USED IN PERFORMANCE
# READ ONLY — NO DATA MODIFICATIONS
# ============================================================

legacy_source_ids = set(
    old_cs_unmatched["player_id"]
    .dropna()
    .astype(str)
)

legacy_cricketiq_ids = set(
    old_cs_unmatched["cricketiq_player_id"]
    .dropna()
    .astype(str)
)

legacy_all_ids = legacy_source_ids | legacy_cricketiq_ids

performance_sources = {
    "batting": (
        "player_batting_before_id_migration",
        "batter_id"
    ),
    "bowling": (
        "player_bowling_before_id_migration",
        "bowler_id"
    ),
    "dismissal": (
        "player_dismissal_before_id_migration",
        "player_id"
    ),
    "dismissal_bowler": (
        "dismissal_bowler_before_id_migration",
        "bowler_id"
    )
}

print("=" * 75)
print("LEGACY ID USAGE IN ORIGINAL PERFORMANCE TABLES")
print("=" * 75)

legacy_performance_results = []

for table_label, (variable_name, id_column) in performance_sources.items():

    if variable_name not in globals():
        print(f"\n{table_label}: BACKUP VARIABLE NOT FOUND")
        continue

    df = globals()[variable_name]

    if id_column not in df.columns:
        print(f"\n{table_label}: ID COLUMN '{id_column}' NOT FOUND")
        print("Available columns:", df.columns.tolist())
        continue

    table_ids = df[id_column].dropna().astype(str)

    matched_mask = table_ids.isin(legacy_all_ids)
    matched_values = set(table_ids[matched_mask])

    matched_rows = int(matched_mask.sum())

    legacy_performance_results.append({
        "table": table_label,
        "total_rows": len(df),
        "matched_rows": matched_rows,
        "unique_legacy_ids_used": len(matched_values)
    })

    print(f"\n{table_label.upper()}")
    print("-" * 75)
    print("Total rows:", len(df))
    print("Rows using legacy IDs:", matched_rows)
    print("Unique legacy IDs used:", len(matched_values))

    if matched_rows:
        sample = df.loc[matched_mask].copy()

        display_columns = [
            c for c in [
                id_column,
                "batter",
                "bowler",
                "player_name",
                "match_id"
            ]
            if c in sample.columns
        ]

        print("\nSample records:")
        print(sample[display_columns].head(10).to_string(index=False))

print("\n" + "=" * 75)
print("OVERALL RESULT")
print("=" * 75)

if legacy_performance_results:
    legacy_performance_summary = pd.DataFrame(
        legacy_performance_results
    )
    print(legacy_performance_summary.to_string(index=False))

    print(
        "\nTotal performance rows using legacy IDs:",
        legacy_performance_summary["matched_rows"].sum()
    )

print("\nAUDIT COMPLETE — NO DATA MODIFIED")


LEGACY ID USAGE IN ORIGINAL PERFORMANCE TABLES

BATTING
---------------------------------------------------------------------------
Total rows: 45432
Rows using legacy IDs: 0
Unique legacy IDs used: 0

BOWLING
---------------------------------------------------------------------------
Total rows: 31186
Rows using legacy IDs: 0
Unique legacy IDs used: 0

dismissal: ID COLUMN 'player_id' NOT FOUND
Available columns: ['match_id', 'innings_number', 'batting_team', 'player_out', 'player_out_id', 'dismissals', 'cricketiq_player_id', 'canonical_player_name']

DISMISSAL_BOWLER
---------------------------------------------------------------------------
Total rows: 34975
Rows using legacy IDs: 0
Unique legacy IDs used: 0

OVERALL RESULT
           table  total_rows  matched_rows  unique_legacy_ids_used
         batting       45432             0                       0
         bowling       31186             0                       0
dismissal_bowler       34975             0                    

In [222]:

# ============================================================
# DISMISSAL TABLE — LEGACY ID USAGE AUDIT
# READ ONLY — NO DATA MODIFICATIONS
# ============================================================

df = player_dismissal_before_id_migration

print("=" * 75)
print("DISMISSAL TABLE LEGACY ID CHECK")
print("=" * 75)

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

# Check the original source ID column
source_ids = df["player_out_id"].dropna().astype(str)
source_matches = source_ids.isin(legacy_source_ids)

print("\n1. player_out_id vs legacy source IDs")
print("-" * 75)
print("Rows using legacy source IDs:", int(source_matches.sum()))
print("Unique legacy source IDs used:",
      source_ids[source_matches].nunique())

# Check the CricketIQ ID column
cricketiq_ids = df["cricketiq_player_id"].dropna().astype(str)
cricketiq_matches = cricketiq_ids.isin(legacy_cricketiq_ids)

print("\n2. cricketiq_player_id vs legacy CricketIQ IDs")
print("-" * 75)
print("Rows using legacy CricketIQ IDs:", int(cricketiq_matches.sum()))
print("Unique legacy CricketIQ IDs used:",
      cricketiq_ids[cricketiq_matches].nunique())

# Inspect any matches, if present
combined_mask = source_matches | cricketiq_matches

if combined_mask.any():
    print("\nMATCHED LEGACY RECORDS — SAMPLE")
    print("-" * 75)
    print(
        df.loc[
            combined_mask,
            [
                "match_id",
                "player_out",
                "player_out_id",
                "cricketiq_player_id",
                "canonical_player_name"
            ]
        ].head(20).to_string(index=False)
    )
else:
    print("\nNo legacy IDs found in either dismissal ID column.")

print("\nAUDIT COMPLETE — NO DATA MODIFIED")


DISMISSAL TABLE LEGACY ID CHECK
Rows: 37711
Columns: ['match_id', 'innings_number', 'batting_team', 'player_out', 'player_out_id', 'dismissals', 'cricketiq_player_id', 'canonical_player_name']

1. player_out_id vs legacy source IDs
---------------------------------------------------------------------------
Rows using legacy source IDs: 0
Unique legacy source IDs used: 0

2. cricketiq_player_id vs legacy CricketIQ IDs
---------------------------------------------------------------------------
Rows using legacy CricketIQ IDs: 0
Unique legacy CricketIQ IDs used: 0

No legacy IDs found in either dismissal ID column.

AUDIT COMPLETE — NO DATA MODIFIED


In [223]:

# ============================================================
# HISTORICAL ALIAS AUDIT — SOURCE INSPECTION
# READ ONLY — NO DATA MODIFICATIONS
# ============================================================

objects_to_inspect = [
    "player_registry_updated",
    "player_registry_final_updated",
    "old_cs",
    "old_cs_unmatched",
    "old_cr_comparison",
    "player_identity_map_updated"
]

for variable_name in objects_to_inspect:
    print("\n" + "=" * 75)
    print(variable_name.upper())
    print("=" * 75)

    if variable_name not in globals():
        print("Variable not found.")
        continue

    obj = globals()[variable_name]

    print("Type:", type(obj).__name__)

    if isinstance(obj, pd.DataFrame):
        print("Shape:", obj.shape)
        print("Columns:", obj.columns.tolist())
        print("\nSample:")
        print(obj.head(3).to_string(index=False))
    else:
        print("Value:", str(obj)[:500])

print("\nINSPECTION COMPLETE — NO DATA MODIFIED")



PLAYER_REGISTRY_UPDATED
Type: DataFrame
Shape: (3635, 10)
Columns: ['cricketiq_player_id', 'player_id', 'cricsheet_name', 'crickrida_name', 'match_status', 'source', 'player_names', 'player_name', 'canonical_player_name', 'normalized_name']

Sample:
cricketiq_player_id player_id cricsheet_name crickrida_name     match_status source player_names player_name canonical_player_name normalized_name
      CRIQ_8a75e999  8a75e999     Babar Azam     Babar Azam exact_name_match   Both          NaN         NaN            Babar Azam             NaN
      CRIQ_2911de16  2911de16      Hasan Ali      Hasan Ali exact_name_match   Both          NaN         NaN             Hasan Ali             NaN
      CRIQ_9cb8d7a6  9cb8d7a6     Imad Wasim     Imad Wasim exact_name_match   Both          NaN         NaN            Imad Wasim             NaN

PLAYER_REGISTRY_FINAL_UPDATED
Type: DataFrame
Shape: (3620, 10)
Columns: ['cricketiq_player_id', 'player_id', 'cricsheet_name', 'crickrida_name', 'normalized_na

In [224]:

# ============================================================
# HISTORICAL ALIAS CANDIDATE AUDIT
# READ ONLY — NO REGISTRY OR IDENTITY MAP MODIFICATIONS
# ============================================================

import pandas as pd
import numpy as np

old_registry = player_registry_updated.copy()
final_map = player_identity_map_updated.copy()

# ------------------------------------------------------------
# 1. Build current identity lookup
# ------------------------------------------------------------

current_ids = set(
    final_map["cricketiq_player_id"].dropna().astype(str)
)

# Map every available Cricsheet source ID to a current ID.
source_to_current = {}

for _, row in final_map.iterrows():
    current_id = row["cricketiq_player_id"]

    for column in ["player_id", "cricsheet_player_id"]:
        source_id = row.get(column)

        if pd.notna(source_id):
            source_id = str(source_id)

            if source_id in source_to_current:
                if source_to_current[source_id] != current_id:
                    raise ValueError(
                        f"Source ID maps to multiple identities: {source_id}"
                    )

            source_to_current[source_id] = current_id


# ------------------------------------------------------------
# 2. Build normalized Crickrida name lookup
# ------------------------------------------------------------

name_to_current = {}

for _, row in final_map.iterrows():
    name = row.get("crickrida_name")

    if pd.isna(name):
        continue

    normalized = normalize_identity(name)

    if not normalized:
        continue

    current_id = row["cricketiq_player_id"]

    if normalized in name_to_current:
        if name_to_current[normalized] != current_id:
            raise ValueError(
                f"Normalized name maps to multiple identities: {normalized}"
            )

    name_to_current[normalized] = current_id


# ------------------------------------------------------------
# 3. Resolve every old registry record
# ------------------------------------------------------------

mapping_rows = []

for _, row in old_registry.iterrows():

    old_id = str(row["cricketiq_player_id"])
    current_id = None
    mapping_method = None

    # Rule A: ID already exists in final map
    if old_id in current_ids:
        current_id = old_id
        mapping_method = "already_current"

    # Rule B: Old Cricsheet ID — translate using source ID
    elif old_id.startswith("CS_"):
        source_id = old_id[3:]
        current_id = source_to_current.get(source_id)

        if current_id is not None:
            mapping_method = "translated_by_source_id"

    # Rule C: Old Crickrida ID — exact normalized name match
    elif old_id.startswith("CR_"):
        old_name = row.get("player_name")

        if pd.notna(old_name):
            normalized = normalize_identity(old_name)
            current_id = name_to_current.get(normalized)

            if current_id is not None:
                mapping_method = "matched_by_crickrida_name"

    mapping_rows.append({
        "old_cricketiq_player_id": old_id,
        "current_cricketiq_player_id": current_id,
        "mapping_method": mapping_method,
        "old_player_name": row.get("player_name"),
        "old_canonical_name": row.get("canonical_player_name"),
        "old_player_names": row.get("player_names")
    })


old_registry_mapping_audit = pd.DataFrame(mapping_rows)

# ------------------------------------------------------------
# 4. Check mapping coverage and ambiguity
# ------------------------------------------------------------

print("=" * 75)
print("OLD REGISTRY IDENTITY MAPPING SUMMARY")
print("=" * 75)

print("Old registry records:", len(old_registry_mapping_audit))

print("\nMapping methods:")
print(
    old_registry_mapping_audit["mapping_method"]
    .fillna("unresolved")
    .value_counts()
    .to_string()
)

print("\nUnresolved records:",
      old_registry_mapping_audit["current_cricketiq_player_id"].isna().sum())

duplicate_old_ids = old_registry_mapping_audit[
    old_registry_mapping_audit["old_cricketiq_player_id"].duplicated(False)
]

duplicate_current_ids = old_registry_mapping_audit[
    old_registry_mapping_audit["current_cricketiq_player_id"].notna()
    & old_registry_mapping_audit["current_cricketiq_player_id"].duplicated(False)
]

print("\nDuplicate old IDs:", len(duplicate_old_ids))
print("Old records sharing a current identity:", len(duplicate_current_ids))


# ------------------------------------------------------------
# 5. Extract historical name candidates
# ------------------------------------------------------------

def extract_historical_names(value):
    if value is None:
        return []

    if isinstance(value, (list, tuple, set)):
        values = list(value)
    elif isinstance(value, str):
        values = [value]
    else:
        try:
            if pd.isna(value):
                return []
        except (TypeError, ValueError):
            pass
        values = [value]

    result = []

    for item in values:
        if item is None:
            continue

        try:
            if pd.isna(item):
                continue
        except (TypeError, ValueError):
            pass

        name = str(item).strip()

        if name and name.lower() not in {"none", "nan"}:
            result.append(name)

    return result


candidate_rows = []

for _, row in old_registry_mapping_audit.iterrows():

    current_id = row["current_cricketiq_player_id"]

    # Do not extract aliases from unresolved historical identities.
    if pd.isna(current_id):
        continue

    candidates = []

    candidates.extend(
        extract_historical_names(row["old_player_names"])
    )
    candidates.extend(
        extract_historical_names(row["old_player_name"])
    )
    candidates.extend(
        extract_historical_names(row["old_canonical_name"])
    )

    for name in candidates:
        candidate_rows.append({
            "current_cricketiq_player_id": current_id,
            "old_cricketiq_player_id": row["old_cricketiq_player_id"],
            "mapping_method": row["mapping_method"],
            "historical_name": name,
            "normalized_historical_name": normalize_identity(name)
        })


historical_name_candidates = pd.DataFrame(candidate_rows)

# Remove exact duplicate candidate records, not different names.
if not historical_name_candidates.empty:
    historical_name_candidates = (
        historical_name_candidates.drop_duplicates(
            subset=[
                "current_cricketiq_player_id",
                "historical_name"
            ]
        ).reset_index(drop=True)
    )


# ------------------------------------------------------------
# 6. Compare candidates with current known names
# ------------------------------------------------------------

current_known_names = {}

for _, row in final_map.iterrows():

    current_id = row["cricketiq_player_id"]
    known = set()

    for column in [
        "cricsheet_name",
        "crickrida_name",
        "crickrida_player_name",
        "cricsheet_player_name"
    ]:
        value = row.get(column)

        if pd.notna(value):
            name = str(value).strip()

            if name:
                known.add(normalize_identity(name))

    current_known_names[current_id] = known


if not historical_name_candidates.empty:

    historical_name_candidates["is_known_current_name"] = (
        historical_name_candidates.apply(
            lambda r: (
                r["normalized_historical_name"]
                in current_known_names.get(
                    r["current_cricketiq_player_id"], set()
                )
            ),
            axis=1
        )
    )

    historical_name_candidates["alias_candidate"] = (
        ~historical_name_candidates["is_known_current_name"]
    )

    alias_candidates = historical_name_candidates[
        historical_name_candidates["alias_candidate"]
    ].copy()

else:
    alias_candidates = historical_name_candidates.copy()


# ------------------------------------------------------------
# 7. Summary and samples
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("HISTORICAL NAME CANDIDATE SUMMARY")
print("=" * 75)

print("Mapped old registry records:",
      old_registry_mapping_audit["current_cricketiq_player_id"].notna().sum())

print("Unique current identities reached:",
      old_registry_mapping_audit["current_cricketiq_player_id"].nunique())

print("Historical name candidates:",
      len(historical_name_candidates))

if not historical_name_candidates.empty:
    print("\nAlready-known current names:",
          int(historical_name_candidates["is_known_current_name"].sum()))

    print("Potential additional aliases:",
          len(alias_candidates))

    print("\nPotential aliases by mapping method:")
    print(
        alias_candidates["mapping_method"]
        .value_counts()
        .to_string()
    )

    print("\nPotential alias samples:")
    print(
        alias_candidates[
            [
                "current_cricketiq_player_id",
                "old_cricketiq_player_id",
                "mapping_method",
                "historical_name"
            ]
        ].head(40).to_string(index=False)
    )

print("\nAUDIT COMPLETE — NO DATA MODIFIED")


OLD REGISTRY IDENTITY MAPPING SUMMARY
Old registry records: 3635

Mapping methods:
mapping_method
already_current              1775
unresolved                   1279
translated_by_source_id       294
matched_by_crickrida_name     287

Unresolved records: 1279

Duplicate old IDs: 0
Old records sharing a current identity: 572

HISTORICAL NAME CANDIDATE SUMMARY
Mapped old registry records: 2356
Unique current identities reached: 2070
Historical name candidates: 2344

Already-known current names: 2339
Potential additional aliases: 5

Potential aliases by mapping method:
mapping_method
already_current            3
translated_by_source_id    2

Potential alias samples:
current_cricketiq_player_id old_cricketiq_player_id          mapping_method    historical_name
              CRIQ_2af838ee             CS_2af838ee translated_by_source_id          N Pradeep
              CRIQ_64839cb3             CS_64839cb3 translated_by_source_id        M Pathirana
                CS_a9231c3f             CS_

In [230]:

# ============================================================
# FINAL FIVE-CANDIDATE SOURCE-ID CHECK
# READ ONLY
# ============================================================

for _, row in alias_candidates.iterrows():

    old_id = row["old_cricketiq_player_id"]
    current_id = row["current_cricketiq_player_id"]
    historical_name = row["historical_name"]

    old_record = old_registry.loc[
        old_registry["cricketiq_player_id"].astype(str) == str(old_id)
    ]

    current_record = final_map.loc[
        final_map["cricketiq_player_id"].astype(str) == str(current_id)
    ]

    old_source_id = (
        old_record["player_id"].iloc[0]
        if not old_record.empty else None
    )

    current_source_id = (
        current_record["player_id"].iloc[0]
        if not current_record.empty else None
    )

    print({
        "historical_name": historical_name,
        "old_id": old_id,
        "current_id": current_id,
        "old_source_id": old_source_id,
        "current_source_id": current_source_id,
        "source_id_matches": (
            pd.notna(old_source_id)
            and pd.notna(current_source_id)
            and str(old_source_id) == str(current_source_id)
        )
    })

print("\nREAD-ONLY CHECK COMPLETE")


{'historical_name': 'N Pradeep', 'old_id': 'CS_2af838ee', 'current_id': 'CRIQ_2af838ee', 'old_source_id': '2af838ee', 'current_source_id': '2af838ee', 'source_id_matches': True}
{'historical_name': 'M Pathirana', 'old_id': 'CS_64839cb3', 'current_id': 'CRIQ_64839cb3', 'old_source_id': '64839cb3', 'current_source_id': '64839cb3', 'source_id_matches': True}
{'historical_name': 'Steven Ryan Taylor', 'old_id': 'CS_a9231c3f', 'current_id': 'CS_a9231c3f', 'old_source_id': 'a9231c3f', 'current_source_id': 'a9231c3f', 'source_id_matches': True}
{'historical_name': 'Usman Khan', 'old_id': 'CS_bafb9238', 'current_id': 'CS_bafb9238', 'old_source_id': 'bafb9238', 'current_source_id': 'bafb9238', 'source_id_matches': True}
{'historical_name': 'Yousuf Youhana', 'old_id': 'CS_e237b28c', 'current_id': 'CS_e237b28c', 'old_source_id': 'e237b28c', 'current_source_id': 'e237b28c', 'source_id_matches': True}

READ-ONLY CHECK COMPLETE


In [231]:

# ============================================================
# HISTORICAL ALIAS TABLE — CANDIDATE CREATION
# NO CHANGES TO ACTIVE REGISTRY OR IDENTITY MAP
# ============================================================

alias_rows = []

for _, row in alias_candidates.iterrows():

    current_id = row["current_cricketiq_player_id"]
    alias_name = str(row["historical_name"]).strip()

    current_row = final_map.loc[
        final_map["cricketiq_player_id"] == current_id
    ].iloc[0]

    old_row = old_registry.loc[
        old_registry["cricketiq_player_id"]
        == row["old_cricketiq_player_id"]
    ].iloc[0]

    source_id = str(old_row["player_id"])

    # Confirm source ID against current identity
    current_source_id = str(current_row["player_id"])

    if source_id != current_source_id:
        raise ValueError(
            f"Source ID mismatch for {alias_name}: "
            f"{source_id} != {current_source_id}"
        )

    # Evidence classification
    if alias_name in ["N Pradeep", "M Pathirana"]:
        evidence_type = "verified_full_name_variation"
        confidence = "high"
    else:
        evidence_type = "source_observed_alternate_name"
        confidence = "high"

    canonical_name = (
        current_row["crickrida_name"]
        if pd.notna(current_row["crickrida_name"])
        else current_row["cricsheet_name"]
    )

    alias_rows.append({
        "cricketiq_player_id": current_id,
        "canonical_player_name": canonical_name,
        "alias_name": alias_name,
        "normalized_alias": normalize_identity(alias_name),
        "source_player_id": source_id,
        "old_cricketiq_player_id": row["old_cricketiq_player_id"],
        "evidence_type": evidence_type,
        "confidence": confidence,
        "review_status": "verified_for_historical_alias"
    })


historical_alias_table_candidate = pd.DataFrame(alias_rows)

# ------------------------------------------------------------
# 1. Check alias uniqueness
# ------------------------------------------------------------

print("=" * 75)
print("HISTORICAL ALIAS TABLE")
print("=" * 75)

print("Alias rows:", len(historical_alias_table_candidate))
print(
    "Unique current identities:",
    historical_alias_table_candidate["cricketiq_player_id"].nunique()
)

duplicate_aliases = historical_alias_table_candidate[
    historical_alias_table_candidate["normalized_alias"].duplicated(False)
]

print("\nDuplicate normalized aliases:", len(duplicate_aliases))

# ------------------------------------------------------------
# 2. Check collisions against other current player names
# ------------------------------------------------------------

current_name_lookup = []

for _, row in final_map.iterrows():

    for column in [
        "cricsheet_name",
        "crickrida_name",
        "crickrida_player_name",
        "cricsheet_player_name"
    ]:
        value = row.get(column)

        if pd.notna(value):
            current_name_lookup.append({
                "current_name": str(value).strip(),
                "normalized_name": normalize_identity(value),
                "current_identity_id": row["cricketiq_player_id"]
            })

current_names_df = pd.DataFrame(current_name_lookup)

collision_rows = []

for _, alias in historical_alias_table_candidate.iterrows():

    matches = current_names_df[
        current_names_df["normalized_name"]
        == alias["normalized_alias"]
    ]

    for _, match in matches.iterrows():

        if match["current_identity_id"] != alias["cricketiq_player_id"]:
            collision_rows.append({
                "alias_name": alias["alias_name"],
                "alias_identity_id": alias["cricketiq_player_id"],
                "other_identity_id": match["current_identity_id"],
                "other_current_name": match["current_name"]
            })

alias_name_collisions = pd.DataFrame(collision_rows)

print("\nCollisions with other current identities:",
      len(alias_name_collisions))

if not alias_name_collisions.empty:
    print(alias_name_collisions.to_string(index=False))

# ------------------------------------------------------------
# 3. Check source ID consistency
# ------------------------------------------------------------

source_id_valid = (
    historical_alias_table_candidate["source_player_id"].notna()
    & historical_alias_table_candidate["source_player_id"].ne("")
)

print("\nAll source IDs present:", bool(source_id_valid.all()))

print("\nAlias evidence breakdown:")
print(
    historical_alias_table_candidate["evidence_type"]
    .value_counts()
    .to_string()
)

print("\nFINAL ALIAS TABLE CANDIDATE")
print("-" * 75)

print(
    historical_alias_table_candidate[
        [
            "cricketiq_player_id",
            "canonical_player_name",
            "alias_name",
            "source_player_id",
            "evidence_type",
            "confidence"
        ]
    ].to_string(index=False)
)

print("\nAUDIT COMPLETE — NO ACTIVE DATA MODIFIED")


HISTORICAL ALIAS TABLE
Alias rows: 5
Unique current identities: 5

Duplicate normalized aliases: 0

Collisions with other current identities: 0

All source IDs present: True

Alias evidence breakdown:
evidence_type
source_observed_alternate_name    3
verified_full_name_variation      2

FINAL ALIAS TABLE CANDIDATE
---------------------------------------------------------------------------
cricketiq_player_id canonical_player_name         alias_name source_player_id                  evidence_type confidence
      CRIQ_2af838ee         Nuwan Pradeep          N Pradeep         2af838ee   verified_full_name_variation       high
      CRIQ_64839cb3   Matheesha Pathirana        M Pathirana         64839cb3   verified_full_name_variation       high
        CS_a9231c3f             SR Taylor Steven Ryan Taylor         a9231c3f source_observed_alternate_name       high
        CS_bafb9238        Usman Khan (2)         Usman Khan         bafb9238 source_observed_alternate_name       high
        

## 9. Active Player Registry Construction and Promotion

Build a registry candidate from the authoritative identity map, validate its structure and identity references, create backups, and promote the validated candidate while preserving legacy registry variables. Perform post-promotion performance checks.

In [232]:

# ============================================================
# BUILD NEW PLAYER REGISTRY CANDIDATE
# SOURCE OF TRUTH: player_identity_map_updated
# NO EXISTING DATAFRAMES OVERWRITTEN
# ============================================================

import pandas as pd

# 1. Start with a separate copy of the final identity map
player_registry_candidate = player_identity_map_updated.copy()

# 2. Helper: collect available names without duplicates
def collect_player_names(*values):
    names = []

    for value in values:
        if pd.notna(value):
            name = str(value).strip()

            if name and name not in names:
                names.append(name)

    return names if names else None


# 3. Determine source coverage and display fields
source_values = []
player_names_values = []
player_name_values = []
canonical_name_values = []

for _, row in player_registry_candidate.iterrows():

    has_cricsheet = any(
        pd.notna(row.get(column))
        for column in [
            "cricsheet_name",
            "cricsheet_player_name",
            "cricsheet_player_id",
            "player_id"
        ]
    )

    has_crickrida = any(
        pd.notna(row.get(column))
        for column in [
            "crickrida_name",
            "crickrida_player_name"
        ]
    )

    if has_cricsheet and has_crickrida:
        source = "Both"
    elif has_cricsheet:
        source = "Cricsheet"
    elif has_crickrida:
        source = "Crickrida"
    else:
        source = "Unclassified"

    source_values.append(source)

    # Preserve available Cricsheet name variations
    player_names_values.append(
        collect_player_names(
            row.get("cricsheet_name"),
            row.get("cricsheet_player_name")
        )
    )

    # Legacy-compatible Crickrida display name
    player_name = (
        row["crickrida_player_name"]
        if pd.notna(row.get("crickrida_player_name"))
        else row.get("crickrida_name")
    )

    player_name_values.append(player_name)

    # Canonical display-name rule:
    # 1. Crickrida full name
    # 2. Cricsheet name
    # 3. Crickrida player name
    # 4. Cricsheet player name
    canonical_name = None

    for column in [
        "crickrida_name",
        "cricsheet_name",
        "crickrida_player_name",
        "cricsheet_player_name"
    ]:
        value = row.get(column)

        if pd.notna(value) and str(value).strip():
            canonical_name = str(value).strip()
            break

    canonical_name_values.append(canonical_name)


# 4. Add compatibility/display columns
player_registry_candidate["source"] = source_values
player_registry_candidate["player_names"] = player_names_values
player_registry_candidate["player_name"] = player_name_values
player_registry_candidate["canonical_player_name"] = canonical_name_values


# 5. Arrange columns
registry_columns = [
    "cricketiq_player_id",
    "player_id",
    "cricsheet_player_id",
    "cricsheet_name",
    "crickrida_name",
    "match_status",
    "source",
    "player_names",
    "player_name",
    "canonical_player_name",
    "normalized_name",
    "crickrida_player_name",
    "cricsheet_player_name",
    "mapping_method",
    "confidence",
    "notes"
]

player_registry_candidate = player_registry_candidate[
    registry_columns
].copy()


# ============================================================
# INITIAL REGISTRY SUMMARY
# ============================================================

print("=" * 75)
print("NEW PLAYER REGISTRY CANDIDATE")
print("=" * 75)

print("Shape:", player_registry_candidate.shape)
print("Unique CricketIQ IDs:",
      player_registry_candidate["cricketiq_player_id"].nunique())

print("\nSource distribution:")
print(player_registry_candidate["source"].value_counts(dropna=False))

print("\nIdentity status distribution:")
print(
    player_registry_candidate["match_status"]
    .value_counts(dropna=False)
)

print("\nCanonical names missing:",
      player_registry_candidate["canonical_player_name"].isna().sum())

print("\nPlayer IDs missing:",
      player_registry_candidate["player_id"].isna().sum())

print("Cricsheet player IDs missing:",
      player_registry_candidate["cricsheet_player_id"].isna().sum())

print("\nSample records:")
print(
    player_registry_candidate[
        [
            "cricketiq_player_id",
            "cricsheet_name",
            "crickrida_name",
            "canonical_player_name",
            "source",
            "match_status"
        ]
    ].head(10).to_string(index=False)
)

print("\nCANDIDATE CREATED — ORIGINAL DATAFRAMES UNCHANGED")


NEW PLAYER REGISTRY CANDIDATE
Shape: (2070, 16)
Unique CricketIQ IDs: 2070

Source distribution:
source
Cricsheet    1593
Both          415
Crickrida      62
Name: count, dtype: int64

Identity status distribution:
match_status
cricsheet_only                                1593
verified_alias                                 279
exact_name_match                                98
crickrida_only                                  62
verified_manual_identity                        23
verified_full_name_and_career                   10
verified_ireland_identity                        2
verified_match_and_career                        1
verified_cross_country_identity                  1
verified_external_career_and_match_records       1
Name: count, dtype: int64

Canonical names missing: 0

Player IDs missing: 63
Cricsheet player IDs missing: 263

Sample records:
cricketiq_player_id  cricsheet_name  crickrida_name canonical_player_name source     match_status
      CRIQ_8a75e999      Babar Azam

In [234]:

# ============================================================
# FULL PLAYER REGISTRY CANDIDATE VALIDATION — CORRECTED
# READ ONLY — NO DATA MODIFICATIONS
# ============================================================

source_map = player_identity_map_updated
candidate = player_registry_candidate

original_columns = source_map.columns.tolist()

results = {}

# 1. Row count
results["row_count_preserved"] = (
    len(candidate) == len(source_map) == 2070
)

# 2. Unique CricketIQ IDs
results["unique_cricketiq_ids"] = (
    candidate["cricketiq_player_id"].notna().all()
    and candidate["cricketiq_player_id"].is_unique
)

# 3. Exact identity coverage
results["all_identity_ids_preserved"] = (
    set(candidate["cricketiq_player_id"])
    == set(source_map["cricketiq_player_id"])
)

# 4. Compare every original identity-map column
source_indexed = source_map.set_index(
    "cricketiq_player_id",
    drop=False
).sort_index()

candidate_indexed = candidate.set_index(
    "cricketiq_player_id",
    drop=False
).sort_index()

for column in original_columns:
    results[f"original_column_unchanged_{column}"] = (
        source_indexed[column].equals(candidate_indexed[column])
    )

# 5. Canonical names
results["no_missing_canonical_names"] = (
    candidate["canonical_player_name"].notna().all()
    and candidate["canonical_player_name"]
        .astype(str).str.strip().ne("").all()
)

# 6. Canonical name rule validation
def expected_canonical_name(row):
    for column in [
        "crickrida_name",
        "cricsheet_name",
        "crickrida_player_name",
        "cricsheet_player_name"
    ]:
        value = row[column]

        if pd.notna(value) and str(value).strip():
            return str(value).strip()

    return None

expected_names = source_map.apply(
    expected_canonical_name,
    axis=1
)

expected_name_lookup = dict(zip(
    source_map["cricketiq_player_id"],
    expected_names
))

actual_names = candidate_indexed["canonical_player_name"]

results["canonical_names_follow_rule"] = all(
    actual_names.loc[player_id] == expected_name_lookup[player_id]
    for player_id in expected_name_lookup
)

# 7. Historical alias links
alias_ids = set(
    historical_alias_table_candidate["cricketiq_player_id"]
)

results["all_alias_ids_exist"] = alias_ids.issubset(
    set(candidate["cricketiq_player_id"])
)

results["five_alias_records_preserved"] = (
    len(historical_alias_table_candidate) == 5
    and historical_alias_table_candidate[
        "cricketiq_player_id"
    ].nunique() == 5
)

# 8. Source classification
results["source_values_valid"] = (
    set(candidate["source"].dropna().unique())
    <= {"Cricsheet", "Crickrida", "Both"}
)

# 9. Normalized names unchanged
results["normalized_names_unchanged"] = (
    source_indexed["normalized_name"].equals(
        candidate_indexed["normalized_name"]
    )
)

# ============================================================
# RESULTS
# ============================================================

print("=" * 75)
print("PLAYER REGISTRY VALIDATION")
print("=" * 75)

for check, passed in results.items():
    print(f"{'PASS' if passed else 'FAIL'} | {check}")

passed_count = sum(results.values())
total_count = len(results)

print("\n" + "=" * 75)
print("FINAL VALIDATION SUMMARY")
print("=" * 75)

print(f"Passed: {passed_count}/{total_count}")
print("Failed:", total_count - passed_count)

if all(results.values()):
    print("\nALL CHECKS PASSED — REGISTRY CANDIDATE VALIDATED")
else:
    print("\nVALIDATION FAILED — DO NOT PROMOTE THE CANDIDATE")

print("\nOriginal registries and identity map remain unchanged.")


PLAYER REGISTRY VALIDATION
PASS | row_count_preserved
PASS | unique_cricketiq_ids
PASS | all_identity_ids_preserved
PASS | original_column_unchanged_cricketiq_player_id
PASS | original_column_unchanged_player_id
PASS | original_column_unchanged_cricsheet_name
PASS | original_column_unchanged_crickrida_name
PASS | original_column_unchanged_normalized_name
PASS | original_column_unchanged_match_status
PASS | original_column_unchanged_crickrida_player_name
PASS | original_column_unchanged_cricsheet_player_name
PASS | original_column_unchanged_cricsheet_player_id
PASS | original_column_unchanged_mapping_method
PASS | original_column_unchanged_confidence
PASS | original_column_unchanged_notes
PASS | no_missing_canonical_names
PASS | canonical_names_follow_rule
PASS | all_alias_ids_exist
PASS | five_alias_records_preserved
PASS | source_values_valid
PASS | normalized_names_unchanged

FINAL VALIDATION SUMMARY
Passed: 21/21
Failed: 0

ALL CHECKS PASSED — REGISTRY CANDIDATE VALIDATED

Original 

In [235]:

# ============================================================
# PLAYER REGISTRY MIGRATION BACKUP
# SAVE + READ-BACK VALIDATION
# NO ACTIVE REGISTRY REPLACEMENT
# ============================================================

from pathlib import Path
from datetime import datetime
import pandas as pd

# 1. Create a unique backup directory
backup_root = (
    Path(r"C:\CricIQ")
    / "backups"
    / "player_identity_reconciliation"
    / "identity_reconciliation_backups"
)

backup_root.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")

migration_folder = backup_root / f"registry_migration_{timestamp}"
migration_folder.mkdir(parents=True, exist_ok=False)

# 2. Collect all important DataFrames
backup_objects = {
    "player_identity_map_final": player_identity_map_updated,
    "player_registry_candidate": player_registry_candidate,
    "historical_alias_table": historical_alias_table_candidate,
    "old_player_registry_updated": player_registry_updated,
    "old_player_registry_final_updated": player_registry_final_updated,
    "old_player_registry_final": player_registry_final
}

# 3. Save each DataFrame
saved_files = {}

for name, df in backup_objects.items():

    if not isinstance(df, pd.DataFrame):
        raise TypeError(f"{name} is not a DataFrame")

    pkl_path = migration_folder / f"{name}.pkl"
    csv_path = migration_folder / f"{name}.csv"

    df.to_pickle(pkl_path)
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    saved_files[name] = {
        "pkl": pkl_path,
        "csv": csv_path,
        "rows": len(df)
    }

# 4. Read back and validate the saved files
print("=" * 75)
print("BACKUP READ-BACK VALIDATION")
print("=" * 75)

all_passed = True

for name, original in backup_objects.items():

    pkl_path = saved_files[name]["pkl"]
    csv_path = saved_files[name]["csv"]

    restored = pd.read_pickle(pkl_path)
    csv_restored = pd.read_csv(csv_path, low_memory=False)

    pickle_ok = restored.equals(original)
    csv_ok = len(csv_restored) == len(original)

    passed = pickle_ok and csv_ok
    all_passed = all_passed and passed

    print(
        f"{'PASS' if passed else 'FAIL'} | {name} | "
        f"Rows: {len(original)} | "
        f"Pickle: {pickle_ok} | CSV: {csv_ok}"
    )

print("\n" + "=" * 75)
print("BACKUP SUMMARY")
print("=" * 75)

print("Backup directory:", migration_folder)
print("DataFrames saved:", len(saved_files))
print("Total rows saved:", sum(len(df) for df in backup_objects.values()))
print("All read-back checks passed:", all_passed)

if not all_passed:
    raise RuntimeError(
        "Backup validation failed. Do not promote the new registry."
    )

print("\nBACKUP COMPLETE — ACTIVE REGISTRY NOT MODIFIED")


BACKUP READ-BACK VALIDATION
PASS | player_identity_map_final | Rows: 2070 | Pickle: True | CSV: True
PASS | player_registry_candidate | Rows: 2070 | Pickle: True | CSV: True
PASS | historical_alias_table | Rows: 5 | Pickle: True | CSV: True
PASS | old_player_registry_updated | Rows: 3635 | Pickle: True | CSV: True
PASS | old_player_registry_final_updated | Rows: 3620 | Pickle: True | CSV: True
PASS | old_player_registry_final | Rows: 3635 | Pickle: True | CSV: True

BACKUP SUMMARY
Backup directory: C:\CricIQ\notebooks\identity_reconciliation_backups\registry_migration_20261116_235903_966751
DataFrames saved: 6
Total rows saved: 15035
All read-back checks passed: True

BACKUP COMPLETE — ACTIVE REGISTRY NOT MODIFIED


In [236]:

# ============================================================
# PROMOTE VALIDATED PLAYER REGISTRY
# PRESERVE ALL LEGACY REGISTRY VARIABLES
# ============================================================

# 1. Confirm backup checkpoint exists
from pathlib import Path

migration_folder = (
    Path(r"C:\CricIQ")
    / "backups"
    / "player_identity_reconciliation"
    / "identity_reconciliation_backups"
    / "registry_migration_20261116_235903_966751"
)

required_backups = [
    "player_identity_map_final.pkl",
    "player_registry_candidate.pkl",
    "historical_alias_table.pkl",
    "old_player_registry_updated.pkl",
    "old_player_registry_final_updated.pkl",
    "old_player_registry_final.pkl"
]

missing_backups = [
    name for name in required_backups
    if not (migration_folder / name).exists()
]

if missing_backups:
    raise FileNotFoundError(
        f"Missing backup files: {missing_backups}"
    )

# 2. Confirm candidate still matches authoritative map
if len(player_registry_candidate) != len(player_identity_map_updated):
    raise ValueError("Candidate and identity map row counts differ.")

if not player_registry_candidate["cricketiq_player_id"].is_unique:
    raise ValueError("Candidate contains duplicate CricketIQ IDs.")

if set(player_registry_candidate["cricketiq_player_id"]) != set(
    player_identity_map_updated["cricketiq_player_id"]
):
    raise ValueError("Candidate identity coverage has changed.")

# 3. Promote using a separate active variable
player_registry_active = player_registry_candidate.copy(deep=True)

# 4. Keep aliases as a separate lookup table
player_aliases_active = historical_alias_table_candidate.copy(deep=True)

# 5. Final promotion checks
assert len(player_registry_active) == 2070
assert player_registry_active["cricketiq_player_id"].is_unique
assert len(player_aliases_active) == 5

assert set(player_aliases_active["cricketiq_player_id"]).issubset(
    set(player_registry_active["cricketiq_player_id"])
)

print("=" * 75)
print("PLAYER REGISTRY PROMOTION")
print("=" * 75)

print("Active registry rows:", len(player_registry_active))
print("Unique player IDs:",
      player_registry_active["cricketiq_player_id"].nunique())
print("Historical aliases:", len(player_aliases_active))

print("\nLegacy variables preserved:")
print("player_registry_updated:", player_registry_updated.shape)
print("player_registry_final_updated:", player_registry_final_updated.shape)
print("player_registry_final:", player_registry_final.shape)

print("\nActive registry created successfully.")
print("No legacy registry variable was overwritten.")
print("Identity map unchanged.")
print("Performance tables unchanged.")


PLAYER REGISTRY PROMOTION
Active registry rows: 2070
Unique player IDs: 2070
Historical aliases: 5

Legacy variables preserved:
player_registry_updated: (3635, 10)
player_registry_final_updated: (3620, 10)
player_registry_final: (3635, 9)

Active registry created successfully.
No legacy registry variable was overwritten.
Identity map unchanged.
Performance tables unchanged.


In [237]:

# ============================================================
# POST-PROMOTION PERFORMANCE RECONCILIATION
# READ ONLY
# ============================================================

performance_tables = {
    "batting": player_batting_canonical,
    "bowling": player_bowling_canonical,
    "dismissal": player_dismissal_canonical,
    "dismissal_bowler": dismissal_bowler_canonical
}

active_ids = set(
    player_registry_active["cricketiq_player_id"].dropna().astype(str)
)

performance_results = []

for table_name, df in performance_tables.items():

    if "cricketiq_player_id" not in df.columns:
        raise KeyError(
            f"{table_name} has no cricketiq_player_id column."
        )

    table_ids = df["cricketiq_player_id"].astype("string")

    missing_id_rows = int(table_ids.isna().sum())

    unknown_mask = (
        table_ids.notna()
        & ~table_ids.isin(active_ids)
    )

    unknown_rows = int(unknown_mask.sum())

    performance_results.append({
        "table": table_name,
        "rows": len(df),
        "unique_players": table_ids.nunique(),
        "missing_ids": missing_id_rows,
        "unknown_ids": unknown_rows,
        "status": (
            "PASS"
            if missing_id_rows == 0 and unknown_rows == 0
            else "FAIL"
        )
    })

performance_reconciliation = pd.DataFrame(performance_results)

print("=" * 75)
print("POST-PROMOTION PERFORMANCE RECONCILIATION")
print("=" * 75)

print(performance_reconciliation.to_string(index=False))

print("\nTotal performance records:",
      performance_reconciliation["rows"].sum())

print("Total missing IDs:",
      performance_reconciliation["missing_ids"].sum())

print("Total unknown IDs:",
      performance_reconciliation["unknown_ids"].sum())

if (performance_reconciliation["status"] == "PASS").all():
    print("\nALL PERFORMANCE RECORDS LINK TO ACTIVE PLAYERS.")
else:
    print("\nFAILURE — INVESTIGATE BEFORE USING THE ACTIVE REGISTRY.")


POST-PROMOTION PERFORMANCE RECONCILIATION
           table  rows  unique_players  missing_ids  unknown_ids status
         batting 45432            1934            0            0   PASS
         bowling 31186            1502            0            0   PASS
       dismissal 37711            1841            0            0   PASS
dismissal_bowler 34975            1281            0            0   PASS

Total performance records: 149304
Total missing IDs: 0
Total unknown IDs: 0

ALL PERFORMANCE RECORDS LINK TO ACTIVE PLAYERS.


## 10. Dataset Publication and Notebook Organization

Prepare versioned player datasets for publication, verify saved files through read-back and integrity checks, and organize supporting backups and notebook-related files. Preserve prior published versions and source backups.

In [238]:

from pathlib import Path
import pandas as pd

ROOT = Path(r"C:\CricIQ")

inventory = []

for path in ROOT.rglob("*"):

    # Ignore virtual environments and Git internals
    if any(
        part in {
            "venv", ".venv", "__pycache__", ".git",
            "node_modules"
        }
        for part in path.parts
    ):
        continue

    if path.is_file():

        inventory.append({
            "relative_path": str(path.relative_to(ROOT)),
            "folder": str(path.parent.relative_to(ROOT)),
            "extension": path.suffix.lower(),
            "file_name": path.name,
            "size_mb": round(path.stat().st_size / (1024 ** 2), 3)
        })

inventory_df = pd.DataFrame(inventory)

print("=" * 75)
print("CRICIQ PROJECT FILE INVENTORY")
print("=" * 75)

print("Total files:", len(inventory_df))

print("\nFile types:")
print(
    inventory_df["extension"]
    .value_counts()
    .head(25)
)

print("\nLargest files:")
print(
    inventory_df.sort_values(
        "size_mb", ascending=False
    ).head(30).to_string(index=False)
)

print("\nProcessed data folder:")
processed = inventory_df[
    inventory_df["relative_path"].str.startswith(
        "data\\processed"
    )
]

print(processed.to_string(index=False))

# Save inventory report outside the project
report_path = Path(
    r"C:\CricIQ\project_inventory.csv"
)

inventory_df.to_csv(report_path, index=False)

print("\nInventory saved to:", report_path)
print("READ-ONLY AUDIT — NO PROJECT FILES MODIFIED")


CRICIQ PROJECT FILE INVENTORY
Total files: 70

File types:
extension
.csv      49
.pkl       6
.ipynb     6
.pdf       4
.zip       2
.gz        2
.txt       1
Name: count, dtype: int64

Largest files:
                                                                                                            relative_path                                                                              folder extension                                        file_name  size_mb
                                                                data\processed\cricsheet_odi\cricsheet_odi_deliveries.csv                                                        data\processed\cricsheet_odi      .csv                     cricsheet_odi_deliveries.csv   95.513
                                                                   notebooks\notebook06_checkpoint_20261116_024853.pkl.gz                                                                           notebooks       .gz     notebook06_checkpoint_20261116

In [239]:

# ============================================================
# CRICIQ PLAYER DATA — VERSIONED DATASET PUBLICATION
# Destination: data/processed/player_data
# Existing files will NOT be overwritten.
# ============================================================

from pathlib import Path
from datetime import datetime
import pandas as pd
import hashlib

# ------------------------------------------------------------
# 1. Define destination
# ------------------------------------------------------------

processed_root = Path(r"C:\CricIQ\data\processed")

publication_folder = (
    processed_root
    / "player_data"
    / "identity_migration_20261117"
)

if publication_folder.exists():
    raise FileExistsError(
        f"Publication folder already exists: {publication_folder}\n"
        "Do not overwrite it. Choose a new versioned folder."
    )

# ------------------------------------------------------------
# 2. Collect validated DataFrames
# ------------------------------------------------------------

datasets = {
    "cricketiq_player_identity_registry": player_registry_active,
    "cricketiq_player_aliases": player_aliases_active,
    "cricketiq_player_identity_map": player_identity_map_updated,
    "cricketiq_player_batting_canonical": player_batting_canonical,
    "cricketiq_player_bowling_canonical": player_bowling_canonical,
    "cricketiq_player_dismissals_canonical": player_dismissal_canonical,
    "cricketiq_dismissal_bowler_canonical": dismissal_bowler_canonical
}

# ------------------------------------------------------------
# 3. Pre-publication checks
# ------------------------------------------------------------

expected_rows = {
    "cricketiq_player_identity_registry": 2070,
    "cricketiq_player_aliases": 5,
    "cricketiq_player_identity_map": 2070,
    "cricketiq_player_batting_canonical": 45432,
    "cricketiq_player_bowling_canonical": 31186,
    "cricketiq_player_dismissals_canonical": 37711,
    "cricketiq_dismissal_bowler_canonical": 34975
}

active_ids = set(
    player_registry_active["cricketiq_player_id"]
    .dropna().astype(str)
)

for name, df in datasets.items():

    if not isinstance(df, pd.DataFrame):
        raise TypeError(f"{name} is not a DataFrame")

    if len(df) != expected_rows[name]:
        raise ValueError(
            f"{name}: expected {expected_rows[name]} rows, "
            f"found {len(df)}"
        )

# Performance tables must have valid player references.
performance_names = [
    "cricketiq_player_batting_canonical",
    "cricketiq_player_bowling_canonical",
    "cricketiq_player_dismissals_canonical",
    "cricketiq_dismissal_bowler_canonical"
]

for name in performance_names:

    df = datasets[name]

    if "cricketiq_player_id" not in df.columns:
        raise KeyError(f"{name} has no CricketIQ player ID column.")

    ids = df["cricketiq_player_id"].dropna().astype(str)

    if len(ids) != len(df):
        raise ValueError(f"{name} contains missing player IDs.")

    if not set(ids).issubset(active_ids):
        raise ValueError(f"{name} contains unknown player IDs.")

# ------------------------------------------------------------
# 4. Create publication folder and save CSVs
# ------------------------------------------------------------

publication_folder.mkdir(parents=True, exist_ok=False)

manifest_rows = []

for name, df in datasets.items():

    file_path = publication_folder / f"{name}.csv"

    df.to_csv(
        file_path,
        index=False,
        encoding="utf-8-sig"
    )

    # Calculate file checksum
    sha256 = hashlib.sha256(
        file_path.read_bytes()
    ).hexdigest()

    manifest_rows.append({
        "dataset": name,
        "file_name": file_path.name,
        "rows": len(df),
        "columns": len(df.columns),
        "size_mb": round(file_path.stat().st_size / (1024 ** 2), 3),
        "sha256": sha256
    })

# ------------------------------------------------------------
# 5. Read back and validate every CSV
# ------------------------------------------------------------

print("=" * 80)
print("CRICIQ PLAYER DATA PUBLICATION VALIDATION")
print("=" * 80)

validation_results = []

for name, original in datasets.items():

    file_path = publication_folder / f"{name}.csv"

    # Preserve player IDs as strings when reading back.
    restored = pd.read_csv(
        file_path,
        low_memory=False,
        dtype={"cricketiq_player_id": "string"}
    )

    row_check = len(restored) == len(original)
    column_check = restored.columns.tolist() == original.columns.tolist()

    if name in performance_names:
        restored_ids = set(
            restored["cricketiq_player_id"].dropna().astype(str)
        )
        identity_check = restored_ids.issubset(active_ids)
    elif name == "cricketiq_player_aliases":
        identity_check = set(
            restored["cricketiq_player_id"].dropna().astype(str)
        ).issubset(active_ids)
    else:
        identity_check = (
            restored["cricketiq_player_id"].nunique() == len(original)
        )

    passed = row_check and column_check and identity_check

    validation_results.append({
        "dataset": name,
        "rows": len(restored),
        "row_count": row_check,
        "columns": column_check,
        "identity_check": identity_check,
        "status": "PASS" if passed else "FAIL"
    })

    print(
        f"{'PASS' if passed else 'FAIL'} | {name} | "
        f"Rows: {len(restored)}"
    )

# ------------------------------------------------------------
# 6. Save manifest
# ------------------------------------------------------------

manifest = pd.DataFrame(manifest_rows)

manifest_path = publication_folder / "publication_manifest.csv"
manifest.to_csv(manifest_path, index=False, encoding="utf-8-sig")

validation_df = pd.DataFrame(validation_results)

validation_path = publication_folder / "validation_report.csv"
validation_df.to_csv(validation_path, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 7. Final result
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PUBLICATION SUMMARY")
print("=" * 80)

print("Destination:", publication_folder)
print("Datasets published:", len(datasets))
print("Total records:", sum(expected_rows.values()))

print("\nValidation summary:")
print(validation_df["status"].value_counts().to_string())

if (validation_df["status"] == "PASS").all():
    print("\nALL PUBLICATION CHECKS PASSED.")
    print("Existing processed datasets were not overwritten.")
else:
    print("\nPUBLICATION VALIDATION FAILED.")
    print("Do not designate these files as official.")

print("\nREAD-BACK VALIDATION COMPLETE")


CRICIQ PLAYER DATA PUBLICATION VALIDATION
PASS | cricketiq_player_identity_registry | Rows: 2070
PASS | cricketiq_player_aliases | Rows: 5
PASS | cricketiq_player_identity_map | Rows: 2070
PASS | cricketiq_player_batting_canonical | Rows: 45432
PASS | cricketiq_player_bowling_canonical | Rows: 31186
PASS | cricketiq_player_dismissals_canonical | Rows: 37711
PASS | cricketiq_dismissal_bowler_canonical | Rows: 34975

PUBLICATION SUMMARY
Destination: C:\CricIQ\data\processed\player_data\identity_migration_20261117
Datasets published: 7
Total records: 153449

Validation summary:
status
PASS    7

ALL PUBLICATION CHECKS PASSED.
Existing processed datasets were not overwritten.

READ-BACK VALIDATION COMPLETE


In [240]:

from pathlib import Path
import pandas as pd

old_folder = Path(r"C:\CricIQ\data\processed\player_data")

new_folder = (
    old_folder / "identity_migration_20261117"
)

old_files = {
    "identity_registry": "cricketiq_player_identity_registry.csv",
    "batting": "cricsheet_player_batting.csv",
    "bowling": "cricsheet_player_bowling.csv",
    "dismissals": "cricsheet_player_dismissals.csv",
    "dismissal_bowler": "cricsheet_bowler_wickets.csv"
}

new_files = {
    "identity_registry": "cricketiq_player_identity_registry.csv",
    "batting": "cricketiq_player_batting_canonical.csv",
    "bowling": "cricketiq_player_bowling_canonical.csv",
    "dismissals": "cricketiq_player_dismissals_canonical.csv",
    "dismissal_bowler": "cricketiq_dismissal_bowler_canonical.csv"
}

print("=" * 75)
print("OLD VS NEW PLAYER DATASET INVENTORY")
print("=" * 75)

for dataset in old_files:

    old_path = old_folder / old_files[dataset]
    new_path = new_folder / new_files[dataset]

    print(f"\nDATASET: {dataset}")

    if old_path.exists():
        old_df = pd.read_csv(old_path, low_memory=False)
        print("Old file rows:", len(old_df))
        print("Old columns:", len(old_df.columns))
    else:
        print("Old file: NOT FOUND")

    if new_path.exists():
        new_df = pd.read_csv(new_path, low_memory=False)
        print("New file rows:", len(new_df))
        print("New columns:", len(new_df.columns))
    else:
        print("New file: NOT FOUND")

print("\nInventory comparison complete.")


OLD VS NEW PLAYER DATASET INVENTORY

DATASET: identity_registry
Old file rows: 3620
Old columns: 10
New file rows: 2070
New columns: 16

DATASET: batting
Old file rows: 45432
Old columns: 12
New file rows: 45432
New columns: 12

DATASET: bowling
Old file rows: 31186
Old columns: 13
New file rows: 31186
New columns: 13

DATASET: dismissals
Old file rows: 37711
Old columns: 8
New file rows: 37711
New columns: 8

DATASET: dismissal_bowler
Old file rows: 34975
Old columns: 8
New file rows: 34975
New columns: 8

Inventory comparison complete.


In [241]:

from pathlib import Path
import json
import pandas as pd

# ------------------------------------------------
# 1. Notebook directory
# ------------------------------------------------

notebooks_root = Path(r"C:\CricIQ\notebooks")

if not notebooks_root.exists():
    raise FileNotFoundError(notebooks_root)

inventory = []

# ------------------------------------------------
# 2. Inspect all files recursively
# ------------------------------------------------

for file_path in sorted(notebooks_root.rglob("*")):

    if not file_path.is_file():
        continue

    relative_path = file_path.relative_to(notebooks_root)

    record = {
        "file": file_path.name,
        "relative_path": str(relative_path),
        "extension": file_path.suffix.lower(),
        "size_mb": round(file_path.stat().st_size / (1024 ** 2), 3),
        "notebook_cells": None,
        "markdown_headings": None,
        "notebook_metadata": None,
        "status": "File"
    }

    # --------------------------------------------
    # 3. Inspect Jupyter notebooks
    # --------------------------------------------

    if file_path.suffix.lower() == ".ipynb":

        try:
            with open(file_path, "r", encoding="utf-8") as f:
                notebook = json.load(f)

            cells = notebook.get("cells", [])

            headings = []

            for cell in cells:
                if cell.get("cell_type") == "markdown":

                    source = cell.get("source", [])

                    if isinstance(source, list):
                        source = "".join(source)

                    for line in source.splitlines():
                        if line.strip().startswith("#"):
                            headings.append(line.strip())

            record["notebook_cells"] = len(cells)
            record["markdown_headings"] = " | ".join(headings[:12])
            record["notebook_metadata"] = notebook.get(
                "metadata", {}
            ).get("kernelspec", {}).get("name", "Unknown")

            record["status"] = "Notebook"

        except Exception as e:
            record["status"] = f"Notebook read error: {e}"

    # --------------------------------------------
    # 4. Identify checkpoint and backup files
    # --------------------------------------------

    if any(word in file_path.name.lower()
           for word in ["checkpoint", "backup", "old", "archive"]):

        record["status"] += " | Possible backup/checkpoint"

    inventory.append(record)

# ------------------------------------------------
# 5. Display inventory
# ------------------------------------------------

notebook_inventory = pd.DataFrame(inventory)

print("=" * 90)
print("CRICIQ NOTEBOOKS FOLDER — READ-ONLY INVENTORY")
print("=" * 90)

print("Total files:", len(notebook_inventory))
print("\nFile types:")
print(notebook_inventory["extension"].value_counts().to_string())

print("\nComplete inventory:")
display(
    notebook_inventory.sort_values(
        ["extension", "relative_path"]
    ).reset_index(drop=True)
)

# ------------------------------------------------
# 6. Show only notebook details
# ------------------------------------------------

print("\n" + "=" * 90)
print("NOTEBOOK CONTENT OVERVIEW")
print("=" * 90)

display(
    notebook_inventory[
        notebook_inventory["extension"] == ".ipynb"
    ][
        [
            "relative_path",
            "size_mb",
            "notebook_cells",
            "markdown_headings",
            "notebook_metadata"
        ]
    ]
)

print("\nInspection complete. No files were moved or modified.")


CRICIQ NOTEBOOKS FOLDER — READ-ONLY INVENTORY
Total files: 32

File types:
extension
.csv      18
.ipynb     6
.pkl       6
.gz        2

Complete inventory:


,file,relative_path,extension,size_mb,notebook_cells,markdown_headings,notebook_metadata,status
0,cricsheet_only_staged_20261116_231047.csv,identity_reconciliation_backups\cricsheet_only...,.csv,0.461,NaN,None,None,File
1,identity_map_backup_20261116_231047.csv,identity_reconciliation_backups\identity_map_b...,.csv,0.099,NaN,None,None,File | Possible backup/checkpoint
2,identity_map_combined_review_20261116_231047.csv,identity_reconciliation_backups\identity_map_c...,.csv,0.560,NaN,None,None,File
3,player_identity_map_final_20261116_231942.csv,identity_reconciliation_backups\player_identit...,.csv,0.560,NaN,None,None,File
4,historical_alias_table.csv,identity_reconciliation_backups\registry_migra...,.csv,0.001,NaN,None,None,File
5,old_player_registry_final.csv,identity_reconciliation_backups\registry_migra...,.csv,0.256,NaN,None,None,File | Possible backup/checkpoint
6,old_player_registry_final_updated.csv,identity_reconciliation_backups\registry_migra...,.csv,0.265,NaN,None,None,File | Possible backup/checkpoint
7,old_player_registry_updated.csv,identity_reconciliation_backups\registry_migra...,.csv,0.260,NaN,None,None,File | Possible backup/checkpoint
8,player_identity_map_final.csv,identity_reconciliation_backups\registry_migra...,.csv,0.560,NaN,None,None,File
9,player_registry_candidate.csv,identity_reconciliation_backups\registry_migra...,.csv,0.639,NaN,None,None,File



NOTEBOOK CONTENT OVERVIEW


,relative_path,size_mb,notebook_cells,markdown_headings,notebook_metadata
0,01_data_exploration.ipynb,0.134,60.0,,python3
1,02_afghanistan_match_collection.ipynb,2.170,492.0,,python3
2,03_cricsheet_odi_processing.ipynb,0.190,42.0,# Notebook 03: Cricsheet ODI Data Processing |...,python3
3,04_dataset_integration.ipynb,0.114,37.0,,python3
4,05_feature_engineering.ipynb,0.382,58.0,,python3
5,06_player_data_processing.ipynb,5.239,506.0,,python3



Inspection complete. No files were moved or modified.


In [242]:

from pathlib import Path
import json
import re

root = Path(r"C:\CricIQ")
notebooks_folder = root / "notebooks"

print("=" * 85)
print("NOTEBOOK PATH DEPENDENCY AUDIT")
print("=" * 85)

patterns = [
    r"read_csv",
    r"to_csv",
    r"read_pickle",
    r"to_pickle",
    r"open\(",
    r"Path\(",
    r"checkpoint",
    r"identity_reconciliation_backups",
    r"player_data",
    r"\.\./",
    r"\.\.\\"
]

for notebook_path in sorted(notebooks_folder.glob("*.ipynb")):

    with open(notebook_path, "r", encoding="utf-8") as f:
        notebook = json.load(f)

    matches = []

    for cell_number, cell in enumerate(notebook.get("cells", [])):

        source = cell.get("source", [])
        source = "".join(source) if isinstance(source, list) else source

        for line_number, line in enumerate(source.splitlines(), start=1):

            if any(re.search(pattern, line, re.IGNORECASE)
                   for pattern in patterns):

                matches.append(
                    (cell_number, line_number, line.strip()[:220])
                )

    print(f"\nNOTEBOOK: {notebook_path.name}")

    if matches:
        for cell_number, line_number, line in matches:
            print(
                f"Cell {cell_number}, Line {line_number}: {line}"
            )
    else:
        print("No matching path references found.")

print("\nAudit complete. No files were modified.")


NOTEBOOK PATH DEPENDENCY AUDIT

NOTEBOOK: 01_data_exploration.ipynb
Cell 2, Line 1: PROJECT_ROOT = Path(r"C:\CricIQ")
Cell 4, Line 5: with archive.open(sample_file) as file:
Cell 13, Line 7: with archive.open(file_name) as file:
Cell 23, Line 7: with archive.open(file_name) as file:
Cell 52, Line 4: REGISTER_DIR = Path(r"C:\CricIQ\data\raw\afghanistan_odi\match_register")
Cell 52, Line 32: register_df.to_csv(register_path, index=False)
Cell 55, Line 1: register_df = pd.read_csv(register_path, parse_dates=["date"])
Cell 55, Line 10: register_df.to_csv(register_path, index=False)
Cell 57, Line 3: project_path = Path(r"C:\CricIQ")
Cell 58, Line 7: with zip_file.open(filename) as file:

NOTEBOOK: 02_afghanistan_match_collection.ipynb
Cell 1, Line 4: REGISTER_DIR = Path(r"C:\CricIQ\data\raw\afghanistan_odi\match_register")
Cell 1, Line 32: register_df.to_csv(register_path, index=False)
Cell 4, Line 1: register_df = pd.read_csv(register_path, parse_dates=["date"])
Cell 4, Line 10: register_d

In [243]:

import json
from pathlib import Path

notebook_path = Path(
    r"C:\CricIQ\notebooks\06_player_data_processing.ipynb"
)

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

target_cells = [206, 207, 289, 290, 467, 473, 499, 500]

for index in target_cells:

    if index >= len(nb["cells"]):
        continue

    cell = nb["cells"][index]

    source = cell.get("source", [])
    source = "".join(source) if isinstance(source, list) else source

    print("\n" + "=" * 80)
    print(f"CELL {index} | {cell.get('cell_type')}")
    print("=" * 80)
    print(source)

print("\nInspection only. No notebook changes made.")



CELL 206 | code
import pickle
import gzip
from pathlib import Path
from datetime import datetime
import types

# Create a separate checkpoint folder.
checkpoint_dir = (
    Path.cwd()
    / "data"
    / "processed"
    / "player_data"
    / "checkpoints"
)
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Create a timestamped checkpoint filename.
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
checkpoint_path = checkpoint_dir / f"notebook06_checkpoint_{timestamp}.pkl.gz"

# Collect serializable notebook variables.
checkpoint_data = {}
skipped_variables = []

excluded_names = {
    "In", "Out", "get_ipython", "exit", "quit"
}

for name, value in list(globals().items()):
    if (
        name.startswith("_")
        or name in excluded_names
        or any(word in name.lower() for word in
               ["password", "secret", "token", "api_key", "apikey"])
        or isinstance(value, (types.ModuleType, types.FunctionType, type))
    ):
        continue

    try:
        pickle.

In [244]:
PROJECT_ROOT = Path(r"C:\CricIQ")

IDENTITY_BACKUP_DIR = (
    PROJECT_ROOT / "backups" / "player_identity_reconciliation"
)

NOTEBOOK_CHECKPOINT_DIR = (
    PROJECT_ROOT / "backups" / "notebook_checkpoints"
)

In [245]:

from pathlib import Path
import pandas as pd

ROOT = Path(r"C:\CricIQ")

paths_to_check = [
    ROOT / "notebooks" / "identity_reconciliation_backups",
    ROOT / "notebooks" / "data" / "processed" / "player_data" / "checkpoints",
    ROOT / "notebooks" / "notebook06_checkpoint_20261116_024853.pkl.gz",
    ROOT / "backups"
]

print("=" * 80)
print("FINAL NOTEBOOK CLEANUP SAFETY CHECK")
print("=" * 80)

for path in paths_to_check:

    print(f"\nPATH: {path}")
    print("Exists:", path.exists())

    if path.is_dir():

        files = [p for p in path.rglob("*") if p.is_file()]

        print("Files:", len(files))

        for file in files[:20]:
            print(
                " -",
                file.relative_to(ROOT),
                f"({file.stat().st_size / (1024**2):.2f} MB)"
            )

    elif path.is_file():

        print(
            "Size:",
            round(path.stat().st_size / (1024**2), 2),
            "MB"
        )

print("\nSafety check completed. No files modified.")


FINAL NOTEBOOK CLEANUP SAFETY CHECK

PATH: C:\CricIQ\notebooks\identity_reconciliation_backups
Exists: True
Files: 16
 - notebooks\identity_reconciliation_backups\cricsheet_only_staged_20261116_231047.csv (0.46 MB)
 - notebooks\identity_reconciliation_backups\identity_map_backup_20261116_231047.csv (0.10 MB)
 - notebooks\identity_reconciliation_backups\identity_map_combined_review_20261116_231047.csv (0.56 MB)
 - notebooks\identity_reconciliation_backups\player_identity_map_final_20261116_231942.csv (0.56 MB)
 - notebooks\identity_reconciliation_backups\registry_migration_20261116_235903_966751\historical_alias_table.csv (0.00 MB)
 - notebooks\identity_reconciliation_backups\registry_migration_20261116_235903_966751\historical_alias_table.pkl (0.00 MB)
 - notebooks\identity_reconciliation_backups\registry_migration_20261116_235903_966751\old_player_registry_final.csv (0.26 MB)
 - notebooks\identity_reconciliation_backups\registry_migration_20261116_235903_966751\old_player_registry_fin

In [246]:

from pathlib import Path
import shutil
import hashlib
import pandas as pd

ROOT = Path(r"C:\CricIQ")
NOTEBOOKS = ROOT / "notebooks"
BACKUPS = ROOT / "backups"

IDENTITY_DEST = BACKUPS / "player_identity_reconciliation"
CHECKPOINT_DEST = BACKUPS / "notebook_checkpoints"

# --------------------------------------------------
# 1. Identify source files
# --------------------------------------------------

identity_source = NOTEBOOKS / "identity_reconciliation_backups"

root_csv_files = sorted(NOTEBOOKS.glob("*.csv"))

checkpoint_sources = [
    NOTEBOOKS / "notebook06_checkpoint_20261116_024853.pkl.gz",
    (
        NOTEBOOKS
        / "data"
        / "processed"
        / "player_data"
        / "checkpoints"
        / "notebook06_checkpoint_20261115_004228.pkl.gz"
    )
]

if not identity_source.is_dir():
    raise FileNotFoundError(identity_source)

if BACKUPS.exists():
    raise FileExistsError(
        f"{BACKUPS} already exists. Stop and inspect it before proceeding."
    )

if len(root_csv_files) != 8:
    raise RuntimeError(
        f"Expected 8 root-level CSV files, found {len(root_csv_files)}. "
        "Inspect the files before copying."
    )

for path in checkpoint_sources:
    if not path.is_file():
        raise FileNotFoundError(path)

# --------------------------------------------------
# 2. Prepare destination folders
# --------------------------------------------------

BACKUPS.mkdir(parents=True, exist_ok=False)
IDENTITY_DEST.mkdir(parents=True)
CHECKPOINT_DEST.mkdir(parents=True)

# Keep the existing identity backup folder intact.
identity_folder_dest = IDENTITY_DEST / "identity_reconciliation_backups"

shutil.copytree(identity_source, identity_folder_dest)

# Copy root-level identity staging CSVs.
for source in root_csv_files:

    destination = IDENTITY_DEST / source.name

    if destination.exists():
        raise FileExistsError(destination)

    shutil.copy2(source, destination)

# Copy both checkpoint files.
for source in checkpoint_sources:

    destination = CHECKPOINT_DEST / source.name

    if destination.exists():
        raise FileExistsError(destination)

    shutil.copy2(source, destination)

# --------------------------------------------------
# 3. Build source-to-destination file pairs
# --------------------------------------------------

file_pairs = []

for source in identity_source.rglob("*"):
    if source.is_file():
        destination = (
            identity_folder_dest / source.relative_to(identity_source)
        )
        file_pairs.append((source, destination))

for source in root_csv_files:
    file_pairs.append((source, IDENTITY_DEST / source.name))

for source in checkpoint_sources:
    file_pairs.append((source, CHECKPOINT_DEST / source.name))

# --------------------------------------------------
# 4. Verify every copied file using SHA-256
# --------------------------------------------------

def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


verification = []

for source, destination in file_pairs:

    source_hash = sha256_file(source)
    destination_hash = sha256_file(destination)

    verification.append({
        "source": str(source.relative_to(ROOT)),
        "destination": str(destination.relative_to(ROOT)),
        "source_size": source.stat().st_size,
        "destination_size": destination.stat().st_size,
        "hash_match": source_hash == destination_hash,
        "status": (
            "PASS" if source_hash == destination_hash else "FAIL"
        )
    })

verification_df = pd.DataFrame(verification)

# --------------------------------------------------
# 5. Save migration manifest
# --------------------------------------------------

manifest_path = BACKUPS / "notebook_cleanup_copy_manifest.csv"

verification_df.to_csv(
    manifest_path,
    index=False,
    encoding="utf-8-sig"
)

# --------------------------------------------------
# 6. Final report
# --------------------------------------------------

print("=" * 80)
print("CRICIQ NOTEBOOK SUPPORT-FILE COPY VALIDATION")
print("=" * 80)

print("Files copied:", len(file_pairs))
print("Identity backup files:", len(list(identity_folder_dest.rglob("*"))))
print("Root-level CSV files:", len(root_csv_files))
print("Checkpoint files:", len(checkpoint_sources))

print("\nHash verification:")
print(verification_df["status"].value_counts().to_string())

display(verification_df)

if not verification_df["hash_match"].all():
    raise RuntimeError(
        "One or more file hashes do not match. "
        "Do not delete or move any original files."
    )

print("\nALL COPIED FILES VERIFIED SUCCESSFULLY.")
print("Original files remain untouched.")
print("Manifest:", manifest_path)


CRICIQ NOTEBOOK SUPPORT-FILE COPY VALIDATION
Files copied: 26
Identity backup files: 17
Root-level CSV files: 8
Checkpoint files: 2

Hash verification:
status
PASS    26


,source,destination,source_size,destination_size,hash_match,status
0,notebooks\identity_reconciliation_backups\cric...,backups\player_identity_reconciliation\identit...,483532,483532,True,PASS
1,notebooks\identity_reconciliation_backups\iden...,backups\player_identity_reconciliation\identit...,103798,103798,True,PASS
2,notebooks\identity_reconciliation_backups\iden...,backups\player_identity_reconciliation\identit...,587144,587144,True,PASS
3,notebooks\identity_reconciliation_backups\play...,backups\player_identity_reconciliation\identit...,587144,587144,True,PASS
4,notebooks\identity_reconciliation_backups\regi...,backups\player_identity_reconciliation\identit...,857,857,True,PASS
5,notebooks\identity_reconciliation_backups\regi...,backups\player_identity_reconciliation\identit...,1305,1305,True,PASS
6,notebooks\identity_reconciliation_backups\regi...,backups\player_identity_reconciliation\identit...,268782,268782,True,PASS
7,notebooks\identity_reconciliation_backups\regi...,backups\player_identity_reconciliation\identit...,334688,334688,True,PASS
8,notebooks\identity_reconciliation_backups\regi...,backups\player_identity_reconciliation\identit...,277456,277456,True,PASS
9,notebooks\identity_reconciliation_backups\regi...,backups\player_identity_reconciliation\identit...,250683,250683,True,PASS



ALL COPIED FILES VERIFIED SUCCESSFULLY.
Original files remain untouched.
Manifest: C:\CricIQ\backups\notebook_cleanup_copy_manifest.csv
